# Imports

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple, Union
import hashlib
import json

import numpy as np
from numba import njit

import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

import os 
import time

from scipy.optimize import differential_evolution, minimize

from scipy.interpolate import CubicSpline

In [ ]:
import time
from functools import wraps

def timeit(label: Optional[str] = None):
    def decorator(func):
        def wrapper(*args, **kwargs):
            t0 = time.perf_counter()
            out = func(*args, **kwargs)
            t1 = time.perf_counter()
            name = label or func.__name__
            print(f"{name} {t1 - t0:.4f} s")
            return out
        return wrapper
    return decorator
    
# def timeit(label=None):
#     def decorator(func):
#         @wraps(func)
#         def wrapper(*args, **kwargs):
#             t0 = time.perf_counter()
#             result = func(*args, **kwargs)
#             t1 = time.perf_counter()
#             name = label or func.__name__
#             print(f"[{name}] took {t1 - t0:.6f} s")
#             return result
#         return wrapper
#     return decorator

In [ ]:
def read_fd(obj, width=None, factor=None):
    index = np.argmax(obj.data[0])
    if width is None:
        width = np.min([index, len(obj.data[0])-index])
    fwd = np.zeros([3, width])
    bkd = np.zeros([3, width])
    for i in range(3):
        fwd[i] = obj.data[i][index-width:index]
        bkd[i] = obj.data[i][index:index+width]
    if factor is not None:
        fwd[1] /= factor
        bkd[1] /= factor
    return fwd, bkd        

def find_all_sign_change_indices(arr):
    signs = np.sign(arr)  # Get sign of elements (-1, 0, or 1)
    changes = np.where(signs[:-1] != signs[1:])[0]  # Compare consecutive elements
    return changes
    
def find_thres(ph, amp, thres=80):
    ph_offset = ph - ph[0] + 90
    index = find_all_sign_change_indices(ph_offset-thres)
    if len(index) == 1:
        return amp[index][0]
    elif len(index) > 1:
        return amp[index[0]]
    else:
        return np.nan
    
def conver_unit(p, limits, reverse=False):
    '''
    Project  p from (0, 1) to (p_min, p_max).
    '''
    out = np.zeros_like(p)
    for i in range(len(p)):
        if reverse is False:
            p_min, p_max = limits[i]
            out[i] = p_min + (p_max - p_min) * p[i]
        else:
            p_min, p_max = limits[i]
            out[i] = (p[i] - p_min) / (p_max - p_min)
    return out

# Helper functions

## Configuration containers

In [ ]:
# ============================================================
# 1) CLEAN RK45 APPROACH SOLVER (NO NOISE IN THE SOLVER)
# ============================================================
@dataclass
class NormalizedSPMParams:
    """
    Dimensionless / normalized SPM parameters.

    The normalized equation of motion is
        z'' + z'/Q + z = F_ts(delta) + F_drive(t)
    with
        delta_hat = z_hat + d_hat .
    """
    Q: float
    C1: float
    C2: float
    C3: float
    a0: float
    omega_drive_hat: float = 1.0
    omega_ref_hat: float = 1.0


@dataclass
class RK45Options:
    """Controls for the RK45 steady-state solver."""
    N_cycles: int = 20
    max_windows: int = 40
    settle_windows: int = 1
    tol_A: float = 1e-4
    tol_phi_deg: float = 1e-2
    consecutive: int = 2
    demod_per_ref_cycle: int = 40
    rtol: float = 1e-6
    atol: float = 1e-9
    dt0: float = 2e-2
    dt_min: float = 1e-10
    dt_max: float = 3e-1
    max_steps_per_window: int = 200000

## Equation of motion

In [ ]:
@njit(cache=True)
def F_DMT_hat(delta_hat, C1, C2, a0):
    if delta_hat > 0.0:
        return -C1 / (delta_hat + a0) ** 2
    else:
        return -C1 / a0 ** 2 + C2 * (a0 - delta_hat) ** 1.5


@njit(cache=True)
def rhs_hat(
    t_hat,
    z_hat,
    v_hat,
    d_hat,
    Q,
    C1,
    C2,
    C3,
    a0,
    omega_drive_hat,
):
    delta_hat = z_hat + d_hat
    F_contact = F_DMT_hat(delta_hat, C1, C2, a0)
    F_drive = -C3 * np.cos(omega_drive_hat * t_hat)
    dz = v_hat
    dv = F_contact + F_drive - v_hat / Q - z_hat
    return dz, dv

## RK45 solver

In [ ]:
@njit(cache=True)
def rk45_step_dopri(
    t,
    z,
    v,
    dt,
    d_hat,
    Q,
    C1,
    C2,
    C3,
    a0,
    omega_drive_hat,
    rtol,
    atol,
):
    """One Dormand-Prince RK45 step with a scalar error estimate."""
    c2 = 1.0 / 5.0
    c3 = 3.0 / 10.0
    c4 = 4.0 / 5.0
    c5 = 8.0 / 9.0
    c6 = 1.0

    a21 = 1.0 / 5.0

    a31 = 3.0 / 40.0
    a32 = 9.0 / 40.0

    a41 = 44.0 / 45.0
    a42 = -56.0 / 15.0
    a43 = 32.0 / 9.0

    a51 = 19372.0 / 6561.0
    a52 = -25360.0 / 2187.0
    a53 = 64448.0 / 6561.0
    a54 = -212.0 / 729.0

    a61 = 9017.0 / 3168.0
    a62 = -355.0 / 33.0
    a63 = 46732.0 / 5247.0
    a64 = 49.0 / 176.0
    a65 = -5103.0 / 18656.0

    b1 = 35.0 / 384.0
    b3 = 500.0 / 1113.0
    b4 = 125.0 / 192.0
    b5 = -2187.0 / 6784.0
    b6 = 11.0 / 84.0

    b1s = 5179.0 / 57600.0
    b3s = 7571.0 / 16695.0
    b4s = 393.0 / 640.0
    b5s = -92097.0 / 339200.0
    b6s = 187.0 / 2100.0
    b7s = 1.0 / 40.0

    dz1, dv1 = rhs_hat(t, z, v, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

    z2 = z + dt * a21 * dz1
    v2 = v + dt * a21 * dv1
    dz2, dv2 = rhs_hat(t + c2 * dt, z2, v2, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

    z3 = z + dt * (a31 * dz1 + a32 * dz2)
    v3 = v + dt * (a31 * dv1 + a32 * dv2)
    dz3, dv3 = rhs_hat(t + c3 * dt, z3, v3, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

    z4 = z + dt * (a41 * dz1 + a42 * dz2 + a43 * dz3)
    v4 = v + dt * (a41 * dv1 + a42 * dv2 + a43 * dv3)
    dz4, dv4 = rhs_hat(t + c4 * dt, z4, v4, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

    z5i = z + dt * (a51 * dz1 + a52 * dz2 + a53 * dz3 + a54 * dz4)
    v5i = v + dt * (a51 * dv1 + a52 * dv2 + a53 * dv3 + a54 * dv4)
    dz5, dv5 = rhs_hat(t + c5 * dt, z5i, v5i, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

    z6 = z + dt * (a61 * dz1 + a62 * dz2 + a63 * dz3 + a64 * dz4 + a65 * dz5)
    v6 = v + dt * (a61 * dv1 + a62 * dv2 + a63 * dv3 + a64 * dv4 + a65 * dv5)
    dz6, dv6 = rhs_hat(t + c6 * dt, z6, v6, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

    z5 = z + dt * (b1 * dz1 + b3 * dz3 + b4 * dz4 + b5 * dz5 + b6 * dz6)
    v5 = v + dt * (b1 * dv1 + b3 * dv3 + b4 * dv4 + b5 * dv5 + b6 * dv6)

    dz7, dv7 = rhs_hat(t + dt, z5, v5, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

    z4s = z + dt * (b1s * dz1 + b3s * dz3 + b4s * dz4 + b5s * dz5 + b6s * dz6 + b7s * dz7)
    v4s = v + dt * (b1s * dv1 + b3s * dv3 + b4s * dv4 + b5s * dv5 + b6s * dv6 + b7s * dv7)

    ez = z5 - z4s
    ev = v5 - v4s

    sz = atol + rtol * max(abs(z), abs(z5))
    sv = atol + rtol * max(abs(v), abs(v5))
    err = np.sqrt(0.5 * ((ez / sz) ** 2 + (ev / sv) ** 2))

    return z5, v5, err


@njit(cache=True)
def demod_add_segment(X, Y, z0, z1, t0, t1, omega_ref_hat):
    """
    Trapezoidal accumulation of the lock-in X/Y components over one segment.
    """
    c0 = np.cos(omega_ref_hat * t0)
    s0 = np.sin(omega_ref_hat * t0)
    c1 = np.cos(omega_ref_hat * t1)
    s1 = np.sin(omega_ref_hat * t1)

    X += 0.5 * (z0 * c0 + z1 * c1) * (t1 - t0)
    Y += 0.5 * (z0 * s0 + z1 * s1) * (t1 - t0)
    return X, Y


@njit(cache=True)
def run_one_window_rk45(
    t_start,
    z_start,
    v_start,
    d_hat,
    Q,
    C1,
    C2,
    C3,
    a0,
    omega_drive_hat,
    omega_ref_hat,
    T_win,
    demod_dt_hat,
    rtol,
    atol,
    dt0,
    dt_min,
    dt_max,
    max_steps,
):
    t = t_start
    z = z_start
    v = v_start
    t_end = t_start + T_win

    X = 0.0
    Y = 0.0

    if demod_dt_hat <= 0.0:
        demod_dt_hat = T_win

    t_s = t_start
    z_s = z_start
    next_s = t_s + demod_dt_hat

    F0 = F_DMT_hat(z + d_hat, C1, C2, a0)
    F_int = 0.0
    F_min = F0
    F_max = F0

    safety = 0.9
    fac_min = 0.2
    fac_max = 5.0
    p = 5.0

    dt = dt0
    steps = 0

    while t < t_end and steps < max_steps:
        if dt < dt_min:
            dt = dt_min
        if dt > dt_max:
            dt = dt_max
        if t + dt > t_end:
            dt = t_end - t

        t_prev = t
        z_prev = z
        F_prev = F_DMT_hat(z_prev + d_hat, C1, C2, a0)

        z_new, v_new, err = rk45_step_dopri(
            t, z, v, dt, d_hat, Q, C1, C2, C3, a0, omega_drive_hat, rtol, atol
        )

        accept = (err <= 1.0) or (dt <= dt_min * 1.0001)
        if accept:
            t = t + dt
            z = z_new
            v = v_new

            F_cur = F_DMT_hat(z + d_hat, C1, C2, a0)
            F_int += 0.5 * (F_prev + F_cur) * (t - t_prev)

            if F_cur < F_min:
                F_min = F_cur
            if F_cur > F_max:
                F_max = F_cur

            while next_s <= t:
                denom = t - t_prev
                if denom <= 0.0:
                    z_b = z
                else:
                    frac = (next_s - t_prev) / denom
                    if frac < 0.0:
                        frac = 0.0
                    elif frac > 1.0:
                        frac = 1.0
                    z_b = z_prev + frac * (z - z_prev)

                X, Y = demod_add_segment(X, Y, z_s, z_b, t_s, next_s, omega_ref_hat)
                t_s = next_s
                z_s = z_b
                next_s = next_s + demod_dt_hat

            steps += 1

        if err == 0.0:
            fac = fac_max
        else:
            fac = safety * (1.0 / err) ** (1.0 / p)
            if fac < fac_min:
                fac = fac_min
            elif fac > fac_max:
                fac = fac_max
        dt = dt * fac

    if t_end > t_s:
        X, Y = demod_add_segment(X, Y, z_s, z, t_s, t_end, omega_ref_hat)

    F_mean = F_int / T_win
    return X, Y, F_mean, F_min, F_max, steps, z, v


@njit(cache=True)
def simulate_steady_state_rk45(
    z0_hat,
    v0_hat,
    d_hat,
    Q,
    C1,
    C2,
    C3,
    a0,
    omega_drive_hat,
    omega_ref_hat,
    N_cycles,
    max_windows,
    settle_windows,
    tol_A,
    tol_phi_deg,
    consecutive,
    demod_per_ref_cycle,
    rtol,
    atol,
    dt0,
    dt_min,
    dt_max,
    max_steps_per_window,
):
    T_ref = 2.0 * np.pi / omega_ref_hat
    T_win = N_cycles * T_ref

    if demod_per_ref_cycle < 5:
        demod_per_ref_cycle = 5
    demod_dt_hat = T_ref / demod_per_ref_cycle

    t = 0.0
    z = z0_hat
    v = v0_hat

    last_A = 1e30
    last_phi = 1e30
    stable = 0

    A_last = 0.0
    phi_last = 0.0
    F_mean_last = 0.0
    F_min_last = 0.0
    F_max_last = 0.0

    for widx in range(max_windows):
        X, Y, F_mean, F_min, F_max, _, z, v = run_one_window_rk45(
            t, z, v,
            d_hat, Q, C1, C2, C3, a0,
            omega_drive_hat, omega_ref_hat,
            T_win, demod_dt_hat,
            rtol, atol, dt0, dt_min, dt_max, max_steps_per_window,
        )
        t = t + T_win

        A = (2.0 / T_win) * np.sqrt(X * X + Y * Y)
        phi = np.degrees(np.arctan2(Y, X))
        if phi < 0.0:
            phi += 180.0
        if phi > 180.0:
            phi -= 180.0

        A_last = A
        phi_last = phi
        F_mean_last = F_mean
        F_min_last = F_min
        F_max_last = F_max

        if widx >= settle_windows:
            dA = abs(A - last_A)
            dphi = abs(phi - last_phi)
            if dA < tol_A and dphi < tol_phi_deg:
                stable += 1
            else:
                stable = 0

            if stable >= consecutive:
                return A, phi, F_mean, F_min, F_max, (widx + 1), z, v

        last_A = A
        last_phi = phi

    return A_last, phi_last, F_mean_last, F_min_last, F_max_last, max_windows, z, v


## Approaching curve

In [ ]:
class SPMRK45ApproachModel:
    """
    Clean approach-curve simulator with no solver-side force noise.
    """

    def __init__(self, params: NormalizedSPMParams, solver: Optional[RK45Options] = None):
        self.params = params
        self.solver = solver if solver is not None else RK45Options()

    def simulate_at_height(
        self,
        d_hat,
        z0_hat=0.0,
        v0_hat=0.0,
        omega_drive_hat=None,
        omega_ref_hat=None,
    ):
        if omega_drive_hat is None:
            omega_drive_hat = self.params.omega_drive_hat
        if omega_ref_hat is None:
            omega_ref_hat = self.params.omega_ref_hat

        so = self.solver
        p = self.params

        A, phi, F_mean, F_min, F_max, windows_used, z_final, v_final = simulate_steady_state_rk45(
            z0_hat=float(z0_hat),
            v0_hat=float(v0_hat),
            d_hat=float(d_hat),
            Q=float(p.Q),
            C1=float(p.C1),
            C2=float(p.C2),
            C3=float(p.C3),
            a0=float(p.a0),
            omega_drive_hat=float(omega_drive_hat),
            omega_ref_hat=float(omega_ref_hat),
            N_cycles=int(so.N_cycles),
            max_windows=int(so.max_windows),
            settle_windows=int(so.settle_windows),
            tol_A=float(so.tol_A),
            tol_phi_deg=float(so.tol_phi_deg),
            consecutive=int(so.consecutive),
            demod_per_ref_cycle=int(so.demod_per_ref_cycle),
            rtol=float(so.rtol),
            atol=float(so.atol),
            dt0=float(so.dt0),
            dt_min=float(so.dt_min),
            dt_max=float(so.dt_max),
            max_steps_per_window=int(so.max_steps_per_window),
        )

        return {
            "d_hat": float(d_hat),
            "A_hat": float(A),
            "phi_deg": float(phi),
            "F_mean_hat": float(F_mean),
            "F_min_hat": float(F_min),
            "F_max_hat": float(F_max),
            "windows_used": int(windows_used),
            "z_final_hat": float(z_final),
            "v_final_hat": float(v_final),
        }

    def approach_curve(
        self,
        d_start_hat,
        d_end_hat,
        n_points=256,
        z0_hat=0.0,
        v0_hat=0.0,
        warm_start=True,
        omega_drive_hat=None,
        omega_ref_hat=None,
    ):
        d_vals = np.linspace(float(d_start_hat), float(d_end_hat), int(n_points))

        A_vals = np.empty_like(d_vals)
        phi_vals = np.empty_like(d_vals)
        F_mean_vals = np.empty_like(d_vals)
        F_min_vals = np.empty_like(d_vals)
        F_max_vals = np.empty_like(d_vals)
        windows_vals = np.empty(d_vals.size, dtype=np.int64)

        z_cur = float(z0_hat)
        v_cur = float(v0_hat)

        for i, d_hat in enumerate(d_vals):
            out = self.simulate_at_height(
                d_hat=float(d_hat),
                z0_hat=z_cur,
                v0_hat=v_cur,
                omega_drive_hat=omega_drive_hat,
                omega_ref_hat=omega_ref_hat,
            )

            A_vals[i] = out["A_hat"]
            phi_vals[i] = out["phi_deg"]
            F_mean_vals[i] = out["F_mean_hat"]
            F_min_vals[i] = out["F_min_hat"]
            F_max_vals[i] = out["F_max_hat"]
            windows_vals[i] = out["windows_used"]

            if warm_start:
                z_cur = out["z_final_hat"]
                v_cur = out["v_final_hat"]

        return {
            "d_hat": d_vals,
            "A_hat": A_vals,
            "phi_deg": phi_vals,
            "F_mean_hat": F_mean_vals,
            "F_min_hat": F_min_vals,
            "F_max_hat": F_max_vals,
            "windows_used": windows_vals,
        }

In [ ]:
# # ============================================================
# # 2) FD LOOKUP + PI SCANNER WITH NOISE IN THE PI CHANNEL ONLY
# # ============================================================
# @dataclass
# class PIDriftNoiseOptions:
#     """
#     Low-frequency noise injected into the PI amplitude channel.

#     This is meant to emulate false feedback / drift-like scans at large
#     setpoint amplitudes, where A_set is close to the free amplitude A0.

#     The controller uses
#         A_ctrl = A_meas + drift + sinusoid + white
#     and then
#         err = A_set - A_ctrl

#     Parameters
#     ----------
#     drift_rel_std : float
#         Stationary standard deviation of the slow OU drift, relative to A0.
#     drift_tau_pixels : float
#         Correlation length of the slow drift in pixels.
#     white_rel_std : float
#         Pixel-to-pixel white measurement noise, relative to A0.
#     sinusoid_rel_amp : float
#         Optional slow periodic baseline modulation, relative to A0.
#     sinusoid_period_pixels : float
#         Period of the sinusoidal baseline in pixels.
#     seed : int
#         RNG seed for reproducibility.
#     """
#     drift_rel_std: float = 2e-3
#     drift_tau_pixels: float = 200.0
#     white_rel_std: float = 0.0
#     sinusoid_rel_amp: float = 0.0
#     sinusoid_period_pixels: float = 512.0
#     seed: int = 0


# class FDCurve1D:
#     """
#     1D lookup model built from a precomputed force-distance / approach curve.

#     Parameters
#     ----------
#     d_hat : array
#         Effective gap coordinate used by the scanner.
#     A_hat, phi_deg, F_max_hat : arrays
#         Amplitude, phase, and maximum force on the same d_hat grid.
#     clip : bool
#         If True, clip queries to the available d_hat range.
#     """

#     def __init__(self, d_hat, A_hat, phi_deg, F_max_hat, clip=True):
#         d_hat = np.asarray(d_hat, dtype=float)
#         A_hat = np.asarray(A_hat, dtype=float)
#         phi_deg = np.asarray(phi_deg, dtype=float)
#         F_max_hat = np.asarray(F_max_hat, dtype=float)

#         if not (d_hat.ndim == A_hat.ndim == phi_deg.ndim == F_max_hat.ndim == 1):
#             raise ValueError("All inputs must be 1D arrays.")
#         if not (d_hat.size == A_hat.size == phi_deg.size == F_max_hat.size):
#             raise ValueError("All arrays must have the same length.")
#         if d_hat.size < 2:
#             raise ValueError("Need at least two FD points.")

#         order = np.argsort(d_hat)
#         self.d_hat = d_hat[order]
#         self.A_hat = A_hat[order]
#         self.phi_deg = phi_deg[order]
#         self.F_max_hat = F_max_hat[order]
#         self.clip = bool(clip)

#     def _prep_query(self, d_query):
#         d_query = np.asarray(d_query, dtype=float)
#         if self.clip:
#             d_query = np.clip(d_query, self.d_hat[0], self.d_hat[-1])
#         return d_query

#     def evaluate(self, d_query):
#         d_query = self._prep_query(d_query)
#         A = np.interp(d_query, self.d_hat, self.A_hat)
#         phi = np.interp(d_query, self.d_hat, self.phi_deg)
#         F = np.interp(d_query, self.d_hat, self.F_max_hat)
#         return A, phi, F


# def _lpf1_step(y, x, dt, tau):
#     if tau is None or tau <= 0.0:
#         return x
#     alpha = dt / (tau + dt)
#     return y + alpha * (x - y)


# def _lpf2_step(y, ydot, x, dt, wn, zeta):
#     if wn is None or wn <= 0.0:
#         return x, 0.0
#     ydd = (wn * wn) * (x - y) - (2.0 * zeta * wn) * ydot
#     ydot_new = ydot + dt * ydd
#     y_new = y + dt * ydot_new
#     return y_new, ydot_new


# def _ou_step_stationary(x, std, tau, dt, rng):
#     """
#     Ornstein-Uhlenbeck update with stationary std = std.
#     """
#     if std <= 0.0:
#         return 0.0
#     if tau is None or tau <= 0.0:
#         return std * rng.standard_normal()

#     alpha = np.exp(-dt / tau)
#     sigma = std * np.sqrt(max(0.0, 1.0 - alpha * alpha))
#     return alpha * x + sigma * rng.standard_normal()

In [ ]:
"""
Approach-curve library + cross-drive interpolation.

Strategy
--------
Every measured approach curve is single-valued in z when parameterized by
the slow z-piezo. Store curves in physical units (nm, deg) per drive A0.

To interpolate to an arbitrary drive A0*:
  1) Resample each curve onto a common normalized grid u = d / A0,
     storing the normalized amplitude a = A / A0 and the phase phi.
  2) Linearly interpolate a(u, A0) and phi(u, A0) in A0 at fixed u.
  3) Convert back: d = u * A0*,  A = a * A0*.

Optional force F(d) is carried through if provided.

Public API
----------
FDCurve1D                 : single-curve interpolator with evaluate(d) -> (A, phi, F)
ApproachCurveLibrary      : add curves, query at any drive, returns FDCurve1D
make_normalized_fd_lookup : adapter that wraps the library in the normalized-units
                            API expected by ScannerFD (matches calibrate_scanner.py)
detect_bistability        : utility to flag curves with phase jumps before fitting
"""
from __future__ import annotations

import numpy as np
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter1d
from typing import Optional, Tuple, Dict, List, Callable


# -----------------------------------------------------------------------------
# Smoothing helper (used by FDCurve1D and ApproachCurveLibrary)
# -----------------------------------------------------------------------------

def _smooth_along_d(values: np.ndarray, method: Optional[str],
                    window: int, polyorder: int, sigma: float) -> np.ndarray:
    """Smooth `values` along its single axis. Returns a new array.

    method = None       : no-op
    method = 'savgol'   : Savitzky–Golay filter (preserves features sharper
                          than `window`, kills noise smaller than that)
    method = 'gaussian' : Gaussian smoothing with std `sigma` (in samples)
    """
    if method is None or method == 'none':
        return np.asarray(values, dtype=float)
    v = np.asarray(values, dtype=float)
    n = v.size
    if n < 5:
        return v
    if method == 'savgol':
        w = int(window)
        if w % 2 == 0:
            w += 1
        w = min(w, n if n % 2 == 1 else n - 1)
        if w <= polyorder + 1:
            return v
        return savgol_filter(v, window_length=w, polyorder=int(polyorder))
    if method == 'gaussian':
        return gaussian_filter1d(v, sigma=float(sigma))
    raise ValueError(f"Unknown smooth method '{method}'")


# -----------------------------------------------------------------------------
# Single-curve evaluator
# -----------------------------------------------------------------------------

class FDCurve1D:
    """Interpolated A(d), phi(d), F(d) at a single drive amplitude.

    Conventions
    -----------
    Inputs in any consistent unit system (typically nm, degrees).
    d increases away from the surface; the largest d in the curve is treated
    as the "free" point, A(d_max) ~ A0.

    Out-of-range behavior
    ---------------------
    d > d_max : A clamped to A0 (free), phi clamped to phi(d_max), F -> 0.
    d < d_min : edge-clamped (no extrapolation into untrusted near-contact regime).
    """

    def __init__(self, d, A, phi, F=None, *, A0: Optional[float] = None,
                 kind: str = 'linear',
                 smooth_method: Optional[str] = None,
                 smooth_window: int = 7,
                 smooth_polyorder: int = 3,
                 smooth_sigma: float = 2.0):
        d = np.asarray(d, dtype=float)
        A = np.asarray(A, dtype=float)
        phi = np.asarray(phi, dtype=float)
        F = np.zeros_like(d) if F is None else np.asarray(F, dtype=float)

        order = np.argsort(d)
        self.d   = d[order]
        self.A   = A[order]
        self.phi = phi[order]
        self.F   = F[order]

        # Optional smoothing along d (after sorting). d itself stays exact;
        # only A, phi, F get smoothed.
        if smooth_method is not None and smooth_method != 'none':
            self.A   = _smooth_along_d(self.A,   smooth_method, smooth_window,
                                       smooth_polyorder, smooth_sigma)
            self.phi = _smooth_along_d(self.phi, smooth_method, smooth_window,
                                       smooth_polyorder, smooth_sigma)
            self.F   = _smooth_along_d(self.F,   smooth_method, smooth_window,
                                       smooth_polyorder, smooth_sigma)

        self.A0 = float(self.A[-1]) if A0 is None else float(A0)
        self.kind = kind

        self._A = interp1d(
            self.d, self.A, kind=kind, bounds_error=False,
            fill_value=(self.A[0], self.A0),  # left = nearest, right = free
        )
        self._phi = interp1d(
            self.d, self.phi, kind=kind, bounds_error=False,
            fill_value=(self.phi[0], self.phi[-1]),
        )
        self._F = interp1d(
            self.d, self.F, kind=kind, bounds_error=False,
            fill_value=(self.F[0], 0.0),
        )

    def evaluate(self, d):
        """Return (A, phi, F). Scalar in -> scalar out; array in -> array out."""
        d_arr = np.atleast_1d(d).astype(float)
        A   = self._A(d_arr)
        phi = self._phi(d_arr)
        F   = self._F(d_arr)
        if np.isscalar(d):
            return float(A[0]), float(phi[0]), float(F[0])
        return A, phi, F


# -----------------------------------------------------------------------------
# Library
# -----------------------------------------------------------------------------

class ApproachCurveLibrary:
    """Collection of approach curves at multiple drive amplitudes (nm).

    Each curve is stored as 1D arrays (d_nm, A_nm, phi_deg, optional F).
    The curves do NOT need to share a common d-grid or have the same length.
    """

    def __init__(self,
                 smooth_method: Optional[str] = None,
                 smooth_window: int = 7,
                 smooth_polyorder: int = 3,
                 smooth_sigma: float = 2.0):
        """Optional smoothing applied to every curve at ingestion.

        smooth_method = 'savgol'   : Savitzky–Golay (default window=7, poly=3)
        smooth_method = 'gaussian' : Gaussian (default sigma=2 samples)
        smooth_method = None       : no smoothing (default)

        Smoothing is applied ONCE at ingestion (in `add` / `from_dict`) so the
        stored arrays — and every curve returned by `fd_at_drive` — are
        already smooth.
        """
        self._curves: Dict[float, Dict[str, Optional[np.ndarray]]] = {}
        self.smooth_method = smooth_method
        self.smooth_window = int(smooth_window)
        self.smooth_polyorder = int(smooth_polyorder)
        self.smooth_sigma = float(smooth_sigma)

    # ----- ingestion -----

    def add(self, drive_nm: float, d_nm, A_nm, phi_deg, F=None) -> None:
        """Add one approach curve at a known drive amplitude (in nm).

        Parameters
        ----------
        drive_nm : free-amplitude drive setting for this curve, nm.
        d_nm     : 1D array of z-piezo positions, nm. Increases away from surface.
        A_nm     : 1D array of measured oscillation amplitudes, nm.
        phi_deg  : 1D array of measured phases, degrees.
        F        : optional 1D array (e.g. peak force or virial), arbitrary units.
        """
        d = np.asarray(d_nm,    dtype=float).ravel()
        A = np.asarray(A_nm,    dtype=float).ravel()
        phi = np.asarray(phi_deg, dtype=float).ravel()
        if not (d.shape == A.shape == phi.shape):
            raise ValueError("d, A, phi must be the same length")
        F_arr = None
        if F is not None:
            F_arr = np.asarray(F, dtype=float).ravel()
            if F_arr.shape != d.shape:
                raise ValueError("F must match d length")

        # Drop NaNs and sort by d ascending
        mask = np.isfinite(d) & np.isfinite(A) & np.isfinite(phi)
        if F_arr is not None:
            mask &= np.isfinite(F_arr)
        d, A, phi = d[mask], A[mask], phi[mask]
        if F_arr is not None:
            F_arr = F_arr[mask]
        order = np.argsort(d)
        d_sorted, A_sorted, phi_sorted = d[order], A[order], phi[order]
        F_sorted = None if F_arr is None else F_arr[order]

        # Apply per-curve smoothing once at ingestion
        if self.smooth_method is not None and self.smooth_method != 'none':
            A_sorted = _smooth_along_d(A_sorted,
                self.smooth_method, self.smooth_window,
                self.smooth_polyorder, self.smooth_sigma)
            phi_sorted = _smooth_along_d(phi_sorted,
                self.smooth_method, self.smooth_window,
                self.smooth_polyorder, self.smooth_sigma)
            if F_sorted is not None:
                F_sorted = _smooth_along_d(F_sorted,
                    self.smooth_method, self.smooth_window,
                    self.smooth_polyorder, self.smooth_sigma)

        self._curves[float(drive_nm)] = {
            'd':   d_sorted,
            'A':   A_sorted,
            'phi': phi_sorted,
            'F':   F_sorted,
        }

    @classmethod
    def from_dict(cls, by_drive: Dict[float, Dict[str, np.ndarray]],
                  smooth_method: Optional[str] = None,
                  smooth_window: int = 7,
                  smooth_polyorder: int = 3,
                  smooth_sigma: float = 2.0) -> 'ApproachCurveLibrary':
        """Bulk-construct from {drive_nm: {'d':..., 'A':..., 'phi':..., 'F':...}}.
        Smoothing kwargs are forwarded to the constructor (applied per-curve at
        ingestion).
        """
        lib = cls(smooth_method=smooth_method, smooth_window=smooth_window,
                  smooth_polyorder=smooth_polyorder, smooth_sigma=smooth_sigma)
        for drv, c in by_drive.items():
            lib.add(drv, c['d'], c['A'], c['phi'], F=c.get('F'))
        return lib

    # ----- inspection -----

    @property
    def drives(self) -> np.ndarray:
        return np.array(sorted(self._curves.keys()))

    def get_raw(self, drive_nm: float) -> Dict[str, Optional[np.ndarray]]:
        return self._curves[float(drive_nm)]

    # ----- normalization -----

    def normalized_grid(
        self,
        n_u: int = 400,
        u_min: Optional[float] = None,
        u_max: Optional[float] = None,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """Resample all curves onto a common u = d/A0 grid.

        Returns
        -------
        drives   : (n_drives,)        sorted drive amplitudes [nm]
        u_grid   : (n_u,)             common normalized distance grid
        a_grid   : (n_drives, n_u)    normalized amplitude  A/A0
        phi_grid : (n_drives, n_u)    phase [deg]
        F_grid   : (n_drives, n_u)    force (zero where F not supplied)
        """
        drives = self.drives
        if drives.size < 2:
            raise ValueError("Library needs at least two drives to interpolate.")
        umins = [self._curves[d]['d'].min() / d for d in drives]
        umaxs = [self._curves[d]['d'].max() / d for d in drives]

        u_min = max(umins) if u_min is None else float(u_min)
        u_max = min(umaxs) if u_max is None else float(u_max)
        if not (u_max > u_min):
            raise ValueError(
                f"No common u-range across curves "
                f"(largest u_min={max(umins):.3g}, smallest u_max={min(umaxs):.3g}). "
                "Trim outlier curves or pass u_min/u_max manually."
            )
        u_grid = np.linspace(u_min, u_max, int(n_u))

        nD = drives.size
        a_grid   = np.empty((nD, n_u))
        phi_grid = np.empty((nD, n_u))
        F_grid   = np.zeros((nD, n_u))

        for i, drv in enumerate(drives):
            c = self._curves[drv]
            u = c['d'] / drv
            a = c['A'] / drv
            a_grid[i]   = np.interp(u_grid, u, a)
            phi_grid[i] = np.interp(u_grid, u, c['phi'])
            if c['F'] is not None:
                F_grid[i] = np.interp(u_grid, u, c['F'])

        return drives, u_grid, a_grid, phi_grid, F_grid

    # ----- interpolation across drives -----

    def _raw_to_fd(self, drv: float) -> FDCurve1D:
        """Wrap a stored curve in an FDCurve1D verbatim (no resampling)."""
        c = self._curves[float(drv)]
        return FDCurve1D(c['d'], c['A'], c['phi'],
                         F=c['F'], A0=float(drv))

    def fd_at_drive(self, drive_target_nm: float, n_u: int = 400) -> FDCurve1D:
        """Build an FDCurve1D for an arbitrary target drive amplitude (nm).

        Strategy:
          - Outside [drives.min, drives.max], clamp to the nearest library
            curve (returned verbatim — full d-range preserved).
          - At an exact library drive, return that curve verbatim.
          - Otherwise, linearly interpolate between the two adjacent library
            curves on the OVERLAP of their physical d-ranges. Each curve is
            evaluated on its own d-axis (no global u-grid intersection),
            so a single small-range curve cannot truncate the rest.

        Note: n_u kept for backward compatibility — used as the d-grid size
        for the local pair LERP.
        """
        drives = self.drives
        drv = float(drive_target_nm)

        # Clamp to ends
        if drv <= drives[0]:
            return self._raw_to_fd(drives[0])
        if drv >= drives[-1]:
            return self._raw_to_fd(drives[-1])

        # Locate adjacent library drives
        idx = int(np.searchsorted(drives, drv) - 1)
        drv_lo, drv_hi = float(drives[idx]), float(drives[idx + 1])

        # Exact-match short-circuit (avoids interpolation noise)
        if np.isclose(drv, drv_lo):
            return self._raw_to_fd(drv_lo)
        if np.isclose(drv, drv_hi):
            return self._raw_to_fd(drv_hi)

        c_lo = self._curves[drv_lo]
        c_hi = self._curves[drv_hi]
        w = (drv - drv_lo) / (drv_hi - drv_lo)

        # Common d-grid on the overlap of the two curves' d-supports
        d_min = float(max(c_lo['d'].min(), c_hi['d'].min()))
        d_max = float(min(c_lo['d'].max(), c_hi['d'].max()))
        if not (d_max > d_min):
            raise ValueError(
                f"Curves at drive={drv_lo} and drive={drv_hi} have no "
                f"overlapping d-range; cannot interpolate."
            )
        d_grid = np.linspace(d_min, d_max, int(n_u))

        A_lo = np.interp(d_grid, c_lo['d'], c_lo['A'])
        A_hi = np.interp(d_grid, c_hi['d'], c_hi['A'])
        phi_lo = np.interp(d_grid, c_lo['d'], c_lo['phi'])
        phi_hi = np.interp(d_grid, c_hi['d'], c_hi['phi'])
        A = (1 - w) * A_lo + w * A_hi
        phi = (1 - w) * phi_lo + w * phi_hi

        F = None
        if (c_lo['F'] is not None) and (c_hi['F'] is not None):
            F_lo = np.interp(d_grid, c_lo['d'], c_lo['F'])
            F_hi = np.interp(d_grid, c_hi['d'], c_hi['F'])
            F = (1 - w) * F_lo + w * F_hi

        return FDCurve1D(d_grid, A, phi, F=F, A0=drv)

    # convenience alias
    def __call__(self, drive_target_nm: float) -> FDCurve1D:
        return self.fd_at_drive(drive_target_nm)


# -----------------------------------------------------------------------------
# Adapter to the normalized-units API expected by ScannerFD
# -----------------------------------------------------------------------------

def make_normalized_fd_lookup(
    library: ApproachCurveLibrary,
    conv_L: float,
    conv_A: Optional[float] = None,
    conv_F: float = 1.0,
    n_u: int = 400,
) -> Callable[[float], FDCurve1D]:
    """Wrap a physical-units library in a callable returning FDCurve1D in
    the *normalized* units the ScannerFD expects internally.

    Parameters
    ----------
    library : ApproachCurveLibrary in physical units (nm, deg).
    conv_L  : length conversion (d_norm = d_nm * conv_L). In ScannerFD all
              length-like params share the same conversion factor, so pass
              scanner.conversion['A'] (or equivalently scanner.conversion['d0']).
    conv_A  : amplitude conversion (A_norm = A_nm * conv_A). Defaults to conv_L,
              which is the right thing since amplitude IS a length.
    conv_F  : force conversion. Pass 1.0 if your library doesn't carry F, or
              the relevant scanner.conversion key if it does.
    n_u     : grid size for the normalized resampling step.

    The returned callable takes a drive in *physical* nm and returns an
    FDCurve1D whose evaluate(d_hat) accepts and returns *normalized* values.
    Use this directly as `fd_lookup` in calibrate_scanner.calibrate(...).
    """
    if conv_A is None:
        conv_A = conv_L

    def lookup(drive_nm: float) -> FDCurve1D:
        fd_phys = library.fd_at_drive(float(drive_nm), n_u=n_u)
        return FDCurve1D(
            d=fd_phys.d * conv_L,
            A=fd_phys.A * conv_A,
            phi=fd_phys.phi,
            F=fd_phys.F * conv_F,
            A0=float(drive_nm) * conv_A,
        )
    return lookup


# -----------------------------------------------------------------------------
# Diagnostics
# -----------------------------------------------------------------------------

def detect_bistability(
    library: ApproachCurveLibrary,
    phase_jump_threshold_deg: float = 25.0,
    amp_jump_threshold_rel: float = 0.15,
) -> Dict[float, Dict[str, float]]:
    """Flag curves where phi or A/A0 jumps more than thresholds between
    adjacent samples in d. These are likely to be bistable and should be
    inspected (or split into branches) before being put into the library.

    Returns {drive: {'max_dphi_deg': ..., 'max_da_rel': ..., 'd_at_jump': ...}}
    only for curves that exceed at least one threshold.
    """
    flags: Dict[float, Dict[str, float]] = {}
    for drv in library.drives:
        c = library.get_raw(drv)
        d, A, phi = c['d'], c['A'], c['phi']
        if d.size < 3:
            continue
        dphi = np.abs(np.diff(phi))
        da_rel = np.abs(np.diff(A)) / float(drv)
        i_phi = int(np.argmax(dphi)) if dphi.size else 0
        i_a   = int(np.argmax(da_rel)) if da_rel.size else 0
        if dphi[i_phi] > phase_jump_threshold_deg or da_rel[i_a] > amp_jump_threshold_rel:
            phi_severity = dphi[i_phi] / phase_jump_threshold_deg
            a_severity   = da_rel[i_a] / amp_jump_threshold_rel
            i_jump = i_phi if phi_severity > a_severity else i_a
            flags[float(drv)] = {
                'max_dphi_deg': float(dphi.max()),
                'max_da_rel':   float(da_rel.max()),
                'd_at_jump':    float(d[i_jump]),
            }
    return flags



In [ ]:
# Log-spaced grid-search calibration for the SPM digital twin.
"""
calibrate_grid.py

Log-spaced grid-search calibration for the SPM digital twin.

Stage 1  (grid_search_HE)
    Search over (H, E*) - Hamaker constant and effective modulus - against
    experimental approach curves at multiple drives. R is held fixed; C1 and
    C2 are recomputed from the scanner_fd convention:
        C1 = H * R / 6
        C2 = (4/3) * E_star * sqrt(R)
    Joint loss = mean[(A_sim - A_meas)/A0]^2  +  w_phi * mean[(phi_sim - phi_meas)/180]^2
    summed across all measured curves.

Stage 2  (build_library_from_fit)
    Regenerate clean approach curves at the fitted (H, E*) on a uniform fine
    d-grid spanning each measured curve's support, and pack them into an
    ApproachCurveLibrary keyed by drive amplitude (nm).

Stage 3  (grid_search_PI / grid_search_PI_multi)
    Search over (P, I) PI gains for a ScannerFD instance against measured
    scan-line data. Loss = mean[(d_sim - d_meas)/A0]^2 + w_A * mean[(A_sim - A_meas)/A0]^2.
    The multi-line variant sums losses across scan-speed / step-shape
    combinations, which is what breaks the lock-in tau vs. z-piezo tau
    degeneracy in stage 3.

Default grid (matches the spec):
    multipliers = np.linspace(1, 9, 9)
    exponents   = np.arange(-6, -2)            # -6, -5, -4, -3
    grid        = concat(multipliers * 10**e for e in exponents) + [10.0]
    -> 37 points spanning [1e-6, 9e-3] U {10}.

Optional joblib parallelism via n_jobs (silently falls back to serial).
"""
from __future__ import annotations

import copy
import numpy as np
from typing import Dict, List, Optional, Sequence, Tuple


# -----------------------------------------------------------------------------
# Log-spaced grid builder
# -----------------------------------------------------------------------------

def make_log_grid(
    multipliers: Sequence[float] = np.linspace(1.0, 9.0, 9),
    exponents: Sequence[int] = np.arange(-6, -2),
    tail: Sequence[float] = (10.0,),
) -> np.ndarray:
    """Build a log-spaced grid: outer of multipliers x 10**exponents, plus tail.

    Default = 37 points spanning [1e-6, 9e-3] U {10}.
    """
    parts = [np.asarray(multipliers, dtype=float) * 10.0 ** int(e)
             for e in exponents]
    if tail:
        parts.append(np.asarray(tail, dtype=float))
    return np.concatenate(parts)


# =============================================================================
# Stage 1 - (H, E*) grid search against measured approach curves
# =============================================================================

def _set_HEstar(base_params, H_new: float, E_star_new: float, R_fixed: float):
    """Return a copy of base_params with C1, C2 recomputed for new H and E*.

    Tries dataclasses.replace first; falls back to a shallow copy + attribute
    write. base_params must expose .C1 and .C2 (NormalizedSPMParams).
    """
    C1 = float(H_new) * float(R_fixed) / 6.0
    C2 = (4.0 / 3.0) * float(E_star_new) * np.sqrt(float(R_fixed))
    try:
        from dataclasses import is_dataclass, replace
        if is_dataclass(base_params):
            return replace(base_params, C1=C1, C2=C2)
    except Exception:
        pass
    p = copy.copy(base_params)
    p.C1 = C1
    p.C2 = C2
    return p


def _simulate_at_d_array(
    model,
    d_array_desc: np.ndarray,
    omega_drive_hat: Optional[float] = None,
    omega_ref_hat: Optional[float] = None,
    z0: float = 0.0,
    v0: float = 0.0,
) -> Tuple[np.ndarray, np.ndarray]:
    """Run model.simulate_at_height at each d in d_array_desc.

    Sweep is in the order given (descending in d is recommended for stable
    warm-starting). Returns (A, phi) in the same order as d_array_desc.
    """
    n = len(d_array_desc)
    A = np.empty(n)
    phi = np.empty(n)
    z_cur, v_cur = float(z0), float(v0)
    for i, d in enumerate(d_array_desc):
        out = model.simulate_at_height(
            d_hat=float(d),
            z0_hat=z_cur,
            v0_hat=v_cur,
            omega_drive_hat=omega_drive_hat,
            omega_ref_hat=omega_ref_hat,
        )
        A[i] = out["A_hat"]
        phi[i] = out["phi_deg"]
        z_cur = out["z_final_hat"]
        v_cur = out["v_final_hat"]
    return A, phi


def _curve_loss(
    model,
    d_meas: np.ndarray,
    A_meas: np.ndarray,
    phi_meas: np.ndarray,
    A0_hat: float,
    omega_drive_hat: Optional[float] = None,
    omega_ref_hat: Optional[float] = None,
    w_phi: float = 1.0,
) -> float:
    """Joint MSE on (A/A0) + w_phi * (phi/180) for one approach curve."""
    d = np.asarray(d_meas, dtype=float)
    A = np.asarray(A_meas, dtype=float)
    phi = np.asarray(phi_meas, dtype=float)

    # Sweep descending in d for warm-start stability
    order = np.argsort(d)[::-1]
    d_desc, A_desc, phi_desc = d[order], A[order], phi[order]

    A_sim, phi_sim = _simulate_at_d_array(
        model, d_desc, omega_drive_hat, omega_ref_hat,
    )

    res_A = (A_sim - A_desc) / float(A0_hat)
    res_phi = (phi_sim - phi_desc) / 180.0
    return float(np.mean(res_A ** 2) + float(w_phi) * np.mean(res_phi ** 2))


def grid_search_HE(
    measured_curves: List[Dict],
    base_params,
    R_fixed: float,
    SPMRK45ApproachModel,
    H_grid: Optional[np.ndarray] = None,
    E_star_grid: Optional[np.ndarray] = None,
    solver=None,
    w_phi: float = 1.0,
    n_jobs: int = 1,
    verbose: bool = True,
) -> Tuple[np.ndarray, float, float]:
    """Grid-search (H, E*) against a list of measured approach curves.

    Parameters
    ----------
    measured_curves : list of dicts, each with keys
        'd_hat'           : (n,) array of normalized z-piezo positions
        'A_hat'           : (n,) array of normalized amplitudes
        'phi_deg'         : (n,) array of phases [deg]
        'A0_hat'          : scalar, normalized free amplitude for this curve
        'omega_drive_hat' : optional, normalized drive freq for this curve
        'omega_ref_hat'   : optional, normalized reference freq for this curve
    base_params : NormalizedSPMParams (passed through to SPMRK45ApproachModel).
    R_fixed     : tip radius in the SAME normalized units used by C1, C2.
    SPMRK45ApproachModel : class (passed in to avoid a circular import).
    H_grid, E_star_grid  : 1D arrays. Default = make_log_grid().
    solver               : RK45Options or None.
    w_phi                : weight on phi residual (default 1.0).
    n_jobs               : if > 1 and joblib is installed, run grid in parallel.
    verbose              : print per-row best so far.

    Returns
    -------
    loss_grid : (nH, nE) total loss summed across measured curves
    best_H, best_E_star : floats at np.nanargmin(loss_grid)
    """
    if H_grid is None:
        H_grid = make_log_grid()
    if E_star_grid is None:
        E_star_grid = make_log_grid()
    H_grid = np.asarray(H_grid, dtype=float)
    E_star_grid = np.asarray(E_star_grid, dtype=float)

    nH, nE = H_grid.size, E_star_grid.size
    loss_grid = np.full((nH, nE), np.inf)

    def _eval_point(iH: int, jE: int) -> Tuple[int, int, float]:
        params = _set_HEstar(base_params, H_grid[iH], E_star_grid[jE], R_fixed)
        try:
            model = SPMRK45ApproachModel(params, solver=solver)
            total = 0.0
            for c in measured_curves:
                total += _curve_loss(
                    model, c['d_hat'], c['A_hat'], c['phi_deg'],
                    c['A0_hat'],
                    c.get('omega_drive_hat'),
                    c.get('omega_ref_hat'),
                    w_phi=w_phi,
                )
            return iH, jE, float(total)
        except Exception:
            return iH, jE, float('inf')

    if n_jobs and n_jobs > 1:
        try:
            from joblib import Parallel, delayed
            idx_pairs = [(i, j) for i in range(nH) for j in range(nE)]
            results = Parallel(n_jobs=n_jobs, verbose=10 if verbose else 0)(
                delayed(_eval_point)(i, j) for i, j in idx_pairs
            )
            for iH, jE, val in results:
                loss_grid[iH, jE] = val
        except ImportError:
            if verbose:
                print("joblib not available; falling back to serial.")
            n_jobs = 1

    if not (n_jobs and n_jobs > 1):
        for iH in range(nH):
            for jE in range(nE):
                _, _, val = _eval_point(iH, jE)
                loss_grid[iH, jE] = val
            if verbose:
                best = np.unravel_index(np.nanargmin(loss_grid), loss_grid.shape)
                print(f"H row {iH+1:>3}/{nH}: best so far "
                      f"H={H_grid[best[0]]:.3g}, E*={E_star_grid[best[1]]:.3g}, "
                      f"loss={loss_grid[best]:.6g}")

    best = np.unravel_index(np.nanargmin(loss_grid), loss_grid.shape)
    return loss_grid, float(H_grid[best[0]]), float(E_star_grid[best[1]])


# =============================================================================
# Stage 2 - build approach-curve library at the fitted (H, E*)
# =============================================================================

def build_library_from_fit(
    measured_curves: List[Dict],
    base_params,
    R_fixed: float,
    H_fit: float,
    E_star_fit: float,
    SPMRK45ApproachModel,
    ApproachCurveLibrary,
    n_points: int = 256,
    conv_L: float = 1.0,
    solver=None,
    smooth_method: Optional[str] = None,
    smooth_window: int = 7,
    smooth_polyorder: int = 3,
    smooth_sigma: float = 2.0,
):
    """Regenerate a clean curve at (H_fit, E_star_fit) for each measured drive,
    then pack into an ApproachCurveLibrary keyed by drive amplitude in nm.

    Parameters
    ----------
    conv_L : length conversion such that d_norm = d_nm * conv_L. The simulator
             works in normalized units; the library stores nm/deg. Pass
             scanner.conversion['A'] (or equivalently 'd0') to integrate with
             ScannerFD via approach_curves.make_normalized_fd_lookup.

    Returns
    -------
    library : ApproachCurveLibrary
    """
    params = _set_HEstar(base_params, H_fit, E_star_fit, R_fixed)
    model = SPMRK45ApproachModel(params, solver=solver)

    by_drive: Dict[float, Dict[str, np.ndarray]] = {}
    for c in measured_curves:
        d_meas = np.asarray(c['d_hat'], dtype=float)
        d_min, d_max = float(d_meas.min()), float(d_meas.max())
        out = model.approach_curve(
            d_start_hat=d_max,
            d_end_hat=d_min,
            n_points=int(n_points),
            warm_start=True,
            omega_drive_hat=c.get('omega_drive_hat'),
            omega_ref_hat=c.get('omega_ref_hat'),
        )
        # normalized -> nm for length-like quantities
        d_nm = np.asarray(out['d_hat'], dtype=float) / float(conv_L)
        A_nm = np.asarray(out['A_hat'], dtype=float) / float(conv_L)
        phi_dg = np.asarray(out['phi_deg'], dtype=float)
        # F is left in whatever normalized units the simulator uses; the library
        # callers convert via conv_F when wrapping into ScannerFD.
        F_arr = np.asarray(out.get('F_mean_hat', np.zeros_like(d_nm)), dtype=float)
        drive_nm = float(c['A0_hat']) / float(conv_L)

        by_drive[drive_nm] = {'d': d_nm, 'A': A_nm, 'phi': phi_dg, 'F': F_arr}

    return ApproachCurveLibrary.from_dict(
        by_drive,
        smooth_method=smooth_method,
        smooth_window=smooth_window,
        smooth_polyorder=smooth_polyorder,
        smooth_sigma=smooth_sigma,
    )


# =============================================================================
# Stage 3 - PI gain grid search against measured scan-line data
# =============================================================================

def _line_loss(
    scanner,
    h_hat: np.ndarray,
    d_meas: np.ndarray,
    A_meas: np.ndarray,
    P: float,
    I: float,
    w_A: float = 0.3,
    line_kwargs: Optional[Dict] = None,
) -> float:
    """Per-line loss for stage 3.

    Loss = mean[(d_sim - d_meas)/A0]^2 + w_A * mean[(A_sim - A_meas)/A0]^2

    Both residuals are scaled by A0 so the d-trace and A-trace are dimensionally
    comparable. line_kwargs holds every scan_line_pi argument other than P and I
    (scan_speed_hat, dx_hat, filter taus/orders, T_I, setpoint, ...).
    """
    line_kwargs = dict(line_kwargs or {})
    line_kwargs.update(P=float(P), I=float(I))
    out = scanner.scan_line_pi(h_hat=h_hat, **line_kwargs)
    d_sim = out['d_hat']
    A_sim = out['A_hat']

    A0 = float(scanner.param_norm['A'])
    d_meas = np.asarray(d_meas, dtype=float)
    A_meas = np.asarray(A_meas, dtype=float)

    res_d = (d_sim - d_meas) / A0
    res_A = (A_sim - A_meas) / A0
    return float(np.mean(res_d ** 2) + float(w_A) * np.mean(res_A ** 2))


def grid_search_PI_multi(
    scanner,
    line_data: List[Dict],
    P_grid: Optional[np.ndarray] = None,
    I_grid: Optional[np.ndarray] = None,
    w_A: float = 0.3,
    n_jobs: int = 1,
    verbose: bool = True,
) -> Tuple[np.ndarray, float, float]:
    """Grid-search PI gains across a list of measured scan lines.

    Parameters
    ----------
    line_data : list of dicts, each with keys
        'h_hat'       : (n,) topography (normalized)
        'd_meas'      : (n,) measured d trace (normalized)
        'A_meas'      : (n,) measured A trace (normalized)
        'line_kwargs' : dict of scan_line_pi kwargs other than P, I
                        (scan_speed_hat, dx_hat, A_filter_order, tau_A,
                         z_filter_order, tau_z, T_I, setpoint, ...).
                        These should be FIXED to the values you've already
                        identified for the controller hardware (lock-in tau,
                        z bandwidth, etc.) and the operating point you ran on
                        the real instrument. P and I are the two unknowns.
        'weight'      : optional scalar (default 1.0)

    Returns
    -------
    loss_grid : (nP, nI) total loss summed across line_data entries
    best_P, best_I : floats at np.nanargmin(loss_grid)
    """
    if P_grid is None: P_grid = make_log_grid()
    if I_grid is None: I_grid = make_log_grid()
    P_grid = np.asarray(P_grid, dtype=float)
    I_grid = np.asarray(I_grid, dtype=float)

    nP, nI = P_grid.size, I_grid.size
    loss_grid = np.full((nP, nI), np.inf)

    def _eval_point(ip: int, ii: int) -> Tuple[int, int, float]:
        try:
            total = 0.0
            for ld in line_data:
                w = float(ld.get('weight', 1.0))
                total += w * _line_loss(
                    scanner, ld['h_hat'], ld['d_meas'], ld['A_meas'],
                    P_grid[ip], I_grid[ii],
                    w_A=w_A, line_kwargs=ld.get('line_kwargs'),
                )
            return ip, ii, float(total)
        except Exception:
            return ip, ii, float('inf')

    if n_jobs and n_jobs > 1:
        try:
            from joblib import Parallel, delayed
            idx_pairs = [(i, j) for i in range(nP) for j in range(nI)]
            results = Parallel(n_jobs=n_jobs, verbose=10 if verbose else 0)(
                delayed(_eval_point)(i, j) for i, j in idx_pairs
            )
            for ip, ii, val in results:
                loss_grid[ip, ii] = val
        except ImportError:
            if verbose:
                print("joblib not available; falling back to serial.")
            n_jobs = 1

    if not (n_jobs and n_jobs > 1):
        for ip in range(nP):
            for ii in range(nI):
                _, _, val = _eval_point(ip, ii)
                loss_grid[ip, ii] = val
            if verbose:
                best = np.unravel_index(np.nanargmin(loss_grid), loss_grid.shape)
                print(f"P row {ip+1:>3}/{nP}: best so far "
                      f"P={P_grid[best[0]]:.3g}, I={I_grid[best[1]]:.3g}, "
                      f"loss={loss_grid[best]:.6g}")

    best = np.unravel_index(np.nanargmin(loss_grid), loss_grid.shape)
    return loss_grid, float(P_grid[best[0]]), float(I_grid[best[1]])


def grid_search_PI(
    scanner,
    h_hat: np.ndarray,
    d_meas: np.ndarray,
    A_meas: np.ndarray,
    P_grid: Optional[np.ndarray] = None,
    I_grid: Optional[np.ndarray] = None,
    w_A: float = 0.3,
    line_kwargs: Optional[Dict] = None,
    n_jobs: int = 1,
    verbose: bool = True,
) -> Tuple[np.ndarray, float, float]:
    """Single-line convenience wrapper around grid_search_PI_multi."""
    return grid_search_PI_multi(
        scanner,
        line_data=[{'h_hat': h_hat, 'd_meas': d_meas, 'A_meas': A_meas,
                    'line_kwargs': line_kwargs}],
        P_grid=P_grid, I_grid=I_grid, w_A=w_A,
        n_jobs=n_jobs, verbose=verbose,
    )


# =============================================================================
# Sub-grid parabolic refinement (cheap post-processing)
# =============================================================================

def parabolic_refine_2d(
    loss_grid: np.ndarray,
    x_grid: np.ndarray,
    y_grid: np.ndarray,
) -> Tuple[float, float]:
    """Separable axis-by-axis parabolic refinement in log10 space.

    Fits a 1D parabola through 3 points along each axis through the grid
    minimum and returns the sub-grid minimizer. Useful for resolving a finer
    estimate than the log-grid spacing without a second optimizer.

    If the minimum is on the boundary or the local curvature is non-positive,
    the bare grid minimum is returned.
    """
    loss_grid = np.asarray(loss_grid, dtype=float)
    x_grid = np.asarray(x_grid, dtype=float)
    y_grid = np.asarray(y_grid, dtype=float)
    nx, ny = loss_grid.shape
    i0, j0 = np.unravel_index(np.nanargmin(loss_grid), loss_grid.shape)

    if i0 in (0, nx - 1) or j0 in (0, ny - 1):
        return float(x_grid[i0]), float(y_grid[j0])

    def _refine_axis(coords3, vals3, fallback):
        lc = np.log10(coords3)
        M = np.column_stack([np.ones(3), lc, lc ** 2])
        try:
            a, b, c = np.linalg.solve(M, vals3)
        except np.linalg.LinAlgError:
            return float(fallback)
        if c <= 0:
            return float(fallback)
        lc_star = -b / (2 * c)
        lc_star = float(np.clip(lc_star, lc[0], lc[-1]))
        return 10.0 ** lc_star

    x_star = _refine_axis(
        x_grid[i0 - 1:i0 + 2], loss_grid[i0 - 1:i0 + 2, j0], x_grid[i0],
    )
    y_star = _refine_axis(
        y_grid[j0 - 1:j0 + 2], loss_grid[i0, j0 - 1:j0 + 2], y_grid[j0],
    )
    return float(x_star), float(y_star)


# =============================================================================
# Visualization helper (optional, requires matplotlib)
# =============================================================================

def plot_loss_grid(
    loss_grid: np.ndarray,
    x_grid: np.ndarray,
    y_grid: np.ndarray,
    x_label: str = 'x',
    y_label: str = 'y',
    title: Optional[str] = None,
    log_loss: bool = True,
    ax=None,
):
    """Heatmap of the 2D loss on log-log axes; marks the minimum with a star."""
    import matplotlib.pyplot as plt
    if ax is None:
        _, ax = plt.subplots(figsize=(6.2, 5.2))
    Z = np.log10(np.where(loss_grid > 0, loss_grid, np.nan)) if log_loss else loss_grid
    X, Y = np.meshgrid(x_grid, y_grid, indexing='ij')
    pcm = ax.pcolormesh(X, Y, Z, shading='auto', cmap='viridis_r')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    plt.colorbar(pcm, ax=ax, label=('log10 loss' if log_loss else 'loss'))
    i, j = np.unravel_index(np.nanargmin(loss_grid), loss_grid.shape)
    ax.plot(x_grid[i], y_grid[j], marker='*', color='red',
            markersize=14, markeredgecolor='white', linestyle='')
    if title:
        ax.set_title(title)
    return ax


# =============================================================================
# Example wiring (commented; adapt to your data structures)
# =============================================================================
#
# import numpy as np
# from approach_curves import (SPMRK45ApproachModel, ApproachCurveLibrary,
#                              make_normalized_fd_lookup, RK45Options,
#                              NormalizedSPMParams)
# from scanner_fd import ScannerFD
# from calibrate_grid import (make_log_grid, grid_search_HE,
#                             build_library_from_fit, grid_search_PI_multi,
#                             parabolic_refine_2d, plot_loss_grid)
#
# # ----- shared grid -----
# multipliers = np.linspace(1, 9, 9)
# exponents   = np.arange(-6, -2)             # -6, -5, -4, -3
# grid        = np.concatenate([multipliers * 10.0**e for e in exponents]
#                              + [[10.0]])    # 37 points
#
# # ----- stage 1 -----
# # measured_curves: list of dicts in NORMALIZED units
# loss_HE, H_fit, E_fit = grid_search_HE(
#     measured_curves, base_params, R_fixed=R_norm,
#     SPMRK45ApproachModel=SPMRK45ApproachModel,
#     H_grid=grid, E_star_grid=grid,
#     solver=RK45Options(),
#     w_phi=1.0, n_jobs=4,
# )
# H_ref, E_ref = parabolic_refine_2d(loss_HE, grid, grid)
# plot_loss_grid(loss_HE, grid, grid, x_label='H', y_label='E*',
#                title='stage 1 loss')
#
# # ----- stage 2 -----
# library = build_library_from_fit(
#     measured_curves, base_params, R_fixed=R_norm,
#     H_fit=H_ref, E_star_fit=E_ref,
#     SPMRK45ApproachModel=SPMRK45ApproachModel,
#     ApproachCurveLibrary=ApproachCurveLibrary,
#     n_points=512, conv_L=scanner.conversion['A'],
#     smooth_method='savgol', smooth_window=7, smooth_polyorder=3,
# )
# fd_lookup = make_normalized_fd_lookup(library, conv_L=scanner.conversion['A'])
#
# # ----- stage 3 -----
# # build a ScannerFD bound to the library at the experimental drive
# scanner = ScannerFD(params, conversions, fd_model=fd_lookup(drive_nm))
#
# line_data = [
#     {'h_hat': h_hat_slow, 'd_meas': d_slow, 'A_meas': A_slow,
#      'line_kwargs': {'scan_speed_hat': v_slow, 'dx_hat': dx,
#                      'A_filter_order': 1, 'tau_A': tau_A_init,
#                      'z_filter_order': 1, 'tau_z': tau_z_init,
#                      'setpoint': 0.8, 'T_I': 0.0}},
#     {'h_hat': h_hat_fast, 'd_meas': d_fast, 'A_meas': A_fast,
#      'line_kwargs': {'scan_speed_hat': v_fast, 'dx_hat': dx,
#                      'A_filter_order': 1, 'tau_A': tau_A_init,
#                      'z_filter_order': 1, 'tau_z': tau_z_init,
#                      'setpoint': 0.8, 'T_I': 0.0}},
# ]
# loss_PI, P_fit, I_fit = grid_search_PI_multi(
#     scanner, line_data, P_grid=grid, I_grid=grid,
#     w_A=0.3, n_jobs=4,
# )
# P_ref, I_ref = parabolic_refine_2d(loss_PI, grid, grid)
# plot_loss_grid(loss_PI, grid, grid, x_label='P', y_label='I',
#                title='stage 3 loss')


## Scanner with LPF

In [ ]:
"""
ScannerFD — FD-lookup digital twin with controller dynamics aligned to the
full ODE-based Scanner (no noise injection).

Differences from the ODE Scanner:
  - measure_from_fd(d_eff) replaces measure_amp_phase_force_fast(...).
    The FD lookup returns the steady-state (A, phi, F) for the given d.
    The cantilever's natural amplitude time constant (~Q / (2 omega0)) is
    NOT modeled by the lookup — set tau_A and tau_phi explicitly to mimic
    the lock-in averaging behavior of the ODE version.
  - No ODE-specific kwargs (omega_*, tau_li_hat, N_cycles, ...).
  - No cantilever state (z, v) is propagated — replaced by A_meas_init_hat
    and phi_meas_init_deg so the trace -> retrace handoff still works.

Identical to the ODE Scanner's scan_line_pi:
  - dt = dx_hat / scan_speed_hat
  - first/second-order LPFs on A, phi, and z
  - PI controller with optional integrator leak (T_I) and rate limit
  - same operation order: measure -> filter -> err -> integ -> PI -> dcmd -> Z LPF
"""
from __future__ import annotations

import numpy as np
from typing import Dict, Optional


# -----------------------------------------------------------------------------
# Filters (must match the ODE Scanner's helpers)
# -----------------------------------------------------------------------------

def _lpf1_step(y, x, dt, tau):
    """1st-order LPF, alpha = dt / (tau + dt). Unconditionally stable."""
    if tau is None or tau <= 0.0:
        return x
    alpha = dt / (tau + dt)
    return y + alpha * (x - y)


# def _lpf2_step(y, ydot, x, dt, wn, zeta):
#     """
#     2nd-order LPF, semi-implicit Euler:
#         y'' + 2 zeta wn y' + wn^2 y = wn^2 x
#     Stable iff dt * wn is not too large; safe rule: dt * wn < 1.5.
#     For larger dt * wn, use a backward-Euler discretization instead.
#     """
#     if wn is None or wn <= 0.0:
#         return x, 0.0
#     ydd = (wn * wn) * (x - y) - (2.0 * zeta * wn) * ydot
#     ydot_new = ydot + dt * ydd
#     y_new = y + dt * ydot_new
#     return y_new, ydot_new

import numpy as np

def _lpf2_step(y, ydot, x, dt, wn, zeta):
    """
    Exact discrete step for the 2nd-order low-pass:
        y'' + 2*zeta*wn*y' + wn**2 * y = wn**2 * x,
    assuming x is held constant over [t, t+dt].

    Unconditionally stable; reduces to identity as dt -> 0,
    and to y -> x as dt -> infinity (for zeta > 0).
    """
    # Work in transient coordinates: e = y - x_ss, x_ss = x for unit-gain LPF
    e = y - x
    edot = ydot

    if zeta < 1.0:
        wd = wn * np.sqrt(1.0 - zeta * zeta)
        decay = np.exp(-zeta * wn * dt)
        c = np.cos(wd * dt)
        s = np.sin(wd * dt)
        e_new    = decay * (c * e + (s / wd) * (edot + zeta * wn * e))
        edot_new = decay * (c * edot - (s / wd) * (wn * wn * e + zeta * wn * edot))
    elif zeta > 1.0:
        wd = wn * np.sqrt(zeta * zeta - 1.0)
        decay = np.exp(-zeta * wn * dt)
        ch = np.cosh(wd * dt)
        sh = np.sinh(wd * dt)
        e_new    = decay * (ch * e + (sh / wd) * (edot + zeta * wn * e))
        edot_new = decay * (ch * edot - (sh / wd) * (wn * wn * e + zeta * wn * edot))
    else:  # critically damped
        decay = np.exp(-wn * dt)
        e_new    = decay * (e + (edot + wn * e) * dt)
        edot_new = decay * (edot - wn * (edot + wn * e) * dt)

    return x + e_new, edot_new
    

import numpy as np

# 4th-order Butterworth = cascade of two 2nd-order sections
_BW4_ZETAS = (np.sin(np.pi / 8), np.sin(3 * np.pi / 8))

def _lpf4_step(s1, s1_dot, s2, s2_dot, x, dt, tau):
    """
    One step of a 4th-order Butterworth low-pass at corner 1/tau.
    State: (s1, s1_dot, s2, s2_dot) — 4 floats per filtered channel.
    Returns: (y, s1_new, s1_dot_new, s2_new, s2_dot_new)  where y = s2_new.
    """
    wn = 1.0 / tau
    z1, z2 = _BW4_ZETAS
    # First (lightly damped) section: input = x, output = s1
    s1_new, s1_dot_new = _lpf2_step(s1, s1_dot, x, dt, wn, z1)
    # Second (heavily damped) section: input = s1_new, output = s2
    s2_new, s2_dot_new = _lpf2_step(s2, s2_dot, s1_new, dt, wn, z2)
    return s2_new, s1_new, s1_dot_new, s2_new, s2_dot_new
    
# -----------------------------------------------------------------------------
# ScannerFD
# -----------------------------------------------------------------------------

class ScannerFD:
    """FD-lookup scanner. Controller code path matches the ODE Scanner."""

    def __init__(self, params: Dict, conversions: Dict, fd_model):
        self.param = dict(params)
        self.conversion = dict(conversions)
        self.fd_model = fd_model

        self.param_norm: Dict[str, float] = {}
        for key in self.param:
            self.param_norm[key] = self.param[key] * self.conversion[key]

        # Derived constants kept for parity with the ODE Scanner; unused here.
        self.param_norm['C1'] = self.param_norm['H'] * self.param_norm['R'] / 6.0
        self.param_norm['C2'] = (4.0 / 3.0) * self.param_norm['E_star'] * np.sqrt(self.param_norm['R'])
        self.param_norm['C3'] = self.param_norm['A'] / self.param_norm['Q']

    # -------------------------------------------------------------------------
    # Measurement
    # -------------------------------------------------------------------------

    def measure_from_fd(self, d_hat: float):
        A, phi, F = self.fd_model.evaluate(float(d_hat))
        return float(A), float(phi), float(F)

    # -------------------------------------------------------------------------
    # Single line scan
    # -------------------------------------------------------------------------

    def scan_line_pi(
        self,
        h_hat,
        setpoint: float = 0.8,
        scan_speed_hat: float = 1.0,
        dx_hat: float = 1.0,

        # PI gains
        P: float = 0.1,
        I: float = 0.01,

        # Amplitude / phase measurement filters (lock-in proxy)
        A_filter_order: Optional[int] = None,
        tau_A: float = 1e-2, wn_A: float = 1e3, zeta_A: float = 0.7,
        phi_filter_order: Optional[int] = None,
        tau_phi: float = 1e-2, wn_phi: float = 1e3, zeta_phi: float = 0.7,

        # Z actuator dynamics
        z_filter_order: Optional[int] = None,
        tau_z: float = 1e-2, wn_z: float = 1e3, zeta_z: float = 0.7,
        z_rate_limit_hat: Optional[float] = None,

        # Integrator memory
        T_I: float = 0.0,
        integ_clip: Optional[float] = None,

        # ----- Initial controller state -----
        # ODE-style single init for d_cmd:
        d_init_hat: Optional[float] = 500,
        # FD-style finer init (override d_init_hat if provided), used for
        # trace -> retrace state hand-off in the calibrator:
        d_cmd_init_hat: Optional[float] = None,
        d_act_init_hat: Optional[float] = None,
        A_meas_init_hat: Optional[float] = None,
        phi_meas_init_deg: float = 0.0,
        integ_init: float = 0.0,
    ):
        h_hat = np.asarray(h_hat, dtype=np.float64)
        n = h_hat.size
        dt = float(dx_hat / scan_speed_hat)

        A0 = float(self.param_norm['A'])
        A_set = float(setpoint * A0)

        # d_cmd init: priority is d_cmd_init_hat > d_init_hat > param_norm['d0']
        if d_cmd_init_hat is not None:
            d_cmd = float(d_cmd_init_hat)
        elif d_init_hat is not None:
            d_cmd = float(d_init_hat)
        else:
            d_cmd = float(self.param_norm['d0'])

        d_act = float(d_cmd if d_act_init_hat is None else d_act_init_hat)
        d_act_dot = 0.0

        A_meas = float(A0 if A_meas_init_hat is None else A_meas_init_hat)
        A_meas_dot = 0.0

        phi_meas = float(phi_meas_init_deg)
        phi_meas_dot = 0.0

        integ = float(integ_init)

        if T_I is None or T_I <= 0.0:
            decay = 1.0
            use_leak = False
        else:
            decay = float(np.exp(-dt / T_I))
            use_leak = True

        out_d_cmd    = np.empty(n)
        out_d_act    = np.empty(n)
        out_A_true   = np.empty(n)
        out_A_meas   = np.empty(n)
        out_phi_true = np.empty(n)
        out_phi_meas = np.empty(n)
        out_F        = np.empty(n)
        out_err      = np.empty(n)

        # initialize once before the loop, per channel
        A1, A1d, A2, A2d         = A_meas, 0.0, A_meas, 0.0
        phi1, phi1d, phi2, phi2d = phi_meas, 0.0, phi_meas, 0.0
        
        
                
        # for i in range(n):
        #     d_eff = d_act - h_hat[i]
        #     A_true, phi_true, F_peak = self.measure_from_fd(d_eff)

        #     # --- Measurement filtering (matches ODE Scanner) ---
        #     if A_filter_order is None:
        #         A_meas = A_true
        #     elif A_filter_order == 1:
        #         A_meas = _lpf1_step(A_meas, A_true, dt, tau_A)
        #     elif A_filter_order == 2:
        #         A_meas, A_meas_dot = _lpf2_step(A_meas, A_meas_dot, A_true, dt, wn_A, zeta_A)
        #     else:
        #         raise ValueError("A_filter_order must be None, 1, or 2")

        #     if phi_filter_order is None:
        #         phi_meas = phi_true
        #     elif phi_filter_order == 1:
        #         phi_meas = _lpf1_step(phi_meas, phi_true, dt, tau_phi)
        #     elif phi_filter_order == 2:
        #         phi_meas, phi_meas_dot = _lpf2_step(phi_meas, phi_meas_dot, phi_true, dt, wn_phi, zeta_phi)
        #     else:
        #         raise ValueError("phi_filter_order must be None, 1, or 2")

        for i in range(n):
            d_eff = d_act - h_hat[i]
            A_true, phi_true, F_peak = self.measure_from_fd(d_eff)
        
            # --- Amplitude filter ---
            if A_filter_order is None:
                A_meas = A_true
            elif A_filter_order == 1:
                A_meas = _lpf1_step(A_meas, A_true, dt, tau_A)
            elif A_filter_order == 2:
                A_meas, A_meas_dot = _lpf2_step(A_meas, A_meas_dot, A_true, dt, wn_A, zeta_A)
            elif A_filter_order == 4:
                A_meas, A1, A1d, A2, A2d = _lpf4_step(A1, A1d, A2, A2d, A_true, dt, tau_A)
            else:
                raise ValueError("A_filter_order must be None, 1, 2, or 4")
        
            # --- Phase filter (same pattern) ---
            if phi_filter_order is None:
                phi_meas = phi_true
            elif phi_filter_order == 1:
                phi_meas = _lpf1_step(phi_meas, phi_true, dt, tau_phi)
            elif phi_filter_order == 2:
                phi_meas, phi_meas_dot = _lpf2_step(phi_meas, phi_meas_dot, phi_true, dt, wn_phi, zeta_phi)
            elif phi_filter_order == 4:
                phi_meas, phi1, phi1d, phi2, phi2d = _lpf4_step(
                    phi1, phi1d, phi2, phi2d, phi_true, dt, tau_phi
                )
            else:
                raise ValueError("phi_filter_order must be None, 1, 2 or 4")

            # --- Error & PI (NO noise injection) ---
            err = A_set - A_meas

            if use_leak:
                integ = decay * integ + err * dt
            else:
                integ = integ + err * dt

            if integ_clip is not None:
                integ = float(np.clip(integ, -integ_clip, integ_clip))

            dd_cmd = P * err + I * integ

            if z_rate_limit_hat is not None:
                max_dd = float(z_rate_limit_hat) * dt
                dd_cmd = float(np.clip(dd_cmd, -max_dd, max_dd))

            d_cmd = d_cmd + dd_cmd

            # --- Z actuator dynamics ---
            if z_filter_order is None:
                d_act = d_cmd
            elif z_filter_order == 1:
                d_act = _lpf1_step(d_act, d_cmd, dt, tau_z)
            elif z_filter_order == 2:
                d_act, d_act_dot = _lpf2_step(d_act, d_act_dot, d_cmd, dt, wn_z, zeta_z)
            else:
                raise ValueError("z_filter_order must be None, 1, or 2")

            out_d_cmd[i]    = d_cmd
            out_d_act[i]    = d_act
            out_A_true[i]   = A_true
            out_A_meas[i]   = A_meas
            out_phi_true[i] = phi_true
            out_phi_meas[i] = phi_meas
            out_F[i]        = F_peak
            out_err[i]      = err

        return {
            # Output keys aligned with the ODE Scanner where they overlap.
            "d_cmd_hat":          out_d_cmd,
            "d_hat":              out_d_act,
            "A_true_hat":         out_A_true,
            "A_hat":              out_A_meas,
            "phi_true_deg":       out_phi_true,
            "phi_deg":            out_phi_meas,
            "F_peak":             out_F,
            "err":                out_err,
            "dt_pixel":           dt,
            # Extra final-state fields needed for trace -> retrace stitching:
            "d_cmd_final_hat":    float(d_cmd),
            "d_final_hat":        float(d_act),
            "A_meas_final_hat":   float(A_meas),
            "phi_meas_final_deg": float(phi_meas),
            "integ_final":        float(integ),
        }
    
    def scan_line_substepped(
        self,
        h_hat,
        drive_nm,
        setpoint,
        *,
        scan_speed_hat,
        dx_hat,
        P=0.0,
        I=1.0,
        T_I=None,
        A_filter_order=4,
        tau_A=3.2e-5,
        phi_filter_order=4,
        tau_phi=3.2e-5,
        z_filter_order=1,
        tau_z=3.2e-5,
        d_init_hat=30.0,
        phi_meas_init_deg=113.0,
        A0=22.88,                  # free-air amplitude (nm)
        integ_clip=None,
        z_rate_limit_hat=None,
        # --- speed control ---
        n_substeps=None,           # explicit number of inner steps per pixel
        dt_inner_max=None,         # OR cap inner timestep (auto-computes n_substeps)
        safety=10.0,               # auto-compute uses dt_inner = tau_min / safety
    ):
        """
        Scan one trace+retrace line with the controller and filters running at
        a finer timestep than the pixel rate.
    
        Parameters
        ----------
        n_substeps : int, optional
            Inner steps per pixel. If None, computed from dt_inner_max or `safety`.
        dt_inner_max : float, optional
            Upper bound on the inner timestep. Overrides `safety` if both given.
        safety : float
            If neither n_substeps nor dt_inner_max is provided,
            choose dt_inner = (smallest tau among tau_A, tau_phi, tau_z) / safety.
            safety=10 is conservative; safety=5 is usually fine.
    
        Returns
        -------
        dict with d_cmd, d_act, A_meas, phi_meas, A_true, phi_true, F, err arrays
        (length = number of pixels).
        """
        # 4th-order Butterworth biquad damping ratios
        _BW4_Z = (np.sin(np.pi / 8), np.sin(3 * np.pi / 8))

        n = len(h_hat)
        dt_pixel = dx_hat / scan_speed_hat
    
        # ----- decide inner timestep -----
        tau_min = min(t for t in (tau_A, tau_phi, tau_z) if t is not None and t > 0)
        if n_substeps is None:
            if dt_inner_max is None:
                dt_inner_max = tau_min / safety
            n_substeps = max(1, int(np.ceil(dt_pixel / dt_inner_max)))
        dt_inner = dt_pixel / n_substeps
        ctrl_scale = dt_inner / dt_pixel  # preserves macro controller behavior
    
        if T_I is None or T_I <= 0:
            use_leak = False
            decay_inner = 1.0
        else:
            use_leak = True
            decay_inner = float(np.exp(-dt_inner / T_I))
    
        # ----- state init -----
        d_cmd = float(d_init_hat)
        d_act = float(d_init_hat)
        integ = 0.0
    
        A_meas    = float(A0)
        phi_meas  = float(phi_meas_init_deg)
        # 2nd-order filter states
        A_meas_dot   = 0.0
        phi_meas_dot = 0.0
        d_act_dot    = 0.0
        # 4th-order filter states (cascade of two biquads)
        A1, A1d, A2, A2d         = A_meas, 0.0, A_meas, 0.0
        p1, p1d, p2, p2d         = phi_meas, 0.0, phi_meas, 0.0
    
        # ----- outputs (sampled once per pixel, after final substep) -----
        out = {k: np.empty(n) for k in
               ("d_cmd", "d_act", "d_hat", "A_hat", "A_meas", "phi_meas",
                "A_true", "phi_true", "phi_deg", "F", "err")}
    
        drive_arr = np.broadcast_to(np.asarray(drive_nm, float), (n,))
        sp_arr    = np.broadcast_to(np.asarray(setpoint,  float), (n,))
    
        for i in range(n):
            h_i      = float(h_hat[i])
            drive_i  = float(drive_arr[i])
            sp_i     = float(sp_arr[i])
            A_set    = sp_i * A0     # adjust if your A_set convention differs
    
            # --- inner loop ---
            for _ in range(n_substeps):
                # 1) tip-sample interaction at current z
                d_eff = d_act - h_i
                A_true, phi_true, F_peak = self.measure_from_fd(d_eff)
    
                # 2) amplitude filter
                if A_filter_order is None:
                    A_meas = A_true
                elif A_filter_order == 1:
                    A_meas = _lpf1_step(A_meas, A_true, dt_inner, tau_A)
                elif A_filter_order == 2:
                    A_meas, A_meas_dot = _lpf2_step(
                        A_meas, A_meas_dot, A_true, dt_inner, 1.0/tau_A, np.sqrt(0.5)
                    )
                elif A_filter_order == 4:
                    wn = 1.0/tau_A
                    A1, A1d = _lpf2_step(A1, A1d, A_true, dt_inner, wn, _BW4_Z[0])
                    A2, A2d = _lpf2_step(A2, A2d, A1,     dt_inner, wn, _BW4_Z[1])
                    A_meas  = A2
    
                # 3) phase filter
                if phi_filter_order is None:
                    phi_meas = phi_true
                elif phi_filter_order == 1:
                    phi_meas = _lpf1_step(phi_meas, phi_true, dt_inner, tau_phi)
                elif phi_filter_order == 2:
                    phi_meas, phi_meas_dot = _lpf2_step(
                        phi_meas, phi_meas_dot, phi_true, dt_inner, 1.0/tau_phi, np.sqrt(0.5)
                    )
                elif phi_filter_order == 4:
                    wn = 1.0/tau_phi
                    p1, p1d = _lpf2_step(p1, p1d, phi_true, dt_inner, wn, _BW4_Z[0])
                    p2, p2d = _lpf2_step(p2, p2d, p1,       dt_inner, wn, _BW4_Z[1])
                    phi_meas = p2
    
                # 4) error and PI update at inner rate
                err = A_set - A_meas
                if use_leak:
                    integ = decay_inner * integ + err * dt_inner
                else:
                    integ = integ + err * dt_inner
                if integ_clip is not None:
                    integ = float(np.clip(integ, -integ_clip, integ_clip))
    
                # 5) controller output, scaled so total per-pixel motion matches
                #    a single-step controller at dt_pixel.
                dd_cmd = (P * err + I * integ) * ctrl_scale
                if z_rate_limit_hat is not None:
                    max_dd = float(z_rate_limit_hat) * dt_inner
                    dd_cmd = float(np.clip(dd_cmd, -max_dd, max_dd))
                d_cmd = d_cmd + dd_cmd
    
                # 6) z-piezo dynamics
                if z_filter_order is None:
                    d_act = d_cmd
                elif z_filter_order == 1:
                    d_act = _lpf1_step(d_act, d_cmd, dt_inner, tau_z)
                elif z_filter_order == 2:
                    d_act, d_act_dot = _lpf2_step(
                        d_act, d_act_dot, d_cmd, dt_inner, 1.0/tau_z, np.sqrt(0.5)
                    )
    
            # --- sample at end of pixel ---
            out["d_cmd"][i]    = d_cmd
            out["d_act"][i]    = d_act
            out["d_hat"][i]    = d_act
            out["A_hat"][i]    = A_meas
            out["A_meas"][i]   = A_meas
            out["phi_meas"][i] = phi_meas
            out["phi_deg"][i] = phi_meas
            out["A_true"][i]   = A_true
            out["phi_true"][i] = phi_true
            out["F"][i]        = F_peak
            out["err"][i]      = err
    
        out["dt_pixel"]   = dt_pixel
        out["dt_inner"]   = dt_inner
        out["n_substeps"] = n_substeps
        return out
    # -------------------------------------------------------------------------
    # 2D map (serpentine, optional state carry across rows)
    # -------------------------------------------------------------------------

    def scan_map_pi(
        self,
        h_map_hat,
        serpentine: bool = True,
        carry_state: bool = True,
        **line_kwargs,
    ):
        h_map_hat = np.asarray(h_map_hat, dtype=float)
        if h_map_hat.ndim != 2:
            raise ValueError("h_map_hat must be 2D.")
        ny, nx = h_map_hat.shape

        d_map   = np.empty((ny, nx))
        phi_map = np.empty((ny, nx))
        F_map   = np.empty((ny, nx))
        A_map   = np.empty((ny, nx))
        err_map = np.empty((ny, nx))

        carry = dict(
            d_cmd_init_hat=line_kwargs.pop("d_cmd_init_hat", None),
            d_act_init_hat=line_kwargs.pop("d_act_init_hat", None),
            A_meas_init_hat=line_kwargs.pop("A_meas_init_hat", None),
            phi_meas_init_deg=line_kwargs.pop("phi_meas_init_deg", 0.0),
            integ_init=line_kwargs.pop("integ_init", 0.0),
        )

        for iy in range(ny):
            reverse = bool(serpentine and (iy % 2 == 1))
            row = h_map_hat[iy, ::-1] if reverse else h_map_hat[iy, :]
            out = self.scan_line_pi(row, **carry, **line_kwargs)

            d_row   = out["d_hat"]
            phi_row = out["phi_deg"]
            F_row   = out["F_peak"]
            A_row   = out["A_hat"]
            err_row = out["err"]

            if reverse:
                d_row, phi_row, F_row, A_row, err_row = (a[::-1] for a in
                    (d_row, phi_row, F_row, A_row, err_row))

            d_map[iy, :]   = d_row
            phi_map[iy, :] = phi_row
            F_map[iy, :]   = F_row
            A_map[iy, :]   = A_row
            err_map[iy, :] = err_row

            if carry_state:
                carry = dict(
                    d_cmd_init_hat=out["d_cmd_final_hat"],
                    d_act_init_hat=out["d_final_hat"],
                    A_meas_init_hat=out["A_meas_final_hat"],
                    phi_meas_init_deg=out["phi_meas_final_deg"],
                    integ_init=out["integ_final"],
                )

        return {
            "d_hat_map":     d_map,
            "phi_deg_map":   phi_map,
            "F_peak_map":    F_map,
            "A_hat_map":     A_map,
            "err_map":       err_map,
        }
    

## Calibration helpers / workflow

In [ ]:
"""
calibrate_grid.py

Log-spaced grid-search calibration for the SPM digital twin.

Stage 1  (grid_search_HE)
    Search over (H, E*) - Hamaker constant and effective modulus - against
    experimental approach curves at multiple drives. R is held fixed; C1 and
    C2 are recomputed from the scanner_fd convention:
        C1 = H * R / 6
        C2 = (4/3) * E_star * sqrt(R)
    Joint loss = mean[(A_sim - A_meas)/A0]^2  +  w_phi * mean[(phi_sim - phi_meas)/180]^2
    summed across all measured curves.

Stage 2  (build_library_from_fit)
    Regenerate clean approach curves at the fitted (H, E*) on a uniform fine
    d-grid spanning each measured curve's support, and pack them into an
    ApproachCurveLibrary keyed by drive amplitude (nm).

Stage 3  (grid_search_PI / grid_search_PI_multi)
    Search over (P, I) PI gains for a ScannerFD instance against measured
    scan-line data. Loss = mean[(d_sim - d_meas)/A0]^2 + w_A * mean[(A_sim - A_meas)/A0]^2.
    The multi-line variant sums losses across scan-speed / step-shape
    combinations, which is what breaks the lock-in tau vs. z-piezo tau
    degeneracy in stage 3.

Default grid (matches the spec):
    multipliers = np.linspace(1, 9, 9)
    exponents   = np.arange(-6, -2)            # -6, -5, -4, -3
    grid        = concat(multipliers * 10**e for e in exponents) + [10.0]
    -> 37 points spanning [1e-6, 9e-3] U {10}.

Optional joblib parallelism via n_jobs (silently falls back to serial).
"""
from __future__ import annotations

import copy
import numpy as np
from typing import Dict, List, Optional, Sequence, Tuple


# -----------------------------------------------------------------------------
# Log-spaced grid builder
# -----------------------------------------------------------------------------

def make_log_grid(
    multipliers: Sequence[float] = np.linspace(1.0, 9.0, 9),
    exponents: Sequence[int] = np.arange(-6, -2),
    tail: Sequence[float] = (10.0,),
) -> np.ndarray:
    """Build a log-spaced grid: outer of multipliers x 10**exponents, plus tail.

    Default = 37 points spanning [1e-6, 9e-3] U {10}.
    """
    parts = [np.asarray(multipliers, dtype=float) * 10.0 ** int(e)
             for e in exponents]
    if tail:
        parts.append(np.asarray(tail, dtype=float))
    return np.concatenate(parts)


# =============================================================================
# Data prep - convert experimental tuples into the dict format expected below
# =============================================================================

def build_measured_curves(
    raw_data,
    conv_L: float,
    A0_method: str = 'far_field_median',
    far_field_frac: float = 0.1,
    A0_explicit: Optional[Dict[float, float]] = None,
    omega_drive_hat: Optional[float] = None,
    omega_ref_hat: Optional[float] = None,
) -> List[Dict]:
    """Convert experimental approach-curve data into the dict format that
    grid_search_HE / build_library_from_fit consume.

    Accepted input formats
    ----------------------
    1. dict of dicts  (e.g. your `measured_fd`):
           {drive_nm: {'d': h_nm, 'A': A_nm, 'phi': phi_deg,
                       # optional per-curve overrides:
                       'omega_drive_hat': ..., 'omega_ref_hat': ...}}
    2. iterable of 4-tuples:
           [(drive_nm, h_nm, A_nm, phi_deg), ...]
    3. iterable of 6-tuples (with per-curve frequencies):
           [(drive_nm, h_nm, A_nm, phi_deg, omega_drive_hat, omega_ref_hat), ...]

    Units throughout the input are physical: nm for d, A, drive; degrees for phi.

    Parameters
    ----------
    conv_L : length conversion s.t. d_norm = d_nm * conv_L. This is the same
        factor your simulator setup uses for every length-like param. If you
        have a ScannerFD around, it's `scanner.conversion['A']` (which equals
        `conversion['d0']`); otherwise pass the scalar directly.
    A0_method : how to set the free amplitude A0 used for normalizing residuals
        'far_field_median' : median of A_nm over the far-field tail
                             (largest far_field_frac of d). Robust to gain
                             drift between drive setting and actual A0.
        'drive'            : trust the commanded drive  ->  A0_nm = drive_nm
        'explicit'         : look up in A0_explicit[drive_nm]
    far_field_frac : fraction of points used for the far-field A0 estimate.
    A0_explicit : dict {drive_nm: A0_nm}, only used when A0_method='explicit'.
    omega_drive_hat, omega_ref_hat : optional global defaults applied to every
        curve. Per-curve values in the input override these.

    Returns
    -------
    measured_curves : list of dicts shaped for grid_search_HE.
    """
    # ---- normalize input to an iterable of (drive, d, A, phi, omd, omr) ----
    def _iter_entries():
        if isinstance(raw_data, dict):
            for drive_nm, c in raw_data.items():
                if not isinstance(c, dict):
                    raise ValueError(f"raw_data[{drive_nm}] must be a dict "
                                     f"with keys 'd', 'A', 'phi'.")
                yield (
                    drive_nm,
                    c['d'], c['A'], c['phi'],
                    c.get('omega_drive_hat', omega_drive_hat),
                    c.get('omega_ref_hat',   omega_ref_hat),
                )
        else:
            for entry in raw_data:
                if len(entry) == 4:
                    drive_nm, h_nm, A_nm, phi_deg = entry
                    yield drive_nm, h_nm, A_nm, phi_deg, omega_drive_hat, omega_ref_hat
                elif len(entry) == 6:
                    yield tuple(entry)
                else:
                    raise ValueError("Tuple entries must have length 4 or 6.")

    out: List[Dict] = []
    for drive_nm, h_in, A_in, phi_in, omd, omr in _iter_entries():
        h_nm = np.asarray(h_in, dtype=float).ravel()
        A_nm = np.asarray(A_in, dtype=float).ravel()
        phi_deg = np.asarray(phi_in, dtype=float).ravel()
        if not (h_nm.shape == A_nm.shape == phi_deg.shape):
            raise ValueError(f"Inconsistent array shapes for drive {drive_nm}: "
                             f"d{h_nm.shape}, A{A_nm.shape}, phi{phi_deg.shape}")

        mask = np.isfinite(h_nm) & np.isfinite(A_nm) & np.isfinite(phi_deg)
        h_nm, A_nm, phi_deg = h_nm[mask], A_nm[mask], phi_deg[mask]
        if h_nm.size < 5:
            raise ValueError(f"drive {drive_nm}: <5 finite samples after NaN drop.")

        # ---- estimate the actual free amplitude A0 for this curve ----
        if A0_method == 'far_field_median':
            n_far = max(1, int(far_field_frac * h_nm.size))
            order_desc = np.argsort(h_nm)[::-1]
            A0_nm = float(np.median(A_nm[order_desc][:n_far]))
        elif A0_method == 'drive':
            A0_nm = float(drive_nm)
        elif A0_method == 'explicit':
            if A0_explicit is None or float(drive_nm) not in A0_explicit:
                raise ValueError(f"A0_explicit missing entry for drive {drive_nm}")
            A0_nm = float(A0_explicit[float(drive_nm)])
        else:
            raise ValueError(f"Unknown A0_method '{A0_method}'")

        # ---- physical -> normalized ----
        c: Dict = {
            'd_hat':    h_nm    * float(conv_L),
            'A_hat':    A_nm    * float(conv_L),
            'phi_deg':  phi_deg,
            'A0_hat':   A0_nm   * float(conv_L),
            'drive_nm': float(drive_nm),  # kept for traceability / library keys
        }
        if omd is not None:
            c['omega_drive_hat'] = float(omd)
        if omr is not None:
            c['omega_ref_hat'] = float(omr)
        out.append(c)

    return out


# =============================================================================
# Stage 1 - (H, E*) grid search against measured approach curves
# =============================================================================

def _set_HEstar(base_params, H_new: float, E_star_new: float, R_fixed: float):
    """Return a copy of base_params with C1, C2 recomputed for new H and E*.

    Tries dataclasses.replace first; falls back to a shallow copy + attribute
    write. base_params must expose .C1 and .C2 (NormalizedSPMParams).
    """
    C1 = float(H_new) * float(R_fixed) / 6.0
    C2 = (4.0 / 3.0) * float(E_star_new) * np.sqrt(float(R_fixed))
    try:
        from dataclasses import is_dataclass, replace
        if is_dataclass(base_params):
            return replace(base_params, C1=C1, C2=C2)
    except Exception:
        pass
    p = copy.copy(base_params)
    p.C1 = C1
    p.C2 = C2
    return p


def _simulate_at_d_array(
    model,
    d_array_desc: np.ndarray,
    omega_drive_hat: Optional[float] = None,
    omega_ref_hat: Optional[float] = None,
    z0: float = 0.0,
    v0: float = 0.0,
) -> Tuple[np.ndarray, np.ndarray]:
    """Run model.simulate_at_height at each d in d_array_desc.

    Sweep is in the order given (descending in d is recommended for stable
    warm-starting). Returns (A, phi) in the same order as d_array_desc.
    """
    n = len(d_array_desc)
    A = np.empty(n)
    phi = np.empty(n)
    z_cur, v_cur = float(z0), float(v0)
    for i, d in enumerate(d_array_desc):
        out = model.simulate_at_height(
            d_hat=float(d),
            z0_hat=z_cur,
            v0_hat=v_cur,
            omega_drive_hat=omega_drive_hat,
            omega_ref_hat=omega_ref_hat,
        )
        A[i] = out["A_hat"]
        phi[i] = out["phi_deg"]
        z_cur = out["z_final_hat"]
        v_cur = out["v_final_hat"]
    return A, phi


def _curve_loss(
    model,
    d_meas: np.ndarray,
    A_meas: np.ndarray,
    phi_meas: np.ndarray,
    A0_hat: float,
    omega_drive_hat: Optional[float] = None,
    omega_ref_hat: Optional[float] = None,
    w_phi: float = 1.0,
) -> float:
    """Joint MSE on (A/A0) + w_phi * (phi/180) for one approach curve."""
    d = np.asarray(d_meas, dtype=float)
    A = np.asarray(A_meas, dtype=float)
    phi = np.asarray(phi_meas, dtype=float)

    # Sweep descending in d for warm-start stability
    order = np.argsort(d)[::-1]
    d_desc, A_desc, phi_desc = d[order], A[order], phi[order]

    A_sim, phi_sim = _simulate_at_d_array(
        model, d_desc, omega_drive_hat, omega_ref_hat,
    )

    res_A = (A_sim - A_desc) / float(A0_hat)
    res_phi = (phi_sim - phi_desc) / 180.0
    return float(np.mean(res_A ** 2) + float(w_phi) * np.mean(res_phi ** 2))


def grid_search_HE(
    measured_curves: List[Dict],
    base_params,
    R_fixed: float,
    SPMRK45ApproachModel,
    H_grid: Optional[np.ndarray] = None,
    E_star_grid: Optional[np.ndarray] = None,
    solver=None,
    w_phi: float = 1.0,
    n_jobs: int = 1,
    verbose: bool = True,
) -> Tuple[np.ndarray, float, float]:
    """Grid-search (H, E*) against a list of measured approach curves.

    Parameters
    ----------
    measured_curves : list of dicts, each with keys
        'd_hat'           : (n,) array of normalized z-piezo positions
        'A_hat'           : (n,) array of normalized amplitudes
        'phi_deg'         : (n,) array of phases [deg]
        'A0_hat'          : scalar, normalized free amplitude for this curve
        'omega_drive_hat' : optional, normalized drive freq for this curve
        'omega_ref_hat'   : optional, normalized reference freq for this curve
    base_params : NormalizedSPMParams (passed through to SPMRK45ApproachModel).
    R_fixed     : tip radius in the SAME normalized units used by C1, C2.
    SPMRK45ApproachModel : class (passed in to avoid a circular import).
    H_grid, E_star_grid  : 1D arrays. Default = make_log_grid().
    solver               : RK45Options or None.
    w_phi                : weight on phi residual (default 1.0).
    n_jobs               : if > 1 and joblib is installed, run grid in parallel.
    verbose              : print per-row best so far.

    Returns
    -------
    loss_grid : (nH, nE) total loss summed across measured curves
    best_H, best_E_star : floats at np.nanargmin(loss_grid)
    """
    if H_grid is None:
        H_grid = make_log_grid()
    if E_star_grid is None:
        E_star_grid = make_log_grid()
    H_grid = np.asarray(H_grid, dtype=float)
    E_star_grid = np.asarray(E_star_grid, dtype=float)

    nH, nE = H_grid.size, E_star_grid.size
    loss_grid = np.full((nH, nE), np.inf)

    def _eval_point(iH: int, jE: int) -> Tuple[int, int, float]:
        params = _set_HEstar(base_params, H_grid[iH], E_star_grid[jE], R_fixed)
        try:
            model = SPMRK45ApproachModel(params, solver=solver)
            total = 0.0
            for c in measured_curves:
                loss_c, offset_c = _curve_loss(
                    model,
                    c['d_hat'],
                    c['A_hat'],
                    c['phi_deg'],
                    c['A0_hat'],
                    c.get('omega_drive_hat'),
                    c.get('omega_ref_hat'),
                    w_phi=w_phi,
                    d_axis_mode=c.get('d_axis_mode', 'approach_extension'),
                )
                total += loss_c
            # for c in measured_curves:
            #     total += _curve_loss(
            #         model, c['d_hat'], c['A_hat'], c['phi_deg'],
            #         c['A0_hat'],
            #         c.get('omega_drive_hat'),
            #         c.get('omega_ref_hat'),
            #         w_phi=w_phi,
            #     )
            return iH, jE, float(total)
        except Exception:
            return iH, jE, float('inf')

    if n_jobs and n_jobs > 1:
        try:
            from joblib import Parallel, delayed
            idx_pairs = [(i, j) for i in range(nH) for j in range(nE)]
            results = Parallel(n_jobs=n_jobs, verbose=10 if verbose else 0)(
                delayed(_eval_point)(i, j) for i, j in idx_pairs
            )
            for iH, jE, val in results:
                loss_grid[iH, jE] = val
        except ImportError:
            if verbose:
                print("joblib not available; falling back to serial.")
            n_jobs = 1

    if not (n_jobs and n_jobs > 1):
        for iH in range(nH):
            for jE in range(nE):
                _, _, val = _eval_point(iH, jE)
                loss_grid[iH, jE] = val
            if verbose:
                best = np.unravel_index(np.nanargmin(loss_grid), loss_grid.shape)
                print(f"H row {iH+1:>3}/{nH}: best so far "
                      f"H={H_grid[best[0]]:.3g}, E*={E_star_grid[best[1]]:.3g}, "
                      f"loss={loss_grid[best]:.6g}")

    best = np.unravel_index(np.nanargmin(loss_grid), loss_grid.shape)
    return loss_grid, float(H_grid[best[0]]), float(E_star_grid[best[1]])


# =============================================================================
# Stage 2 - build approach-curve library at the fitted (H, E*)
# =============================================================================

def build_library_from_fit(
    measured_curves: List[Dict],
    base_params,
    R_fixed: float,
    H_fit: float,
    E_star_fit: float,
    SPMRK45ApproachModel,
    ApproachCurveLibrary,
    n_points: int = 256,
    conv_L: float = 1.0,
    solver=None,
    smooth_method: Optional[str] = None,
    smooth_window: int = 7,
    smooth_polyorder: int = 3,
    smooth_sigma: float = 2.0,
):
    """Regenerate a clean curve at (H_fit, E_star_fit) for each measured drive,
    then pack into an ApproachCurveLibrary keyed by drive amplitude in nm.

    Parameters
    ----------
    conv_L : length conversion such that d_norm = d_nm * conv_L. The simulator
             works in normalized units; the library stores nm/deg. Pass
             scanner.conversion['A'] (or equivalently 'd0') to integrate with
             ScannerFD via approach_curves.make_normalized_fd_lookup.

    Returns
    -------
    library : ApproachCurveLibrary
    """
    params = _set_HEstar(base_params, H_fit, E_star_fit, R_fixed)
    model = SPMRK45ApproachModel(params, solver=solver)

    by_drive: Dict[float, Dict[str, np.ndarray]] = {}
    for c in measured_curves:
        d_meas = np.asarray(c['d_hat'], dtype=float)
        d_min, d_max = float(d_meas.min()), float(d_meas.max())
        out = model.approach_curve(
            d_start_hat=d_max,
            d_end_hat=d_min,
            n_points=int(n_points),
            warm_start=True,
            omega_drive_hat=c.get('omega_drive_hat'),
            omega_ref_hat=c.get('omega_ref_hat'),
        )
        # normalized -> nm for length-like quantities
        d_nm = np.asarray(out['d_hat'], dtype=float) / float(conv_L)
        A_nm = np.asarray(out['A_hat'], dtype=float) / float(conv_L)
        phi_dg = np.asarray(out['phi_deg'], dtype=float)
        # F is left in whatever normalized units the simulator uses; the library
        # callers convert via conv_F when wrapping into ScannerFD.
        F_arr = np.asarray(out.get('F_mean_hat', np.zeros_like(d_nm)), dtype=float)
        drive_nm = float(c['A0_hat']) / float(conv_L)

        by_drive[drive_nm] = {'d': d_nm, 'A': A_nm, 'phi': phi_dg, 'F': F_arr}

    return ApproachCurveLibrary.from_dict(
        by_drive,
        smooth_method=smooth_method,
        smooth_window=smooth_window,
        smooth_polyorder=smooth_polyorder,
        smooth_sigma=smooth_sigma,
    )


# =============================================================================
# Stage 3 - PI gain grid search against measured scan-line data
# =============================================================================

def _line_loss(
    scanner,
    h_hat: np.ndarray,
    d_meas: np.ndarray,
    A_meas: np.ndarray,
    P: float,
    I: float,
    w_A: float = 0.3,
    line_kwargs: Optional[Dict] = None,
) -> float:
    """Per-line loss for stage 3.

    Loss = mean[(d_sim - d_meas)/A0]^2 + w_A * mean[(A_sim - A_meas)/A0]^2

    Both residuals are scaled by A0 so the d-trace and A-trace are dimensionally
    comparable. line_kwargs holds every scan_line_pi argument other than P and I
    (scan_speed_hat, dx_hat, filter taus/orders, T_I, setpoint, ...).
    """
    line_kwargs = dict(line_kwargs or {})
    line_kwargs.update(P=float(P), I=float(I))
    out = scanner.scan_line_pi(h_hat=h_hat, **line_kwargs)
    d_sim = out['d_hat']
    A_sim = out['A_hat']

    A0 = float(scanner.param_norm['A'])
    d_meas = np.asarray(d_meas, dtype=float)
    A_meas = np.asarray(A_meas, dtype=float)

    res_d = (d_sim - d_meas) / A0
    res_A = (A_sim - A_meas) / A0
    return float(np.mean(res_d ** 2) + float(w_A) * np.mean(res_A ** 2))


def grid_search_PI_multi(
    scanner,
    line_data: List[Dict],
    P_grid: Optional[np.ndarray] = None,
    I_grid: Optional[np.ndarray] = None,
    w_A: float = 0.3,
    n_jobs: int = 1,
    verbose: bool = True,
) -> Tuple[np.ndarray, float, float]:
    """Grid-search PI gains across a list of measured scan lines.

    Parameters
    ----------
    line_data : list of dicts, each with keys
        'h_hat'       : (n,) topography (normalized)
        'd_meas'      : (n,) measured d trace (normalized)
        'A_meas'      : (n,) measured A trace (normalized)
        'line_kwargs' : dict of scan_line_pi kwargs other than P, I
                        (scan_speed_hat, dx_hat, A_filter_order, tau_A,
                         z_filter_order, tau_z, T_I, setpoint, ...).
                        These should be FIXED to the values you've already
                        identified for the controller hardware (lock-in tau,
                        z bandwidth, etc.) and the operating point you ran on
                        the real instrument. P and I are the two unknowns.
        'weight'      : optional scalar (default 1.0)

    Returns
    -------
    loss_grid : (nP, nI) total loss summed across line_data entries
    best_P, best_I : floats at np.nanargmin(loss_grid)
    """
    if P_grid is None: P_grid = make_log_grid()
    if I_grid is None: I_grid = make_log_grid()
    P_grid = np.asarray(P_grid, dtype=float)
    I_grid = np.asarray(I_grid, dtype=float)

    nP, nI = P_grid.size, I_grid.size
    loss_grid = np.full((nP, nI), np.inf)

    def _eval_point(ip: int, ii: int) -> Tuple[int, int, float]:
        try:
            total = 0.0
            for ld in line_data:
                w = float(ld.get('weight', 1.0))
                total += w * _line_loss(
                    scanner, ld['h_hat'], ld['d_meas'], ld['A_meas'],
                    P_grid[ip], I_grid[ii],
                    w_A=w_A, line_kwargs=ld.get('line_kwargs'),
                )
            return ip, ii, float(total)
        except Exception:
            return ip, ii, float('inf')

    if n_jobs and n_jobs > 1:
        try:
            from joblib import Parallel, delayed
            idx_pairs = [(i, j) for i in range(nP) for j in range(nI)]
            results = Parallel(n_jobs=n_jobs, verbose=10 if verbose else 0)(
                delayed(_eval_point)(i, j) for i, j in idx_pairs
            )
            for ip, ii, val in results:
                loss_grid[ip, ii] = val
        except ImportError:
            if verbose:
                print("joblib not available; falling back to serial.")
            n_jobs = 1

    if not (n_jobs and n_jobs > 1):
        for ip in range(nP):
            for ii in range(nI):
                _, _, val = _eval_point(ip, ii)
                loss_grid[ip, ii] = val
            if verbose:
                best = np.unravel_index(np.nanargmin(loss_grid), loss_grid.shape)
                print(f"P row {ip+1:>3}/{nP}: best so far "
                      f"P={P_grid[best[0]]:.3g}, I={I_grid[best[1]]:.3g}, "
                      f"loss={loss_grid[best]:.6g}")

    best = np.unravel_index(np.nanargmin(loss_grid), loss_grid.shape)
    return loss_grid, float(P_grid[best[0]]), float(I_grid[best[1]])


def grid_search_PI(
    scanner,
    h_hat: np.ndarray,
    d_meas: np.ndarray,
    A_meas: np.ndarray,
    P_grid: Optional[np.ndarray] = None,
    I_grid: Optional[np.ndarray] = None,
    w_A: float = 0.3,
    line_kwargs: Optional[Dict] = None,
    n_jobs: int = 1,
    verbose: bool = True,
) -> Tuple[np.ndarray, float, float]:
    """Single-line convenience wrapper around grid_search_PI_multi."""
    return grid_search_PI_multi(
        scanner,
        line_data=[{'h_hat': h_hat, 'd_meas': d_meas, 'A_meas': A_meas,
                    'line_kwargs': line_kwargs}],
        P_grid=P_grid, I_grid=I_grid, w_A=w_A,
        n_jobs=n_jobs, verbose=verbose,
    )


# =============================================================================
# Sub-grid parabolic refinement (cheap post-processing)
# =============================================================================

def parabolic_refine_2d(
    loss_grid: np.ndarray,
    x_grid: np.ndarray,
    y_grid: np.ndarray,
) -> Tuple[float, float]:
    """Separable axis-by-axis parabolic refinement in log10 space.

    Fits a 1D parabola through 3 points along each axis through the grid
    minimum and returns the sub-grid minimizer. Useful for resolving a finer
    estimate than the log-grid spacing without a second optimizer.

    If the minimum is on the boundary or the local curvature is non-positive,
    the bare grid minimum is returned.
    """
    loss_grid = np.asarray(loss_grid, dtype=float)
    x_grid = np.asarray(x_grid, dtype=float)
    y_grid = np.asarray(y_grid, dtype=float)
    nx, ny = loss_grid.shape
    i0, j0 = np.unravel_index(np.nanargmin(loss_grid), loss_grid.shape)

    if i0 in (0, nx - 1) or j0 in (0, ny - 1):
        return float(x_grid[i0]), float(y_grid[j0])

    def _refine_axis(coords3, vals3, fallback):
        lc = np.log10(coords3)
        M = np.column_stack([np.ones(3), lc, lc ** 2])
        try:
            a, b, c = np.linalg.solve(M, vals3)
        except np.linalg.LinAlgError:
            return float(fallback)
        if c <= 0:
            return float(fallback)
        lc_star = -b / (2 * c)
        lc_star = float(np.clip(lc_star, lc[0], lc[-1]))
        return 10.0 ** lc_star

    x_star = _refine_axis(
        x_grid[i0 - 1:i0 + 2], loss_grid[i0 - 1:i0 + 2, j0], x_grid[i0],
    )
    y_star = _refine_axis(
        y_grid[j0 - 1:j0 + 2], loss_grid[i0, j0 - 1:j0 + 2], y_grid[j0],
    )
    return float(x_star), float(y_star)


# =============================================================================
# Visualization helper (optional, requires matplotlib)
# =============================================================================

def plot_loss_grid(
    loss_grid: np.ndarray,
    x_grid: np.ndarray,
    y_grid: np.ndarray,
    x_label: str = 'x',
    y_label: str = 'y',
    title: Optional[str] = None,
    log_loss: bool = True,
    ax=None,
):
    """Heatmap of the 2D loss on log-log axes; marks the minimum with a star."""
    import matplotlib.pyplot as plt
    if ax is None:
        _, ax = plt.subplots(figsize=(6.2, 5.2))
    Z = np.log10(np.where(loss_grid > 0, loss_grid, np.nan)) if log_loss else loss_grid
    X, Y = np.meshgrid(x_grid, y_grid, indexing='ij')
    pcm = ax.pcolormesh(X, Y, Z, shading='auto', cmap='viridis_r')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    plt.colorbar(pcm, ax=ax, label=('log10 loss' if log_loss else 'loss'))
    i, j = np.unravel_index(np.nanargmin(loss_grid), loss_grid.shape)
    ax.plot(x_grid[i], y_grid[j], marker='*', color='red',
            markersize=14, markeredgecolor='white', linestyle='')
    if title:
        ax.set_title(title)
    return ax


# =============================================================================
# Example wiring (commented; adapt to your data structures)
# =============================================================================
#
# import numpy as np
# from approach_curves import (SPMRK45ApproachModel, ApproachCurveLibrary,
#                              make_normalized_fd_lookup, RK45Options,
#                              NormalizedSPMParams)
# from scanner_fd import ScannerFD
# from calibrate_grid import (make_log_grid, grid_search_HE,
#                             build_library_from_fit, grid_search_PI_multi,
#                             parabolic_refine_2d, plot_loss_grid)
#
# # ----- shared grid -----
# multipliers = np.linspace(1, 9, 9)
# exponents   = np.arange(-6, -2)             # -6, -5, -4, -3
# grid        = np.concatenate([multipliers * 10.0**e for e in exponents]
#                              + [[10.0]])    # 37 points
#
# # ----- stage 1 -----
# # measured_curves: list of dicts in NORMALIZED units
# loss_HE, H_fit, E_fit = grid_search_HE(
#     measured_curves, base_params, R_fixed=R_norm,
#     SPMRK45ApproachModel=SPMRK45ApproachModel,
#     H_grid=grid, E_star_grid=grid,
#     solver=RK45Options(),
#     w_phi=1.0, n_jobs=4,
# )
# H_ref, E_ref = parabolic_refine_2d(loss_HE, grid, grid)
# plot_loss_grid(loss_HE, grid, grid, x_label='H', y_label='E*',
#                title='stage 1 loss')
#
# # ----- stage 2 -----
# library = build_library_from_fit(
#     measured_curves, base_params, R_fixed=R_norm,
#     H_fit=H_ref, E_star_fit=E_ref,
#     SPMRK45ApproachModel=SPMRK45ApproachModel,
#     ApproachCurveLibrary=ApproachCurveLibrary,
#     n_points=512, conv_L=scanner.conversion['A'],
#     smooth_method='savgol', smooth_window=7, smooth_polyorder=3,
# )
# fd_lookup = make_normalized_fd_lookup(library, conv_L=scanner.conversion['A'])
#
# # ----- stage 3 -----
# # build a ScannerFD bound to the library at the experimental drive
# scanner = ScannerFD(params, conversions, fd_model=fd_lookup(drive_nm))
#
# line_data = [
#     {'h_hat': h_hat_slow, 'd_meas': d_slow, 'A_meas': A_slow,
#      'line_kwargs': {'scan_speed_hat': v_slow, 'dx_hat': dx,
#                      'A_filter_order': 1, 'tau_A': tau_A_init,
#                      'z_filter_order': 1, 'tau_z': tau_z_init,
#                      'setpoint': 0.8, 'T_I': 0.0}},
#     {'h_hat': h_hat_fast, 'd_meas': d_fast, 'A_meas': A_fast,
#      'line_kwargs': {'scan_speed_hat': v_fast, 'dx_hat': dx,
#                      'A_filter_order': 1, 'tau_A': tau_A_init,
#                      'z_filter_order': 1, 'tau_z': tau_z_init,
#                      'setpoint': 0.8, 'T_I': 0.0}},
# ]
# loss_PI, P_fit, I_fit = grid_search_PI_multi(
#     scanner, line_data, P_grid=grid, I_grid=grid,
#     w_A=0.3, n_jobs=4,
# )
# P_ref, I_ref = parabolic_refine_2d(loss_PI, grid, grid)
# plot_loss_grid(loss_PI, grid, grid, x_label='P', y_label='I',
#                title='stage 3 loss')


In [ ]:
# Stage-2 FD-curve surface interpolation for ScannerFD.

"""
Stage-2 FD-curve surface interpolation for ScannerFD.

Purpose
-------
Start from measured force-distance / amplitude-distance / phase-distance curves
at discrete drive amplitudes and build a smooth lookup that supports continuous
changes of drive amplitude and setpoint during scanning.

Main objects
------------
FDCurve1D
    Single-drive interpolator, evaluate(d) -> (A, phi, F).

FDDriveSurface
    2D surface over (drive, distance_coordinate). It is built once from the
    measured FD curves, then queried cheaply at arbitrary drive.

make_normalized_fd_lookup
    Adapter from physical nm/degree units to the normalized units used inside
    ScannerFD.

attach_continuous_fd_to_scanner
    Monkey-patch helper for an existing ScannerFD instance. It adds
    scanner.set_drive(...), scanner.scan_line_pi_drive(...), and
    scanner.scan_line_trace_retrace_pi_drive(...). If you prefer permanent
    changes, copy these methods into your ScannerFD class.

Conventions
-----------
- d increases away from the sample. Large d is far field.
- The measured d origin can be arbitrary; if needed, pre-shift each measured
  curve before adding it, or pass d_offset_nm in add_curve().
- A0 is the free amplitude for each curve. By default it is estimated from the
  largest-d tail, which matches separation-like FD curves.
- The recommended interpolation coordinate is x = d / A0, because AM-AFM
  amplitude/phase curves often collapse better in drive-normalized distance.
  For fixed calibrated absolute separation axes, use x_mode='absolute_d'.
"""
from __future__ import annotations

from dataclasses import dataclass
from types import MethodType
from typing import Callable, Dict, Iterable, List, Optional, Tuple, Union

import numpy as np
from scipy.interpolate import PchipInterpolator, RegularGridInterpolator, interp1d

ArrayLike = Union[float, np.ndarray, Iterable[float]]


def _as_1d(x: ArrayLike, name: str) -> np.ndarray:
    arr = np.asarray(x, dtype=float).ravel()
    if arr.size == 0:
        raise ValueError(f"{name} is empty")
    return arr


def _finite_mask(*arrays: np.ndarray) -> np.ndarray:
    mask = np.ones_like(arrays[0], dtype=bool)
    for a in arrays:
        mask &= np.isfinite(a)
    return mask


def _average_duplicate_x(x: np.ndarray, *ys: np.ndarray) -> Tuple[np.ndarray, ...]:
    """Sort by x and average duplicate x values for stable interpolation."""
    order = np.argsort(x)
    x = x[order]
    ys = [np.asarray(y)[order] for y in ys]

    xu, inv = np.unique(x, return_inverse=True)
    if xu.size == x.size:
        return (x, *ys)

    y_out = []
    for y in ys:
        accum = np.zeros_like(xu, dtype=float)
        count = np.zeros_like(xu, dtype=float)
        np.add.at(accum, inv, y)
        np.add.at(count, inv, 1.0)
        y_out.append(accum / np.maximum(count, 1.0))
    return (xu, *y_out)


def _unwrap_phase_deg(phi_deg: np.ndarray) -> np.ndarray:
    return np.rad2deg(np.unwrap(np.deg2rad(phi_deg)))


def _safe_pchip(x: np.ndarray, y: np.ndarray, left: float, right: float):
    """PCHIP when possible; linear fallback for very short curves."""
    if x.size >= 3:
        return PchipInterpolator(x, y, extrapolate=False)
    return interp1d(x, y, kind="linear", bounds_error=False, fill_value=(left, right))


@dataclass
class FDCurve1D:
    """Single-drive FD evaluator.

    d, A and F use the same unit system supplied at construction. phi is degrees.
    d must increase away from the sample. Out-of-range behavior is clamped:
    near-contact side uses the first sampled value; far-field side uses A0,
    far-field phase, and F=0.
    """

    d: np.ndarray
    A: np.ndarray
    phi: np.ndarray
    F: Optional[np.ndarray] = None
    A0: Optional[float] = None
    kind: str = "pchip"

    def __post_init__(self) -> None:
        d = _as_1d(self.d, "d")
        A = _as_1d(self.A, "A")
        phi = _as_1d(self.phi, "phi")
        F = np.zeros_like(d) if self.F is None else _as_1d(self.F, "F")
        if not (d.shape == A.shape == phi.shape == F.shape):
            raise ValueError("d, A, phi, and F must have the same shape")

        mask = _finite_mask(d, A, phi, F)
        d, A, phi, F = d[mask], A[mask], phi[mask], F[mask]
        if d.size < 2:
            raise ValueError("Need at least two finite samples for FDCurve1D")

        d, A, phi, F = _average_duplicate_x(d, A, phi, F)
        if d.size < 2 or not np.all(np.diff(d) > 0):
            raise ValueError("d must contain at least two unique increasing values")

        self.d = d
        self.A = A
        self.phi = phi
        self.F = F
        self.A0 = float(A[-1] if self.A0 is None else self.A0)

        phi_unwrapped = _unwrap_phase_deg(phi)

        if self.kind == "pchip":
            self._A_interp = _safe_pchip(d, A, left=A[0], right=self.A0)
            self._phi_interp = _safe_pchip(d, phi_unwrapped, left=phi_unwrapped[0], right=phi_unwrapped[-1])
            self._F_interp = _safe_pchip(d, F, left=F[0], right=0.0)
        else:
            self._A_interp = interp1d(d, A, kind=self.kind, bounds_error=False, fill_value=(A[0], self.A0))
            self._phi_interp = interp1d(d, phi_unwrapped, kind=self.kind, bounds_error=False,
                                        fill_value=(phi_unwrapped[0], phi_unwrapped[-1]))
            self._F_interp = interp1d(d, F, kind=self.kind, bounds_error=False, fill_value=(F[0], 0.0))

    # def _fit_far_tail(self, frac=0.15, model='exp'):
    #     """Fit asymptotic decay parameters for A, phi, F using the last
    #     `frac` fraction of measured points."""
    #     n = len(self.d)
    #     k = max(5, int(frac * n))
    #     d_tail = self.d[-k:]
    #     if model == 'exp':
    #         # A0 - A = alpha * exp(-(d-d_ref)/lambda)
    #         d_ref = self.d[-1]
    #         y = np.maximum(self.A0 - self.A[-k:], 1e-12)
    #         # log y = log alpha - (d - d_ref)/lambda
    #         coeffs = np.polyfit(d_tail - d_ref, np.log(y), 1)
    #         lam_A = -1.0 / coeffs[0] if coeffs[0] < 0 else np.inf
    #         alpha_A = np.exp(coeffs[1])
    #         # similar for phi (deviation from phi_free) and F
    #     return dict(d_ref=d_ref, alpha_A=alpha_A, lam_A=lam_A, ...)
    
    def _eval_far(self, dq, params):
        d_ref = params['d_ref']
        A   = self.A0     - params['alpha_A']   * np.exp(-(dq - d_ref) / params['lam_A'])
        phi = self.phi_free - params['alpha_phi'] * np.exp(-(dq - d_ref) / params['lam_phi'])
        F   =                params['alpha_F']   * np.exp(-(dq - d_ref) / params['lam_F'])
        return A, phi, F
        
    def evaluate(self, d_query: ArrayLike):
        scalar = np.isscalar(d_query)
        dq = np.atleast_1d(np.asarray(d_query, dtype=float))
        dq_clip = np.clip(dq, self.d[0], self.d[-1])

        A = np.asarray(self._A_interp(dq_clip), dtype=float)
        phi = np.asarray(self._phi_interp(dq_clip), dtype=float)
        F = np.asarray(self._F_interp(dq_clip), dtype=float)

        # Explicit far-field clamp. This avoids PCHIP extrapolation surprises.
        far = dq > self.d[-1]
        if np.any(far):
            A[far] = self.A0
            phi[far] = self.phi[-1]
            F[far] = 0.0

        # far = dq > self.d[-1]
        # if np.any(far):
        #     A_far, phi_far, F_far = self._eval_far(dq[far], self.tail_params)
        #     A[far], phi[far], F[far] = A_far, phi_far, F_far

        near = dq < self.d[0]
        if np.any(near):
            A[near] = self.A[0]
            phi[near] = self.phi[0]
            F[near] = self.F[0]

        if scalar:
            return float(A[0]), float(phi[0]), float(F[0])
        return A, phi, F

    def __call__(self, d_query: ArrayLike):
        return self.evaluate(d_query)


class FDDriveSurface:
    """Continuous FD surface over drive and distance.

    The surface stores normalized amplitude a=A/A0 and unwrapped phase. The
    drive interpolation is linear. The per-curve resampling along distance is
    PCHIP by default.
    """

    def __init__(
        self,
        *,
        x_mode: str = "d_over_A0",
        far_field_frac: float = 0.10,
        common_range: str = "intersection",
        n_x: int = 512,
        interp_kind_1d: str = "pchip",
        right_fill_free: bool = True,
    ) -> None:
        if x_mode not in {"d_over_A0", "absolute_d"}:
            raise ValueError("x_mode must be 'd_over_A0' or 'absolute_d'")
        if common_range not in {"intersection", "union"}:
            raise ValueError("common_range must be 'intersection' or 'union'")

        self.x_mode = x_mode
        self.far_field_frac = float(far_field_frac)
        self.common_range = common_range
        self.n_x = int(n_x)
        self.interp_kind_1d = interp_kind_1d
        self.right_fill_free = bool(right_fill_free)

        self._raw: Dict[float, Dict[str, np.ndarray]] = {}
        self._built = False

    @property
    def drives(self) -> np.ndarray:
        return np.array(sorted(self._raw.keys()), dtype=float)

    @property
    def x_grid(self) -> np.ndarray:
        self._require_built()
        return self._x_grid

    def add_curve(
        self,
        drive_nm: float,
        d_nm: ArrayLike,
        A_nm: ArrayLike,
        phi_deg: ArrayLike,
        F: Optional[ArrayLike] = None,
        *,
        A0_nm: Optional[float] = None,
        d_offset_nm: float = 0.0,
        far_field_side: str = "largest_d",
    ) -> None:
        """Add one measured curve.

        Parameters
        ----------
        drive_nm
            Commanded or calibrated drive/free-amplitude label in nm.
        d_nm
            Separation-like coordinate. Larger d should mean farther from the
            surface. If your measured origin is arbitrary, use d_offset_nm.
        A_nm, phi_deg
            Measured amplitude and phase.
        A0_nm
            Optional explicit free amplitude. If omitted, it is estimated from
            the far-field tail specified by far_field_side.
        far_field_side
            'largest_d', 'smallest_d', 'start', or 'end'. For your screenshot,
            use 'largest_d' or the default.
        """
        drv = float(drive_nm)
        d = _as_1d(d_nm, "d_nm") + float(d_offset_nm)
        A = _as_1d(A_nm, "A_nm")
        phi = _as_1d(phi_deg, "phi_deg")
        F_arr = np.zeros_like(d) if F is None else _as_1d(F, "F")
        if not (d.shape == A.shape == phi.shape == F_arr.shape):
            raise ValueError("d, A, phi, and F must have the same shape")

        mask = _finite_mask(d, A, phi, F_arr)
        d, A, phi, F_arr = d[mask], A[mask], phi[mask], F_arr[mask]
        if d.size < 3:
            raise ValueError(f"drive {drv}: need at least 3 finite samples")

        d, A, phi, F_arr = _average_duplicate_x(d, A, phi, F_arr)

        if A0_nm is None:
            n_far = max(1, int(round(self.far_field_frac * d.size)))
            if far_field_side == "largest_d":
                idx_far = np.argsort(d)[-n_far:]
            elif far_field_side == "smallest_d":
                idx_far = np.argsort(d)[:n_far]
            elif far_field_side == "start":
                idx_far = np.arange(n_far)
            elif far_field_side == "end":
                idx_far = np.arange(d.size - n_far, d.size)
            else:
                raise ValueError("far_field_side must be largest_d, smallest_d, start, or end")
            A0 = float(np.nanmedian(A[idx_far]))
        else:
            A0 = float(A0_nm)

        if not np.isfinite(A0) or A0 <= 0:
            raise ValueError(f"drive {drv}: invalid A0={A0}")

        self._raw[drv] = {
            "d": d,
            "A": A,
            "phi": phi,
            "F": F_arr,
            "A0": np.array([A0], dtype=float),
        }
        self._built = False

    @classmethod
    def from_measured_fd(
        cls,
        measured_fd: Dict[float, Dict[str, ArrayLike]],
        *,
        x_mode: str = "d_over_A0",
        far_field_frac: float = 0.10,
        common_range: str = "intersection",
        n_x: int = 512,
        far_field_side: str = "largest_d",
        drive_key_is_A0: bool = False,
    ) -> "FDDriveSurface":
        """Build from {drive: {'d':..., 'A':..., 'phi':..., optional 'F':...}}."""
        surf = cls(
            x_mode=x_mode,
            far_field_frac=far_field_frac,
            common_range=common_range,
            n_x=n_x,
        )
        for drv, c in measured_fd.items():
            A0 = float(drv) if drive_key_is_A0 else c.get("A0_nm", None)
            surf.add_curve(
                float(drv), c["d"], c["A"], c["phi"], F=c.get("F"),
                A0_nm=A0, far_field_side=far_field_side,
            )
        surf.build()
        return surf

    def build(self) -> "FDDriveSurface":
        drives = self.drives
        if drives.size < 1:
            raise ValueError("No curves have been added")
        if drives.size < 2:
            raise ValueError("Need at least two drives for continuous drive interpolation")

        x_mins: List[float] = []
        x_maxs: List[float] = []
        prepared: List[Tuple[float, np.ndarray, np.ndarray, np.ndarray, np.ndarray, float]] = []

        for drv in drives:
            c = self._raw[float(drv)]
            d = c["d"]
            A = c["A"]
            phi = _unwrap_phase_deg(c["phi"])
            F = c["F"]
            A0 = float(c["A0"][0])

            x = d / A0 if self.x_mode == "d_over_A0" else d.copy()
            a = A / A0

            x, a, phi, F = _average_duplicate_x(x, a, phi, F)
            x_mins.append(float(x[0]))
            x_maxs.append(float(x[-1]))
            prepared.append((float(drv), x, a, phi, F, A0))

        if self.common_range == "intersection":
            x_min = max(x_mins)
            x_max = min(x_maxs)
            if not x_max > x_min:
                raise ValueError(
                    "No overlapping distance range across curves. Use common_range='union', "
                    "trim curves, or align d-offsets first."
                )
        else:
            x_min = min(x_mins)
            x_max = max(x_maxs)

        self._x_grid = np.linspace(x_min, x_max, self.n_x)
        nD, nX = drives.size, self._x_grid.size
        a_grid = np.empty((nD, nX), dtype=float)
        phi_grid = np.empty((nD, nX), dtype=float)
        F_grid = np.empty((nD, nX), dtype=float)
        A0_by_drive = np.empty(nD, dtype=float)

        for i, (drv, x, a, phi, F, A0) in enumerate(prepared):
            # Far-field fill: a -> 1, phase -> far-field phase, F -> 0.
            a_left, a_right = float(a[0]), 1.0 if self.right_fill_free else float(a[-1])
            phi_left, phi_right = float(phi[0]), float(phi[-1])
            F_left, F_right = float(F[0]), 0.0

            if self.interp_kind_1d == "pchip":
                a_interp = _safe_pchip(x, a, a_left, a_right)
                p_interp = _safe_pchip(x, phi, phi_left, phi_right)
                f_interp = _safe_pchip(x, F, F_left, F_right)
                xq = np.clip(self._x_grid, x[0], x[-1])
                a_grid[i] = a_interp(xq)
                phi_grid[i] = p_interp(xq)
                F_grid[i] = f_interp(xq)
                # Manual edge fill after clipping.
                left = self._x_grid < x[0]
                right = self._x_grid > x[-1]
                a_grid[i, left] = a_left
                phi_grid[i, left] = phi_left
                F_grid[i, left] = F_left
                a_grid[i, right] = a_right
                phi_grid[i, right] = phi_right
                F_grid[i, right] = F_right
            else:
                a_grid[i] = np.interp(self._x_grid, x, a, left=a_left, right=a_right)
                phi_grid[i] = np.interp(self._x_grid, x, phi, left=phi_left, right=phi_right)
                F_grid[i] = np.interp(self._x_grid, x, F, left=F_left, right=F_right)

            A0_by_drive[i] = A0

        self._drives = drives
        self._A0_by_drive = A0_by_drive
        self._a_grid = a_grid
        self._phi_grid = phi_grid
        self._F_grid = F_grid

        self._a_rgi = RegularGridInterpolator((drives, self._x_grid), a_grid,
                                               method="linear", bounds_error=False, fill_value=None)
        self._phi_rgi = RegularGridInterpolator((drives, self._x_grid), phi_grid,
                                                 method="linear", bounds_error=False, fill_value=None)
        self._F_rgi = RegularGridInterpolator((drives, self._x_grid), F_grid,
                                               method="linear", bounds_error=False, fill_value=None)
        self._A0_rgi = interp1d(drives, A0_by_drive, kind="linear", bounds_error=False,
                                fill_value=(A0_by_drive[0], A0_by_drive[-1]))
        self._built = True
        return self

    def _require_built(self) -> None:
        if not self._built:
            self.build()

    def evaluate(self, d_nm: ArrayLike, drive_nm: float):
        """Evaluate the FD surface at arbitrary physical drive and distance.

        Returns A_nm, phi_deg, F. Scalar input returns scalars; array input
        returns arrays.
        """
        self._require_built()
        scalar = np.isscalar(d_nm)
        d_arr = np.atleast_1d(np.asarray(d_nm, dtype=float))
        drv = float(drive_nm)
        drv_clip = float(np.clip(drv, self._drives[0], self._drives[-1]))

        # Use the target drive as A0 scaling. This lets the nearest normalized
        # curve scale reasonably if drive is slightly outside the measured range.
        A0_target = float(self._A0_rgi(drv_clip))
        if self.x_mode == "d_over_A0":
            xq = d_arr / max(A0_target, 1e-30)
        else:
            xq = d_arr
        xq = np.clip(xq, self._x_grid[0], self._x_grid[-1])

        pts = np.column_stack([np.full_like(xq, drv_clip, dtype=float), xq])
        a = np.asarray(self._a_rgi(pts), dtype=float)
        phi = np.asarray(self._phi_rgi(pts), dtype=float)
        F = np.asarray(self._F_rgi(pts), dtype=float)

        A = a * A0_target
        if scalar:
            return float(A[0]), float(phi[0]), float(F[0])
        return A, phi, F

    def fd_at_drive(self, drive_nm: float, n_points: Optional[int] = None) -> FDCurve1D:
        """Return a single-drive FDCurve1D at any target drive."""
        self._require_built()
        n = self.n_x if n_points is None else int(n_points)
        x = self._x_grid if n == self._x_grid.size else np.linspace(self._x_grid[0], self._x_grid[-1], n)
        drv = float(drive_nm)
        drv_clip = float(np.clip(drv, self._drives[0], self._drives[-1]))
        A0_target = float(self._A0_rgi(drv_clip))

        d = x * A0_target if self.x_mode == "d_over_A0" else x.copy()
        A, phi, F = self.evaluate(d, drv)
        return FDCurve1D(d=d, A=A, phi=phi, F=F, A0=A0_target, kind="pchip")

    def __call__(self, drive_nm: float) -> FDCurve1D:
        return self.fd_at_drive(drive_nm)

    def diagnostics(self) -> Dict[str, np.ndarray]:
        self._require_built()
        return {
            "drives": self._drives.copy(),
            "A0_by_drive": self._A0_by_drive.copy(),
            "x_grid": self._x_grid.copy(),
            "a_grid": self._a_grid.copy(),
            "phi_grid": self._phi_grid.copy(),
            "F_grid": self._F_grid.copy(),
        }


def make_normalized_fd_lookup(
    surface: FDDriveSurface,
    *,
    conv_L: float,
    conv_A: Optional[float] = None,
    conv_F: float = 1.0,
    n_points: Optional[int] = None,
) -> Callable[[float], FDCurve1D]:
    """Return lookup(drive_nm) -> FDCurve1D in ScannerFD normalized units.

    conv_L: d_hat = d_nm * conv_L
    conv_A: A_hat = A_nm * conv_A. Defaults to conv_L.
    conv_F: F_hat = F_physical_or_library * conv_F.
    """
    conv_A = conv_L if conv_A is None else float(conv_A)
    conv_L = float(conv_L)
    conv_F = float(conv_F)

    def lookup(drive_nm: float) -> FDCurve1D:
        fd = surface.fd_at_drive(float(drive_nm), n_points=n_points)
        return FDCurve1D(
            d=fd.d * conv_L,
            A=fd.A * conv_A,
            phi=fd.phi,
            F=fd.F * conv_F,
            A0=fd.A0 * conv_A,
            kind="pchip",
        )

    return lookup


def attach_continuous_fd_to_scanner(
    scanner,
    fd_lookup: Callable[[float], FDCurve1D],
    *,
    conv_A: Optional[float] = None,
) -> object:
    """Attach continuous-drive methods to an existing ScannerFD instance.

    Added methods
    -------------
    scanner.set_drive(drive_nm=None, drive_hat=None)
    scanner.scan_line_pi_drive(h_hat, drive_nm=..., setpoint=..., **kwargs)
    scanner.scan_line_trace_retrace_pi_drive(h_hat, drive_nm=..., setpoint=..., **kwargs)

    This does not overwrite your existing scan_line_pi methods.
    """
    if conv_A is None:
        conv_A = float(scanner.conversion.get("A", 1.0))
    conv_A = float(conv_A)

    scanner.fd_lookup = fd_lookup

    def set_drive(self, drive_nm: Optional[float] = None, drive_hat: Optional[float] = None):
        if drive_nm is None and drive_hat is None:
            raise ValueError("Pass either drive_nm or drive_hat")
        if drive_nm is None:
            drive_nm = float(drive_hat) / conv_A
        drive_nm = float(drive_nm)
        drive_hat = drive_nm * conv_A

        fd_model = self.fd_lookup(drive_nm)
        self.fd_model = fd_model

        # Important: the lookup may use drive_nm as a command/label, while the
        # actual free amplitude A0 comes from the measured far-field curve. Use
        # fd_model.A0, not drive_nm, for the scanner feedback normalization.
        A_hat_effective = float(getattr(fd_model, "A0", drive_hat))
        A_nm_effective = A_hat_effective / conv_A

        if hasattr(self, "param") and isinstance(self.param, dict):
            self.param["A"] = A_nm_effective
        if hasattr(self, "param_norm") and isinstance(self.param_norm, dict):
            self.param_norm["A"] = A_hat_effective
            if "Q" in self.param_norm:
                self.param_norm["C3"] = A_hat_effective / float(self.param_norm["Q"])

        self.drive_command_nm = drive_nm
        self.drive_nm = A_nm_effective
        self.drive_hat = A_hat_effective
        return fd_model

    def scan_line_pi_drive(self, h_hat, *, drive_nm: Optional[float] = None,
                           drive_hat: Optional[float] = None,
                           setpoint: Optional[float] = 0.6,
                           setpoint_hat: Optional[float] = None,
                           **kwargs):
        if drive_nm is not None or drive_hat is not None:
            self.set_drive(drive_nm=drive_nm, drive_hat=drive_hat)
        if setpoint_hat is not None:
            A_hat = float(self.param_norm["A"])
            setpoint = float(setpoint_hat) / A_hat
        if setpoint is None:
            raise ValueError("Pass setpoint fraction or setpoint_hat")
        return self.scan_line_pi(h_hat=h_hat, setpoint=float(setpoint), **kwargs)
        # return self.scan_line_substepped(h_hat=h_hat, setpoint=float(setpoint), **kwargs)
        
    def scan_line_trace_retrace_pi_drive(self, h_hat, *, drive_nm: Optional[float] = None,
                                         drive_hat: Optional[float] = None,
                                         setpoint: Optional[float] = 0.6,
                                         setpoint_hat: Optional[float] = None,
                                         **kwargs):
        if drive_nm is not None or drive_hat is not None:
            self.set_drive(drive_nm=drive_nm, drive_hat=drive_hat)
        if setpoint_hat is not None:
            A_hat = float(self.param_norm["A"])
            setpoint = float(setpoint_hat) / A_hat
        if setpoint is None:
            raise ValueError("Pass setpoint fraction or setpoint_hat")
        return self.scan_line_trace_retrace_pi(h_hat=h_hat, setpoint=float(setpoint), **kwargs)

    scanner.set_drive = MethodType(set_drive, scanner)
    scanner.scan_line_pi_drive = MethodType(scan_line_pi_drive, scanner)
    scanner.scan_line_trace_retrace_pi_drive = MethodType(scan_line_trace_retrace_pi_drive, scanner)
    return scanner


def detect_bad_curves(
    measured_fd: Dict[float, Dict[str, ArrayLike]],
    *,
    phase_jump_threshold_deg: float = 30.0,
    amp_jump_threshold_rel: float = 0.20,
) -> Dict[float, Dict[str, float]]:
    """Flag curves with abrupt phase or normalized-amplitude jumps."""
    flags: Dict[float, Dict[str, float]] = {}
    for drv, c in measured_fd.items():
        d = _as_1d(c["d"], "d")
        A = _as_1d(c["A"], "A")
        phi = _as_1d(c["phi"], "phi")
        mask = _finite_mask(d, A, phi)
        d, A, phi = d[mask], A[mask], phi[mask]
        if d.size < 3:
            continue
        order = np.argsort(d)
        d, A, phi = d[order], A[order], phi[order]
        A0 = float(c.get("A0_nm", np.nanmedian(A[np.argsort(d)[-max(1, int(0.1 * d.size)):]])))
        dphi = np.abs(np.diff(_unwrap_phase_deg(phi)))
        da = np.abs(np.diff(A / max(A0, 1e-30)))
        if np.nanmax(dphi) > phase_jump_threshold_deg or np.nanmax(da) > amp_jump_threshold_rel:
            i = int(np.nanargmax(dphi)) if np.nanmax(dphi) / phase_jump_threshold_deg >= np.nanmax(da) / amp_jump_threshold_rel else int(np.nanargmax(da))
            flags[float(drv)] = {
                "max_dphi_deg": float(np.nanmax(dphi)),
                "max_da_rel": float(np.nanmax(da)),
                "d_at_jump_nm": float(d[i]),
            }
    return flags


# -----------------------------------------------------------------------------
# Minimal usage example
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    # measured_fd should look like:
    # measured_fd = {
    #     2.0: {"d": d_nm_2, "A": A_nm_2, "phi": phi_deg_2},
    #     8.0: {"d": d_nm_8, "A": A_nm_8, "phi": phi_deg_8},
    # }
    pass


In [ ]:
import numpy as np
from types import MethodType


def attach_continuous_fd_to_scanner(
    scanner,
    fd_lookup,
    conv_A=None,
):
    """
    Attach continuous-drive FD lookup to an existing ScannerFD instance.

    This works even if scanner only has scan_line_pi() and does not already
    have scan_line_trace_retrace_pi().

    Parameters
    ----------
    scanner : existing ScannerFD-like object
        Must have scan_line_pi().
    fd_lookup : callable
        fd_lookup(drive_nm) -> FDCurve1D in normalized units.
    conv_A : float
        Conversion from drive_nm to scanner-normalized amplitude.
        Usually scanner.conversion["A"].
    """

    if conv_A is None:
        conv_A = scanner.conversion.get("A", 1.0)

    scanner._continuous_fd_lookup = fd_lookup
    scanner._continuous_fd_conv_A = float(conv_A)

    def set_drive(self, drive_nm=None, drive_hat=None):
        if drive_nm is None and drive_hat is None:
            raise ValueError("Pass either drive_nm or drive_hat.")

        if drive_nm is None:
            drive_hat = float(drive_hat)
            drive_nm = drive_hat / self._continuous_fd_conv_A
        else:
            drive_nm = float(drive_nm)
            drive_hat = drive_nm * self._continuous_fd_conv_A

        fd_model = self._continuous_fd_lookup(drive_nm)

        # Classic ScannerFD expects fd_model directly.
        self.fd_model = fd_model

        # Keep amplitude bookkeeping consistent.
        if hasattr(self, "param"):
            self.param["A"] = drive_nm

        if hasattr(self, "param_norm"):
            self.param_norm["A"] = drive_hat
            if "Q" in self.param_norm:
                self.param_norm["C3"] = self.param_norm["A"] / self.param_norm["Q"]

        self.drive_nm_current = drive_nm
        self.drive_hat_current = drive_hat

        return fd_model

    def _pack_carry_state_from_trace(out):
        """
        Convert final state from trace into initial state for retrace.
        Only includes keys that actually exist in your scan_line_pi output.
        """
        pairs = {
            "d_cmd_init_hat": "d_cmd_final_hat",
            "d_act_init_hat": "d_final_hat",
            "A_meas_init_hat": "A_meas_final_hat",
            "phi_meas_init_deg": "phi_meas_final_deg",
            "integ_init": "integ_final",
            "drift_init_hat": "drift_final_hat",
        }

        carry = {}
        for init_key, final_key in pairs.items():
            if final_key in out:
                carry[init_key] = out[final_key]

        return carry

    def _flip_line_output(out):
        """
        Reverse 1D pixel arrays so retrace is returned in the same x-order
        as trace.
        """
        flipped = {}
        for k, v in out.items():
            if isinstance(v, np.ndarray) and v.ndim == 1:
                flipped[k] = v[::-1].copy()
            else:
                flipped[k] = v
        return flipped

    def scan_line_trace_retrace_pi_fallback(
        self,
        h_hat,
        retrace_carry_state=True,
        retrace_use_reversed_topography=True,
        **line_kwargs,
    ):
        h_hat = np.asarray(h_hat, dtype=float)

        # Trace
        out_trace = self.scan_line_pi(
            h_hat=h_hat,
            **line_kwargs,
        )

        # Retrace topography
        if retrace_use_reversed_topography:
            h_retrace = h_hat[::-1]
        else:
            h_retrace = h_hat.copy()

        # Carry state if available
        retrace_kwargs = dict(line_kwargs)
        if retrace_carry_state:
            retrace_kwargs.update(_pack_carry_state_from_trace(out_trace))

        out_retrace_raw = self.scan_line_pi(
            h_hat=h_retrace,
            **retrace_kwargs,
        )

        out_retrace = _flip_line_output(out_retrace_raw)

        return {
            "trace": out_trace,
            "retrace": out_retrace,
        }

    def scan_line_trace_retrace_pi_drive(
        self,
        h_hat,
        drive_nm=None,
        drive_hat=None,
        setpoint=None,
        setpoint_hat=None,
        **kwargs,
    ):
        fd_model = self.set_drive(drive_nm=drive_nm, drive_hat=drive_hat)

        if setpoint is None:
            if setpoint_hat is None:
                raise ValueError("Pass either setpoint fraction or setpoint_hat.")

            if hasattr(fd_model, "A0"):
                A0_hat = float(fd_model.A0)
            elif hasattr(fd_model, "free_amplitude"):
                A0_hat = float(fd_model.free_amplitude())
            else:
                A0_hat = float(self.param_norm["A"])

            setpoint = float(setpoint_hat) / A0_hat

        # If your scanner already has a native trace/retrace method, use it.
        # Otherwise use the fallback we attach below.
        return self.scan_line_trace_retrace_pi(
            h_hat=h_hat,
            setpoint=float(setpoint),
            # drive_nm=drive_nm,
            **kwargs,
        )

    scanner.set_drive = MethodType(set_drive, scanner)

    if not hasattr(scanner, "scan_line_trace_retrace_pi"):
        scanner.scan_line_trace_retrace_pi = MethodType(
            scan_line_trace_retrace_pi_fallback,
            scanner,
        )

    scanner.scan_line_trace_retrace_pi_drive = MethodType(
        scan_line_trace_retrace_pi_drive,
        scanner,
    )

    return scanner

## Digital twin SPM

In [ ]:
# # ============================================================
# # Online integration / orchestration workflow
# # ============================================================
# class DigitalTwinSPM2D:
#     """
#     High-level workflow wrapper for:
#     - building a multi-drive FD library
#     - creating ScannerFD objects for online scans
#     - calibration against drive/setpoint grids
#     - quick operating-point evaluation for instrument integration
#     """

#     def __init__(
#         self,
#         params: Dict[str, float],
#         conversions: Dict[str, float],
#         omega_drive_hat: float = 1.0,
#         omega_ref_hat: float = 1.0,
#         solver: Optional[RK45Options] = None,
#         pi_noise: Optional[PIDriftNoiseOptions] = None,
#         attrs: Optional[Dict[str, Any]] = None,
#     ):
#         self.param_book = ParamBook.from_dicts(params, conversions)
#         self.norm_params = NormalizedSPMParams.from_param_book(
#             self.param_book,
#             omega_drive_hat=omega_drive_hat,
#             omega_ref_hat=omega_ref_hat,
#         )
#         self.solver = solver if solver is not None else RK45Options()
#         self.pi_noise = pi_noise if pi_noise is not None else PIDriftNoiseOptions()
#         self.attrs = {
#             "type": "DigitalTwinSPM2D",
#             "omega_drive_hat": float(omega_drive_hat),
#             "omega_ref_hat": float(omega_ref_hat),
#             "params_raw": dict(params),
#             "conversions": dict(conversions),
#         }
#         self.attrs.update(_copy_dict(attrs))

#         self.approach_model = SPMRK45ApproachModel(self.norm_params, self.solver)
#         self.fd_library: Optional[FDCurve2D] = None

#     def build_fd_library(
#         self,
#         drive_hat_grid: Sequence[float],
#         d_start_hat: float,
#         d_end_hat: float,
#         n_points: int = 256,
#         z0_hat: float = 0.0,
#         v0_hat: float = 0.0,
#         warm_start: bool = True,
#         clip: bool = True,
#         attrs: Optional[Dict[str, Any]] = None,
#     ) -> FDCurve2D:
#         builder = FDLibraryBuilder(self.approach_model, attrs={"workflow": "DigitalTwinSPM2D"})
#         self.fd_library = builder.build(
#             drive_hat_grid=drive_hat_grid,
#             d_start_hat=d_start_hat,
#             d_end_hat=d_end_hat,
#             n_points=n_points,
#             z0_hat=z0_hat,
#             v0_hat=v0_hat,
#             warm_start=warm_start,
#             clip=clip,
#             attrs=attrs,
#         )
#         return self.fd_library

#     def get_scanner(self, fd_library: Optional[FDCurve2D] = None, pi_noise: Optional[PIDriftNoiseOptions] = None) -> ScannerFD:
#         lib = self.fd_library if fd_library is None else fd_library
#         if lib is None:
#             raise ValueError("Build or provide an FD library first.")
#         return ScannerFD(
#             params=self.param_book.params_raw,
#             conversions=self.param_book.conversions,
#             fd_model=lib,
#             pi_noise=self.pi_noise if pi_noise is None else pi_noise,
#             attrs={"workflow": "DigitalTwinSPM2D"},
#         )

#     def simulate_line(self, h_hat, drive_hat, setpoint, **scan_kwargs):
#         scanner = self.get_scanner()
#         return scanner.scan_line_pi(h_hat=h_hat, drive_hat=drive_hat, setpoint=setpoint, **scan_kwargs)

#     def simulate_map(self, h_map_hat, drive_hat, setpoint, **scan_kwargs):
#         scanner = self.get_scanner()
#         return scanner.scan_map_pi(h_map_hat=h_map_hat, drive_hat=drive_hat, setpoint=setpoint, **scan_kwargs)

#     def calibrator(
#         self,
#         data: FDGridData,
#         surface_ref_hat: np.ndarray,
#         fd_library: Optional[FDCurve2D] = None,
#         fd_library_factory: Optional[Callable[[Dict[str, Any]], FDCurve2D]] = None,
#         line_kwargs_base: Optional[Dict[str, Any]] = None,
#         loss_weights: Optional[Dict[str, float]] = None,
#         remove_tilt_in_height: bool = True,
#         fit_subset: Optional[Sequence[Tuple[int, int]]] = None,
#     ) -> FDGridCalibrator2D:
#         lib = self.fd_library if fd_library is None else fd_library
#         return FDGridCalibrator2D(
#             data=data,
#             params_template=self.param_book.params_raw,
#             conversions=self.param_book.conversions,
#             surface_ref_hat=surface_ref_hat,
#             fd_library=lib,
#             fd_library_factory=fd_library_factory,
#             pi_noise=self.pi_noise,
#             line_kwargs_base=line_kwargs_base,
#             loss_weights=loss_weights,
#             remove_tilt_in_height=remove_tilt_in_height,
#             fit_subset=fit_subset,
#         )

#     def rank_operating_points(
#         self,
#         d_eff_hat: float,
#         drive_hat_candidates: Sequence[float],
#         setpoint_candidates: Sequence[float],
#         force_limit_hat: Optional[float] = None,
#     ) -> List[Dict[str, float]]:
#         if self.fd_library is None:
#             raise ValueError("Build or provide an FD library first.")
#         results: List[Dict[str, float]] = []
#         for drive_hat in drive_hat_candidates:
#             A0 = float(np.asarray(self.fd_library.free_amplitude(drive_hat)))
#             A_true, phi_true, F_max = self.fd_library.evaluate(d_eff_hat, drive_hat)
#             A_true = float(np.asarray(A_true))
#             phi_true = float(np.asarray(phi_true))
#             F_max = float(np.asarray(F_max))
#             for sp in setpoint_candidates:
#                 A_set = float(sp * A0)
#                 err = A_set - A_true
#                 feasible = True if force_limit_hat is None else (F_max <= force_limit_hat)
#                 score = abs(err)
#                 results.append({
#                     "drive_hat": float(drive_hat),
#                     "setpoint": float(sp),
#                     "A0_hat": float(A0),
#                     "A_true_hat": float(A_true),
#                     "phi_deg": float(phi_true),
#                     "F_max_hat": float(F_max),
#                     "err_hat": float(err),
#                     "score": float(score),
#                     "feasible": bool(feasible),
#                 })
#         results.sort(key=lambda x: (not x["feasible"], x["score"]))
#         return results


# # ============================================================
# # Example theta mapping for calibration
# # ============================================================
# def default_vector_to_theta_2d(x: np.ndarray) -> Dict[str, Any]:
#     """
#     Example mapping for transform-only calibration over a fixed FD library.

#     x = [
#         drive_scale,
#         drive_shift,
#         setpoint_scale,
#         setpoint_shift,
#         phase_scale,
#         phase_offset,
#         log_tau_A,
#         log_tau_phi,
#         log_tau_z,
#         P,
#         I,
#     ]
#     """
#     x = np.asarray(x, dtype=float)
#     return {
#         "drive_scale": x[0],
#         "drive_shift": x[1],
#         "setpoint_scale": x[2],
#         "setpoint_shift": x[3],
#         "phase_scale": x[4],
#         "phase_offset": x[5],
#         "tau_A": float(np.exp(x[6])),
#         "tau_phi": float(np.exp(x[7])),
#         "tau_z": float(np.exp(x[8])),
#         "P": float(x[9]),
#         "I": float(x[10]),
#     }

In [ ]:



# # ============================================================
# # Example usage
# # ============================================================
# if __name__ == "__main__":
#     # Example parameter set in the same style used earlier.
#     drive = 100e-9
#     params = {
#         "k": 1.0,
#         "A": drive,
#         "d0": 125e-9,
#         "a0": 0.164e-9,
#         "R": 20e-9,
#         "f0": 75e3,
#         "Q": 200.0,
#         "E_star": 1 / ((1 - 0.3**2) / (130e9) + (1 - 0.3**2) / (1.2e9)),
#         "H": 7.1e-20,
#     }
#     conversions = {
#         "k": 1.0,
#         "A": 1e9,
#         "d0": 1e9,
#         "a0": 1e9,
#         "R": 1e9,
#         "f0": 1 / (2 * np.pi * params["f0"]),
#         "Q": 1.0,
#         "E_star": 1e-9 / params["k"],
#         "H": 1 / (params["k"] * (1e-9) ** 2),
#     }

#     twin = DigitalTwinSPM2D(
#         params=params,
#         conversions=conversions,
#         solver=RK45Options(N_cycles=12, max_windows=20, demod_per_ref_cycle=25),
#         pi_noise=PIDriftNoiseOptions(drift_rel_std=1e-3, drift_tau_pixels=200.0, seed=1),
#     )

#     drive_grid = np.array([40.0, 70.0, 100.0, 130.0])
#     fd_library = twin.build_fd_library(
#         drive_hat_grid=drive_grid,
#         d_start_hat=twin.param_book.params_norm["d0"],
#         d_end_hat=-25.0,
#         n_points=128,
#     )
#     print("FD library summary:", fd_library.summary())

#     # Online line simulation example.
#     x = np.linspace(0, 4 * np.pi, 256)
#     h_line = 80.0 + 20.0 * (np.sin(x) > 0).astype(float)
#     out = twin.simulate_line(
#         h_hat=h_line,
#         drive_hat=100.0,
#         setpoint=0.97,
#         scan_speed_hat=1.0,
#         dx_hat=1.0,
#         P=0.1,
#         I=0.01,
#         A_filter_order=1,
#         tau_A=1e-2,
#         z_filter_order=1,
#         tau_z=1e-2,
#         d_cmd_clip=(-1000.0, 1000.0),
#     )
#     print("Line simulation keys:", sorted(out.keys()))

#     ranked = twin.rank_operating_points(
#         d_eff_hat=10.0,
#         drive_hat_candidates=[60.0, 80.0, 100.0, 120.0],
#         setpoint_candidates=[0.90, 0.95, 0.97, 0.99],
#         force_limit_hat=0.0,
#     )
#     print("Top 5 operating points:")
#     for row in ranked[:5]:
#         print(row)


# # ============================================================
# # Regenerating FD-library workflow extensions
# # ============================================================
# def default_vector_to_theta_2d_physics(x: np.ndarray) -> Dict[str, Any]:
#     """
#     Example mapping for joint calibration of:
#     - FD physics scales (C1, C2, a0)
#     - drive/setpoint/phase transforms
#     - filter/controller parameters

#     x = [
#         log_C1_scale,
#         log_C2_scale,
#         log_a0_scale,
#         drive_scale,
#         drive_shift,
#         setpoint_scale,
#         setpoint_shift,
#         phase_scale,
#         phase_offset,
#         log_tau_A,
#         log_tau_phi,
#         log_tau_z,
#         P,
#         I,
#     ]
#     """
#     x = np.asarray(x, dtype=float)
#     return {
#         "fd_params": {
#             "C1_scale": float(np.exp(x[0])),
#             "C2_scale": float(np.exp(x[1])),
#             "a0_scale": float(np.exp(x[2])),
#         },
#         "drive_scale": float(x[3]),
#         "drive_shift": float(x[4]),
#         "setpoint_scale": float(x[5]),
#         "setpoint_shift": float(x[6]),
#         "phase_scale": float(x[7]),
#         "phase_offset": float(x[8]),
#         "tau_A": float(np.exp(x[9])),
#         "tau_phi": float(np.exp(x[10])),
#         "tau_z": float(np.exp(x[11])),
#         "P": float(x[12]),
#         "I": float(x[13]),
#     }


# class RegeneratingDigitalTwinSPM2D(DigitalTwinSPM2D):
#     """
#     Extension of DigitalTwinSPM2D that can rebuild the FD library from
#     trial FD-physics parameters during calibration.

#     The intended use is:
#     1) set one FD-library recipe (drive grid, d range, resolution)
#     2) fit fd_params such as C1_scale, C2_scale, a0_scale
#     3) automatically regenerate an FDCurve2D on each objective evaluation

#     Notes
#     -----
#     - fd_params are applied directly to the *normalized* effective contact
#       parameters used by the solver.
#     - This avoids re-deriving H, E_star, R separately when those are not
#       individually identifiable from the experiment.
#     """

#     def __init__(self, *args, **kwargs):
#         super().__init__(*args, **kwargs)
#         self._fd_recipe: Optional[Dict[str, Any]] = None
#         self._regen_library_cache: Dict[str, FDCurve2D] = {}

#     def set_fd_library_recipe(
#         self,
#         drive_hat_grid: Sequence[float],
#         d_start_hat: float,
#         d_end_hat: float,
#         n_points: int = 256,
#         z0_hat: float = 0.0,
#         v0_hat: float = 0.0,
#         warm_start: bool = True,
#         clip: bool = True,
#         attrs: Optional[Dict[str, Any]] = None,
#     ) -> Dict[str, Any]:
#         recipe = {
#             "drive_hat_grid": np.asarray(drive_hat_grid, dtype=float),
#             "d_start_hat": float(d_start_hat),
#             "d_end_hat": float(d_end_hat),
#             "n_points": int(n_points),
#             "z0_hat": float(z0_hat),
#             "v0_hat": float(v0_hat),
#             "warm_start": bool(warm_start),
#             "clip": bool(clip),
#             "attrs": _copy_dict(attrs),
#         }
#         self._fd_recipe = recipe
#         return recipe

#     def get_fd_library_recipe(self) -> Dict[str, Any]:
#         if self._fd_recipe is None:
#             raise ValueError("FD-library recipe has not been set. Call set_fd_library_recipe(...) first.")
#         return dict(self._fd_recipe)

#     def _scaled_norm_params(self, fd_params: Optional[Dict[str, Any]] = None) -> NormalizedSPMParams:
#         fd_params = {} if fd_params is None else dict(fd_params)
#         base = self.norm_params

#         C1 = float(base.C1) * float(fd_params.get("C1_scale", 1.0))
#         C2 = float(base.C2) * float(fd_params.get("C2_scale", 1.0))
#         a0 = float(base.a0) * float(fd_params.get("a0_scale", 1.0))
#         drive_hat_default = float(base.drive_hat_default) * float(fd_params.get("drive_library_scale", 1.0))

#         attrs = dict(base.attrs)
#         attrs.update({
#             "fd_params": dict(fd_params),
#             "source": "RegeneratingDigitalTwinSPM2D._scaled_norm_params",
#         })
#         return NormalizedSPMParams(
#             Q=float(base.Q),
#             C1=C1,
#             C2=C2,
#             a0=a0,
#             drive_hat_default=drive_hat_default,
#             omega_drive_hat=float(base.omega_drive_hat),
#             omega_ref_hat=float(base.omega_ref_hat),
#             attrs=attrs,
#         )

#     def build_fd_library_from_fd_params(
#         self,
#         fd_params: Optional[Dict[str, Any]] = None,
#         recipe: Optional[Dict[str, Any]] = None,
#         use_cache: bool = True,
#     ) -> FDCurve2D:
#         recipe = self.get_fd_library_recipe() if recipe is None else dict(recipe)
#         fd_params = {} if fd_params is None else dict(fd_params)

#         cache_key = _json_hashable({
#             "fd_params": fd_params,
#             "recipe": {
#                 **recipe,
#                 "drive_hat_grid": np.asarray(recipe["drive_hat_grid"], dtype=float).tolist(),
#             },
#         })
#         if use_cache and cache_key in self._regen_library_cache:
#             return self._regen_library_cache[cache_key]

#         scaled_norm = self._scaled_norm_params(fd_params)
#         approach_model = SPMRK45ApproachModel(scaled_norm, self.solver)
#         builder = FDLibraryBuilder(approach_model, attrs={
#             "workflow": "RegeneratingDigitalTwinSPM2D",
#             "fd_params": fd_params,
#         })
#         lib = builder.build(
#             drive_hat_grid=recipe["drive_hat_grid"],
#             d_start_hat=recipe["d_start_hat"],
#             d_end_hat=recipe["d_end_hat"],
#             n_points=recipe["n_points"],
#             z0_hat=recipe["z0_hat"],
#             v0_hat=recipe["v0_hat"],
#             warm_start=recipe["warm_start"],
#             clip=recipe["clip"],
#             attrs={**_copy_dict(recipe.get("attrs")), "fd_params": fd_params},
#         )
#         if use_cache:
#             self._regen_library_cache[cache_key] = lib
#         return lib

#     def make_fd_library_factory(
#         self,
#         recipe: Optional[Dict[str, Any]] = None,
#         use_cache: bool = True,
#     ) -> Callable[[Dict[str, Any]], FDCurve2D]:
#         recipe_local = self.get_fd_library_recipe() if recipe is None else dict(recipe)

#         def factory(fd_params: Dict[str, Any]) -> FDCurve2D:
#             return self.build_fd_library_from_fd_params(
#                 fd_params=fd_params,
#                 recipe=recipe_local,
#                 use_cache=use_cache,
#             )

#         return factory

#     def calibrator_regenerating_fd(
#         self,
#         data: FDGridData,
#         surface_ref_hat: np.ndarray,
#         recipe: Optional[Dict[str, Any]] = None,
#         line_kwargs_base: Optional[Dict[str, Any]] = None,
#         loss_weights: Optional[Dict[str, float]] = None,
#         remove_tilt_in_height: bool = True,
#         fit_subset: Optional[Sequence[Tuple[int, int]]] = None,
#         use_cache: bool = True,
#     ) -> FDGridCalibrator2D:
#         factory = self.make_fd_library_factory(recipe=recipe, use_cache=use_cache)
#         return self.calibrator(
#             data=data,
#             surface_ref_hat=surface_ref_hat,
#             fd_library=None,
#             fd_library_factory=factory,
#             line_kwargs_base=line_kwargs_base,
#             loss_weights=loss_weights,
#             remove_tilt_in_height=remove_tilt_in_height,
#             fit_subset=fit_subset,
#         )

#     def simulate_line_with_fd_params(
#         self,
#         h_hat,
#         drive_hat,
#         setpoint,
#         fd_params: Optional[Dict[str, Any]] = None,
#         recipe: Optional[Dict[str, Any]] = None,
#         **scan_kwargs,
#     ):
#         lib = self.build_fd_library_from_fd_params(fd_params=fd_params, recipe=recipe, use_cache=True)
#         scanner = self.get_scanner(fd_library=lib)
#         return scanner.scan_line_pi(h_hat=h_hat, drive_hat=drive_hat, setpoint=setpoint, **scan_kwargs)

#     def simulate_map_with_fd_params(
#         self,
#         h_map_hat,
#         drive_hat,
#         setpoint,
#         fd_params: Optional[Dict[str, Any]] = None,
#         recipe: Optional[Dict[str, Any]] = None,
#         **scan_kwargs,
#     ):
#         lib = self.build_fd_library_from_fd_params(fd_params=fd_params, recipe=recipe, use_cache=True)
#         scanner = self.get_scanner(fd_library=lib)
#         return scanner.scan_map_pi(h_map_hat=h_map_hat, drive_hat=drive_hat, setpoint=setpoint, **scan_kwargs)


# def example_bounds_vector_to_theta_2d_physics() -> List[Tuple[float, float]]:
#     """
#     Suggested bounds for default_vector_to_theta_2d_physics.
#     """
#     return [
#         (-1.0, 1.0),    # log_C1_scale
#         (-1.0, 1.0),    # log_C2_scale
#         (-0.7, 0.7),    # log_a0_scale
#         (0.7, 1.3),     # drive_scale
#         (-20.0, 20.0),  # drive_shift
#         (0.8, 1.2),     # setpoint_scale
#         (-0.2, 0.2),    # setpoint_shift
#         (0.7, 1.3),     # phase_scale
#         (-40.0, 40.0),  # phase_offset
#         (-8.0, 0.0),    # log_tau_A
#         (-8.0, 0.0),    # log_tau_phi
#         (-8.0, 0.0),    # log_tau_z
#         (0.0, 1.0),     # P
#         (0.0, 0.2),     # I
#     ]

# Example uses

## Example approaching curves

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# # ------------------------------------------------------------------
# # User input
# # ------------------------------------------------------------------
# drive = 117.9e-9

# params = {
#     'k': 1,
#     'A': drive,
#     'd0': drive*1.25,
#     'a0': 0.164e-9,
#     'R': 15e-9,
#     'f0': 300e3,
#     'Q': 175,
#     'E_star': 1 / ((1 - 0.3**2) / (130e9) + (1 - 0.3**2) / (1.2e9)),
#     'H': 7.1e-20,
# }

# conversions = {
#     'k': 1,
#     'A': 1e9,
#     'd0': 1e9,
#     'a0': 1e9,
#     'R': 1e9,
#     'f0': 1 / (2 * np.pi * params['f0']),
#     'Q': 1,
#     'E_star': 1e-9 / params['k'],
#     'H': 1 / (params['k'] * (1e-9)**2),
# }

# # ------------------------------------------------------------------
# # Convert to normalized parameters
# # ------------------------------------------------------------------
# param_norm = {}
# for key in params:
#     param_norm[key] = params[key] * conversions[key]

# C1 = param_norm['H'] * param_norm['R'] / 6.0
# C2 = (4.0 / 3.0) * param_norm['E_star'] * np.sqrt(param_norm['R'])
# C3 = param_norm['A'] / param_norm['Q']

# omega_drive_hat = 1.0
# omega_ref_hat = 1.0

# spm_params = NormalizedSPMParams(
#     Q=param_norm['Q'],
#     C1=C1,
#     C2=C2,
#     C3=C3,
#     a0=param_norm['a0'],
#     omega_drive_hat=omega_drive_hat,
#     omega_ref_hat=omega_ref_hat,
# )

# solver = RK45Options(
#     N_cycles=20,
#     max_windows=40,
#     settle_windows=1,
#     tol_A=1e-4,
#     tol_phi_deg=1e-2,
#     consecutive=2,
#     demod_per_ref_cycle=30,
#     rtol=1e-6,
#     atol=1e-9,
#     dt0=0.02,
#     dt_min=1e-10,
#     dt_max=0.3,
#     max_steps_per_window=200000,
# )

# model = SPMRK45ApproachModel(
#     params=spm_params,
#     solver=solver,
# )

# # ------------------------------------------------------------------
# # Generate approach curve
# # ------------------------------------------------------------------
# d_start_hat = param_norm['d0']
# d_end_hat = -5.0
# n_points = 256

# curve = model.approach_curve(
#     d_start_hat=d_start_hat,
#     d_end_hat=d_end_hat,
#     n_points=n_points,
#     z0_hat=0.0,
#     v0_hat=0.0,
#     warm_start=True,
# )

# # ------------------------------------------------------------------
# # Extract results
# # ------------------------------------------------------------------
# height_nm = curve["d_hat"]
# A_nm = curve["A_hat"]
# phi_deg = curve["phi_deg"]
# F_mean = curve["F_mean_hat"]
# F_min = curve["F_min_hat"]
# F_max = curve["F_max_hat"]

# # ------------------------------------------------------------------
# # Plot
# # ------------------------------------------------------------------
# fig, axes = plt.subplots(3, 1, figsize=(6, 9), sharex=True)

# axes[0].plot(height_nm, A_nm, lw=2)
# axes[0].set_ylabel("Amplitude (nm)")
# axes[0].set_title("Approach curve")

# axes[1].plot(height_nm, phi_deg, lw=2)
# axes[1].set_ylabel("Phase (deg)")

# axes[2].plot(height_nm, F_mean, lw=2, label="Mean force")
# axes[2].fill_between(height_nm, F_min, F_max, alpha=0.25, label="Min-Max force")
# axes[2].set_ylabel("Force ($\\hat{F}$)")
# axes[2].set_xlabel("Height (nm)")
# axes[2].legend()

# for ax in axes:
#     ax.invert_xaxis()
#     ax.grid(True, alpha=0.3)

# plt.tight_layout()
# # plt.show()
# # plt.savefig('output/example approach curve.png', dpi=400, bbox_inches='tight', pad_inches=0.1)

In [ ]:
# @timeit("Approaching simulations take:")
# def simulate_FD(drives):
#     out = []
#     for i in range(len(drives)):
#         drive = drives[i]   # example: 100 nm free amplitude in SI units
#                         # replace this with your desired drive amplitude
#         params = {
#             'k': 1,
#             'A': drive,
#             'd0': drive * 1.25,
#             'a0': 0.164e-9,
#             'R': 20e-9,
#             'f0': 75e3,
#             'Q': 200,
#             'E_star': 1 / ((1 - 0.3**2) / (130e9) + (1 - 0.3**2) / (1.2e9)),
#             'H': 7.1e-20,
#         }
        
#         conversions = {
#             'k': 1,
#             'A': 1e9,
#             'd0': 1e9,
#             'a0': 1e9,
#             'R': 1e9,
#             'f0': 1 / (2 * np.pi * params['f0']),   # t_hat = omega0 * t
#             'Q': 1,
#             'E_star': 1e-9 / params['k'],
#             'H': 1 / (params['k'] * (1e-9)**2),
#         }
        
#         # ------------------------------------------------------------------
#         # Convert to normalized parameters
#         # ------------------------------------------------------------------
#         param_norm = {}
#         for key in params:
#             param_norm[key] = params[key] * conversions[key]
        
#         C1 = param_norm['H'] * param_norm['R'] / 6.0
#         C2 = (4.0 / 3.0) * param_norm['E_star'] * np.sqrt(param_norm['R'])
#         C3 = param_norm['A'] / param_norm['Q']
        
#         # If driving at resonance:
#         omega_drive_hat = 1.0
#         omega_ref_hat = 1.0
        
#         # If you want an off-resonance drive at f_drive:
#         # f_drive = 70e3
#         # omega_drive_hat = f_drive / params['f0']
#         # omega_ref_hat   = omega_drive_hat   # or keep 1.0 depending on your demod choice
        
#         spm_params = NormalizedSPMParams(
#             Q=param_norm['Q'],
#             C1=C1,
#             C2=C2,
#             C3=C3,
#             a0=param_norm['a0'],
#             omega_drive_hat=omega_drive_hat,
#             omega_ref_hat=omega_ref_hat,
#         )
        
#         solver = RK45Options(
#             N_cycles=20,
#             max_windows=40,
#             settle_windows=1,
#             tol_A=1e-4,
#             tol_phi_deg=1e-2,
#             consecutive=2,
#             demod_per_ref_cycle=30,
#             rtol=1e-6,
#             atol=1e-9,
#             dt0=0.02,
#             dt_min=1e-10,
#             dt_max=0.3,
#             max_steps_per_window=200000,
#         )
        
#         model = SPMRK45ApproachModel(
#             params=spm_params,
#             solver=solver,
#         )
        
#         # ------------------------------------------------------------------
#         # Generate approach curve
#         # ------------------------------------------------------------------
#         d_start_hat = param_norm['d0']
#         d_end_hat = -5.0
#         n_points = 256
        
#         curve = model.approach_curve(
#             d_start_hat=d_start_hat,
#             d_end_hat=d_end_hat,
#             n_points=n_points,
#             z0_hat=0.0,
#             v0_hat=0.0,
#             warm_start=True,
#         )
#         out.append(curve)
#     return out

In [ ]:
# # ------------------------------------------------------------------
# # User input
# # ------------------------------------------------------------------
# drive = 150e-9   # example: 100 nm free amplitude in SI units
#                 # replace this with your desired drive amplitude

# params = {
#     'k': 1,
#     'A': drive,
#     'd0': drive * 1.25,
#     'a0': 0.164e-9,
#     'R': 20e-9,
#     'f0': 75e3,
#     'Q': 200,
#     'E_star': 1 / ((1 - 0.3**2) / (130e9) + (1 - 0.3**2) / (1.2e9)),
#     'H': 7.1e-20,
# }

# conversions = {
#     'k': 1,
#     'A': 1e9,
#     'd0': 1e9,
#     'a0': 1e9,
#     'R': 1e9,
#     'f0': 1 / (2 * np.pi * params['f0']),   # t_hat = omega0 * t
#     'Q': 1,
#     'E_star': 1e-9 / params['k'],
#     'H': 1 / (params['k'] * (1e-9)**2),
# }

# # ------------------------------------------------------------------
# # Convert to normalized parameters
# # ------------------------------------------------------------------
# param_norm = {}
# for key in params:
#     param_norm[key] = params[key] * conversions[key]

# C1 = param_norm['H'] * param_norm['R'] / 6.0
# C2 = (4.0 / 3.0) * param_norm['E_star'] * np.sqrt(param_norm['R'])
# C3 = param_norm['A'] / param_norm['Q']

# # If driving at resonance:
# omega_drive_hat = 1.0
# omega_ref_hat = 1.0

# # If you want an off-resonance drive at f_drive:
# # f_drive = 70e3
# # omega_drive_hat = f_drive / params['f0']
# # omega_ref_hat   = omega_drive_hat   # or keep 1.0 depending on your demod choice

# spm_params = NormalizedSPMParams(
#     Q=param_norm['Q'],
#     C1=C1,
#     C2=C2,
#     C3=C3,
#     a0=param_norm['a0'],
#     omega_drive_hat=omega_drive_hat,
#     omega_ref_hat=omega_ref_hat,
# )

# solver = RK45Options(
#     N_cycles=20,
#     max_windows=40,
#     settle_windows=1,
#     tol_A=1e-4,
#     tol_phi_deg=1e-2,
#     consecutive=2,
#     demod_per_ref_cycle=30,
#     rtol=1e-6,
#     atol=1e-9,
#     dt0=0.02,
#     dt_min=1e-10,
#     dt_max=0.3,
#     max_steps_per_window=200000,
# )

# noise = SmoothNoiseOptions(
#     noise_rel_amp=-5e-1,                 # small, slowly varying force noise
#     freq_factors=(0.005, 0.011, 0.017),
#     weights=(1.0, 0.5, 0.25),
#     seed=1,
# )

# model = SPMRK45ApproachModel(
#     params=spm_params,
#     solver=solver,
#     noise=noise,
# )

# # ------------------------------------------------------------------
# # Generate approach curve
# # ------------------------------------------------------------------
# # In your normalization, distance-like variables are effectively in nm
# # because you multiply lengths by 1e9.
# #
# # Start far from surface and approach into interaction/contact.
# # You can tune d_end_hat depending on how far into contact you want to go.
# d_start_hat = param_norm['d0']   # e.g. 150 nm
# d_end_hat = -25.0                 # go slightly into contact
# n_points = 256

# curve = model.approach_curve(
#     d_start_hat=d_start_hat,
#     d_end_hat=d_end_hat,
#     n_points=n_points,
#     z0_hat=0.0,
#     v0_hat=0.0,
#     warm_start=True,
# )

# # ------------------------------------------------------------------
# # Extract results
# # ------------------------------------------------------------------
# height_nm = curve["d_hat"]          # numerically in nm with your conversion
# A_nm = curve["A_hat"]               # normalized amplitude, also nm here
# phi_deg = curve["phi_deg"]
# F_mean = curve["F_mean_hat"]
# F_min = curve["F_min_hat"]
# F_max = curve["F_max_hat"]

# # ------------------------------------------------------------------
# # Plot
# # ------------------------------------------------------------------
# fig, axes = plt.subplots(3, 1, figsize=(6, 9), sharex=True)

# # Amplitude
# axes[0].plot(height_nm, A_nm, lw=2)
# axes[0].set_ylabel("Amplitude (nm)")
# axes[0].set_title("Approach curve")

# # Phase
# axes[1].plot(height_nm, phi_deg, lw=2)
# axes[1].set_ylabel("Phase (deg)")

# # Force
# axes[2].plot(height_nm, F_mean, lw=2, label="Mean force")
# axes[2].fill_between(height_nm, F_min, F_max, alpha=0.25, label="Min-Max force")
# axes[2].set_ylabel("Force (normalized)")
# axes[2].set_xlabel("Height (nm)")
# axes[2].legend()

# # Optional: show approach from large height -> small height visually
# for ax in axes:
#     ax.invert_xaxis()
#     ax.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

In [ ]:
# drives = np.array([200, 175, 150, 125, 100, 75, 50]) * 1e-9

# curves = simulate_FD(drives)

In [ ]:
# # ------------------------------------------------------------------
# # Plot
# # ------------------------------------------------------------------
# fig, axes = plt.subplots(3, 1, figsize=(6, 9), sharex=True)

# # ------------------------------------------------------------------
# # Extract results
# # ------------------------------------------------------------------

# for i in range(len(drives)):
#     curve = curves[i]
#     height_nm = curve["d_hat"] - drives[i] * 1e9          # numerically in nm with your conversion
#     A_nm = curve["A_hat"]               # normalized amplitude, also nm here
#     phi_deg = curve["phi_deg"]
#     F_mean = curve["F_mean_hat"]
#     F_min = curve["F_min_hat"]
#     F_max = curve["F_max_hat"]
    
#     # Amplitude
#     axes[0].plot(height_nm, A_nm, lw=2)
#     axes[0].set_ylabel("Amplitude (nm)")
#     axes[0].set_title("Approach curve")
    
#     # Phase
#     axes[1].plot(height_nm, phi_deg, lw=2, label="Drive={} nm".format(int(drives[i]*1e9)))
#     axes[1].set_ylabel("Phase (deg)")
#     axes[1].legend()
    
#     # Force
#     axes[2].plot(height_nm, F_mean, lw=2, label="Mean force")
#     # axes[2].fill_between(height_nm, F_min, F_max, alpha=0.25, label="Min-Max force")
#     axes[2].set_ylabel("Force (normalized)")
#     axes[2].set_xlabel("Height (nm)")
#     # axes[2].legend()

# # Optional: show approach from large height -> small height visually
# for ax in axes:
#     ax.invert_xaxis()
#     ax.grid(True, alpha=0.3)

# plt.tight_layout()
# # plt.show()
# # plt.savefig('output/example approach curve different drives.png', dpi=400, bbox_inches='tight', pad_inches=0.1)


## Example scan lines

In [ ]:
# Define the groud-truth line profile
indices = np.array([10, 89, 137, 216, 264, 343, 392, 470])
h1 = np.zeros(512)
for i in range(4):
    h1[indices[2*i]:indices[2*i+1]] = 1
    # h1[(i*50):(i*50)+25] = 1
h_hat = h1 * 100 + 52 / 512 * (512-np.arange(512))

plt.figure(figsize=[5,4])
plt.plot(h_hat[::4], label='Simulated')
# plt.plot((tp[ind1,ind2,0]-np.min(tp[ind1,ind2,0]))*1e9, label='Expt')
plt.xlabel('Pixels')
plt.ylabel('Height (nm)')
plt.legend()

In [ ]:
# fd_model = FDCurve1D(
#     d_hat=curve["d_hat"],
#     A_hat=curve["A_hat"],
#     phi_deg=curve["phi_deg"],
#     F_max_hat=curve["F_max_hat"],
# )

# pi_noise = PIDriftNoiseOptions(
#     drift_rel_std=2e-3,
#     drift_tau_pixels=200.0,
#     white_rel_std=0.0,
#     sinusoid_rel_amp=0.0,
#     seed=1,
# )

# scanner = ScannerFD(
#     params=params,
#     conversions=conversions,
#     fd_model=fd_model,
#     pi_noise=pi_noise,
# )

In [ ]:
# out = scanner.scan_line_pi(
#     h_hat=h_hat[::4],
#     setpoint=0.3,
#     scan_speed_hat=1.0,
#     dx_hat=1.0,
#     P=1e0,
#     I=1e-1,
#     A_filter_order=1,
#     tau_A=1e-2,
#     z_filter_order=1,
#     tau_z=1e-2,
# )
# plt.plot(out['d_hat'], label='simulation')
# plt.plot(h_hat[::4], label='ground truth')
# # plt.plot(out['phi_deg'])
# plt.legend()

In [ ]:
# out_pair = scanner.scan_line_trace_retrace_pi(
#     h_hat=h_hat,
#     setpoint=0.85,
#     scan_speed_hat=1.0,
#     dx_hat=1.0,
#     P=1,
#     I=1e-2,
#     A_filter_order=1,
#     tau_A=1e-2,
#     phi_filter_order=1,
#     tau_phi=1e-2,
#     z_filter_order=1,
#     tau_z=1e-2,
# )

# plot_trace_retrace_channels(
#     out_pair,
#     h_hat=h_hat,
#     center_height=True,
# )

In [ ]:
# out_pair = scanner.scan_line_trace_retrace_pi(
#     h_hat=h_hat[::4],
#     setpoint=0.8,
#     scan_speed_hat=1.0,
#     dx_hat=1.0,
#     P=1,
#     I=1e-2,
#     A_filter_order=1,
#     tau_A=1e-2,
#     phi_filter_order=1,
#     tau_phi=1e-2,
#     z_filter_order=1,
#     tau_z=1e-2,
# )

# plot_trace_retrace_channels(
#     out_pair,
#     h_hat=h_hat[::4],
#     center_height=True,
# )

## Build scanner at different drives

In [ ]:
# import numpy as np

# def build_scanner_for_drive(
#     drive_si,
#     params,
#     conversions,
#     solver,
#     pi_noise,
#     d_end_hat=-25.0,
#     n_points=256,
# ):
#     # copy params and set drive
#     params_local = dict(params)
#     params_local["A"] = drive_si

#     # normalize
#     param_norm = {k: params_local[k] * conversions[k] for k in params_local}

#     C1 = param_norm["H"] * param_norm["R"] / 6.0
#     C2 = (4.0 / 3.0) * param_norm["E_star"] * np.sqrt(param_norm["R"])
#     C3 = param_norm["A"] / param_norm["Q"]

#     spm_params = NormalizedSPMParams(
#         Q=param_norm["Q"],
#         C1=C1,
#         C2=C2,
#         C3=C3,
#         a0=param_norm["a0"],
#         omega_drive_hat=1.0,
#         omega_ref_hat=1.0,
#     )

#     model = SPMRK45ApproachModel(
#         params=spm_params,
#         solver=solver,
#     )

#     curve = model.approach_curve(
#         d_start_hat=param_norm["d0"],
#         d_end_hat=d_end_hat,
#         n_points=n_points,
#         z0_hat=0.0,
#         v0_hat=0.0,
#         warm_start=True,
#     )

#     fd_model = FDCurve1D(
#         d_hat=curve["d_hat"],
#         A_hat=curve["A_hat"],
#         phi_deg=curve["phi_deg"],
#         F_max_hat=curve["F_max_hat"],
#     )

#     scanner = ScannerFD(
#         params=params_local,
#         conversions=conversions,
#         fd_model=fd_model,
#         pi_noise=pi_noise,
#     )

#     return scanner, curve

In [ ]:
# drive = 80e-9   # 80 nm

# h_line = h_hat[::4]
# scanner, curve = build_scanner_for_drive(
#     drive_si=drive,
#     params=params,
#     conversions=conversions,
#     solver=solver,
#     pi_noise=PIDriftNoiseOptions(
#         drift_rel_std=2e-3,
#         drift_tau_pixels=200.0,
#         white_rel_std=0.0,
#         sinusoid_rel_amp=0.0,
#         seed=1,
#     ),
# )

# out_pair = scanner.scan_line_trace_retrace_pi(
#     h_hat=h_line,
#     setpoint=1.0,
#     scan_speed_hat=1.0,
#     dx_hat=1.0,
#     P=1,
#     I=0.01,
#     A_filter_order=1,
#     tau_A=1e-2,
#     phi_filter_order=1,
#     tau_phi=1e-2,
#     z_filter_order=1,
#     tau_z=1e-2,
# )

# plot_trace_retrace_channels(
#     out_pair,
#     h_hat=h_line,
#     center_height=True,
# )

# Calibrate with grid-data

## Use AC MOBO results for the calibration

In [ ]:
folder = os.path.join('data', '250315', 'pickles')

fname = '250315_Cali1_MOBO training.pickle'

save = '250315_Cali1_MOBO'

import pickle

with open(os.path.join(folder, fname), 'rb') as fopen:
    param = pickle.load(fopen)

train_obj = param['y_measured_tensor']

# visualize the topograph
import aespm as ae

map_name = 'Cali_res1_0001.ibw'

folder = os.path.join('data', '250315', 'CaliSample01')

save = '250315_Cali_MOBO_01'


# import pickle

topo = ae.tools.load_ibw(os.path.join(folder, map_name))

factor = param['factor'] * topo.header['InvOLS'] * 1e9

In [ ]:
folder = os.path.join('data', '250315', 'pickles')


with open(os.path.join(folder, '250315_Cali1_MOBO.pickle'), 'rb') as fopen:
    param_clean = pickle.load(fopen)

param_clean.keys()

In [ ]:
# Load FD curves from experiment

folder = os.path.join('data', '250315', 'CaliSample01')

data = []
files = []

for file in os.listdir(folder):
    if file.startswith('Cali_FD'):
        files.append(os.path.join(folder, file))

files = sorted(files)

for file in files:
    data.append(ae.tools.load_ibw(file))

In [ ]:
fwds = []
bkds = []
drives = []
thresholds = []

x = []

for i in range(len(data)):
    temp = data[i]
    drive = temp.header['DriveAmplitude']
    drives.append(drive)
    fwd, bkd = read_fd(temp, factor=1e-9)
    ending = np.min(fwd[1])
    setpoint = ending * 4
    fwds.append(fwd)
    bkds.append(bkd)
    amp_thres = find_thres(fwd[2], fwd[1], thres=85)
    thresholds.append(amp_thres / setpoint * 1e2)
    x.append(fwd[1] / setpoint * 1e2)

In [ ]:
factor1 =  param['factor'] * topo.header['AmpInvOLS'] *1e9
print(topo.header['AmpInvOLS'])

In [ ]:
# index = 1, 5, 8, 13

# colors = ['r', 'g', 'b', 'orange']
fig, ax = plt.subplots(1, 2, figsize=[9., 3])
for i in range(len(fwds)):
    ax[0].plot(x[i], fwds[i][2], label='Drive={:.0f} nm'.format(drives[i]*factor1))
    ax[1].plot(x[i], fwds[i][1], label='Drive={:.0f} nm'.format(drives[i]*factor1))
    
    # plt.axvline(thresholds[ind], linestyle='--')
ax[1].legend(bbox_to_anchor=[1.1, 1])
ax[0].set_xlim(20, 100)
ax[0].set_ylim(20, 180)
ax[0].axhline(85, linestyle='--', color='gray')
ax[0].set_ylabel('Phase (degrees)')
ax[0].set_xlabel('Setpoint (%)')
ax[1].set_xlim(20, 100)
ax[1].set_ylabel('Amplitude (nm)')
ax[1].set_xlabel('Setpoint (%)')
plt.tight_layout()
# plt.savefig('output/{} threhold FD curve.png'.format(save), dpi=400, bbox_inches='tight', pad_inches=0.1)


In [ ]:
tp = fwds[0]

L = 50

height = np.zeros([len(fwds), L])
amp = np.zeros([len(fwds), L])
phase = np.zeros([len(fwds), L])

for i in range(len(fwds)):
    tp = fwds[i]
    height[i] = (tp[0] - np.min(tp[0][-L:]))[-L:]*1e9
    amp[i] = tp[1][-L:][::-1]
    phase[i] = tp[2][-L:][::-1]


fig, ax = plt.subplots(1,2,figsize=[10,3])
for i in range(len(fwds)):
    ax[0].plot(height[i], amp[i])
ax[0].set_xlabel('Height (nm)')
ax[0].set_ylabel('Amplitude (nm)')

for i in range(len(fwds)):
    ax[1].plot(height[i], phase[i])
ax[1].set_xlabel('Height (nm)')
ax[1].set_ylabel('Phase (degrees)')

plt.tight_layout()

# plt.figure()
# for i in range(len(fwds)):
#     plt.plot(height[i], amp[i])
# plt.xlabel('Height (nm)')
# plt.ylabel('Amplitude (nm)')

# plt.figure()
# for i in range(len(fwds)):
#     plt.plot(height[i], phase[i])
# plt.xlabel('Height (nm)')
# plt.ylabel('Phase (degrees)')

In [ ]:
np.savez('output/cali_fd.npz', height=height, amp=amp, phase=phase, drive=(np.array(drives) * factor1))

In [ ]:
for i in range(len(fwds)):
    plt.plot(amp[i] / drives[i]/factor1, phase[i], '.-', label=drives[i]*factor1)
plt.legend()

In [ ]:
params = {
    'k': 25,
    'A': drive,
    'd0': drive * 1.25,
    'a0': 0.164e-9,
    'R': 10e-9,
    'f0': 300e3,
    'Q': 350,
    'E_star': 1 / ((1 - 0.3**2) / (130e9) + (1 - 0.3**2) / (1.2e9)),
    'H': 7.1e-20,
}

conversions = {
    'k': 1,
    'A': 1e9,
    'd0': 1e9,
    'a0': 1e9,
    'R': 1e9,
    'f0': 1 / (2 * np.pi * params['f0']),   # t_hat = omega0 * t
    'Q': 1,
    'E_star': 1e-9 / params['k'],
    'H': 1 / (params['k'] * (1e-9)**2),
}

# ------------------------------------------------------------------
# Convert to normalized parameters
# ------------------------------------------------------------------
param_norm = {}
for key in params:
    param_norm[key] = params[key] * conversions[key]

C1 = param_norm['H'] * param_norm['R'] / 6.0
C2 = (4.0 / 3.0) * param_norm['E_star'] * np.sqrt(param_norm['R'])
C3 = param_norm['A'] / param_norm['Q']

# If driving at resonance:
omega_drive_hat = 1.0
omega_ref_hat = 1.0

# If you want an off-resonance drive at f_drive:
# f_drive = 70e3
# omega_drive_hat = f_drive / params['f0']
# omega_ref_hat   = omega_drive_hat   # or keep 1.0 depending on your demod choice

spm_params = NormalizedSPMParams(
    Q=param_norm['Q'],
    C1=C1,
    C2=C2,
    C3=C3,
    a0=param_norm['a0'],
    omega_drive_hat=omega_drive_hat,
    omega_ref_hat=omega_ref_hat,
)

In [ ]:
drives_si = np.array(drives)*factor1
measured_fd = {}


# Assume these are lists/arrays where index i corresponds to drives[i]
# z_arrays[i], A_arrays[i], phi_arrays[i]
# each could be a 1D FD curve, a scan line, or an image
measured_fd = {
    float(drive): {
        "d": height[i],
        "A": amp[i],
        "phi": phase[i],
    }
    for i, drive in enumerate(drives_si)
}

measured_fd

In [ ]:
traces = param_clean['traces']
# Define the groud-truth line profile
indices = np.array([0, 20, 44, 83, 108, 147, 172, 211, 235, 256]) * 2
h1 = np.zeros(512)
for i in range(5):
    h1[indices[2*i]:indices[2*i+1]] = 1
    # h1[(i*50):(i*50)+25] = 1
h_hat = h1 * 100 + 142 / 512 * (512-np.arange(512))

# h_truth_hat = h_hat[::2]
h_truth_hat = h_hat


fig, ax = plt.subplots(2,1,figsize=[4, 6])
tp  = traces[30, 4, 0]
ax[0].plot(h_truth_hat)
ax[1].plot((tp-np.min(tp))*1e9)

In [ ]:
h_truth = h_truth_hat-np.min(h_truth_hat)
plt.plot(h_truth[::2])
plt.plot((traces[5,4,0]-traces[5,4,0].min()) * 1e9)

### Construct scanner using FD curves

In [ ]:
import numpy as np

idx = -4

drive_nm = drives_si[idx]
d_nm     = height[idx]    # ascending, tip–sample distance
A_nm     = amp[idx]    # rising toward drive_nm at large d
phi_deg  = phase[idx]

fd = FDCurve1D(d_nm, A_nm, phi_deg, A0=drive_nm)

# Eyeball check before plugging into the loop
d_grid = np.linspace(d_nm.min(), d_nm.max(), 200)
A_grid = np.array([fd.evaluate(d)[0] for d in d_grid])
plt.plot(d_grid, A_grid); plt.axhline(drive_nm, ls='--')

# Build a scanner that always uses this one curve
params = {'k':40, 'A': drive_nm, 'd0': d_nm.max()*0.8, 'R': 10, 'H': 1e-19,
          'E_star': 1e9, 'Q': 150}
scanner = ScannerFD(params=params,
                    conversions={k: 1.0 for k in params},
                    fd_model=fd)

# Trivial fd_lookup: ignore the drive argument, always return `fd`
fd_lookup = lambda _drive: fd

phi_grid = np.array([fd.evaluate(d)[1] for d in d_grid])

plt.plot(d_grid, phi_grid); plt.axhline(drive_nm, ls='--')
# plt.plot(d_grid, A_grid); plt.axhline(drive_nm, ls='--')


In [ ]:
# ------------------------------------------------------------
# 2. Build continuous FD surface from measured FD curves
# ------------------------------------------------------------
fd_surface = FDDriveSurface.from_measured_fd(
    measured_fd,
    x_mode="d_over_A0",          # recommended for your plotted curves
    common_range="intersection", # safest; use "union" if range overlap is too small
    n_x=800,
    far_field_frac=0.1,
    far_field_side="largest_d",  # because large height is far field in your plot

    # Use True if measured_fd keys are actual free amplitudes in nm.
    # For your plot, this is probably True.
    drive_key_is_A0=True,
)

# ------------------------------------------------------------
# 3. Convert physical FD curves into scanner-normalized units
# ------------------------------------------------------------
fd_lookup = make_normalized_fd_lookup(
    fd_surface,
    conv_L=1,   # or scanner.conversion["d0"]
    conv_A=1,
    conv_F=conversions.get("F", 1.0),
    n_points=800,
)

# ------------------------------------------------------------
# 4. Attach continuous-drive FD model to scanner
# ------------------------------------------------------------
scanner = attach_continuous_fd_to_scanner(
    scanner,
    fd_lookup,
    conv_A=scanner.conversion["A"],
)

In [ ]:
# # scanner.scan_line_substepped()


# h_hat,
# drive_nm,
# setpoint,
# *,
# scan_speed_hat,
# dx_hat,
# P=0.0,
# I=1.0,
# T_I=None,
# A_filter_order=4,
# tau_A=3.2e-5,
# phi_filter_order=4,
# tau_phi=3.2e-5,
# z_filter_order=1,
# tau_z=3.2e-5,
# d_init_hat=30.0,
# phi_meas_init_deg=113.0,
# A0=22.88,                  # free-air amplitude (nm)
# integ_clip=None,
# z_rate_limit_hat=None,
# # --- speed control ---
# n_substeps=None,           # explicit number of inner steps per pixel
# dt_inner_max=None,         # OR cap inner timestep (auto-computes n_substeps)
# safety=10.0,               # auto-compute uses dt_inner = tau_min / safety

In [ ]:
out = scanner.scan_line_trace_retrace_pi_drive(
    h_hat=h_truth,
    drive_nm=100,        # continuous drive, not limited to measured curves
    setpoint=0.6,        # continuous setpoint fraction of current A0

    scan_speed_hat=1e3,
    dx_hat=1.0,
    P=0,
    I=50,

    A_filter_order=4,
    tau_A=3e-5,
    phi_filter_order=4,
    tau_phi=3e-5,
    z_filter_order=1,
    tau_z=1e-5,

    d_init_hat=350,
    T_I=3e-3,
)

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out['trace']['d_hat'])
ax[0].plot(out['retrace']['d_hat'])
ax[0].plot(h_truth, 'g--')
ax[1].plot(out['trace']['phi_deg'])
ax[1].plot(out['retrace']['phi_deg'])
ax[2].plot(out['trace']['A_hat'])
ax[2].plot(out['retrace']['A_hat'])
ax[0].set_ylabel('Height (nm)')
ax[1].set_ylabel('Phase (deg)')
ax[2].set_ylabel('Amplitude (nm)')

plt.tight_layout()


In [ ]:
scanner.scan_line_pi()

In [ ]:
drive = 20
setpoint = 0.6

out = scanner.scan_line_trace_retrace_pi_drive(
    h_hat=h_truth[::2],
    drive_nm=drive,        # continuous drive, not limited to measured curves
    setpoint=setpoint,        # continuous setpoint fraction of current A0

    scan_speed_hat=1e3,
    dx_hat=1.0,
    P=3.2,
    I=0,

    A_filter_order=1,
    tau_A=8e-4,
    phi_filter_order=1,
    tau_phi=5e-4,
    z_filter_order=1,
    tau_z=1e-5,
    z_rate_limit_hat = 1e6,
    
    d_init_hat=250,
    phi_meas_init_deg=120,
    T_I=3e-3,
)

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out['trace']['d_hat']-drive*setpoint*0)
ax[0].plot(out['retrace']['d_hat']-drive*setpoint*0)
ax[0].plot(h_truth[::2], 'g--')
ax[1].plot(out['trace']['phi_deg'])
ax[1].plot(out['retrace']['phi_deg'])
ax[2].plot(out['trace']['A_hat'])
ax[2].plot(out['retrace']['A_hat'])
ax[0].set_ylabel('Height (nm)')
ax[1].set_ylabel('Phase (deg)')
ax[2].set_ylabel('Amplitude (nm)')

plt.tight_layout()


### Pure explorative calibration

In [ ]:
import numpy as np
from scipy.optimize import differential_evolution, minimize
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Robust loss utilities
# ------------------------------------------------------------
def mad_std(x, eps=1e-12):
    x = np.asarray(x)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    return 1.4826 * mad + eps


def huber_loss(r, delta=2.0):
    r = np.asarray(r)
    a = np.abs(r)
    return np.where(a <= delta, 0.5 * r**2, delta * (a - 0.5 * delta))


def align_by_constant_offset(y, ref):
    """
    Remove best constant offset between simulated d_hat and h_truth.
    This lets the fitting focus on shape/alignment rather than absolute z offset.
    """
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)
    offset = np.nanmedian(y - ref)
    return y - offset, offset


def score_scan_no_reverse(
    out,
    h_truth_line,
    drive,
    setpoint,
    trim=20,
    w_height=1.0,
    w_trrt=0.75,
    w_ringing=0.15,
    w_amp=0.05,
    w_phase=0.02,
    huber_delta=2.0,
):
    h = np.asarray(h_truth_line, dtype=float)

    trace_h_raw = np.asarray(out["trace"]["d_hat"], dtype=float)
    retrace_h_raw = np.asarray(out["retrace"]["d_hat"], dtype=float)

    trace_h, trace_offset = align_by_constant_offset(trace_h_raw, h)
    retrace_h, retrace_offset = align_by_constant_offset(retrace_h_raw, h)

    # sl = slice(trim, len(h) - trim if trim > 0 else len(h))

    # h_use = h[sl]
    # tr_use = trace_h[sl]
    # rt_use = retrace_h[sl]

    h_use = h
    tr_use = trace_h
    rt_use = retrace_h
    
    # h_scale = max(mad_std(h_use), 0.05 * np.ptp(h_use), 1.0)
    h_scale = 1
    
    # Height tracking
    r_trace = (tr_use - h_use) / h_scale
    r_retrace = (rt_use - h_use) / h_scale
    
    loss_height = np.nanmean(
        huber_loss(np.r_[r_trace, r_retrace], delta=huber_delta)
    )

    # Trace/retrace mismatch
    r_trrt = (tr_use - rt_use) / h_scale
    loss_trrt = np.nanmean(huber_loss(r_trrt, delta=huber_delta))

    # Ringing penalty from curvature of residual
    res_trace = tr_use - h_use
    res_retrace = rt_use - h_use

    if len(res_trace) > 4:
        ring_trace = np.diff(res_trace, n=2)
        ring_retrace = np.diff(res_retrace, n=2)
        ring_scale = max(mad_std(np.diff(h_use, n=2)), 0.1)
        loss_ringing = np.nanmean(
            huber_loss(
                np.r_[ring_trace, ring_retrace] / ring_scale,
                delta=huber_delta,
            )
        )
    else:
        loss_ringing = 0.0

    # Amplitude regulation
    A_target = drive * setpoint
    A_scale = max(0.05 * drive, 0.5)

    A_trace = np.asarray(out["trace"]["A_hat"], dtype=float)
    A_retrace = np.asarray(out["retrace"]["A_hat"], dtype=float)

    loss_amp = np.nanmean(
        huber_loss(
            np.r_[A_trace - A_target, A_retrace - A_target] / A_scale,
            delta=huber_delta,
        )
    )

    # Phase trace/retrace mismatch
    phi_trace = np.asarray(out["trace"]["phi_deg"], dtype=float)
    phi_retrace = np.asarray(out["retrace"]["phi_deg"], dtype=float)

    loss_phase = np.nanmean(
        huber_loss((phi_trace - phi_retrace) / 30.0, delta=huber_delta)
    )

    total = (
        w_height * loss_height
        + w_trrt * loss_trrt
        + w_ringing * loss_ringing
        + w_amp * loss_amp
        + w_phase * loss_phase
    )

    details = {
        "total": total,
        "loss_height": loss_height,
        "loss_trrt": loss_trrt,
        "loss_ringing": loss_ringing,
        "loss_amp": loss_amp,
        "loss_phase": loss_phase,
        "trace_offset": trace_offset,
        "retrace_offset": retrace_offset,
        "h_scale": h_scale,
    }

    return total, details


def unpack_controller_params_4(x):
    log10_P, log10_I, log10_tau_A, log10_tau_phi = x

    return {
        "P": 10.0 ** log10_P,
        "I": 10.0 ** log10_I,
        "tau_A": 10.0 ** log10_tau_A,
        "tau_phi": 10.0 ** log10_tau_phi,
    }


def make_scan_objective_4param(
    scanner,
    h_truth_line,
    drive=20,
    setpoint=0.6,
    scan_speed_hat=1e3,
    dx_hat=1.0,
    trim=20,

    # Fixed scanner parameters
    tau_z=1e-5,
    z_rate_limit_hat=1e6,
    d_init_hat=250,
    phi_meas_init_deg=120,
    T_I=3e-3,

    # Loss weights
    w_height=1.0,
    w_trrt=0.75,
    w_ringing=0.15,
    w_amp=0.05,
    w_phase=0.02,
    huber_delta=2.0,

    verbose=False,
):
    h_truth_line = np.asarray(h_truth_line, dtype=float)

    def objective(x, return_full=False):
        p = unpack_controller_params_4(x)

        try:
            out = scanner.scan_line_trace_retrace_pi_drive(
                h_hat=h_truth_line,
                drive_nm=drive,
                setpoint=setpoint,

                scan_speed_hat=scan_speed_hat,
                dx_hat=dx_hat,

                P=p["P"],
                I=p["I"],

                A_filter_order=1,
                tau_A=p["tau_A"],

                phi_filter_order=1,
                tau_phi=p["tau_phi"],

                z_filter_order=1,
                tau_z=tau_z,

                z_rate_limit_hat=z_rate_limit_hat,

                d_init_hat=d_init_hat,
                phi_meas_init_deg=phi_meas_init_deg,
                T_I=T_I,
                
            )

            loss, details = score_scan_no_reverse(
                out,
                h_truth_line,
                drive=drive,
                setpoint=setpoint,
                trim=trim,
                w_height=w_height,
                w_trrt=w_trrt,
                w_ringing=w_ringing,
                w_amp=w_amp,
                w_phase=w_phase,
                huber_delta=huber_delta,
            )

            if not np.isfinite(loss):
                loss = 1e12

        except Exception as e:
            if verbose:
                print("Simulation failed:", e)

            loss = 1e12
            details = {"error": str(e)}
            out = None

        if return_full:
            return loss, details, out, p
        else:
            return loss

    return objective

In [ ]:
def fit_controller_4param(
    scanner,
    h_truth_line,
    drive=20,
    setpoint=0.6,
    scan_speed_hat=1e3,
    dx_hat=1.0,

    # Fixed parameters
    tau_z=1e-5,
    z_rate_limit_hat=1e6,
    d_init_hat=250,
    phi_meas_init_deg=120,
    T_I=3e-3,
    trim=20,

    # Loss weights
    w_height=1.0,
    w_trrt=0.75,
    w_ringing=0.15,
    w_amp=0.05,
    w_phase=0.02,
    huber_delta=2.0,
    
    # Optimizer controls
    maxiter_global=80,
    maxiter_local=300,
    global_tol=0.02,
    local_xatol=1e-3,
    local_fatol=1e-4,
    loss_target=None,
    seed=0,
    popsize=10,

    
    # Bounds
    bounds=None,

    verbose=True,
):
    if bounds is None:
        bounds = [
            (-2.0, 2.0),    # log10_P:     1e-2 to ~32
            (-12.0, 4.0),   # log10_I:     ~0 to 1e4
            (-5.5, -2.0),   # log10_tau_A: 3e-6 to 1e-2
            (-5.5, -2.0),   # log10_tau_z: 1e-7 to 1e-3
        ]

    objective = make_scan_objective_4param(
        scanner=scanner,
        h_truth_line=h_truth_line,
        drive=drive,
        setpoint=setpoint,
        scan_speed_hat=scan_speed_hat,
        dx_hat=dx_hat,
        trim=trim,
        tau_z=tau_z,
        z_rate_limit_hat=z_rate_limit_hat,
        d_init_hat=d_init_hat,
        phi_meas_init_deg=phi_meas_init_deg,
        T_I=T_I,
        # Loss weights
        w_height= w_height,
        w_trrt=   w_trrt,
        w_ringing=w_ringing,
        w_amp=    w_amp,
        w_phase=  w_phase,
        huber_delta=huber_delta,
    )

    best_seen = {"loss": np.inf, "x": None}

    def callback(xk, convergence):
        loss = objective(xk)

        if loss < best_seen["loss"]:
            best_seen["loss"] = loss
            best_seen["x"] = np.array(xk, dtype=float)

            if verbose:
                p = unpack_controller_params_4(xk)
                print(
                    f"loss={loss:.5g}, "
                    f"P={p['P']:.4g}, I={p['I']:.4g}, "
                    f"tau_A={p['tau_A']:.4g}, tau_phi={p['tau_phi']:.4g}"
                )

        if loss_target is not None and loss <= loss_target:
            if verbose:
                print(f"Stopping early because loss <= loss_target = {loss_target}")
            return True

        return False

    res_global = differential_evolution(
        objective,
        bounds=bounds,
        seed=seed,
        maxiter=maxiter_global,
        popsize=popsize,
        tol=global_tol,
        polish=False,
        updating="immediate",
        workers=1,
        callback=callback,
    )

    if best_seen["x"] is not None and best_seen["loss"] < res_global.fun:
        x0 = best_seen["x"]
    else:
        x0 = res_global.x

    res_local = minimize(
        objective,
        x0,
        method="Nelder-Mead",
        options={
            "maxiter": maxiter_local,
            "xatol": local_xatol,
            "fatol": local_fatol,
        },
    )

    if res_local.fun < res_global.fun:
        best_x = res_local.x
        best_source = "local"
    else:
        best_x = res_global.x
        best_source = "global"

    best_loss, best_details, best_out, best_params = objective(
        best_x,
        return_full=True,
    )

    result = {
        "best_x": best_x,
        "best_params": best_params,
        "best_loss": best_loss,
        "best_details": best_details,
        "best_out": best_out,
        "best_source": best_source,
        "res_global": res_global,
        "res_local": res_local,
    }

    return result

In [ ]:
# scanner.scan_line_pi()

In [ ]:
h_line = h_truth[::2]

fit = fit_controller_4param(
    scanner=scanner,
    h_truth_line=h_line,
    drive=20,
    setpoint=0.6,

    scan_speed_hat=1e3,
    dx_hat=1.0,

    tau_z=3e-5,
    z_rate_limit_hat=1e5,
    d_init_hat=250,
    phi_meas_init_deg=120,
    T_I=None,

    maxiter_global=1000,
    maxiter_local=1000,
    global_tol=0.02,
    local_xatol=1e-3,
    local_fatol=1e-4,

    w_trrt=0.,
    w_ringing=0.,
    w_amp=0.,
    w_phase=0.,
    
    # Optional. Set to None if you do not want early stopping.
    loss_target=1,

    seed=0,
    popsize=8,
    verbose=True,
)

print("Best source:", fit["best_source"])
print("Best loss:", fit["best_loss"])
print("Best parameters:")
for k, v in fit["best_params"].items():
    print(f"{k}: {v}")
print("Loss details:")
for k, v in fit["best_details"].items():
    print(f"{k}: {v}")

In [ ]:
def plot_fit_result_no_reverse(fit, h_truth_line, drive=20, setpoint=0.6):
    out = fit["best_out"]
    h = np.asarray(h_truth_line)

    trace_h = np.asarray(out["trace"]["d_hat"])
    retrace_h = np.asarray(out["retrace"]["d_hat"])

    trace_h_aligned, off_t = align_by_constant_offset(trace_h, h)
    retrace_h_aligned, off_r = align_by_constant_offset(retrace_h, h)

    A_trace = np.asarray(out["trace"]["A_hat"])
    A_retrace = np.asarray(out["retrace"]["A_hat"])

    phi_trace = np.asarray(out["trace"]["phi_deg"])
    phi_retrace = np.asarray(out["retrace"]["phi_deg"])

    x = np.arange(len(h))

    fig, ax = plt.subplots(1, 4, figsize=(14, 3.2))

    ax[0].plot(x, trace_h_aligned, label="trace")
    ax[0].plot(x, retrace_h_aligned, label="retrace")
    ax[0].plot(x, h, "k--", label="h_truth")
    ax[0].set_ylabel("Height / d_hat")
    ax[0].set_title("Height tracking")
    ax[0].legend()

    ax[1].plot(x, trace_h_aligned - h, label="trace residual")
    ax[1].plot(x, retrace_h_aligned - h, label="retrace residual")
    ax[1].axhline(0, color="k", lw=1)
    ax[1].set_ylabel("Residual")
    ax[1].set_title("Height error")
    ax[1].legend()

    ax[2].plot(x, phi_trace, label="trace")
    ax[2].plot(x, phi_retrace, label="retrace")
    ax[2].set_ylabel("Phase deg")
    ax[2].set_title("Phase")
    ax[2].legend()

    ax[3].plot(x, A_trace, label="trace")
    ax[3].plot(x, A_retrace, label="retrace")
    ax[3].axhline(drive * setpoint, color="k", ls="--", label="target A")
    ax[3].set_ylabel("Amplitude")
    ax[3].set_title("Amplitude")
    ax[3].legend()

    plt.suptitle(
        f"loss={fit['best_loss']:.4g}, "
        f"P={fit['best_params']['P']:.3g}, "
        f"I={fit['best_params']['I']:.3g}, "
        f"tau_A={fit['best_params']['tau_A']:.3g}, "
        f"tau_phi={fit['best_params']['tau_phi']:.3g}",
        y=1.05,
    )

    plt.tight_layout()
    plt.show()


plot_fit_result_no_reverse(fit, h_line, drive=20, setpoint=0.6)

In [ ]:
import numpy as np
from scipy.optimize import differential_evolution, minimize
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Robust loss utilities
# ------------------------------------------------------------
def mad_std(x, eps=1e-12):
    x = np.asarray(x)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    return 1.4826 * mad + eps


def huber_loss(r, delta=2.0):
    r = np.asarray(r)
    a = np.abs(r)
    return np.where(a <= delta, 0.5 * r**2, delta * (a - 0.5 * delta))


def align_by_constant_offset(y, ref):
    """
    Remove best constant offset between simulated d_hat and h_truth.
    This lets the fitting focus on shape/alignment rather than absolute z offset.
    """
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)
    offset = np.nanmedian(y - ref)
    return y - offset, offset

def diff_shifted(y1, y2, thres=0.1):
    '''
    Find the optimal x-shift of y2 with respect to y1 that minimizes the absolute
    difference between y1 and y2_shifted. If the amount of shift is larger than
    thres * len(y1), the original y1 and y2 will be returned.

    Input:
        y1   - list: 1D scan trace/retrace
        y2   - list: 1D scan trace/retrace
        thres- list: threshold to determine if fitting is failed. If x_shift is
                    greater than thres * len(y1), original y1 and y2 will be returned.

    Returns:
        y1_cut     - y1 is cut to keep overlapped part with y2_shifted
        y2_shifted - shifted y2 array

    Usage:
        y1_shifted, y2_shifted = diff_shifted(y1, y2)

    '''
    # Initial guess for the shift
    initial_shift = 0

    x = np.arange(len(y1))

    # Run the optimization
    result = minimize(mse_shift, initial_shift, args=(x, y1, y2))

    # Best shift found
    best_shift = result.x

    # Create a spline of the second trace
    spline = CubicSpline(x, y2)

    # Shift x values by the found optimal shift
    x_shifted = x - best_shift

    # Evaluate the shifted trace at the original x values
    y2_optimal_shifted = spline(x_shifted)

    if np.abs(best_shift) > thres * len(y1):
        return y1, y2
    elif best_shift > 0:
        return y1[int(best_shift)+1:], y2_optimal_shifted[int(best_shift)+1:]
    elif best_shift < -1:
        return y1[:int(best_shift)-1], y2_optimal_shifted[:int(best_shift)-1]
    else:
        return y1, y2

def mse_shift(shift, x, y1, y2):
    # Create a spline of the second trace
    spline = CubicSpline(x, y2)

    # Shift x values
    x_shifted = x - shift

    # Evaluate the shifted trace at the original x values
    y2_shifted = spline(x_shifted)

    # Compute the MSE between the original and shifted traces
    mse = np.mean((y1 - y2_shifted)**2)
    return mse
    

def score_scan_no_reverse(
    out,
    h_truth_line,
    h_exp_line, 
    drive,
    setpoint,
    w_height=1.0,
    huber_delta=2.0,
):
    h = np.asarray(h_truth_line, dtype=float)

    trace_h_raw = np.asarray(out["trace"]["d_hat"], dtype=float)
    retrace_h_raw = np.asarray(out["retrace"]["d_hat"], dtype=float)

    # trace_h, trace_offset = align_by_constant_offset(trace_h_raw, h)
    # retrace_h, retrace_offset = align_by_constant_offset(retrace_h_raw, h)
    trace_h, trace_offset = trace_h_raw-np.min(trace_h_raw),np.min(trace_h_raw) 
    retrace_h, retrace_offset = retrace_h_raw-np.min(retrace_h_raw),np.min(retrace_h_raw) 
    # sl = slice(trim, len(h) - trim if trim > 0 else len(h))

    # h_use = h[sl]
    # tr_use = trace_h[sl]
    # rt_use = retrace_h[sl]

    h_use_tr, h_use_rt = h_exp_line
    # h_use_rt = h_exp_line[1]
    tr_use = trace_h
    rt_use = retrace_h
    
    # h_scale = max(mad_std(h_use), 0.05 * np.ptp(h_use), 1.0)
    h_scale = 1
    
    # Height tracking
    tr_use, h_use_tr = diff_shifted(tr_use, h_use_tr)
    rt_use, h_use_rt = diff_shifted(rt_use, h_use_rt)

    r_trace = (tr_use - h_use_tr) / h_scale
    r_retrace = (rt_use - h_use_rt) / h_scale
    
    loss_height = np.nanmean(
        huber_loss(np.r_[r_trace, r_retrace], delta=huber_delta)
    )

    
    total = (
        w_height * loss_height
    )

    details = {
        "total": total,
        "loss_height": loss_height,
        "trace_offset": trace_offset,
        "retrace_offset": retrace_offset,
        "h_scale": h_scale,
    }

    return total, details


def unpack_controller_params_4(x):
    log10_P, log10_I, log10_tau_A, log10_tau_phi = x

    return {
        "P": 10.0 ** log10_P,
        "I": 10.0 ** log10_I,
        "tau_A": 10.0 ** log10_tau_A,
        "tau_phi": 10.0 ** log10_tau_phi,
    }


def make_scan_objective_4param(
    scanner,
    h_truth_line,
    h_exp_line,
    drive=20,
    setpoint=0.6,
    scan_speed_hat=1e3,
    dx_hat=1.0,

    # Fixed scanner parameters
    tau_z=1e-5,
    z_rate_limit_hat=1e6,
    d_init_hat=250,
    phi_meas_init_deg=120,
    T_I=3e-3,

    # Loss weights
    w_height=1.0,
    huber_delta=2.0,

    verbose=False,
):
    h_truth_line = np.asarray(h_truth_line, dtype=float)

    def objective(x, return_full=False):
        p = unpack_controller_params_4(x)

        try:
            out = scanner.scan_line_trace_retrace_pi_drive(
                h_hat=h_truth_line,
                drive_nm=drive,
                setpoint=setpoint,

                scan_speed_hat=scan_speed_hat,
                dx_hat=dx_hat,

                P=p["P"],
                I=p["I"],

                A_filter_order=1,
                tau_A=p["tau_A"],

                phi_filter_order=1,
                tau_phi=p["tau_phi"],

                z_filter_order=1,
                tau_z=tau_z,

                z_rate_limit_hat=z_rate_limit_hat,

                d_init_hat=d_init_hat,
                phi_meas_init_deg=phi_meas_init_deg,
                T_I=T_I,
            )

            loss, details = score_scan_no_reverse(
                out,
                h_truth_line,
                h_exp_line,
                drive=drive,
                setpoint=setpoint,
                w_height=w_height,
                huber_delta=huber_delta,
            )

            if not np.isfinite(loss):
                loss = 1e12

        except Exception as e:
            if verbose:
                print("Simulation failed:", e)

            loss = 1e12
            details = {"error": str(e)}
            out = None

        if return_full:
            return loss, details, out, p
        else:
            return loss

    return objective

In [ ]:
import numpy as np
from scipy.optimize import differential_evolution, minimize
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Robust utilities
# ------------------------------------------------------------
def mad_std(x, eps=1e-12):
    x = np.asarray(x, dtype=float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    return 1.4826 * mad + eps


def huber_loss(r, delta=2.0):
    r = np.asarray(r, dtype=float)
    a = np.abs(r)
    return np.where(a <= delta, 0.5 * r**2, delta * (a - 0.5 * delta))


def crop_to_common_length(*arrays):
    arrays = [np.asarray(a, dtype=float) for a in arrays]
    n = min(len(a) for a in arrays)
    return [a[:n] for a in arrays]


def parse_h_exp(h_exp):
    """
    Accepts either:
        h_exp = (h_exp_tr, h_exp_rt)

    or:
        h_exp = {"trace": h_exp_tr, "retrace": h_exp_rt}
    """
    if isinstance(h_exp, dict):
        return np.asarray(h_exp["trace"], dtype=float), np.asarray(h_exp["retrace"], dtype=float)
    else:
        h_exp_tr, h_exp_rt = h_exp
        return np.asarray(h_exp_tr, dtype=float), np.asarray(h_exp_rt, dtype=float)


def apply_common_height_offset(sim_tr, sim_rt, exp_tr, exp_rt):
    """
    Apply one shared offset to both simulated trace and retrace.

    This preserves the simulated trace-retrace difference, which is important
    because the trace-retrace MSE is part of the objective.
    """
    sim_mid = 0.5 * (sim_tr + sim_rt)
    exp_mid = 0.5 * (exp_tr + exp_rt)

    offset = np.nanmedian(sim_mid - exp_mid)

    return sim_tr - offset, sim_rt - offset, offset

In [ ]:
def score_scan_against_experiment(
    out,
    h_exp,
    trim=20,

    # Main loss weights
    w_height_match=1.0,
    w_mse_match=0.5,

    # Optional regularizers
    w_diff_profile=0.1,
    w_ringing=0.05,
    w_amp=0.02,

    drive=20,
    setpoint=0.6,
    huber_delta=2.0,
    mse_floor_frac=0.03,
):
    """
    Objective:
    1. simulated trace/retrace height lines agree with experimental trace/retrace;
    2. simulated trace-retrace MSE is close to experimental trace-retrace MSE.

    No retrace reversal is applied.
    """

    exp_tr, exp_rt = parse_h_exp(h_exp)

    sim_tr = np.asarray(out["trace"]["d_hat"], dtype=float)
    sim_rt = np.asarray(out["retrace"]["d_hat"], dtype=float)

    # sim_tr, exp_tr = diff_shifted(sim_tr, exp_tr)
    # sim_rt, exp_rt = diff_shifted(sim_rt, exp_rt)

    sim_tr_use = sim_tr
    sim_rt_use = sim_rt
    exp_tr_use = exp_tr
    exp_rt_use = exp_rt

    exp_all = np.r_[exp_tr_use, exp_rt_use]
    exp_mid = 0.5 * (exp_tr_use + exp_rt_use)

    h_scale = 1

    # --------------------------------------------------------
    # 1. Height matching loss
    # --------------------------------------------------------
    r_height = np.r_[
        (sim_tr_use - exp_tr_use) / h_scale,
        (sim_rt_use - exp_rt_use) / h_scale,
    ]

    loss_height_match = np.nanmean(
        huber_loss(r_height, delta=huber_delta)
    )

    # --------------------------------------------------------
    # 2. Trace-retrace MSE matching loss
    # --------------------------------------------------------
    diff_sim = sim_tr_use - sim_rt_use
    diff_exp = exp_tr_use - exp_rt_use

    mse_sim = np.nanmean(diff_sim**2)
    mse_exp = np.nanmean(diff_exp**2)

    mse_floor = (mse_floor_frac * h_scale) ** 2

    # Log-ratio loss is better than raw MSE difference because MSE is positive
    # and can span orders of magnitude.
    loss_mse_match = (
        np.log((mse_sim + mse_floor) / (mse_exp + mse_floor))
    ) ** 2


    total = (
        1 * loss_height_match
        + 0.5 * loss_mse_match
        # + w_diff_profile * loss_diff_profile
        # + w_ringing * loss_ringing
        # + w_amp * loss_amp
    )
    
    # total = (
    #     w_height_match * loss_height_match
    #     + w_mse_match * loss_mse_match
    #     + w_diff_profile * loss_diff_profile
    #     + w_ringing * loss_ringing
    #     + w_amp * loss_amp
    # )

    details = {
        "total": total,
        "loss_height_match": loss_height_match,
        "loss_mse_match": loss_mse_match,
        # "loss_diff_profile": loss_diff_profile,
        # "loss_ringing": loss_ringing,
        # "loss_amp": loss_amp,
        "mse_sim": mse_sim,
        "mse_exp": mse_exp,
        "mse_ratio": mse_sim / (mse_exp + 1e-12),
        # "common_offset": common_offset,
        "h_scale": h_scale,
    }

    return total, details

In [ ]:
def unpack_controller_params_4(x):
    log10_P, log10_I, log10_tau_A, log10_tau_phi = x

    return {
        "P": 10.0 ** log10_P,
        "I": 10.0 ** log10_I,
        "tau_A": 10.0 ** log10_tau_A,
        "tau_phi": 10.0 ** log10_tau_phi,
    }


def make_scan_objective_exp_4param(
    scanner,
    h_input,
    h_exp,

    drive=20,
    setpoint=0.6,
    scan_speed_hat=1e3,
    dx_hat=1.0,

    # Fixed scanner parameters
    tau_z=1e-5,
    z_rate_limit_hat=1e6,
    d_init_hat=250,
    phi_meas_init_deg=120,
    T_I=3e-3,

    # Loss parameters
    trim=20,
    w_height_match=1.0,
    w_mse_match=0.5,
    w_diff_profile=0.1,
    w_ringing=0.05,
    w_amp=0.02,
    huber_delta=2.0,
    mse_floor_frac=0.03,

    verbose=False,
):
    h_input = np.asarray(h_input, dtype=float)

    def objective(x, return_full=False):
        p = unpack_controller_params_4(x)

        try:
            out = scanner.scan_line_trace_retrace_pi_drive(
                h_hat=h_input,
                drive_nm=drive,
                setpoint=setpoint,

                scan_speed_hat=scan_speed_hat,
                dx_hat=dx_hat,

                P=p["P"],
                I=p["I"],

                A_filter_order=1,
                tau_A=p["tau_A"],

                phi_filter_order=1,
                tau_phi=p["tau_phi"],

                z_filter_order=1,
                tau_z=tau_z,

                z_rate_limit_hat=z_rate_limit_hat,

                d_init_hat=d_init_hat,
                phi_meas_init_deg=phi_meas_init_deg,
                T_I=T_I,
            )

            loss, details = score_scan_against_experiment(
                out=out,
                h_exp=h_exp,
                trim=trim,
                w_height_match=w_height_match,
                w_mse_match=w_mse_match,
                w_diff_profile=w_diff_profile,
                w_ringing=w_ringing,
                w_amp=w_amp,
                drive=drive,
                setpoint=setpoint,
                huber_delta=huber_delta,
                mse_floor_frac=mse_floor_frac,
            )

            if not np.isfinite(loss):
                loss = 1e12

        except Exception as e:
            if verbose:
                print("Simulation failed:", e)

            loss = 1e12
            details = {"error": str(e)}
            out = None

        if return_full:
            return loss, details, out, p
        else:
            return loss

    return objective

In [ ]:
def fit_controller_to_experiment_4param(
    scanner,
    h_input,
    h_exp,

    drive=20,
    setpoint=0.6,
    scan_speed_hat=1e3,
    dx_hat=1.0,

    # Fixed scanner parameters
    tau_z=1e-5,
    z_rate_limit_hat=1e6,
    d_init_hat=250,
    phi_meas_init_deg=120,
    T_I=3e-3,

    # Loss parameters
    trim=20,
    w_height_match=1.0,
    w_mse_match=0.5,
    w_diff_profile=0.1,
    w_ringing=0.05,
    w_amp=0.02,
    huber_delta=2.0,
    mse_floor_frac=0.03,

    # Optimizer controls
    maxiter_global=80,
    maxiter_local=300,
    global_tol=0.02,
    local_xatol=1e-3,
    local_fatol=1e-4,
    loss_target=None,
    seed=0,
    popsize=10,

    # Bounds in log10-space
    bounds=None,

    verbose=True,
):
    if bounds is None:
        bounds = [
            (-2.0, 6.0),    # log10_P:     1e-2 to ~32
            (-2.0, 6.0),   # log10_I:     effectively 0 to 1e4
            (-5.5, -2.0),   # log10_tau_A: 3e-6 to 1e-2
            (-5.5, -2.0),   # log10_tau_phi: 1e-7 to 1e-3
        ]

    objective = make_scan_objective_exp_4param(
        scanner=scanner,
        h_input=h_input,
        h_exp=h_exp,
        drive=drive,
        setpoint=setpoint,
        scan_speed_hat=scan_speed_hat,
        dx_hat=dx_hat,
        tau_z=tau_z,
        z_rate_limit_hat=z_rate_limit_hat,
        d_init_hat=d_init_hat,
        phi_meas_init_deg=phi_meas_init_deg,
        T_I=T_I,
        trim=trim,
        w_height_match=w_height_match,
        w_mse_match=w_mse_match,
        w_diff_profile=w_diff_profile,
        w_ringing=w_ringing,
        w_amp=w_amp,
        huber_delta=huber_delta,
        mse_floor_frac=mse_floor_frac,
        verbose=verbose,
    )

    best_seen = {"loss": np.inf, "x": None}

    def callback(xk, convergence):
        loss = objective(xk)

        if loss < best_seen["loss"]:
            best_seen["loss"] = loss
            best_seen["x"] = np.array(xk, dtype=float)

            if verbose:
                p = unpack_controller_params_4(xk)
                _, details, _, _ = objective(xk, return_full=True)

                print(
                    f"loss={loss:.5g}, "
                    f"P={p['P']:.4g}, I={p['I']:.4g}, "
                    f"tau_A={p['tau_A']:.4g}, tau_phi={p['tau_phi']:.4g}, "
                    f"mse_ratio={details.get('mse_ratio', np.nan):.4g}"
                )

        if loss_target is not None and loss <= loss_target:
            if verbose:
                print(f"Stopping early because loss <= loss_target = {loss_target}")
            return True

        return False

    res_global = differential_evolution(
        objective,
        bounds=bounds,
        seed=seed,
        maxiter=maxiter_global,
        popsize=popsize,
        tol=global_tol,
        polish=False,
        updating="immediate",
        workers=1,
        callback=callback,
    )

    if best_seen["x"] is not None and best_seen["loss"] < res_global.fun:
        x0 = best_seen["x"]
    else:
        x0 = res_global.x

    res_local = minimize(
        objective,
        x0,
        method="Nelder-Mead",
        options={
            "maxiter": maxiter_local,
            "xatol": local_xatol,
            "fatol": local_fatol,
        },
    )

    if res_local.fun < res_global.fun:
        best_x = res_local.x
        best_source = "local"
    else:
        best_x = res_global.x
        best_source = "global"

    best_loss, best_details, best_out, best_params = objective(
        best_x,
        return_full=True,
    )

    return {
        "best_x": best_x,
        "best_params": best_params,
        "best_loss": best_loss,
        "best_details": best_details,
        "best_out": best_out,
        "best_source": best_source,
        "res_global": res_global,
        "res_local": res_local,
        "objective": objective,
    }

In [ ]:
traces = param_clean['traces']
traces.shape

In [ ]:
param_clean['x_measured']

In [ ]:
plt.plot((traces[0,-1,:2]-np.min(traces[0,-1,:2]))[0]*1e9)
plt.plot((traces[0,-1,:2]-np.min(traces[0,-1,:2]))[1]*1e9)


In [ ]:
drive_exp = param_clean['x_measured'][:,0]
setpoint_exp = param_clean['x_measured'][:,1] / param_clean['x_measured'][:,0]
igain_exp = param_clean['x_measured'][:,2]

In [ ]:
traces_exp = np.zeros([60, 2, 256])

for i in range(60):
    for j in range(2):
        traces_exp[i,j] = (traces[i,-1,j]-np.min(traces[i,-1,j]))*1e9

traces_exp.shape

In [ ]:
idx = 15
# h_line = traces_exp[idx,1]
plt.plot(traces_exp[idx,1])
plt.plot(h_line)

In [ ]:
def plot_fit_to_experiment(fit, h_exp):
    out = fit["best_out"]

    exp_tr, exp_rt = parse_h_exp(h_exp)

    sim_tr = np.asarray(out["trace"]["d_hat"], dtype=float)
    sim_rt = np.asarray(out["retrace"]["d_hat"], dtype=float)

    sim_tr, sim_rt, exp_tr, exp_rt = crop_to_common_length(
        sim_tr, sim_rt, exp_tr, exp_rt
    )

    sim_tr_aligned, sim_rt_aligned, offset = apply_common_height_offset(
        sim_tr,
        sim_rt,
        exp_tr,
        exp_rt,
    )

    x = np.arange(len(exp_tr))

    fig, ax = plt.subplots(1, 4, figsize=(15, 3.3))

    ax[0].plot(x, exp_tr, "k--", label="exp trace")
    ax[0].plot(x, exp_rt, "k:", label="exp retrace")
    ax[0].plot(x, sim_tr_aligned, label="sim trace")
    ax[0].plot(x, sim_rt_aligned, label="sim retrace")
    ax[0].set_title("Height lines")
    ax[0].set_ylabel("Height")
    ax[0].legend()

    ax[1].plot(x, sim_tr_aligned - exp_tr, label="trace residual")
    ax[1].plot(x, sim_rt_aligned - exp_rt, label="retrace residual")
    ax[1].axhline(0, color="k", lw=1)
    ax[1].set_title("Residuals")
    ax[1].legend()

    diff_exp = exp_tr - exp_rt
    diff_sim = sim_tr_aligned - sim_rt_aligned

    ax[2].plot(x, diff_exp, "k--", label="exp trace-retrace")
    ax[2].plot(x, diff_sim, label="sim trace-retrace")
    ax[2].set_title("Trace-retrace difference")
    ax[2].legend()

    mse_exp = np.mean(diff_exp**2)
    mse_sim = np.mean(diff_sim**2)

    ax[3].bar([0, 1], [mse_exp, mse_sim])
    ax[3].set_xticks([0, 1])
    ax[3].set_xticklabels(["exp", "sim"])
    ax[3].set_title("Trace-retrace MSE")

    title = (
        f"loss={fit['best_loss']:.4g}, "
        f"P={fit['best_params']['P']:.3g}, "
        f"I={fit['best_params']['I']:.3g}, "
        f"tau_A={fit['best_params']['tau_A']:.3g}, "
        f"tau_phi={fit['best_params']['tau_phi']:.3g}, "
        f"MSE ratio={fit['best_details']['mse_ratio']:.3g}"
    )

    fig.suptitle(title, y=1.06)
    plt.tight_layout()
    plt.show()




In [ ]:
idx = 4
fit = fit_controller_to_experiment_4param(
    scanner=scanner,
    h_input=h_line,
    h_exp = traces_exp[idx],
    drive=drive_exp[idx],
    setpoint=setpoint_exp[idx],

    scan_speed_hat=1e3,
    dx_hat=1.0,

    tau_z=3e-5,
    z_rate_limit_hat=1e5,
    d_init_hat=250,
    phi_meas_init_deg=120,
    T_I=None,

    maxiter_global=10000,
    maxiter_local=1000,
    global_tol=0.02,
    local_xatol=1e-3,
    local_fatol=1e-4,
    
    # Optional. Set to None if you do not want early stopping.
    loss_target=1,

    seed=0,
    popsize=8,
    verbose=True,
)

In [ ]:
plot_fit_to_experiment(fit, traces_exp[idx])

In [ ]:
setpoint_exp

In [ ]:
out = []

h_line = h_truth[::2]

for i in range(len(traces)):
    print('Working on {}/{}'.format(i, 60), end='\r')
    fit = fit_controller_4param(
        scanner=scanner,
        h_truth_line=h_line,
        h_exp_line = traces_exp[i],
        drive=param_clean['x_measured'][i,0],
        setpoint=param_clean['x_measured'][i,1]/param_clean['x_measured'][i,0],
    
        scan_speed_hat=1e3,
        dx_hat=1.0,
    
        tau_z=3e-5,
        z_rate_limit_hat=1e5,
        d_init_hat=250,
        phi_meas_init_deg=120,
        T_I=None,
    
        maxiter_global=1000,
        maxiter_local=1000,
        global_tol=0.02,
        local_xatol=1e-3,
        local_fatol=1e-4,
    
        w_trrt=0.,
        w_ringing=0.,
        w_amp=0.,
        w_phase=0.,
        
        # Optional. Set to None if you do not want early stopping.
        loss_target=1,
    
        seed=0,
        popsize=8,
        verbose=True,
    )
    out.append(fit)

In [ ]:
out

In [ ]:
# drive_exp = param_clean['x_measured'][:,0]
# setpoint_exp = param_clean['x_measured'][:,1] / param_clean['x_measured'][:,0]
# igain_exp = param_clean['x_measured'][:,2]

In [ ]:
idx = 10

print("Drive={} nm, setpoint={}, I Gain={}".format(drive_exp[idx],setpoint_exp[idx],igain_exp[idx]))

out = scanner.scan_line_trace_retrace_pi_drive(
    h_hat=h_truth,
    drive_nm=drive_exp[idx],        # continuous drive, not limited to measured curves
    setpoint=setpoint_exp[idx],        # continuous setpoint fraction of current A0

    scan_speed_hat=1e3,
    dx_hat=1.0,
    P=3.93,
    I=0,

    A_filter_order=1,
    tau_A=8e-4,
    phi_filter_order=1,
    tau_phi=3e-5,
    z_filter_order=1,
    tau_z=3e-5,

    d_init_hat=250,
    # T_I = 3e-3,
    phi_meas_init_deg=120,
    z_rate_limit_hat=1e5,
)

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out['trace']['d_hat'][::2]-drive*setpoint*0)
ax[0].plot(out['retrace']['d_hat'][::2]-drive*setpoint*0)
# ax[0].plot(h_truth[::2], 'g--')
ax[1].plot(out['trace']['phi_deg'][::2])
ax[1].plot(out['retrace']['phi_deg'][::2])
ax[2].plot(out['trace']['A_hat'][::2])
ax[2].plot(out['retrace']['A_hat'][::2])
ax[0].set_ylabel('Height (nm)')
ax[1].set_ylabel('Phase (deg)')
ax[2].set_ylabel('Amplitude (nm)')

plt.tight_layout()

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot((traces[idx, -1, 0]-np.min(traces[idx, -1, 0]))*1e9)
ax[0].plot((traces[idx, -1, 1]-np.min(traces[idx, -1, 1]))*1e9)
ax[1].plot(traces[idx, -1, 4])
ax[1].plot(traces[idx, -1, 5])
ax[2].plot(traces[idx, -1, 2]*1e9)
ax[2].plot(traces[idx, -1, 3]*1e9)
ax[0].set_ylabel('Height (nm)')
ax[1].set_ylabel('Phase (deg)')
ax[2].set_ylabel('Amplitude (nm)')

plt.tight_layout()


In [ ]:
n1 = 21
n2 = 21

drive_grid = np.linspace(5, 100, num=n1)
setpoint_grid = np.linspace(0.2, 0.9, num=n2)
rew1 = np.zeros([n1, n2])
rew2 = np.zeros([n1, n2])
rew3 = np.zeros([n1, n2])
rew4 = np.zeros([n1, n2])




out_array = np.zeros([n1, n2, 6, 256])

for i in range(n1):
    for j in range(n2):
        print('Working on {}/{}'.format(i*n1+j, n1*n2), end='\r')
        out = scanner.scan_line_trace_retrace_pi_drive(
            h_hat=h_truth[::2],
            drive_nm=drive_grid[i],        # continuous drive, not limited to measured curves
            setpoint=setpoint_grid[j],        # continuous setpoint fraction of current A0
        
            scan_speed_hat=1e3,
            dx_hat=1.0,
            P=2.8,
            I=4.8,
        
            A_filter_order=1,
            tau_A=8e-4,
            phi_filter_order=1,
            tau_phi=5e-3,
            z_filter_order=1,
            tau_z=1e-5,
        
            d_init_hat=350,
            T_I=3e-3,
        )
        out_array[i,j,0] = out['trace']['d_hat']
        out_array[i,j,1] = out['retrace']['d_hat']
        out_array[i,j,2] = out['trace']['phi_deg']
        out_array[i,j,3] = out['retrace']['phi_deg']
        out_array[i,j,4] = out['trace']['A_hat']
        out_array[i,j,5] = out['retrace']['A_hat']

        rew1[i,j] = np.mean(np.abs((out['trace']['d_hat']+out['retrace']['d_hat'])/2-
                                   np.min((out['trace']['d_hat']+out['retrace']['d_hat'])/2)-h_truth[::2]))
        rew2[i,j] = np.mean(np.abs((out['trace']['d_hat']-out['retrace']['d_hat'])))

        ph1 = len(np.where(out['trace']['phi_deg']<90)[0])
        ph2 = len(np.where(out['retrace']['phi_deg']<90)[0])
        rew3[i,j] = ph1 + ph2
        
        rew4[i,j] = np.mean(np.abs((out['trace']['A_hat']+out['retrace']['A_hat'])/2-drive_grid[i]*setpoint_grid[j]))



In [ ]:
imgs = []
rews = [rew1, rew2, rew3, rew4]

fig, ax = plt.subplots(2, 2, figsize=(9, 7))
axes = ax.flatten()

for i in range(4):
    im = axes[i].imshow(
        np.log(rews[i]),
        extent=[20, 80, 5, 100],
        aspect='auto',
        origin='lower'
    )
    imgs.append(im)

    fig.colorbar(im, ax=axes[i])
    axes[i].set_xlabel('Setpoint (%)')
    axes[i].set_ylabel('Drive Amplitude (nm)')

plt.tight_layout()
plt.show()
np.where(rew4==np.min(rew4))

In [ ]:
drive = 20
setpoint = 0.6

idx1 = 1
idx2 = 4

print("Drive={}, Setpoint={}".format(drive_grid[idx1], setpoint_grid[idx2]))

out = scanner.scan_line_trace_retrace_pi_drive(
    h_hat=h_truth[::2],
    drive_nm=drive_grid[idx1],        # continuous drive, not limited to measured curves
    setpoint=setpoint_grid[idx2],        # continuous setpoint fraction of current A0

    scan_speed_hat=1e3,
    dx_hat=1.0,
    P=2.8,
    I=4.8,

    A_filter_order=1,
    tau_A=8e-4,
    phi_filter_order=1,
    tau_phi=5e-3,
    z_filter_order=1,
    tau_z=1e-5,

    d_init_hat=350,
    T_I=3e-3,
)

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out['trace']['d_hat']-drive*setpoint*0)
ax[0].plot(out['retrace']['d_hat']-drive*setpoint*0)
ax[0].plot(h_truth[::2], 'g--')
ax[1].plot(out['trace']['phi_deg'])
ax[1].plot(out['retrace']['phi_deg'])
ax[2].plot(out['trace']['A_hat'])
ax[2].plot(out['retrace']['A_hat'])
ax[0].set_ylabel('Height (nm)')
ax[1].set_ylabel('Phase (deg)')
ax[2].set_ylabel('Amplitude (nm)')

plt.tight_layout()


In [ ]:
# n1 = 21
# n2 = 21

# drive_grid = np.linspace(5, 100, num=n1)
# setpoint_grid = np.linspace(0.2, 0.9, num=n2)
# rew = np.zeros([n1, n2])

# out_array = np.zeros([n1, n2, 6, 256])

# for i in range(n1):
#     for j in range(n2):
#         print('Working on {}/{}'.format(i*n1+j, n1*n2), end='\r')
#         out = scanner.scan_line_trace_retrace_pi_drive(
#             h_hat=h_truth[::2],
#             drive_nm=drive_grid[i],        # continuous drive, not limited to measured curves
#             setpoint=setpoint_grid[j],        # continuous setpoint fraction of current A0
        
#             scan_speed_hat=1e3,
#             dx_hat=1.0,
#             P=1.8,
#             I=4.8,
        
#             A_filter_order=1,
#             tau_A=8e-4,
#             phi_filter_order=1,
#             tau_phi=5e-3,
#             z_filter_order=1,
#             tau_z=1e-5,
        
#             d_init_hat=350,
#             T_I=3e-3,
#         )
#         out_array[i,j,0] = out['trace']['d_hat']
#         out_array[i,j,1] = out['retrace']['d_hat']
#         out_array[i,j,2] = out['trace']['phi_deg']
#         out_array[i,j,3] = out['retrace']['phi_deg']
#         out_array[i,j,4] = out['trace']['A_hat']
#         out_array[i,j,5] = out['retrace']['A_hat']

#         rew[i,j] = np.mean(np.abs((out['trace']['d_hat']+out['retrace']['d_hat'])/2-h_truth[::2]))
#         # rew2[i,j] = np.mean(np.abs((out['trace']['phi_deg']+out['retrace']['phi_deg'])/2-131.4))
#         # rew3[i,j] = np.mean(np.abs((out['trace']['A_hat']+out['retrace']['A_hat'])/2-11.81))


In [ ]:
# plt.imshow(np.log(rew), extent=[20, 80, 5, 100], aspect='auto', origin='lower')
# plt.colorbar()
# plt.xlabel('Setpoint (%)')
# plt.ylabel('Drive Amplitude (nm)')

In [ ]:
# n1 = 21
# n2 = 21

# drive_grid = np.linspace(5, 100, num=n1)
# setpoint_grid = np.linspace(0.2, 0.9, num=n2)
# rew = np.zeros([n1, n2])

# out_array = np.zeros([n1, n2, 6, 256])

# for i in range(n1):
#     for j in range(n2):
#         print('Working on {}/{}'.format(i*n1+j, n1*n2), end='\r')
#         out = scanner.scan_line_trace_retrace_pi_drive(
#             h_hat=h_truth[::2],
#             drive_nm=drive_grid[i],        # continuous drive, not limited to measured curves
#             setpoint=setpoint_grid[j],        # continuous setpoint fraction of current A0
        
#             scan_speed_hat=1e3,
#             dx_hat=1.0,
#             P=0,
#             I=5e3,
        
#             A_filter_order=1,
#             tau_A=8e-4,
#             phi_filter_order=1,
#             tau_phi=5e-3,
#             z_filter_order=1,
#             tau_z=1e-5,
        
#             d_init_hat=350,
#             T_I=3e-3,
#         )
#         out_array[i,j,0] = out['trace']['d_hat']
#         out_array[i,j,1] = out['retrace']['d_hat']
#         out_array[i,j,2] = out['trace']['phi_deg']
#         out_array[i,j,3] = out['retrace']['phi_deg']
#         out_array[i,j,4] = out['trace']['A_hat']
#         out_array[i,j,5] = out['retrace']['A_hat']

#         rew[i,j] = np.mean(np.abs((out['trace']['d_hat']+out['retrace']['d_hat'])/2-h_truth[::2]))
#         # rew2[i,j] = np.mean(np.abs((out['trace']['phi_deg']+out['retrace']['phi_deg'])/2-131.4))
#         # rew3[i,j] = np.mean(np.abs((out['trace']['A_hat']+out['retrace']['A_hat'])/2-11.81))

# plt.imshow(np.log(rew), extent=[20, 80, 5, 100], aspect='auto', origin='lower')
# plt.colorbar()
# plt.xlabel('Setpoint (%)')
# plt.ylabel('Drive Amplitude (nm)')

In [ ]:
param_clean.keys()

In [ ]:
param_clean['x_measured']

In [ ]:
traces.shape

In [ ]:
drive_exp = param_clean['x_measured'][:,0]
setpoint_exp = param_clean['x_measured'][:,1] / param_clean['x_measured'][:,0]
igain_exp = param_clean['x_measured'][:,2]

In [ ]:
rew_exp = np.zeros(60)

for i in range(60):
    tp1 = traces[i,-1, 0] * 1e9
    tp2 = traces[i,-1, 1] * 1e9
    rew_exp[i] = np.mean(np.abs(tp1-tp2))

In [ ]:
plt.scatter(setpoint_exp, drive_exp, c=rew_exp, clim=[0, 5])
plt.colorbar()

In [ ]:
scanner.scan_line_pi()

In [ ]:
idx = 15

print("Drive={} nm, setpoint={}, I Gain={}".format(drive_exp[idx],setpoint_exp[idx],igain_exp[idx]))

out = scanner.scan_line_trace_retrace_pi_drive(
    h_hat=h_truth,
    drive_nm=drive_exp[idx],        # continuous drive, not limited to measured curves
    setpoint=setpoint_exp[idx],        # continuous setpoint fraction of current A0

    scan_speed_hat=1e3,
    dx_hat=1.0,
    P=2.2,
    I=4.8,

    A_filter_order=1,
    tau_A=8e-4,
    phi_filter_order=1,
    tau_phi=5e-5,
    z_filter_order=1,
    tau_z=3e-5,

    d_init_hat=250,
    # T_I = 3e-3,
    phi_meas_init_deg=120,
    z_rate_limit_hat=1e5,
)

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out['trace']['d_hat'][::2]-drive*setpoint*0)
ax[0].plot(out['retrace']['d_hat'][::2]-drive*setpoint*0)
# ax[0].plot(h_truth[::2], 'g--')
ax[1].plot(out['trace']['phi_deg'][::2])
ax[1].plot(out['retrace']['phi_deg'][::2])
ax[2].plot(out['trace']['A_hat'][::2])
ax[2].plot(out['retrace']['A_hat'][::2])
ax[0].set_ylabel('Height (nm)')
ax[1].set_ylabel('Phase (deg)')
ax[2].set_ylabel('Amplitude (nm)')

plt.tight_layout()

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot((traces[idx, -1, 0]-np.min(traces[idx, -1, 0]))*1e9)
ax[0].plot((traces[idx, -1, 1]-np.min(traces[idx, -1, 1]))*1e9)
ax[1].plot(traces[idx, -1, 4])
ax[1].plot(traces[idx, -1, 5])
ax[2].plot(traces[idx, -1, 2]*1e9)
ax[2].plot(traces[idx, -1, 3]*1e9)
ax[0].set_ylabel('Height (nm)')
ax[1].set_ylabel('Phase (deg)')
ax[2].set_ylabel('Amplitude (nm)')

plt.tight_layout()


In [ ]:
topo.header

In [ ]:
idx = 15

print("Drive={} nm, setpoint={}, I Gain={}".format(drive_exp[idx],setpoint_exp[idx],igain_exp[idx]))

out = scanner.scan_line_trace_retrace_pi_drive(
    h_hat=h_truth[::2],
    drive_nm=drive_exp[idx],        # continuous drive, not limited to measured curves
    setpoint=setpoint_exp[idx],        # continuous setpoint fraction of current A0

    scan_speed_hat=1e3,
    dx_hat=1.0,
    P=0.1,
    I=5e2,

    A_filter_order=1,
    tau_A=8e-3,
    phi_filter_order=1,
    tau_phi=5e-3,
    z_filter_order=1,
    tau_z=1e-3,

    d_init_hat=250,
    # T_I=3e-2,
    phi_meas_init_deg=120
)

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out['trace']['d_hat']-drive*setpoint*0)
ax[0].plot(out['retrace']['d_hat']-drive*setpoint*0)
ax[0].plot(h_truth[::2], 'g--')
ax[1].plot(out['trace']['phi_deg'])
ax[1].plot(out['retrace']['phi_deg'])
ax[2].plot(out['trace']['A_hat'])
ax[2].plot(out['retrace']['A_hat'])
ax[0].set_ylabel('Height (nm)')
ax[1].set_ylabel('Phase (deg)')
ax[2].set_ylabel('Amplitude (nm)')

plt.tight_layout()

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot((traces[idx, -1, 0]-np.min(traces[idx, -1, 0]))*1e9)
ax[0].plot((traces[idx, -1, 1]-np.min(traces[idx, -1, 1]))*1e9)
ax[1].plot(traces[idx, -1, 4])
ax[1].plot(traces[idx, -1, 5])
ax[2].plot(traces[idx, -1, 2]*1e9)
ax[2].plot(traces[idx, -1, 3]*1e9)
ax[0].set_ylabel('Height (nm)')
ax[1].set_ylabel('Phase (deg)')
ax[2].set_ylabel('Amplitude (nm)')

plt.tight_layout()


In [ ]:
import numpy as np
from scipy.optimize import minimize_scalar


def fit_I_gain(
    scanner,
    h_truth,
    drive_nm,
    setpoint,
    z_trace_exp,
    z_retrace_exp,
    base_params,
    I_bounds=(0.1, 100.0),
    n_grid=15,
    skip_pixels=20,
    weight_retrace=1.0,
    instability_penalty=True,
    verbose=False,
):
    """
    Fit the integral gain `I` of the simulator by minimizing MSE between the
    simulated z scan lines and an experimental reference.

    The search uses a log-spaced coarse grid (robust to instability cliffs) and
    then a bounded scalar refinement around the best grid point.

    Parameters
    ----------
    scanner : object exposing `scan_line_trace_retrace_pi_drive(...)`.
    h_truth : 1D array, nm
        Ground-truth height profile of the calibration sample.
    drive_nm : float or 1D array, nm
        Drive amplitude trace used during the experimental line.
    setpoint : float or 1D array
        Setpoint (fraction of free amplitude) used during the experimental line.
    z_trace_exp, z_retrace_exp : 1D arrays, nm
        Measured forward and backward z traces.
    base_params : dict
        All other kwargs to scan_line_trace_retrace_pi_drive *except* I,
        e.g. P, tau_A, tau_phi, tau_z, scan_speed_hat, dx_hat, d_init_hat,
        T_I, phi_meas_init_deg, filter orders, etc.
    I_bounds : (lo, hi)
        Search interval for I (log-spaced).
    n_grid : int
        Number of points in the coarse grid scan.
    skip_pixels : int
        Pixels to discard at the start of each direction (filter settle-in).
    weight_retrace : float
        Relative weight of retrace MSE vs trace MSE in the loss.
    instability_penalty : bool
        Add a soft penalty when the simulated z range vastly exceeds the
        experimental range (catches limit cycles that don't NaN out).
    verbose : bool
        Print each evaluation.

    Returns
    -------
    I_best : float
    mse_best : float
    history : list[(I, mse)]
        Every evaluation, in call order. Useful for diagnostics / plotting.
    """
    history = []

    exp_range = np.ptp(z_trace_exp) + np.ptp(z_retrace_exp)
    BIG = 1e12

    def loss(I_value):
        try:
            out = scanner.scan_line_trace_retrace_pi_drive(
                h_hat=h_truth,
                drive_nm=drive_nm,
                setpoint=setpoint,
                I=float(I_value),
                **base_params,
            )
        except Exception as e:
            if verbose:
                print(f"  I={I_value:8.3f}  sim raised {type(e).__name__}: {e}")
            history.append((float(I_value), BIG))
            return BIG

        # Adjust these keys to whatever your scanner actually returns.
        z_t = np.asarray(out["trace"]['d_hat'], dtype=float)
        z_r = np.asarray(out["retrace"]['d_hat'], dtype=float)

        z_t, z_r = z_t - np.min(z_t), z_r - np.min(z_r)
        
        if not (np.all(np.isfinite(z_t)) and np.all(np.isfinite(z_r))):
            history.append((float(I_value), BIG))
            if verbose:
                print(f"  I={I_value:8.3f}  non-finite output (unstable)")
            return BIG

        s = skip_pixels
        mse_t = np.mean((z_t[s:] - z_trace_exp[s:]) ** 2)
        mse_r = np.mean((z_r[s:] - z_retrace_exp[s:]) ** 2)
        mse = mse_t + weight_retrace * mse_r

        if instability_penalty:
            sim_range = np.ptp(z_t) + np.ptp(z_r)
            if sim_range > 3.0 * exp_range:
                mse += 1e3 * (sim_range / max(exp_range, 1e-9))

        history.append((float(I_value), float(mse)))
        if verbose:
            print(f"  I={I_value:8.3f}  MSE={mse:.4e}")
        return mse

    # ---- Phase 1: log-spaced coarse grid ----
    if verbose:
        print(f"Phase 1: coarse grid over [{I_bounds[0]}, {I_bounds[1]}], n={n_grid}")
    I_grid = np.logspace(np.log10(I_bounds[0]), np.log10(I_bounds[1]), n_grid)
    mse_grid = np.array([loss(I) for I in I_grid])
    best_idx = int(np.argmin(mse_grid))

    # ---- Phase 2: bounded scalar refinement ----
    lo = I_grid[max(0, best_idx - 1)]
    hi = I_grid[min(n_grid - 1, best_idx + 1)]
    if verbose:
        print(f"Phase 2: refining in ({lo:.4g}, {hi:.4g})")
    res = minimize_scalar(
        loss,
        bounds=(lo, hi),
        method="bounded",
        options={"xatol": max(1e-3, 1e-3 * lo)},
    )

    return float(res.x), float(res.fun), history

In [ ]:
h_exp = (traces[idx, -1, 0]-np.min(traces[idx, -1, 0]))*1e9
plt.plot(h_exp)

In [ ]:
base = dict(
    scan_speed_hat=500,         # 2 ms/pixel to match the 1 Hz line scan
    dx_hat=1.0,
    P=0.0,                       # experiment runs pure-I
    A_filter_order=1,            # match Asylum lock-in (4th-order if available)
    tau_A=3.2e-5,
    phi_filter_order=1,
    tau_phi=3.2e-5,
    z_filter_order=1,
    tau_z=3.2e-5,
    # d_init_hat=30,               # just above A0 = 22.88 nm
    T_I=3e-1,
    phi_meas_init_deg=113,
)

I_best, mse, hist = fit_I_gain(
    scanner=scanner,
    h_truth=h_exp,
    drive_nm=drive_exp[idx],
    setpoint=setpoint_exp[idx],
    z_trace_exp=(traces[idx, -1, 0]-np.min(traces[idx, -1, 0]))*1e9,
    z_retrace_exp=(traces[idx, -1, 1]-np.min(traces[idx, -1, 1]))*1e9,
    base_params=base,
    I_bounds=(0.1, 1000.0),
    verbose=True,
    n_grid=101,
)

In [ ]:
plt.plot(hist)

In [ ]:
scanner.scan_line_pi()

In [ ]:
idx = 15

print("Drive={} nm, setpoint={}, I Gain={}".format(drive_exp[idx],setpoint_exp[idx],igain_exp[idx]))

out = scanner.scan_line_trace_retrace_pi_drive(
    h_hat=h_truth[::2],
    drive_nm=drive_exp[idx],        # continuous drive, not limited to measured curves
    setpoint=setpoint_exp[idx],        # continuous setpoint fraction of current A0

    scan_speed_hat=1e3/2,
    dx_hat=1.0,
    P=1.2,
    I=0.,
    A_filter_order=1,
    tau_A=1e-3,
    phi_filter_order=1,
    tau_phi=1e-3,
    z_filter_order=1,
    tau_z=3e-4,
    d_init_hat=250,
    T_I = 3e-1,
    phi_meas_init_deg=120,
    z_rate_limit_hat = 1e6
    
)

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out['trace']['d_hat']-drive*setpoint*0)
ax[0].plot(out['retrace']['d_hat']-drive*setpoint*0)
ax[0].plot(h_truth[::2], 'g--')
ax[1].plot(out['trace']['phi_deg'])
ax[1].plot(out['retrace']['phi_deg'])
ax[2].plot(out['trace']['A_hat'])
ax[2].plot(out['retrace']['A_hat'])
ax[0].set_ylabel('Height (nm)')
ax[1].set_ylabel('Phase (deg)')
ax[2].set_ylabel('Amplitude (nm)')

plt.tight_layout()

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot((traces[idx, -1, 0]-np.min(traces[idx, -1, 0]))*1e9)
ax[0].plot((traces[idx, -1, 1]-np.min(traces[idx, -1, 1]))*1e9)
ax[1].plot(traces[idx, -1, 4])
ax[1].plot(traces[idx, -1, 5])
ax[2].plot(traces[idx, -1, 2]*1e9)
ax[2].plot(traces[idx, -1, 3]*1e9)
ax[0].set_ylabel('Height (nm)')
ax[1].set_ylabel('Phase (deg)')
ax[2].set_ylabel('Amplitude (nm)')

plt.tight_layout()


In [ ]:
from scipy.optimize import differential_evolution

def loss(params, scanner, h_truth, drive, setpoint, z_t_exp, z_r_exp, base):
    P, tau_eff, tau_z_eff = params
    out = scanner.scan_line_trace_retrace_pi_drive(
        h_hat=h_truth, drive_nm=drive, setpoint=setpoint,
        P=P, I=0.0, T_I=None,
        A_filter_order=2, tau_A=tau_eff,
        phi_filter_order=2, tau_phi=tau_eff,
        z_filter_order=1, tau_z=tau_z_eff,
        **base,
    )
    z_t = np.asarray(out['z_trace']); z_r = np.asarray(out['z_retrace'])
    if not np.all(np.isfinite(z_t)): return 1e12
    s = 20
    return (np.mean((z_t[s:] - z_t_exp[s:])**2)
            + np.mean((z_r[s:] - z_r_exp[s:])**2))

bounds = [(0.1, 10.0),       # P
          (2e-4, 5e-3),       # tau_eff
          (1e-4, 2e-3)]       # tau_z_eff

result = differential_evolution(
    loss, bounds, args=(scanner, h_truth, drive, sp, z_t_exp, z_r_exp, base),
    maxiter=40, popsize=15, tol=1e-4, seed=0,
)

## Calibrate the FD curves

In [ ]:
params = {
    'k': 40,
    'A': drive,
    'd0': drive * 1.25,
    'a0': 0.164e-9,
    'R': 10e-9,
    'f0': 300e3,
    'Q': 150,
    'E_star': 1 / ((1 - 0.3**2) / (130e9) + (1 - 0.3**2) / (1.2e9)),
    'H': 7.1e-20,
}

conversions = {
    'k': 1,
    'A': 1e9,
    'd0': 1e9,
    'a0': 1e9,
    'R': 1e9,
    'f0': 1 / (2 * np.pi * params['f0']),   # t_hat = omega0 * t
    'Q': 1,
    'E_star': 1e-9 / params['k'],
    'H': 1 / (params['k'] * (1e-9)**2),
}

# ------------------------------------------------------------------
# Convert to normalized parameters
# ------------------------------------------------------------------
param_norm = {}
for key in params:
    param_norm[key] = params[key] * conversions[key]

C1 = param_norm['H'] * param_norm['R'] / 6.0
C2 = (4.0 / 3.0) * param_norm['E_star'] * np.sqrt(param_norm['R'])
C3 = param_norm['A'] / param_norm['Q']

# If driving at resonance:
omega_drive_hat = 1.0
omega_ref_hat = 1.0

# If you want an off-resonance drive at f_drive:
# f_drive = 70e3
# omega_drive_hat = f_drive / params['f0']
# omega_ref_hat   = omega_drive_hat   # or keep 1.0 depending on your demod choice

spm_params = NormalizedSPMParams(
    Q=param_norm['Q'],
    C1=C1,
    C2=C2,
    C3=C3,
    a0=param_norm['a0'],
    omega_drive_hat=omega_drive_hat,
    omega_ref_hat=omega_ref_hat,
)

In [ ]:
param_norm

In [ ]:
drives_si = np.array(drives)*factor1
measured_fd = {}


# Assume these are lists/arrays where index i corresponds to drives[i]
# z_arrays[i], A_arrays[i], phi_arrays[i]
# each could be a 1D FD curve, a scan line, or an image
measured_fd = {
    float(drive): {
        "d": height[i],
        "A": amp[i],
        "phi": phase[i],
    }
    for i, drive in enumerate(drives_si)
}

measured_fd

In [ ]:
spm_params

In [ ]:
param_norm

In [ ]:
def build_measured_curves(
    raw_data,
    conv_L: float,
    A0_method: str = 'far_field_median',
    far_field_frac: float = 0.1,
    far_field_side: str = 'start',  # 'start', 'end', 'largest_d', 'smallest_d', or 'auto'
    d_axis_mode: str = 'approach_extension',  # important
    A0_explicit=None,
    omega_drive_hat=None,
    omega_ref_hat=None,
):
    """
    d_axis_mode:
        'approach_extension':
            measured d starts at 0 in far field and increases as the tip approaches.
            true separation is modeled as d_true = d0 - d_meas.

        'separation_relative':
            measured d starts at arbitrary zero but larger d means farther away.
            true separation is modeled as d_true = d0 + d_meas.
    """

    def _iter_entries():
        if isinstance(raw_data, dict):
            for drive_nm, c in raw_data.items():
                yield (
                    drive_nm,
                    c['d'], c['A'], c['phi'],
                    c.get('omega_drive_hat', omega_drive_hat),
                    c.get('omega_ref_hat', omega_ref_hat),
                )
        else:
            for entry in raw_data:
                if len(entry) == 4:
                    drive_nm, h_nm, A_nm, phi_deg = entry
                    yield drive_nm, h_nm, A_nm, phi_deg, omega_drive_hat, omega_ref_hat
                elif len(entry) == 6:
                    yield tuple(entry)
                else:
                    raise ValueError("Tuple entries must have length 4 or 6.")

    out = []

    for drive_nm, h_in, A_in, phi_in, omd, omr in _iter_entries():
        h_nm = np.asarray(h_in, dtype=float).ravel()
        A_nm = np.asarray(A_in, dtype=float).ravel()
        phi_deg = np.asarray(phi_in, dtype=float).ravel()

        mask = np.isfinite(h_nm) & np.isfinite(A_nm) & np.isfinite(phi_deg)
        h_nm, A_nm, phi_deg = h_nm[mask], A_nm[mask], phi_deg[mask]

        if h_nm.size < 5:
            raise ValueError(f"drive {drive_nm}: <5 finite samples after NaN drop.")

        # Make measured coordinate relative.
        # For your case, h_nm[0] is usually 0 already, but this makes it robust.
        if d_axis_mode == 'approach_extension':
            d_rel_nm = h_nm - h_nm[0]
        elif d_axis_mode == 'separation_relative':
            d_rel_nm = h_nm - np.nanmin(h_nm)
        else:
            raise ValueError("d_axis_mode must be 'approach_extension' or 'separation_relative'.")

        # Estimate A0 from the actual far-field side.
        if A0_method == 'far_field_median':
            n_far = max(1, int(far_field_frac * h_nm.size))

            if far_field_side == 'start':
                idx_far = np.arange(n_far)
            elif far_field_side == 'end':
                idx_far = np.arange(h_nm.size - n_far, h_nm.size)
            elif far_field_side == 'largest_d':
                idx_far = np.argsort(h_nm)[::-1][:n_far]
            elif far_field_side == 'smallest_d':
                idx_far = np.argsort(h_nm)[:n_far]
            elif far_field_side == 'auto':
                idx_start = np.arange(n_far)
                idx_end = np.arange(h_nm.size - n_far, h_nm.size)
                # In tapping-mode approach curves, the far field usually has larger amplitude.
                if np.nanmedian(A_nm[idx_start]) >= np.nanmedian(A_nm[idx_end]):
                    idx_far = idx_start
                else:
                    idx_far = idx_end
            else:
                raise ValueError(
                    "far_field_side must be 'start', 'end', 'largest_d', "
                    "'smallest_d', or 'auto'."
                )

            A0_nm = float(np.nanmedian(A_nm[idx_far]))

        elif A0_method == 'drive':
            A0_nm = float(drive_nm)

        elif A0_method == 'explicit':
            if A0_explicit is None or float(drive_nm) not in A0_explicit:
                raise ValueError(f"A0_explicit missing entry for drive {drive_nm}")
            A0_nm = float(A0_explicit[float(drive_nm)])

        else:
            raise ValueError(f"Unknown A0_method '{A0_method}'")

        c = {
            'd_hat': d_rel_nm * float(conv_L),      # relative coordinate, not true separation
            'A_hat': A_nm * float(conv_L),
            'phi_deg': phi_deg,
            'A0_hat': A0_nm * float(conv_L),
            'drive_nm': float(drive_nm),
            'd_axis_mode': d_axis_mode,
            'far_field_side': far_field_side,
        }

        if omd is not None:
            c['omega_drive_hat'] = float(omd)
        if omr is not None:
            c['omega_ref_hat'] = float(omr)

        out.append(c)

    return out

def _curve_loss(
    model,
    d_meas: np.ndarray,
    A_meas: np.ndarray,
    phi_meas: np.ndarray,
    A0_hat: float,
    omega_drive_hat=None,
    omega_ref_hat=None,
    w_phi: float = 1.0,
    fit_offset: bool = True,
    offset_bounds=None,
    fixed_offset_hat: float = 0.0,
    d_axis_mode: str = 'approach_extension',
    d_grid_n: int = 128,
    d_grid_pad: float = 2.0,
    d_grid_lo_floor: float = 0.05,
):
    """
    Fits an unknown physical separation offset.

    For d_axis_mode='approach_extension':
        measured d is a relative piezo extension.
        d=0 is far field.
        d increases as the tip approaches.
        true separation = d0 - d_meas.

    For d_axis_mode='separation_relative':
        measured d is a relative separation-like coordinate.
        larger d means farther from the surface.
        true separation = d0 + d_meas.
    """

    d_meas_arr = np.asarray(d_meas, dtype=float)
    A_meas_arr = np.asarray(A_meas, dtype=float)
    phi_meas_arr = np.asarray(phi_meas, dtype=float)
    A0 = float(A0_hat)

    d_floor = float(d_grid_lo_floor) * A0
    d_pad = float(d_grid_pad) * A0

    d_min = float(np.min(d_meas_arr))
    d_max = float(np.max(d_meas_arr))

    if d_axis_mode == 'approach_extension':
        # offset is the true far-field separation at measured d=0.
        # The closest true separation is offset - d_max.
        if offset_bounds is None:
            closest_lo = d_floor
            closest_hi = 2.0 * A0
            offset_bounds = (d_max + closest_lo, d_max + closest_hi)

        off_lo, off_hi = map(float, offset_bounds)

        d_lo_grid = max(d_floor, off_lo - d_max)
        d_hi_grid = off_hi - d_min + d_pad

        def map_to_true_d(offset):
            return float(offset) - d_meas_arr

    elif d_axis_mode == 'separation_relative':
        # offset is the unknown true separation at measured d=0.
        if offset_bounds is None:
            offset_bounds = (d_floor, 2.0 * A0)

        off_lo, off_hi = map(float, offset_bounds)

        d_lo_grid = max(d_floor, d_min + off_lo)
        d_hi_grid = d_max + off_hi + d_pad

        def map_to_true_d(offset):
            return d_meas_arr + float(offset)

    else:
        raise ValueError(
            "d_axis_mode must be 'approach_extension' or 'separation_relative'."
        )

    d_grid_asc, A_grid, phi_grid = _simulate_on_d_grid(
        model,
        d_lo_grid,
        d_hi_grid,
        d_grid_n,
        omega_drive_hat,
        omega_ref_hat,
    )

    def _residual(offset):
        d_true = map_to_true_d(offset)

        A_pred = np.interp(
            d_true,
            d_grid_asc,
            A_grid,
            left=A_grid[0],
            right=A_grid[-1],
        )
        phi_pred = np.interp(
            d_true,
            d_grid_asc,
            phi_grid,
            left=phi_grid[0],
            right=phi_grid[-1],
        )

        res_A = (A_pred - A_meas_arr) / A0
        res_phi = (phi_pred - phi_meas_arr) / 180.0

        return float(np.mean(res_A**2) + float(w_phi) * np.mean(res_phi**2))

    if not fit_offset:
        return _residual(float(fixed_offset_hat)), float(fixed_offset_hat)

    try:
        from scipy.optimize import minimize_scalar

        result = minimize_scalar(
            _residual,
            bounds=(off_lo, off_hi),
            method='bounded',
            options={'xatol': 1e-4 * A0, 'maxiter': 100},
        )
        return float(result.fun), float(result.x)

    except Exception:
        scan = np.linspace(off_lo, off_hi, 51)
        losses = np.array([_residual(o) for o in scan])
        k = int(np.argmin(losses))
        return float(losses[k]), float(scan[k])

In [ ]:
measured_curves = build_measured_curves(
    measured_fd,
    conv_L=1,
    A0_method='far_field_median',
    far_field_frac=0.1,
    far_field_side='start',
    d_axis_mode='approach_extension',
    omega_drive_hat=1.0,
    omega_ref_hat=1.0,
)

grid1 = make_log_grid(
    multipliers=np.linspace(1, 9, 9),
    exponents=np.arange(-5, 0),
    tail=(),
)

grid2 = make_log_grid(
    multipliers=np.linspace(1, 9, 9),
    exponents=np.arange(-5, 0),
    tail=(),
)

loss_HE, H_fit, E_fit = grid_search_HE(
    measured_curves,
    spm_params,
    R_fixed=10,
    SPMRK45ApproachModel=SPMRK45ApproachModel,
    H_grid=grid1,
    E_star_grid=grid2,
    solver=RK45Options(),
    w_phi=1.0,
    n_jobs=8,
)

In [ ]:
import numpy as np
# from approach_curves import (
#     SPMRK45ApproachModel, RK45Options, NormalizedSPMParams,
# )
# from calibrate_grid import make_log_grid, build_measured_curves, grid_search_HE

# ---------------------------------------------------------------
# 1. measured_curves  <- your (drive, h, A, phi) tuples
# ---------------------------------------------------------------
# raw_data is what you already have:
#   raw_data = [(drive_nm_i, h_nm_i, A_nm_i, phi_deg_i), ...]
#
# conv_L converts physical (nm) into the simulator's normalized units.
# It's the same factor you use everywhere else for length-like params:
# conv_L = scanner.conversion['A']        # (or scanner.conversion['d0'])

measured_curves = build_measured_curves(
    measured_fd,
    conv_L=1,
    A0_method='far_field_median',   # or 'drive' if your drive setting is the calibrated A0
    far_field_frac=0.1,
    omega_drive_hat=1.0,            # global default; only needed if you drive off-resonance
    omega_ref_hat=1.0,
)

# measured_curves = build_measured_curves(
#     measured_fd,
#     conv_L=1,
#     A0_method='far_field_median',
#     far_field_frac=0.1,
#     far_field_side='start',          # because d starts at 0 in far field
#     d_axis_mode='approach_extension',
#     omega_drive_hat=1.0,
#     omega_ref_hat=1.0,
# )


# ---------------------------------------------------------------
# 2. base_params  <- your already-optimized NormalizedSPMParams
# ---------------------------------------------------------------
# This is the dataclass you feed to SPMRK45ApproachModel today.
# It must expose .C1, .C2, .C3, .Q, .a0, .omega_drive_hat, .omega_ref_hat.
# The grid search will overwrite C1 and C2 each step; everything else
# (Q, C3, a0, frequencies) stays at the values you've already tuned.

# base_params = NormalizedSPMParams(
#     Q=scanner.param_norm['Q'], C1=scanner.param_norm['C1'], 
#     C2=scanner.param_norm['C2'], C3=scanner.param_norm['C3'],
#     a0=0.164,
#     omega_drive_hat=1.0, omega_ref_hat=1.0,
# )

base_params = spm_params

# base_params = scanner.param_norm
# If you already have one in memory, just reuse it.

# ---------------------------------------------------------------
# 3. R_norm  <- tip radius in the SAME normalized units as C1, C2
# ---------------------------------------------------------------
# scanner_fd.py defines  param_norm['R'] = param['R'] * conversion['R']
# so the cleanest source is:
R_norm = 10

# or equivalently:
# R_norm = R_nm * scanner.conversion['R']

# ---------------------------------------------------------------
# 4. grid (your snippet, verbatim)
# ---------------------------------------------------------------
multipliers = np.linspace(1, 9, num=9)
exponents1   = np.arange(-5, 0)         # -6, -5, -4, -3
grid1 = np.concatenate([multipliers * 10.0**e for e in exponents1])

exponents2   = np.arange(-5, 0)         # -6, -5, -4, -3
grid2 = np.concatenate([multipliers * 10.0**e for e in exponents2])

# ---------------------------------------------------------------
# 5. The call
# ---------------------------------------------------------------
loss_HE, H_fit, E_fit = grid_search_HE(
    measured_curves,
    base_params,
    R_fixed=R_norm,
    SPMRK45ApproachModel=SPMRK45ApproachModel,
    H_grid=grid1,
    E_star_grid=grid2,
    solver=RK45Options(),               # use your existing solver settings
    w_phi=1.0,
    n_jobs=8,
)

In [ ]:
H_ref, E_ref = parabolic_refine_2d(loss_HE, grid1, grid2)
plot_loss_grid(loss_HE, grid1, grid2, x_label='H', y_label='E*',
               title='stage 1 loss')

print(H_ref, E_ref)

In [ ]:
params = {'A': drive_nm, 'd0': d_nm.max()*0.8, 'R': 10, 'H': 1e-19,
          'E_star': 1e9, 'Q': 200}

In [ ]:
# ----- stage 1 -----
# measured_curves: list of dicts in NORMALIZED units
loss_HE, H_fit, E_fit = grid_search_HE(
    measured_curves, base_params, R_fixed=R_norm,
    SPMRK45ApproachModel=SPMRK45ApproachModel,
    H_grid=grid, E_star_grid=grid,
    solver=RK45Options(),
    w_phi=1.0, n_jobs=4,
)
H_ref, E_ref = parabolic_refine_2d(loss_HE, grid, grid)
plot_loss_grid(loss_HE, grid, grid, x_label='H', y_label='E*',
               title='stage 1 loss')

In [ ]:
# Let's fit the materials properties by solving the ODE

# =============================================================================
# Example wiring (commented; adapt to your data structures)
# =============================================================================
#
# import numpy as np
# from approach_curves import (SPMRK45ApproachModel, ApproachCurveLibrary,
#                              make_normalized_fd_lookup, RK45Options,
#                              NormalizedSPMParams)
# from scanner_fd import ScannerFD
# from calibrate_grid import (make_log_grid, grid_search_HE,
#                             build_library_from_fit, grid_search_PI_multi,
#                             parabolic_refine_2d, plot_loss_grid)
#
# # ----- shared grid -----
# multipliers = np.linspace(1, 9, 9)
# exponents   = np.arange(-6, -2)             # -6, -5, -4, -3
# grid        = np.concatenate([multipliers * 10.0**e for e in exponents]
#                              + [[10.0]])    # 37 points
#
# # ----- stage 1 -----
# # measured_curves: list of dicts in NORMALIZED units
# loss_HE, H_fit, E_fit = grid_search_HE(
#     measured_curves, base_params, R_fixed=R_norm,
#     SPMRK45ApproachModel=SPMRK45ApproachModel,
#     H_grid=grid, E_star_grid=grid,
#     solver=RK45Options(),
#     w_phi=1.0, n_jobs=4,
# )
# H_ref, E_ref = parabolic_refine_2d(loss_HE, grid, grid)
# plot_loss_grid(loss_HE, grid, grid, x_label='H', y_label='E*',
#                title='stage 1 loss')
#
# # ----- stage 2 -----
# library = build_library_from_fit(
#     measured_curves, base_params, R_fixed=R_norm,
#     H_fit=H_ref, E_star_fit=E_ref,
#     SPMRK45ApproachModel=SPMRK45ApproachModel,
#     ApproachCurveLibrary=ApproachCurveLibrary,
#     n_points=512, conv_L=scanner.conversion['A'],
#     smooth_method='savgol', smooth_window=7, smooth_polyorder=3,
# )
# fd_lookup = make_normalized_fd_lookup(library, conv_L=scanner.conversion['A'])
#
# # ----- stage 3 -----
# # build a ScannerFD bound to the library at the experimental drive
# scanner = ScannerFD(params, conversions, fd_model=fd_lookup(drive_nm))
#
# line_data = [
#     {'h_hat': h_hat_slow, 'd_meas': d_slow, 'A_meas': A_slow,
#      'line_kwargs': {'scan_speed_hat': v_slow, 'dx_hat': dx,
#                      'A_filter_order': 1, 'tau_A': tau_A_init,
#                      'z_filter_order': 1, 'tau_z': tau_z_init,
#                      'setpoint': 0.8, 'T_I': 0.0}},
#     {'h_hat': h_hat_fast, 'd_meas': d_fast, 'A_meas': A_fast,
#      'line_kwargs': {'scan_speed_hat': v_fast, 'dx_hat': dx,
#                      'A_filter_order': 1, 'tau_A': tau_A_init,
#                      'z_filter_order': 1, 'tau_z': tau_z_init,
#                      'setpoint': 0.8, 'T_I': 0.0}},
# ]
# loss_PI, P_fit, I_fit = grid_search_PI_multi(
#     scanner, line_data, P_grid=grid, I_grid=grid,
#     w_A=0.3, n_jobs=4,
# )
# P_ref, I_ref = parabolic_refine_2d(loss_PI, grid, grid)
# plot_loss_grid(loss_PI, grid, grid, x_label='P', y_label='I',
#                title='stage 3 loss')


In [ ]:
# # ============================================================
# # 1) CLEAN RK45 APPROACH SOLVER (NO SOLVER-SIDE NOISE)
# # ============================================================
# @dataclass
# class NormalizedSPMParams:
#     """Dimensionless / normalized SPM parameters."""
#     Q: float
#     C1: float
#     C2: float
#     C3: float
#     a0: float
#     omega_drive_hat: float = 1.0
#     omega_ref_hat: float = 1.0
#     attrs: Dict[str, Any] = field(default_factory=dict)


# @dataclass
# class RK45Options:
#     """Controls for the RK45 steady-state solver."""
#     N_cycles: int = 20
#     max_windows: int = 40
#     settle_windows: int = 1
#     tol_A: float = 1e-4
#     tol_phi_deg: float = 1e-2
#     consecutive: int = 2
#     demod_per_ref_cycle: int = 40
#     rtol: float = 1e-6
#     atol: float = 1e-9
#     dt0: float = 2e-2
#     dt_min: float = 1e-10
#     dt_max: float = 3e-1
#     max_steps_per_window: int = 200000


# @njit(cache=True)
# def F_DMT_hat(delta_hat, C1, C2, a0):
#     if delta_hat > 0.0:
#         return -C1 / (delta_hat + a0) ** 2
#     else:
#         return -C1 / a0 ** 2 + C2 * (a0 - delta_hat) ** 1.5


# @njit(cache=True)
# def rhs_hat(t_hat, z_hat, v_hat, d_hat, Q, C1, C2, C3, a0, omega_drive_hat):
#     delta_hat = z_hat + d_hat
#     F_contact = F_DMT_hat(delta_hat, C1, C2, a0)
#     F_drive = -C3 * np.cos(omega_drive_hat * t_hat)
#     dz = v_hat
#     dv = F_contact + F_drive - v_hat / Q - z_hat
#     return dz, dv


# @njit(cache=True)
# def rk45_step_dopri(t, z, v, dt, d_hat, Q, C1, C2, C3, a0, omega_drive_hat, rtol, atol):
#     c2 = 1.0 / 5.0
#     c3 = 3.0 / 10.0
#     c4 = 4.0 / 5.0
#     c5 = 8.0 / 9.0
#     c6 = 1.0

#     a21 = 1.0 / 5.0
#     a31 = 3.0 / 40.0
#     a32 = 9.0 / 40.0
#     a41 = 44.0 / 45.0
#     a42 = -56.0 / 15.0
#     a43 = 32.0 / 9.0
#     a51 = 19372.0 / 6561.0
#     a52 = -25360.0 / 2187.0
#     a53 = 64448.0 / 6561.0
#     a54 = -212.0 / 729.0
#     a61 = 9017.0 / 3168.0
#     a62 = -355.0 / 33.0
#     a63 = 46732.0 / 5247.0
#     a64 = 49.0 / 176.0
#     a65 = -5103.0 / 18656.0

#     b1 = 35.0 / 384.0
#     b3 = 500.0 / 1113.0
#     b4 = 125.0 / 192.0
#     b5 = -2187.0 / 6784.0
#     b6 = 11.0 / 84.0

#     b1s = 5179.0 / 57600.0
#     b3s = 7571.0 / 16695.0
#     b4s = 393.0 / 640.0
#     b5s = -92097.0 / 339200.0
#     b6s = 187.0 / 2100.0
#     b7s = 1.0 / 40.0

#     dz1, dv1 = rhs_hat(t, z, v, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)
#     z2 = z + dt * a21 * dz1
#     v2 = v + dt * a21 * dv1
#     dz2, dv2 = rhs_hat(t + c2 * dt, z2, v2, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

#     z3 = z + dt * (a31 * dz1 + a32 * dz2)
#     v3 = v + dt * (a31 * dv1 + a32 * dv2)
#     dz3, dv3 = rhs_hat(t + c3 * dt, z3, v3, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

#     z4 = z + dt * (a41 * dz1 + a42 * dz2 + a43 * dz3)
#     v4 = v + dt * (a41 * dv1 + a42 * dv2 + a43 * dv3)
#     dz4, dv4 = rhs_hat(t + c4 * dt, z4, v4, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

#     z5i = z + dt * (a51 * dz1 + a52 * dz2 + a53 * dz3 + a54 * dz4)
#     v5i = v + dt * (a51 * dv1 + a52 * dv2 + a53 * dv3 + a54 * dv4)
#     dz5, dv5 = rhs_hat(t + c5 * dt, z5i, v5i, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

#     z6 = z + dt * (a61 * dz1 + a62 * dz2 + a63 * dz3 + a64 * dz4 + a65 * dz5)
#     v6 = v + dt * (a61 * dv1 + a62 * dv2 + a63 * dv3 + a64 * dv4 + a65 * dv5)
#     dz6, dv6 = rhs_hat(t + c6 * dt, z6, v6, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

#     z5 = z + dt * (b1 * dz1 + b3 * dz3 + b4 * dz4 + b5 * dz5 + b6 * dz6)
#     v5 = v + dt * (b1 * dv1 + b3 * dv3 + b4 * dv4 + b5 * dv5 + b6 * dv6)

#     dz7, dv7 = rhs_hat(t + dt, z5, v5, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

#     z4s = z + dt * (b1s * dz1 + b3s * dz3 + b4s * dz4 + b5s * dz5 + b6s * dz6 + b7s * dz7)
#     v4s = v + dt * (b1s * dv1 + b3s * dv3 + b4s * dv4 + b5s * dv5 + b6s * dv6 + b7s * dv7)

#     ez = z5 - z4s
#     ev = v5 - v4s
#     sz = atol + rtol * max(abs(z), abs(z5))
#     sv = atol + rtol * max(abs(v), abs(v5))
#     err = np.sqrt(0.5 * ((ez / sz) ** 2 + (ev / sv) ** 2))
#     return z5, v5, err


# @njit(cache=True)
# def demod_add_segment(X, Y, z0, z1, t0, t1, omega_ref_hat):
#     c0 = np.cos(omega_ref_hat * t0)
#     s0 = np.sin(omega_ref_hat * t0)
#     c1 = np.cos(omega_ref_hat * t1)
#     s1 = np.sin(omega_ref_hat * t1)
#     dt = t1 - t0
#     X += 0.5 * (z0 * c0 + z1 * c1) * dt
#     Y += 0.5 * (z0 * s0 + z1 * s1) * dt
#     return X, Y


# @njit(cache=True)
# def run_one_window_rk45(
#     t_start, z_start, v_start, d_hat, Q, C1, C2, C3, a0,
#     omega_drive_hat, omega_ref_hat,
#     T_win, demod_dt_hat, rtol, atol, dt0, dt_min, dt_max, max_steps,
# ):
#     t = t_start
#     z = z_start
#     v = v_start
#     t_end = t_start + T_win

#     X = 0.0
#     Y = 0.0
#     if demod_dt_hat <= 0.0:
#         demod_dt_hat = T_win

#     t_s = t_start
#     z_s = z_start
#     next_s = t_s + demod_dt_hat

#     F0 = F_DMT_hat(z + d_hat, C1, C2, a0)
#     F_int = 0.0
#     F_min = F0
#     F_max = F0

#     safety = 0.9
#     fac_min = 0.2
#     fac_max = 5.0
#     p = 5.0
#     dt = dt0
#     steps = 0

#     while t < t_end and steps < max_steps:
#         if dt < dt_min:
#             dt = dt_min
#         if dt > dt_max:
#             dt = dt_max
#         if t + dt > t_end:
#             dt = t_end - t

#         t_prev = t
#         z_prev = z
#         F_prev = F_DMT_hat(z_prev + d_hat, C1, C2, a0)

#         z_new, v_new, err = rk45_step_dopri(
#             t, z, v, dt, d_hat, Q, C1, C2, C3, a0, omega_drive_hat, rtol, atol
#         )
#         accept = (err <= 1.0) or (dt <= dt_min * 1.0001)
#         if accept:
#             t = t + dt
#             z = z_new
#             v = v_new

#             F_cur = F_DMT_hat(z + d_hat, C1, C2, a0)
#             F_int += 0.5 * (F_prev + F_cur) * (t - t_prev)
#             if F_cur < F_min:
#                 F_min = F_cur
#             if F_cur > F_max:
#                 F_max = F_cur

#             while next_s <= t:
#                 denom = t - t_prev
#                 if denom <= 0.0:
#                     z_b = z
#                 else:
#                     frac = (next_s - t_prev) / denom
#                     if frac < 0.0:
#                         frac = 0.0
#                     elif frac > 1.0:
#                         frac = 1.0
#                     z_b = z_prev + frac * (z - z_prev)
#                 X, Y = demod_add_segment(X, Y, z_s, z_b, t_s, next_s, omega_ref_hat)
#                 t_s = next_s
#                 z_s = z_b
#                 next_s = next_s + demod_dt_hat
#             steps += 1

#         if err == 0.0:
#             fac = fac_max
#         else:
#             fac = safety * (1.0 / err) ** (1.0 / p)
#             if fac < fac_min:
#                 fac = fac_min
#             elif fac > fac_max:
#                 fac = fac_max
#         dt = dt * fac

#     if t_end > t_s:
#         X, Y = demod_add_segment(X, Y, z_s, z, t_s, t_end, omega_ref_hat)

#     F_mean = F_int / T_win
#     return X, Y, F_mean, F_min, F_max, steps, z, v


# @njit(cache=True)
# def simulate_steady_state_rk45(
#     z0_hat, v0_hat, d_hat, Q, C1, C2, C3, a0, omega_drive_hat, omega_ref_hat,
#     N_cycles, max_windows, settle_windows, tol_A, tol_phi_deg, consecutive,
#     demod_per_ref_cycle, rtol, atol, dt0, dt_min, dt_max, max_steps_per_window,
# ):
#     T_ref = 2.0 * np.pi / omega_ref_hat
#     T_win = N_cycles * T_ref
#     if demod_per_ref_cycle < 5:
#         demod_per_ref_cycle = 5
#     demod_dt_hat = T_ref / demod_per_ref_cycle

#     t = 0.0
#     z = z0_hat
#     v = v0_hat
#     last_A = 1e30
#     last_phi = 1e30
#     stable = 0
#     A_last = 0.0
#     phi_last = 0.0
#     F_mean_last = 0.0
#     F_min_last = 0.0
#     F_max_last = 0.0

#     for widx in range(max_windows):
#         X, Y, F_mean, F_min, F_max, _, z, v = run_one_window_rk45(
#             t, z, v, d_hat, Q, C1, C2, C3, a0,
#             omega_drive_hat, omega_ref_hat,
#             T_win, demod_dt_hat, rtol, atol, dt0, dt_min, dt_max, max_steps_per_window,
#         )
#         t = t + T_win
#         A = (2.0 / T_win) * np.sqrt(X * X + Y * Y)
#         phi = np.degrees(np.arctan2(Y, X))
#         if phi < 0.0:
#             phi += 180.0
#         if phi > 180.0:
#             phi -= 180.0

#         A_last = A
#         phi_last = phi
#         F_mean_last = F_mean
#         F_min_last = F_min
#         F_max_last = F_max

#         if widx >= settle_windows:
#             dA = abs(A - last_A)
#             dphi = abs(phi - last_phi)
#             if dA < tol_A and dphi < tol_phi_deg:
#                 stable += 1
#             else:
#                 stable = 0
#             if stable >= consecutive:
#                 return A, phi, F_mean, F_min, F_max, (widx + 1), z, v

#         last_A = A
#         last_phi = phi

#     return A_last, phi_last, F_mean_last, F_min_last, F_max_last, max_windows, z, v


# class SPMRK45ApproachModel:
#     """Clean approach-curve simulator with no solver-side force noise."""

#     def __init__(self, params: NormalizedSPMParams, solver: Optional[RK45Options] = None):
#         self.params = params
#         self.solver = solver if solver is not None else RK45Options()

#     def simulate_at_height(self, d_hat, z0_hat=0.0, v0_hat=0.0, omega_drive_hat=None, omega_ref_hat=None):
#         if omega_drive_hat is None:
#             omega_drive_hat = self.params.omega_drive_hat
#         if omega_ref_hat is None:
#             omega_ref_hat = self.params.omega_ref_hat
#         so = self.solver
#         p = self.params
#         A, phi, F_mean, F_min, F_max, windows_used, z_final, v_final = simulate_steady_state_rk45(
#             z0_hat=float(z0_hat),
#             v0_hat=float(v0_hat),
#             d_hat=float(d_hat),
#             Q=float(p.Q), C1=float(p.C1), C2=float(p.C2), C3=float(p.C3), a0=float(p.a0),
#             omega_drive_hat=float(omega_drive_hat), omega_ref_hat=float(omega_ref_hat),
#             N_cycles=int(so.N_cycles), max_windows=int(so.max_windows), settle_windows=int(so.settle_windows),
#             tol_A=float(so.tol_A), tol_phi_deg=float(so.tol_phi_deg), consecutive=int(so.consecutive),
#             demod_per_ref_cycle=int(so.demod_per_ref_cycle),
#             rtol=float(so.rtol), atol=float(so.atol), dt0=float(so.dt0), dt_min=float(so.dt_min), dt_max=float(so.dt_max),
#             max_steps_per_window=int(so.max_steps_per_window),
#         )
#         return {
#             "d_hat": float(d_hat),
#             "A_hat": float(A),
#             "phi_deg": float(phi),
#             "F_mean_hat": float(F_mean),
#             "F_min_hat": float(F_min),
#             "F_max_hat": float(F_max),
#             "windows_used": int(windows_used),
#             "z_final_hat": float(z_final),
#             "v_final_hat": float(v_final),
#         }

#     def approach_curve(self, d_start_hat, d_end_hat, n_points=256, z0_hat=0.0, v0_hat=0.0, warm_start=True, omega_drive_hat=None, omega_ref_hat=None):
#         d_vals = np.linspace(float(d_start_hat), float(d_end_hat), int(n_points))
#         A_vals = np.empty_like(d_vals)
#         phi_vals = np.empty_like(d_vals)
#         F_mean_vals = np.empty_like(d_vals)
#         F_min_vals = np.empty_like(d_vals)
#         F_max_vals = np.empty_like(d_vals)
#         windows_vals = np.empty(d_vals.size, dtype=np.int64)
#         z_cur = float(z0_hat)
#         v_cur = float(v0_hat)

#         for i, d_hat in enumerate(d_vals):
#             out = self.simulate_at_height(d_hat=float(d_hat), z0_hat=z_cur, v0_hat=v_cur, omega_drive_hat=omega_drive_hat, omega_ref_hat=omega_ref_hat)
#             A_vals[i] = out["A_hat"]
#             phi_vals[i] = out["phi_deg"]
#             F_mean_vals[i] = out["F_mean_hat"]
#             F_min_vals[i] = out["F_min_hat"]
#             F_max_vals[i] = out["F_max_hat"]
#             windows_vals[i] = out["windows_used"]
#             if warm_start:
#                 z_cur = out["z_final_hat"]
#                 v_cur = out["v_final_hat"]

#         return {
#             "d_hat": d_vals,
#             "A_hat": A_vals,
#             "phi_deg": phi_vals,
#             "F_mean_hat": F_mean_vals,
#             "F_min_hat": F_min_vals,
#             "F_max_hat": F_max_vals,
#             "windows_used": windows_vals,
#         }


# # ============================================================
# # 2) PARAMETER / LIBRARY BUILDERS
# # ============================================================
# def normalize_params(params: Dict[str, float], conversions: Dict[str, float]) -> Dict[str, float]:
#     return {k: float(params[k] * conversions[k]) for k in params}


# def build_normalized_spm_params(
#     params: Dict[str, float],
#     conversions: Dict[str, float],
#     drive_si: float,
#     omega_drive_hat: float = 1.0,
#     omega_ref_hat: float = 1.0,
#     attrs: Optional[Dict[str, Any]] = None,
# ) -> Tuple[NormalizedSPMParams, Dict[str, float]]:
#     params_local = dict(params)
#     params_local["A"] = float(drive_si)
#     param_norm = normalize_params(params_local, conversions)
#     C1 = param_norm['H'] * param_norm['R'] / 6.0
#     C2 = (4.0 / 3.0) * param_norm['E_star'] * np.sqrt(param_norm['R'])
#     C3 = param_norm['A'] / param_norm['Q']
#     meta = dict(attrs or {})
#     meta.update({
#         "drive_si": float(drive_si),
#         "drive_hat": float(param_norm['A']),
#         "omega_drive_hat": float(omega_drive_hat),
#         "omega_ref_hat": float(omega_ref_hat),
#     })
#     spm_params = NormalizedSPMParams(
#         Q=float(param_norm['Q']), C1=float(C1), C2=float(C2), C3=float(C3), a0=float(param_norm['a0']),
#         omega_drive_hat=float(omega_drive_hat), omega_ref_hat=float(omega_ref_hat), attrs=meta,
#     )
#     return spm_params, param_norm


# class FDCurve1D:
#     """1D lookup model built from one precomputed approach curve."""

#     def __init__(self, d_hat, A_hat, phi_deg, F_max_hat, F_mean_hat=None, clip=True, attrs: Optional[Dict[str, Any]] = None):
#         d_hat = np.asarray(d_hat, dtype=float)
#         A_hat = np.asarray(A_hat, dtype=float)
#         phi_deg = np.asarray(phi_deg, dtype=float)
#         F_max_hat = np.asarray(F_max_hat, dtype=float)
#         if F_mean_hat is None:
#             F_mean_hat = F_max_hat.copy()
#         else:
#             F_mean_hat = np.asarray(F_mean_hat, dtype=float)

#         if not (d_hat.ndim == A_hat.ndim == phi_deg.ndim == F_max_hat.ndim == F_mean_hat.ndim == 1):
#             raise ValueError("All inputs must be 1D arrays.")
#         if not (d_hat.size == A_hat.size == phi_deg.size == F_max_hat.size == F_mean_hat.size):
#             raise ValueError("All arrays must have the same length.")
#         if d_hat.size < 2:
#             raise ValueError("Need at least two FD points.")
#         order = np.argsort(d_hat)
#         self.d_hat = d_hat[order]
#         self.A_hat = A_hat[order]
#         self.phi_deg = phi_deg[order]
#         self.F_max_hat = F_max_hat[order]
#         self.F_mean_hat = F_mean_hat[order]
#         self.clip = bool(clip)
#         self.attrs = dict(attrs or {})

#     def _prep_query(self, d_query):
#         d_query = np.asarray(d_query, dtype=float)
#         if self.clip:
#             d_query = np.clip(d_query, self.d_hat[0], self.d_hat[-1])
#         return d_query

#     def evaluate(self, d_query):
#         d_query = self._prep_query(d_query)
#         A = np.interp(d_query, self.d_hat, self.A_hat)
#         phi = np.interp(d_query, self.d_hat, self.phi_deg)
#         F = np.interp(d_query, self.d_hat, self.F_max_hat)
#         return A, phi, F

#     def free_amplitude(self):
#         return float(self.A_hat[-1])


# class FDLibrary2D:
#     """
#     Multi-drive FD lookup library.

#     Uses bilinear interpolation in (drive_hat, d_hat) for amplitude, phase, and force.
#     """

#     def __init__(self, drive_hat_grid, d_hat_grid, A_table, phi_table, F_max_table, F_mean_table=None, clip=True, attrs: Optional[Dict[str, Any]] = None):
#         drive_hat_grid = np.asarray(drive_hat_grid, dtype=float)
#         d_hat_grid = np.asarray(d_hat_grid, dtype=float)
#         A_table = np.asarray(A_table, dtype=float)
#         phi_table = np.asarray(phi_table, dtype=float)
#         F_max_table = np.asarray(F_max_table, dtype=float)
    
#         if F_mean_table is None:
#             F_mean_table = F_max_table.copy()
#         else:
#             F_mean_table = np.asarray(F_mean_table, dtype=float)
    
#         if drive_hat_grid.ndim != 1 or d_hat_grid.ndim != 1:
#             raise ValueError("drive_hat_grid and d_hat_grid must be 1D.")
    
#         expected = (drive_hat_grid.size, d_hat_grid.size)
#         for arr in (A_table, phi_table, F_max_table, F_mean_table):
#             if arr.shape != expected:
#                 raise ValueError(f"Table shape must be {expected}, got {arr.shape}.")
    
#         # sort drive axis
#         drive_order = np.argsort(drive_hat_grid)
#         drive_hat_grid = drive_hat_grid[drive_order]
#         A_table = A_table[drive_order]
#         phi_table = phi_table[drive_order]
#         F_max_table = F_max_table[drive_order]
#         F_mean_table = F_mean_table[drive_order]
    
#         # sort d axis
#         d_order = np.argsort(d_hat_grid)
#         d_hat_grid = d_hat_grid[d_order]
#         A_table = A_table[:, d_order]
#         phi_table = phi_table[:, d_order]
#         F_max_table = F_max_table[:, d_order]
#         F_mean_table = F_mean_table[:, d_order]
    
#         self.drive_hat_grid = drive_hat_grid
#         self.d_hat_grid = d_hat_grid
#         self.A_table = A_table
#         self.phi_table = phi_table
#         self.F_max_table = F_max_table
#         self.F_mean_table = F_mean_table
#         self.clip = bool(clip)
#         self.attrs = dict(attrs or {})

#     @classmethod
#     def from_curves(cls, drive_hat_grid, curves: Sequence[Dict[str, np.ndarray]], clip=True, attrs: Optional[Dict[str, Any]] = None):
#         d0 = np.asarray(curves[0]["d_hat"], dtype=float)
#         order0 = np.argsort(d0)
#         d0_sorted = d0[order0]
    
#         A_list, phi_list, Fmax_list, Fmean_list = [], [], [], []
    
#         for curve in curves:
#             d = np.asarray(curve["d_hat"], dtype=float)
#             order = np.argsort(d)
#             d_sorted = d[order]
    
#             if d_sorted.shape != d0_sorted.shape or not np.allclose(d_sorted, d0_sorted):
#                 raise ValueError("All curves must share the same d_hat grid after sorting.")
    
#             A_list.append(np.asarray(curve["A_hat"], dtype=float)[order])
#             phi_list.append(np.asarray(curve["phi_deg"], dtype=float)[order])
#             Fmax_list.append(np.asarray(curve["F_max_hat"], dtype=float)[order])
    
#             if "F_mean_hat" in curve:
#                 Fmean_list.append(np.asarray(curve["F_mean_hat"], dtype=float)[order])
#             else:
#                 Fmean_list.append(np.asarray(curve["F_max_hat"], dtype=float)[order])
    
#         return cls(
#             drive_hat_grid=np.asarray(drive_hat_grid, dtype=float),
#             d_hat_grid=d0_sorted,
#             A_table=np.vstack(A_list),
#             phi_table=np.vstack(phi_list),
#             F_max_table=np.vstack(Fmax_list),
#             F_mean_table=np.vstack(Fmean_list),
#             clip=clip,
#             attrs=attrs,
#         )

#     def _prep_query(self, d_query, drive_query):
#         d_query = np.asarray(d_query, dtype=float)
#         drive_query = np.asarray(drive_query, dtype=float)
#         if self.clip:
#             d_query = np.clip(d_query, self.d_hat_grid[0], self.d_hat_grid[-1])
#             drive_query = np.clip(drive_query, self.drive_hat_grid[0], self.drive_hat_grid[-1])
#         return d_query, drive_query

#     def _interp_drive_table(self, table, d_query, drive_hat):
#         vals_per_drive = np.array([np.interp(d_query, self.d_hat_grid, row) for row in table], dtype=float)
#         return float(np.interp(drive_hat, self.drive_hat_grid, vals_per_drive))

#     def evaluate(self, d_query, drive_hat):
#         d_query, drive_hat = self._prep_query(d_query, drive_hat)
#         if np.ndim(d_query) == 0 and np.ndim(drive_hat) == 0:
#             A = self._interp_drive_table(self.A_table, float(d_query), float(drive_hat))
#             phi = self._interp_drive_table(self.phi_table, float(d_query), float(drive_hat))
#             F = self._interp_drive_table(self.F_max_table, float(d_query), float(drive_hat))
#             return float(A), float(phi), float(F)
#         d_query = np.broadcast_to(d_query, np.shape(np.broadcast(d_query, drive_hat)))
#         drive_hat = np.broadcast_to(drive_hat, d_query.shape)
#         A = np.empty_like(d_query, dtype=float)
#         phi = np.empty_like(d_query, dtype=float)
#         F = np.empty_like(d_query, dtype=float)
#         it = np.nditer([d_query, drive_hat, A, phi, F], flags=['multi_index'], op_flags=[['readonly']]*2 + [['writeonly']]*3)
#         for d_i, dr_i, Ao, phio, Fo in it:
#             Ao[...] = self._interp_drive_table(self.A_table, float(d_i), float(dr_i))
#             phio[...] = self._interp_drive_table(self.phi_table, float(d_i), float(dr_i))
#             Fo[...] = self._interp_drive_table(self.F_max_table, float(d_i), float(dr_i))
#         return A, phi, F

#     def free_amplitude(self, drive_hat):
#         far_idx = np.argmax(self.d_hat_grid)
#         return float(np.interp(float(drive_hat), self.drive_hat_grid, self.A_table[:, far_idx]))

#     def nearest_drive_index(self, drive_hat):
#         return int(np.argmin(np.abs(self.drive_hat_grid - float(drive_hat))))


# @dataclass
# class FDLibraryBuildResult:
#     library: FDLibrary2D
#     curves_by_drive: Dict[float, Dict[str, np.ndarray]]
#     attrs: Dict[str, Any] = field(default_factory=dict)


# def build_fd_library_from_drives(
#     params: Dict[str, float],
#     conversions: Dict[str, float],
#     drive_si_values: Sequence[float],
#     solver: Optional[RK45Options] = None,
#     d_start_hat: Optional[float] = None,
#     d_end_hat: float = -25.0,
#     n_points: int = 256,
#     omega_drive_hat: float = 1.0,
#     omega_ref_hat: float = 1.0,
#     warm_start: bool = True,
#     clip: bool = True,
#     extra_attrs: Optional[Dict[str, Any]] = None,
# ) -> FDLibraryBuildResult:
#     drive_si_values = [float(v) for v in drive_si_values]
#     curves = []
#     drive_hat_grid = []
#     curves_by_drive: Dict[float, Dict[str, np.ndarray]] = {}
#     for drive_si in drive_si_values:
#         spm_params, param_norm = build_normalized_spm_params(
#             params=params, conversions=conversions, drive_si=drive_si,
#             omega_drive_hat=omega_drive_hat, omega_ref_hat=omega_ref_hat,
#             attrs={"source": "build_fd_library_from_drives"},
#         )
#         model = SPMRK45ApproachModel(spm_params, solver=solver)
#         d0 = float(param_norm['d0']) if d_start_hat is None else float(d_start_hat)

#         curve_raw = model.approach_curve(
#             d_start_hat=d0,
#             d_end_hat=float(d_end_hat),
#             n_points=int(n_points),
#             z0_hat=0.0,
#             v0_hat=0.0,
#             warm_start=warm_start,
#         )
        
#         order = np.argsort(curve_raw["d_hat"])
#         curve = {k: (np.asarray(v)[order] if isinstance(v, np.ndarray) and np.asarray(v).shape == np.asarray(curve_raw["d_hat"]).shape else v)
#                  for k, v in curve_raw.items()}
        
#         curves.append(curve)
#         drive_hat_grid.append(float(param_norm['A']))
#         curves_by_drive[drive_si] = curve
        
#         # curve = model.approach_curve(d_start_hat=d0, d_end_hat=float(d_end_hat), n_points=int(n_points), z0_hat=0.0, v0_hat=0.0, warm_start=warm_start)
#         # curves.append(curve)
#         # drive_hat_grid.append(float(param_norm['A']))
#         # curves_by_drive[drive_si] = curve

    

#     attrs = dict(extra_attrs or {})
#     attrs.update({
#         "drive_si_values": drive_si_values,
#         "d_end_hat": float(d_end_hat),
#         "n_points": int(n_points),
#         "solver": asdict(solver if solver is not None else RK45Options()),
#     })
#     library = FDLibrary2D.from_curves(drive_hat_grid=drive_hat_grid, curves=curves, clip=clip, attrs=attrs)
#     return FDLibraryBuildResult(library=library, curves_by_drive=curves_by_drive, attrs=attrs)


# # ============================================================
# # 3) PI-SIDE DRIFT + MULTI-DRIVE SCANNER
# # ============================================================
# @dataclass
# class PIDriftNoiseOptions:
#     drift_rel_std: float = 2e-3
#     drift_tau_pixels: float = 200.0
#     white_rel_std: float = 0.0
#     sinusoid_rel_amp: float = 0.0
#     sinusoid_period_pixels: float = 512.0
#     seed: int = 0


# def _lpf1_step(y, x, dt, tau):
#     if tau is None or tau <= 0.0:
#         return x
#     alpha = dt / (tau + dt)
#     return y + alpha * (x - y)


# def _lpf2_step(y, ydot, x, dt, wn, zeta):
#     if wn is None or wn <= 0.0:
#         return x, 0.0
#     ydd = (wn * wn) * (x - y) - (2.0 * zeta * wn) * ydot
#     ydot_new = ydot + dt * ydd
#     y_new = y + dt * ydot_new
#     return y_new, ydot_new


# def _ou_step_stationary(x, std, tau, dt, rng):
#     if std <= 0.0:
#         return 0.0
#     if tau is None or tau <= 0.0:
#         return std * rng.standard_normal()
#     alpha = np.exp(-dt / tau)
#     sigma = std * np.sqrt(max(1.0 - alpha * alpha, 0.0))
#     return alpha * x + sigma * rng.standard_normal()


# class ScannerFD2D:
#     """
#     Multi-drive scanner using a (drive_hat, d_hat) FD library.

#     PI / LPF dynamics are simulated online. Optional low-frequency drift/noise is
#     injected into the amplitude channel seen by the PI controller.
#     """

#     def __init__(self, params: Dict[str, float], conversions: Dict[str, float], fd_library: FDLibrary2D, pi_noise: Optional[PIDriftNoiseOptions] = None, attrs: Optional[Dict[str, Any]] = None):
#         self.param = dict(params)
#         self.conversion = dict(conversions)
#         self.fd_library = fd_library
#         self.pi_noise = pi_noise if pi_noise is not None else PIDriftNoiseOptions()
#         self.rng = np.random.default_rng(self.pi_noise.seed)
#         self.param_norm = normalize_params(self.param, self.conversion)
#         self.param_norm['C1'] = self.param_norm['H'] * self.param_norm['R'] / 6.0
#         self.param_norm['C2'] = 4.0 / 3.0 * self.param_norm['E_star'] * np.sqrt(self.param_norm['R'])
#         self.param_norm['C3'] = self.param_norm['A'] / self.param_norm['Q']
#         self.attrs = dict(attrs or {})

#     def drive_si_to_hat(self, drive_si: float) -> float:
#         return float(drive_si * self.conversion['A'])

#     def measure_from_fd(self, d_hat: float, drive_hat: float):
#         A, phi, F = self.fd_library.evaluate(float(d_hat), float(drive_hat))
#         return float(A), float(phi), float(F)

#     def _pi_noise_step(self, drift_hat, A0_hat, dt_pixel, pixel_index):
#         nopt = self.pi_noise
#         drift_std_hat = float(nopt.drift_rel_std * A0_hat)
#         white_std_hat = float(nopt.white_rel_std * A0_hat)
#         sinusoid_amp_hat = float(nopt.sinusoid_rel_amp * A0_hat)
#         tau_hat = float(max(nopt.drift_tau_pixels, 1e-12) * dt_pixel)
#         drift_hat = _ou_step_stationary(drift_hat, std=drift_std_hat, tau=tau_hat, dt=dt_pixel, rng=self.rng)
#         white_hat = white_std_hat * self.rng.standard_normal() if white_std_hat > 0.0 else 0.0
#         if sinusoid_amp_hat > 0.0 and nopt.sinusoid_period_pixels > 0.0:
#             sinusoid_hat = sinusoid_amp_hat * np.sin(2.0 * np.pi * pixel_index / nopt.sinusoid_period_pixels)
#         else:
#             sinusoid_hat = 0.0
#         total_hat = drift_hat + white_hat + sinusoid_hat
#         return drift_hat, white_hat, sinusoid_hat, total_hat

#     def _pack_carry_state(self, out):
#         return dict(
#             d_cmd_init_hat=out["d_cmd_final_hat"],
#             d_act_init_hat=out["d_final_hat"],
#             A_meas_init_hat=out["A_meas_final_hat"],
#             phi_meas_init_deg=out["phi_meas_final_deg"],
#             integ_init=out["integ_final"],
#             drift_init_hat=out["drift_final_hat"],
#         )

#     def _flip_line_output(self, out):
#         flipped = {}
#         for k, v in out.items():
#             if isinstance(v, np.ndarray) and v.ndim == 1:
#                 flipped[k] = v[::-1].copy()
#             else:
#                 flipped[k] = v
#         return flipped

#     def scan_line_pi(
#         self,
#         h_hat,
#         drive_hat: Optional[float] = None,
#         drive_si: Optional[float] = None,
#         setpoint: float = 0.8,
#         scan_speed_hat: float = 1.0,
#         dx_hat: float = 1.0,
#         P: float = 0.1,
#         I: float = 0.01,
#         A_filter_order=None,
#         tau_A=1e-2,
#         wn_A=1e3,
#         zeta_A=0.7,
#         phi_filter_order=None,
#         tau_phi=1e-2,
#         wn_phi=1e3,
#         zeta_phi=0.7,
#         z_filter_order=None,
#         tau_z=1e-2,
#         wn_z=1e3,
#         zeta_z=0.7,
#         z_rate_limit_hat=None,
#         T_I=0.0,
#         integ_clip=None,
#         d_cmd_init_hat=None,
#         d_act_init_hat=None,
#         A_meas_init_hat=None,
#         phi_meas_init_deg=0.0,
#         integ_init=0.0,
#         drift_init_hat=0.0,
#         anti_windup=True,
#     ):
#         h_hat = np.asarray(h_hat, dtype=float)
#         n = h_hat.size
#         dt = float(dx_hat / scan_speed_hat)

#         if drive_hat is None:
#             if drive_si is not None:
#                 drive_hat = self.drive_si_to_hat(drive_si)
#             else:
#                 drive_hat = float(self.param_norm['A'])
#         drive_hat = float(drive_hat)

#         A0 = float(self.fd_library.free_amplitude(drive_hat))
#         A_set = float(setpoint * A0)
#         d_cmd = float(self.param_norm['d0'] if d_cmd_init_hat is None else d_cmd_init_hat)
#         d_act = float(d_cmd if d_act_init_hat is None else d_act_init_hat)
#         d_act_dot = 0.0
#         A_meas = float(A0 if A_meas_init_hat is None else A_meas_init_hat)
#         A_meas_dot = 0.0
#         phi_meas = float(phi_meas_init_deg)
#         phi_meas_dot = 0.0
#         integ = float(integ_init)
#         drift_hat = float(drift_init_hat)

#         if T_I is None or T_I <= 0.0:
#             decay = 1.0
#             use_leak = False
#         else:
#             decay = float(np.exp(-dt / T_I))
#             use_leak = True

#         d_min = float(self.fd_library.d_hat_grid[0])
#         d_max = float(self.fd_library.d_hat_grid[-1])

#         out_d_eff = np.empty(n)
#         out_d_cmd = np.empty(n)
#         out_d_act = np.empty(n)
#         out_A_true = np.empty(n)
#         out_A_meas = np.empty(n)
#         out_A_ctrl = np.empty(n)
#         out_phi_true = np.empty(n)
#         out_phi_meas = np.empty(n)
#         out_Fmax = np.empty(n)
#         out_err = np.empty(n)
#         out_drift = np.empty(n)
#         out_white = np.empty(n)
#         out_sinusoid = np.empty(n)
#         out_noise_total = np.empty(n)
#         out_drive_hat = np.full(n, drive_hat)
#         out_A_set = np.full(n, A_set)

#         for i in range(n):
#             d_eff = d_act - h_hat[i]
#             d_eff_eval = float(np.clip(d_eff, d_min, d_max))
#             A_true, phi_true, F_max = self.measure_from_fd(d_eff_eval, drive_hat)

#             if A_filter_order is None:
#                 A_meas = A_true
#             elif A_filter_order == 1:
#                 A_meas = _lpf1_step(A_meas, A_true, dt, tau_A)
#             elif A_filter_order == 2:
#                 A_meas, A_meas_dot = _lpf2_step(A_meas, A_meas_dot, A_true, dt, wn_A, zeta_A)
#             else:
#                 raise ValueError("A_filter_order must be None, 1, or 2")

#             if phi_filter_order is None:
#                 phi_meas = phi_true
#             elif phi_filter_order == 1:
#                 phi_meas = _lpf1_step(phi_meas, phi_true, dt, tau_phi)
#             elif phi_filter_order == 2:
#                 phi_meas, phi_meas_dot = _lpf2_step(phi_meas, phi_meas_dot, phi_true, dt, wn_phi, zeta_phi)
#             else:
#                 raise ValueError("phi_filter_order must be None, 1, or 2")

#             drift_hat, white_hat, sinusoid_hat, total_noise_hat = self._pi_noise_step(drift_hat=drift_hat, A0_hat=A0, dt_pixel=dt, pixel_index=i)
#             A_ctrl = A_meas + total_noise_hat
#             err = A_set - A_ctrl

#             sat_hi = d_eff > d_max
#             sat_lo = d_eff < d_min
#             would_wind_up = anti_windup and ((sat_hi and err > 0.0) or (sat_lo and err < 0.0))
#             if not would_wind_up:
#                 if use_leak:
#                     integ = decay * integ + err * dt
#                 else:
#                     integ = integ + err * dt
#             if integ_clip is not None:
#                 integ = float(np.clip(integ, -integ_clip, integ_clip))

#             dd_cmd = P * err + I * integ
#             if z_rate_limit_hat is not None:
#                 max_dd = float(z_rate_limit_hat) * dt
#                 dd_cmd = float(np.clip(dd_cmd, -max_dd, max_dd))
#             d_cmd = d_cmd + dd_cmd

#             if z_filter_order is None:
#                 d_act = d_cmd
#             elif z_filter_order == 1:
#                 d_act = _lpf1_step(d_act, d_cmd, dt, tau_z)
#             elif z_filter_order == 2:
#                 d_act, d_act_dot = _lpf2_step(d_act, d_act_dot, d_cmd, dt, wn_z, zeta_z)
#             else:
#                 raise ValueError("z_filter_order must be None, 1, or 2")

#             out_d_eff[i] = d_eff
#             out_d_cmd[i] = d_cmd
#             out_d_act[i] = d_act
#             out_A_true[i] = A_true
#             out_A_meas[i] = A_meas
#             out_A_ctrl[i] = A_ctrl
#             out_phi_true[i] = phi_true
#             out_phi_meas[i] = phi_meas
#             out_Fmax[i] = F_max
#             out_err[i] = err
#             out_drift[i] = drift_hat
#             out_white[i] = white_hat
#             out_sinusoid[i] = sinusoid_hat
#             out_noise_total[i] = total_noise_hat

#         return {
#             "d_eff_hat": out_d_eff,
#             "d_cmd_hat": out_d_cmd,
#             "d_hat": out_d_act,
#             "A_true_hat": out_A_true,
#             "A_hat": out_A_meas,
#             "A_ctrl_hat": out_A_ctrl,
#             "A_set_hat": out_A_set,
#             "phi_true_deg": out_phi_true,
#             "phi_deg": out_phi_meas,
#             "F_max_hat": out_Fmax,
#             "err": out_err,
#             "drift_hat": out_drift,
#             "white_hat": out_white,
#             "sinusoid_hat": out_sinusoid,
#             "noise_total_hat": out_noise_total,
#             "drive_hat": out_drive_hat,
#             "dt_pixel": dt,
#             "d_cmd_final_hat": float(d_cmd),
#             "d_final_hat": float(d_act),
#             "A_meas_final_hat": float(A_meas),
#             "phi_meas_final_deg": float(phi_meas),
#             "integ_final": float(integ),
#             "drift_final_hat": float(drift_hat),
#         }

#     def scan_line_trace_retrace_pi(self, h_hat, retrace_carry_state=True, retrace_use_reversed_topography=True, **line_kwargs):
#         h_hat = np.asarray(h_hat, dtype=float)
#         out_tr = self.scan_line_pi(h_hat, **line_kwargs)
#         h_rt = h_hat[::-1] if retrace_use_reversed_topography else h_hat.copy()
#         retrace_init = self._pack_carry_state(out_tr) if retrace_carry_state else {}
#         out_rt_raw = self.scan_line_pi(h_rt, **retrace_init, **line_kwargs)
#         out_rt = self._flip_line_output(out_rt_raw)
#         return {"trace": out_tr, "retrace": out_rt}

#     def scan_map_trace_retrace_pi(self, h_map_hat, carry_state_between_lines=True, retrace_carry_state=True, **line_kwargs):
#         h_map_hat = np.asarray(h_map_hat, dtype=float)
#         if h_map_hat.ndim != 2:
#             raise ValueError("h_map_hat must be 2D.")
#         ny, nx = h_map_hat.shape

#         channels = {
#             "height_trace_map": np.empty((ny, nx), dtype=float),
#             "height_retrace_map": np.empty((ny, nx), dtype=float),
#             "phase_trace_map": np.empty((ny, nx), dtype=float),
#             "phase_retrace_map": np.empty((ny, nx), dtype=float),
#             "amplitude_trace_map": np.empty((ny, nx), dtype=float),
#             "amplitude_retrace_map": np.empty((ny, nx), dtype=float),
#             "amplitude_ctrl_trace_map": np.empty((ny, nx), dtype=float),
#             "amplitude_ctrl_retrace_map": np.empty((ny, nx), dtype=float),
#             "force_trace_map": np.empty((ny, nx), dtype=float),
#             "force_retrace_map": np.empty((ny, nx), dtype=float),
#             "err_trace_map": np.empty((ny, nx), dtype=float),
#             "err_retrace_map": np.empty((ny, nx), dtype=float),
#             "drift_trace_map": np.empty((ny, nx), dtype=float),
#             "drift_retrace_map": np.empty((ny, nx), dtype=float),
#         }
#         carry = dict(
#             d_cmd_init_hat=line_kwargs.pop("d_cmd_init_hat", None),
#             d_act_init_hat=line_kwargs.pop("d_act_init_hat", None),
#             A_meas_init_hat=line_kwargs.pop("A_meas_init_hat", None),
#             phi_meas_init_deg=line_kwargs.pop("phi_meas_init_deg", 0.0),
#             integ_init=line_kwargs.pop("integ_init", 0.0),
#             drift_init_hat=line_kwargs.pop("drift_init_hat", 0.0),
#         )
#         for iy in range(ny):
#             row = h_map_hat[iy, :]
#             out_pair = self.scan_line_trace_retrace_pi(row, retrace_carry_state=retrace_carry_state, **carry, **line_kwargs)
#             tr = out_pair["trace"]
#             rt = out_pair["retrace"]
#             channels["height_trace_map"][iy, :] = tr["d_hat"]
#             channels["height_retrace_map"][iy, :] = rt["d_hat"]
#             channels["phase_trace_map"][iy, :] = tr["phi_deg"]
#             channels["phase_retrace_map"][iy, :] = rt["phi_deg"]
#             channels["amplitude_trace_map"][iy, :] = tr["A_hat"]
#             channels["amplitude_retrace_map"][iy, :] = rt["A_hat"]
#             channels["amplitude_ctrl_trace_map"][iy, :] = tr["A_ctrl_hat"]
#             channels["amplitude_ctrl_retrace_map"][iy, :] = rt["A_ctrl_hat"]
#             channels["force_trace_map"][iy, :] = tr["F_max_hat"]
#             channels["force_retrace_map"][iy, :] = rt["F_max_hat"]
#             channels["err_trace_map"][iy, :] = tr["err"]
#             channels["err_retrace_map"][iy, :] = rt["err"]
#             channels["drift_trace_map"][iy, :] = tr["drift_hat"]
#             channels["drift_retrace_map"][iy, :] = rt["drift_hat"]
#             if carry_state_between_lines:
#                 carry = self._pack_carry_state(rt)
#         return channels


# # ============================================================
# # 4) PLOTTING HELPERS
# # ============================================================
# def plot_trace_retrace_channels(out_pair, h_hat=None, center_height=True, figsize=(10, 9), title=None):
#     tr = out_pair["trace"]
#     rt = out_pair["retrace"]
#     x = np.arange(tr["d_hat"].size)

#     fig, axes = plt.subplots(3, 2, figsize=figsize, sharex=True)

#     h_tr = tr["d_hat"].copy()
#     h_rt = rt["d_hat"].copy()
#     gt = None if h_hat is None else np.asarray(h_hat, dtype=float).copy()
#     if center_height:
#         h_tr -= np.mean(h_tr)
#         h_rt -= np.mean(h_rt)
#         if gt is not None:
#             gt -= np.mean(gt)

#     axes[0, 0].plot(x, h_tr, lw=2, label="Trace")
#     if gt is not None:
#         axes[0, 0].plot(x, gt, lw=1.5, ls="--", label="Ground truth")
#     axes[0, 0].set_ylabel("Height")
#     axes[0, 0].set_title("Trace height")
#     axes[0, 0].grid(alpha=0.3)
#     axes[0, 0].legend()

#     axes[0, 1].plot(x, h_rt, lw=2, label="Retrace")
#     if gt is not None:
#         axes[0, 1].plot(x, gt, lw=1.5, ls="--", label="Ground truth")
#     axes[0, 1].set_ylabel("Height")
#     axes[0, 1].set_title("Retrace height")
#     axes[0, 1].grid(alpha=0.3)
#     axes[0, 1].legend()

#     axes[1, 0].plot(x, tr["phi_deg"], lw=2)
#     axes[1, 0].set_ylabel("Phase (deg)")
#     axes[1, 0].set_title("Trace phase")
#     axes[1, 0].grid(alpha=0.3)

#     axes[1, 1].plot(x, rt["phi_deg"], lw=2)
#     axes[1, 1].set_ylabel("Phase (deg)")
#     axes[1, 1].set_title("Retrace phase")
#     axes[1, 1].grid(alpha=0.3)

#     axes[2, 0].plot(x, tr["A_hat"], lw=2, label="A_meas")
#     if "A_ctrl_hat" in tr:
#         axes[2, 0].plot(x, tr["A_ctrl_hat"], lw=1.5, ls="--", label="A_ctrl")
#     axes[2, 0].set_ylabel("Amplitude")
#     axes[2, 0].set_xlabel("Pixel")
#     axes[2, 0].set_title("Trace amplitude")
#     axes[2, 0].grid(alpha=0.3)
#     axes[2, 0].legend()

#     axes[2, 1].plot(x, rt["A_hat"], lw=2, label="A_meas")
#     if "A_ctrl_hat" in rt:
#         axes[2, 1].plot(x, rt["A_ctrl_hat"], lw=1.5, ls="--", label="A_ctrl")
#     axes[2, 1].set_ylabel("Amplitude")
#     axes[2, 1].set_xlabel("Pixel")
#     axes[2, 1].set_title("Retrace amplitude")
#     axes[2, 1].grid(alpha=0.3)
#     axes[2, 1].legend()

#     if title is not None:
#         fig.suptitle(title)
#         fig.tight_layout(rect=(0, 0, 1, 0.97))
#     else:
#         fig.tight_layout()
#     return fig, axes


# # ============================================================
# # 5) ONLINE INSTRUMENT INTEGRATION WRAPPER
# # ============================================================
# class OnlineInstrumentBridge:
#     """
#     Maintains controller state across sequential scan lines for online use.
#     """

#     def __init__(self, scanner: ScannerFD2D):
#         self.scanner = scanner
#         self.reset()

#     def reset(self):
#         self.state = dict(
#             d_cmd_init_hat=None,
#             d_act_init_hat=None,
#             A_meas_init_hat=None,
#             phi_meas_init_deg=0.0,
#             integ_init=0.0,
#             drift_init_hat=0.0,
#         )

#     def scan_line(self, h_hat, drive_hat=None, drive_si=None, trace_retrace=True, **kwargs):
#         if trace_retrace:
#             out = self.scanner.scan_line_trace_retrace_pi(h_hat, drive_hat=drive_hat, drive_si=drive_si, **self.state, **kwargs)
#             self.state = self.scanner._pack_carry_state(out["retrace"])
#             return out
#         out = self.scanner.scan_line_pi(h_hat, drive_hat=drive_hat, drive_si=drive_si, **self.state, **kwargs)
#         self.state = self.scanner._pack_carry_state(out)
#         return out


# # ============================================================
# # 6) EXPERIMENT GRID DATA + ALIGNMENT / CALIBRATION
# # ============================================================
# @dataclass
# class ExperimentGridData:
#     drives_exp: np.ndarray
#     setpoints_exp: np.ndarray
#     I_gains_exp: np.ndarray
#     height_trace: np.ndarray
#     height_retrace: np.ndarray
#     phase_trace: np.ndarray
#     phase_retrace: np.ndarray
#     amplitude_trace: Optional[np.ndarray] = None
#     amplitude_retrace: Optional[np.ndarray] = None
#     attrs: Dict[str, Any] = field(default_factory=dict)

#     def __post_init__(self):
#         self.drives_exp = np.asarray(self.drives_exp, dtype=float)
#         self.setpoints_exp = np.asarray(self.setpoints_exp, dtype=float)
#         self.I_gains_exp = np.asarray(self.I_gains_exp, dtype=float)
#         self.height_trace = np.asarray(self.height_trace, dtype=float)
#         self.height_retrace = np.asarray(self.height_retrace, dtype=float)
#         self.phase_trace = np.asarray(self.phase_trace, dtype=float)
#         self.phase_retrace = np.asarray(self.phase_retrace, dtype=float)
#         if self.amplitude_trace is not None:
#             self.amplitude_trace = np.asarray(self.amplitude_trace, dtype=float)
#         if self.amplitude_retrace is not None:
#             self.amplitude_retrace = np.asarray(self.amplitude_retrace, dtype=float)

#         expected = (len(self.drives_exp), len(self.setpoints_exp), len(self.I_gains_exp))
#         for name, arr in [
#             ("height_trace", self.height_trace),
#             ("height_retrace", self.height_retrace),
#             ("phase_trace", self.phase_trace),
#             ("phase_retrace", self.phase_retrace),
#         ]:
#             if arr.ndim != 4 or arr.shape[:3] != expected:
#                 raise ValueError(f"{name} must have shape (n_drive, n_setpoint, n_I, n_pixel).")
#         if self.amplitude_trace is not None and (self.amplitude_trace.ndim != 4 or self.amplitude_trace.shape[:3] != expected):
#             raise ValueError("amplitude_trace must have shape (n_drive, n_setpoint, n_I, n_pixel).")
#         if self.amplitude_retrace is not None and (self.amplitude_retrace.ndim != 4 or self.amplitude_retrace.shape[:3] != expected):
#             raise ValueError("amplitude_retrace must have shape (n_drive, n_setpoint, n_I, n_pixel).")

#     @property
#     def n_pixel(self):
#         return self.height_trace.shape[-1]


# @dataclass
# class AlignmentConfig:
#     compare_amplitude: bool = False
#     height_weight: float = 1.0
#     phase_weight: float = 1.0
#     amplitude_weight: float = 0.5
#     hysteresis_weight: float = 0.5
#     remove_height_offset: bool = True
#     remove_height_tilt: bool = True
#     phase_wrap_deg: float = 180.0
#     robust_eps: float = 1e-8
#     line_kwargs_base: Dict[str, Any] = field(default_factory=lambda: dict(
#         scan_speed_hat=1.0,
#         dx_hat=1.0,
#         P=0.1,
#         I=0.01,
#         A_filter_order=1,
#         tau_A=1e-2,
#         phi_filter_order=1,
#         tau_phi=1e-2,
#         z_filter_order=1,
#         tau_z=1e-2,
#         T_I=0.0,
#     ))


# def _mad_scale(x: np.ndarray, eps: float = 1e-8) -> float:
#     med = np.median(x)
#     mad = np.median(np.abs(x - med))
#     return float(max(1.4826 * mad, eps))


# def _remove_offset_tilt(y: np.ndarray, remove_offset: bool = True, remove_tilt: bool = True) -> np.ndarray:
#     y = np.asarray(y, dtype=float)
#     if not (remove_offset or remove_tilt):
#         return y.copy()
#     x = np.arange(y.size, dtype=float)
#     if remove_tilt:
#         A = np.column_stack([np.ones_like(x), x])
#         coef, *_ = np.linalg.lstsq(A, y, rcond=None)
#         trend = A @ coef
#         return y - trend
#     if remove_offset:
#         return y - np.mean(y)
#     return y.copy()


# def _wrap_phase_diff_deg(a: np.ndarray, b: np.ndarray, period: float = 180.0) -> np.ndarray:
#     diff = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
#     return ((diff + 0.5 * period) % period) - 0.5 * period


# def _pseudo_huber(residual: np.ndarray, delta: float = 1.0) -> float:
#     r = np.asarray(residual, dtype=float)
#     return float(np.mean(delta * delta * (np.sqrt(1.0 + (r / delta) ** 2) - 1.0)))


# def default_vector_to_theta(x: np.ndarray) -> Dict[str, float]:
#     """
#     Default alignment parameterization.

#     x = [
#         drive_scale, drive_shift_hat,
#         setpoint_scale, setpoint_shift,
#         I_scale,
#         phase_scale, phase_offset_deg,
#         tau_A, tau_phi, tau_z,
#         P_scale
#     ]
#     """
#     return {
#         "drive_scale": float(x[0]),
#         "drive_shift_hat": float(x[1]),
#         "setpoint_scale": float(x[2]),
#         "setpoint_shift": float(x[3]),
#         "I_scale": float(x[4]),
#         "phase_scale": float(x[5]),
#         "phase_offset_deg": float(x[6]),
#         "tau_A": float(x[7]),
#         "tau_phi": float(x[8]),
#         "tau_z": float(x[9]),
#         "P_scale": float(x[10]),
#     }


# class GridCalibrator2D:
#     """
#     Align multi-drive FD scanner simulations to an experimental drive/setpoint/I grid.
#     """

#     def __init__(
#         self,
#         experiment: ExperimentGridData,
#         scanner: ScannerFD2D,
#         surface_ref_hat: np.ndarray,
#         config: Optional[AlignmentConfig] = None,
#         attrs: Optional[Dict[str, Any]] = None,
#     ):
#         self.experiment = experiment
#         self.scanner = scanner
#         self.surface_ref_hat = np.asarray(surface_ref_hat, dtype=float)
#         self.config = config if config is not None else AlignmentConfig()
#         self.attrs = dict(attrs or {})
#         if self.surface_ref_hat.ndim != 1 or self.surface_ref_hat.size != self.experiment.n_pixel:
#             raise ValueError("surface_ref_hat must be 1D with length equal to n_pixel.")

#     def _line_kwargs_from_theta(self, theta: Dict[str, float], I_exp: float) -> Dict[str, Any]:
#         kw = dict(self.config.line_kwargs_base)
#         kw["I"] = float(I_exp * theta.get("I_scale", 1.0))
#         kw["P"] = float(kw.get("P", 0.1) * theta.get("P_scale", 1.0))
#         kw["tau_A"] = float(theta.get("tau_A", kw.get("tau_A", 1e-2)))
#         kw["tau_phi"] = float(theta.get("tau_phi", kw.get("tau_phi", 1e-2)))
#         kw["tau_z"] = float(theta.get("tau_z", kw.get("tau_z", 1e-2)))
#         return kw

#     def _map_drive_setpoint(self, drive_exp: float, setpoint_exp: float, theta: Dict[str, float]) -> Tuple[float, float]:
#         drive_hat = theta.get("drive_scale", 1.0) * float(drive_exp) + theta.get("drive_shift_hat", 0.0)
#         setpoint = theta.get("setpoint_scale", 1.0) * float(setpoint_exp) + theta.get("setpoint_shift", 0.0)
#         return drive_hat, setpoint

#     def simulate_condition(self, drive_exp: float, setpoint_exp: float, I_exp: float, theta: Dict[str, float], carry_state: bool = False):
#         drive_hat, setpoint = self._map_drive_setpoint(drive_exp, setpoint_exp, theta)
#         line_kwargs = self._line_kwargs_from_theta(theta, I_exp)
#         out_pair = self.scanner.scan_line_trace_retrace_pi(
#             h_hat=self.surface_ref_hat,
#             drive_hat=drive_hat,
#             setpoint=setpoint,
#             retrace_carry_state=carry_state,
#             **line_kwargs,
#         )
#         # affine phase correction
#         for key in ("trace", "retrace"):
#             out_pair[key]["phi_deg"] = theta.get("phase_scale", 1.0) * out_pair[key]["phi_deg"] + theta.get("phase_offset_deg", 0.0)
#         return out_pair

#     def condition_loss(self, out_pair: Dict[str, Dict[str, np.ndarray]], exp_ht_tr, exp_ht_rt, exp_ph_tr, exp_ph_rt, exp_amp_tr=None, exp_amp_rt=None) -> float:
#         cfg = self.config
#         loss = 0.0

#         sim_ht_tr = _remove_offset_tilt(out_pair["trace"]["d_hat"], cfg.remove_height_offset, cfg.remove_height_tilt)
#         sim_ht_rt = _remove_offset_tilt(out_pair["retrace"]["d_hat"], cfg.remove_height_offset, cfg.remove_height_tilt)
#         exp_ht_tr = _remove_offset_tilt(exp_ht_tr, cfg.remove_height_offset, cfg.remove_height_tilt)
#         exp_ht_rt = _remove_offset_tilt(exp_ht_rt, cfg.remove_height_offset, cfg.remove_height_tilt)

#         hs = 0.5 * (_mad_scale(exp_ht_tr, cfg.robust_eps) + _mad_scale(exp_ht_rt, cfg.robust_eps))
#         ps = 0.5 * (_mad_scale(exp_ph_tr, cfg.robust_eps) + _mad_scale(exp_ph_rt, cfg.robust_eps))

#         loss += cfg.height_weight * _pseudo_huber((sim_ht_tr - exp_ht_tr) / hs)
#         loss += cfg.height_weight * _pseudo_huber((sim_ht_rt - exp_ht_rt) / hs)

#         ph_tr_diff = _wrap_phase_diff_deg(out_pair["trace"]["phi_deg"], exp_ph_tr, cfg.phase_wrap_deg) / ps
#         ph_rt_diff = _wrap_phase_diff_deg(out_pair["retrace"]["phi_deg"], exp_ph_rt, cfg.phase_wrap_deg) / ps
#         loss += cfg.phase_weight * _pseudo_huber(ph_tr_diff)
#         loss += cfg.phase_weight * _pseudo_huber(ph_rt_diff)

#         # hysteresis matching
#         sim_dh = (sim_ht_tr - sim_ht_rt) / hs
#         exp_dh = (exp_ht_tr - exp_ht_rt) / hs
#         loss += cfg.hysteresis_weight * _pseudo_huber(sim_dh - exp_dh)

#         sim_dp = _wrap_phase_diff_deg(out_pair["trace"]["phi_deg"], out_pair["retrace"]["phi_deg"], cfg.phase_wrap_deg) / ps
#         exp_dp = _wrap_phase_diff_deg(exp_ph_tr, exp_ph_rt, cfg.phase_wrap_deg) / ps
#         loss += cfg.hysteresis_weight * _pseudo_huber(sim_dp - exp_dp)

#         if cfg.compare_amplitude and exp_amp_tr is not None and exp_amp_rt is not None:
#             sim_amp_tr = out_pair["trace"]["A_hat"]
#             sim_amp_rt = out_pair["retrace"]["A_hat"]
#             ascale = 0.5 * (_mad_scale(exp_amp_tr, cfg.robust_eps) + _mad_scale(exp_amp_rt, cfg.robust_eps))
#             loss += cfg.amplitude_weight * _pseudo_huber((sim_amp_tr - exp_amp_tr) / ascale)
#             loss += cfg.amplitude_weight * _pseudo_huber((sim_amp_rt - exp_amp_rt) / ascale)

#         return float(loss)

#     def total_loss(self, theta: Dict[str, float], subset: Optional[Sequence[Tuple[int, int, int]]] = None) -> float:
#         exp = self.experiment
#         if subset is None:
#             subset = [(i, j, k) for i in range(len(exp.drives_exp)) for j in range(len(exp.setpoints_exp)) for k in range(len(exp.I_gains_exp))]
#         total = 0.0
#         for i, j, k in subset:
#             out_pair = self.simulate_condition(exp.drives_exp[i], exp.setpoints_exp[j], exp.I_gains_exp[k], theta)
#             amp_tr = None if exp.amplitude_trace is None else exp.amplitude_trace[i, j, k]
#             amp_rt = None if exp.amplitude_retrace is None else exp.amplitude_retrace[i, j, k]
#             total += self.condition_loss(
#                 out_pair,
#                 exp.height_trace[i, j, k],
#                 exp.height_retrace[i, j, k],
#                 exp.phase_trace[i, j, k],
#                 exp.phase_retrace[i, j, k],
#                 amp_tr,
#                 amp_rt,
#             )
#         return float(total / max(len(subset), 1))

#     def fit(
#         self,
#         bounds: Sequence[Tuple[float, float]],
#         vector_to_theta=default_vector_to_theta,
#         x0: Optional[np.ndarray] = None,
#         subset: Optional[Sequence[Tuple[int, int, int]]] = None,
#         maxiter_de: int = 30,
#         polish: bool = True,
#         seed: int = 0,
#         workers: int = 1,
#     ) -> Dict[str, Any]:
#         def obj(x):
#             theta = vector_to_theta(np.asarray(x, dtype=float))
#             return self.total_loss(theta, subset=subset)

#         result_de = differential_evolution(obj, bounds=bounds, maxiter=maxiter_de, seed=seed, workers=workers, polish=False)
#         x_best = result_de.x.copy()
#         f_best = float(result_de.fun)
#         result_local = None
#         if polish:
#             result_local = minimize(obj, x_best, method="Powell")
#             if float(result_local.fun) < f_best:
#                 x_best = result_local.x.copy()
#                 f_best = float(result_local.fun)
#         theta_best = vector_to_theta(np.asarray(x_best, dtype=float))
#         return {
#             "x_best": x_best,
#             "theta_best": theta_best,
#             "loss_best": f_best,
#             "de_result": result_de,
#             "local_result": result_local,
#         }


# # ============================================================
# # 7) VISUAL DIAGNOSTICS FOR ALIGNMENT
# # ============================================================
# def plot_alignment_condition(out_pair, exp_height_trace, exp_height_retrace, exp_phase_trace, exp_phase_retrace, title=None, center_height=True, figsize=(10, 9)):
#     tr = out_pair["trace"]
#     rt = out_pair["retrace"]
#     x = np.arange(len(exp_height_trace))

#     sim_h_tr = tr["d_hat"].copy()
#     sim_h_rt = rt["d_hat"].copy()
#     exp_h_tr = np.asarray(exp_height_trace, dtype=float).copy()
#     exp_h_rt = np.asarray(exp_height_retrace, dtype=float).copy()
#     if center_height:
#         sim_h_tr -= np.mean(sim_h_tr)
#         sim_h_rt -= np.mean(sim_h_rt)
#         exp_h_tr -= np.mean(exp_h_tr)
#         exp_h_rt -= np.mean(exp_h_rt)

#     fig, axes = plt.subplots(3, 2, figsize=figsize, sharex=True)
#     axes[0, 0].plot(x, exp_h_tr, label="Exp", lw=1.8)
#     axes[0, 0].plot(x, sim_h_tr, label="Sim", lw=1.8)
#     axes[0, 0].set_title("Trace height")
#     axes[0, 0].legend()
#     axes[0, 0].grid(alpha=0.3)

#     axes[0, 1].plot(x, exp_h_rt, label="Exp", lw=1.8)
#     axes[0, 1].plot(x, sim_h_rt, label="Sim", lw=1.8)
#     axes[0, 1].set_title("Retrace height")
#     axes[0, 1].legend()
#     axes[0, 1].grid(alpha=0.3)

#     axes[1, 0].plot(x, exp_phase_trace, label="Exp", lw=1.8)
#     axes[1, 0].plot(x, tr["phi_deg"], label="Sim", lw=1.8)
#     axes[1, 0].set_title("Trace phase")
#     axes[1, 0].legend()
#     axes[1, 0].grid(alpha=0.3)

#     axes[1, 1].plot(x, exp_phase_retrace, label="Exp", lw=1.8)
#     axes[1, 1].plot(x, rt["phi_deg"], label="Sim", lw=1.8)
#     axes[1, 1].set_title("Retrace phase")
#     axes[1, 1].legend()
#     axes[1, 1].grid(alpha=0.3)

#     axes[2, 0].plot(x, tr["A_hat"], label="Trace A_meas", lw=1.8)
#     axes[2, 0].plot(x, tr["A_ctrl_hat"], label="Trace A_ctrl", lw=1.4, ls="--")
#     axes[2, 0].set_title("Trace amplitude")
#     axes[2, 0].legend()
#     axes[2, 0].grid(alpha=0.3)

#     axes[2, 1].plot(x, rt["A_hat"], label="Retrace A_meas", lw=1.8)
#     axes[2, 1].plot(x, rt["A_ctrl_hat"], label="Retrace A_ctrl", lw=1.4, ls="--")
#     axes[2, 1].set_title("Retrace amplitude")
#     axes[2, 1].legend()
#     axes[2, 1].grid(alpha=0.3)

#     if title is not None:
#         fig.suptitle(title)
#         fig.tight_layout(rect=(0, 0, 1, 0.97))
#     else:
#         fig.tight_layout()
#     return fig, axes


# # ============================================================
# # 8) EXAMPLE WORKFLOW HELPERS
# # ============================================================
# def build_scanner_from_drive_library(
#     params: Dict[str, float],
#     conversions: Dict[str, float],
#     drive_si_values: Sequence[float],
#     solver: Optional[RK45Options] = None,
#     pi_noise: Optional[PIDriftNoiseOptions] = None,
#     d_start_hat: Optional[float] = None,
#     d_end_hat: float = -25.0,
#     n_points: int = 256,
#     omega_drive_hat: float = 1.0,
#     omega_ref_hat: float = 1.0,
#     attrs: Optional[Dict[str, Any]] = None,
# ):
#     build_result = build_fd_library_from_drives(
#         params=params,
#         conversions=conversions,
#         drive_si_values=drive_si_values,
#         solver=solver,
#         d_start_hat=d_start_hat,
#         d_end_hat=d_end_hat,
#         n_points=n_points,
#         omega_drive_hat=omega_drive_hat,
#         omega_ref_hat=omega_ref_hat,
#         extra_attrs=attrs,
#     )
#     scanner = ScannerFD2D(params=params, conversions=conversions, fd_library=build_result.library, pi_noise=pi_noise)
#     return build_result, scanner

In [ ]:
@dataclass
class NormalizedSPMParams:
    """Dimensionless / normalized SPM parameters."""
    Q: float
    C1: float
    C2: float
    C3: float
    a0: float
    omega_drive_hat: float = 1.0
    omega_ref_hat: float = 1.0
    attrs: Dict[str, Any] = field(default_factory=dict)

def build_normalized_spm_params(
    params: Dict[str, float],
    conversions: Dict[str, float],
    drive_si: float,
    omega_drive_hat: float = 1.0,
    omega_ref_hat: float = 1.0,
    attrs: Optional[Dict[str, Any]] = None,
) -> Tuple[NormalizedSPMParams, Dict[str, float]]:
    params_local = dict(params)
    params_local["A"] = float(drive_si)
    param_norm = normalize_params(params_local, conversions)
    C1 = param_norm['H'] * param_norm['R'] / 6.0
    C2 = (4.0 / 3.0) * param_norm['E_star'] * np.sqrt(param_norm['R'])
    C3 = param_norm['A'] / param_norm['Q']
    meta = dict(attrs or {})
    meta.update({
        "drive_si": float(drive_si),
        "drive_hat": float(param_norm['A']),
        "omega_drive_hat": float(omega_drive_hat),
        "omega_ref_hat": float(omega_ref_hat),
    })
    spm_params = NormalizedSPMParams(
        Q=float(param_norm['Q']), C1=float(C1), C2=float(C2), C3=float(C3), a0=float(param_norm['a0']),
        omega_drive_hat=float(omega_drive_hat), omega_ref_hat=float(omega_ref_hat), attrs=meta,
    )
    return spm_params, param_norm

def normalize_params(params: Dict[str, float], conversions: Dict[str, float]) -> Dict[str, float]:
    return {k: float(params[k] * conversions[k]) for k in params}

In [ ]:
# FD-only calibration for the SPM digital twin.
"""
fd_only_calibration.py
======================

FD-only calibration for the SPM digital twin.

Inputs are a set of *experimental* point-FD curves taken at different
drive amplitudes.  Each curve is a 1D array (length N, e.g. 1007) of
amplitude and phase, optionally with z-piezo position.  The "setpoint
axis" used for fitting is the normalized amplitude:

        setpoint_pct  =  A_meas / A_free * 100

This avoids needing absolute z-piezo calibration and matches how the
attractive-to-repulsive transition is usually plotted (see Drive vs
Threshold-Setpoint plot).

The calibrator runs the existing RK45 approach-curve simulator from
`spm_fd_multidrive_alignment.py`, reparameterizes the simulated FD
curve onto the same setpoint axis, and minimises a weighted loss made
of three terms:

    L = w_phase  * MSE( phi_sim(setpoint) - phi_exp(setpoint) )
      + w_free   * (phi_sim_free - phi_exp_free)^2
      + w_thresh * (sp_thresh_sim - sp_thresh_exp)^2

Parameters that can be fit (any subset, controlled by FDCalibrationConfig.fit_keys):

    Material:   E_star, H, a0, R
    Cantilever: Q, k
    Aux:        phase_offset_deg, drive_scale

Recommended first pass:  fit ("E_star", "H", "a0", "phase_offset_deg")
with Q locked (use a thermal-tune Q) and drive_scale = 1.

Author: digital-twin alignment helper.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, minimize

# # Re-use the existing simulator. Adjust the import path if needed.
# from spm_fd_multidrive_alignment import (
#     NormalizedSPMParams,
#     RK45Options,
#     SPMRK45ApproachModel,
#     build_normalized_spm_params,
# )


# ============================================================
# 1) DATA CONTAINER
# ============================================================
@dataclass
class FDCalibrationCurve:
    """One experimental FD curve at a fixed drive amplitude.

    The fitter only needs (amplitude_nm, phase_deg) and the drive in SI.
    Setpoint% is auto-computed from amplitude / A_free, and the threshold
    setpoint (attractive→repulsive transition) is auto-detected from a
    sharp downward jump in phase across `threshold_phase_deg` if not
    supplied explicitly.

    All arrays must be 1-D and the same length.
    """
    drive_si: float                                  # m, drive amplitude at the cantilever
    amplitude_nm: np.ndarray                         # nm
    phase_deg: np.ndarray                            # deg (in [0, 180] convention)
    z_piezo_nm: Optional[np.ndarray] = None          # optional, z-piezo position (nm)
    setpoint_pct: Optional[np.ndarray] = None        # optional, default = A/A_free*100
    threshold_setpoint_pct: Optional[float] = None   # optional; auto-detected if None
    A_free_nm: Optional[float] = None                # optional; auto = high-percentile of A
    free_phase_deg: Optional[float] = None           # optional; auto = mean phase at high SP
    label: str = ""

    def __post_init__(self):
        self.amplitude_nm = np.asarray(self.amplitude_nm, dtype=float).ravel()
        self.phase_deg = np.asarray(self.phase_deg, dtype=float).ravel()
        if self.amplitude_nm.shape != self.phase_deg.shape:
            raise ValueError("amplitude_nm and phase_deg must have the same shape")
        if self.z_piezo_nm is not None:
            self.z_piezo_nm = np.asarray(self.z_piezo_nm, dtype=float).ravel()
        if self.A_free_nm is None:
            # robust free-amplitude estimate: mean of top 5% of amplitude
            top = np.percentile(self.amplitude_nm, 95.0)
            self.A_free_nm = float(np.mean(self.amplitude_nm[self.amplitude_nm >= top]))
        if self.A_free_nm <= 0:
            raise ValueError(f"Could not estimate A_free_nm for drive={self.drive_si}")
        if self.setpoint_pct is None:
            self.setpoint_pct = self.amplitude_nm / self.A_free_nm * 100.0
        else:
            self.setpoint_pct = np.asarray(self.setpoint_pct, dtype=float).ravel()
        if self.free_phase_deg is None:
            top_sp = np.percentile(self.setpoint_pct, 95.0)
            mask = self.setpoint_pct >= top_sp
            if np.any(mask):
                self.free_phase_deg = float(np.mean(self.phase_deg[mask]))
            else:
                self.free_phase_deg = float(np.mean(self.phase_deg))


# ============================================================
# 2) TRANSITION DETECTION
# ============================================================
def detect_threshold_setpoint(
    setpoint_pct: np.ndarray,
    phase_deg: np.ndarray,
    threshold_phase_deg: float = 90.0,
    jump_min_deg: float = 15.0,
    smooth_window: int = 3,
) -> Optional[float]:
    """
    Find the setpoint at which phase jumps downward across `threshold_phase_deg`.

    Walks the curve in approach order (descending setpoint) and looks for
    a discrete drop of at least `jump_min_deg` over `smooth_window` samples
    that crosses `threshold_phase_deg`.  Returns the linearly interpolated
    setpoint at the crossing, or None if no clear jump exists.
    """
    sp = np.asarray(setpoint_pct, dtype=float)
    ph = np.asarray(phase_deg, dtype=float)
    order = np.argsort(-sp)
    sp_o = sp[order]
    ph_o = ph[order]
    w = max(1, int(smooth_window))
    if ph_o.size < w + 1:
        return None
    diff = ph_o[w:] - ph_o[:-w]
    crossings = np.where(
        (ph_o[:-w] >= threshold_phase_deg)
        & (ph_o[w:] < threshold_phase_deg)
        & (diff <= -jump_min_deg)
    )[0]
    if crossings.size == 0:
        return None
    i = int(crossings[0])
    a = ph_o[i]
    b = ph_o[i + w]
    if b == a:
        return float(0.5 * (sp_o[i] + sp_o[i + w]))
    t = (threshold_phase_deg - a) / (b - a)
    t = float(np.clip(t, 0.0, 1.0))
    return float(sp_o[i] + t * (sp_o[i + w] - sp_o[i]))


# ============================================================
# 3) FIT CONFIG
# ============================================================
@dataclass
class FDCalibrationConfig:
    # Which keys to fit. Anything not listed is held at base_params value.
    fit_keys: Tuple[str, ...] = ("E_star", "H", "a0", "phase_offset_deg")

    # Loss weights
    phase_weight: float = 1.0
    free_phase_weight: float = 2.0
    threshold_weight: float = 6.0

    # Region of phase curve to compare. Avoids the bistable jump itself
    # (handled by the threshold term) and the noisy pre-engagement tail.
    setpoint_min_pct: float = 30.0
    setpoint_max_pct: float = 92.0

    # Threshold detection on experimental + simulated curves.
    threshold_phase_deg: float = 90.0
    jump_min_deg: float = 15.0
    threshold_smooth_window: int = 3

    # Branch selection for sim → exp interpolation
    # "auto"        : sort by setpoint and interp (works above threshold)
    # "attractive"  : keep only the high-d_hat branch (before transition)
    # "repulsive"   : keep only the low-d_hat branch (after transition)
    interp_branch: str = "auto"

    # Simulator grid
    n_d_hat_grid: int = 160
    d_end_hat: float = -8.0
    d_start_factor: float = 1.25  # d_start_hat = drive_hat * d_start_factor

    # Numerics
    bistable_jump_penalty: float = 100.0   # penalty if exp shows a jump but sim doesn't


# ============================================================
# 4) FD-ONLY CALIBRATOR
# ============================================================
class FDOnlyCalibrator:
    """
    Hierarchical FD-only calibration:
        1. Auto-detect threshold setpoints in experimental curves.
        2. Forward-simulate each FD curve under candidate parameters.
        3. Score (phi_sim vs phi_exp on the setpoint axis,
                  free_phase asymptote,
                  threshold setpoint).
        4. Optimise via differential evolution + Powell polish.
    """

    _MATERIAL_KEYS = ("E_star", "H", "a0", "R", "Q", "k")
    _AUX_KEYS = ("phase_offset_deg", "drive_scale")

    def __init__(
        self,
        fd_curves: Sequence[FDCalibrationCurve],
        base_params: Dict[str, float],
        conversions: Dict[str, float],
        solver: Optional[RK45Options] = None,
        config: Optional[FDCalibrationConfig] = None,
    ):
        self.fd_curves = list(fd_curves)
        self.base_params = dict(base_params)
        self.conversions = dict(conversions)
        self.solver = solver if solver is not None else RK45Options()
        self.config = config if config is not None else FDCalibrationConfig()

        # Auto-fill missing thresholds from data
        for fd in self.fd_curves:
            if fd.threshold_setpoint_pct is None:
                fd.threshold_setpoint_pct = detect_threshold_setpoint(
                    fd.setpoint_pct, fd.phase_deg,
                    self.config.threshold_phase_deg,
                    self.config.jump_min_deg,
                    self.config.threshold_smooth_window,
                )

    # ---------------- parameter assembly ----------------
    def _build_sim_params(self, theta: Dict[str, float]) -> Dict[str, float]:
        p = dict(self.base_params)
        for k in self._MATERIAL_KEYS:
            if k in theta:
                p[k] = float(theta[k])
        return p

    # ---------------- forward simulation ----------------
    def simulate_curve(self, drive_si: float, theta: Dict[str, float]):
        """Run one approach curve.  Returns d_hat, A_hat, phi_deg (with phase offset applied)."""
        p = self._build_sim_params(theta)
        drive_eff = float(drive_si) * float(theta.get("drive_scale", 1.0))
        spm_params, param_norm = build_normalized_spm_params(
            params=p,
            conversions=self.conversions,
            drive_si=drive_eff,
            omega_drive_hat=1.0,
            omega_ref_hat=1.0,
        )
        model = SPMRK45ApproachModel(spm_params, solver=self.solver)
        d_start = float(param_norm["A"]) * float(self.config.d_start_factor)
        curve = model.approach_curve(
            d_start_hat=d_start,
            d_end_hat=float(self.config.d_end_hat),
            n_points=int(self.config.n_d_hat_grid),
            z0_hat=0.0, v0_hat=0.0, warm_start=True,
        )
        phi = curve["phi_deg"] + float(theta.get("phase_offset_deg", 0.0))
        return curve["d_hat"], curve["A_hat"], phi

    # ---------------- sim → setpoint axis ----------------
    @staticmethod
    def _sim_to_setpoint_axis(d_hat: np.ndarray, A_hat: np.ndarray, phi: np.ndarray):
        """Re-parameterize (d_hat, A, phi) onto setpoint% = A/A_free*100."""
        far = int(np.argmax(d_hat))
        A_free_hat = float(A_hat[far])
        if A_free_hat <= 0 or not np.isfinite(A_free_hat):
            return None, None, None, np.nan
        sp_sim = A_hat / A_free_hat * 100.0
        return sp_sim, phi, d_hat, A_free_hat

    @staticmethod
    def _split_branches(sp_sim, phi_sim, d_sim, threshold_phase_deg: float = 90.0):
        """Split the natural approach trajectory at the first phase-jump across threshold."""
        order = np.argsort(-d_sim)
        sp = sp_sim[order]
        ph = phi_sim[order]
        ds = d_sim[order]
        trans = None
        for i in range(1, ph.size):
            if ph[i - 1] >= threshold_phase_deg > ph[i]:
                trans = i
                break
        if trans is None:
            return (sp, ph, ds), (None, None, None)
        return (sp[:trans], ph[:trans], ds[:trans]), (sp[trans:], ph[trans:], ds[trans:])

    @classmethod
    def _interp_phase(cls, sp_sim, phi_sim, d_sim, sp_query, branch: str = "auto",
                      threshold_phase_deg: float = 90.0):
        """Interpolate sim phase at query setpoints."""
        if branch == "auto":
            order = np.argsort(sp_sim)
            sp_s = sp_sim[order]
            ph_s = phi_sim[order]
            keep = np.concatenate([[True], np.diff(sp_s) > 1e-9])
            sp_s = sp_s[keep]
            ph_s = ph_s[keep]
            return np.interp(sp_query, sp_s, ph_s, left=np.nan, right=np.nan)
        attr, rep = cls._split_branches(sp_sim, phi_sim, d_sim, threshold_phase_deg)
        if branch == "attractive":
            sp_b, ph_b, _ = attr
        elif branch == "repulsive":
            if rep[0] is None:
                return np.full_like(np.asarray(sp_query, dtype=float), np.nan)
            sp_b, ph_b, _ = rep
        else:
            raise ValueError(f"Unknown branch: {branch}")
        order = np.argsort(sp_b)
        sp_s = sp_b[order]
        ph_s = ph_b[order]
        keep = np.concatenate([[True], np.diff(sp_s) > 1e-9])
        return np.interp(sp_query, sp_s[keep], ph_s[keep], left=np.nan, right=np.nan)

    @classmethod
    def _find_sim_threshold(cls, sp_sim, phi_sim, d_sim, threshold_phase_deg: float = 90.0):
        """Find the setpoint at which the sim's phase first crosses threshold downward."""
        order = np.argsort(-d_sim)
        sp = sp_sim[order]
        ph = phi_sim[order]
        for i in range(1, ph.size):
            if ph[i - 1] >= threshold_phase_deg > ph[i]:
                if ph[i] == ph[i - 1]:
                    return float(0.5 * (sp[i - 1] + sp[i]))
                t = (threshold_phase_deg - ph[i - 1]) / (ph[i] - ph[i - 1])
                return float(sp[i - 1] + t * (sp[i] - sp[i - 1]))
        return np.nan

    # ---------------- loss ----------------
    def loss(self, theta: Dict[str, float]) -> float:
        cfg = self.config
        total = 0.0
        n_terms = 0
        for fd in self.fd_curves:
            try:
                d_hat, A_hat, phi = self.simulate_curve(fd.drive_si, theta)
            except Exception:
                return 1e20
            sp_sim, phi_sim, d_sim, A_free_hat = self._sim_to_setpoint_axis(d_hat, A_hat, phi)
            if sp_sim is None:
                return 1e20

            # (a) phase curve term
            if cfg.phase_weight > 0:
                m = (fd.setpoint_pct >= cfg.setpoint_min_pct) & (fd.setpoint_pct <= cfg.setpoint_max_pct)
                if np.any(m):
                    phi_at = self._interp_phase(
                        sp_sim, phi_sim, d_sim, fd.setpoint_pct[m],
                        branch=cfg.interp_branch,
                        threshold_phase_deg=cfg.threshold_phase_deg,
                    )
                    valid = np.isfinite(phi_at)
                    if np.any(valid):
                        r = phi_at[valid] - fd.phase_deg[m][valid]
                        total += cfg.phase_weight * float(np.mean(r * r))
                        n_terms += 1

            # (b) free phase asymptote
            if cfg.free_phase_weight > 0:
                far_sim = float(phi_sim[int(np.argmax(d_sim))])
                r = far_sim - float(fd.free_phase_deg)
                total += cfg.free_phase_weight * float(r * r)
                n_terms += 1

            # (c) threshold term
            if cfg.threshold_weight > 0 and fd.threshold_setpoint_pct is not None:
                sp_th_sim = self._find_sim_threshold(sp_sim, phi_sim, d_sim, cfg.threshold_phase_deg)
                if np.isfinite(sp_th_sim):
                    r = sp_th_sim - float(fd.threshold_setpoint_pct)
                    total += cfg.threshold_weight * float(r * r)
                else:
                    total += cfg.threshold_weight * float(cfg.bistable_jump_penalty)
                n_terms += 1

        return float(total / max(n_terms, 1))

    # ---------------- fit ----------------
    def fit(
        self,
        bounds: Dict[str, Tuple[float, float]],
        x0: Optional[Dict[str, float]] = None,
        method: str = "differential_evolution",
        de_maxiter: int = 50,
        seed: int = 0,
        workers: int = 1,
        polish: bool = True,
        verbose: bool = False,
        callback=None,
    ) -> Dict[str, Any]:
        names = list(bounds.keys())
        bnds = [bounds[n] for n in names]

        def vec_to_theta(x):
            return {n: float(x[i]) for i, n in enumerate(names)}

        def obj(x):
            v = self.loss(vec_to_theta(x))
            if verbose:
                print(f"  loss={v:.5f}  theta={vec_to_theta(x)}")
            return v

        if method == "differential_evolution":
            res = differential_evolution(
                obj, bounds=bnds, maxiter=de_maxiter, seed=seed,
                workers=workers, polish=False, callback=callback,
            )
            x_best = res.x.copy()
            f_best = float(res.fun)
            if polish:
                res_local = minimize(obj, x_best, method="Powell", bounds=bnds)
                if float(res_local.fun) < f_best:
                    x_best = res_local.x.copy()
                    f_best = float(res_local.fun)
        else:
            x0_arr = (np.array([x0[n] for n in names])
                      if x0 is not None else
                      np.array([0.5 * (b[0] + b[1]) for b in bnds]))
            res = minimize(obj, x0_arr, method=method, bounds=bnds)
            x_best = res.x.copy()
            f_best = float(res.fun)

        theta_best = vec_to_theta(x_best)
        return {
            "theta_best": theta_best,
            "loss_best": f_best,
            "names": names,
            "x_best": x_best,
            "result": res,
            "calibrated_params": self.locked_params(theta_best),
        }

    def locked_params(self, theta: Dict[str, float]) -> Dict[str, float]:
        """Return the base_params dict with material values overridden by theta."""
        out = dict(self.base_params)
        for k in self._MATERIAL_KEYS:
            if k in theta:
                out[k] = float(theta[k])
        return out

    # ---------------- visualization ----------------
    def plot_fit(self, theta, figsize=(11, 8), only_drives=None, ax=None):
        if ax is None:
            fig, ax = plt.subplots(figsize=figsize)
        else:
            fig = ax.figure
        curves = self.fd_curves if only_drives is None else [
            c for c in self.fd_curves
            if any(np.isclose(c.drive_si, d, rtol=1e-3) for d in only_drives)
        ]
        colors = plt.cm.viridis(np.linspace(0, 0.92, max(len(curves), 1)))
        for fd, col in zip(curves, colors):
            d_hat, A_hat, phi = self.simulate_curve(fd.drive_si, theta)
            sp_sim, phi_sim, d_sim, _ = self._sim_to_setpoint_axis(d_hat, A_hat, phi)
            label_drive = fd.label or f"drive={fd.drive_si*1e9:.1f} nm"
            ax.plot(fd.setpoint_pct, fd.phase_deg, "-", color=col, lw=1.6,
                    label=f"Exp {label_drive}")
            order = np.argsort(-d_sim)
            ax.plot(sp_sim[order], phi_sim[order], "--", color=col, lw=1.2, alpha=0.7)
            if fd.threshold_setpoint_pct is not None:
                ax.axvline(fd.threshold_setpoint_pct, color=col, ls=":", alpha=0.4)
            sp_th_sim = self._find_sim_threshold(sp_sim, phi_sim, d_sim,
                                                 self.config.threshold_phase_deg)
            if np.isfinite(sp_th_sim):
                ax.axvline(sp_th_sim, color=col, ls="-.", alpha=0.4)
        ax.set_xlabel("Setpoint (%)")
        ax.set_ylabel("Phase (deg)")
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 180)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, loc="best", ncol=2)
        ax.set_title(f"FD-only calibration  (loss={self.loss(theta):.3g})")
        fig.tight_layout()
        return fig, ax

    def plot_threshold_curve(self, theta, figsize=(6.5, 6.5), ax=None):
        if ax is None:
            fig, ax = plt.subplots(figsize=figsize)
        else:
            fig = ax.figure
        drives_nm = []
        sp_th_exp = []
        sp_th_sim = []
        for fd in self.fd_curves:
            if fd.threshold_setpoint_pct is None:
                continue
            d_hat, A_hat, phi = self.simulate_curve(fd.drive_si, theta)
            sp_sim, phi_sim, d_sim, _ = self._sim_to_setpoint_axis(d_hat, A_hat, phi)
            sp_th = self._find_sim_threshold(sp_sim, phi_sim, d_sim,
                                             self.config.threshold_phase_deg)
            drives_nm.append(fd.drive_si * 1e9)
            sp_th_exp.append(fd.threshold_setpoint_pct)
            sp_th_sim.append(sp_th)
        drives_nm = np.array(drives_nm)
        sp_th_exp = np.array(sp_th_exp)
        sp_th_sim = np.array(sp_th_sim)
        ax.plot(sp_th_exp, drives_nm, "ro", ms=10, label="Exp threshold")
        ax.plot(sp_th_sim, drives_nm, "b^", ms=10, label="Sim threshold")
        for x_e, x_s, y in zip(sp_th_exp, sp_th_sim, drives_nm):
            if np.isfinite(x_s):
                ax.plot([x_e, x_s], [y, y], "k-", lw=0.5, alpha=0.4)
        ax.set_xlabel("Threshold setpoint (%)")
        ax.set_ylabel("Drive amplitude (nm)")
        ax.legend()
        ax.grid(alpha=0.3)
        ax.set_title("Attractive→repulsive threshold")
        fig.tight_layout()
        return fig, ax

    def report(self, theta):
        """Per-curve diagnostics — useful for spotting which drives don't match."""
        rows = []
        for fd in self.fd_curves:
            d_hat, A_hat, phi = self.simulate_curve(fd.drive_si, theta)
            sp_sim, phi_sim, d_sim, A_free_hat = self._sim_to_setpoint_axis(d_hat, A_hat, phi)
            far_sim = float(phi_sim[int(np.argmax(d_sim))])
            sp_th_sim = self._find_sim_threshold(
                sp_sim, phi_sim, d_sim, self.config.threshold_phase_deg
            )
            m = (fd.setpoint_pct >= self.config.setpoint_min_pct) & (
                fd.setpoint_pct <= self.config.setpoint_max_pct
            )
            phi_at = self._interp_phase(
                sp_sim, phi_sim, d_sim, fd.setpoint_pct[m],
                branch=self.config.interp_branch,
                threshold_phase_deg=self.config.threshold_phase_deg,
            )
            valid = np.isfinite(phi_at)
            rmse = (
                float(np.sqrt(np.mean((phi_at[valid] - fd.phase_deg[m][valid]) ** 2)))
                if np.any(valid)
                else float("nan")
            )
            rows.append({
                "drive_nm": fd.drive_si * 1e9,
                "A_free_exp_nm": fd.A_free_nm,
                "A_free_sim_nm": A_free_hat,  # already in nm units in this normalization
                "free_phase_exp": fd.free_phase_deg,
                "free_phase_sim": far_sim,
                "thresh_exp_pct": fd.threshold_setpoint_pct,
                "thresh_sim_pct": sp_th_sim,
                "phase_rmse_deg": rmse,
            })
        return rows


# ============================================================
# 5) BOUNDS HELPER
# ============================================================
def suggest_bounds(
    base_params: Dict[str, float],
    fit_keys: Sequence[str] = ("E_star", "H", "a0", "phase_offset_deg"),
) -> Dict[str, Tuple[float, float]]:
    """Conservative bounds keyed off the user's base_params."""
    b: Dict[str, Tuple[float, float]] = {}
    if "E_star" in fit_keys:
        E0 = base_params["E_star"]
        b["E_star"] = (E0 * 0.2, E0 * 5.0)
    if "H" in fit_keys:
        H0 = base_params["H"]
        b["H"] = (H0 * 0.2, H0 * 5.0)
    if "a0" in fit_keys:
        a0_0 = base_params["a0"]
        b["a0"] = (a0_0 * 0.3, a0_0 * 3.0)
    if "R" in fit_keys:
        R0 = base_params["R"]
        b["R"] = (R0 * 0.5, R0 * 2.0)
    if "Q" in fit_keys:
        Q0 = base_params["Q"]
        b["Q"] = (Q0 * 0.7, Q0 * 1.3)
    if "k" in fit_keys:
        k0 = base_params["k"]
        b["k"] = (k0 * 0.5, k0 * 2.0)
    if "phase_offset_deg" in fit_keys:
        b["phase_offset_deg"] = (-30.0, 30.0)
    if "drive_scale" in fit_keys:
        b["drive_scale"] = (0.7, 1.4)
    return b


# ============================================================
# 6) DATA LOADER FOR THE COMMON 15-CURVES LAYOUT
# ============================================================
def build_curves_from_arrays(
    drives_si: Sequence[float],
    amplitude_nm: np.ndarray,        # shape (n_drive, n_pixel)
    phase_deg: np.ndarray,           # shape (n_drive, n_pixel)
    z_piezo_nm: Optional[np.ndarray] = None,    # shape (n_drive, n_pixel) or None
    threshold_pct: Optional[Sequence[Optional[float]]] = None,
    A_free_nm: Optional[Sequence[float]] = None,
    labels: Optional[Sequence[str]] = None,
) -> List[FDCalibrationCurve]:
    """Convenience loader: turn 2D arrays of (n_drive, n_pixel) into FDCalibrationCurve list."""
    drives_si = list(drives_si)
    amp = np.asarray(amplitude_nm, dtype=float)
    ph = np.asarray(phase_deg, dtype=float)
    if amp.shape != ph.shape:
        raise ValueError("amplitude_nm and phase_deg must share shape")
    if amp.shape[0] != len(drives_si):
        raise ValueError("amplitude_nm.shape[0] must equal len(drives_si)")
    z = np.asarray(z_piezo_nm, dtype=float) if z_piezo_nm is not None else None
    out: List[FDCalibrationCurve] = []
    for i, d in enumerate(drives_si):
        out.append(FDCalibrationCurve(
            drive_si=float(d),
            amplitude_nm=amp[i],
            phase_deg=ph[i],
            z_piezo_nm=None if z is None else z[i],
            threshold_setpoint_pct=None if threshold_pct is None else threshold_pct[i],
            A_free_nm=None if A_free_nm is None else float(A_free_nm[i]),
            label="" if labels is None else str(labels[i]),
        ))
    return out





In [ ]:
# Surrogate-accelerated FD calibration of material parameters (E_star, H, a0).
"""
fd_surrogate_calibration.py
===========================

Surrogate-accelerated FD calibration of material parameters (E_star, H, a0).

Strategy
--------
1. BUILD a coarse parameter library ONCE:
     for each (E_star_i, H_j, a0_k) on a regular grid,
     for each drive in drives_si,
        run the RK45 approach simulator -> store (A_hat[d_hat], phi[d_hat])
   The result is a 5-D table:  (n_E, n_H, n_a0, n_drive, n_d_hat)
   This step is parallelizable and cacheable to .npz.

2. AT OPTIMIZATION TIME, evaluate the loss by interpolating in the
   library (scipy.RegularGridInterpolator over (E_star, H, a0)).  No
   ODE solves.  A loss eval costs milliseconds, so DE / Powell run
   in seconds.

3. OPTIONAL VERIFICATION: at the end, run one full-fidelity simulation
   at the optimum and confirm the surrogate's prediction.

This pairs with `fd_only_calibration.py` (same FDCalibrationCurve schema,
same loss decomposition: phase-curve + free-phase + threshold).

Compute budget guideline
------------------------
Default grid 6 x 6 x 6 = 216 param combos x 15 drives = 3240 RK45 sims.
At ~0.5 s per sim (numba-jit, 160 d_hat points) -> ~25 min serial,
~3 min on 8 cores.  Re-use forever.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple

import os
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, minimize
from scipy.interpolate import RegularGridInterpolator

# ============================================================
# 1) PARAMETER LIBRARY
# ============================================================
@dataclass
class FDParamLibrary:
    """5-D pre-computed FD library indexed by (E_star, H, a0, drive, d_hat)."""
    E_grid: np.ndarray         # (n_E,)
    H_grid: np.ndarray         # (n_H,)
    a0_grid: np.ndarray        # (n_a0,)
    drives_si: np.ndarray      # (n_drive,)  drive amplitudes in m
    d_hat_grid: np.ndarray     # (n_d_hat,)  shared distance grid
    A_table: np.ndarray        # (n_E, n_H, n_a0, n_drive, n_d_hat)
    phi_table: np.ndarray      # same shape, phase in deg (NO offset applied)
    base_params: Dict[str, float]
    conversions: Dict[str, float]
    solver: Dict[str, Any] = field(default_factory=dict)
    attrs: Dict[str, Any] = field(default_factory=dict)

    # ---------- save / load ----------
    def save(self, path: str) -> None:
        np.savez_compressed(
            path,
            E_grid=self.E_grid, H_grid=self.H_grid, a0_grid=self.a0_grid,
            drives_si=self.drives_si, d_hat_grid=self.d_hat_grid,
            A_table=self.A_table, phi_table=self.phi_table,
            base_params_keys=np.array(list(self.base_params.keys())),
            base_params_vals=np.array(list(self.base_params.values()), dtype=float),
            conversions_keys=np.array(list(self.conversions.keys())),
            conversions_vals=np.array(list(self.conversions.values()), dtype=float),
        )

    @classmethod
    def load(cls, path: str) -> "FDParamLibrary":
        z = np.load(path, allow_pickle=False)
        bp = {str(k): float(v) for k, v in zip(z["base_params_keys"], z["base_params_vals"])}
        cv = {str(k): float(v) for k, v in zip(z["conversions_keys"], z["conversions_vals"])}
        return cls(
            E_grid=z["E_grid"], H_grid=z["H_grid"], a0_grid=z["a0_grid"],
            drives_si=z["drives_si"], d_hat_grid=z["d_hat_grid"],
            A_table=z["A_table"], phi_table=z["phi_table"],
            base_params=bp, conversions=cv,
        )

    # ---------- geometry ----------
    @property
    def shape(self):
        return self.A_table.shape  # (n_E, n_H, n_a0, n_drive, n_d_hat)


# ============================================================
# 2) BUILDER
# ============================================================
def _simulate_one(args):
    """Top-level helper for joblib / multiprocessing pickling."""
    (E, H, a0, drv, base_params, conversions, solver_kw,
     d_start_hat, d_end_hat, n_d_hat) = args
    p = dict(base_params)
    p["E_star"] = float(E)
    p["H"] = float(H)
    p["a0"] = float(a0)
    spm_params, param_norm = build_normalized_spm_params(
        params=p, conversions=conversions, drive_si=float(drv),
        omega_drive_hat=1.0, omega_ref_hat=1.0,
    )
    solver = RK45Options(**solver_kw)
    model = SPMRK45ApproachModel(spm_params, solver=solver)
    d0 = float(param_norm["A"]) * float(d_start_hat)
    curve = model.approach_curve(
        d_start_hat=d0, d_end_hat=float(d_end_hat),
        n_points=int(n_d_hat), z0_hat=0.0, v0_hat=0.0, warm_start=True,
    )
    # Re-sample onto a fixed *ascending* d_hat grid so all curves share the same axis
    # and so that np.interp / RegularGridInterpolator behave correctly downstream.
    d_grid = np.linspace(float(d_end_hat), float(d_start_hat * float(param_norm["A"])), int(n_d_hat))
    order = np.argsort(curve["d_hat"])
    d_sorted = curve["d_hat"][order]
    A_sorted = curve["A_hat"][order]
    phi_sorted = curve["phi_deg"][order]
    A_resamp = np.interp(d_grid, d_sorted, A_sorted)
    phi_resamp = np.interp(d_grid, d_sorted, phi_sorted)
    return A_resamp, phi_resamp, d_grid


def build_fd_param_library(
    base_params: Dict[str, float],
    conversions: Dict[str, float],
    E_grid: np.ndarray,
    H_grid: np.ndarray,
    a0_grid: np.ndarray,
    drives_si: np.ndarray,
    solver: Optional[RK45Options] = None,
    n_d_hat: int = 160,
    d_start_factor: float = 1.25,   # d_start_hat = drive_hat * factor
    d_end_hat: float = -8.0,
    n_jobs: int = 1,
    verbose: bool = True,
    cache_path: Optional[str] = None,
) -> FDParamLibrary:
    """
    Build an FD library on the (E_star, H, a0) x drive grid.

    If `cache_path` exists, the library is loaded from there and the build
    is skipped.  After building, the library is saved to `cache_path`.
    """
    if cache_path is not None and os.path.exists(cache_path):
        if verbose:
            print(f"[lib] loading cached library from {cache_path}")
        return FDParamLibrary.load(cache_path)

    E_grid = np.asarray(E_grid, dtype=float).ravel()
    H_grid = np.asarray(H_grid, dtype=float).ravel()
    a0_grid = np.asarray(a0_grid, dtype=float).ravel()
    drives_si = np.asarray(drives_si, dtype=float).ravel()

    n_E, n_H, n_a0, n_drv = E_grid.size, H_grid.size, a0_grid.size, drives_si.size
    total = n_E * n_H * n_a0 * n_drv

    if solver is None:
        solver = RK45Options()
    solver_kw = dict(
        N_cycles=solver.N_cycles, max_windows=solver.max_windows,
        settle_windows=solver.settle_windows, tol_A=solver.tol_A,
        tol_phi_deg=solver.tol_phi_deg, consecutive=solver.consecutive,
        demod_per_ref_cycle=solver.demod_per_ref_cycle, rtol=solver.rtol,
        atol=solver.atol, dt0=solver.dt0, dt_min=solver.dt_min, dt_max=solver.dt_max,
        max_steps_per_window=solver.max_steps_per_window,
    )

    # the d_hat grid depends on drive (via d_start_hat = drive_hat * factor),
    # so per-drive we'll use a normalized [0,1] axis and store d_hat too.
    # For interpolation simplicity we pick a SINGLE d_hat grid that spans
    # the largest drive's d_start; smaller drives get clipped/extrapolated.
    largest_drive_hat = float(drives_si.max() * conversions["A"])
    d_start_hat_global = largest_drive_hat * float(d_start_factor)
    # ASCENDING d_hat grid (low to high). Smaller drives only cover the lower
    # portion of this range; the upper portion will be NaN for those drives.
    d_hat_global = np.linspace(float(d_end_hat), d_start_hat_global, int(n_d_hat))

    A_table = np.full((n_E, n_H, n_a0, n_drv, n_d_hat), np.nan, dtype=float)
    phi_table = np.full_like(A_table, np.nan)

    # build job list
    jobs = []
    idx_map = []
    for iE, E in enumerate(E_grid):
        for iH, H in enumerate(H_grid):
            for ia, a0v in enumerate(a0_grid):
                for idr, drv in enumerate(drives_si):
                    jobs.append((
                        E, H, a0v, drv,
                        base_params, conversions, solver_kw,
                        d_start_factor, d_end_hat, n_d_hat,
                    ))
                    idx_map.append((iE, iH, ia, idr))

    if verbose:
        print(f"[lib] building {total} sims  "
              f"(n_E={n_E}, n_H={n_H}, n_a0={n_a0}, n_drive={n_drv}, n_d_hat={n_d_hat})  "
              f"on {n_jobs} job(s)")

    t0 = time.time()
    if n_jobs == 1:
        for k, args in enumerate(jobs):
            A_resamp, phi_resamp, d_grid = _simulate_one(args)
            iE, iH, ia, idr = idx_map[k]
            A_table[iE, iH, ia, idr, :] = np.interp(d_hat_global, d_grid, A_resamp,
                                                     left=np.nan, right=np.nan)
            phi_table[iE, iH, ia, idr, :] = np.interp(d_hat_global, d_grid, phi_resamp,
                                                       left=np.nan, right=np.nan)
            if verbose and (k + 1) % max(1, total // 20) == 0:
                el = time.time() - t0
                eta = el / (k + 1) * (total - k - 1)
                print(f"  [{k+1:5d}/{total}]  elapsed {el:6.1f}s  ETA {eta:6.1f}s")
    else:
        try:
            from joblib import Parallel, delayed
        except ImportError as exc:
            raise RuntimeError("Install joblib (pip install joblib) to use n_jobs > 1") from exc
        results = Parallel(n_jobs=n_jobs, verbose=10 if verbose else 0)(
            delayed(_simulate_one)(args) for args in jobs
        )
        for k, (A_resamp, phi_resamp, d_grid) in enumerate(results):
            iE, iH, ia, idr = idx_map[k]
            A_table[iE, iH, ia, idr, :] = np.interp(d_hat_global, d_grid, A_resamp,
                                                     left=np.nan, right=np.nan)
            phi_table[iE, iH, ia, idr, :] = np.interp(d_hat_global, d_grid, phi_resamp,
                                                       left=np.nan, right=np.nan)

    el = time.time() - t0
    if verbose:
        print(f"[lib] done in {el:.1f}s  ({el/total*1000:.1f} ms/sim)")

    lib = FDParamLibrary(
        E_grid=E_grid, H_grid=H_grid, a0_grid=a0_grid,
        drives_si=drives_si, d_hat_grid=d_hat_global,
        A_table=A_table, phi_table=phi_table,
        base_params=dict(base_params), conversions=dict(conversions),
        solver=solver_kw,
        attrs={"d_start_factor": float(d_start_factor),
               "d_end_hat": float(d_end_hat), "n_d_hat": int(n_d_hat),
               "build_time_s": float(el)},
    )
    if cache_path is not None:
        lib.save(cache_path)
        if verbose:
            print(f"[lib] saved to {cache_path}")
    return lib


# ============================================================
# 3) SURROGATE INTERPOLATOR
# ============================================================
class FDSurrogate:
    """
    Cheap evaluator: given (E_star, H, a0) -> per-drive (A_hat[d], phi[d]) curves.

    Uses scipy RegularGridInterpolator (linear) over the 3-D parameter grid;
    drive index and d_hat are sampled discretely.
    """

    def __init__(self, library: FDParamLibrary, method: str = "linear"):
        self.lib = library
        # Interpolators: one per (drive_idx, channel) returning the (n_d_hat,) vector.
        # Trick: pack (n_drive, n_d_hat) into the trailing dims and let RGI broadcast.
        # values shape -> (n_E, n_H, n_a0, n_drive*n_d_hat)
        n_E, n_H, n_a0, n_drv, n_d = library.A_table.shape
        A_flat = library.A_table.reshape(n_E, n_H, n_a0, n_drv * n_d)
        phi_flat = library.phi_table.reshape(n_E, n_H, n_a0, n_drv * n_d)
        pts = (library.E_grid, library.H_grid, library.a0_grid)
        self._interp_A = RegularGridInterpolator(pts, A_flat, method=method,
                                                   bounds_error=False, fill_value=np.nan)
        self._interp_phi = RegularGridInterpolator(pts, phi_flat, method=method,
                                                    bounds_error=False, fill_value=np.nan)
        self._n_drv = n_drv
        self._n_d = n_d

    def evaluate(self, E_star: float, H: float, a0: float):
        """Return (A_table[n_drive, n_d], phi_table[n_drive, n_d]) at (E*, H, a0)."""
        q = np.array([[float(E_star), float(H), float(a0)]])
        A = self._interp_A(q).reshape(self._n_drv, self._n_d)
        phi = self._interp_phi(q).reshape(self._n_drv, self._n_d)
        return A, phi


# ============================================================
# 4) SURROGATE CALIBRATOR
# ============================================================
@dataclass
class SurrogateCalibrationConfig:
    fit_keys: Tuple[str, ...] = ("E_star", "H", "a0", "phase_offset_deg")
    phase_weight: float = 1.0
    free_phase_weight: float = 2.0
    threshold_weight: float = 6.0
    setpoint_min_pct: float = 30.0
    setpoint_max_pct: float = 92.0
    threshold_phase_deg: float = 90.0
    jump_min_deg: float = 15.0
    bistable_jump_penalty: float = 100.0
    interp_branch: str = "auto"


class FDSurrogateCalibrator:
    """
    Same loss decomposition as FDOnlyCalibrator, but evaluates the forward
    model via interpolation in the precomputed library, NOT via the ODE.
    """

    def __init__(self,
                 library: FDParamLibrary,
                 fd_curves: Sequence[FDCalibrationCurve],
                 config: Optional[SurrogateCalibrationConfig] = None,
                 surrogate_method: str = "linear"):
        self.lib = library
        self.fd_curves = list(fd_curves)
        self.config = config if config is not None else SurrogateCalibrationConfig()
        self.surr = FDSurrogate(library, method=surrogate_method)

        # Map each experimental curve to a library drive index (nearest).
        self._drive_idx = []
        for fd in self.fd_curves:
            i = int(np.argmin(np.abs(self.lib.drives_si - float(fd.drive_si))))
            err_pct = abs(self.lib.drives_si[i] - fd.drive_si) / fd.drive_si * 100.0
            if err_pct > 5.0:
                print(f"[warn] drive {fd.drive_si*1e9:.2f} nm not in library "
                      f"(nearest = {self.lib.drives_si[i]*1e9:.2f} nm, err {err_pct:.1f}%)")
            self._drive_idx.append(i)

        # Auto-detect thresholds in the experimental data once.
        for fd in self.fd_curves:
            if fd.threshold_setpoint_pct is None:
                fd.threshold_setpoint_pct = detect_threshold_setpoint(
                    fd.setpoint_pct, fd.phase_deg,
                    self.config.threshold_phase_deg,
                    self.config.jump_min_deg,
                )

    # ---- branch-aware sim → setpoint axis interpolation ----
    @staticmethod
    def _sim_to_setpoint(d_hat, A_hat, phi):
        # Mask out any NaN entries (small drives have NaN at high d_hat in the
        # global library grid because they don't reach those distances).
        finite = np.isfinite(A_hat) & np.isfinite(phi) & np.isfinite(d_hat)
        if not np.any(finite):
            return None, None, None, np.nan
        d_v = d_hat[finite]
        A_v = A_hat[finite]
        phi_v = phi[finite]
        far = int(np.argmax(d_v))
        A_free = float(A_v[far])
        if A_free <= 0 or not np.isfinite(A_free):
            return None, None, None, np.nan
        return A_v / A_free * 100.0, phi_v, d_v, A_free

    @staticmethod
    def _interp_phase_at(sp_sim, phi_sim, sp_query):
        order = np.argsort(sp_sim)
        sp_s = sp_sim[order]
        phi_s = phi_sim[order]
        keep = np.concatenate([[True], np.diff(sp_s) > 1e-9])
        return np.interp(sp_query, sp_s[keep], phi_s[keep], left=np.nan, right=np.nan)

    @staticmethod
    def _find_threshold_sp(sp_sim, phi_sim, d_sim, threshold_phase_deg=90.0):
        order = np.argsort(-d_sim)
        sp = sp_sim[order]
        ph = phi_sim[order]
        for i in range(1, ph.size):
            if ph[i - 1] >= threshold_phase_deg > ph[i]:
                if ph[i] == ph[i - 1]:
                    return float(0.5 * (sp[i - 1] + sp[i]))
                t = (threshold_phase_deg - ph[i - 1]) / (ph[i] - ph[i - 1])
                return float(sp[i - 1] + t * (sp[i] - sp[i - 1]))
        return np.nan

    # ---- loss ----
    def loss(self, theta: Dict[str, float]) -> float:
        cfg = self.config
        E = float(theta.get("E_star", self.lib.base_params["E_star"]))
        H = float(theta.get("H", self.lib.base_params["H"]))
        a0 = float(theta.get("a0", self.lib.base_params["a0"]))
        phase_offset = float(theta.get("phase_offset_deg", 0.0))

        try:
            A_all, phi_all = self.surr.evaluate(E, H, a0)
        except Exception:
            return 1e20

        d_sim = self.lib.d_hat_grid
        total = 0.0
        n_terms = 0

        for k, fd in enumerate(self.fd_curves):
            j = self._drive_idx[k]
            A_hat = A_all[j]
            phi = phi_all[j] + phase_offset
            sp_sim, phi_sim, d_arr, A_free_hat = self._sim_to_setpoint(d_sim, A_hat, phi)
            if sp_sim is None:
                # surrogate returned no valid samples for this drive; skip it
                continue

            # phase curve term
            if cfg.phase_weight > 0:
                m = (fd.setpoint_pct >= cfg.setpoint_min_pct) & (fd.setpoint_pct <= cfg.setpoint_max_pct)
                if np.any(m):
                    phi_at = self._interp_phase_at(sp_sim, phi_sim, fd.setpoint_pct[m])
                    valid = np.isfinite(phi_at)
                    if np.any(valid):
                        r = phi_at[valid] - fd.phase_deg[m][valid]
                        total += cfg.phase_weight * float(np.mean(r * r))
                        n_terms += 1

            # free-phase term
            if cfg.free_phase_weight > 0:
                far_sim = float(phi_sim[int(np.argmax(d_arr))])
                r = far_sim - float(fd.free_phase_deg)
                total += cfg.free_phase_weight * float(r * r)
                n_terms += 1

            # threshold term
            if cfg.threshold_weight > 0 and fd.threshold_setpoint_pct is not None:
                sp_th_sim = self._find_threshold_sp(sp_sim, phi_sim, d_arr, cfg.threshold_phase_deg)
                if np.isfinite(sp_th_sim):
                    r = sp_th_sim - float(fd.threshold_setpoint_pct)
                    total += cfg.threshold_weight * float(r * r)
                else:
                    total += cfg.threshold_weight * cfg.bistable_jump_penalty
                n_terms += 1

        return float(total / max(n_terms, 1))

    # ---- fit ----
    def fit(self, bounds: Dict[str, Tuple[float, float]],
            method: str = "differential_evolution",
            de_maxiter: int = 60, seed: int = 0, polish: bool = True,
            x0: Optional[Dict[str, float]] = None, verbose: bool = False):
        names = list(bounds.keys())
        bnds = [bounds[n] for n in names]

        def vec_to_theta(x):
            return {n: float(x[i]) for i, n in enumerate(names)}

        def obj(x):
            v = self.loss(vec_to_theta(x))
            if verbose:
                print(f"  loss={v:.5f}  theta={vec_to_theta(x)}")
            return v

        if method == "differential_evolution":
            res = differential_evolution(obj, bounds=bnds, maxiter=de_maxiter, seed=seed,
                                          polish=False)
            x_best = res.x.copy()
            f_best = float(res.fun)
            if polish:
                res2 = minimize(obj, x_best, method="Powell", bounds=bnds)
                if float(res2.fun) < f_best:
                    x_best, f_best = res2.x.copy(), float(res2.fun)
        else:
            x0_arr = (np.array([x0[n] for n in names])
                      if x0 is not None else
                      np.array([0.5 * (b[0] + b[1]) for b in bnds]))
            res = minimize(obj, x0_arr, method=method, bounds=bnds)
            x_best = res.x.copy()
            f_best = float(res.fun)

        return {
            "theta_best": vec_to_theta(x_best),
            "loss_best": f_best,
            "result": res,
        }

    # ---- diagnostic plot ----
    def plot_fit(self, theta, figsize=(11, 8), only_drive_indices=None):
        E = float(theta.get("E_star", self.lib.base_params["E_star"]))
        H = float(theta.get("H", self.lib.base_params["H"]))
        a0 = float(theta.get("a0", self.lib.base_params["a0"]))
        po = float(theta.get("phase_offset_deg", 0.0))
        A_all, phi_all = self.surr.evaluate(E, H, a0)
        d_sim = self.lib.d_hat_grid

        fig, ax = plt.subplots(figsize=figsize)
        idxs = range(len(self.fd_curves)) if only_drive_indices is None else only_drive_indices
        colors = plt.cm.viridis(np.linspace(0, 0.92, len(list(idxs))))
        for col, k in zip(colors, idxs):
            fd = self.fd_curves[k]
            j = self._drive_idx[k]
            A_hat, phi = A_all[j], phi_all[j] + po
            sp_sim, phi_sim, d_arr, _ = self._sim_to_setpoint(d_sim, A_hat, phi)
            ax.plot(fd.setpoint_pct, fd.phase_deg, "-", color=col, lw=1.6,
                    label=f"Exp d={fd.drive_si*1e9:.0f}nm")
            if sp_sim is None:
                # surrogate had no valid samples for this drive; show exp only
                continue
            order = np.argsort(-d_arr)
            ax.plot(sp_sim[order], phi_sim[order], "--", color=col, lw=1.2, alpha=0.75)
        ax.set_xlabel("Setpoint (%)")
        ax.set_ylabel("Phase (deg)")
        ax.set_xlim(0, 100); ax.set_ylim(0, 180)
        ax.grid(alpha=0.3); ax.legend(fontsize=8, ncol=2)
        ax.set_title(f"Surrogate fit  loss={self.loss(theta):.3g}  "
                      f"(E={E:.2g}, H={H:.2g}, a0={a0:.2e})")
        fig.tight_layout()
        return fig, ax




In [ ]:

# ============================================================
# 5) END-TO-END WORKFLOW EXAMPLE
# ============================================================
if __name__ == "__main__":
    # ---- (i) base setup, identical to your main script ----
    base_params = {
        "k": 40.0, "A": 100e-9, "d0": 100e-9 * 1.25,
        "a0": 0.164e-9, "R": 10e-9, "f0": 300e3, "Q": 150,
        "E_star": 1 / ((1 - 0.3 ** 2) / 130e9 + (1 - 0.3 ** 2) / 1.2e9),
        "H": 7.1e-20,
    }
    conversions = {
        "k": 1, "A": 1e9, "d0": 1e9, "a0": 1e9, "R": 1e9,
        "f0": 1 / (2 * np.pi * base_params["f0"]), "Q": 1,
        "E_star": 1e-9 / base_params["k"],
        "H": 1 / (base_params["k"] * (1e-9) ** 2),
    }
    solver = RK45Options(N_cycles=15, max_windows=25, settle_windows=1,
                          tol_A=2e-4, tol_phi_deg=2e-2, consecutive=2,
                          demod_per_ref_cycle=24, rtol=1e-5, atol=1e-8,
                          dt0=0.02, dt_min=1e-10, dt_max=0.3,
                          max_steps_per_window=120000)

    # drives_si = np.array([7, 13, 18, 24, 29, 34, 39, 45, 50, 53,
    #                       66, 79, 92, 105, 117], dtype=float) * 1e-9

    # ---- (ii) BUILD library ONCE (cached after first run) ----
    # Use a SUBSET of 5-6 drives for material fitting; library stores them all.
    E0 = base_params["E_star"]
    H0 = base_params["H"]
    a0_0 = base_params["a0"]
    E_grid = np.geomspace(E0 * 1e-2, E0 * 1e2, 21)     # log-spaced (~3 decades robust)
    H_grid = np.geomspace(H0 * 1e-2, H0 * 1e2, 21)
    a0_grid = np.linspace(a0_0 * 0.25, a0_0 * 2.0, 11)  # linear
    lib = build_fd_param_library(
        base_params=base_params, conversions=conversions,
        E_grid=E_grid, H_grid=H_grid, a0_grid=a0_grid,
        drives_si=drives_si, solver=solver,
        n_d_hat=51, d_start_factor=1.25, d_end_hat=10,
        n_jobs=8,                    # bump to 4-8 if joblib available
        cache_path=None,
        verbose=True,
    )

    # ---- (iii) load YOUR experimental FD curves ----
    # Replace this stub with your actual loader (see fast-calibration script).
    n_pix = 128
    # amplitude_nm = np.empty((drives_si.size, n_pix))
    # phase_deg = np.empty((drives_si.size, n_pix))
    # for i, drv in enumerate(drives_si):
    #     sp = np.linspace(95, 5, n_pix)
    #     amplitude_nm[i] = drv * 1e9 * sp / 100.0
    #     phase_deg[i] = 90.0 + 18.0 * np.exp(-((sp - 70) ** 2) / 200.0) - 55.0 * (sp < 50)
    fd_curves = [
        FDCalibrationCurve(drive_si=float(drives_si[i]),
                            amplitude_nm=amplitude_nm[i], phase_deg=phase_deg[i])
        for i in range(drives_si.size)
    ]

    # ---- (iv) instantiate surrogate calibrator and FIT (seconds!) ----
    cfg = SurrogateCalibrationConfig(
        fit_keys=("E_star", "H", "a0", "phase_offset_deg"),
        phase_weight=1.0, free_phase_weight=2.0, threshold_weight=8.0,
        setpoint_min_pct=30.0, setpoint_max_pct=92.0,
    )
    cal = FDSurrogateCalibrator(lib, fd_curves, cfg, surrogate_method="linear")
    bounds = {
        "E_star": (E_grid.min(), E_grid.max()),
        "H":      (H_grid.min(), H_grid.max()),
        "a0":     (a0_grid.min(), a0_grid.max()),
        "phase_offset_deg": (-20.0, 20.0),
    }

    t0 = time.time()
    res = cal.fit(bounds, method="differential_evolution", de_maxiter=80,
                   seed=0, polish=True, verbose=False)
    print(f"[fit] done in {time.time()-t0:.1f}s    loss={res['loss_best']:.4f}")
    print("theta_best =", res["theta_best"])

    cal.plot_fit(res["theta_best"])
    plt.show()

In [ ]:
for i in range(len(fwds)):
    tp = fwds[i]
    height[i] = (tp[0] - np.min(tp[0][-L:]))[-L:]*1e9
    amp[i] = tp[1][-L:][::-1]
    phase[i] = tp[2][-L:][::-1]

In [ ]:
fd_curves = [
    FDCalibrationCurve(drive_si=float(drives_si[i]),
                        amplitude_nm=amp[i], phase_deg=phase[i])
    for i in range(drives_si.size)
]

# ---- (iv) instantiate surrogate calibrator and FIT (seconds!) ----
cfg = SurrogateCalibrationConfig(
    fit_keys=("E_star", "H", "a0", "phase_offset_deg"),
    phase_weight=1.0, free_phase_weight=1.0, threshold_weight=8.0,
    setpoint_min_pct=30.0, setpoint_max_pct=92.0,
)
cal = FDSurrogateCalibrator(lib, fd_curves, cfg, surrogate_method="linear")
bounds = {
    "E_star": (E_grid.min(), E_grid.max()),
    "H":      (H_grid.min(), H_grid.max()),
    "a0":     (a0_grid.min(), a0_grid.max()),
    "phase_offset_deg": (-20.0, 20.0),
}

t0 = time.time()
res = cal.fit(bounds, method="differential_evolution", de_maxiter=80,
               seed=0, polish=True, verbose=False)
print(f"[fit] done in {time.time()-t0:.1f}s    loss={res['loss_best']:.4f}")
print("theta_best =", res["theta_best"])

cal.plot_fit(res["theta_best"])
# plt.xlim(40, 120)
plt.show()

In [ ]:
# Approximate H and E from fd directly
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, Optional, Tuple, Any

try:
    from scipy.signal import savgol_filter
except Exception:
    savgol_filter = None


def _wrap_phase_deg(x):
    x = np.asarray(x, dtype=float)
    return (x + 180.0) % 360.0 - 180.0


def _unwrap_phase_deg(phi_deg):
    phi_deg = np.asarray(phi_deg, dtype=float)
    return np.rad2deg(np.unwrap(np.deg2rad(phi_deg)))


def _valid_window(n: int, requested: int, polyorder: int = 3) -> int:
    if n < 5:
        return max(3, n if n % 2 == 1 else n - 1)
    w = min(requested, n if n % 2 == 1 else n - 1)
    if w <= polyorder:
        w = polyorder + 2
        if w % 2 == 0:
            w += 1
        if w >= n:
            w = n if n % 2 == 1 else n - 1
    return max(3, w)


def smooth_1d(y, window: int = 11, polyorder: int = 3):
    y = np.asarray(y, dtype=float)
    if y.size < 5 or savgol_filter is None:
        return y.copy()
    w = _valid_window(len(y), window, polyorder)
    if w < 3:
        return y.copy()
    p = min(polyorder, w - 1)
    return savgol_filter(y, window_length=w, polyorder=p, mode="interp")


def _far_field_indices(d: np.ndarray, n_far: int, side: str):
    n = len(d)
    if side == "start":
        return np.arange(n_far)
    if side == "end":
        return np.arange(n - n_far, n)
    if side == "largest_d":
        return np.argsort(d)[::-1][:n_far]
    if side == "smallest_d":
        return np.argsort(d)[:n_far]
    raise ValueError("side must be one of: start, end, largest_d, smallest_d")


def _estimate_transition_index(
    d: np.ndarray,
    s_smooth: np.ndarray,
    phi_unwrap_smooth: np.ndarray,
    dphi_dd: np.ndarray,
    ds_dd: np.ndarray,
    phi_free_deg: float,
    mode: str = "phase_derivative",
) -> int:
    # Restrict to the actual transition zone: avoid flat far-field/near-field tails.
    phi_dev = np.abs(_wrap_phase_deg(phi_unwrap_smooth - phi_free_deg))
    s = np.asarray(s_smooth, dtype=float)
    score_mask = (phi_dev > max(3.0, 0.05 * np.nanmax(phi_dev))) | (
        np.abs(ds_dd) > 0.05 * np.nanmax(np.abs(ds_dd))
    )
    if not np.any(score_mask):
        return int(np.argmax(np.abs(dphi_dd)))

    idx = np.where(score_mask)[0]
    if mode == "phase_derivative":
        loc = idx[np.argmax(np.abs(dphi_dd[idx]))]
    elif mode == "amp_derivative":
        loc = idx[np.argmax(np.abs(ds_dd[idx]))]
    elif mode == "combined":
        p = np.abs(dphi_dd)
        a = np.abs(ds_dd)
        p = p / (np.nanmax(p) + 1e-12)
        a = a / (np.nanmax(a) + 1e-12)
        score = p + a
        loc = idx[np.argmax(score[idx])]
    elif mode == "mid_setpoint":
        loc = int(np.argmin(np.abs(s - 0.5 * (np.nanmax(s) + np.nanmin(s)))))
    else:
        raise ValueError("Unknown transition mode")
    return int(loc)


def analyze_single_fd_curve(
    d,
    A,
    phi,
    *,
    drive_nm: float,
    Q: float,
    R: Optional[float] = None,
    a0: float = 0.0,
    k_cantilever: float = 1.0,
    far_field_frac: float = 0.1,
    far_field_side: str = "largest_d",
    smooth_window: int = 11,
    smooth_polyorder: int = 3,
    transition_mode: str = "phase_derivative",
    separation_offset: Optional[float] = None,
    repulsive_eval: str = "closest",
) -> Dict[str, Any]:
    """Analyze one measured FD curve.

    Parameters
    ----------
    d, A, phi : arrays in physical units (nm, nm, deg).
    drive_nm  : nominal/free drive amplitude for this curve.
    Q         : cantilever quality factor used in the approximations.
    R         : tip radius in the same length unit as d and a0.
    a0        : DMT cutoff distance in same length unit as d.
    k_cantilever : cantilever stiffness (only used for E* proxy if requested).
    separation_offset : if provided, true separation D = d + separation_offset.
                        If None, only relative-distance proxies are reported.
    repulsive_eval : how to choose the point used for the E* proxy.
        'closest'       : use the smallest-d point, usually deepest indentation.
        'max_phase_dev' : use the point with largest |phi - phi_free| on the
                          closer-than-transition side.

    Returns
    -------
    dict of scalar descriptors + selected smoothed arrays for diagnostics.
    """
    d = np.asarray(d, dtype=float).ravel()
    A = np.asarray(A, dtype=float).ravel()
    phi = np.asarray(phi, dtype=float).ravel()
    mask = np.isfinite(d) & np.isfinite(A) & np.isfinite(phi)
    d, A, phi = d[mask], A[mask], phi[mask]
    if d.size < 7:
        raise ValueError("Need at least 7 valid samples in each curve")

    order = np.argsort(d)
    d, A, phi = d[order], A[order], phi[order]

    n_far = max(3, int(np.ceil(far_field_frac * d.size)))
    idx_far = _far_field_indices(d, n_far, far_field_side)

    A0 = float(np.nanmedian(A[idx_far]))
    phi_free = float(np.nanmedian(phi[idx_far]))

    s = A / max(A0, 1e-12)
    phi_unwrap = _unwrap_phase_deg(phi)
    phi_unwrap = phi_unwrap - np.nanmedian(phi_unwrap[idx_far]) + phi_free

    s_s = smooth_1d(s, window=smooth_window, polyorder=smooth_polyorder)
    A_s = s_s * A0
    phi_u_s = smooth_1d(phi_unwrap, window=smooth_window, polyorder=smooth_polyorder)

    ds_dd = np.gradient(s_s, d)
    dA_dd = np.gradient(A_s, d)
    dphi_dd = np.gradient(phi_u_s, d)

    i_tr = _estimate_transition_index(
        d,
        s_s,
        phi_u_s,
        dphi_dd,
        ds_dd,
        phi_free_deg=phi_free,
        mode=transition_mode,
    )

    d_tr = float(d[i_tr])
    A_tr = float(A_s[i_tr])
    s_tr = float(s_s[i_tr])
    phi_tr = float(phi_u_s[i_tr])
    dphi_tr = float(_wrap_phase_deg(phi_tr - phi_free))

    slope_s_tr = float(ds_dd[i_tr])
    slope_A_tr = float(dA_dd[i_tr])
    slope_phi_tr = float(dphi_dd[i_tr])
    max_abs_slope_s = float(np.nanmax(np.abs(ds_dd)))
    max_abs_slope_A = float(np.nanmax(np.abs(dA_dd)))
    max_abs_slope_phi = float(np.nanmax(np.abs(dphi_dd)))

    # Effective gradient estimates from the approximations.
    s_clip = float(np.clip(s_tr, 1e-6, 0.999999))
    kappa_from_setpoint = float(np.sqrt(1.0 / s_clip**2 - 1.0) / Q)
    kappa_from_phase = float(np.tan(np.deg2rad(abs(dphi_tr))) / Q)

    # Distance proxies. If offset is provided, use absolute separation D = d + offset.
    d_rel = d - np.min(d)
    D_rel_tr = float(d_rel[i_tr])
    if separation_offset is None:
        D_abs_tr = np.nan
    else:
        D_abs_tr = float(d_tr + separation_offset)

    # Attractive-branch Hamaker proxies.
    # Using |kappa| ~ 2 C1 / (D + a0)^3, C1 = H R / 6 -> H ~ 3*kappa*(D+a0)^3 / R
    if R is not None and R > 0:
        H_proxy_rel_setpoint = float(3.0 * kappa_from_setpoint * (D_rel_tr + a0) ** 3 / R)
        H_proxy_rel_phase = float(3.0 * kappa_from_phase * (D_rel_tr + a0) ** 3 / R)
        if np.isfinite(D_abs_tr):
            H_proxy_abs_setpoint = float(3.0 * kappa_from_setpoint * (D_abs_tr + a0) ** 3 / R)
            H_proxy_abs_phase = float(3.0 * kappa_from_phase * (D_abs_tr + a0) ** 3 / R)
        else:
            H_proxy_abs_setpoint = np.nan
            H_proxy_abs_phase = np.nan
    else:
        H_proxy_rel_setpoint = np.nan
        H_proxy_rel_phase = np.nan
        H_proxy_abs_setpoint = np.nan
        H_proxy_abs_phase = np.nan

    # Relative repulsive stiffness proxy from the phase-slope sharpness.
    # This is not an absolute E* unless indentation/contact is known.
    repulsive_sharpness_proxy = float(max_abs_slope_phi)

    # ------------------------------------------------------------------
    # Repulsive-branch E* proxy
    # ------------------------------------------------------------------
    # Approximation:
    #     kappa_rep = k_ts / k_cantilever ~ 2 E* sqrt(R * delta) / k_cantilever
    # so:
    #     E* ~ k_cantilever * kappa_rep / [2 sqrt(R * delta)]
    #
    # Here delta is only a proxy unless the true contact point is known.
    # We use the detected transition point as a contact/onset proxy and evaluate
    # the repulsive branch at a closer point. For separation-like curves where
    # smaller d is closer to the sample:
    #     delta_proxy = d_transition - d_repulsive_eval.
    #
    # If d and R are in nm and k is in N/m, this E* proxy is not SI Pa until
    # lengths are converted to meters. If d and R are normalized, E* is in your
    # simulator-normalized units.
    if repulsive_eval == "closest":
        i_rep = int(np.argmin(d))
    elif repulsive_eval == "max_phase_dev":
        closer = np.where(d <= d_tr)[0]
        if closer.size == 0:
            closer = np.arange(d.size)
        phi_dev_all = np.abs(_wrap_phase_deg(phi_u_s - phi_free))
        i_rep = int(closer[np.nanargmax(phi_dev_all[closer])])
    else:
        raise ValueError("repulsive_eval must be 'closest' or 'max_phase_dev'")

    d_rep = float(d[i_rep])
    s_rep = float(np.clip(s_s[i_rep], 1e-6, 0.999999))
    phi_rep = float(phi_u_s[i_rep])
    dphi_rep = float(_wrap_phase_deg(phi_rep - phi_free))

    delta_rep_proxy = float(max(d_tr - d_rep, np.finfo(float).eps))
    kappa_rep_from_setpoint = float(np.sqrt(1.0 / s_rep**2 - 1.0) / Q)
    kappa_rep_from_phase = float(np.tan(np.deg2rad(abs(dphi_rep))) / Q)

    if R is not None and R > 0 and delta_rep_proxy > 0:
        E_star_proxy_rel_from_setpoint = float(
            k_cantilever * kappa_rep_from_setpoint
            / (2.0 * np.sqrt(float(R) * delta_rep_proxy))
        )
        E_star_proxy_rel_from_phase = float(
            k_cantilever * kappa_rep_from_phase
            / (2.0 * np.sqrt(float(R) * delta_rep_proxy))
        )
    else:
        E_star_proxy_rel_from_setpoint = np.nan
        E_star_proxy_rel_from_phase = np.nan

    # Included for symmetry with the H outputs. A constant separation offset does
    # not change delta_proxy if the transition point is used as contact onset.
    E_star_proxy_abs_from_setpoint = E_star_proxy_rel_from_setpoint
    E_star_proxy_abs_from_phase = E_star_proxy_rel_from_phase

    out = {
        "drive_nm": float(drive_nm),
        "A0_nm": A0,
        "phi_free_deg": phi_free,
        "d_transition_nm": d_tr,
        "d_transition_rel_nm": D_rel_tr,
        "A_transition_nm": A_tr,
        "setpoint_transition": s_tr,
        "phi_transition_deg": phi_tr,
        "delta_phi_transition_deg": dphi_tr,
        "kappa_from_setpoint": kappa_from_setpoint,
        "kappa_from_phase": kappa_from_phase,
        "slope_s_transition_per_nm": slope_s_tr,
        "slope_A_transition_nm_per_nm": slope_A_tr,
        "slope_phi_transition_deg_per_nm": slope_phi_tr,
        "max_abs_slope_s_per_nm": max_abs_slope_s,
        "max_abs_slope_A_nm_per_nm": max_abs_slope_A,
        "max_abs_slope_phi_deg_per_nm": max_abs_slope_phi,
        "H_proxy_rel_from_setpoint": H_proxy_rel_setpoint,
        "H_proxy_rel_from_phase": H_proxy_rel_phase,
        "H_proxy_abs_from_setpoint": H_proxy_abs_setpoint,
        "H_proxy_abs_from_phase": H_proxy_abs_phase,
        "d_repulsive_eval_nm": d_rep,
        "delta_repulsive_proxy_nm": delta_rep_proxy,
        "setpoint_repulsive_eval": s_rep,
        "delta_phi_repulsive_eval_deg": dphi_rep,
        "kappa_rep_from_setpoint": kappa_rep_from_setpoint,
        "kappa_rep_from_phase": kappa_rep_from_phase,
        "E_star_proxy_rel_from_setpoint": E_star_proxy_rel_from_setpoint,
        "E_star_proxy_rel_from_phase": E_star_proxy_rel_from_phase,
        "E_star_proxy_abs_from_setpoint": E_star_proxy_abs_from_setpoint,
        "E_star_proxy_abs_from_phase": E_star_proxy_abs_from_phase,
        "repulsive_sharpness_proxy": repulsive_sharpness_proxy,
        # diagnostic arrays
        "_d": d,
        "_A": A,
        "_phi": phi,
        "_s_smooth": s_s,
        "_A_smooth": A_s,
        "_phi_unwrap_smooth": phi_u_s,
        "_ds_dd": ds_dd,
        "_dA_dd": dA_dd,
        "_dphi_dd": dphi_dd,
        "_i_transition": i_tr,
    }
    return out


def analyze_fd_collection(
    measured_fd: Dict[float, Dict[str, np.ndarray]],
    *,
    Q: float,
    R: Optional[float] = None,
    a0: float = 0.0,
    k_cantilever: float = 1.0,
    far_field_frac: float = 0.1,
    far_field_side: str = "largest_d",
    smooth_window: int = 11,
    smooth_polyorder: int = 3,
    transition_mode: str = "phase_derivative",
    separation_offsets: Optional[Dict[float, float]] = None,
    repulsive_eval: str = "closest",
) -> Tuple[pd.DataFrame, Dict[float, Dict[str, Any]]]:
    """Analyze all curves in measured_fd.

    measured_fd format:
        {drive_nm: {'d': ..., 'A': ..., 'phi': ...}, ...}
    """
    rows = []
    diag = {}
    for drive_nm in sorted(measured_fd.keys()):
        c = measured_fd[drive_nm]
        sep_off = None if separation_offsets is None else separation_offsets.get(float(drive_nm), None)
        res = analyze_single_fd_curve(
            c["d"], c["A"], c["phi"],
            drive_nm=float(drive_nm),
            Q=Q,
            R=R,
            a0=a0,
            k_cantilever=k_cantilever,
            far_field_frac=far_field_frac,
            far_field_side=far_field_side,
            smooth_window=smooth_window,
            smooth_polyorder=smooth_polyorder,
            transition_mode=transition_mode,
            separation_offset=sep_off,
            repulsive_eval=repulsive_eval,
        )
        diag[float(drive_nm)] = res
        scalar_res = {k: v for k, v in res.items() if not k.startswith("_")}
        rows.append(scalar_res)
    df = pd.DataFrame(rows).sort_values("drive_nm").reset_index(drop=True)
    return df, diag


def plot_fd_parameter_summary(df: pd.DataFrame):
    """Return a dict of matplotlib figures for key derived parameters vs drive."""
    figs = {}
    drive = df["drive_nm"].to_numpy()

    def _make_plot(ycols, ylabel, title):
        fig, ax = plt.subplots(figsize=(6.5, 4.5))
        for col in ycols:
            if col in df.columns:
                ax.plot(drive, df[col].to_numpy(), marker="o", label=col)
        ax.set_xlabel("Drive amplitude (nm)")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        if len(ycols) > 1:
            ax.legend(frameon=False)
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        return fig

    figs["transition_setpoint"] = _make_plot(
        ["setpoint_transition"], "Transition setpoint A/A0", "Transition setpoint vs drive"
    )
    figs["transition_phase"] = _make_plot(
        ["delta_phi_transition_deg"], "Transition phase shift (deg)", "Transition phase shift vs drive"
    )
    figs["transition_distance"] = _make_plot(
        ["d_transition_nm", "d_transition_rel_nm"], "Distance (nm)", "Transition distance vs drive"
    )
    figs["amp_slope"] = _make_plot(
        ["max_abs_slope_A_nm_per_nm", "slope_A_transition_nm_per_nm"],
        "Amplitude-distance slope", "Amplitude-distance slope vs drive"
    )
    figs["effective_gradient"] = _make_plot(
        ["kappa_from_setpoint", "kappa_from_phase"],
        "Effective gradient proxy", "Effective force-gradient proxy vs drive"
    )
    h_cols = [c for c in ["H_proxy_rel_from_setpoint", "H_proxy_rel_from_phase",
                          "H_proxy_abs_from_setpoint", "H_proxy_abs_from_phase"]
              if c in df.columns and np.any(np.isfinite(df[c].to_numpy()))]
    if h_cols:
        figs["hamaker_proxy"] = _make_plot(
            h_cols, "Hamaker proxy", "Attractive-branch H proxy vs drive"
        )

    e_cols = [c for c in ["E_star_proxy_rel_from_setpoint", "E_star_proxy_rel_from_phase",
                          "E_star_proxy_abs_from_setpoint", "E_star_proxy_abs_from_phase"]
              if c in df.columns and np.any(np.isfinite(df[c].to_numpy()))]
    if e_cols:
        figs["modulus_proxy"] = _make_plot(
            e_cols, "E* proxy", "Repulsive-branch E* proxy vs drive"
        )

    figs["repulsive_sharpness"] = _make_plot(
        ["repulsive_sharpness_proxy", "max_abs_slope_phi_deg_per_nm"],
        "Repulsive sharpness proxy", "Repulsive-branch sharpness vs drive"
    )
    return figs


def plot_curve_diagnostics(diag: Dict[float, Dict[str, Any]], drive_nm: float):
    """Plot one curve with transition markers and derivatives."""
    key = float(drive_nm)
    if key not in diag:
        raise KeyError(f"drive {drive_nm} not found in diagnostics")
    r = diag[key]
    d = r["_d"]
    i = r["_i_transition"]

    fig1, ax1 = plt.subplots(figsize=(6.5, 4.5))
    ax1.plot(d, r["_A"], label="A raw")
    ax1.plot(d, r["_A_smooth"], label="A smooth")
    ax1.axvline(d[i], linestyle="--", label="transition")
    ax1.set_xlabel("Distance (nm)")
    ax1.set_ylabel("Amplitude (nm)")
    ax1.set_title(f"Amplitude curve diagnostics @ drive={drive_nm:g} nm")
    ax1.legend(frameon=False)
    ax1.grid(True, alpha=0.3)
    fig1.tight_layout()

    fig2, ax2 = plt.subplots(figsize=(6.5, 4.5))
    ax2.plot(d, r["_phi"], label="phase raw")
    ax2.plot(d, r["_phi_unwrap_smooth"], label="phase smooth")
    ax2.axvline(d[i], linestyle="--", label="transition")
    ax2.set_xlabel("Distance (nm)")
    ax2.set_ylabel("Phase (deg)")
    ax2.set_title(f"Phase curve diagnostics @ drive={drive_nm:g} nm")
    ax2.legend(frameon=False)
    ax2.grid(True, alpha=0.3)
    fig2.tight_layout()

    fig3, ax3 = plt.subplots(figsize=(6.5, 4.5))
    ax3.plot(d, np.abs(r["_dA_dd"]), label="|dA/dd|")
    ax3.plot(d, np.abs(r["_dphi_dd"]), label="|dphi/dd|")
    ax3.axvline(d[i], linestyle="--", label="transition")
    ax3.set_xlabel("Distance (nm)")
    ax3.set_ylabel("Derivative magnitude")
    ax3.set_title(f"Derivative diagnostics @ drive={drive_nm:g} nm")
    ax3.legend(frameon=False)
    ax3.grid(True, alpha=0.3)
    fig3.tight_layout()

    return {"amplitude": fig1, "phase": fig2, "derivatives": fig3}


# -----------------------------
# Example usage
# -----------------------------
# df, diag = analyze_fd_collection(
#     measured_fd,
#     Q=scanner.param_norm['Q'],
#     R=scanner.param_norm.get('R', None),
#     a0=getattr(spm_params, 'a0', 0.0),
#     far_field_side='largest_d',
#     transition_mode='phase_derivative',
#     repulsive_eval='closest',
# )
# print(df)
# figs = plot_fd_parameter_summary(df)
# for name, fig in figs.items():
#     fig.show()


In [ ]:
# from fd_parameter_analysis_with_E import (
#     analyze_fd_collection,
#     plot_fd_parameter_summary,
#     plot_curve_diagnostics,
# )

df_fd, diag_fd = analyze_fd_collection(
    measured_fd,
    Q=param_norm["Q"],
    R=param_norm["R"],
    a0=spm_params.a0,
    k_cantilever=1.0,
    far_field_frac=0.1,
    far_field_side="largest_d",
    transition_mode="phase_derivative",
    repulsive_eval="closest",
)

print(df_fd)

In [ ]:
df_fd.keys()

In [ ]:
plt.plot(df_fd['H_proxy_rel_from_phase'], '.-')
plt.plot(df_fd['H_proxy_rel_from_setpoint'], '.-')


In [ ]:
plt.plot(df_fd['kappa_from_phase'], '.-')
plt.plot(df_fd['kappa_from_setpoint'], '.-')


In [ ]:
figs = plot_fd_parameter_summary(df_fd)

import matplotlib.pyplot as plt
plt.show()

In [ ]:
drives_si = np.array(drives)*factor1
measured = {}


# Assume these are lists/arrays where index i corresponds to drives[i]
# z_arrays[i], A_arrays[i], phi_arrays[i]
# each could be a 1D FD curve, a scan line, or an image
measured = {
    float(drive): {
        "d": height[i],
        "A": amp[i],
        "phi": phase[i],
    }
    for i, drive in enumerate(drives_si)
}

In [ ]:
traces = param_clean['traces']
# Define the groud-truth line profile
indices = np.array([0, 20, 44, 83, 108, 147, 172, 211, 235, 256]) * 2
h1 = np.zeros(512)
for i in range(5):
    h1[indices[2*i]:indices[2*i+1]] = 1
    # h1[(i*50):(i*50)+25] = 1
h_hat = h1 * 100 + 142 / 512 * (512-np.arange(512))

# h_truth_hat = h_hat[::2]
h_truth_hat = h_hat


fig, ax = plt.subplots(2,1,figsize=[4, 6])
tp  = traces[30, 4, 0]
ax[0].plot(h_truth_hat)
ax[1].plot((tp-np.min(tp))*1e9)

In [ ]:
h_truth = h_truth_hat-np.min(h_truth_hat)
plt.plot(h_truth[::2])
plt.plot((traces[5,4,0]-traces[5,4,0].min()) * 1e9)

In [ ]:
import numpy as np

idx = -4

drive_nm = drives_si[idx]
d_nm     = height[idx]    # ascending, tip–sample distance
A_nm     = amp[idx]    # rising toward drive_nm at large d
phi_deg  = phase[idx]

fd = FDCurve1D(d_nm, A_nm, phi_deg, A0=drive_nm)

# Eyeball check before plugging into the loop
d_grid = np.linspace(d_nm.min(), d_nm.max(), 200)
A_grid = np.array([fd.evaluate(d)[0] for d in d_grid])
plt.plot(d_grid, A_grid); plt.axhline(drive_nm, ls='--')

# Build a scanner that always uses this one curve
params = {'A': drive_nm, 'd0': d_nm.max()*0.8, 'R': 10, 'H': 1e-19,
          'E_star': 1e9, 'Q': 200}
scanner = ScannerFD(params=params,
                    conversions={k: 1.0 for k in params},
                    fd_model=fd)

# Trivial fd_lookup: ignore the drive argument, always return `fd`
fd_lookup = lambda _drive: fd

phi_grid = np.array([fd.evaluate(d)[1] for d in d_grid])

plt.plot(d_grid, phi_grid); plt.axhline(drive_nm, ls='--')
# plt.plot(d_grid, A_grid); plt.axhline(drive_nm, ls='--')


In [ ]:
# scanner.scan_line_pi(d_init_hat=)

In [ ]:
P0 = 2.5
I0 = 1

out = scanner.scan_line_trace_retrace_pi(
    h_hat=h_truth,
    setpoint=0.6,
    scan_speed_hat=1e3,
    dx_hat=1.0,
    P=P0,
    I=I0,
    A_filter_order=1,
    tau_A=5e-4,
    phi_filter_order=1,
    tau_phi=2e-3,
    z_filter_order=None,
    tau_z=1e-5,
    d_init_hat=300
)

plot_trace_retrace_channels(out, h_hat=h_truth)

In [ ]:
np.mean(out['trace']['phi_deg']+out['retrace']['phi_deg']) / 2

In [ ]:
P0 = 0.5
I0 = 0.0

out = scanner.scan_line_trace_retrace_pi(
    h_hat=h_truth,
    setpoint=0.6,
    scan_speed_hat=1e3,
    dx_hat=1.0,

    P=P0,
    I=I0,

    A_filter_order=1,
    tau_A=1e-2,

    phi_filter_order=1,
    tau_phi=1e-2,

    z_filter_order=1,
    tau_z=5e-3,

    z_rate_limit_hat=None,
    integ_clip=20.0,
    T_I=0.02,

    d_init_hat=300
    
)
plot_trace_retrace_channels(out, h_hat=h_truth)

In [ ]:

P0 = 1.8
I0 = 0.9
out = scanner.scan_line_trace_retrace_pi(
    h_hat=h_truth,
    setpoint=0.5,
    scan_speed_hat=1e0, dx_hat=1.0,
    P=1.1, I=1.9,                          # ← your original, leave alone
    A_filter_order=1, tau_A=5e-4,           # ← your original, leave alone
    phi_filter_order=1, tau_phi=1e-3,       # ← NEW: cosmetic, free
    z_filter_order=None, tau_z=1e-5,                    # ← your original, leave alone
    integ_clip=10,
    T_I=1e-2,
    d_init_hat=300
    
)

plot_trace_retrace_channels(out, h_hat=h_truth)


In [ ]:

P0 = 1.8
I0 = 0.9
out = scanner.scan_line_trace_retrace_pi(
    h_hat=h_truth,
    setpoint=0.6,
    scan_speed_hat=1e3, dx_hat=1.0,
    P=1.25e0, I=1e1,                          # ← your original, leave alone
    A_filter_order=None, tau_A=5e-4,           # ← your original, leave alone
    phi_filter_order=None, tau_phi=3e-3,       # ← NEW: cosmetic, free
    z_filter_order=None,                    # ← your original, leave alone
    T_I=1e2,
    integ_clip=10,
    d_init_hat=300
    
)

plot_trace_retrace_channels(out, h_hat=h_truth)

# scanner.scan_line_pi()


## Grid-mode to search for best parameters

### Best PI parameters

In [ ]:
import numpy as np

# define the multipliers within each decade
# multipliers = np.array([1, 2.5, 5, 7.5])
# multipliers = np.linspace(1,9,num=41)
multipliers = np.linspace(1,9,num=9)

# define the decades (from 1e-4 to 1e1)
exponents = np.arange(-6, -2)  # -4, -3, -2, -1, 0

# build the array
# p_grid = np.concatenate([multipliers * 10.0**e for e in exponents] + [[10]])
# i_grid = np.concatenate([multipliers * 10.0**e for e in exponents] + [[10]])

A_grid = np.concatenate([multipliers * 10.0**e for e in exponents] + [[10]])
phi_grid = np.concatenate([multipliers * 10.0**e for e in exponents] + [[10]])

len(A_grid)

In [ ]:
# p_grid

In [ ]:
n1 = len(p_grid)
n2 = len(i_grid)

out_array = np.zeros([n1, n2, 6, 512])

rew1 = np.zeros([n1,n2])
rew2 = np.zeros([n1,n2])
rew3 = np.zeros([n1,n2])


for i in range(n1):
    for j in range(n2):
        print('Working on {}/{}'.format(i*n2+j, n1*n2), end='\r')
        p0 = p_grid[i]
        i0 = i_grid[j]
        out = scanner.scan_line_trace_retrace_pi(
            h_hat=h_truth,
            setpoint=0.6,
            scan_speed_hat=1e3,
            dx_hat=1.0,
            P=2.8,
            I=4.8,
            A_filter_order=1,
            tau_A=5e-4,
            phi_filter_order=1,
            tau_phi=2e-3,
            z_filter_order=None,
            tau_z=1e-6,
            d_init_hat=250
        )
        out_array[i,j,0] = out['trace']['d_hat']
        out_array[i,j,1] = out['retrace']['d_hat']
        out_array[i,j,2] = out['trace']['phi_deg']
        out_array[i,j,3] = out['retrace']['phi_deg']
        out_array[i,j,4] = out['trace']['A_hat']
        out_array[i,j,5] = out['retrace']['A_hat']

        rew1[i,j] = np.mean(np.abs((out['trace']['d_hat']+out['retrace']['d_hat'])/2-h_truth))
        rew2[i,j] = np.mean(np.abs((out['trace']['phi_deg']+out['retrace']['phi_deg'])/2-131.4))
        rew3[i,j] = np.mean(np.abs((out['trace']['A_hat']+out['retrace']['A_hat'])/2-11.81))
        

In [ ]:
np.where(rew3==np.min(rew3))

In [ ]:
# np.savez('output/sparse grid search.npz', p_grid=p_grid, i_grid=i_grid, data=out_array, rew1=rew1, rew2=rew2, rew3=rew3)
# np.savez('output/fine grid search 1.npz', p_grid=p_grid, i_grid=i_grid, data=out_array, rew1=rew1, rew2=rew2, rew3=rew3)


In [ ]:
# idx1 = 39
# idx2 = 2

# idx1 = 38
# idx2 = 41

idx1 = 91
idx2 = 101

print("P={}, I={}".format(p_grid[idx1], i_grid[idx2]))

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out_array[idx1, idx2, 0])
ax[0].plot(out_array[idx1, idx2, 1])
ax[1].plot(out_array[idx1, idx2, 2])
ax[1].plot(out_array[idx1, idx2, 3])
ax[2].plot(out_array[idx1, idx2, 4])
ax[2].plot(out_array[idx1, idx2, 5])


In [ ]:
# idx1 = 39
# idx2 = 2

# idx1 = 38
# idx2 = 41

idx1 = 37
idx2 = 32

print("P={}, I={}".format(p_grid[idx1], i_grid[idx2]))

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out_array[idx1, idx2, 0])
ax[0].plot(out_array[idx1, idx2, 1])
ax[1].plot(out_array[idx1, idx2, 2])
ax[1].plot(out_array[idx1, idx2, 3])
ax[2].plot(out_array[idx1, idx2, 4])
ax[2].plot(out_array[idx1, idx2, 5])


In [ ]:
np.mean(out_array[idx1, idx2, 4]+out_array[idx1, idx2, 5])/2

In [ ]:
plt.imshow(np.log(rew1), origin='lower')
plt.colorbar()

In [ ]:
plt.imshow(np.log(rew2), origin='lower')
plt.colorbar()

### Best LPF parameters

In [ ]:
import numpy as np

# define the multipliers within each decade
# multipliers = np.array([1, 2.5, 5, 7.5])
multipliers = np.linspace(1,9,num=41)
# multipliers = np.linspace(1,9,num=9)

# define the decades (from 1e-4 to 1e1)
exponents = np.arange(-5, -2)  # -4, -3, -2, -1, 0

# build the array
# p_grid = np.concatenate([multipliers * 10.0**e for e in exponents] + [[10]])
# i_grid = np.concatenate([multipliers * 10.0**e for e in exponents] + [[10]])

A_grid = np.concatenate([multipliers * 10.0**e for e in exponents] )
phi_grid = np.concatenate([multipliers * 10.0**e for e in exponents] )

len(A_grid)

In [ ]:
A_grid

In [ ]:
n1 = len(A_grid)
n2 = len(phi_grid)

out_array = np.zeros([n1, n2, 6, 512])

rew1 = np.zeros([n1,n2])
rew2 = np.zeros([n1,n2])
rew3 = np.zeros([n1,n2])


for i in range(n1):
    for j in range(n2):
        print('Working on {}/{}'.format(i*n2+j, n1*n2), end='\r')
        # p0 = p_grid[i]
        # i0 = i_grid[j]
        out = scanner.scan_line_trace_retrace_pi(
            h_hat=h_truth,
            setpoint=0.6,
            scan_speed_hat=1e3,
            dx_hat=1.0,
            P=2.8,
            I=4.8,
            A_filter_order=1,
            tau_A=A_grid[i],
            phi_filter_order=1,
            tau_phi=phi_grid[j],
            z_filter_order=1,
            tau_z=1e-5,
            d_init_hat=250
        )
        out_array[i,j,0] = out['trace']['d_hat']
        out_array[i,j,1] = out['retrace']['d_hat']
        out_array[i,j,2] = out['trace']['phi_deg']
        out_array[i,j,3] = out['retrace']['phi_deg']
        out_array[i,j,4] = out['trace']['A_hat']
        out_array[i,j,5] = out['retrace']['A_hat']

        rew1[i,j] = np.mean(np.abs((out['trace']['d_hat']+out['retrace']['d_hat'])/2-h_truth))
        rew2[i,j] = np.mean(np.abs((out['trace']['phi_deg']+out['retrace']['phi_deg'])/2-131.4))
        rew3[i,j] = np.mean(np.abs((out['trace']['A_hat']+out['retrace']['A_hat'])/2-11.81))
        

In [ ]:
np.where(rew2==np.min(rew2))

In [ ]:
# idx1 = 25
# idx2 = 31

idx1 = 76
idx2 = 104

print("tau_A={}, tau_phi={}".format(A_grid[idx1], phi_grid[idx2]))

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out_array[idx1, idx2, 0])
ax[0].plot(out_array[idx1, idx2, 1])
ax[1].plot(out_array[idx1, idx2, 2])
ax[1].plot(out_array[idx1, idx2, 3])
ax[2].plot(out_array[idx1, idx2, 4])
ax[2].plot(out_array[idx1, idx2, 5])


In [ ]:
plt.plot(traces[5,4,4])
plt.plot(traces[5,4,5])


### Best simulated scan lines (under this drive amplitude)

In [ ]:
out = scanner.scan_line_trace_retrace_pi(
    h_hat=h_truth,
    setpoint=0.6,
    scan_speed_hat=1e3,
    dx_hat=1.0,
    P=2.8,
    I=4.8,
    A_filter_order=1,
    tau_A=8e-4,
    phi_filter_order=1,
    tau_phi=5e-3,
    z_filter_order=1,
    tau_z=1e-5,
    d_init_hat=350
)

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out['trace']['d_hat'])
ax[0].plot(out['retrace']['d_hat'])
ax[1].plot(out['trace']['phi_deg'])
ax[1].plot(out['retrace']['phi_deg'])
ax[2].plot(out['trace']['A_hat'])
ax[2].plot(out['retrace']['A_hat'])



In [ ]:
scanner.scan_line_pi()

In [ ]:
# Compare with the experimental result

out = scanner.scan_line_trace_retrace_pi(
    h_hat=h_truth,
    setpoint=0.6,
    scan_speed_hat=1e3,
    dx_hat=1.0,
    P=2.8,
    I=4.8,
    A_filter_order=1,
    tau_A=8e-4,
    phi_filter_order=1,
    tau_phi=5e-3,
    z_filter_order=None,
    tau_z=1e-5,
    d_init_hat=350,
    z_rate_limit_hat=1e6
)


tp = traces[5,4]

conv = np.mean(out['trace']['phi_deg'][::2]) / np.mean(tp[2])


fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out['trace']['d_hat'][::2])
ax[0].plot(out['retrace']['d_hat'][::2])
ax[1].plot(out['trace']['phi_deg'][::2])
ax[1].plot(out['retrace']['phi_deg'][::2])
ax[2].plot(out['trace']['A_hat'][::2])
ax[2].plot(out['retrace']['A_hat'][::2])

ax[0].plot((tp[0]-np.min(tp[0]))*1e9)
ax[2].plot(tp[2]*1e9)
ax[1].plot(tp[4])
ax[1].plot(tp[5])


In [ ]:
data[0].header['IntegralGain']

In [ ]:
param.keys()

In [ ]:
scanner.param_norm

In [ ]:
param_clean.keys()

In [ ]:
param_clean['x_measured']

In [ ]:
out = scanner.scan_line_trace_retrace_pi(
    h_hat=h_truth,
    setpoint=0.6,
    scan_speed_hat=1e2,
    dx_hat=1.0,
    P=0,
    I=1e2,
    A_filter_order=1,
    tau_A=8e-4,
    phi_filter_order=1,
    tau_phi=5e-3,
    z_filter_order=1,
    tau_z=1e-5,
    d_init_hat=350,
    T_I=1e-2,
)

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out['trace']['d_hat'])
ax[0].plot(out['retrace']['d_hat'])
ax[1].plot(out['trace']['phi_deg'])
ax[1].plot(out['retrace']['phi_deg'])
ax[2].plot(out['trace']['A_hat'])
ax[2].plot(out['retrace']['A_hat'])



In [ ]:
out = scanner.scan_line_trace_retrace_pi(
    h_hat=h_truth,
    setpoint=0.6,
    scan_speed_hat=1e3,
    dx_hat=1.0,
    P=0,
    I=1e3,
    A_filter_order=1,
    tau_A=8e-4/1,
    phi_filter_order=1,
    tau_phi=5e-3/1,
    z_filter_order=1,
    tau_z=1e-5/1,
    d_init_hat=350,
    T_I=1e-3,
)

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out['trace']['d_hat'])
ax[0].plot(out['retrace']['d_hat'])
ax[1].plot(out['trace']['phi_deg'])
ax[1].plot(out['retrace']['phi_deg'])
ax[2].plot(out['trace']['A_hat'])
ax[2].plot(out['retrace']['A_hat'])



## Calibrate without P gain

In [ ]:
import numpy as np

# define the multipliers within each decade
# multipliers = np.array([1, 2.5, 5, 7.5])
# multipliers = np.linspace(1,9,num=41)
multipliers = np.linspace(1,9,num=9)


# define the decades (from 1e-4 to 1e1)
exponents = np.arange(0, 6)  # -4, -3, -2, -1, 0

# build the array
# p_grid = np.concatenate([multipliers * 10.0**e for e in exponents] + [[10]])
i_grid = np.concatenate([multipliers * 10.0**e for e in exponents])
# define the decades (from 1e-4 to 1e1)
exponents = np.arange(-5, 0)  # -4, -3, -2, -1, 0

TI_grid = np.concatenate([multipliers * 10.0**e for e in exponents])

# define the decades (from 1e-4 to 1e1)
exponents = np.arange(-6, -2)  # -4, -3, -2, -1, 0

A_grid = np.concatenate([multipliers * 10.0**e for e in exponents])
phi_grid = np.concatenate([multipliers * 10.0**e for e in exponents])

len(TI_grid)

In [ ]:
i_grid

In [ ]:
n1 = len(i_grid)
n2 = len(TI_grid)
# n3 = len(phi_grid)

out_array = np.zeros([n1, n2, 6, 512])

rew1 = np.zeros([n1,n2])
rew2 = np.zeros([n1,n2])
rew3 = np.zeros([n1,n2])


for i in range(n1):
    for j in range(n2):
        # for k in range(n3):
        print('Working on {}/{}'.format(i*n2+j, n1*n2*1), end='\r')
        # p0 = p_grid[i]
        # i0 = i_grid[j]
        out = scanner.scan_line_trace_retrace_pi(
            h_hat=h_truth,
            setpoint=0.6,
            scan_speed_hat=1e3,
            dx_hat=1.0,
            P=0,
            I=i_grid[i],
            A_filter_order=1,
            tau_A=8e-4,
            phi_filter_order=1,
            tau_phi=5e-3,
            z_filter_order=1,
            tau_z=1e-5,
            d_init_hat=350,
            T_I=TI_grid[j],
        )
        out_array[i,j,0] = out['trace']['d_hat']
        out_array[i,j,1] = out['retrace']['d_hat']
        out_array[i,j,2] = out['trace']['phi_deg']
        out_array[i,j,3] = out['retrace']['phi_deg']
        out_array[i,j,4] = out['trace']['A_hat']
        out_array[i,j,5] = out['retrace']['A_hat']

        rew1[i,j] = np.mean(np.abs((out['trace']['d_hat']+out['retrace']['d_hat'])/2-h_truth))
        rew2[i,j] = np.mean(np.abs((out['trace']['phi_deg']+out['retrace']['phi_deg'])/2-131.4))
        rew3[i,j] = np.mean(np.abs((out['trace']['A_hat']+out['retrace']['A_hat'])/2-11.81))
        

In [ ]:
np.where(rew1==np.min(rew1))

In [ ]:
# idx1 = 25
# idx2 = 31

idx1 = 30
idx2 = 11
idx3 = 0

print("I Gain={}, T_I={}".format(i_grid[idx1], TI_grid[idx2]))

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out_array[idx1, idx2, 0])
ax[0].plot(out_array[idx1, idx2, 1])
ax[1].plot(out_array[idx1, idx2, 2])
ax[1].plot(out_array[idx1, idx2, 3])
ax[2].plot(out_array[idx1, idx2, 4])
ax[2].plot(out_array[idx1, idx2, 5])


In [ ]:
4e3
3e-3
1e-3

In [ ]:
out = scanner.scan_line_trace_retrace_pi(
            h_hat=h_truth,
            setpoint=0.6,
            scan_speed_hat=1e3,
            dx_hat=1.0,
            P=0,
            I=4e3,
            A_filter_order=1,
            tau_A=8e-4,
            phi_filter_order=1,
            tau_phi=5e-3,
            z_filter_order=1,
            tau_z=1e-5,
            d_init_hat=350,
            T_I=3e-3,
        )

In [ ]:
np.log(8000)

In [ ]:
out = scanner.scan_line_trace_retrace_pi(
    h_hat=h_truth,
    setpoint=0.6,
    scan_speed_hat=1e3,
    dx_hat=1.0,
    P=0,
    I=1e3,
    A_filter_order=1,
    tau_A=8e-4/1,
    phi_filter_order=1,
    tau_phi=5e-3/1,
    z_filter_order=1,
    tau_z=1e-5/1,
    d_init_hat=350,
    T_I=1e-2/10,
)

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out['trace']['d_hat'])
ax[0].plot(out['retrace']['d_hat'])
ax[1].plot(out['trace']['phi_deg'])
ax[1].plot(out['retrace']['phi_deg'])
ax[2].plot(out['trace']['A_hat'])
ax[2].plot(out['retrace']['A_hat'])



In [ ]:
out['trace'].keys()

In [ ]:
P0 = 2.5
I0 = 1

out = scanner.scan_line_trace_retrace_pi(
    h_hat=h_truth,
    setpoint=0.6,
    scan_speed_hat=1e3,
    dx_hat=1.0,
    P=P0,
    I=I0,
    A_filter_order=1,
    tau_A=5e-4,
    phi_filter_order=1,
    tau_phi=2e-3,
    z_filter_order=None,
    tau_z=1e-5,
    d_init_hat=300
)

plot_trace_retrace_channels(out, h_hat=h_truth)

## Simulate the effect of drive and setpoints

In [ ]:
# Stage-2 FD-curve surface interpolation for ScannerFD.

"""
Stage-2 FD-curve surface interpolation for ScannerFD.

Purpose
-------
Start from measured force-distance / amplitude-distance / phase-distance curves
at discrete drive amplitudes and build a smooth lookup that supports continuous
changes of drive amplitude and setpoint during scanning.

Main objects
------------
FDCurve1D
    Single-drive interpolator, evaluate(d) -> (A, phi, F).

FDDriveSurface
    2D surface over (drive, distance_coordinate). It is built once from the
    measured FD curves, then queried cheaply at arbitrary drive.

make_normalized_fd_lookup
    Adapter from physical nm/degree units to the normalized units used inside
    ScannerFD.

attach_continuous_fd_to_scanner
    Monkey-patch helper for an existing ScannerFD instance. It adds
    scanner.set_drive(...), scanner.scan_line_pi_drive(...), and
    scanner.scan_line_trace_retrace_pi_drive(...). If you prefer permanent
    changes, copy these methods into your ScannerFD class.

Conventions
-----------
- d increases away from the sample. Large d is far field.
- The measured d origin can be arbitrary; if needed, pre-shift each measured
  curve before adding it, or pass d_offset_nm in add_curve().
- A0 is the free amplitude for each curve. By default it is estimated from the
  largest-d tail, which matches separation-like FD curves.
- The recommended interpolation coordinate is x = d / A0, because AM-AFM
  amplitude/phase curves often collapse better in drive-normalized distance.
  For fixed calibrated absolute separation axes, use x_mode='absolute_d'.
"""
from __future__ import annotations

from dataclasses import dataclass
from types import MethodType
from typing import Callable, Dict, Iterable, List, Optional, Tuple, Union

import numpy as np
from scipy.interpolate import PchipInterpolator, RegularGridInterpolator, interp1d

ArrayLike = Union[float, np.ndarray, Iterable[float]]


def _as_1d(x: ArrayLike, name: str) -> np.ndarray:
    arr = np.asarray(x, dtype=float).ravel()
    if arr.size == 0:
        raise ValueError(f"{name} is empty")
    return arr


def _finite_mask(*arrays: np.ndarray) -> np.ndarray:
    mask = np.ones_like(arrays[0], dtype=bool)
    for a in arrays:
        mask &= np.isfinite(a)
    return mask


def _average_duplicate_x(x: np.ndarray, *ys: np.ndarray) -> Tuple[np.ndarray, ...]:
    """Sort by x and average duplicate x values for stable interpolation."""
    order = np.argsort(x)
    x = x[order]
    ys = [np.asarray(y)[order] for y in ys]

    xu, inv = np.unique(x, return_inverse=True)
    if xu.size == x.size:
        return (x, *ys)

    y_out = []
    for y in ys:
        accum = np.zeros_like(xu, dtype=float)
        count = np.zeros_like(xu, dtype=float)
        np.add.at(accum, inv, y)
        np.add.at(count, inv, 1.0)
        y_out.append(accum / np.maximum(count, 1.0))
    return (xu, *y_out)


def _unwrap_phase_deg(phi_deg: np.ndarray) -> np.ndarray:
    return np.rad2deg(np.unwrap(np.deg2rad(phi_deg)))


def _safe_pchip(x: np.ndarray, y: np.ndarray, left: float, right: float):
    """PCHIP when possible; linear fallback for very short curves."""
    if x.size >= 3:
        return PchipInterpolator(x, y, extrapolate=False)
    return interp1d(x, y, kind="linear", bounds_error=False, fill_value=(left, right))


@dataclass
class FDCurve1D:
    """Single-drive FD evaluator.

    d, A and F use the same unit system supplied at construction. phi is degrees.
    d must increase away from the sample. Out-of-range behavior is clamped:
    near-contact side uses the first sampled value; far-field side uses A0,
    far-field phase, and F=0.
    """

    d: np.ndarray
    A: np.ndarray
    phi: np.ndarray
    F: Optional[np.ndarray] = None
    A0: Optional[float] = None
    kind: str = "pchip"

    def __post_init__(self) -> None:
        d = _as_1d(self.d, "d")
        A = _as_1d(self.A, "A")
        phi = _as_1d(self.phi, "phi")
        F = np.zeros_like(d) if self.F is None else _as_1d(self.F, "F")
        if not (d.shape == A.shape == phi.shape == F.shape):
            raise ValueError("d, A, phi, and F must have the same shape")

        mask = _finite_mask(d, A, phi, F)
        d, A, phi, F = d[mask], A[mask], phi[mask], F[mask]
        if d.size < 2:
            raise ValueError("Need at least two finite samples for FDCurve1D")

        d, A, phi, F = _average_duplicate_x(d, A, phi, F)
        if d.size < 2 or not np.all(np.diff(d) > 0):
            raise ValueError("d must contain at least two unique increasing values")

        self.d = d
        self.A = A
        self.phi = phi
        self.F = F
        self.A0 = float(A[-1] if self.A0 is None else self.A0)

        phi_unwrapped = _unwrap_phase_deg(phi)

        if self.kind == "pchip":
            self._A_interp = _safe_pchip(d, A, left=A[0], right=self.A0)
            self._phi_interp = _safe_pchip(d, phi_unwrapped, left=phi_unwrapped[0], right=phi_unwrapped[-1])
            self._F_interp = _safe_pchip(d, F, left=F[0], right=0.0)
        else:
            self._A_interp = interp1d(d, A, kind=self.kind, bounds_error=False, fill_value=(A[0], self.A0))
            self._phi_interp = interp1d(d, phi_unwrapped, kind=self.kind, bounds_error=False,
                                        fill_value=(phi_unwrapped[0], phi_unwrapped[-1]))
            self._F_interp = interp1d(d, F, kind=self.kind, bounds_error=False, fill_value=(F[0], 0.0))

    def evaluate(self, d_query: ArrayLike):
        scalar = np.isscalar(d_query)
        dq = np.atleast_1d(np.asarray(d_query, dtype=float))
        dq_clip = np.clip(dq, self.d[0], self.d[-1])

        A = np.asarray(self._A_interp(dq_clip), dtype=float)
        phi = np.asarray(self._phi_interp(dq_clip), dtype=float)
        F = np.asarray(self._F_interp(dq_clip), dtype=float)

        # Explicit far-field clamp. This avoids PCHIP extrapolation surprises.
        far = dq > self.d[-1]
        if np.any(far):
            A[far] = self.A0
            phi[far] = self.phi[-1]
            F[far] = 0.0

        near = dq < self.d[0]
        if np.any(near):
            A[near] = self.A[0]
            phi[near] = self.phi[0]
            F[near] = self.F[0]

        if scalar:
            return float(A[0]), float(phi[0]), float(F[0])
        return A, phi, F

    def __call__(self, d_query: ArrayLike):
        return self.evaluate(d_query)


class FDDriveSurface:
    """Continuous FD surface over drive and distance.

    The surface stores normalized amplitude a=A/A0 and unwrapped phase. The
    drive interpolation is linear. The per-curve resampling along distance is
    PCHIP by default.
    """

    def __init__(
        self,
        *,
        x_mode: str = "d_over_A0",
        far_field_frac: float = 0.10,
        common_range: str = "intersection",
        n_x: int = 512,
        interp_kind_1d: str = "pchip",
        right_fill_free: bool = True,
    ) -> None:
        if x_mode not in {"d_over_A0", "absolute_d"}:
            raise ValueError("x_mode must be 'd_over_A0' or 'absolute_d'")
        if common_range not in {"intersection", "union"}:
            raise ValueError("common_range must be 'intersection' or 'union'")

        self.x_mode = x_mode
        self.far_field_frac = float(far_field_frac)
        self.common_range = common_range
        self.n_x = int(n_x)
        self.interp_kind_1d = interp_kind_1d
        self.right_fill_free = bool(right_fill_free)

        self._raw: Dict[float, Dict[str, np.ndarray]] = {}
        self._built = False

    @property
    def drives(self) -> np.ndarray:
        return np.array(sorted(self._raw.keys()), dtype=float)

    @property
    def x_grid(self) -> np.ndarray:
        self._require_built()
        return self._x_grid

    def add_curve(
        self,
        drive_nm: float,
        d_nm: ArrayLike,
        A_nm: ArrayLike,
        phi_deg: ArrayLike,
        F: Optional[ArrayLike] = None,
        *,
        A0_nm: Optional[float] = None,
        d_offset_nm: float = 0.0,
        far_field_side: str = "largest_d",
    ) -> None:
        """Add one measured curve.

        Parameters
        ----------
        drive_nm
            Commanded or calibrated drive/free-amplitude label in nm.
        d_nm
            Separation-like coordinate. Larger d should mean farther from the
            surface. If your measured origin is arbitrary, use d_offset_nm.
        A_nm, phi_deg
            Measured amplitude and phase.
        A0_nm
            Optional explicit free amplitude. If omitted, it is estimated from
            the far-field tail specified by far_field_side.
        far_field_side
            'largest_d', 'smallest_d', 'start', or 'end'. For your screenshot,
            use 'largest_d' or the default.
        """
        drv = float(drive_nm)
        d = _as_1d(d_nm, "d_nm") + float(d_offset_nm)
        A = _as_1d(A_nm, "A_nm")
        phi = _as_1d(phi_deg, "phi_deg")
        F_arr = np.zeros_like(d) if F is None else _as_1d(F, "F")
        if not (d.shape == A.shape == phi.shape == F_arr.shape):
            raise ValueError("d, A, phi, and F must have the same shape")

        mask = _finite_mask(d, A, phi, F_arr)
        d, A, phi, F_arr = d[mask], A[mask], phi[mask], F_arr[mask]
        if d.size < 3:
            raise ValueError(f"drive {drv}: need at least 3 finite samples")

        d, A, phi, F_arr = _average_duplicate_x(d, A, phi, F_arr)

        if A0_nm is None:
            n_far = max(1, int(round(self.far_field_frac * d.size)))
            if far_field_side == "largest_d":
                idx_far = np.argsort(d)[-n_far:]
            elif far_field_side == "smallest_d":
                idx_far = np.argsort(d)[:n_far]
            elif far_field_side == "start":
                idx_far = np.arange(n_far)
            elif far_field_side == "end":
                idx_far = np.arange(d.size - n_far, d.size)
            else:
                raise ValueError("far_field_side must be largest_d, smallest_d, start, or end")
            A0 = float(np.nanmedian(A[idx_far]))
        else:
            A0 = float(A0_nm)

        if not np.isfinite(A0) or A0 <= 0:
            raise ValueError(f"drive {drv}: invalid A0={A0}")

        self._raw[drv] = {
            "d": d,
            "A": A,
            "phi": phi,
            "F": F_arr,
            "A0": np.array([A0], dtype=float),
        }
        self._built = False

    @classmethod
    def from_measured_fd(
        cls,
        measured_fd: Dict[float, Dict[str, ArrayLike]],
        *,
        x_mode: str = "d_over_A0",
        far_field_frac: float = 0.10,
        common_range: str = "intersection",
        n_x: int = 512,
        far_field_side: str = "largest_d",
        drive_key_is_A0: bool = False,
    ) -> "FDDriveSurface":
        """Build from {drive: {'d':..., 'A':..., 'phi':..., optional 'F':...}}."""
        surf = cls(
            x_mode=x_mode,
            far_field_frac=far_field_frac,
            common_range=common_range,
            n_x=n_x,
        )
        for drv, c in measured_fd.items():
            A0 = float(drv) if drive_key_is_A0 else c.get("A0_nm", None)
            surf.add_curve(
                float(drv), c["d"], c["A"], c["phi"], F=c.get("F"),
                A0_nm=A0, far_field_side=far_field_side,
            )
        surf.build()
        return surf

    def build(self) -> "FDDriveSurface":
        drives = self.drives
        if drives.size < 1:
            raise ValueError("No curves have been added")
        if drives.size < 2:
            raise ValueError("Need at least two drives for continuous drive interpolation")

        x_mins: List[float] = []
        x_maxs: List[float] = []
        prepared: List[Tuple[float, np.ndarray, np.ndarray, np.ndarray, np.ndarray, float]] = []

        for drv in drives:
            c = self._raw[float(drv)]
            d = c["d"]
            A = c["A"]
            phi = _unwrap_phase_deg(c["phi"])
            F = c["F"]
            A0 = float(c["A0"][0])

            x = d / A0 if self.x_mode == "d_over_A0" else d.copy()
            a = A / A0

            x, a, phi, F = _average_duplicate_x(x, a, phi, F)
            x_mins.append(float(x[0]))
            x_maxs.append(float(x[-1]))
            prepared.append((float(drv), x, a, phi, F, A0))

        if self.common_range == "intersection":
            x_min = max(x_mins)
            x_max = min(x_maxs)
            if not x_max > x_min:
                raise ValueError(
                    "No overlapping distance range across curves. Use common_range='union', "
                    "trim curves, or align d-offsets first."
                )
        else:
            x_min = min(x_mins)
            x_max = max(x_maxs)

        self._x_grid = np.linspace(x_min, x_max, self.n_x)
        nD, nX = drives.size, self._x_grid.size
        a_grid = np.empty((nD, nX), dtype=float)
        phi_grid = np.empty((nD, nX), dtype=float)
        F_grid = np.empty((nD, nX), dtype=float)
        A0_by_drive = np.empty(nD, dtype=float)

        for i, (drv, x, a, phi, F, A0) in enumerate(prepared):
            # Far-field fill: a -> 1, phase -> far-field phase, F -> 0.
            a_left, a_right = float(a[0]), 1.0 if self.right_fill_free else float(a[-1])
            phi_left, phi_right = float(phi[0]), float(phi[-1])
            F_left, F_right = float(F[0]), 0.0

            if self.interp_kind_1d == "pchip":
                a_interp = _safe_pchip(x, a, a_left, a_right)
                p_interp = _safe_pchip(x, phi, phi_left, phi_right)
                f_interp = _safe_pchip(x, F, F_left, F_right)
                xq = np.clip(self._x_grid, x[0], x[-1])
                a_grid[i] = a_interp(xq)
                phi_grid[i] = p_interp(xq)
                F_grid[i] = f_interp(xq)
                # Manual edge fill after clipping.
                left = self._x_grid < x[0]
                right = self._x_grid > x[-1]
                a_grid[i, left] = a_left
                phi_grid[i, left] = phi_left
                F_grid[i, left] = F_left
                a_grid[i, right] = a_right
                phi_grid[i, right] = phi_right
                F_grid[i, right] = F_right
            else:
                a_grid[i] = np.interp(self._x_grid, x, a, left=a_left, right=a_right)
                phi_grid[i] = np.interp(self._x_grid, x, phi, left=phi_left, right=phi_right)
                F_grid[i] = np.interp(self._x_grid, x, F, left=F_left, right=F_right)

            A0_by_drive[i] = A0

        self._drives = drives
        self._A0_by_drive = A0_by_drive
        self._a_grid = a_grid
        self._phi_grid = phi_grid
        self._F_grid = F_grid

        self._a_rgi = RegularGridInterpolator((drives, self._x_grid), a_grid,
                                               method="linear", bounds_error=False, fill_value=None)
        self._phi_rgi = RegularGridInterpolator((drives, self._x_grid), phi_grid,
                                                 method="linear", bounds_error=False, fill_value=None)
        self._F_rgi = RegularGridInterpolator((drives, self._x_grid), F_grid,
                                               method="linear", bounds_error=False, fill_value=None)
        self._A0_rgi = interp1d(drives, A0_by_drive, kind="linear", bounds_error=False,
                                fill_value=(A0_by_drive[0], A0_by_drive[-1]))
        self._built = True
        return self

    def _require_built(self) -> None:
        if not self._built:
            self.build()

    def evaluate(self, d_nm: ArrayLike, drive_nm: float):
        """Evaluate the FD surface at arbitrary physical drive and distance.

        Returns A_nm, phi_deg, F. Scalar input returns scalars; array input
        returns arrays.
        """
        self._require_built()
        scalar = np.isscalar(d_nm)
        d_arr = np.atleast_1d(np.asarray(d_nm, dtype=float))
        drv = float(drive_nm)
        drv_clip = float(np.clip(drv, self._drives[0], self._drives[-1]))

        # Use the target drive as A0 scaling. This lets the nearest normalized
        # curve scale reasonably if drive is slightly outside the measured range.
        A0_target = float(self._A0_rgi(drv_clip))
        if self.x_mode == "d_over_A0":
            xq = d_arr / max(A0_target, 1e-30)
        else:
            xq = d_arr
        xq = np.clip(xq, self._x_grid[0], self._x_grid[-1])

        pts = np.column_stack([np.full_like(xq, drv_clip, dtype=float), xq])
        a = np.asarray(self._a_rgi(pts), dtype=float)
        phi = np.asarray(self._phi_rgi(pts), dtype=float)
        F = np.asarray(self._F_rgi(pts), dtype=float)

        A = a * A0_target
        if scalar:
            return float(A[0]), float(phi[0]), float(F[0])
        return A, phi, F

    def fd_at_drive(self, drive_nm: float, n_points: Optional[int] = None) -> FDCurve1D:
        """Return a single-drive FDCurve1D at any target drive."""
        self._require_built()
        n = self.n_x if n_points is None else int(n_points)
        x = self._x_grid if n == self._x_grid.size else np.linspace(self._x_grid[0], self._x_grid[-1], n)
        drv = float(drive_nm)
        drv_clip = float(np.clip(drv, self._drives[0], self._drives[-1]))
        A0_target = float(self._A0_rgi(drv_clip))

        d = x * A0_target if self.x_mode == "d_over_A0" else x.copy()
        A, phi, F = self.evaluate(d, drv)
        return FDCurve1D(d=d, A=A, phi=phi, F=F, A0=A0_target, kind="pchip")

    def __call__(self, drive_nm: float) -> FDCurve1D:
        return self.fd_at_drive(drive_nm)

    def diagnostics(self) -> Dict[str, np.ndarray]:
        self._require_built()
        return {
            "drives": self._drives.copy(),
            "A0_by_drive": self._A0_by_drive.copy(),
            "x_grid": self._x_grid.copy(),
            "a_grid": self._a_grid.copy(),
            "phi_grid": self._phi_grid.copy(),
            "F_grid": self._F_grid.copy(),
        }


def make_normalized_fd_lookup(
    surface: FDDriveSurface,
    *,
    conv_L: float,
    conv_A: Optional[float] = None,
    conv_F: float = 1.0,
    n_points: Optional[int] = None,
) -> Callable[[float], FDCurve1D]:
    """Return lookup(drive_nm) -> FDCurve1D in ScannerFD normalized units.

    conv_L: d_hat = d_nm * conv_L
    conv_A: A_hat = A_nm * conv_A. Defaults to conv_L.
    conv_F: F_hat = F_physical_or_library * conv_F.
    """
    conv_A = conv_L if conv_A is None else float(conv_A)
    conv_L = float(conv_L)
    conv_F = float(conv_F)

    def lookup(drive_nm: float) -> FDCurve1D:
        fd = surface.fd_at_drive(float(drive_nm), n_points=n_points)
        return FDCurve1D(
            d=fd.d * conv_L,
            A=fd.A * conv_A,
            phi=fd.phi,
            F=fd.F * conv_F,
            A0=fd.A0 * conv_A,
            kind="pchip",
        )

    return lookup


def attach_continuous_fd_to_scanner(
    scanner,
    fd_lookup: Callable[[float], FDCurve1D],
    *,
    conv_A: Optional[float] = None,
) -> object:
    """Attach continuous-drive methods to an existing ScannerFD instance.

    Added methods
    -------------
    scanner.set_drive(drive_nm=None, drive_hat=None)
    scanner.scan_line_pi_drive(h_hat, drive_nm=..., setpoint=..., **kwargs)
    scanner.scan_line_trace_retrace_pi_drive(h_hat, drive_nm=..., setpoint=..., **kwargs)

    This does not overwrite your existing scan_line_pi methods.
    """
    if conv_A is None:
        conv_A = float(scanner.conversion.get("A", 1.0))
    conv_A = float(conv_A)

    scanner.fd_lookup = fd_lookup

    def set_drive(self, drive_nm: Optional[float] = None, drive_hat: Optional[float] = None):
        if drive_nm is None and drive_hat is None:
            raise ValueError("Pass either drive_nm or drive_hat")
        if drive_nm is None:
            drive_nm = float(drive_hat) / conv_A
        drive_nm = float(drive_nm)
        drive_hat = drive_nm * conv_A

        fd_model = self.fd_lookup(drive_nm)
        self.fd_model = fd_model

        # Important: the lookup may use drive_nm as a command/label, while the
        # actual free amplitude A0 comes from the measured far-field curve. Use
        # fd_model.A0, not drive_nm, for the scanner feedback normalization.
        A_hat_effective = float(getattr(fd_model, "A0", drive_hat))
        A_nm_effective = A_hat_effective / conv_A

        if hasattr(self, "param") and isinstance(self.param, dict):
            self.param["A"] = A_nm_effective
        if hasattr(self, "param_norm") and isinstance(self.param_norm, dict):
            self.param_norm["A"] = A_hat_effective
            if "Q" in self.param_norm:
                self.param_norm["C3"] = A_hat_effective / float(self.param_norm["Q"])

        self.drive_command_nm = drive_nm
        self.drive_nm = A_nm_effective
        self.drive_hat = A_hat_effective
        return fd_model

    def scan_line_pi_drive(self, h_hat, *, drive_nm: Optional[float] = None,
                           drive_hat: Optional[float] = None,
                           setpoint: Optional[float] = 0.6,
                           setpoint_hat: Optional[float] = None,
                           **kwargs):
        if drive_nm is not None or drive_hat is not None:
            self.set_drive(drive_nm=drive_nm, drive_hat=drive_hat)
        if setpoint_hat is not None:
            A_hat = float(self.param_norm["A"])
            setpoint = float(setpoint_hat) / A_hat
        if setpoint is None:
            raise ValueError("Pass setpoint fraction or setpoint_hat")
        return self.scan_line_pi(h_hat=h_hat, setpoint=float(setpoint), **kwargs)

    def scan_line_trace_retrace_pi_drive(self, h_hat, *, drive_nm: Optional[float] = None,
                                         drive_hat: Optional[float] = None,
                                         setpoint: Optional[float] = 0.6,
                                         setpoint_hat: Optional[float] = None,
                                         **kwargs):
        if drive_nm is not None or drive_hat is not None:
            self.set_drive(drive_nm=drive_nm, drive_hat=drive_hat)
        if setpoint_hat is not None:
            A_hat = float(self.param_norm["A"])
            setpoint = float(setpoint_hat) / A_hat
        if setpoint is None:
            raise ValueError("Pass setpoint fraction or setpoint_hat")
        return self.scan_line_trace_retrace_pi(h_hat=h_hat, setpoint=float(setpoint), **kwargs)

    scanner.set_drive = MethodType(set_drive, scanner)
    scanner.scan_line_pi_drive = MethodType(scan_line_pi_drive, scanner)
    scanner.scan_line_trace_retrace_pi_drive = MethodType(scan_line_trace_retrace_pi_drive, scanner)
    return scanner


def detect_bad_curves(
    measured_fd: Dict[float, Dict[str, ArrayLike]],
    *,
    phase_jump_threshold_deg: float = 30.0,
    amp_jump_threshold_rel: float = 0.20,
) -> Dict[float, Dict[str, float]]:
    """Flag curves with abrupt phase or normalized-amplitude jumps."""
    flags: Dict[float, Dict[str, float]] = {}
    for drv, c in measured_fd.items():
        d = _as_1d(c["d"], "d")
        A = _as_1d(c["A"], "A")
        phi = _as_1d(c["phi"], "phi")
        mask = _finite_mask(d, A, phi)
        d, A, phi = d[mask], A[mask], phi[mask]
        if d.size < 3:
            continue
        order = np.argsort(d)
        d, A, phi = d[order], A[order], phi[order]
        A0 = float(c.get("A0_nm", np.nanmedian(A[np.argsort(d)[-max(1, int(0.1 * d.size)):]])))
        dphi = np.abs(np.diff(_unwrap_phase_deg(phi)))
        da = np.abs(np.diff(A / max(A0, 1e-30)))
        if np.nanmax(dphi) > phase_jump_threshold_deg or np.nanmax(da) > amp_jump_threshold_rel:
            i = int(np.nanargmax(dphi)) if np.nanmax(dphi) / phase_jump_threshold_deg >= np.nanmax(da) / amp_jump_threshold_rel else int(np.nanargmax(da))
            flags[float(drv)] = {
                "max_dphi_deg": float(np.nanmax(dphi)),
                "max_da_rel": float(np.nanmax(da)),
                "d_at_jump_nm": float(d[i]),
            }
    return flags


# -----------------------------------------------------------------------------
# Minimal usage example
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    # measured_fd should look like:
    # measured_fd = {
    #     2.0: {"d": d_nm_2, "A": A_nm_2, "phi": phi_deg_2},
    #     8.0: {"d": d_nm_8, "A": A_nm_8, "phi": phi_deg_8},
    # }
    pass


In [ ]:
import numpy as np
from types import MethodType


def attach_continuous_fd_to_scanner(
    scanner,
    fd_lookup,
    conv_A=None,
):
    """
    Attach continuous-drive FD lookup to an existing ScannerFD instance.

    This works even if scanner only has scan_line_pi() and does not already
    have scan_line_trace_retrace_pi().

    Parameters
    ----------
    scanner : existing ScannerFD-like object
        Must have scan_line_pi().
    fd_lookup : callable
        fd_lookup(drive_nm) -> FDCurve1D in normalized units.
    conv_A : float
        Conversion from drive_nm to scanner-normalized amplitude.
        Usually scanner.conversion["A"].
    """

    if conv_A is None:
        conv_A = scanner.conversion.get("A", 1.0)

    scanner._continuous_fd_lookup = fd_lookup
    scanner._continuous_fd_conv_A = float(conv_A)

    def set_drive(self, drive_nm=None, drive_hat=None):
        if drive_nm is None and drive_hat is None:
            raise ValueError("Pass either drive_nm or drive_hat.")

        if drive_nm is None:
            drive_hat = float(drive_hat)
            drive_nm = drive_hat / self._continuous_fd_conv_A
        else:
            drive_nm = float(drive_nm)
            drive_hat = drive_nm * self._continuous_fd_conv_A

        fd_model = self._continuous_fd_lookup(drive_nm)

        # Classic ScannerFD expects fd_model directly.
        self.fd_model = fd_model

        # Keep amplitude bookkeeping consistent.
        if hasattr(self, "param"):
            self.param["A"] = drive_nm

        if hasattr(self, "param_norm"):
            self.param_norm["A"] = drive_hat
            if "Q" in self.param_norm:
                self.param_norm["C3"] = self.param_norm["A"] / self.param_norm["Q"]

        self.drive_nm_current = drive_nm
        self.drive_hat_current = drive_hat

        return fd_model

    def _pack_carry_state_from_trace(out):
        """
        Convert final state from trace into initial state for retrace.
        Only includes keys that actually exist in your scan_line_pi output.
        """
        pairs = {
            "d_cmd_init_hat": "d_cmd_final_hat",
            "d_act_init_hat": "d_final_hat",
            "A_meas_init_hat": "A_meas_final_hat",
            "phi_meas_init_deg": "phi_meas_final_deg",
            "integ_init": "integ_final",
            "drift_init_hat": "drift_final_hat",
        }

        carry = {}
        for init_key, final_key in pairs.items():
            if final_key in out:
                carry[init_key] = out[final_key]

        return carry

    def _flip_line_output(out):
        """
        Reverse 1D pixel arrays so retrace is returned in the same x-order
        as trace.
        """
        flipped = {}
        for k, v in out.items():
            if isinstance(v, np.ndarray) and v.ndim == 1:
                flipped[k] = v[::-1].copy()
            else:
                flipped[k] = v
        return flipped

    def scan_line_trace_retrace_pi_fallback(
        self,
        h_hat,
        retrace_carry_state=True,
        retrace_use_reversed_topography=True,
        **line_kwargs,
    ):
        h_hat = np.asarray(h_hat, dtype=float)

        # Trace
        out_trace = self.scan_line_pi(
            h_hat=h_hat,
            **line_kwargs,
        )

        # Retrace topography
        if retrace_use_reversed_topography:
            h_retrace = h_hat[::-1]
        else:
            h_retrace = h_hat.copy()

        # Carry state if available
        retrace_kwargs = dict(line_kwargs)
        if retrace_carry_state:
            retrace_kwargs.update(_pack_carry_state_from_trace(out_trace))

        out_retrace_raw = self.scan_line_pi(
            h_hat=h_retrace,
            **retrace_kwargs,
        )

        out_retrace = _flip_line_output(out_retrace_raw)

        return {
            "trace": out_trace,
            "retrace": out_retrace,
        }

    def scan_line_trace_retrace_pi_drive(
        self,
        h_hat,
        drive_nm=None,
        drive_hat=None,
        setpoint=None,
        setpoint_hat=None,
        **kwargs,
    ):
        fd_model = self.set_drive(drive_nm=drive_nm, drive_hat=drive_hat)

        if setpoint is None:
            if setpoint_hat is None:
                raise ValueError("Pass either setpoint fraction or setpoint_hat.")

            if hasattr(fd_model, "A0"):
                A0_hat = float(fd_model.A0)
            elif hasattr(fd_model, "free_amplitude"):
                A0_hat = float(fd_model.free_amplitude())
            else:
                A0_hat = float(self.param_norm["A"])

            setpoint = float(setpoint_hat) / A0_hat

        # If your scanner already has a native trace/retrace method, use it.
        # Otherwise use the fallback we attach below.
        return self.scan_line_trace_retrace_pi(
            h_hat=h_hat,
            setpoint=float(setpoint),
            **kwargs,
        )

    scanner.set_drive = MethodType(set_drive, scanner)

    if not hasattr(scanner, "scan_line_trace_retrace_pi"):
        scanner.scan_line_trace_retrace_pi = MethodType(
            scan_line_trace_retrace_pi_fallback,
            scanner,
        )

    scanner.scan_line_trace_retrace_pi_drive = MethodType(
        scan_line_trace_retrace_pi_drive,
        scanner,
    )

    return scanner

In [ ]:
import numpy as np

idx = -4

drive_nm = drives_si[idx]
d_nm     = height[idx]    # ascending, tip–sample distance
A_nm     = amp[idx]    # rising toward drive_nm at large d
phi_deg  = phase[idx]

fd = FDCurve1D(d_nm, A_nm, phi_deg, A0=drive_nm)

# Eyeball check before plugging into the loop
d_grid = np.linspace(d_nm.min(), d_nm.max(), 200)
A_grid = np.array([fd.evaluate(d)[0] for d in d_grid])
plt.plot(d_grid, A_grid); plt.axhline(drive_nm, ls='--')

# Build a scanner that always uses this one curve
params = {'k':40, 'A': drive_nm, 'd0': d_nm.max()*0.8, 'R': 10, 'H': 1e-19,
          'E_star': 1e9, 'Q': 150}
scanner = ScannerFD(params=params,
                    conversions={k: 1.0 for k in params},
                    fd_model=fd)

# Trivial fd_lookup: ignore the drive argument, always return `fd`
fd_lookup = lambda _drive: fd

phi_grid = np.array([fd.evaluate(d)[1] for d in d_grid])

plt.plot(d_grid, phi_grid); plt.axhline(drive_nm, ls='--')
# plt.plot(d_grid, A_grid); plt.axhline(drive_nm, ls='--')


In [ ]:
# from fd_surface_stage2 import (
#     FDDriveSurface,
#     make_normalized_fd_lookup,
#     attach_continuous_fd_to_scanner,
#     detect_bad_curves,
# )

# ------------------------------------------------------------
# 1. Check measured curves before building the surface
# ------------------------------------------------------------
flags = detect_bad_curves(
    measured_fd,
    phase_jump_threshold_deg=30,
    amp_jump_threshold_rel=0.20,
)

print(flags)

In [ ]:
# ------------------------------------------------------------
# 2. Build continuous FD surface from measured FD curves
# ------------------------------------------------------------
fd_surface = FDDriveSurface.from_measured_fd(
    measured_fd,
    x_mode="d_over_A0",          # recommended for your plotted curves
    common_range="intersection", # safest; use "union" if range overlap is too small
    n_x=800,
    far_field_frac=0.1,
    far_field_side="largest_d",  # because large height is far field in your plot

    # Use True if measured_fd keys are actual free amplitudes in nm.
    # For your plot, this is probably True.
    drive_key_is_A0=True,
)

In [ ]:
conversions

In [ ]:
# ------------------------------------------------------------
# 3. Convert physical FD curves into scanner-normalized units
# ------------------------------------------------------------
fd_lookup = make_normalized_fd_lookup(
    fd_surface,
    conv_L=1,   # or scanner.conversion["d0"]
    conv_A=1,
    conv_F=conversions.get("F", 1.0),
    n_points=800,
)

In [ ]:
# ------------------------------------------------------------
# 4. Attach continuous-drive FD model to scanner
# ------------------------------------------------------------
scanner = attach_continuous_fd_to_scanner(
    scanner,
    fd_lookup,
    conv_A=scanner.conversion["A"],
)

In [ ]:
scanner.scan_line_trace_retrace_pi_drive

In [ ]:
out = scanner.scan_line_trace_retrace_pi_drive(
    h_hat=h_truth,
    drive_nm=100,        # continuous drive, not limited to measured curves
    setpoint=0.6,        # continuous setpoint fraction of current A0

    scan_speed_hat=1e3,
    dx_hat=1.0,
    P=2.8,
    I=4.8,

    A_filter_order=1,
    tau_A=8e-4,
    phi_filter_order=1,
    tau_phi=5e-3,
    z_filter_order=1,
    tau_z=1e-5,

    d_init_hat=350,
    T_I=3e-3,
)

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out['trace']['d_hat'])
ax[0].plot(out['retrace']['d_hat'])
ax[1].plot(out['trace']['phi_deg'])
ax[1].plot(out['retrace']['phi_deg'])
ax[2].plot(out['trace']['A_hat'])
ax[2].plot(out['retrace']['A_hat'])

In [ ]:
# out = scanner.scan_line_trace_retrace_pi_drive(
#     h_hat=h_truth,
#     drive_nm=37.5,        # continuous drive, not limited to measured curves
#     setpoint=0.63,        # continuous setpoint fraction of current A0

#     scan_speed_hat=1e3,
#     dx_hat=1.0,
#     P=0.0,
#     I=4e3,

#     A_filter_order=1,
#     tau_A=8e-4,
#     phi_filter_order=1,
#     tau_phi=5e-3,
#     z_filter_order=1,
#     tau_z=1e-5,

#     d_init_hat=350,
#     T_I=3e-3,
# )

# Archived codes (05-03-2026)

In [ ]:
import numpy as np

def robust_rmse(x, clip_quantile=0.95):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.inf
    s = np.nanquantile(np.abs(x), clip_quantile)
    if s <= 0:
        return np.sqrt(np.nanmean(x**2))
    x_clip = np.clip(x, -s, s)
    return np.sqrt(np.nanmean(x_clip**2))


def best_offset_rmse(y, ref):
    """
    Compare y and ref after removing constant offset.
    """
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)
    offset = np.nanmedian(y - ref)
    return robust_rmse(y - offset - ref)


def edge_mask_from_truth(h, percentile=90, pad=3):
    """
    Build a mask around sharp topographic edges.
    """
    h = np.asarray(h, dtype=float)
    dh = np.abs(np.gradient(h))
    threshold = np.nanpercentile(dh, percentile)
    mask = dh > threshold

    if pad > 0:
        mask2 = mask.copy()
        idx = np.where(mask)[0]
        for k in idx:
            lo = max(0, k - pad)
            hi = min(len(h), k + pad + 1)
            mask2[lo:hi] = True
        mask = mask2

    return mask


def score_scan_quality(out, h_truth, setpoint, A0=None, ignore_first=20):
    """
    Lower is better.
    Assumes out has trace/retrace dictionaries or arrays.
    You may need to adjust key names depending on your output format.
    """

    h_truth = np.asarray(h_truth, dtype=float)

    # Adjust these keys to match your scan_line_trace_retrace_pi output
    h_tr = np.asarray(out["trace"]["d_hat"], dtype=float)
    h_rt = np.asarray(out["retrace"]["d_hat"], dtype=float)

    A_tr = np.asarray(out["trace"]["A_hat"], dtype=float)
    A_rt = np.asarray(out["retrace"]["A_hat"], dtype=float)

    # If retrace is stored in reverse scan order, flip it before comparison.
    # If your plotting function already returns it aligned, remove these two lines.
    if h_rt.shape == h_tr.shape:
        h_rt_aligned = h_rt
        A_rt_aligned = A_rt
    else:
        h_rt_aligned = h_rt
        A_rt_aligned = A_rt

    sl = slice(ignore_first, None)

    # Height tracking
    loss_h_tr = best_offset_rmse(h_tr[sl], h_truth[sl])
    loss_h_rt = best_offset_rmse(h_rt_aligned[sl], h_truth[sl])

    # Trace-retrace consistency
    loss_trrt = robust_rmse((h_tr - h_rt_aligned)[sl])

    # Edge overshoot penalty
    edge_mask = edge_mask_from_truth(h_truth)
    edge_mask[:ignore_first] = False
    if np.any(edge_mask):
        loss_edge = robust_rmse((h_tr - h_rt_aligned)[edge_mask])
    else:
        loss_edge = 0.0

    # Amplitude setpoint error
    if A0 is None:
        A0 = np.nanmax([np.nanmax(A_tr), np.nanmax(A_rt)])
    A_set = setpoint * A0
    loss_A = 0.5 * (
        robust_rmse((A_tr - A_set)[sl]) +
        robust_rmse((A_rt_aligned - A_set)[sl])
    )

    # Weights: adjust based on what you care about most
    total = (
        1.0 * (loss_h_tr + loss_h_rt) / 2.0
        + 0.5 * loss_trrt
        + 1.0 * loss_edge
        + 0.2 * loss_A
    )

    return {
        "total": float(total),
        "height": float((loss_h_tr + loss_h_rt) / 2.0),
        "trace_retrace": float(loss_trrt),
        "edge": float(loss_edge),
        "amplitude": float(loss_A),
    }

In [ ]:
from itertools import product
import pandas as pd
import numpy as np


def run_quality_grid(scanner, h_truth, setpoint=0.6, scan_speed_hat=1e3, dx_hat=1.0):
    dt = dx_hat / scan_speed_hat

    P_grid = np.array([0.03, 0.06, 0.1, 0.2, 0.4, 0.8])
    I_grid = np.array([0.0, 0.005, 0.015, 0.05, 0.15, 0.4])

    tau_A_grid = dt * np.array([2, 5, 10, 20])
    tau_z_grid = dt * np.array([1, 2, 5, 10])

    z_rate_grid = [None, 10.0, 20.0, 50.0]
    integ_clip_grid = [5.0, 10.0, 20.0, 50.0]

    rows = []

    for P, I, tau_A, tau_z, z_rate, integ_clip in product(
        P_grid,
        I_grid,
        tau_A_grid,
        tau_z_grid,
        z_rate_grid,
        integ_clip_grid,
    ):
        try:
            out = scanner.scan_line_trace_retrace_pi(
                h_hat=h_truth,
                setpoint=setpoint,
                scan_speed_hat=scan_speed_hat,
                dx_hat=dx_hat,

                P=float(P),
                I=float(I),

                A_filter_order=1,
                tau_A=float(tau_A),

                phi_filter_order=1,
                tau_phi=float(tau_A),

                z_filter_order=1,
                tau_z=float(tau_z),

                z_rate_limit_hat=z_rate,
                integ_clip=integ_clip,
                T_I=20 * dt,
            )

            score = score_scan_quality(
                out=out,
                h_truth=h_truth,
                setpoint=setpoint,
                A0=None,
                ignore_first=20,
            )

            rows.append({
                "P": P,
                "I": I,
                "tau_A": tau_A,
                "tau_z": tau_z,
                "z_rate_limit_hat": z_rate,
                "integ_clip": integ_clip,
                **score,
            })

        except Exception as e:
            rows.append({
                "P": P,
                "I": I,
                "tau_A": tau_A,
                "tau_z": tau_z,
                "z_rate_limit_hat": z_rate,
                "integ_clip": integ_clip,
                "total": np.inf,
                "height": np.inf,
                "trace_retrace": np.inf,
                "edge": np.inf,
                "amplitude": np.inf,
                "error": repr(e),
            })

    df = pd.DataFrame(rows)
    df = df.sort_values("total").reset_index(drop=True)
    return df

In [ ]:
# Approach-curve library + cross-drive interpolation.

"""
Approach-curve library + cross-drive interpolation.

Strategy
--------
Every measured approach curve is single-valued in z when parameterized by
the slow z-piezo. Store curves in physical units (nm, deg) per drive A0.

To interpolate to an arbitrary drive A0*:
  1) Resample each curve onto a common normalized grid u = d / A0,
     storing the normalized amplitude a = A / A0 and the phase phi.
  2) Linearly interpolate a(u, A0) and phi(u, A0) in A0 at fixed u.
  3) Convert back: d = u * A0*,  A = a * A0*.

Optional force F(d) is carried through if provided.

Public API
----------
FDCurve1D                 : single-curve interpolator with evaluate(d) -> (A, phi, F)
ApproachCurveLibrary      : add curves, query at any drive, returns FDCurve1D
make_normalized_fd_lookup : adapter that wraps the library in the normalized-units
                            API expected by ScannerFD (matches calibrate_scanner.py)
detect_bistability        : utility to flag curves with phase jumps before fitting
"""
from __future__ import annotations

import numpy as np
from scipy.interpolate import interp1d
from typing import Optional, Tuple, Dict, List, Callable


# -----------------------------------------------------------------------------
# Single-curve evaluator
# -----------------------------------------------------------------------------

class FDCurve1D:
    """Interpolated A(d), phi(d), F(d) at a single drive amplitude.

    Conventions
    -----------
    Inputs in any consistent unit system (typically nm, degrees).
    d increases away from the surface; the largest d in the curve is treated
    as the "free" point, A(d_max) ~ A0.

    Out-of-range behavior
    ---------------------
    d > d_max : A clamped to A0 (free), phi clamped to phi(d_max), F -> 0.
    d < d_min : edge-clamped (no extrapolation into untrusted near-contact regime).
    """

    def __init__(self, d, A, phi, F=None, *, A0: Optional[float] = None,
                 kind: str = 'linear'):
        d = np.asarray(d, dtype=float)
        A = np.asarray(A, dtype=float)
        phi = np.asarray(phi, dtype=float)
        F = np.zeros_like(d) if F is None else np.asarray(F, dtype=float)

        order = np.argsort(d)
        self.d   = d[order]
        self.A   = A[order]
        self.phi = phi[order]
        self.F   = F[order]
        self.A0 = float(self.A[-1]) if A0 is None else float(A0)
        self.kind = kind

        self._A = interp1d(
            self.d, self.A, kind=kind, bounds_error=False,
            fill_value=(self.A[0], self.A0),  # left = nearest, right = free
        )
        self._phi = interp1d(
            self.d, self.phi, kind=kind, bounds_error=False,
            fill_value=(self.phi[0], self.phi[-1]),
        )
        self._F = interp1d(
            self.d, self.F, kind=kind, bounds_error=False,
            fill_value=(self.F[0], 0.0),
        )

    def evaluate(self, d):
        """Return (A, phi, F). Scalar in -> scalar out; array in -> array out."""
        d_arr = np.atleast_1d(d).astype(float)
        A   = self._A(d_arr)
        phi = self._phi(d_arr)
        F   = self._F(d_arr)
        if np.isscalar(d):
            return float(A[0]), float(phi[0]), float(F[0])
        return A, phi, F


# -----------------------------------------------------------------------------
# Library
# -----------------------------------------------------------------------------

class ApproachCurveLibrary:
    """Collection of approach curves at multiple drive amplitudes (nm).

    Each curve is stored as 1D arrays (d_nm, A_nm, phi_deg, optional F).
    The curves do NOT need to share a common d-grid or have the same length.
    """

    def __init__(self):
        self._curves: Dict[float, Dict[str, Optional[np.ndarray]]] = {}

    # ----- ingestion -----

    def add(self, drive_nm: float, d_nm, A_nm, phi_deg, F=None) -> None:
        """Add one approach curve at a known drive amplitude (in nm).

        Parameters
        ----------
        drive_nm : free-amplitude drive setting for this curve, nm.
        d_nm     : 1D array of z-piezo positions, nm. Increases away from surface.
        A_nm     : 1D array of measured oscillation amplitudes, nm.
        phi_deg  : 1D array of measured phases, degrees.
        F        : optional 1D array (e.g. peak force or virial), arbitrary units.
        """
        d = np.asarray(d_nm,    dtype=float).ravel()
        A = np.asarray(A_nm,    dtype=float).ravel()
        phi = np.asarray(phi_deg, dtype=float).ravel()
        if not (d.shape == A.shape == phi.shape):
            raise ValueError("d, A, phi must be the same length")
        F_arr = None
        if F is not None:
            F_arr = np.asarray(F, dtype=float).ravel()
            if F_arr.shape != d.shape:
                raise ValueError("F must match d length")

        # Drop NaNs and sort by d ascending
        mask = np.isfinite(d) & np.isfinite(A) & np.isfinite(phi)
        if F_arr is not None:
            mask &= np.isfinite(F_arr)
        d, A, phi = d[mask], A[mask], phi[mask]
        if F_arr is not None:
            F_arr = F_arr[mask]
        order = np.argsort(d)

        self._curves[float(drive_nm)] = {
            'd':   d[order],
            'A':   A[order],
            'phi': phi[order],
            'F':   None if F_arr is None else F_arr[order],
        }

    @classmethod
    def from_dict(cls, by_drive: Dict[float, Dict[str, np.ndarray]]) -> 'ApproachCurveLibrary':
        """Bulk-construct from {drive_nm: {'d':..., 'A':..., 'phi':..., 'F':...}}."""
        lib = cls()
        for drv, c in by_drive.items():
            lib.add(drv, c['d'], c['A'], c['phi'], F=c.get('F'))
        return lib

    # ----- inspection -----

    @property
    def drives(self) -> np.ndarray:
        return np.array(sorted(self._curves.keys()))

    def get_raw(self, drive_nm: float) -> Dict[str, Optional[np.ndarray]]:
        return self._curves[float(drive_nm)]

    # ----- normalization -----

    def normalized_grid(
        self,
        n_u: int = 400,
        u_min: Optional[float] = None,
        u_max: Optional[float] = None,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """Resample all curves onto a common u = d/A0 grid.

        Returns
        -------
        drives   : (n_drives,)        sorted drive amplitudes [nm]
        u_grid   : (n_u,)             common normalized distance grid
        a_grid   : (n_drives, n_u)    normalized amplitude  A/A0
        phi_grid : (n_drives, n_u)    phase [deg]
        F_grid   : (n_drives, n_u)    force (zero where F not supplied)
        """
        drives = self.drives
        if drives.size < 2:
            raise ValueError("Library needs at least two drives to interpolate.")
        umins = [self._curves[d]['d'].min() / d for d in drives]
        umaxs = [self._curves[d]['d'].max() / d for d in drives]

        u_min = max(umins) if u_min is None else float(u_min)
        u_max = min(umaxs) if u_max is None else float(u_max)
        if not (u_max > u_min):
            raise ValueError(
                f"No common u-range across curves "
                f"(largest u_min={max(umins):.3g}, smallest u_max={min(umaxs):.3g}). "
                "Trim outlier curves or pass u_min/u_max manually."
            )
        u_grid = np.linspace(u_min, u_max, int(n_u))

        nD = drives.size
        a_grid   = np.empty((nD, n_u))
        phi_grid = np.empty((nD, n_u))
        F_grid   = np.zeros((nD, n_u))

        for i, drv in enumerate(drives):
            c = self._curves[drv]
            u = c['d'] / drv
            a = c['A'] / drv
            a_grid[i]   = np.interp(u_grid, u, a)
            phi_grid[i] = np.interp(u_grid, u, c['phi'])
            if c['F'] is not None:
                F_grid[i] = np.interp(u_grid, u, c['F'])

        return drives, u_grid, a_grid, phi_grid, F_grid

    # ----- interpolation across drives -----

    def fd_at_drive(self, drive_target_nm: float, n_u: int = 400) -> FDCurve1D:
        """Build an FDCurve1D for an arbitrary target drive amplitude (nm).

        Linear interpolation in (u, A0). Outside [drives.min, drives.max],
        clamps to the nearest curve (no extrapolation across drive).
        """
        drives, u_grid, a_grid, phi_grid, F_grid = self.normalized_grid(n_u=n_u)

        if drive_target_nm <= drives[0]:
            a_t, phi_t, F_t = a_grid[0], phi_grid[0], F_grid[0]
        elif drive_target_nm >= drives[-1]:
            a_t, phi_t, F_t = a_grid[-1], phi_grid[-1], F_grid[-1]
        else:
            idx = int(np.searchsorted(drives, drive_target_nm) - 1)
            d_lo, d_hi = drives[idx], drives[idx + 1]
            w = (drive_target_nm - d_lo) / (d_hi - d_lo)
            a_t   = (1 - w) * a_grid[idx]   + w * a_grid[idx + 1]
            phi_t = (1 - w) * phi_grid[idx] + w * phi_grid[idx + 1]
            F_t   = (1 - w) * F_grid[idx]   + w * F_grid[idx + 1]

        # Convert back to physical d, A
        d_phys = u_grid * float(drive_target_nm)
        A_phys = a_t   * float(drive_target_nm)
        return FDCurve1D(d_phys, A_phys, phi_t, F=F_t, A0=float(drive_target_nm))

    # convenience alias
    def __call__(self, drive_target_nm: float) -> FDCurve1D:
        return self.fd_at_drive(drive_target_nm)


# -----------------------------------------------------------------------------
# Adapter to the normalized-units API expected by ScannerFD
# -----------------------------------------------------------------------------

def make_normalized_fd_lookup(
    library: ApproachCurveLibrary,
    conv_L: float,
    conv_A: Optional[float] = None,
    conv_F: float = 1.0,
    n_u: int = 400,
) -> Callable[[float], FDCurve1D]:
    """Wrap a physical-units library in a callable returning FDCurve1D in
    the *normalized* units the ScannerFD expects internally.

    Parameters
    ----------
    library : ApproachCurveLibrary in physical units (nm, deg).
    conv_L  : length conversion (d_norm = d_nm * conv_L). In ScannerFD all
              length-like params share the same conversion factor, so pass
              scanner.conversion['A'] (or equivalently scanner.conversion['d0']).
    conv_A  : amplitude conversion (A_norm = A_nm * conv_A). Defaults to conv_L,
              which is the right thing since amplitude IS a length.
    conv_F  : force conversion. Pass 1.0 if your library doesn't carry F, or
              the relevant scanner.conversion key if it does.
    n_u     : grid size for the normalized resampling step.

    The returned callable takes a drive in *physical* nm and returns an
    FDCurve1D whose evaluate(d_hat) accepts and returns *normalized* values.
    Use this directly as `fd_lookup` in calibrate_scanner.calibrate(...).
    """
    if conv_A is None:
        conv_A = conv_L

    def lookup(drive_nm: float) -> FDCurve1D:
        fd_phys = library.fd_at_drive(float(drive_nm), n_u=n_u)
        return FDCurve1D(
            d=fd_phys.d * conv_L,
            A=fd_phys.A * conv_A,
            phi=fd_phys.phi,
            F=fd_phys.F * conv_F,
            A0=float(drive_nm) * conv_A,
        )
    return lookup


# -----------------------------------------------------------------------------
# Diagnostics
# -----------------------------------------------------------------------------

def detect_bistability(
    library: ApproachCurveLibrary,
    phase_jump_threshold_deg: float = 25.0,
    amp_jump_threshold_rel: float = 0.15,
) -> Dict[float, Dict[str, float]]:
    """Flag curves where phi or A/A0 jumps more than thresholds between
    adjacent samples in d. These are likely to be bistable and should be
    inspected (or split into branches) before being put into the library.

    Returns {drive: {'max_dphi_deg': ..., 'max_da_rel': ..., 'd_at_jump': ...}}
    only for curves that exceed at least one threshold.
    """
    flags: Dict[float, Dict[str, float]] = {}
    for drv in library.drives:
        c = library.get_raw(drv)
        d, A, phi = c['d'], c['A'], c['phi']
        if d.size < 3:
            continue
        dphi = np.abs(np.diff(phi))
        da_rel = np.abs(np.diff(A)) / float(drv)
        i_phi = int(np.argmax(dphi)) if dphi.size else 0
        i_a   = int(np.argmax(da_rel)) if da_rel.size else 0
        if dphi[i_phi] > phase_jump_threshold_deg or da_rel[i_a] > amp_jump_threshold_rel:
            phi_severity = dphi[i_phi] / phase_jump_threshold_deg
            a_severity   = da_rel[i_a] / amp_jump_threshold_rel
            i_jump = i_phi if phi_severity > a_severity else i_a
            flags[float(drv)] = {
                'max_dphi_deg': float(dphi.max()),
                'max_da_rel':   float(da_rel.max()),
                'd_at_jump':    float(d[i_jump]),
            }
    return flags




In [ ]:
# Calibrate a ScannerFD digital twin against experimental scan lines.
"""
Calibrate a ScannerFD digital twin against experimental scan lines.

Free parameters fit (theta, length 6):
    a_drive, b_drive          drive_sim    = a_drive * drive_exp    + b_drive
    a_setpoint, b_setpoint    setpoint_sim = a_setpoint * setpoint_exp + b_setpoint
    tau_A                     amplitude / phase demodulator time constant (1st-order LPF)
    tau_z                     Z-actuator time constant (1st-order LPF)

Held fixed per scan line:
    I_gain                    taken from each experiment record
    P_gain                    0 by default; expose as an extra theta entry if needed

Inputs the user supplies:
    scanner       : a ScannerFD instance configured with nominal physical params
    fd_lookup(A)  : callable returning an FDCurve1D for a given drive amplitude (nm)
                    (interpolate your approach-curve library here)
    h_truth_hat   : 1D ground-truth topography (already in the normalized length
                    units used by the scanner, i.e. h * conversion['L'])
    experiments   : list of dicts (one per scan line). Use flatten_experiments(...)
                    to build it from a packed (D, S, I, 4, P) array.
"""
from __future__ import annotations

import numpy as np
from scipy.optimize import least_squares
from typing import Callable, Iterable, List, Dict, Optional, Tuple


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def wrap_phase_diff(phi_a_deg, phi_b_deg) -> np.ndarray:
    """Phase difference (a - b) wrapped to (-180, 180] degrees."""
    d = np.asarray(phi_a_deg, dtype=float) - np.asarray(phi_b_deg, dtype=float)
    return (d + 180.0) % 360.0 - 180.0


def set_scanner_drive(scanner, A_nm: float, fd_model) -> None:
    """Switch scanner free amplitude and FD lookup in place."""
    scanner.param['A'] = float(A_nm)
    scanner.param_norm['A'] = float(A_nm) * scanner.conversion['A']
    scanner.param_norm['C3'] = scanner.param_norm['A'] / scanner.param_norm['Q']
    scanner.fd_model = fd_model


# -----------------------------------------------------------------------------
# Forward model: one simulated line, both directions
# -----------------------------------------------------------------------------

def simulate_trace_retrace(
    scanner,
    fd_lookup: Callable[[float], object],
    h_truth_hat: np.ndarray,
    drive_nm: float,
    setpoint_frac: float,
    I_gain: float,
    tau_A: float,
    tau_z: float,
    P_gain: float = 0.0,
    tau_phi: Optional[float] = None,
    burn_in: int = 128,
    scan_speed_hat: float = 1.0,
    dx_hat: float = 1.0,
) -> Dict[str, np.ndarray]:
    """Simulate one scan line forward (trace) and backward (retrace).

    Returns a dict with h_{trace,retrace}, A_{trace,retrace}, phi_{trace,retrace},
    each of length len(h_truth_hat). The retrace arrays are returned in the
    same x-order as the trace (i.e. left-to-right) for direct comparison.
    """
    fd_model = fd_lookup(drive_nm)
    set_scanner_drive(scanner, drive_nm, fd_model)

    if tau_phi is None:
        tau_phi = tau_A

    common = dict(
        setpoint=setpoint_frac,
        scan_speed_hat=scan_speed_hat,
        dx_hat=dx_hat,
        P=P_gain,
        I=I_gain,
        A_filter_order=1, tau_A=tau_A,
        phi_filter_order=1, tau_phi=tau_phi,
        z_filter_order=1, tau_z=tau_z,
    )

    # --- Trace: pad with first sample, scan left-to-right, drop the burn-in ---
    pad_t = np.full(burn_in, h_truth_hat[0], dtype=float)
    h_in_t = np.concatenate([pad_t, h_truth_hat])
    out_t = scanner.scan_line_pi(h_in_t, **common)

    h_trace   = out_t['d_hat'][burn_in:]
    A_trace   = out_t['A_hat'][burn_in:]
    phi_trace = out_t['phi_deg'][burn_in:]

    # --- Retrace: scan reversed h, carry controller state from end of trace ---
    pad_r = np.full(burn_in, h_truth_hat[-1], dtype=float)
    h_in_r = np.concatenate([pad_r, h_truth_hat[::-1]])
    out_r = scanner.scan_line_pi(
        h_in_r,
        d_cmd_init_hat=out_t['d_cmd_final_hat'],
        d_act_init_hat=out_t['d_final_hat'],
        A_meas_init_hat=out_t['A_meas_final_hat'],
        phi_meas_init_deg=out_t['phi_meas_final_deg'],
        integ_init=out_t['integ_final'],
        **common,
    )
    # Drop burn-in, then reverse so x-order matches the trace.
    h_retrace   = out_r['d_hat'][burn_in:][::-1]
    A_retrace   = out_r['A_hat'][burn_in:][::-1]
    phi_retrace = out_r['phi_deg'][burn_in:][::-1]

    return dict(
        h_trace=h_trace,     h_retrace=h_retrace,
        A_trace=A_trace,     A_retrace=A_retrace,
        phi_trace=phi_trace, phi_retrace=phi_retrace,
    )


# -----------------------------------------------------------------------------
# Pack / unpack experimental data
# -----------------------------------------------------------------------------

def flatten_experiments(
    data: np.ndarray,
    drives: np.ndarray,
    setpoints: np.ndarray,
    I_gains: np.ndarray,
    channel_order: Tuple[str, ...] = (
        'h_trace', 'h_retrace', 'phi_trace', 'phi_retrace'
    ),
) -> List[Dict]:
    """Convert a packed (n_drive, n_setpoint, n_Igain, n_channel, n_pix) array
    into a flat list of per-line dicts.

    The default channel_order matches your description "trace and retrace of
    heights + phases" interpreted as (h_trace, h_retrace, phi_trace, phi_retrace).
    Override channel_order if your packing differs.
    """
    data = np.asarray(data)
    nD, nS, nI, nC, nP = data.shape
    if nC != len(channel_order):
        raise ValueError(f"Expected {len(channel_order)} channels, got {nC}")
    if not (len(drives) == nD and len(setpoints) == nS and len(I_gains) == nI):
        raise ValueError("drives / setpoints / I_gains lengths must match data shape")

    rows: List[Dict] = []
    for i in range(nD):
        for j in range(nS):
            for k in range(nI):
                line = {ch: np.asarray(data[i, j, k, ci, :], dtype=float)
                        for ci, ch in enumerate(channel_order)}
                rows.append({
                    'drive':    float(drives[i]),
                    'setpoint': float(setpoints[j]),
                    'I_gain':   float(I_gains[k]),
                    **line,
                })
    return rows


# -----------------------------------------------------------------------------
# Cost function
# -----------------------------------------------------------------------------

def make_residual_fn(
    scanner,
    fd_lookup: Callable[[float], object],
    h_truth_hat: np.ndarray,
    experiments: List[Dict],
    weights: Optional[Dict[str, float]] = None,
    scan_speed_hat: float = 1.0,
    dx_hat: float = 1.0,
    burn_in: int = 128,
    huge_penalty: float = 1e6,
):
    """Build a residual vector: stacked (h_trace, h_retrace, phi_trace, phi_retrace)
    differences across all experimental lines. Phase residuals are wrapped.

    Default weights make 1 deg of phase ~ 0.01 normalized length unit.
    """
    if weights is None:
        weights = {'h': 1.0, 'phi': 0.01}

    h_truth_hat = np.asarray(h_truth_hat, dtype=float)
    n_pix = h_truth_hat.size

    def residual(theta: np.ndarray) -> np.ndarray:
        a_d, b_d, a_s, b_s, log_tau_A, log_tau_z = theta
        tau_A = float(np.exp(log_tau_A))
        tau_z = float(np.exp(log_tau_z))

        chunks: List[np.ndarray] = []
        for exp in experiments:
            drive_sim    = a_d * exp['drive']    + b_d
            setpoint_sim = a_s * exp['setpoint'] + b_s

            # Refuse pathological setpoints early (saves a sim call)
            if not (0.01 < setpoint_sim < 0.999) or drive_sim <= 0.0:
                chunks.append(np.full(4 * n_pix, huge_penalty))
                continue

            try:
                sim = simulate_trace_retrace(
                    scanner, fd_lookup, h_truth_hat,
                    drive_nm=float(drive_sim),
                    setpoint_frac=float(setpoint_sim),
                    I_gain=float(exp['I_gain']),
                    tau_A=tau_A, tau_z=tau_z,
                    burn_in=burn_in,
                    scan_speed_hat=scan_speed_hat,
                    dx_hat=dx_hat,
                )
            except Exception:
                # FD lookup outside library, NaN on a branch jump, etc.
                chunks.append(np.full(4 * n_pix, huge_penalty))
                continue

            r_h_t = (sim['h_trace']   - exp['h_trace'])   * weights['h']
            r_h_r = (sim['h_retrace'] - exp['h_retrace']) * weights['h']
            r_p_t = wrap_phase_diff(sim['phi_trace'],   exp['phi_trace'])   * weights['phi']
            r_p_r = wrap_phase_diff(sim['phi_retrace'], exp['phi_retrace']) * weights['phi']
            chunks.extend([r_h_t, r_h_r, r_p_t, r_p_r])

        return np.concatenate(chunks)

    return residual


# -----------------------------------------------------------------------------
# Top-level driver
# -----------------------------------------------------------------------------

def calibrate(
    scanner,
    fd_lookup: Callable[[float], object],
    h_truth_hat: np.ndarray,
    experiments: List[Dict],
    x0: Optional[np.ndarray] = None,
    bounds: Optional[Tuple[Iterable[float], Iterable[float]]] = None,
    weights: Optional[Dict[str, float]] = None,
    scan_speed_hat: float = 1.0,
    dx_hat: float = 1.0,
    burn_in: int = 128,
    verbose: int = 2,
    max_nfev: int = 200,
) -> Dict:
    """Fit (a_drive, b_drive, a_setpoint, b_setpoint, tau_A, tau_z).

    h_truth_hat : 1D ground-truth profile in the same normalized length units
                  the scanner uses internally (i.e. physical_h * conversion['L']).
    experiments : list of dicts with keys
                  drive (nm, in EXPERIMENTAL units), setpoint (fraction, in
                  EXPERIMENTAL units), I_gain, h_trace, h_retrace,
                  phi_trace, phi_retrace.
    fd_lookup   : drive_nm_simulator -> FDCurve1D (interpolate your library).

    Returns a dict with the fitted parameters and the scipy result object.
    """
    if x0 is None:
        # Identity transforms; tau_A, tau_z log-space initial guesses.
        x0 = np.array([1.0, 0.0, 1.0, 0.0, np.log(1e-3), np.log(1e-4)])

    if bounds is None:
        # Generous but bounded; tighten if you have priors on calibration error.
        lo = [0.5, -20.0, 0.7, -0.20, np.log(1e-7), np.log(1e-7)]
        hi = [2.0,  20.0, 1.5,  0.20, np.log(1e0),  np.log(1e0)]
        bounds = (lo, hi)

    res_fn = make_residual_fn(
        scanner, fd_lookup, h_truth_hat, experiments,
        weights=weights, scan_speed_hat=scan_speed_hat,
        dx_hat=dx_hat, burn_in=burn_in,
    )

    sol = least_squares(
        res_fn, x0, bounds=bounds,
        method='trf', x_scale='jac',
        verbose=verbose, max_nfev=max_nfev,
    )

    a_d, b_d, a_s, b_s, log_tau_A, log_tau_z = sol.x
    return {
        'a_drive':    float(a_d),
        'b_drive':    float(b_d),
        'a_setpoint': float(a_s),
        'b_setpoint': float(b_s),
        'tau_A':      float(np.exp(log_tau_A)),
        'tau_z':      float(np.exp(log_tau_z)),
        'cost':       float(sol.cost),
        'theta':      sol.x,
        'result':     sol,
    }


# -----------------------------------------------------------------------------
# Diagnostics
# -----------------------------------------------------------------------------

def evaluate_fit(
    fit: Dict,
    scanner,
    fd_lookup: Callable[[float], object],
    h_truth_hat: np.ndarray,
    experiments: List[Dict],
    scan_speed_hat: float = 1.0,
    dx_hat: float = 1.0,
    burn_in: int = 128,
) -> List[Dict]:
    """Re-run the simulator at the fitted theta for each experiment line.
    Useful for plotting sim-vs-exp diagnostics after calibration.
    """
    a_d, b_d = fit['a_drive'], fit['b_drive']
    a_s, b_s = fit['a_setpoint'], fit['b_setpoint']
    tau_A, tau_z = fit['tau_A'], fit['tau_z']

    out = []
    for exp in experiments:
        sim = simulate_trace_retrace(
            scanner, fd_lookup, h_truth_hat,
            drive_nm=a_d * exp['drive'] + b_d,
            setpoint_frac=a_s * exp['setpoint'] + b_s,
            I_gain=exp['I_gain'],
            tau_A=tau_A, tau_z=tau_z,
            burn_in=burn_in,
            scan_speed_hat=scan_speed_hat, dx_hat=dx_hat,
        )
        out.append({'exp': exp, 'sim': sim})
    return out




In [ ]:
# Updated: Calibrate a ScannerFD digital twin against experimental scan lines.
"""
Calibrate a ScannerFD digital twin against experimental scan lines.

Free parameters fit (theta, length 7):
    a_drive, b_drive          drive_sim    = a_drive * drive_exp    + b_drive
    a_setpoint, b_setpoint    setpoint_sim = a_setpoint * setpoint_exp + b_setpoint
    b_I_log10                 log-space offset on integral gain
                              I_sim = 10**(b_I_log10 + log10_I_exp)   if i_gain_is_log10=True
                              I_sim = 10**b_I_log10 * I_exp           if i_gain_is_log10=False
                              (i.e., a single multiplicative factor c_I = 10**b_I_log10)
    tau_A                     amplitude / phase demodulator time constant (1st-order LPF)
    tau_z                     Z-actuator time constant (1st-order LPF)

Held fixed per scan line:
    P_gain                    0 by default; expose as an extra theta entry if needed

Inputs the user supplies:
    scanner       : a ScannerFD instance configured with nominal physical params
    fd_lookup(A)  : callable returning an FDCurve1D for a given drive amplitude (nm)
                    (interpolate your approach-curve library here)
    h_truth_hat   : 1D ground-truth topography (already in the normalized length
                    units used by the scanner, i.e. h * conversion['L'])
    experiments   : list of dicts (one per scan line). Use flatten_experiments(...)
                    to build it from a packed (D, S, I, 4, P) array.
"""
from __future__ import annotations

import numpy as np
from scipy.optimize import least_squares
from typing import Callable, Iterable, List, Dict, Optional, Tuple


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def wrap_phase_diff(phi_a_deg, phi_b_deg) -> np.ndarray:
    """Phase difference (a - b) wrapped to (-180, 180] degrees."""
    d = np.asarray(phi_a_deg, dtype=float) - np.asarray(phi_b_deg, dtype=float)
    return (d + 180.0) % 360.0 - 180.0


def set_scanner_drive(scanner, A_nm: float, fd_model) -> None:
    """Switch scanner free amplitude and FD lookup in place."""
    scanner.param['A'] = float(A_nm)
    scanner.param_norm['A'] = float(A_nm) * scanner.conversion['A']
    scanner.param_norm['C3'] = scanner.param_norm['A'] / scanner.param_norm['Q']
    scanner.fd_model = fd_model


# -----------------------------------------------------------------------------
# Forward model: one simulated line, both directions
# -----------------------------------------------------------------------------

def simulate_trace_retrace(
    scanner,
    fd_lookup: Callable[[float], object],
    h_truth_hat: np.ndarray,
    drive_nm: float,
    setpoint_frac: float,
    I_gain: float,
    tau_A: float,
    tau_z: float,
    P_gain: float = 0.0,
    tau_phi: Optional[float] = None,
    burn_in: int = 128,
    scan_speed_hat: float = 1.0,
    dx_hat: float = 1.0,
) -> Dict[str, np.ndarray]:
    """Simulate one scan line forward (trace) and backward (retrace).

    Returns a dict with h_{trace,retrace}, A_{trace,retrace}, phi_{trace,retrace},
    each of length len(h_truth_hat). The retrace arrays are returned in the
    same x-order as the trace (i.e. left-to-right) for direct comparison.
    """
    fd_model = fd_lookup(drive_nm)
    set_scanner_drive(scanner, drive_nm, fd_model)

    if tau_phi is None:
        tau_phi = tau_A

    common = dict(
        setpoint=setpoint_frac,
        scan_speed_hat=scan_speed_hat,
        dx_hat=dx_hat,
        P=P_gain,
        I=I_gain,
        A_filter_order=1, tau_A=tau_A,
        phi_filter_order=1, tau_phi=tau_phi,
        z_filter_order=1, tau_z=tau_z,
    )

    # --- Trace: pad with first sample, scan left-to-right, drop the burn-in ---
    pad_t = np.full(burn_in, h_truth_hat[0], dtype=float)
    h_in_t = np.concatenate([pad_t, h_truth_hat])
    out_t = scanner.scan_line_pi(h_in_t, **common)

    h_trace   = out_t['d_hat'][burn_in:]
    A_trace   = out_t['A_hat'][burn_in:]
    phi_trace = out_t['phi_deg'][burn_in:]

    # --- Retrace: scan reversed h, carry controller state from end of trace ---
    pad_r = np.full(burn_in, h_truth_hat[-1], dtype=float)
    h_in_r = np.concatenate([pad_r, h_truth_hat[::-1]])
    out_r = scanner.scan_line_pi(
        h_in_r,
        d_cmd_init_hat=out_t['d_cmd_final_hat'],
        d_act_init_hat=out_t['d_final_hat'],
        A_meas_init_hat=out_t['A_meas_final_hat'],
        phi_meas_init_deg=out_t['phi_meas_final_deg'],
        integ_init=out_t['integ_final'],
        **common,
    )
    # Drop burn-in, then reverse so x-order matches the trace.
    h_retrace   = out_r['d_hat'][burn_in:][::-1]
    A_retrace   = out_r['A_hat'][burn_in:][::-1]
    phi_retrace = out_r['phi_deg'][burn_in:][::-1]

    return dict(
        h_trace=h_trace,     h_retrace=h_retrace,
        A_trace=A_trace,     A_retrace=A_retrace,
        phi_trace=phi_trace, phi_retrace=phi_retrace,
    )


# -----------------------------------------------------------------------------
# Pack / unpack experimental data
# -----------------------------------------------------------------------------

def flatten_experiments(
    data: np.ndarray,
    drives: np.ndarray,
    setpoints: np.ndarray,
    I_gains: np.ndarray,
    channel_order: Tuple[str, ...] = (
        'h_trace', 'h_retrace', 'phi_trace', 'phi_retrace'
    ),
) -> List[Dict]:
    """Convert a packed (n_drive, n_setpoint, n_Igain, n_channel, n_pix) array
    into a flat list of per-line dicts.

    The default channel_order matches your description "trace and retrace of
    heights + phases" interpreted as (h_trace, h_retrace, phi_trace, phi_retrace).
    Override channel_order if your packing differs.
    """
    data = np.asarray(data)
    nD, nS, nI, nC, nP = data.shape
    if nC != len(channel_order):
        raise ValueError(f"Expected {len(channel_order)} channels, got {nC}")
    if not (len(drives) == nD and len(setpoints) == nS and len(I_gains) == nI):
        raise ValueError("drives / setpoints / I_gains lengths must match data shape")

    rows: List[Dict] = []
    for i in range(nD):
        for j in range(nS):
            for k in range(nI):
                line = {ch: np.asarray(data[i, j, k, ci, :], dtype=float)
                        for ci, ch in enumerate(channel_order)}
                rows.append({
                    'drive':    float(drives[i]),
                    'setpoint': float(setpoints[j]),
                    'I_gain':   float(I_gains[k]),
                    **line,
                })
    return rows


# -----------------------------------------------------------------------------
# Cost function
# -----------------------------------------------------------------------------

def make_residual_fn(
    scanner,
    fd_lookup: Callable[[float], object],
    h_truth_hat: np.ndarray,
    experiments: List[Dict],
    weights: Optional[Dict[str, float]] = None,
    scan_speed_hat: float = 1.0,
    dx_hat: float = 1.0,
    burn_in: int = 128,
    huge_penalty: float = 1e6,
    i_gain_is_log10: bool = False,
):
    """Build a residual vector: stacked (h_trace, h_retrace, phi_trace, phi_retrace)
    differences across all experimental lines. Phase residuals are wrapped.

    Default weights make 1 deg of phase ~ 0.01 normalized length unit.

    i_gain_is_log10 : if True, exp['I_gain'] is interpreted as log10 of the
                      raw instrument gain. The fit's b_I_log10 is then a
                      pure offset in log space.
                      If False, exp['I_gain'] is linear and b_I_log10 acts as
                      a multiplicative scale factor c_I = 10**b_I_log10.
    """
    if weights is None:
        weights = {'h': 1.0, 'phi': 0.01}

    h_truth_hat = np.asarray(h_truth_hat, dtype=float)
    n_pix = h_truth_hat.size

    def residual(theta: np.ndarray) -> np.ndarray:
        a_d, b_d, a_s, b_s, b_I_log10, log_tau_A, log_tau_z = theta
        tau_A = float(np.exp(log_tau_A))
        tau_z = float(np.exp(log_tau_z))

        chunks: List[np.ndarray] = []
        for exp in experiments:
            drive_sim    = a_d * exp['drive']    + b_d
            setpoint_sim = a_s * exp['setpoint'] + b_s

            # I-gain conversion: experiment -> simulator
            I_exp = float(exp['I_gain'])
            if i_gain_is_log10:
                I_sim = 10.0 ** (b_I_log10 + I_exp)
            else:
                I_sim = (10.0 ** b_I_log10) * I_exp

            # Refuse pathological setpoints early (saves a sim call)
            if not (0.01 < setpoint_sim < 0.999) or drive_sim <= 0.0 or I_sim <= 0.0:
                chunks.append(np.full(4 * n_pix, huge_penalty))
                continue

            try:
                sim = simulate_trace_retrace(
                    scanner, fd_lookup, h_truth_hat,
                    drive_nm=float(drive_sim),
                    setpoint_frac=float(setpoint_sim),
                    I_gain=float(I_sim),
                    tau_A=tau_A, tau_z=tau_z,
                    burn_in=burn_in,
                    scan_speed_hat=scan_speed_hat,
                    dx_hat=dx_hat,
                )
            except Exception:
                # FD lookup outside library, NaN on a branch jump, etc.
                chunks.append(np.full(4 * n_pix, huge_penalty))
                continue

            r_h_t = (sim['h_trace']   - exp['h_trace'])   * weights['h']
            r_h_r = (sim['h_retrace'] - exp['h_retrace']) * weights['h']
            r_p_t = wrap_phase_diff(sim['phi_trace'],   exp['phi_trace'])   * weights['phi']
            r_p_r = wrap_phase_diff(sim['phi_retrace'], exp['phi_retrace']) * weights['phi']
            chunks.extend([r_h_t, r_h_r, r_p_t, r_p_r])

        return np.concatenate(chunks)

    return residual


# -----------------------------------------------------------------------------
# Top-level driver
# -----------------------------------------------------------------------------

def calibrate(
    scanner,
    fd_lookup: Callable[[float], object],
    h_truth_hat: np.ndarray,
    experiments: List[Dict],
    x0: Optional[np.ndarray] = None,
    bounds: Optional[Tuple[Iterable[float], Iterable[float]]] = None,
    weights: Optional[Dict[str, float]] = None,
    scan_speed_hat: float = 1.0,
    dx_hat: float = 1.0,
    burn_in: int = 128,
    verbose: int = 2,
    max_nfev: int = 200,
    i_gain_is_log10: bool = False,
) -> Dict:
    """Fit (a_drive, b_drive, a_setpoint, b_setpoint, b_I_log10, tau_A, tau_z).

    h_truth_hat : 1D ground-truth profile in the same normalized length units
                  the scanner uses internally (i.e. physical_h * conversion['A']).
    experiments : list of dicts with keys
                  drive (nm, in EXPERIMENTAL units), setpoint (fraction, in
                  EXPERIMENTAL units), I_gain (linear OR log10 — see flag),
                  h_trace, h_retrace, phi_trace, phi_retrace.
    fd_lookup   : drive_nm_simulator -> FDCurve1D (interpolate your library).
    i_gain_is_log10 : True if exp['I_gain'] is stored as log10(raw_gain).

    Returns a dict with the fitted parameters and the scipy result object.
    """
    if x0 is None:
        # Identity transforms; b_I_log10=0 (no I rescaling); log time-constants.
        x0 = np.array([1.0, 0.0, 1.0, 0.0, 0.0, np.log(1e-3), np.log(1e-4)])

    if bounds is None:
        # Generous but bounded; tighten with priors on calibration error.
        # b_I_log10 default range -3..3 = scale factor 10^-3 .. 10^3.
        lo = [0.5, -20.0, 0.7, -0.20, -3.0, np.log(1e-7), np.log(1e-7)]
        hi = [2.0,  20.0, 1.5,  0.20,  3.0, np.log(1e0),  np.log(1e0)]
        bounds = (lo, hi)

    res_fn = make_residual_fn(
        scanner, fd_lookup, h_truth_hat, experiments,
        weights=weights, scan_speed_hat=scan_speed_hat,
        dx_hat=dx_hat, burn_in=burn_in,
        i_gain_is_log10=i_gain_is_log10,
    )

    sol = least_squares(
        res_fn, x0, bounds=bounds,
        method='trf', x_scale='jac',
        verbose=verbose, max_nfev=max_nfev,
    )

    a_d, b_d, a_s, b_s, b_I_log10, log_tau_A, log_tau_z = sol.x
    return {
        'a_drive':    float(a_d),
        'b_drive':    float(b_d),
        'a_setpoint': float(a_s),
        'b_setpoint': float(b_s),
        'b_I_log10':  float(b_I_log10),
        'c_I':        float(10.0 ** b_I_log10),  # multiplicative I scale factor
        'tau_A':      float(np.exp(log_tau_A)),
        'tau_z':      float(np.exp(log_tau_z)),
        'cost':       float(sol.cost),
        'theta':      sol.x,
        'i_gain_is_log10': bool(i_gain_is_log10),
        'result':     sol,
    }


# -----------------------------------------------------------------------------
# Diagnostics
# -----------------------------------------------------------------------------

def evaluate_fit(
    fit: Dict,
    scanner,
    fd_lookup: Callable[[float], object],
    h_truth_hat: np.ndarray,
    experiments: List[Dict],
    scan_speed_hat: float = 1.0,
    dx_hat: float = 1.0,
    burn_in: int = 128,
) -> List[Dict]:
    """Re-run the simulator at the fitted theta for each experiment line.
    Useful for plotting sim-vs-exp diagnostics after calibration.
    """
    a_d, b_d = fit['a_drive'], fit['b_drive']
    a_s, b_s = fit['a_setpoint'], fit['b_setpoint']
    b_I_log10 = fit['b_I_log10']
    is_log = fit.get('i_gain_is_log10', False)
    tau_A, tau_z = fit['tau_A'], fit['tau_z']

    out = []
    for exp in experiments:
        I_exp = float(exp['I_gain'])
        I_sim = 10.0 ** (b_I_log10 + I_exp) if is_log else (10.0 ** b_I_log10) * I_exp
        sim = simulate_trace_retrace(
            scanner, fd_lookup, h_truth_hat,
            drive_nm=a_d * exp['drive'] + b_d,
            setpoint_frac=a_s * exp['setpoint'] + b_s,
            I_gain=I_sim,
            tau_A=tau_A, tau_z=tau_z,
            burn_in=burn_in,
            scan_speed_hat=scan_speed_hat, dx_hat=dx_hat,
        )
        out.append({'exp': exp, 'sim': sim})
    return out



In [ ]:
"""
Approach-curve library + cross-drive interpolation.

Strategy
--------
Every measured approach curve is single-valued in z when parameterized by
the slow z-piezo. Store curves in physical units (nm, deg) per drive A0.

To interpolate to an arbitrary drive A0*:
  1) Resample each curve onto a common normalized grid u = d / A0,
     storing the normalized amplitude a = A / A0 and the phase phi.
  2) Linearly interpolate a(u, A0) and phi(u, A0) in A0 at fixed u.
  3) Convert back: d = u * A0*,  A = a * A0*.

Optional force F(d) is carried through if provided.

Public API
----------
FDCurve1D                 : single-curve interpolator with evaluate(d) -> (A, phi, F)
ApproachCurveLibrary      : add curves, query at any drive, returns FDCurve1D
make_normalized_fd_lookup : adapter that wraps the library in the normalized-units
                            API expected by ScannerFD (matches calibrate_scanner.py)
detect_bistability        : utility to flag curves with phase jumps before fitting
"""
from __future__ import annotations

import numpy as np
from scipy.interpolate import interp1d
from typing import Optional, Tuple, Dict, List, Callable


# -----------------------------------------------------------------------------
# Single-curve evaluator
# -----------------------------------------------------------------------------

class FDCurve1D:
    """Interpolated A(d), phi(d), F(d) at a single drive amplitude.

    Conventions
    -----------
    Inputs in any consistent unit system (typically nm, degrees).
    d increases away from the surface; the largest d in the curve is treated
    as the "free" point, A(d_max) ~ A0.

    Out-of-range behavior
    ---------------------
    d > d_max : A clamped to A0 (free), phi clamped to phi(d_max), F -> 0.
    d < d_min : edge-clamped (no extrapolation into untrusted near-contact regime).
    """

    def __init__(self, d, A, phi, F=None, *, A0: Optional[float] = None,
                 kind: str = 'linear'):
        d = np.asarray(d, dtype=float)
        A = np.asarray(A, dtype=float)
        phi = np.asarray(phi, dtype=float)
        F = np.zeros_like(d) if F is None else np.asarray(F, dtype=float)

        order = np.argsort(d)
        self.d   = d[order]
        self.A   = A[order]
        self.phi = phi[order]
        self.F   = F[order]
        self.A0 = float(self.A[-1]) if A0 is None else float(A0)
        self.kind = kind

        self._A = interp1d(
            self.d, self.A, kind=kind, bounds_error=False,
            fill_value=(self.A[0], self.A0),  # left = nearest, right = free
        )
        self._phi = interp1d(
            self.d, self.phi, kind=kind, bounds_error=False,
            fill_value=(self.phi[0], self.phi[-1]),
        )
        self._F = interp1d(
            self.d, self.F, kind=kind, bounds_error=False,
            fill_value=(self.F[0], 0.0),
        )

    def evaluate(self, d):
        """Return (A, phi, F). Scalar in -> scalar out; array in -> array out."""
        d_arr = np.atleast_1d(d).astype(float)
        A   = self._A(d_arr)
        phi = self._phi(d_arr)
        F   = self._F(d_arr)
        if np.isscalar(d):
            return float(A[0]), float(phi[0]), float(F[0])
        return A, phi, F


# -----------------------------------------------------------------------------
# Library
# -----------------------------------------------------------------------------

class ApproachCurveLibrary:
    """Collection of approach curves at multiple drive amplitudes (nm).

    Each curve is stored as 1D arrays (d_nm, A_nm, phi_deg, optional F).
    The curves do NOT need to share a common d-grid or have the same length.
    """

    def __init__(self):
        self._curves: Dict[float, Dict[str, Optional[np.ndarray]]] = {}

    # ----- ingestion -----

    def add(self, drive_nm: float, d_nm, A_nm, phi_deg, F=None) -> None:
        """Add one approach curve at a known drive amplitude (in nm).

        Parameters
        ----------
        drive_nm : free-amplitude drive setting for this curve, nm.
        d_nm     : 1D array of z-piezo positions, nm. Increases away from surface.
        A_nm     : 1D array of measured oscillation amplitudes, nm.
        phi_deg  : 1D array of measured phases, degrees.
        F        : optional 1D array (e.g. peak force or virial), arbitrary units.
        """
        d = np.asarray(d_nm,    dtype=float).ravel()
        A = np.asarray(A_nm,    dtype=float).ravel()
        phi = np.asarray(phi_deg, dtype=float).ravel()
        if not (d.shape == A.shape == phi.shape):
            raise ValueError("d, A, phi must be the same length")
        F_arr = None
        if F is not None:
            F_arr = np.asarray(F, dtype=float).ravel()
            if F_arr.shape != d.shape:
                raise ValueError("F must match d length")

        # Drop NaNs and sort by d ascending
        mask = np.isfinite(d) & np.isfinite(A) & np.isfinite(phi)
        if F_arr is not None:
            mask &= np.isfinite(F_arr)
        d, A, phi = d[mask], A[mask], phi[mask]
        if F_arr is not None:
            F_arr = F_arr[mask]
        order = np.argsort(d)

        self._curves[float(drive_nm)] = {
            'd':   d[order],
            'A':   A[order],
            'phi': phi[order],
            'F':   None if F_arr is None else F_arr[order],
        }

    @classmethod
    def from_dict(cls, by_drive: Dict[float, Dict[str, np.ndarray]]) -> 'ApproachCurveLibrary':
        """Bulk-construct from {drive_nm: {'d':..., 'A':..., 'phi':..., 'F':...}}."""
        lib = cls()
        for drv, c in by_drive.items():
            lib.add(drv, c['d'], c['A'], c['phi'], F=c.get('F'))
        return lib

    # ----- inspection -----

    @property
    def drives(self) -> np.ndarray:
        return np.array(sorted(self._curves.keys()))

    def get_raw(self, drive_nm: float) -> Dict[str, Optional[np.ndarray]]:
        return self._curves[float(drive_nm)]

    # ----- normalization -----

    def normalized_grid(
        self,
        n_u: int = 400,
        u_min: Optional[float] = None,
        u_max: Optional[float] = None,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """Resample all curves onto a common u = d/A0 grid.

        Returns
        -------
        drives   : (n_drives,)        sorted drive amplitudes [nm]
        u_grid   : (n_u,)             common normalized distance grid
        a_grid   : (n_drives, n_u)    normalized amplitude  A/A0
        phi_grid : (n_drives, n_u)    phase [deg]
        F_grid   : (n_drives, n_u)    force (zero where F not supplied)
        """
        drives = self.drives
        if drives.size < 2:
            raise ValueError("Library needs at least two drives to interpolate.")
        umins = [self._curves[d]['d'].min() / d for d in drives]
        umaxs = [self._curves[d]['d'].max() / d for d in drives]

        u_min = max(umins) if u_min is None else float(u_min)
        u_max = min(umaxs) if u_max is None else float(u_max)
        if not (u_max > u_min):
            raise ValueError(
                f"No common u-range across curves "
                f"(largest u_min={max(umins):.3g}, smallest u_max={min(umaxs):.3g}). "
                "Trim outlier curves or pass u_min/u_max manually."
            )
        u_grid = np.linspace(u_min, u_max, int(n_u))

        nD = drives.size
        a_grid   = np.empty((nD, n_u))
        phi_grid = np.empty((nD, n_u))
        F_grid   = np.zeros((nD, n_u))

        for i, drv in enumerate(drives):
            c = self._curves[drv]
            u = c['d'] / drv
            a = c['A'] / drv
            a_grid[i]   = np.interp(u_grid, u, a)
            phi_grid[i] = np.interp(u_grid, u, c['phi'])
            if c['F'] is not None:
                F_grid[i] = np.interp(u_grid, u, c['F'])

        return drives, u_grid, a_grid, phi_grid, F_grid

    # ----- interpolation across drives -----

    def _raw_to_fd(self, drv: float) -> FDCurve1D:
        """Wrap a stored curve in an FDCurve1D verbatim (no resampling)."""
        c = self._curves[float(drv)]
        return FDCurve1D(c['d'], c['A'], c['phi'],
                         F=c['F'], A0=float(drv))

    def fd_at_drive(self, drive_target_nm: float, n_u: int = 400) -> FDCurve1D:
        """Build an FDCurve1D for an arbitrary target drive amplitude (nm).

        Strategy:
          - Outside [drives.min, drives.max], clamp to the nearest library
            curve (returned verbatim — full d-range preserved).
          - At an exact library drive, return that curve verbatim.
          - Otherwise, linearly interpolate between the two adjacent library
            curves on the OVERLAP of their physical d-ranges. Each curve is
            evaluated on its own d-axis (no global u-grid intersection),
            so a single small-range curve cannot truncate the rest.

        Note: n_u kept for backward compatibility — used as the d-grid size
        for the local pair LERP.
        """
        drives = self.drives
        drv = float(drive_target_nm)

        # Clamp to ends
        if drv <= drives[0]:
            return self._raw_to_fd(drives[0])
        if drv >= drives[-1]:
            return self._raw_to_fd(drives[-1])

        # Locate adjacent library drives
        idx = int(np.searchsorted(drives, drv) - 1)
        drv_lo, drv_hi = float(drives[idx]), float(drives[idx + 1])

        # Exact-match short-circuit (avoids interpolation noise)
        if np.isclose(drv, drv_lo):
            return self._raw_to_fd(drv_lo)
        if np.isclose(drv, drv_hi):
            return self._raw_to_fd(drv_hi)

        c_lo = self._curves[drv_lo]
        c_hi = self._curves[drv_hi]
        w = (drv - drv_lo) / (drv_hi - drv_lo)

        # Common d-grid on the overlap of the two curves' d-supports
        d_min = float(max(c_lo['d'].min(), c_hi['d'].min()))
        d_max = float(min(c_lo['d'].max(), c_hi['d'].max()))
        if not (d_max > d_min):
            raise ValueError(
                f"Curves at drive={drv_lo} and drive={drv_hi} have no "
                f"overlapping d-range; cannot interpolate."
            )
        d_grid = np.linspace(d_min, d_max, int(n_u))

        A_lo = np.interp(d_grid, c_lo['d'], c_lo['A'])
        A_hi = np.interp(d_grid, c_hi['d'], c_hi['A'])
        phi_lo = np.interp(d_grid, c_lo['d'], c_lo['phi'])
        phi_hi = np.interp(d_grid, c_hi['d'], c_hi['phi'])
        A = (1 - w) * A_lo + w * A_hi
        phi = (1 - w) * phi_lo + w * phi_hi

        F = None
        if (c_lo['F'] is not None) and (c_hi['F'] is not None):
            F_lo = np.interp(d_grid, c_lo['d'], c_lo['F'])
            F_hi = np.interp(d_grid, c_hi['d'], c_hi['F'])
            F = (1 - w) * F_lo + w * F_hi

        return FDCurve1D(d_grid, A, phi, F=F, A0=drv)

    # convenience alias
    def __call__(self, drive_target_nm: float) -> FDCurve1D:
        return self.fd_at_drive(drive_target_nm)


# -----------------------------------------------------------------------------
# Adapter to the normalized-units API expected by ScannerFD
# -----------------------------------------------------------------------------

def make_normalized_fd_lookup(
    library: ApproachCurveLibrary,
    conv_L: float,
    conv_A: Optional[float] = None,
    conv_F: float = 1.0,
    n_u: int = 400,
) -> Callable[[float], FDCurve1D]:
    """Wrap a physical-units library in a callable returning FDCurve1D in
    the *normalized* units the ScannerFD expects internally.

    Parameters
    ----------
    library : ApproachCurveLibrary in physical units (nm, deg).
    conv_L  : length conversion (d_norm = d_nm * conv_L). In ScannerFD all
              length-like params share the same conversion factor, so pass
              scanner.conversion['A'] (or equivalently scanner.conversion['d0']).
    conv_A  : amplitude conversion (A_norm = A_nm * conv_A). Defaults to conv_L,
              which is the right thing since amplitude IS a length.
    conv_F  : force conversion. Pass 1.0 if your library doesn't carry F, or
              the relevant scanner.conversion key if it does.
    n_u     : grid size for the normalized resampling step.

    The returned callable takes a drive in *physical* nm and returns an
    FDCurve1D whose evaluate(d_hat) accepts and returns *normalized* values.
    Use this directly as `fd_lookup` in calibrate_scanner.calibrate(...).
    """
    if conv_A is None:
        conv_A = conv_L

    def lookup(drive_nm: float) -> FDCurve1D:
        fd_phys = library.fd_at_drive(float(drive_nm), n_u=n_u)
        return FDCurve1D(
            d=fd_phys.d * conv_L,
            A=fd_phys.A * conv_A,
            phi=fd_phys.phi,
            F=fd_phys.F * conv_F,
            A0=float(drive_nm) * conv_A,
        )
    return lookup


# -----------------------------------------------------------------------------
# Diagnostics
# -----------------------------------------------------------------------------

def detect_bistability(
    library: ApproachCurveLibrary,
    phase_jump_threshold_deg: float = 25.0,
    amp_jump_threshold_rel: float = 0.15,
) -> Dict[float, Dict[str, float]]:
    """Flag curves where phi or A/A0 jumps more than thresholds between
    adjacent samples in d. These are likely to be bistable and should be
    inspected (or split into branches) before being put into the library.

    Returns {drive: {'max_dphi_deg': ..., 'max_da_rel': ..., 'd_at_jump': ...}}
    only for curves that exceed at least one threshold.
    """
    flags: Dict[float, Dict[str, float]] = {}
    for drv in library.drives:
        c = library.get_raw(drv)
        d, A, phi = c['d'], c['A'], c['phi']
        if d.size < 3:
            continue
        dphi = np.abs(np.diff(phi))
        da_rel = np.abs(np.diff(A)) / float(drv)
        i_phi = int(np.argmax(dphi)) if dphi.size else 0
        i_a   = int(np.argmax(da_rel)) if da_rel.size else 0
        if dphi[i_phi] > phase_jump_threshold_deg or da_rel[i_a] > amp_jump_threshold_rel:
            phi_severity = dphi[i_phi] / phase_jump_threshold_deg
            a_severity   = da_rel[i_a] / amp_jump_threshold_rel
            i_jump = i_phi if phi_severity > a_severity else i_a
            flags[float(drv)] = {
                'max_dphi_deg': float(dphi.max()),
                'max_da_rel':   float(da_rel.max()),
                'd_at_jump':    float(d[i_jump]),
            }
    return flags



In [ ]:
tp = fwds[0]

L = 50

# plt.figure()
# for i in range(len(fwds)):
#     tp = fwds[i]
#     plt.plot((tp[0] - np.min(tp[0][-L:]))[-L:]*1e9, tp[1][-L:])
# plt.xlabel('Height (nm)')
# plt.ylabel('Amplitude (nm)')

# plt.figure()
# for i in range(len(fwds)):
#     tp = fwds[i]
#     plt.plot((tp[0] - np.min(tp[0][-L:]))[-L:]*1e9, tp[2][-L:])
# plt.xlabel('Height (nm)')
# plt.ylabel('Phase (degrees)')

height = np.zeros([len(fwds), L])
amp = np.zeros([len(fwds), L])
phase = np.zeros([len(fwds), L])

for i in range(len(fwds)):
    tp = fwds[i]
    height[i] = (tp[0] - np.min(tp[0][-L:]))[-L:]*1e9
    amp[i] = tp[1][-L:][::-1]
    phase[i] = tp[2][-L:][::-1]

plt.figure()
for i in range(len(fwds)):
    plt.plot(height[i], amp[i])
plt.xlabel('Height (nm)')
plt.ylabel('Amplitude (nm)')

plt.figure()
for i in range(len(fwds)):
    plt.plot(height[i], phase[i])
plt.xlabel('Height (nm)')
plt.ylabel('Phase (degrees)')

In [ ]:
drives_si = np.array(drives)*factor1
measured = {}


# Assume these are lists/arrays where index i corresponds to drives[i]
# z_arrays[i], A_arrays[i], phi_arrays[i]
# each could be a 1D FD curve, a scan line, or an image
measured = {
    float(drive): {
        "d": height[i],
        "A": amp[i],
        "phi": phase[i],
    }
    for i, drive in enumerate(drives_si)
}

In [ ]:
# Define the groud-truth line profile
indices = np.array([0, 20, 44, 83, 108, 147, 172, 211, 235, 256]) * 2
h1 = np.zeros(512)
for i in range(5):
    h1[indices[2*i]:indices[2*i+1]] = 1
    # h1[(i*50):(i*50)+25] = 1
h_hat = h1 * 100 + 142 / 512 * (512-np.arange(512))

h_truth_hat = h_hat[::2]

fig, ax = plt.subplots(2,1,figsize=[4, 6])
tp  = traces[30, 4, 0]
ax[0].plot(h_truth_hat)
ax[1].plot((tp-np.min(tp))*1e9)

In [ ]:
param_clean.keys()

In [ ]:
param_clean['x_measured']

In [ ]:
traces.shape

In [ ]:
# exps = []
# for record in raw_records:                  # whatever your loader yields
#     exps.append({
#         'drive':       float(param_clean['x_measured'][:,0]),
#         'setpoint':    float(param_clean['x_measured'][:,1] / param_clean['x_measured'][:,0]) / 100.0,
#         'I_gain':      float(param_clean['x_measured'][:,2]),
#         'h_trace':     np.asarray(traces[:,4,0],  float) * conv_L,
#         'h_retrace':   np.asarray(traces[:,4,1], float) * conv_L,
#         'phi_trace':   np.asarray(traces[:,4,4],  float),
#         'phi_retrace': np.asarray(traces[:,4,5], float),
#     })

In [ ]:
traces.shape

In [ ]:
idx = 0
plt.plot(traces[i, 4, 0])

In [ ]:
exps = []
n_records = param_clean['x_measured'].shape[0]   # 60

for i in range(n_records):
    drive_nm = float(param_clean['x_measured'][i, 0])
    A_sp_nm  = float(param_clean['x_measured'][i, 1])
    log10_I  = float(param_clean['x_measured'][i, 2])
    tp1 = traces[i, 4, 0, :]
    tp2 = traces[i, 4, 1, :]

    exps.append({
        'drive':       drive_nm,
        'setpoint':    A_sp_nm / drive_nm,                       # fraction in (0, 1)
        'I_gain':      log10_I,                                  # log10; pass flag below
        'h_trace':     np.asarray((tp1-np.min(tp1))*1e9, float) * 1,
        'h_retrace':   np.asarray((tp2-np.min(tp2))*1e9, float) * 1,
        'phi_trace':   np.asarray(traces[i, 4, 4, :], float),
        'phi_retrace': np.asarray(traces[i, 4, 5, :], float),
    })



In [ ]:
fit = calibrate(
    scanner, fd_lookup, h_truth_hat-np.min(h_truth_hat), exps,
    i_gain_is_log10=True,
    scan_speed_hat=512, dx_hat=1,         # dt = 1.953e-3 s, matches 1 Hz @ 256 px
    fix={
        'a_drive':   1.0,                  # no drive scaling
        'b_drive':   0.0,
        'tau_A':     5e-3,                 # > dt -> stable LPF
        'tau_z':     2e-3,                 # > dt -> stable LPF
    },
    # only a_setpoint, b_setpoint, b_I_log10 are now free
    bounds={
        'a_setpoint': (0.2, 1.75),
        'b_setpoint': (-0.10, 0.10),
        'b_I_log10':  (-2, 2),
    },
)
print('free:',  fit['free'])               # ['a_setpoint', 'b_setpoint', 'b_I_log10']
print('fixed:', fit['fixed'])              # ['a_drive', 'b_drive', 'tau_A', 'tau_z']
print('a_set:', fit['a_setpoint'], 'b_set:', fit['b_setpoint'], 'c_I:', fit['c_I'])

In [ ]:
fit = calibrate(
    scanner, fd_lookup, h_truth_hat-np.min(h_truth_hat), exps,
    i_gain_is_log10=True,
    scan_speed_hat=512, dx_hat=1,         # dt = 1.953e-3 s, matches 1 Hz @ 256 px
    fix={
        'a_drive':   1.0,                  # no drive scaling
        'b_drive':   0.0,
        'tau_A':     5e-3,                 # > dt -> stable LPF
        'tau_z':     2e-3,                 # > dt -> stable LPF
    },
    # only a_setpoint, b_setpoint, b_I_log10 are now free
    bounds={
        'a_setpoint': (0.8, 1.25),
        'b_setpoint': (-0.10, 0.10),
        'b_I_log10':  (-2, 2),
    },
)
print('free:',  fit['free'])               # ['a_setpoint', 'b_setpoint', 'b_I_log10']
print('fixed:', fit['fixed'])              # ['a_drive', 'b_drive', 'tau_A', 'tau_z']
print('a_set:', fit['a_setpoint'], 'b_set:', fit['b_setpoint'], 'c_I:', fit['c_I'])

In [ ]:
# from calibrate_scanner import simulate_trace_retrace
ex = exps[10]
sim = simulate_trace_retrace(
    scanner, fd_lookup, h_truth_hat-np.min(h_truth_hat),
    drive_nm=ex['drive'], setpoint_frac=ex['setpoint'],
    I_gain=10**ex['I_gain'],
    tau_A=5e-3, tau_z=2e-3,
    scan_speed_hat=512, dx_hat=1,
)
for k, v in sim.items():
    print(k, np.isfinite(v).all(), float(np.nanmin(v)), float(np.nanmax(v)))

In [ ]:
plt.plot(h_truth_hat-np.min(h_truth_hat))
plt.plot(exps[10]['h_retrace'])

In [ ]:
plt.plot(exps[10]['phi_retrace'])

In [ ]:
drives_si

In [ ]:
idx = 8

In [ ]:
import numpy as np

idx = -4

drive_nm = drives_si[idx]
d_nm     = height[idx]    # ascending, tip–sample distance
A_nm     = amp[idx]    # rising toward drive_nm at large d
phi_deg  = phase[idx]

fd = FDCurve1D(d_nm, A_nm, phi_deg, A0=drive_nm)

# Eyeball check before plugging into the loop
d_grid = np.linspace(d_nm.min(), d_nm.max(), 200)
A_grid = np.array([fd.evaluate(d)[0] for d in d_grid])
plt.plot(d_grid, A_grid); plt.axhline(drive_nm, ls='--')

# Build a scanner that always uses this one curve
params = {'A': drive_nm, 'd0': d_nm.max()*0.8, 'R': 10, 'H': 1e-19,
          'E_star': 1e9, 'Q': 200}
scanner = ScannerFD(params=params,
                    conversions={k: 1.0 for k in params},
                    fd_model=fd)

# Trivial fd_lookup: ignore the drive argument, always return `fd`
fd_lookup = lambda _drive: fd

In [ ]:
phi_grid = np.array([fd.evaluate(d)[1] for d in d_grid])

plt.plot(d_grid, phi_grid); plt.axhline(drive_nm, ls='--')
plt.plot(d_grid, A_grid); plt.axhline(drive_nm, ls='--')


In [ ]:
out.keys()

In [ ]:
# A_filter_order, wn_A, zeta_A             # 2nd-order amp LPF  (default: 1st)
# phi_filter_order, wn_phi, zeta_phi       # 2nd-order phase LPF (independent of amp)
# z_filter_order, wn_z, zeta_z             # 2nd-order Z actuator (default: 1st)
# z_rate_limit_hat                         # slew limit on Z per step
# T_I                                      # integrator leak (anti-windup)
# integ_clip                               # hard cap on integrator
# d_cmd_init_hat, d_act_init_hat,          # initial controller state
# A_meas_init_hat, phi_meas_init_deg,
# integ_init

In [ ]:
out = scanner.scan_line_trace_retrace_pi(
    h_hat=h_truth_hat-np.min(h_truth_hat),
    setpoint=0.5,
    scan_speed_hat=1,
    dx_hat=1.0,
    P=3.2068,
    I=0.8092,
    A_filter_order=None,
    tau_A=1e-4,
    z_filter_order=None,
    tau_z=1e-5,
)

plot_trace_retrace_channels(
    out,
    h_hat=h_truth_hat,
    center_height=True,
)

In [ ]:
N1 = 101
N2 = 101

p_test = np.linspace(1e-4, 1e1, num=N1)
i_test = np.linspace(1e-4, 1e1, num=N2)

p_test

In [ ]:
N1 = 101
N2 = 101

p_test = np.linspace(1e-2, 1e1, num=N1)
i_test = np.linspace(1e-2, 1e1, num=N2)

rew1_out = np.zeros([N1, N2])
rew2_out = np.zeros([N1, N2])


for i in range(N1):
    for j in range(N2):
        print('Working on {}/{}'.format(i*N2 + j, N1 * N2), end='\r')
        out = scanner.scan_line_trace_retrace_pi(
            h_hat=h_truth_hat-np.min(h_truth_hat),
            setpoint=0.5,
            scan_speed_hat=1,
            dx_hat=1.0,
            P=p_test[i],
            I=i_test[j],
            A_filter_order=None,
            tau_A=1e-4,
            z_filter_order=None,
            tau_z=1e-5,
        )
        rew1_out[i,j] = np.mean(np.abs((out['trace']['d_hat']+out['retrace']['d_hat'])/2-(h_truth_hat-np.min(h_truth_hat))))
        rew2_out[i,j] = np.mean(np.abs((out['trace']['d_hat']+out['retrace']['d_hat'])/2-(h_truth_hat-np.min(h_truth_hat))) + 
                                np.abs(out['trace']['phi_deg']-out['retrace']['phi_deg']))
        

In [ ]:
np.argmin(rew1_out)

In [ ]:
print(i_test[3240//N1], p_test[3240%N2])

In [ ]:
plt.imshow(np.log(rew2_out), origin='lower')
plt.colorbar()
plt.xlabel('P')
plt.ylabel('I')
# plt.xscale('log')
# plt.yscale('log')

In [ ]:
out.keys()

In [ ]:
out['trace'].keys()

In [ ]:
plt.plot(out['trace']['d_hat'])

In [ ]:
dt = 1.953e-3
out = scanner.scan_line_trace_retrace_pi(
    h_hat=h_truth_hat - np.min(h_truth_hat),
    setpoint=0.8,
    scan_speed_hat=1.0/dt, dx_hat=1.0,

    P=0.0,                              # match real instrument
    I=100.0,                            # use the value from your controller

    # Lock-in: 2nd-order, BW ≈ 200 Hz is typical for tapping AFM
    A_filter_order=2,
    wn_A=2*np.pi*200,  zeta_A=0.7,
    phi_filter_order=2,
    wn_phi=2*np.pi*200, zeta_phi=0.7,

    # Z piezo: 2nd-order, ~1 kHz, slightly over-damped
    z_filter_order=2,
    wn_z=2*np.pi*1000, zeta_z=0.9,
)

plot_trace_retrace_channels(
    out,
    h_hat=h_truth_hat,
    center_height=True,
)

In [ ]:
dt = 1.953e-3

out = scanner.scan_line_pi(
    h_hat=h_truth_hat - np.min(h_truth_hat),
    setpoint=0.6,
    scan_speed_hat=1.0/dt, dx_hat=1.0,

    # PI: keep your aggressive P, raise I so the integrator does its share
    P=5.0,
    I=50.0,

    # 2nd-order amplitude LPF — the lock-in's true response
    A_filter_order=2,
    wn_A=1.0/(2*dt), zeta_A=0.7,

    # 2nd-order phase LPF — independently smooth the phase READING
    phi_filter_order=2,
    wn_phi=1.0/(5*dt), zeta_phi=0.7,

    # 2nd-order Z, slightly over-damped so no ringing on a step
    z_filter_order=2,
    wn_z=1.0/(3*dt), zeta_z=0.9,

    # Physical slew limit: a few nm per pixel.  Cuts pixel-to-pixel chatter
    # without affecting legitimate edge response (~10–20 nm/px).
    z_rate_limit_hat=5.0,

    # Integrator leak: prevents windup in the burn-in transient
    T_I=0.5,
)

In [ ]:
fig, ax = plt.subplots(1,2,figsize=[8,2.5])
ax[0].plot(exps[10]['h_trace'])
ax[0].plot(exps[10]['h_retrace'])
ax[1].plot(exps[10]['phi_trace'])
ax[1].plot(exps[10]['phi_retrace'])
ax[0].set_title('Experimental traces')
ax[0].set_ylabel('Height (nm)')
ax[1].set_ylabel('Phase (degree')

plt.tight_layout()

In [ ]:
exps[10]

In [ ]:
# -----------------------------------------------------------------------------
# Example
# -----------------------------------------------------------------------------
if __name__ == '__main__':

    lib = ApproachCurveLibrary.from_dict(measured)
    
    # quick sanity check
    bist = detect_bistability(lib)
    for drv, info in bist.items():
        print(f"Drive {drv} nm: jump near d={info['d_at_jump']:.2f} nm, "
              f"max d-phi={info['max_dphi_deg']:.1f} deg")
    
    # query at any drive
    fd_phys = lib(40.0)               # FDCurve1D in nm/deg
    A, phi, F = fd_phys.evaluate(2.5) # at d = 2.5 nm
    
    # plug into calibrate_scanner
    # from calibrate_scanner import calibrate
    fd_lookup = make_normalized_fd_lookup(
        lib,
        conv_L=scanner.conversion['A'],
    )
    # ---------- 5. Calibrate ----------
    import numpy as np
    dt = 1.953e-3

    # Option B — physical units in nm, nm/s  (more readable, same dt)
    dx_phys        = 10_000.0 / 256     # 39.06 nm/pixel
    v_phys         = 10_000.0 / 0.5     # 20 000 nm/s
    dx_hat         = dx_phys
    scan_speed_hat = v_phys

    # Identity transforms; tau_A ~ 5*dt, tau_z ~ 2*dt.
    x0 = np.array([1.0, 0.0, 1.0, 0.0, 0.0, np.log(5*dt), np.log(2*dt)])
    lo = [0.5, -20.0, 0.7, -0.20, -3.0, np.log(dt),    np.log(dt)]
    hi = [2.0,  20.0, 1.5,  0.20,  3.0, np.log(1e0),   np.log(1e0)]

    fit = calibrate(
        scanner, fd_lookup, h_truth_hat, exps,
        i_gain_is_log10=True,        # << because log10_I_gains is in log10
        scan_speed_hat=scan_speed_hat,          # your experiment's scan speed in normalized units
        dx_hat=dx_phys,                  # pixel pitch in normalized units
        x0=x0, bounds=(lo, hi),
    )

In [ ]:
import numpy as np

idx = -4

d_nm    = height[idx]       # tip-sample distance, ascending away from surface
A_nm    = amp[idx]
phi_deg = phase[idx]

# Use measured free amplitude, not necessarily the instrument drive setting.
# Use the far-away portion of the approach curve.
A0_nm = np.nanmedian(A_nm[-20:])

fd = FDCurve1D(d_nm, A_nm, phi_deg, A0=A0_nm)

# If your setpoint is already a fraction, keep it.
# If your experimental setpoint is an amplitude in nm, use:
# setpoint_frac = A_set_nm / A0_nm
setpoint_frac = 0.6

def find_equilibrium_d(fd, setpoint_frac):
    """Find d where A(d) = setpoint_frac * A0."""
    A_set = setpoint_frac * fd.A0

    d = np.asarray(fd.d, float)
    A = np.asarray(fd.A, float)

    m = np.isfinite(d) & np.isfinite(A)
    d, A = d[m], A[m]

    # Interpolate using A as the independent variable.
    # This assumes the selected branch is mostly monotonic.
    order = np.argsort(A)
    return float(np.interp(A_set, A[order], d[order]))

d_eq = find_equilibrium_d(fd, setpoint_frac)

print("A0_nm =", A0_nm)
print("setpoint amplitude =", setpoint_frac * A0_nm)
print("equilibrium d_eq =", d_eq)
print("FD d-range =", fd.d.min(), fd.d.max())

In [ ]:
test.header['ScanSize']

In [ ]:
# -----------------------------------------------------------------------------
# Example usage
# -----------------------------------------------------------------------------
if __name__ == '__main__':
    # 1) Build scanner with nominal (uncalibrated) physical params
    # scanner = ScannerFD(params=..., conversions=..., fd_model=<placeholder>)
    #
    # 2) Build an FD lookup over your library, e.g. nearest-drive selection
    #    or a SciPy interpolator that returns an FDCurve1D-like object.
    # def fd_lookup(drive_nm):
    #     return interpolate_fd_library(drive_nm)
    #
    # 3) Pack your data and metadata
    # data shape: (n_drive, n_setpoint, n_Igain, 4, n_pix)
    # channels in order: h_trace, h_retrace, phi_trace, phi_retrace
    # exps = flatten_experiments(data, drives_nm, setpoints_frac, I_gains)
    #
    # 4) Run calibration
    # fit = calibrate(scanner, fd_lookup, h_truth_hat, exps,
    #                 scan_speed_hat=..., dx_hat=...)
    # print(fit)
    #
    # 5) Inspect residuals on a held-out condition (different drive or setpoint
    #    than any in `exps`) before trusting the fit.
    pass


## Simulate drive-setpoint parameter map

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field, asdict
from typing import Any, Dict, Iterable, Optional, Sequence, Tuple

import numpy as np
from numba import njit
from scipy.optimize import differential_evolution, minimize
import matplotlib.pyplot as plt


# ============================================================
# 1) CLEAN RK45 APPROACH SOLVER (NO SOLVER-SIDE NOISE)
# ============================================================
@dataclass
class NormalizedSPMParams:
    """Dimensionless / normalized SPM parameters."""
    Q: float
    C1: float
    C2: float
    C3: float
    a0: float
    omega_drive_hat: float = 1.0
    omega_ref_hat: float = 1.0
    attrs: Dict[str, Any] = field(default_factory=dict)


@dataclass
class RK45Options:
    """Controls for the RK45 steady-state solver."""
    N_cycles: int = 20
    max_windows: int = 40
    settle_windows: int = 1
    tol_A: float = 1e-4
    tol_phi_deg: float = 1e-2
    consecutive: int = 2
    demod_per_ref_cycle: int = 40
    rtol: float = 1e-6
    atol: float = 1e-9
    dt0: float = 2e-2
    dt_min: float = 1e-10
    dt_max: float = 3e-1
    max_steps_per_window: int = 200000


@njit(cache=True)
def F_DMT_hat(delta_hat, C1, C2, a0):
    if delta_hat > 0.0:
        return -C1 / (delta_hat + a0) ** 2
    else:
        return -C1 / a0 ** 2 + C2 * (a0 - delta_hat) ** 1.5


@njit(cache=True)
def rhs_hat(t_hat, z_hat, v_hat, d_hat, Q, C1, C2, C3, a0, omega_drive_hat):
    delta_hat = z_hat + d_hat
    F_contact = F_DMT_hat(delta_hat, C1, C2, a0)
    F_drive = -C3 * np.cos(omega_drive_hat * t_hat)
    dz = v_hat
    dv = F_contact + F_drive - v_hat / Q - z_hat
    return dz, dv


@njit(cache=True)
def rk45_step_dopri(t, z, v, dt, d_hat, Q, C1, C2, C3, a0, omega_drive_hat, rtol, atol):
    c2 = 1.0 / 5.0
    c3 = 3.0 / 10.0
    c4 = 4.0 / 5.0
    c5 = 8.0 / 9.0
    c6 = 1.0

    a21 = 1.0 / 5.0
    a31 = 3.0 / 40.0
    a32 = 9.0 / 40.0
    a41 = 44.0 / 45.0
    a42 = -56.0 / 15.0
    a43 = 32.0 / 9.0
    a51 = 19372.0 / 6561.0
    a52 = -25360.0 / 2187.0
    a53 = 64448.0 / 6561.0
    a54 = -212.0 / 729.0
    a61 = 9017.0 / 3168.0
    a62 = -355.0 / 33.0
    a63 = 46732.0 / 5247.0
    a64 = 49.0 / 176.0
    a65 = -5103.0 / 18656.0

    b1 = 35.0 / 384.0
    b3 = 500.0 / 1113.0
    b4 = 125.0 / 192.0
    b5 = -2187.0 / 6784.0
    b6 = 11.0 / 84.0

    b1s = 5179.0 / 57600.0
    b3s = 7571.0 / 16695.0
    b4s = 393.0 / 640.0
    b5s = -92097.0 / 339200.0
    b6s = 187.0 / 2100.0
    b7s = 1.0 / 40.0

    dz1, dv1 = rhs_hat(t, z, v, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)
    z2 = z + dt * a21 * dz1
    v2 = v + dt * a21 * dv1
    dz2, dv2 = rhs_hat(t + c2 * dt, z2, v2, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

    z3 = z + dt * (a31 * dz1 + a32 * dz2)
    v3 = v + dt * (a31 * dv1 + a32 * dv2)
    dz3, dv3 = rhs_hat(t + c3 * dt, z3, v3, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

    z4 = z + dt * (a41 * dz1 + a42 * dz2 + a43 * dz3)
    v4 = v + dt * (a41 * dv1 + a42 * dv2 + a43 * dv3)
    dz4, dv4 = rhs_hat(t + c4 * dt, z4, v4, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

    z5i = z + dt * (a51 * dz1 + a52 * dz2 + a53 * dz3 + a54 * dz4)
    v5i = v + dt * (a51 * dv1 + a52 * dv2 + a53 * dv3 + a54 * dv4)
    dz5, dv5 = rhs_hat(t + c5 * dt, z5i, v5i, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

    z6 = z + dt * (a61 * dz1 + a62 * dz2 + a63 * dz3 + a64 * dz4 + a65 * dz5)
    v6 = v + dt * (a61 * dv1 + a62 * dv2 + a63 * dv3 + a64 * dv4 + a65 * dv5)
    dz6, dv6 = rhs_hat(t + c6 * dt, z6, v6, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

    z5 = z + dt * (b1 * dz1 + b3 * dz3 + b4 * dz4 + b5 * dz5 + b6 * dz6)
    v5 = v + dt * (b1 * dv1 + b3 * dv3 + b4 * dv4 + b5 * dv5 + b6 * dv6)

    dz7, dv7 = rhs_hat(t + dt, z5, v5, d_hat, Q, C1, C2, C3, a0, omega_drive_hat)

    z4s = z + dt * (b1s * dz1 + b3s * dz3 + b4s * dz4 + b5s * dz5 + b6s * dz6 + b7s * dz7)
    v4s = v + dt * (b1s * dv1 + b3s * dv3 + b4s * dv4 + b5s * dv5 + b6s * dv6 + b7s * dv7)

    ez = z5 - z4s
    ev = v5 - v4s
    sz = atol + rtol * max(abs(z), abs(z5))
    sv = atol + rtol * max(abs(v), abs(v5))
    err = np.sqrt(0.5 * ((ez / sz) ** 2 + (ev / sv) ** 2))
    return z5, v5, err


@njit(cache=True)
def demod_add_segment(X, Y, z0, z1, t0, t1, omega_ref_hat):
    c0 = np.cos(omega_ref_hat * t0)
    s0 = np.sin(omega_ref_hat * t0)
    c1 = np.cos(omega_ref_hat * t1)
    s1 = np.sin(omega_ref_hat * t1)
    dt = t1 - t0
    X += 0.5 * (z0 * c0 + z1 * c1) * dt
    Y += 0.5 * (z0 * s0 + z1 * s1) * dt
    return X, Y


@njit(cache=True)
def run_one_window_rk45(
    t_start, z_start, v_start, d_hat, Q, C1, C2, C3, a0,
    omega_drive_hat, omega_ref_hat,
    T_win, demod_dt_hat, rtol, atol, dt0, dt_min, dt_max, max_steps,
):
    t = t_start
    z = z_start
    v = v_start
    t_end = t_start + T_win

    X = 0.0
    Y = 0.0
    if demod_dt_hat <= 0.0:
        demod_dt_hat = T_win

    t_s = t_start
    z_s = z_start
    next_s = t_s + demod_dt_hat

    F0 = F_DMT_hat(z + d_hat, C1, C2, a0)
    F_int = 0.0
    F_min = F0
    F_max = F0

    safety = 0.9
    fac_min = 0.2
    fac_max = 5.0
    p = 5.0
    dt = dt0
    steps = 0

    while t < t_end and steps < max_steps:
        if dt < dt_min:
            dt = dt_min
        if dt > dt_max:
            dt = dt_max
        if t + dt > t_end:
            dt = t_end - t

        t_prev = t
        z_prev = z
        F_prev = F_DMT_hat(z_prev + d_hat, C1, C2, a0)

        z_new, v_new, err = rk45_step_dopri(
            t, z, v, dt, d_hat, Q, C1, C2, C3, a0, omega_drive_hat, rtol, atol
        )
        accept = (err <= 1.0) or (dt <= dt_min * 1.0001)
        if accept:
            t = t + dt
            z = z_new
            v = v_new

            F_cur = F_DMT_hat(z + d_hat, C1, C2, a0)
            F_int += 0.5 * (F_prev + F_cur) * (t - t_prev)
            if F_cur < F_min:
                F_min = F_cur
            if F_cur > F_max:
                F_max = F_cur

            while next_s <= t:
                denom = t - t_prev
                if denom <= 0.0:
                    z_b = z
                else:
                    frac = (next_s - t_prev) / denom
                    if frac < 0.0:
                        frac = 0.0
                    elif frac > 1.0:
                        frac = 1.0
                    z_b = z_prev + frac * (z - z_prev)
                X, Y = demod_add_segment(X, Y, z_s, z_b, t_s, next_s, omega_ref_hat)
                t_s = next_s
                z_s = z_b
                next_s = next_s + demod_dt_hat
            steps += 1

        if err == 0.0:
            fac = fac_max
        else:
            fac = safety * (1.0 / err) ** (1.0 / p)
            if fac < fac_min:
                fac = fac_min
            elif fac > fac_max:
                fac = fac_max
        dt = dt * fac

    if t_end > t_s:
        X, Y = demod_add_segment(X, Y, z_s, z, t_s, t_end, omega_ref_hat)

    F_mean = F_int / T_win
    return X, Y, F_mean, F_min, F_max, steps, z, v


@njit(cache=True)
def simulate_steady_state_rk45(
    z0_hat, v0_hat, d_hat, Q, C1, C2, C3, a0, omega_drive_hat, omega_ref_hat,
    N_cycles, max_windows, settle_windows, tol_A, tol_phi_deg, consecutive,
    demod_per_ref_cycle, rtol, atol, dt0, dt_min, dt_max, max_steps_per_window,
):
    T_ref = 2.0 * np.pi / omega_ref_hat
    T_win = N_cycles * T_ref
    if demod_per_ref_cycle < 5:
        demod_per_ref_cycle = 5
    demod_dt_hat = T_ref / demod_per_ref_cycle

    t = 0.0
    z = z0_hat
    v = v0_hat
    last_A = 1e30
    last_phi = 1e30
    stable = 0
    A_last = 0.0
    phi_last = 0.0
    F_mean_last = 0.0
    F_min_last = 0.0
    F_max_last = 0.0

    for widx in range(max_windows):
        X, Y, F_mean, F_min, F_max, _, z, v = run_one_window_rk45(
            t, z, v, d_hat, Q, C1, C2, C3, a0,
            omega_drive_hat, omega_ref_hat,
            T_win, demod_dt_hat, rtol, atol, dt0, dt_min, dt_max, max_steps_per_window,
        )
        t = t + T_win
        A = (2.0 / T_win) * np.sqrt(X * X + Y * Y)
        phi = np.degrees(np.arctan2(Y, X))
        if phi < 0.0:
            phi += 180.0
        if phi > 180.0:
            phi -= 180.0

        A_last = A
        phi_last = phi
        F_mean_last = F_mean
        F_min_last = F_min
        F_max_last = F_max

        if widx >= settle_windows:
            dA = abs(A - last_A)
            dphi = abs(phi - last_phi)
            if dA < tol_A and dphi < tol_phi_deg:
                stable += 1
            else:
                stable = 0
            if stable >= consecutive:
                return A, phi, F_mean, F_min, F_max, (widx + 1), z, v

        last_A = A
        last_phi = phi

    return A_last, phi_last, F_mean_last, F_min_last, F_max_last, max_windows, z, v


class SPMRK45ApproachModel:
    """Clean approach-curve simulator with no solver-side force noise."""

    def __init__(self, params: NormalizedSPMParams, solver: Optional[RK45Options] = None):
        self.params = params
        self.solver = solver if solver is not None else RK45Options()

    def simulate_at_height(self, d_hat, z0_hat=0.0, v0_hat=0.0, omega_drive_hat=None, omega_ref_hat=None):
        if omega_drive_hat is None:
            omega_drive_hat = self.params.omega_drive_hat
        if omega_ref_hat is None:
            omega_ref_hat = self.params.omega_ref_hat
        so = self.solver
        p = self.params
        A, phi, F_mean, F_min, F_max, windows_used, z_final, v_final = simulate_steady_state_rk45(
            z0_hat=float(z0_hat),
            v0_hat=float(v0_hat),
            d_hat=float(d_hat),
            Q=float(p.Q), C1=float(p.C1), C2=float(p.C2), C3=float(p.C3), a0=float(p.a0),
            omega_drive_hat=float(omega_drive_hat), omega_ref_hat=float(omega_ref_hat),
            N_cycles=int(so.N_cycles), max_windows=int(so.max_windows), settle_windows=int(so.settle_windows),
            tol_A=float(so.tol_A), tol_phi_deg=float(so.tol_phi_deg), consecutive=int(so.consecutive),
            demod_per_ref_cycle=int(so.demod_per_ref_cycle),
            rtol=float(so.rtol), atol=float(so.atol), dt0=float(so.dt0), dt_min=float(so.dt_min), dt_max=float(so.dt_max),
            max_steps_per_window=int(so.max_steps_per_window),
        )
        return {
            "d_hat": float(d_hat),
            "A_hat": float(A),
            "phi_deg": float(phi),
            "F_mean_hat": float(F_mean),
            "F_min_hat": float(F_min),
            "F_max_hat": float(F_max),
            "windows_used": int(windows_used),
            "z_final_hat": float(z_final),
            "v_final_hat": float(v_final),
        }

    def approach_curve(self, d_start_hat, d_end_hat, n_points=256, z0_hat=0.0, v0_hat=0.0, warm_start=True, omega_drive_hat=None, omega_ref_hat=None):
        d_vals = np.linspace(float(d_start_hat), float(d_end_hat), int(n_points))
        A_vals = np.empty_like(d_vals)
        phi_vals = np.empty_like(d_vals)
        F_mean_vals = np.empty_like(d_vals)
        F_min_vals = np.empty_like(d_vals)
        F_max_vals = np.empty_like(d_vals)
        windows_vals = np.empty(d_vals.size, dtype=np.int64)
        z_cur = float(z0_hat)
        v_cur = float(v0_hat)

        for i, d_hat in enumerate(d_vals):
            out = self.simulate_at_height(d_hat=float(d_hat), z0_hat=z_cur, v0_hat=v_cur, omega_drive_hat=omega_drive_hat, omega_ref_hat=omega_ref_hat)
            A_vals[i] = out["A_hat"]
            phi_vals[i] = out["phi_deg"]
            F_mean_vals[i] = out["F_mean_hat"]
            F_min_vals[i] = out["F_min_hat"]
            F_max_vals[i] = out["F_max_hat"]
            windows_vals[i] = out["windows_used"]
            if warm_start:
                z_cur = out["z_final_hat"]
                v_cur = out["v_final_hat"]

        return {
            "d_hat": d_vals,
            "A_hat": A_vals,
            "phi_deg": phi_vals,
            "F_mean_hat": F_mean_vals,
            "F_min_hat": F_min_vals,
            "F_max_hat": F_max_vals,
            "windows_used": windows_vals,
        }


# ============================================================
# 2) PARAMETER / LIBRARY BUILDERS
# ============================================================
def normalize_params(params: Dict[str, float], conversions: Dict[str, float]) -> Dict[str, float]:
    return {k: float(params[k] * conversions[k]) for k in params}


def build_normalized_spm_params(
    params: Dict[str, float],
    conversions: Dict[str, float],
    drive_si: float,
    omega_drive_hat: float = 1.0,
    omega_ref_hat: float = 1.0,
    attrs: Optional[Dict[str, Any]] = None,
) -> Tuple[NormalizedSPMParams, Dict[str, float]]:
    params_local = dict(params)
    params_local["A"] = float(drive_si)
    param_norm = normalize_params(params_local, conversions)
    C1 = param_norm['H'] * param_norm['R'] / 6.0
    C2 = (4.0 / 3.0) * param_norm['E_star'] * np.sqrt(param_norm['R'])
    C3 = param_norm['A'] / param_norm['Q']
    meta = dict(attrs or {})
    meta.update({
        "drive_si": float(drive_si),
        "drive_hat": float(param_norm['A']),
        "omega_drive_hat": float(omega_drive_hat),
        "omega_ref_hat": float(omega_ref_hat),
    })
    spm_params = NormalizedSPMParams(
        Q=float(param_norm['Q']), C1=float(C1), C2=float(C2), C3=float(C3), a0=float(param_norm['a0']),
        omega_drive_hat=float(omega_drive_hat), omega_ref_hat=float(omega_ref_hat), attrs=meta,
    )
    return spm_params, param_norm


class FDCurve1D:
    """1D lookup model built from one precomputed approach curve."""

    def __init__(self, d_hat, A_hat, phi_deg, F_max_hat, F_mean_hat=None, clip=True, attrs: Optional[Dict[str, Any]] = None):
        d_hat = np.asarray(d_hat, dtype=float)
        A_hat = np.asarray(A_hat, dtype=float)
        phi_deg = np.asarray(phi_deg, dtype=float)
        F_max_hat = np.asarray(F_max_hat, dtype=float)
        if F_mean_hat is None:
            F_mean_hat = F_max_hat.copy()
        else:
            F_mean_hat = np.asarray(F_mean_hat, dtype=float)

        if not (d_hat.ndim == A_hat.ndim == phi_deg.ndim == F_max_hat.ndim == F_mean_hat.ndim == 1):
            raise ValueError("All inputs must be 1D arrays.")
        if not (d_hat.size == A_hat.size == phi_deg.size == F_max_hat.size == F_mean_hat.size):
            raise ValueError("All arrays must have the same length.")
        if d_hat.size < 2:
            raise ValueError("Need at least two FD points.")
        order = np.argsort(d_hat)
        self.d_hat = d_hat[order]
        self.A_hat = A_hat[order]
        self.phi_deg = phi_deg[order]
        self.F_max_hat = F_max_hat[order]
        self.F_mean_hat = F_mean_hat[order]
        self.clip = bool(clip)
        self.attrs = dict(attrs or {})

    def _prep_query(self, d_query):
        d_query = np.asarray(d_query, dtype=float)
        if self.clip:
            d_query = np.clip(d_query, self.d_hat[0], self.d_hat[-1])
        return d_query

    def evaluate(self, d_query):
        d_query = self._prep_query(d_query)
        A = np.interp(d_query, self.d_hat, self.A_hat)
        phi = np.interp(d_query, self.d_hat, self.phi_deg)
        F = np.interp(d_query, self.d_hat, self.F_max_hat)
        return A, phi, F

    def free_amplitude(self):
        return float(self.A_hat[-1])


class FDLibrary2D:
    """
    Multi-drive FD lookup library.

    Uses bilinear interpolation in (drive_hat, d_hat) for amplitude, phase, and force.
    """

    def __init__(self, drive_hat_grid, d_hat_grid, A_table, phi_table, F_max_table, F_mean_table=None, clip=True, attrs: Optional[Dict[str, Any]] = None):
        drive_hat_grid = np.asarray(drive_hat_grid, dtype=float)
        d_hat_grid = np.asarray(d_hat_grid, dtype=float)
        A_table = np.asarray(A_table, dtype=float)
        phi_table = np.asarray(phi_table, dtype=float)
        F_max_table = np.asarray(F_max_table, dtype=float)
    
        if F_mean_table is None:
            F_mean_table = F_max_table.copy()
        else:
            F_mean_table = np.asarray(F_mean_table, dtype=float)
    
        if drive_hat_grid.ndim != 1 or d_hat_grid.ndim != 1:
            raise ValueError("drive_hat_grid and d_hat_grid must be 1D.")
    
        expected = (drive_hat_grid.size, d_hat_grid.size)
        for arr in (A_table, phi_table, F_max_table, F_mean_table):
            if arr.shape != expected:
                raise ValueError(f"Table shape must be {expected}, got {arr.shape}.")
    
        # sort drive axis
        drive_order = np.argsort(drive_hat_grid)
        drive_hat_grid = drive_hat_grid[drive_order]
        A_table = A_table[drive_order]
        phi_table = phi_table[drive_order]
        F_max_table = F_max_table[drive_order]
        F_mean_table = F_mean_table[drive_order]
    
        # sort d axis
        d_order = np.argsort(d_hat_grid)
        d_hat_grid = d_hat_grid[d_order]
        A_table = A_table[:, d_order]
        phi_table = phi_table[:, d_order]
        F_max_table = F_max_table[:, d_order]
        F_mean_table = F_mean_table[:, d_order]
    
        self.drive_hat_grid = drive_hat_grid
        self.d_hat_grid = d_hat_grid
        self.A_table = A_table
        self.phi_table = phi_table
        self.F_max_table = F_max_table
        self.F_mean_table = F_mean_table
        self.clip = bool(clip)
        self.attrs = dict(attrs or {})

    @classmethod
    def from_curves(cls, drive_hat_grid, curves: Sequence[Dict[str, np.ndarray]], clip=True, attrs: Optional[Dict[str, Any]] = None):
        d0 = np.asarray(curves[0]["d_hat"], dtype=float)
        order0 = np.argsort(d0)
        d0_sorted = d0[order0]
    
        A_list, phi_list, Fmax_list, Fmean_list = [], [], [], []
    
        for curve in curves:
            d = np.asarray(curve["d_hat"], dtype=float)
            order = np.argsort(d)
            d_sorted = d[order]
    
            if d_sorted.shape != d0_sorted.shape or not np.allclose(d_sorted, d0_sorted):
                raise ValueError("All curves must share the same d_hat grid after sorting.")
    
            A_list.append(np.asarray(curve["A_hat"], dtype=float)[order])
            phi_list.append(np.asarray(curve["phi_deg"], dtype=float)[order])
            Fmax_list.append(np.asarray(curve["F_max_hat"], dtype=float)[order])
    
            if "F_mean_hat" in curve:
                Fmean_list.append(np.asarray(curve["F_mean_hat"], dtype=float)[order])
            else:
                Fmean_list.append(np.asarray(curve["F_max_hat"], dtype=float)[order])
    
        return cls(
            drive_hat_grid=np.asarray(drive_hat_grid, dtype=float),
            d_hat_grid=d0_sorted,
            A_table=np.vstack(A_list),
            phi_table=np.vstack(phi_list),
            F_max_table=np.vstack(Fmax_list),
            F_mean_table=np.vstack(Fmean_list),
            clip=clip,
            attrs=attrs,
        )

    def _prep_query(self, d_query, drive_query):
        d_query = np.asarray(d_query, dtype=float)
        drive_query = np.asarray(drive_query, dtype=float)
        if self.clip:
            d_query = np.clip(d_query, self.d_hat_grid[0], self.d_hat_grid[-1])
            drive_query = np.clip(drive_query, self.drive_hat_grid[0], self.drive_hat_grid[-1])
        return d_query, drive_query

    def _interp_drive_table(self, table, d_query, drive_hat):
        vals_per_drive = np.array([np.interp(d_query, self.d_hat_grid, row) for row in table], dtype=float)
        return float(np.interp(drive_hat, self.drive_hat_grid, vals_per_drive))

    def evaluate(self, d_query, drive_hat):
        d_query, drive_hat = self._prep_query(d_query, drive_hat)
        if np.ndim(d_query) == 0 and np.ndim(drive_hat) == 0:
            A = self._interp_drive_table(self.A_table, float(d_query), float(drive_hat))
            phi = self._interp_drive_table(self.phi_table, float(d_query), float(drive_hat))
            F = self._interp_drive_table(self.F_max_table, float(d_query), float(drive_hat))
            return float(A), float(phi), float(F)
        d_query = np.broadcast_to(d_query, np.shape(np.broadcast(d_query, drive_hat)))
        drive_hat = np.broadcast_to(drive_hat, d_query.shape)
        A = np.empty_like(d_query, dtype=float)
        phi = np.empty_like(d_query, dtype=float)
        F = np.empty_like(d_query, dtype=float)
        it = np.nditer([d_query, drive_hat, A, phi, F], flags=['multi_index'], op_flags=[['readonly']]*2 + [['writeonly']]*3)
        for d_i, dr_i, Ao, phio, Fo in it:
            Ao[...] = self._interp_drive_table(self.A_table, float(d_i), float(dr_i))
            phio[...] = self._interp_drive_table(self.phi_table, float(d_i), float(dr_i))
            Fo[...] = self._interp_drive_table(self.F_max_table, float(d_i), float(dr_i))
        return A, phi, F

    def free_amplitude(self, drive_hat):
        far_idx = np.argmax(self.d_hat_grid)
        return float(np.interp(float(drive_hat), self.drive_hat_grid, self.A_table[:, far_idx]))

    def nearest_drive_index(self, drive_hat):
        return int(np.argmin(np.abs(self.drive_hat_grid - float(drive_hat))))


@dataclass
class FDLibraryBuildResult:
    library: FDLibrary2D
    curves_by_drive: Dict[float, Dict[str, np.ndarray]]
    attrs: Dict[str, Any] = field(default_factory=dict)


def build_fd_library_from_drives(
    params: Dict[str, float],
    conversions: Dict[str, float],
    drive_si_values: Sequence[float],
    solver: Optional[RK45Options] = None,
    d_start_hat: Optional[float] = None,
    d_end_hat: float = -25.0,
    n_points: int = 256,
    omega_drive_hat: float = 1.0,
    omega_ref_hat: float = 1.0,
    warm_start: bool = True,
    clip: bool = True,
    extra_attrs: Optional[Dict[str, Any]] = None,
) -> FDLibraryBuildResult:
    drive_si_values = [float(v) for v in drive_si_values]
    curves = []
    drive_hat_grid = []
    curves_by_drive: Dict[float, Dict[str, np.ndarray]] = {}
    for drive_si in drive_si_values:
        spm_params, param_norm = build_normalized_spm_params(
            params=params, conversions=conversions, drive_si=drive_si,
            omega_drive_hat=omega_drive_hat, omega_ref_hat=omega_ref_hat,
            attrs={"source": "build_fd_library_from_drives"},
        )
        model = SPMRK45ApproachModel(spm_params, solver=solver)
        d0 = float(param_norm['d0']) if d_start_hat is None else float(d_start_hat)

        curve_raw = model.approach_curve(
            d_start_hat=d0,
            d_end_hat=float(d_end_hat),
            n_points=int(n_points),
            z0_hat=0.0,
            v0_hat=0.0,
            warm_start=warm_start,
        )
        
        order = np.argsort(curve_raw["d_hat"])
        curve = {k: (np.asarray(v)[order] if isinstance(v, np.ndarray) and np.asarray(v).shape == np.asarray(curve_raw["d_hat"]).shape else v)
                 for k, v in curve_raw.items()}
        
        curves.append(curve)
        drive_hat_grid.append(float(param_norm['A']))
        curves_by_drive[drive_si] = curve
        
        # curve = model.approach_curve(d_start_hat=d0, d_end_hat=float(d_end_hat), n_points=int(n_points), z0_hat=0.0, v0_hat=0.0, warm_start=warm_start)
        # curves.append(curve)
        # drive_hat_grid.append(float(param_norm['A']))
        # curves_by_drive[drive_si] = curve

    

    attrs = dict(extra_attrs or {})
    attrs.update({
        "drive_si_values": drive_si_values,
        "d_end_hat": float(d_end_hat),
        "n_points": int(n_points),
        "solver": asdict(solver if solver is not None else RK45Options()),
    })
    library = FDLibrary2D.from_curves(drive_hat_grid=drive_hat_grid, curves=curves, clip=clip, attrs=attrs)
    return FDLibraryBuildResult(library=library, curves_by_drive=curves_by_drive, attrs=attrs)


# ============================================================
# 3) PI-SIDE DRIFT + MULTI-DRIVE SCANNER
# ============================================================
@dataclass
class PIDriftNoiseOptions:
    drift_rel_std: float = 2e-3
    drift_tau_pixels: float = 200.0
    white_rel_std: float = 0.0
    sinusoid_rel_amp: float = 0.0
    sinusoid_period_pixels: float = 512.0
    seed: int = 0


def _lpf1_step(y, x, dt, tau):
    if tau is None or tau <= 0.0:
        return x
    alpha = dt / (tau + dt)
    return y + alpha * (x - y)


def _lpf2_step(y, ydot, x, dt, wn, zeta):
    if wn is None or wn <= 0.0:
        return x, 0.0
    ydd = (wn * wn) * (x - y) - (2.0 * zeta * wn) * ydot
    ydot_new = ydot + dt * ydd
    y_new = y + dt * ydot_new
    return y_new, ydot_new


def _ou_step_stationary(x, std, tau, dt, rng):
    if std <= 0.0:
        return 0.0
    if tau is None or tau <= 0.0:
        return std * rng.standard_normal()
    alpha = np.exp(-dt / tau)
    sigma = std * np.sqrt(max(1.0 - alpha * alpha, 0.0))
    return alpha * x + sigma * rng.standard_normal()


class ScannerFD2D:
    """
    Multi-drive scanner using a (drive_hat, d_hat) FD library.

    PI / LPF dynamics are simulated online. Optional low-frequency drift/noise is
    injected into the amplitude channel seen by the PI controller.
    """

    def __init__(self, params: Dict[str, float], conversions: Dict[str, float], fd_library: FDLibrary2D, pi_noise: Optional[PIDriftNoiseOptions] = None, attrs: Optional[Dict[str, Any]] = None):
        self.param = dict(params)
        self.conversion = dict(conversions)
        self.fd_library = fd_library
        self.pi_noise = pi_noise if pi_noise is not None else PIDriftNoiseOptions()
        self.rng = np.random.default_rng(self.pi_noise.seed)
        self.param_norm = normalize_params(self.param, self.conversion)
        self.param_norm['C1'] = self.param_norm['H'] * self.param_norm['R'] / 6.0
        self.param_norm['C2'] = 4.0 / 3.0 * self.param_norm['E_star'] * np.sqrt(self.param_norm['R'])
        self.param_norm['C3'] = self.param_norm['A'] / self.param_norm['Q']
        self.attrs = dict(attrs or {})

    def drive_si_to_hat(self, drive_si: float) -> float:
        return float(drive_si * self.conversion['A'])

    def measure_from_fd(self, d_hat: float, drive_hat: float):
        A, phi, F = self.fd_library.evaluate(float(d_hat), float(drive_hat))
        return float(A), float(phi), float(F)

    def _pi_noise_step(self, drift_hat, A0_hat, dt_pixel, pixel_index):
        nopt = self.pi_noise
        drift_std_hat = float(nopt.drift_rel_std * A0_hat)
        white_std_hat = float(nopt.white_rel_std * A0_hat)
        sinusoid_amp_hat = float(nopt.sinusoid_rel_amp * A0_hat)
        tau_hat = float(max(nopt.drift_tau_pixels, 1e-12) * dt_pixel)
        drift_hat = _ou_step_stationary(drift_hat, std=drift_std_hat, tau=tau_hat, dt=dt_pixel, rng=self.rng)
        white_hat = white_std_hat * self.rng.standard_normal() if white_std_hat > 0.0 else 0.0
        if sinusoid_amp_hat > 0.0 and nopt.sinusoid_period_pixels > 0.0:
            sinusoid_hat = sinusoid_amp_hat * np.sin(2.0 * np.pi * pixel_index / nopt.sinusoid_period_pixels)
        else:
            sinusoid_hat = 0.0
        total_hat = drift_hat + white_hat + sinusoid_hat
        return drift_hat, white_hat, sinusoid_hat, total_hat

    def _pack_carry_state(self, out):
        return dict(
            d_cmd_init_hat=out["d_cmd_final_hat"],
            d_act_init_hat=out["d_final_hat"],
            A_meas_init_hat=out["A_meas_final_hat"],
            phi_meas_init_deg=out["phi_meas_final_deg"],
            integ_init=out["integ_final"],
            drift_init_hat=out["drift_final_hat"],
        )

    def _flip_line_output(self, out):
        flipped = {}
        for k, v in out.items():
            if isinstance(v, np.ndarray) and v.ndim == 1:
                flipped[k] = v[::-1].copy()
            else:
                flipped[k] = v
        return flipped

    def scan_line_pi(
        self,
        h_hat,
        drive_hat: Optional[float] = None,
        drive_si: Optional[float] = None,
        setpoint: float = 0.8,
        scan_speed_hat: float = 1.0,
        dx_hat: float = 1.0,
        P: float = 0.1,
        I: float = 0.01,
        A_filter_order=None,
        tau_A=1e-2,
        wn_A=1e3,
        zeta_A=0.7,
        phi_filter_order=None,
        tau_phi=1e-2,
        wn_phi=1e3,
        zeta_phi=0.7,
        z_filter_order=None,
        tau_z=1e-2,
        wn_z=1e3,
        zeta_z=0.7,
        z_rate_limit_hat=None,
        T_I=0.0,
        integ_clip=None,
        d_cmd_init_hat=None,
        d_act_init_hat=None,
        A_meas_init_hat=None,
        phi_meas_init_deg=0.0,
        integ_init=0.0,
        drift_init_hat=0.0,
        anti_windup=True,
    ):
        h_hat = np.asarray(h_hat, dtype=float)
        n = h_hat.size
        dt = float(dx_hat / scan_speed_hat)

        if drive_hat is None:
            if drive_si is not None:
                drive_hat = self.drive_si_to_hat(drive_si)
            else:
                drive_hat = float(self.param_norm['A'])
        drive_hat = float(drive_hat)

        A0 = float(self.fd_library.free_amplitude(drive_hat))
        A_set = float(setpoint * A0)
        d_cmd = float(self.param_norm['d0'] if d_cmd_init_hat is None else d_cmd_init_hat)
        d_act = float(d_cmd if d_act_init_hat is None else d_act_init_hat)
        d_act_dot = 0.0
        A_meas = float(A0 if A_meas_init_hat is None else A_meas_init_hat)
        A_meas_dot = 0.0
        phi_meas = float(phi_meas_init_deg)
        phi_meas_dot = 0.0
        integ = float(integ_init)
        drift_hat = float(drift_init_hat)

        if T_I is None or T_I <= 0.0:
            decay = 1.0
            use_leak = False
        else:
            decay = float(np.exp(-dt / T_I))
            use_leak = True

        d_min = float(self.fd_library.d_hat_grid[0])
        d_max = float(self.fd_library.d_hat_grid[-1])

        out_d_eff = np.empty(n)
        out_d_cmd = np.empty(n)
        out_d_act = np.empty(n)
        out_A_true = np.empty(n)
        out_A_meas = np.empty(n)
        out_A_ctrl = np.empty(n)
        out_phi_true = np.empty(n)
        out_phi_meas = np.empty(n)
        out_Fmax = np.empty(n)
        out_err = np.empty(n)
        out_drift = np.empty(n)
        out_white = np.empty(n)
        out_sinusoid = np.empty(n)
        out_noise_total = np.empty(n)
        out_drive_hat = np.full(n, drive_hat)
        out_A_set = np.full(n, A_set)

        for i in range(n):
            d_eff = d_act - h_hat[i]
            d_eff_eval = float(np.clip(d_eff, d_min, d_max))
            A_true, phi_true, F_max = self.measure_from_fd(d_eff_eval, drive_hat)

            if A_filter_order is None:
                A_meas = A_true
            elif A_filter_order == 1:
                A_meas = _lpf1_step(A_meas, A_true, dt, tau_A)
            elif A_filter_order == 2:
                A_meas, A_meas_dot = _lpf2_step(A_meas, A_meas_dot, A_true, dt, wn_A, zeta_A)
            else:
                raise ValueError("A_filter_order must be None, 1, or 2")

            if phi_filter_order is None:
                phi_meas = phi_true
            elif phi_filter_order == 1:
                phi_meas = _lpf1_step(phi_meas, phi_true, dt, tau_phi)
            elif phi_filter_order == 2:
                phi_meas, phi_meas_dot = _lpf2_step(phi_meas, phi_meas_dot, phi_true, dt, wn_phi, zeta_phi)
            else:
                raise ValueError("phi_filter_order must be None, 1, or 2")

            drift_hat, white_hat, sinusoid_hat, total_noise_hat = self._pi_noise_step(drift_hat=drift_hat, A0_hat=A0, dt_pixel=dt, pixel_index=i)
            A_ctrl = A_meas + total_noise_hat
            err = A_set - A_ctrl

            sat_hi = d_eff > d_max
            sat_lo = d_eff < d_min
            would_wind_up = anti_windup and ((sat_hi and err > 0.0) or (sat_lo and err < 0.0))
            if not would_wind_up:
                if use_leak:
                    integ = decay * integ + err * dt
                else:
                    integ = integ + err * dt
            if integ_clip is not None:
                integ = float(np.clip(integ, -integ_clip, integ_clip))

            dd_cmd = P * err + I * integ
            if z_rate_limit_hat is not None:
                max_dd = float(z_rate_limit_hat) * dt
                dd_cmd = float(np.clip(dd_cmd, -max_dd, max_dd))
            d_cmd = d_cmd + dd_cmd

            if z_filter_order is None:
                d_act = d_cmd
            elif z_filter_order == 1:
                d_act = _lpf1_step(d_act, d_cmd, dt, tau_z)
            elif z_filter_order == 2:
                d_act, d_act_dot = _lpf2_step(d_act, d_act_dot, d_cmd, dt, wn_z, zeta_z)
            else:
                raise ValueError("z_filter_order must be None, 1, or 2")

            out_d_eff[i] = d_eff
            out_d_cmd[i] = d_cmd
            out_d_act[i] = d_act
            out_A_true[i] = A_true
            out_A_meas[i] = A_meas
            out_A_ctrl[i] = A_ctrl
            out_phi_true[i] = phi_true
            out_phi_meas[i] = phi_meas
            out_Fmax[i] = F_max
            out_err[i] = err
            out_drift[i] = drift_hat
            out_white[i] = white_hat
            out_sinusoid[i] = sinusoid_hat
            out_noise_total[i] = total_noise_hat

        return {
            "d_eff_hat": out_d_eff,
            "d_cmd_hat": out_d_cmd,
            "d_hat": out_d_act,
            "A_true_hat": out_A_true,
            "A_hat": out_A_meas,
            "A_ctrl_hat": out_A_ctrl,
            "A_set_hat": out_A_set,
            "phi_true_deg": out_phi_true,
            "phi_deg": out_phi_meas,
            "F_max_hat": out_Fmax,
            "err": out_err,
            "drift_hat": out_drift,
            "white_hat": out_white,
            "sinusoid_hat": out_sinusoid,
            "noise_total_hat": out_noise_total,
            "drive_hat": out_drive_hat,
            "dt_pixel": dt,
            "d_cmd_final_hat": float(d_cmd),
            "d_final_hat": float(d_act),
            "A_meas_final_hat": float(A_meas),
            "phi_meas_final_deg": float(phi_meas),
            "integ_final": float(integ),
            "drift_final_hat": float(drift_hat),
        }

    def scan_line_trace_retrace_pi(self, h_hat, retrace_carry_state=True, retrace_use_reversed_topography=True, **line_kwargs):
        h_hat = np.asarray(h_hat, dtype=float)
        out_tr = self.scan_line_pi(h_hat, **line_kwargs)
        h_rt = h_hat[::-1] if retrace_use_reversed_topography else h_hat.copy()
        retrace_init = self._pack_carry_state(out_tr) if retrace_carry_state else {}
        out_rt_raw = self.scan_line_pi(h_rt, **retrace_init, **line_kwargs)
        out_rt = self._flip_line_output(out_rt_raw)
        return {"trace": out_tr, "retrace": out_rt}

    def scan_map_trace_retrace_pi(self, h_map_hat, carry_state_between_lines=True, retrace_carry_state=True, **line_kwargs):
        h_map_hat = np.asarray(h_map_hat, dtype=float)
        if h_map_hat.ndim != 2:
            raise ValueError("h_map_hat must be 2D.")
        ny, nx = h_map_hat.shape

        channels = {
            "height_trace_map": np.empty((ny, nx), dtype=float),
            "height_retrace_map": np.empty((ny, nx), dtype=float),
            "phase_trace_map": np.empty((ny, nx), dtype=float),
            "phase_retrace_map": np.empty((ny, nx), dtype=float),
            "amplitude_trace_map": np.empty((ny, nx), dtype=float),
            "amplitude_retrace_map": np.empty((ny, nx), dtype=float),
            "amplitude_ctrl_trace_map": np.empty((ny, nx), dtype=float),
            "amplitude_ctrl_retrace_map": np.empty((ny, nx), dtype=float),
            "force_trace_map": np.empty((ny, nx), dtype=float),
            "force_retrace_map": np.empty((ny, nx), dtype=float),
            "err_trace_map": np.empty((ny, nx), dtype=float),
            "err_retrace_map": np.empty((ny, nx), dtype=float),
            "drift_trace_map": np.empty((ny, nx), dtype=float),
            "drift_retrace_map": np.empty((ny, nx), dtype=float),
        }
        carry = dict(
            d_cmd_init_hat=line_kwargs.pop("d_cmd_init_hat", None),
            d_act_init_hat=line_kwargs.pop("d_act_init_hat", None),
            A_meas_init_hat=line_kwargs.pop("A_meas_init_hat", None),
            phi_meas_init_deg=line_kwargs.pop("phi_meas_init_deg", 0.0),
            integ_init=line_kwargs.pop("integ_init", 0.0),
            drift_init_hat=line_kwargs.pop("drift_init_hat", 0.0),
        )
        for iy in range(ny):
            row = h_map_hat[iy, :]
            out_pair = self.scan_line_trace_retrace_pi(row, retrace_carry_state=retrace_carry_state, **carry, **line_kwargs)
            tr = out_pair["trace"]
            rt = out_pair["retrace"]
            channels["height_trace_map"][iy, :] = tr["d_hat"]
            channels["height_retrace_map"][iy, :] = rt["d_hat"]
            channels["phase_trace_map"][iy, :] = tr["phi_deg"]
            channels["phase_retrace_map"][iy, :] = rt["phi_deg"]
            channels["amplitude_trace_map"][iy, :] = tr["A_hat"]
            channels["amplitude_retrace_map"][iy, :] = rt["A_hat"]
            channels["amplitude_ctrl_trace_map"][iy, :] = tr["A_ctrl_hat"]
            channels["amplitude_ctrl_retrace_map"][iy, :] = rt["A_ctrl_hat"]
            channels["force_trace_map"][iy, :] = tr["F_max_hat"]
            channels["force_retrace_map"][iy, :] = rt["F_max_hat"]
            channels["err_trace_map"][iy, :] = tr["err"]
            channels["err_retrace_map"][iy, :] = rt["err"]
            channels["drift_trace_map"][iy, :] = tr["drift_hat"]
            channels["drift_retrace_map"][iy, :] = rt["drift_hat"]
            if carry_state_between_lines:
                carry = self._pack_carry_state(rt)
        return channels


# ============================================================
# 4) PLOTTING HELPERS
# ============================================================
def plot_trace_retrace_channels(out_pair, h_hat=None, center_height=True, figsize=(10, 9), title=None):
    tr = out_pair["trace"]
    rt = out_pair["retrace"]
    x = np.arange(tr["d_hat"].size)

    fig, axes = plt.subplots(3, 2, figsize=figsize, sharex=True)

    h_tr = tr["d_hat"].copy()
    h_rt = rt["d_hat"].copy()
    gt = None if h_hat is None else np.asarray(h_hat, dtype=float).copy()
    if center_height:
        h_tr -= np.mean(h_tr)
        h_rt -= np.mean(h_rt)
        if gt is not None:
            gt -= np.mean(gt)

    axes[0, 0].plot(x, h_tr, lw=2, label="Trace")
    if gt is not None:
        axes[0, 0].plot(x, gt, lw=1.5, ls="--", label="Ground truth")
    axes[0, 0].set_ylabel("Height")
    axes[0, 0].set_title("Trace height")
    axes[0, 0].grid(alpha=0.3)
    axes[0, 0].legend()

    axes[0, 1].plot(x, h_rt, lw=2, label="Retrace")
    if gt is not None:
        axes[0, 1].plot(x, gt, lw=1.5, ls="--", label="Ground truth")
    axes[0, 1].set_ylabel("Height")
    axes[0, 1].set_title("Retrace height")
    axes[0, 1].grid(alpha=0.3)
    axes[0, 1].legend()

    axes[1, 0].plot(x, tr["phi_deg"], lw=2)
    axes[1, 0].set_ylabel("Phase (deg)")
    axes[1, 0].set_title("Trace phase")
    axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(x, rt["phi_deg"], lw=2)
    axes[1, 1].set_ylabel("Phase (deg)")
    axes[1, 1].set_title("Retrace phase")
    axes[1, 1].grid(alpha=0.3)

    axes[2, 0].plot(x, tr["A_hat"], lw=2, label="A_meas")
    if "A_ctrl_hat" in tr:
        axes[2, 0].plot(x, tr["A_ctrl_hat"], lw=1.5, ls="--", label="A_ctrl")
    axes[2, 0].set_ylabel("Amplitude")
    axes[2, 0].set_xlabel("Pixel")
    axes[2, 0].set_title("Trace amplitude")
    axes[2, 0].grid(alpha=0.3)
    axes[2, 0].legend()

    axes[2, 1].plot(x, rt["A_hat"], lw=2, label="A_meas")
    if "A_ctrl_hat" in rt:
        axes[2, 1].plot(x, rt["A_ctrl_hat"], lw=1.5, ls="--", label="A_ctrl")
    axes[2, 1].set_ylabel("Amplitude")
    axes[2, 1].set_xlabel("Pixel")
    axes[2, 1].set_title("Retrace amplitude")
    axes[2, 1].grid(alpha=0.3)
    axes[2, 1].legend()

    if title is not None:
        fig.suptitle(title)
        fig.tight_layout(rect=(0, 0, 1, 0.97))
    else:
        fig.tight_layout()
    return fig, axes


# ============================================================
# 5) ONLINE INSTRUMENT INTEGRATION WRAPPER
# ============================================================
class OnlineInstrumentBridge:
    """
    Maintains controller state across sequential scan lines for online use.
    """

    def __init__(self, scanner: ScannerFD2D):
        self.scanner = scanner
        self.reset()

    def reset(self):
        self.state = dict(
            d_cmd_init_hat=None,
            d_act_init_hat=None,
            A_meas_init_hat=None,
            phi_meas_init_deg=0.0,
            integ_init=0.0,
            drift_init_hat=0.0,
        )

    def scan_line(self, h_hat, drive_hat=None, drive_si=None, trace_retrace=True, **kwargs):
        if trace_retrace:
            out = self.scanner.scan_line_trace_retrace_pi(h_hat, drive_hat=drive_hat, drive_si=drive_si, **self.state, **kwargs)
            self.state = self.scanner._pack_carry_state(out["retrace"])
            return out
        out = self.scanner.scan_line_pi(h_hat, drive_hat=drive_hat, drive_si=drive_si, **self.state, **kwargs)
        self.state = self.scanner._pack_carry_state(out)
        return out


# ============================================================
# 6) EXPERIMENT GRID DATA + ALIGNMENT / CALIBRATION
# ============================================================
@dataclass
class ExperimentGridData:
    drives_exp: np.ndarray
    setpoints_exp: np.ndarray
    I_gains_exp: np.ndarray
    height_trace: np.ndarray
    height_retrace: np.ndarray
    phase_trace: np.ndarray
    phase_retrace: np.ndarray
    amplitude_trace: Optional[np.ndarray] = None
    amplitude_retrace: Optional[np.ndarray] = None
    attrs: Dict[str, Any] = field(default_factory=dict)

    def __post_init__(self):
        self.drives_exp = np.asarray(self.drives_exp, dtype=float)
        self.setpoints_exp = np.asarray(self.setpoints_exp, dtype=float)
        self.I_gains_exp = np.asarray(self.I_gains_exp, dtype=float)
        self.height_trace = np.asarray(self.height_trace, dtype=float)
        self.height_retrace = np.asarray(self.height_retrace, dtype=float)
        self.phase_trace = np.asarray(self.phase_trace, dtype=float)
        self.phase_retrace = np.asarray(self.phase_retrace, dtype=float)
        if self.amplitude_trace is not None:
            self.amplitude_trace = np.asarray(self.amplitude_trace, dtype=float)
        if self.amplitude_retrace is not None:
            self.amplitude_retrace = np.asarray(self.amplitude_retrace, dtype=float)

        expected = (len(self.drives_exp), len(self.setpoints_exp), len(self.I_gains_exp))
        for name, arr in [
            ("height_trace", self.height_trace),
            ("height_retrace", self.height_retrace),
            ("phase_trace", self.phase_trace),
            ("phase_retrace", self.phase_retrace),
        ]:
            if arr.ndim != 4 or arr.shape[:3] != expected:
                raise ValueError(f"{name} must have shape (n_drive, n_setpoint, n_I, n_pixel).")
        if self.amplitude_trace is not None and (self.amplitude_trace.ndim != 4 or self.amplitude_trace.shape[:3] != expected):
            raise ValueError("amplitude_trace must have shape (n_drive, n_setpoint, n_I, n_pixel).")
        if self.amplitude_retrace is not None and (self.amplitude_retrace.ndim != 4 or self.amplitude_retrace.shape[:3] != expected):
            raise ValueError("amplitude_retrace must have shape (n_drive, n_setpoint, n_I, n_pixel).")

    @property
    def n_pixel(self):
        return self.height_trace.shape[-1]


@dataclass
class AlignmentConfig:
    compare_amplitude: bool = False
    height_weight: float = 1.0
    phase_weight: float = 1.0
    amplitude_weight: float = 0.5
    hysteresis_weight: float = 0.5
    remove_height_offset: bool = True
    remove_height_tilt: bool = True
    phase_wrap_deg: float = 180.0
    robust_eps: float = 1e-8
    line_kwargs_base: Dict[str, Any] = field(default_factory=lambda: dict(
        scan_speed_hat=1.0,
        dx_hat=1.0,
        P=0.1,
        I=0.01,
        A_filter_order=1,
        tau_A=1e-2,
        phi_filter_order=1,
        tau_phi=1e-2,
        z_filter_order=1,
        tau_z=1e-2,
        T_I=0.0,
    ))


def _mad_scale(x: np.ndarray, eps: float = 1e-8) -> float:
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    return float(max(1.4826 * mad, eps))


def _remove_offset_tilt(y: np.ndarray, remove_offset: bool = True, remove_tilt: bool = True) -> np.ndarray:
    y = np.asarray(y, dtype=float)
    if not (remove_offset or remove_tilt):
        return y.copy()
    x = np.arange(y.size, dtype=float)
    if remove_tilt:
        A = np.column_stack([np.ones_like(x), x])
        coef, *_ = np.linalg.lstsq(A, y, rcond=None)
        trend = A @ coef
        return y - trend
    if remove_offset:
        return y - np.mean(y)
    return y.copy()


def _wrap_phase_diff_deg(a: np.ndarray, b: np.ndarray, period: float = 180.0) -> np.ndarray:
    diff = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    return ((diff + 0.5 * period) % period) - 0.5 * period


def _pseudo_huber(residual: np.ndarray, delta: float = 1.0) -> float:
    r = np.asarray(residual, dtype=float)
    return float(np.mean(delta * delta * (np.sqrt(1.0 + (r / delta) ** 2) - 1.0)))


def default_vector_to_theta(x: np.ndarray) -> Dict[str, float]:
    """
    Default alignment parameterization.

    x = [
        drive_scale, drive_shift_hat,
        setpoint_scale, setpoint_shift,
        I_scale,
        phase_scale, phase_offset_deg,
        tau_A, tau_phi, tau_z,
        P_scale
    ]
    """
    return {
        "drive_scale": float(x[0]),
        "drive_shift_hat": float(x[1]),
        "setpoint_scale": float(x[2]),
        "setpoint_shift": float(x[3]),
        "I_scale": float(x[4]),
        "phase_scale": float(x[5]),
        "phase_offset_deg": float(x[6]),
        "tau_A": float(x[7]),
        "tau_phi": float(x[8]),
        "tau_z": float(x[9]),
        "P_scale": float(x[10]),
    }


class GridCalibrator2D:
    """
    Align multi-drive FD scanner simulations to an experimental drive/setpoint/I grid.
    """

    def __init__(
        self,
        experiment: ExperimentGridData,
        scanner: ScannerFD2D,
        surface_ref_hat: np.ndarray,
        config: Optional[AlignmentConfig] = None,
        attrs: Optional[Dict[str, Any]] = None,
    ):
        self.experiment = experiment
        self.scanner = scanner
        self.surface_ref_hat = np.asarray(surface_ref_hat, dtype=float)
        self.config = config if config is not None else AlignmentConfig()
        self.attrs = dict(attrs or {})
        if self.surface_ref_hat.ndim != 1 or self.surface_ref_hat.size != self.experiment.n_pixel:
            raise ValueError("surface_ref_hat must be 1D with length equal to n_pixel.")

    def _line_kwargs_from_theta(self, theta: Dict[str, float], I_exp: float) -> Dict[str, Any]:
        kw = dict(self.config.line_kwargs_base)
        kw["I"] = float(I_exp * theta.get("I_scale", 1.0))
        kw["P"] = float(kw.get("P", 0.1) * theta.get("P_scale", 1.0))
        kw["tau_A"] = float(theta.get("tau_A", kw.get("tau_A", 1e-2)))
        kw["tau_phi"] = float(theta.get("tau_phi", kw.get("tau_phi", 1e-2)))
        kw["tau_z"] = float(theta.get("tau_z", kw.get("tau_z", 1e-2)))
        return kw

    def _map_drive_setpoint(self, drive_exp: float, setpoint_exp: float, theta: Dict[str, float]) -> Tuple[float, float]:
        drive_hat = theta.get("drive_scale", 1.0) * float(drive_exp) + theta.get("drive_shift_hat", 0.0)
        setpoint = theta.get("setpoint_scale", 1.0) * float(setpoint_exp) + theta.get("setpoint_shift", 0.0)
        return drive_hat, setpoint

    def simulate_condition(self, drive_exp: float, setpoint_exp: float, I_exp: float, theta: Dict[str, float], carry_state: bool = False):
        drive_hat, setpoint = self._map_drive_setpoint(drive_exp, setpoint_exp, theta)
        line_kwargs = self._line_kwargs_from_theta(theta, I_exp)
        out_pair = self.scanner.scan_line_trace_retrace_pi(
            h_hat=self.surface_ref_hat,
            drive_hat=drive_hat,
            setpoint=setpoint,
            retrace_carry_state=carry_state,
            **line_kwargs,
        )
        # affine phase correction
        for key in ("trace", "retrace"):
            out_pair[key]["phi_deg"] = theta.get("phase_scale", 1.0) * out_pair[key]["phi_deg"] + theta.get("phase_offset_deg", 0.0)
        return out_pair

    def condition_loss(self, out_pair: Dict[str, Dict[str, np.ndarray]], exp_ht_tr, exp_ht_rt, exp_ph_tr, exp_ph_rt, exp_amp_tr=None, exp_amp_rt=None) -> float:
        cfg = self.config
        loss = 0.0

        sim_ht_tr = _remove_offset_tilt(out_pair["trace"]["d_hat"], cfg.remove_height_offset, cfg.remove_height_tilt)
        sim_ht_rt = _remove_offset_tilt(out_pair["retrace"]["d_hat"], cfg.remove_height_offset, cfg.remove_height_tilt)
        exp_ht_tr = _remove_offset_tilt(exp_ht_tr, cfg.remove_height_offset, cfg.remove_height_tilt)
        exp_ht_rt = _remove_offset_tilt(exp_ht_rt, cfg.remove_height_offset, cfg.remove_height_tilt)

        hs = 0.5 * (_mad_scale(exp_ht_tr, cfg.robust_eps) + _mad_scale(exp_ht_rt, cfg.robust_eps))
        ps = 0.5 * (_mad_scale(exp_ph_tr, cfg.robust_eps) + _mad_scale(exp_ph_rt, cfg.robust_eps))

        loss += cfg.height_weight * _pseudo_huber((sim_ht_tr - exp_ht_tr) / hs)
        loss += cfg.height_weight * _pseudo_huber((sim_ht_rt - exp_ht_rt) / hs)

        ph_tr_diff = _wrap_phase_diff_deg(out_pair["trace"]["phi_deg"], exp_ph_tr, cfg.phase_wrap_deg) / ps
        ph_rt_diff = _wrap_phase_diff_deg(out_pair["retrace"]["phi_deg"], exp_ph_rt, cfg.phase_wrap_deg) / ps
        loss += cfg.phase_weight * _pseudo_huber(ph_tr_diff)
        loss += cfg.phase_weight * _pseudo_huber(ph_rt_diff)

        # hysteresis matching
        sim_dh = (sim_ht_tr - sim_ht_rt) / hs
        exp_dh = (exp_ht_tr - exp_ht_rt) / hs
        loss += cfg.hysteresis_weight * _pseudo_huber(sim_dh - exp_dh)

        sim_dp = _wrap_phase_diff_deg(out_pair["trace"]["phi_deg"], out_pair["retrace"]["phi_deg"], cfg.phase_wrap_deg) / ps
        exp_dp = _wrap_phase_diff_deg(exp_ph_tr, exp_ph_rt, cfg.phase_wrap_deg) / ps
        loss += cfg.hysteresis_weight * _pseudo_huber(sim_dp - exp_dp)

        if cfg.compare_amplitude and exp_amp_tr is not None and exp_amp_rt is not None:
            sim_amp_tr = out_pair["trace"]["A_hat"]
            sim_amp_rt = out_pair["retrace"]["A_hat"]
            ascale = 0.5 * (_mad_scale(exp_amp_tr, cfg.robust_eps) + _mad_scale(exp_amp_rt, cfg.robust_eps))
            loss += cfg.amplitude_weight * _pseudo_huber((sim_amp_tr - exp_amp_tr) / ascale)
            loss += cfg.amplitude_weight * _pseudo_huber((sim_amp_rt - exp_amp_rt) / ascale)

        return float(loss)

    def total_loss(self, theta: Dict[str, float], subset: Optional[Sequence[Tuple[int, int, int]]] = None) -> float:
        exp = self.experiment
        if subset is None:
            subset = [(i, j, k) for i in range(len(exp.drives_exp)) for j in range(len(exp.setpoints_exp)) for k in range(len(exp.I_gains_exp))]
        total = 0.0
        for i, j, k in subset:
            out_pair = self.simulate_condition(exp.drives_exp[i], exp.setpoints_exp[j], exp.I_gains_exp[k], theta)
            amp_tr = None if exp.amplitude_trace is None else exp.amplitude_trace[i, j, k]
            amp_rt = None if exp.amplitude_retrace is None else exp.amplitude_retrace[i, j, k]
            total += self.condition_loss(
                out_pair,
                exp.height_trace[i, j, k],
                exp.height_retrace[i, j, k],
                exp.phase_trace[i, j, k],
                exp.phase_retrace[i, j, k],
                amp_tr,
                amp_rt,
            )
        return float(total / max(len(subset), 1))

    def fit(
        self,
        bounds: Sequence[Tuple[float, float]],
        vector_to_theta=default_vector_to_theta,
        x0: Optional[np.ndarray] = None,
        subset: Optional[Sequence[Tuple[int, int, int]]] = None,
        maxiter_de: int = 30,
        polish: bool = True,
        seed: int = 0,
        workers: int = 1,
    ) -> Dict[str, Any]:
        def obj(x):
            theta = vector_to_theta(np.asarray(x, dtype=float))
            return self.total_loss(theta, subset=subset)

        result_de = differential_evolution(obj, bounds=bounds, maxiter=maxiter_de, seed=seed, workers=workers, polish=False)
        x_best = result_de.x.copy()
        f_best = float(result_de.fun)
        result_local = None
        if polish:
            result_local = minimize(obj, x_best, method="Powell")
            if float(result_local.fun) < f_best:
                x_best = result_local.x.copy()
                f_best = float(result_local.fun)
        theta_best = vector_to_theta(np.asarray(x_best, dtype=float))
        return {
            "x_best": x_best,
            "theta_best": theta_best,
            "loss_best": f_best,
            "de_result": result_de,
            "local_result": result_local,
        }


# ============================================================
# 7) VISUAL DIAGNOSTICS FOR ALIGNMENT
# ============================================================
def plot_alignment_condition(out_pair, exp_height_trace, exp_height_retrace, exp_phase_trace, exp_phase_retrace, title=None, center_height=True, figsize=(10, 9)):
    tr = out_pair["trace"]
    rt = out_pair["retrace"]
    x = np.arange(len(exp_height_trace))

    sim_h_tr = tr["d_hat"].copy()
    sim_h_rt = rt["d_hat"].copy()
    exp_h_tr = np.asarray(exp_height_trace, dtype=float).copy()
    exp_h_rt = np.asarray(exp_height_retrace, dtype=float).copy()
    if center_height:
        sim_h_tr -= np.mean(sim_h_tr)
        sim_h_rt -= np.mean(sim_h_rt)
        exp_h_tr -= np.mean(exp_h_tr)
        exp_h_rt -= np.mean(exp_h_rt)

    fig, axes = plt.subplots(3, 2, figsize=figsize, sharex=True)
    axes[0, 0].plot(x, exp_h_tr, label="Exp", lw=1.8)
    axes[0, 0].plot(x, sim_h_tr, label="Sim", lw=1.8)
    axes[0, 0].set_title("Trace height")
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(x, exp_h_rt, label="Exp", lw=1.8)
    axes[0, 1].plot(x, sim_h_rt, label="Sim", lw=1.8)
    axes[0, 1].set_title("Retrace height")
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)

    axes[1, 0].plot(x, exp_phase_trace, label="Exp", lw=1.8)
    axes[1, 0].plot(x, tr["phi_deg"], label="Sim", lw=1.8)
    axes[1, 0].set_title("Trace phase")
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(x, exp_phase_retrace, label="Exp", lw=1.8)
    axes[1, 1].plot(x, rt["phi_deg"], label="Sim", lw=1.8)
    axes[1, 1].set_title("Retrace phase")
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)

    axes[2, 0].plot(x, tr["A_hat"], label="Trace A_meas", lw=1.8)
    axes[2, 0].plot(x, tr["A_ctrl_hat"], label="Trace A_ctrl", lw=1.4, ls="--")
    axes[2, 0].set_title("Trace amplitude")
    axes[2, 0].legend()
    axes[2, 0].grid(alpha=0.3)

    axes[2, 1].plot(x, rt["A_hat"], label="Retrace A_meas", lw=1.8)
    axes[2, 1].plot(x, rt["A_ctrl_hat"], label="Retrace A_ctrl", lw=1.4, ls="--")
    axes[2, 1].set_title("Retrace amplitude")
    axes[2, 1].legend()
    axes[2, 1].grid(alpha=0.3)

    if title is not None:
        fig.suptitle(title)
        fig.tight_layout(rect=(0, 0, 1, 0.97))
    else:
        fig.tight_layout()
    return fig, axes


# ============================================================
# 8) EXAMPLE WORKFLOW HELPERS
# ============================================================
def build_scanner_from_drive_library(
    params: Dict[str, float],
    conversions: Dict[str, float],
    drive_si_values: Sequence[float],
    solver: Optional[RK45Options] = None,
    pi_noise: Optional[PIDriftNoiseOptions] = None,
    d_start_hat: Optional[float] = None,
    d_end_hat: float = -25.0,
    n_points: int = 256,
    omega_drive_hat: float = 1.0,
    omega_ref_hat: float = 1.0,
    attrs: Optional[Dict[str, Any]] = None,
):
    build_result = build_fd_library_from_drives(
        params=params,
        conversions=conversions,
        drive_si_values=drive_si_values,
        solver=solver,
        d_start_hat=d_start_hat,
        d_end_hat=d_end_hat,
        n_points=n_points,
        omega_drive_hat=omega_drive_hat,
        omega_ref_hat=omega_ref_hat,
        extra_attrs=attrs,
    )
    scanner = ScannerFD2D(params=params, conversions=conversions, fd_library=build_result.library, pi_noise=pi_noise)
    return build_result, scanner


# __all__ = [
#     "NormalizedSPMParams",
#     "RK45Options",
#     "SPMRK45ApproachModel",
#     "normalize_params",
#     "build_normalized_spm_params",
#     "FDCurve1D",
#     "FDLibrary2D",
#     "FDLibraryBuildResult",
#     "build_fd_library_from_drives",
#     "PIDriftNoiseOptions",
#     "ScannerFD2D",
#     "OnlineInstrumentBridge",
#     "ExperimentGridData",
#     "AlignmentConfig",
#     "default_vector_to_theta",
#     "GridCalibrator2D",
#     "plot_trace_retrace_channels",
#     "plot_alignment_condition",
#     "build_scanner_from_drive_library",
# ]


In [ ]:
# import numpy as np
# # from spm_fd_multidrive_alignment import (
# #     RK45Options,
# #     PIDriftNoiseOptions,
# #     build_scanner_from_drive_library,
# #     ExperimentGridData,
# #     AlignmentConfig,
# #     GridCalibrator2D,
# #     plot_trace_retrace_channels,
# #     plot_alignment_condition,
# # )

# # ------------------------------------------------------------
# # 1) Build multi-drive FD library + scanner
# # ------------------------------------------------------------
# solver = RK45Options(
#     N_cycles=20,
#     max_windows=40,
#     settle_windows=1,
#     tol_A=1e-4,
#     tol_phi_deg=1e-2,
#     consecutive=2,
#     demod_per_ref_cycle=30,
#     rtol=1e-6,
#     atol=1e-9,
#     dt0=0.02,
#     dt_min=1e-10,
#     dt_max=0.3,
#     max_steps_per_window=200000,
# )

# pi_noise = PIDriftNoiseOptions(
#     drift_rel_std=2e-3,
#     drift_tau_pixels=200.0,
#     white_rel_std=0.0,
#     sinusoid_rel_amp=0.0,
#     seed=1,
# )

# # drive_si_values = [40e-9, 60e-9, 80e-9, 100e-9, 120e-9]

# drive_si_values = np.linspace(25e-9, 175e-9, num=21)

# build_result, scanner = build_scanner_from_drive_library(
#     params=params,
#     conversions=conversions,
#     drive_si_values=drive_si_values,
#     solver=solver,
#     pi_noise=pi_noise,
#     d_end_hat=-10.0,
#     n_points=256,
# )



In [ ]:
drive_si_values

In [ ]:
# # ------------------------------------------------------------
# # 2) Simulate one trace/retrace line at a chosen drive
# # ------------------------------------------------------------
# out_pair = scanner.scan_line_trace_retrace_pi(
#     h_hat=h_line,             # use nm-like normalized units
#     drive_si=100e-9,          # choose drive here
#     setpoint=0.9,
#     scan_speed_hat=1.0,
#     dx_hat=1.0,
#     P=1,
#     I=1e-2,
#     A_filter_order=1,
#     tau_A=1e-2,
#     phi_filter_order=1,
#     tau_phi=1e-2,
#     z_filter_order=1,
#     tau_z=1e-2,
# )

# plot_trace_retrace_channels(out_pair, h_hat=h_line, center_height=True)

In [ ]:
# # ------------------------------------------------------------
# # 3) Pack experimental grid
# # shape of each channel:
# # (n_drive, n_setpoint, n_I, n_pixel)
# # ------------------------------------------------------------
# exp_data = ExperimentGridData(
#     drives_exp=drives_exp,              # same units you want the calibrator to consume
#     setpoints_exp=setpoints_exp,
#     I_gains_exp=I_gains_exp,
#     height_trace=height_trace_exp,
#     height_retrace=height_retrace_exp,
#     phase_trace=phase_trace_exp,
#     phase_retrace=phase_retrace_exp,
#     # amplitude_trace=amp_trace_exp,    # optional
#     # amplitude_retrace=amp_retrace_exp # optional
# )

# # surface_ref_hat should be the reference sample line used by the simulator
# # often a weak-interaction / high-setpoint condition is a good starting point
# surface_ref_hat = 0.5 * (
#     exp_data.height_trace[0, -1, 0] + exp_data.height_retrace[0, -1, 0]
# )

# cfg = AlignmentConfig(
#     compare_amplitude=False,
#     height_weight=1.0,
#     phase_weight=1.0,
#     hysteresis_weight=0.5,
# )

# cal = GridCalibrator2D(
#     experiment=exp_data,
#     scanner=scanner,
#     surface_ref_hat=surface_ref_hat,
#     config=cfg,
# )

# # default_vector_to_theta expects:
# # [drive_scale, drive_shift_hat,
# #  setpoint_scale, setpoint_shift,
# #  I_scale,
# #  phase_scale, phase_offset_deg,
# #  tau_A, tau_phi, tau_z,
# #  P_scale]

# bounds = [
#     (0.8, 1.2),    # drive_scale
#     (-20.0, 20.0), # drive_shift_hat
#     (0.8, 1.2),    # setpoint_scale
#     (-0.1, 0.1),   # setpoint_shift
#     (0.5, 2.0),    # I_scale
#     (0.8, 1.2),    # phase_scale
#     (-30.0, 30.0), # phase_offset_deg
#     (1e-4, 1e-1),  # tau_A
#     (1e-4, 1e-1),  # tau_phi
#     (1e-4, 1e-1),  # tau_z
#     (0.5, 2.0),    # P_scale
# ]

# result = cal.fit(bounds=bounds, maxiter_de=20, polish=True, seed=0)

# print(result["theta_best"])
# print(result["loss_best"])

In [ ]:
# theta = result["theta_best"]

# out_pair_fit = cal.simulate_condition(
#     drive_exp=exp_data.drives_exp[0],
#     setpoint_exp=exp_data.setpoints_exp[0],
#     I_exp=exp_data.I_gains_exp[0],
#     theta=theta,
# )

# plot_alignment_condition(
#     out_pair_fit,
#     exp_height_trace=exp_data.height_trace[0, 0, 0],
#     exp_height_retrace=exp_data.height_retrace[0, 0, 0],
#     exp_phase_trace=exp_data.phase_trace[0, 0, 0],
#     exp_phase_retrace=exp_data.phase_retrace[0, 0, 0],
#     title="Aligned condition",
# )

## Load experimental grid data

In [ ]:
import numpy as np
import os

file1 = os.path.join('data', 'AR_Height_AE.npz')

data1 = np.load(file1)

print('scan_rate:', data1['scan_rate'],)
print('v_ac:', data1['v_ac'],)
print('setpoint:', data1['setpoint'],)
print('i_gain:', data1['i_gain'])
print('data:', data1['data'].shape)

# # Outputs:

# scan_rate: [0.25 0.5  0.75 1.   2.   3.   4.   5.   6.  ]
# v_ac: [0.15   0.1375 0.125  0.1125 0.1    0.0875 0.075  0.0625 0.05   0.0375
#  0.025 ]
# setpoint: [4.2 4.1 4.  3.9 3.8 3.7 3.6 3.5 3.4 3.3 3.2 3.1 3.  2.9 2.8 2.7]
# i_gain: [ 10.  25.  40.  55.  70.  85. 100. 115. 130. 145. 160.]
# data: (9, 11, 16, 11, 8, 512)

In [ ]:
# Let's verify the optimization results

n_drive = 11
n_set = 16

DRIVES = np.linspace(3.91, 23.47, n_drive)

SETPOINTS = np.linspace(80, 0.01, n_set) * 1e-2

n_drive = 11
n_set = 16

drives = DRIVES * 1e-9
# setpoints = 0.8545345811910205 * SETPOINTS - 0.2066306487444366
setpoints = 0.5 * SETPOINTS

n_drive = len(DRIVES)
n_set = len(SETPOINTS)
exp_channels_grid = [[None for _ in range(n_set)] for _ in range(n_drive)]
np.shape(exp_channels_grid)


In [ ]:
# Extract FD curves from a grid of experimental scan lines.
"""
Extract FD curves from a grid of experimental scan lines.

Idea
----
At each (drive, setpoint), the steady-state operating point of the controller
sits on the FD curve at d* where A(d*) = setpoint*drive. Aggregating across
the pixels of slow, high-I scan lines at that (drive, setpoint) gives us
(A*, phi*, d*) for the curve at that drive. Sweeping setpoint at fixed drive
maps out the full FD curve at that drive.

Inputs (matching the user's grid):
    data       : (n_scan, n_drive, n_set, n_I, 6, n_pix)
                 channel order (configurable):
                 h_trace, h_retrace, phi_trace, phi_retrace, A_trace, A_retrace
    h_truth    : (n_pix,) ground-truth topography (nm), same x as the data
    scan_rates : (n_scan,)   scan rate per axis-0 entry (Hz)
    drives_nm  : (n_drive,)  drive amplitudes (nm)
    setpoints  : (n_set,)    setpoint fractions in (0, 1)
    I_gains    : (n_I,)      I-gain values (log10 if i_gain_is_log10 set later)
"""
from __future__ import annotations

import numpy as np
from typing import Dict, Optional, Sequence


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def histogram_mode(x: np.ndarray, n_bins: int = 51) -> float:
    """Modal value via histogram peak — robust against bimodal / heavy-tail."""
    x = np.asarray(x, dtype=float).ravel()
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    if x.size < 5:
        return float(np.median(x))
    bins = max(8, min(int(n_bins), x.size // 2))
    counts, edges = np.histogram(x, bins=bins)
    i = int(np.argmax(counts))
    return float(0.5 * (edges[i] + edges[i + 1]))


# -----------------------------------------------------------------------------
# Main extraction
# -----------------------------------------------------------------------------

DEFAULT_CHANNEL_ORDER = {
    'h_trace':    0, 'h_retrace':   1,
    'phi_trace':  2, 'phi_retrace': 3,
    'A_trace':    4, 'A_retrace':   5,
}


def extract_operating_points(
    data: np.ndarray,
    h_truth: np.ndarray,
    scan_rates: np.ndarray,
    drives_nm: np.ndarray,
    setpoints: np.ndarray,
    I_gains: np.ndarray,
    *,
    channel_order: Dict[str, int] = None,
    aggregator: str = 'median',
    edge_margin: int = 32,
    scan_rate_indices: Optional[Sequence[int]] = None,
    I_gain_indices:    Optional[Sequence[int]] = None,
    A_window_rel: Optional[float] = 0.10,
) -> Dict[float, Dict[str, np.ndarray]]:
    """Aggregate the grid into one (d, A, phi) per (drive, setpoint).

    Parameters
    ----------
    aggregator : 'median' (default) or 'mode' (histogram peak).
    edge_margin : drop this many pixels at each line end (transients).
    scan_rate_indices : subset to use; default = slowest scan only.
    I_gain_indices    : subset to use; default = largest log10 I (fastest loop).
    A_window_rel : if not None, keep only pixels where A is within this
                   relative window around setpoint*drive (e.g. 0.10 = ±10 %).
                   Excludes pixels still in transient at step edges.
                   Set to None to skip this filter.

    Returns
    -------
    {drive_nm : {'d', 'A', 'phi', 'setpoints'}} — arrays of length n_set.
    """
    if channel_order is None:
        channel_order = DEFAULT_CHANNEL_ORDER

    n_scan, n_drive, n_set, n_I, n_chan, n_pix = data.shape
    if n_chan != 6:
        raise ValueError(f"Expected 6 channels, got {n_chan}")
    if scan_rate_indices is None:
        scan_rate_indices = [int(np.argmin(scan_rates))]
    if I_gain_indices is None:
        I_gain_indices = [int(np.argmax(I_gains))]

    h_truth = np.asarray(h_truth, dtype=float)
    sl   = slice(edge_margin, n_pix - edge_margin)
    h_sl = h_truth[sl]

    if aggregator == 'median':
        agg = lambda a: float(np.nanmedian(a))
    elif aggregator == 'mode':
        agg = histogram_mode
    else:
        raise ValueError("aggregator must be 'median' or 'mode'")

    ch = channel_order
    out: Dict[float, Dict[str, np.ndarray]] = {}

    for di, drv in enumerate(drives_nm):
        d_list, A_list, phi_list = [], [], []
        for si, sp in enumerate(setpoints):
            d_pool, A_pool, phi_pool = [], [], []
            for s_idx in scan_rate_indices:
                for i_idx in I_gain_indices:
                    sub = data[s_idx, di, si, i_idx]   # (6, n_pix)
                    for direction in ('trace', 'retrace'):
                        h   = sub[ch[f'h_{direction}']][sl]
                        phi = sub[ch[f'phi_{direction}']][sl]
                        A   = sub[ch[f'A_{direction}']][sl]
                        d_eff = h - h_sl

                        if A_window_rel is not None:
                            target = float(sp) * float(drv)
                            keep = np.abs(A - target) <= A_window_rel * target
                            d_eff, A, phi = d_eff[keep], A[keep], phi[keep]

                        if d_eff.size:
                            d_pool.append(d_eff); A_pool.append(A); phi_pool.append(phi)

            if not d_pool:
                d_list.append(np.nan); A_list.append(np.nan); phi_list.append(np.nan)
                continue
            d_all   = np.concatenate(d_pool)
            A_all   = np.concatenate(A_pool)
            phi_all = np.concatenate(phi_pool)
            d_list.append(agg(d_all))
            A_list.append(agg(A_all))
            phi_list.append(agg(phi_all))

        out[float(drv)] = {
            'd':         np.array(d_list),
            'A':         np.array(A_list),
            'phi':       np.array(phi_list),
            'setpoints': np.asarray(setpoints, dtype=float),
        }
    return out


def build_library_from_extracted(
    extracted: Dict[float, Dict[str, np.ndarray]],
    anchor_setpoint: Optional[float] = None,
):
    """Construct an ApproachCurveLibrary from extracted operating points.

    Optionally anchor d=0 at a chosen reference setpoint (e.g. the highest
    in your sweep) so each curve shares a comparable d origin. The simulator
    doesn't require this, but plots become interpretable.
    """
    # from approach_curve_library import ApproachCurveLibrary

    lib = ApproachCurveLibrary()
    for drv, info in extracted.items():
        d = np.asarray(info['d'], dtype=float)
        A = np.asarray(info['A'], dtype=float)
        phi = np.asarray(info['phi'], dtype=float)
        keep = np.isfinite(d) & np.isfinite(A) & np.isfinite(phi)
        d, A, phi = d[keep], A[keep], phi[keep]
        if d.size < 3:
            continue
        if anchor_setpoint is not None:
            sp = np.asarray(info['setpoints'])[keep]
            i_anchor = int(np.argmin(np.abs(sp - anchor_setpoint)))
            d = d - d[i_anchor]
        order = np.argsort(d)
        lib.add(float(drv), d[order], A[order], phi[order])
    return lib


# -----------------------------------------------------------------------------
# Diagnostic: validate by overplotting library curves on raw cloud
# -----------------------------------------------------------------------------

def collect_pixel_cloud(
    data: np.ndarray,
    h_truth: np.ndarray,
    drives_nm: np.ndarray,
    setpoints: np.ndarray,
    I_gains: np.ndarray,
    *,
    drive_nm: float,
    channel_order: Dict[str, int] = None,
    edge_margin: int = 32,
    A_window_rel: Optional[float] = 0.10,
):
    """Pool every (d_eff, A, phi) pixel for a chosen drive, across all
    setpoints / I-gains / scan rates. Useful to overlay against the
    extracted FD curve and eyeball the fit.

    Returns: (d_arr, A_arr, phi_arr, setpoint_arr) — flat 1D arrays.
    """
    if channel_order is None:
        channel_order = DEFAULT_CHANNEL_ORDER
    n_scan, n_drive, n_set, n_I, _, n_pix = data.shape
    di = int(np.argmin(np.abs(np.asarray(drives_nm) - drive_nm)))
    sl = slice(edge_margin, n_pix - edge_margin)
    h_sl = np.asarray(h_truth)[sl]
    ch = channel_order

    d_all, A_all, phi_all, sp_all = [], [], [], []
    for si, sp in enumerate(setpoints):
        for s_idx in range(n_scan):
            for i_idx in range(n_I):
                sub = data[s_idx, di, si, i_idx]
                for direction in ('trace', 'retrace'):
                    h   = sub[ch[f'h_{direction}']][sl]
                    phi = sub[ch[f'phi_{direction}']][sl]
                    A   = sub[ch[f'A_{direction}']][sl]
                    d_eff = h - h_sl
                    if A_window_rel is not None:
                        target = float(sp) * float(drives_nm[di])
                        keep = np.abs(A - target) <= A_window_rel * target
                        d_eff, A, phi = d_eff[keep], A[keep], phi[keep]
                    if d_eff.size:
                        d_all.append(d_eff); A_all.append(A); phi_all.append(phi)
                        sp_all.append(np.full(d_eff.size, float(sp)))
    return (np.concatenate(d_all),    np.concatenate(A_all),
            np.concatenate(phi_all),  np.concatenate(sp_all))


In [ ]:
# Calibrate a ScannerFD digital twin against experimental scan lines.
"""
Calibrate a ScannerFD digital twin against experimental scan lines.

Free parameters fit (theta, length 7):
    a_drive, b_drive          drive_sim    = a_drive * drive_exp    + b_drive
    a_setpoint, b_setpoint    setpoint_sim = a_setpoint * setpoint_exp + b_setpoint
    b_I_log10                 log-space offset on integral gain
                              I_sim = 10**(b_I_log10 + log10_I_exp)   if i_gain_is_log10=True
                              I_sim = 10**b_I_log10 * I_exp           if i_gain_is_log10=False
                              (i.e., a single multiplicative factor c_I = 10**b_I_log10)
    tau_A                     amplitude / phase demodulator time constant (1st-order LPF)
    tau_z                     Z-actuator time constant (1st-order LPF)

Held fixed per scan line:
    P_gain                    0 by default; expose as an extra theta entry if needed

Inputs the user supplies:
    scanner       : a ScannerFD instance configured with nominal physical params
    fd_lookup(A)  : callable returning an FDCurve1D for a given drive amplitude (nm)
                    (interpolate your approach-curve library here)
    h_truth_hat   : 1D ground-truth topography (already in the normalized length
                    units used by the scanner, i.e. h * conversion['L'])
    experiments   : list of dicts (one per scan line). Use flatten_experiments(...)
                    to build it from a packed (D, S, I, 4, P) array.
"""
from __future__ import annotations

import numpy as np
from scipy.optimize import least_squares
from typing import Callable, Iterable, List, Dict, Optional, Tuple


# -----------------------------------------------------------------------------
# Parameter layout
# -----------------------------------------------------------------------------

# Internal theta order; tau_A and tau_z are stored as log(tau) inside theta
# but exposed to users in linear form via the `fix` / `x0` interfaces.
PARAM_NAMES = (
    'a_drive', 'b_drive',
    'a_setpoint', 'b_setpoint',
    'b_I_log10',
    'tau_A', 'tau_z',
)
PARAM_INDEX = {n: i for i, n in enumerate(PARAM_NAMES)}
LOG_PARAMS = {'tau_A', 'tau_z'}  # stored as log internally


def _user_to_internal(name: str, value: float) -> float:
    """Convert a user-facing value (linear tau, etc.) to internal theta value."""
    if name in LOG_PARAMS:
        if value <= 0:
            raise ValueError(f"{name} must be positive, got {value}")
        return float(np.log(value))
    return float(value)


def _internal_to_user(name: str, value: float) -> float:
    if name in LOG_PARAMS:
        return float(np.exp(value))
    return float(value)


# -----------------------------------------------------------------------------
# Subsampling helpers — for grids that are too large to fit on directly
# -----------------------------------------------------------------------------

def subsample_random(experiments: List[Dict], n: int = 200, seed: int = 0) -> List[Dict]:
    """Uniform random sample of n lines from the full grid."""
    if len(experiments) <= n:
        return list(experiments)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(experiments), size=n, replace=False)
    return [experiments[int(i)] for i in idx]


def subsample_stratified(
    experiments: List[Dict],
    keys: Tuple[str, ...] = ('drive', 'setpoint', 'I_gain', 'scan_rate_hz'),
    n_per_stratum: int = 1,
    seed: int = 0,
) -> List[Dict]:
    """Take n_per_stratum lines per unique combination of `keys`.
    Default: one line per (drive, setpoint, I_gain, scan_rate) cell, which is
    typically what a calibration fit needs from a dense grid.
    """
    rng = np.random.default_rng(seed)
    groups: Dict[Tuple, List[int]] = {}
    for i, e in enumerate(experiments):
        try:
            k = tuple(e[name] for name in keys)
        except KeyError as exc:
            raise KeyError(f"Stratification key {exc} missing from experiment dict") from exc
        groups.setdefault(k, []).append(i)
    chosen: List[int] = []
    for k, idxs in groups.items():
        if len(idxs) <= n_per_stratum:
            chosen.extend(idxs)
        else:
            chosen.extend(int(j) for j in rng.choice(idxs, size=n_per_stratum, replace=False))
    return [experiments[i] for i in chosen]


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def wrap_phase_diff(phi_a_deg, phi_b_deg) -> np.ndarray:
    """Phase difference (a - b) wrapped to (-180, 180] degrees."""
    d = np.asarray(phi_a_deg, dtype=float) - np.asarray(phi_b_deg, dtype=float)
    return (d + 180.0) % 360.0 - 180.0


def set_scanner_drive(scanner, A_nm: float, fd_model) -> None:
    """Switch scanner free amplitude and FD lookup in place."""
    scanner.param['A'] = float(A_nm)
    scanner.param_norm['A'] = float(A_nm) * scanner.conversion['A']
    scanner.param_norm['C3'] = scanner.param_norm['A'] / scanner.param_norm['Q']
    scanner.fd_model = fd_model


# -----------------------------------------------------------------------------
# Forward model: one simulated line, both directions
# -----------------------------------------------------------------------------

def simulate_trace_retrace(
    scanner,
    fd_lookup: Callable[[float], object],
    h_truth_hat: np.ndarray,
    drive_nm: float,
    setpoint_frac: float,
    I_gain: float,
    tau_A: float,
    tau_z: float,
    P_gain: float = 0.0,
    tau_phi: Optional[float] = None,
    burn_in: int = 128,
    scan_speed_hat: float = 1.0,
    dx_hat: float = 1.0,
) -> Dict[str, np.ndarray]:
    """Simulate one scan line forward (trace) and backward (retrace).

    Returns a dict with h_{trace,retrace}, A_{trace,retrace}, phi_{trace,retrace},
    each of length len(h_truth_hat). The retrace arrays are returned in the
    same x-order as the trace (i.e. left-to-right) for direct comparison.
    """
    fd_model = fd_lookup(drive_nm)
    set_scanner_drive(scanner, drive_nm, fd_model)

    if tau_phi is None:
        tau_phi = tau_A

    common = dict(
        setpoint=setpoint_frac,
        scan_speed_hat=scan_speed_hat,
        dx_hat=dx_hat,
        P=P_gain,
        I=I_gain,
        A_filter_order=1, tau_A=tau_A,
        phi_filter_order=1, tau_phi=tau_phi,
        z_filter_order=1, tau_z=tau_z,
    )

    # --- Trace: pad with first sample, scan left-to-right, drop the burn-in ---
    pad_t = np.full(burn_in, h_truth_hat[0], dtype=float)
    h_in_t = np.concatenate([pad_t, h_truth_hat])
    out_t = scanner.scan_line_pi(h_in_t, **common)

    h_trace   = out_t['d_hat'][burn_in:]
    A_trace   = out_t['A_hat'][burn_in:]
    phi_trace = out_t['phi_deg'][burn_in:]

    # --- Retrace: scan reversed h, carry controller state from end of trace ---
    pad_r = np.full(burn_in, h_truth_hat[-1], dtype=float)
    h_in_r = np.concatenate([pad_r, h_truth_hat[::-1]])
    out_r = scanner.scan_line_pi(
        h_in_r,
        d_cmd_init_hat=out_t['d_cmd_final_hat'],
        d_act_init_hat=out_t['d_final_hat'],
        A_meas_init_hat=out_t['A_meas_final_hat'],
        phi_meas_init_deg=out_t['phi_meas_final_deg'],
        integ_init=out_t['integ_final'],
        **common,
    )
    # Drop burn-in, then reverse so x-order matches the trace.
    h_retrace   = out_r['d_hat'][burn_in:][::-1]
    A_retrace   = out_r['A_hat'][burn_in:][::-1]
    phi_retrace = out_r['phi_deg'][burn_in:][::-1]

    return dict(
        h_trace=h_trace,     h_retrace=h_retrace,
        A_trace=A_trace,     A_retrace=A_retrace,
        phi_trace=phi_trace, phi_retrace=phi_retrace,
    )


# -----------------------------------------------------------------------------
# Pack / unpack experimental data
# -----------------------------------------------------------------------------

def flatten_grid_experiments(
    data: np.ndarray,
    scan_rates_hz: np.ndarray,
    drives: np.ndarray,
    setpoints: np.ndarray,
    I_gains: np.ndarray,
    n_pix_per_line: int = 256,
    line_period_convention: str = 'full',  # 'full' = forward+reverse per period
    channel_order: Tuple[str, ...] = (
        'h_trace', 'h_retrace', 'phi_trace', 'phi_retrace',
    ),
    extra_channels: Tuple[str, ...] = ('A_trace', 'A_retrace'),  # ignored, just dropped
) -> List[Dict]:
    """For grids of shape (n_scan, n_drive, n_set, n_I, n_chan, n_pix).

    Computes scan_speed_hat per line from the scan rate, assuming
    dx_hat = 1.0 in the simulator and dt_phys = 1 / (factor * f_scan * n_pix)
    where factor = 2 for 'full' triangle-period convention or 1 for
    'one_direction'. Stores 'scan_speed_hat' and 'dx_hat' in each dict so
    calibrate(...) uses the correct dt for each line.

    `extra_channels` lets you keep A_trace, A_retrace etc. in the same
    record if you want to fit against amplitude too — currently the cost
    function ignores anything beyond h/phi, so it's safe to skip.
    """
    data = np.asarray(data)
    n_scan, n_drv, n_set, n_I, n_chan, n_pix = data.shape
    if n_chan < len(channel_order):
        raise ValueError(f"Need at least {len(channel_order)} channels, got {n_chan}")
    if not (len(scan_rates_hz) == n_scan and len(drives) == n_drv
            and len(setpoints) == n_set and len(I_gains) == n_I):
        raise ValueError("Axis arrays don't match data shape")

    factor = 2.0 if line_period_convention == 'full' else 1.0
    rows: List[Dict] = []
    for si, f_scan in enumerate(scan_rates_hz):
        dt_phys = 1.0 / (factor * float(f_scan) * n_pix)
        v_hat = 1.0 / dt_phys
        for di, drv in enumerate(drives):
            for spi, sp in enumerate(setpoints):
                for ii, ig in enumerate(I_gains):
                    line = {ch: np.asarray(data[si, di, spi, ii, ci, :], dtype=float)
                            for ci, ch in enumerate(channel_order)}
                    rows.append({
                        'drive':          float(drv),
                        'setpoint':       float(sp),
                        'I_gain':         float(ig),
                        'scan_rate_hz':   float(f_scan),
                        'scan_speed_hat': v_hat,
                        'dx_hat':         1.0,
                        **line,
                    })
    return rows


def flatten_experiments(
    data: np.ndarray,
    drives: np.ndarray,
    setpoints: np.ndarray,
    I_gains: np.ndarray,
    channel_order: Tuple[str, ...] = (
        'h_trace', 'h_retrace', 'phi_trace', 'phi_retrace'
    ),
) -> List[Dict]:
    """Convert a packed (n_drive, n_setpoint, n_Igain, n_channel, n_pix) array
    into a flat list of per-line dicts.

    The default channel_order matches your description "trace and retrace of
    heights + phases" interpreted as (h_trace, h_retrace, phi_trace, phi_retrace).
    Override channel_order if your packing differs.
    """
    data = np.asarray(data)
    nD, nS, nI, nC, nP = data.shape
    if nC != len(channel_order):
        raise ValueError(f"Expected {len(channel_order)} channels, got {nC}")
    if not (len(drives) == nD and len(setpoints) == nS and len(I_gains) == nI):
        raise ValueError("drives / setpoints / I_gains lengths must match data shape")

    rows: List[Dict] = []
    for i in range(nD):
        for j in range(nS):
            for k in range(nI):
                line = {ch: np.asarray(data[i, j, k, ci, :], dtype=float)
                        for ci, ch in enumerate(channel_order)}
                rows.append({
                    'drive':    float(drives[i]),
                    'setpoint': float(setpoints[j]),
                    'I_gain':   float(I_gains[k]),
                    **line,
                })
    return rows


# -----------------------------------------------------------------------------
# Cost function
# -----------------------------------------------------------------------------

def make_residual_fn(
    scanner,
    fd_lookup: Callable[[float], object],
    h_truth_hat: np.ndarray,
    experiments: List[Dict],
    weights: Optional[Dict[str, float]] = None,
    scan_speed_hat: float = 1.0,
    dx_hat: float = 1.0,
    burn_in: int = 128,
    huge_penalty: float = 1e6,
    i_gain_is_log10: bool = False,
):
    """Build a residual vector: stacked (h_trace, h_retrace, phi_trace, phi_retrace)
    differences across all experimental lines. Phase residuals are wrapped.

    Default weights make 1 deg of phase ~ 0.01 normalized length unit.

    i_gain_is_log10 : if True, exp['I_gain'] is interpreted as log10 of the
                      raw instrument gain. The fit's b_I_log10 is then a
                      pure offset in log space.
                      If False, exp['I_gain'] is linear and b_I_log10 acts as
                      a multiplicative scale factor c_I = 10**b_I_log10.
    """
    if weights is None:
        weights = {'h': 1.0, 'phi': 0.01}

    h_truth_hat = np.asarray(h_truth_hat, dtype=float)
    n_pix = h_truth_hat.size

    def residual(theta: np.ndarray) -> np.ndarray:
        a_d, b_d, a_s, b_s, b_I_log10, log_tau_A, log_tau_z = theta
        tau_A = float(np.exp(log_tau_A))
        tau_z = float(np.exp(log_tau_z))

        chunks: List[np.ndarray] = []
        for exp in experiments:
            drive_sim    = a_d * exp['drive']    + b_d
            setpoint_sim = a_s * exp['setpoint'] + b_s

            # I-gain conversion: experiment -> simulator
            I_exp = float(exp['I_gain'])
            if i_gain_is_log10:
                I_sim = 10.0 ** (b_I_log10 + I_exp)
            else:
                I_sim = (10.0 ** b_I_log10) * I_exp

            # Refuse pathological setpoints early (saves a sim call)
            if not (0.01 < setpoint_sim < 0.999) or drive_sim <= 0.0 or I_sim <= 0.0:
                chunks.append(np.full(4 * n_pix, huge_penalty))
                continue

            # Per-line scan_speed_hat overrides the global one if present.
            v_line = float(exp.get('scan_speed_hat', scan_speed_hat))
            dx_line = float(exp.get('dx_hat', dx_hat))
            try:
                sim = simulate_trace_retrace(
                    scanner, fd_lookup, h_truth_hat,
                    drive_nm=float(drive_sim),
                    setpoint_frac=float(setpoint_sim),
                    I_gain=float(I_sim),
                    tau_A=tau_A, tau_z=tau_z,
                    burn_in=burn_in,
                    scan_speed_hat=v_line,
                    dx_hat=dx_line,
                )
            except Exception:
                # FD lookup outside library, NaN on a branch jump, etc.
                chunks.append(np.full(4 * n_pix, huge_penalty))
                continue

            # Reject silently divergent sims (PI loop blowup -> inf/nan).
            if not (np.all(np.isfinite(sim['h_trace']))
                    and np.all(np.isfinite(sim['h_retrace']))
                    and np.all(np.isfinite(sim['phi_trace']))
                    and np.all(np.isfinite(sim['phi_retrace']))):
                chunks.append(np.full(4 * n_pix, huge_penalty))
                continue

            r_h_t = (sim['h_trace']   - exp['h_trace'])   * weights['h']
            r_h_r = (sim['h_retrace'] - exp['h_retrace']) * weights['h']
            r_p_t = wrap_phase_diff(sim['phi_trace'],   exp['phi_trace'])   * weights['phi']
            r_p_r = wrap_phase_diff(sim['phi_retrace'], exp['phi_retrace']) * weights['phi']
            # Final guard: clip any straggler non-finites in the residual itself.
            for r in (r_h_t, r_h_r, r_p_t, r_p_r):
                np.nan_to_num(r, copy=False, nan=huge_penalty,
                              posinf=huge_penalty, neginf=-huge_penalty)
            chunks.extend([r_h_t, r_h_r, r_p_t, r_p_r])

        return np.concatenate(chunks)

    return residual


# -----------------------------------------------------------------------------
# Top-level driver
# -----------------------------------------------------------------------------

def calibrate(
    scanner,
    fd_lookup: Callable[[float], object],
    h_truth_hat: np.ndarray,
    experiments: List[Dict],
    x0: Optional[Dict[str, float]] = None,
    bounds: Optional[Dict[str, Tuple[float, float]]] = None,
    fix: Optional[Dict[str, float]] = None,
    weights: Optional[Dict[str, float]] = None,
    scan_speed_hat: float = 1.0,
    dx_hat: float = 1.0,
    burn_in: int = 128,
    verbose: int = 2,
    max_nfev: int = 200,
    i_gain_is_log10: bool = False,
) -> Dict:
    """Fit any subset of (a_drive, b_drive, a_setpoint, b_setpoint,
    b_I_log10, tau_A, tau_z).

    Parameters
    ----------
    x0     : optional dict of starting values, e.g. {'tau_A': 5e-3, 'tau_z': 2e-3}.
             Missing names use the defaults below. Pass tau_* in linear seconds
             (or whatever units match your dt); they're converted to log internally.
    bounds : optional dict {name: (lo, hi)}. For tau_*, the bounds are also
             linear; converted to log internally.
    fix    : optional dict {name: value}. Pinned parameters are NOT fit; the
             optimizer only varies the remaining names. Use this to lock the
             LPF time constants while you sort out the linear transforms.
    i_gain_is_log10 : True if exp['I_gain'] is stored as log10(raw_gain).

    Default starting point (when x0 is None):
        a_drive=1, b_drive=0, a_setpoint=1, b_setpoint=0, b_I_log10=0,
        tau_A=1e-3, tau_z=1e-4.

    Default bounds (when bounds is None):
        a_drive    : [0.5, 2.0]      b_drive    : [-20, 20]
        a_setpoint : [0.7, 1.5]      b_setpoint : [-0.2, 0.2]
        b_I_log10  : [-3, 3]         tau_A, tau_z : [1e-7, 1e0]

    Returns
    -------
    dict with the fitted parameters (including pinned values), 'free' /
    'fixed' parameter lists, the scipy result object, and 'cost'.
    """
    # ---- defaults (always length 7, in internal theta layout) ----
    default_x0 = {
        'a_drive': 1.0, 'b_drive': 0.0,
        'a_setpoint': 1.0, 'b_setpoint': 0.0,
        'b_I_log10': 0.0,
        'tau_A': 1e-3, 'tau_z': 1e-4,
    }
    default_bounds = {
        'a_drive':    (0.5, 2.0),
        'b_drive':    (-20.0, 20.0),
        'a_setpoint': (0.7, 1.5),
        'b_setpoint': (-0.20, 0.20),
        'b_I_log10':  (-3.0, 3.0),
        'tau_A':      (1e-7, 1e0),
        'tau_z':      (1e-7, 1e0),
    }
    x0_user     = {**default_x0,     **(x0 or {})}
    bounds_user = {**default_bounds, **(bounds or {})}
    fix         = fix or {}

    # ---- validate names ----
    for name in list(x0_user) + list(bounds_user) + list(fix):
        if name not in PARAM_INDEX:
            raise ValueError(
                f"Unknown parameter '{name}'. Allowed: {list(PARAM_NAMES)}"
            )

    # ---- build full internal theta vectors ----
    theta_full = np.array([
        _user_to_internal(n, x0_user[n]) for n in PARAM_NAMES
    ])
    lo_full = np.array([
        _user_to_internal(n, bounds_user[n][0]) for n in PARAM_NAMES
    ])
    hi_full = np.array([
        _user_to_internal(n, bounds_user[n][1]) for n in PARAM_NAMES
    ])

    # ---- apply fixes: pin theta values, remove from search space ----
    fixed_mask = np.zeros(len(PARAM_NAMES), dtype=bool)
    for name, val in fix.items():
        idx = PARAM_INDEX[name]
        theta_full[idx] = _user_to_internal(name, val)
        fixed_mask[idx] = True
    free_mask = ~fixed_mask
    free_names = [PARAM_NAMES[i] for i in range(len(PARAM_NAMES)) if free_mask[i]]
    fixed_names = [PARAM_NAMES[i] for i in range(len(PARAM_NAMES)) if fixed_mask[i]]

    if free_mask.sum() == 0:
        raise ValueError("All parameters fixed — nothing left to fit.")

    # ---- build full-theta residual, then wrap in free-theta residual ----
    res_full = make_residual_fn(
        scanner, fd_lookup, h_truth_hat, experiments,
        weights=weights, scan_speed_hat=scan_speed_hat,
        dx_hat=dx_hat, burn_in=burn_in,
        i_gain_is_log10=i_gain_is_log10,
    )

    def res_free(theta_free: np.ndarray) -> np.ndarray:
        full = theta_full.copy()
        full[free_mask] = theta_free
        return res_full(full)

    x0_free = theta_full[free_mask]
    lo_free = lo_full[free_mask]
    hi_free = hi_full[free_mask]

    sol = least_squares(
        res_free, x0_free, bounds=(lo_free, hi_free),
        method='trf', x_scale='jac',
        verbose=verbose, max_nfev=max_nfev,
    )

    # ---- reconstruct full theta and unpack ----
    theta_full[free_mask] = sol.x
    a_d, b_d, a_s, b_s, b_I_log10, log_tau_A, log_tau_z = theta_full

    return {
        'a_drive':    float(a_d),
        'b_drive':    float(b_d),
        'a_setpoint': float(a_s),
        'b_setpoint': float(b_s),
        'b_I_log10':  float(b_I_log10),
        'c_I':        float(10.0 ** b_I_log10),
        'tau_A':      float(np.exp(log_tau_A)),
        'tau_z':      float(np.exp(log_tau_z)),
        'cost':       float(sol.cost),
        'theta':      theta_full,
        'free':       free_names,
        'fixed':      fixed_names,
        'i_gain_is_log10': bool(i_gain_is_log10),
        'result':     sol,
    }


# -----------------------------------------------------------------------------
# Diagnostics
# -----------------------------------------------------------------------------

def evaluate_fit(
    fit: Dict,
    scanner,
    fd_lookup: Callable[[float], object],
    h_truth_hat: np.ndarray,
    experiments: List[Dict],
    scan_speed_hat: float = 1.0,
    dx_hat: float = 1.0,
    burn_in: int = 128,
) -> List[Dict]:
    """Re-run the simulator at the fitted theta for each experiment line.
    Useful for plotting sim-vs-exp diagnostics after calibration.
    """
    a_d, b_d = fit['a_drive'], fit['b_drive']
    a_s, b_s = fit['a_setpoint'], fit['b_setpoint']
    b_I_log10 = fit['b_I_log10']
    is_log = fit.get('i_gain_is_log10', False)
    tau_A, tau_z = fit['tau_A'], fit['tau_z']

    out = []
    for exp in experiments:
        I_exp = float(exp['I_gain'])
        I_sim = 10.0 ** (b_I_log10 + I_exp) if is_log else (10.0 ** b_I_log10) * I_exp
        sim = simulate_trace_retrace(
            scanner, fd_lookup, h_truth_hat,
            drive_nm=a_d * exp['drive'] + b_d,
            setpoint_frac=a_s * exp['setpoint'] + b_s,
            I_gain=I_sim,
            tau_A=tau_A, tau_z=tau_z,
            burn_in=burn_in,
            scan_speed_hat=scan_speed_hat, dx_hat=dx_hat,
        )
        out.append({'exp': exp, 'sim': sim})
    return out


# -----------------------------------------------------------------------------
# Example usage
# -----------------------------------------------------------------------------
if __name__ == '__main__':
    # 1) Build scanner with nominal (uncalibrated) physical params
    # scanner = ScannerFD(params=..., conversions=..., fd_model=<placeholder>)
    #
    # 2) Build an FD lookup over your library, e.g. nearest-drive selection
    #    or a SciPy interpolator that returns an FDCurve1D-like object.
    # def fd_lookup(drive_nm):
    #     return interpolate_fd_library(drive_nm)
    #
    # 3) Pack your data and metadata
    # data shape: (n_drive, n_setpoint, n_Igain, 4, n_pix)
    # channels in order: h_trace, h_retrace, phi_trace, phi_retrace
    # exps = flatten_experiments(data, drives_nm, setpoints_frac, I_gains)
    #
    # 4) Run calibration
    # fit = calibrate(scanner, fd_lookup, h_truth_hat, exps,
    #                 scan_speed_hat=..., dx_hat=...)
    # print(fit)
    #
    # 5) Inspect residuals on a held-out condition (different drive or setpoint
    #    than any in `exps`) before trusting the fit.
    pass


In [ ]:
scan_rates_hz = data1['scan_rate']
drives_nm = np.linspace(3.91, 23.47, n_drive)
setpoints = np.linspace(80, 0.01, n_set) * 1e-2
log10_I_gains = data1['i_gain']
h_truth_nm = h_truth_hat

data_traces = data1['data'][:,:,:,:,:6,::2]


In [ ]:
# data:        (9, 11, 16, 11, 6, 256)
# axis order:  (scan_rate, drive, setpoint, I_gain, channel, pixel)
# channels:    h_trace, h_retrace, phi_trace, phi_retrace, A_trace, A_retrace

# ---------- Step 1: extract FD library from the grid ----------
extracted = extract_operating_points(
    data_traces, h_truth_nm,
    scan_rates_hz, drives_nm, setpoints, log10_I_gains,
    aggregator='median',     # robust against transient pixels
    edge_margin=32,
    A_window_rel=0.10,       # keep pixels with A within 10% of setpoint*drive
    # defaults: slowest scan and largest log10_I (most settled, fastest loop)
)
lib = build_library_from_extracted(extracted, anchor_setpoint=max(setpoints))


In [ ]:
# import numpy as np
# from extract_fd_from_grid import (
#     extract_operating_points, build_library_from_extracted, collect_pixel_cloud,
# )
# from approach_curve_library import make_normalized_fd_lookup
# from calibrate_scanner import (
#     flatten_grid_experiments, subsample_stratified, calibrate, simulate_trace_retrace,
# )


# Sanity-check by overplotting library curves on raw pixel cloud:
import matplotlib.pyplot as plt
for drv in drives_nm:
    d_cloud, A_cloud, _, _ = collect_pixel_cloud(
        data_traces, h_truth_nm, drives_nm, setpoints, log10_I_gains, drive_nm=drv,
    )
    plt.scatter(d_cloud, A_cloud, s=1, alpha=0.05)
    fd = lib(drv)
    d_g = np.linspace(fd.d.min(), fd.d.max(), 200)
    plt.plot(d_g, [fd.evaluate(d)[0] for d in d_g], 'k-')
    plt.title(f'drive = {drv} nm'); plt.show()

# ---------- Step 2: build experiments list ----------
exps_full = flatten_grid_experiments(
    data_traces, scan_rates_hz, drives_nm, setpoints, log10_I_gains,
    line_period_convention='full',     # 1 Hz means trace+retrace per second
)
# 17424 lines is too many to fit on directly. Stratify so every cell is represented:
exps_fit = subsample_stratified(
    exps_full,
    keys=('drive', 'setpoint', 'I_gain', 'scan_rate_hz'),
    n_per_stratum=1,           # one record per cell -> still 17424, drop a key:
)
# Or, more aggressive — pool over scan rate at first:
exps_fit = subsample_stratified(
    exps_full, keys=('drive', 'setpoint', 'I_gain'), n_per_stratum=1,
)
# That's 11 * 16 * 11 = 1936 lines. Random-sample down further if needed:
from calibrate_scanner import subsample_random
exps_fit = subsample_random(exps_fit, n=200)

# ---------- Step 3: build fd_lookup ----------
fd_lookup = make_normalized_fd_lookup(lib, conv_L=scanner.conversion['A'])

# ---------- Step 4: stage-1 fit (LPFs fixed) ----------
fit1 = calibrate(
    scanner, fd_lookup, h_truth_nm, exps_fit,
    i_gain_is_log10=True,
    fix={'a_drive': 1.0, 'b_drive': 0.0,
         'tau_A': 1e-3, 'tau_z': 1e-3},   # locked while we sort the linears
)
# Stage 2: feed result in, unfix LPFs
fit2 = calibrate(
    scanner, fd_lookup, h_truth_nm, exps_fit,
    i_gain_is_log10=True,
    x0={k: fit1[k] for k in ('a_setpoint', 'b_setpoint', 'b_I_log10')},
    fix={'a_drive': 1.0, 'b_drive': 0.0},
)

In [ ]:
# Let's verify the optimization results

n_drive = 11
n_set = 16

DRIVES = np.linspace(3.91, 23.47, n_drive)

SETPOINTS = np.linspace(80, 0.01, n_set) * 1e-2

n_drive = 11
n_set = 16

drives = DRIVES * 1e-9
# setpoints = 0.8545345811910205 * SETPOINTS - 0.2066306487444366
setpoints = 0.5 * SETPOINTS

n_drive = len(DRIVES)
n_set = len(SETPOINTS)
exp_channels_grid = [[None for _ in range(n_set)] for _ in range(n_drive)]
np.shape(exp_channels_grid)


In [ ]:
n_skip = 4

exp_channels_grid = data1['data'][3, :, :, 5, [0, 1, 4, 5], ::n_skip]
exp_channels_grid = np.moveaxis(exp_channels_grid, 0, 2)   # (4,11,16,128) -> (11,16,4,128)

exp_channels_grid = exp_channels_grid * 1e9
exp_channels_grid -= exp_channels_grid.min(axis=-1, keepdims=True)

print(exp_channels_grid.shape)

## Load the calibration FD curves

In [ ]:
def read_fd(obj, width=None, factor=None):
    index = np.argmax(obj.data[0])
    if width is None:
        width = np.min([index, len(obj.data[0])-index])
    fwd = np.zeros([3, width])
    bkd = np.zeros([3, width])
    for i in range(3):
        fwd[i] = obj.data[i][index-width:index]
        bkd[i] = obj.data[i][index:index+width]
    if factor is not None:
        fwd[1] /= factor
        bkd[1] /= factor
    return fwd, bkd        

def find_all_sign_change_indices(arr):
    signs = np.sign(arr)  # Get sign of elements (-1, 0, or 1)
    changes = np.where(signs[:-1] != signs[1:])[0]  # Compare consecutive elements
    return changes
    
def find_thres(ph, amp, thres=80):
    ph_offset = ph - ph[0] + 90
    index = find_all_sign_change_indices(ph_offset-thres)
    if len(index) == 1:
        return amp[index][0]
    elif len(index) > 1:
        return amp[index[0]]
    else:
        return np.nan
    
def conver_unit(p, limits, reverse=False):
    '''
    Project  p from (0, 1) to (p_min, p_max).
    '''
    out = np.zeros_like(p)
    for i in range(len(p)):
        if reverse is False:
            p_min, p_max = limits[i]
            out[i] = p_min + (p_max - p_min) * p[i]
        else:
            p_min, p_max = limits[i]
            out[i] = (p[i] - p_min) / (p_max - p_min)
    return out

In [ ]:
folder = os.path.join('data', '250315', 'pickles')

fname = '250315_Cali1_MOBO training.pickle'

save = '250315_Cali1_MOBO'

import pickle

with open(os.path.join(folder, fname), 'rb') as fopen:
    param = pickle.load(fopen)

train_obj = param['y_measured_tensor']

# visualize the topograph
import aespm as ae

map_name = 'Cali_res1_0001.ibw'

folder = os.path.join('data', '250315', 'CaliSample01')

save = '250315_Cali_MOBO_01'


# import pickle

topo = ae.tools.load_ibw(os.path.join(folder, map_name))

factor = param['factor'] * topo.header['InvOLS'] * 1e9


In [ ]:
folder = os.path.join('data', '250315', 'pickles')


with open(os.path.join(folder, '250315_Cali1_MOBO.pickle'), 'rb') as fopen:
    param_clean = pickle.load(fopen)

In [ ]:
param_clean.keys()

In [ ]:
param_clean['traces'].shape

In [ ]:
param.keys()

In [ ]:
# visualize the topograph
import aespm as ae

map_name = 'Cali_res1_0001.ibw'

folder = os.path.join('data', '250315', 'CaliSample01')

save = '250315_Cali_MOBO_01'


# import pickle

topo = ae.tools.load_ibw(os.path.join(folder, map_name))

factor = param['factor'] * topo.header['InvOLS'] * 1e9

In [ ]:
folder = os.path.join('data', '250315', 'CaliSample01')

data = []
files = []

for file in os.listdir(folder):
    if file.startswith('Cali_FD'):
        files.append(os.path.join(folder, file))

files = sorted(files)

for file in files:
    data.append(ae.tools.load_ibw(file))

In [ ]:
def read_fd(obj, width=None, factor=None):
    index = np.argmax(obj.data[0])
    if width is None:
        width = np.min([index, len(obj.data[0])-index])
    fwd = np.zeros([3, width])
    bkd = np.zeros([3, width])
    for i in range(3):
        fwd[i] = obj.data[i][index-width:index]
        bkd[i] = obj.data[i][index:index+width]
    if factor is not None:
        fwd[1] /= factor
        bkd[1] /= factor
    return fwd, bkd        

def find_all_sign_change_indices(arr):
    signs = np.sign(arr)  # Get sign of elements (-1, 0, or 1)
    changes = np.where(signs[:-1] != signs[1:])[0]  # Compare consecutive elements
    return changes
    
def find_thres(ph, amp, thres=80):
    ph_offset = ph - ph[0] + 90
    index = find_all_sign_change_indices(ph_offset-thres)
    if len(index) == 1:
        return amp[index][0]
    elif len(index) > 1:
        return amp[index[0]]
    else:
        return np.nan
    
def conver_unit(p, limits, reverse=False):
    '''
    Project  p from (0, 1) to (p_min, p_max).
    '''
    out = np.zeros_like(p)
    for i in range(len(p)):
        if reverse is False:
            p_min, p_max = limits[i]
            out[i] = p_min + (p_max - p_min) * p[i]
        else:
            p_min, p_max = limits[i]
            out[i] = (p[i] - p_min) / (p_max - p_min)
    return out

In [ ]:
test = data[0]

fwd, bkd = read_fd(test, factor=None)

drive = test.header['DriveAmplitude'] * factor
print(drive)

plt.plot(fwd[1] * 1e9, fwd[2])
plt.plot(bkd[1] * 1e9, bkd[2])

In [ ]:
# test.header['FreeAirAmplitude'] / test.header['DriveAmplitude'] 

In [ ]:
# param['factor']

In [ ]:
amp_thres = find_thres(fwd[2], fwd[1], thres=85)
print(amp_thres / (drive * factor * 1e-9))
factor1 =  param['factor'] * topo.header['AmpInvOLS'] *1e9
print(factor1)

In [ ]:
plt.plot(fwd[1])

In [ ]:
fwd[1][-1] / test.header['AmpInvOLS'] 

In [ ]:
factor1

In [ ]:
fwds = []
bkds = []
drives = []
thresholds = []

x = []

for i in range(len(data)):
    temp = data[i]
    drive = temp.header['DriveAmplitude']
    drives.append(drive)
    fwd, bkd = read_fd(temp, factor=factor1)
    ending = np.min(fwd[1])
    setpoint = ending * 4
    fwds.append(fwd)
    bkds.append(bkd)
    amp_thres = find_thres(fwd[2], fwd[1], thres=85)
    thresholds.append(amp_thres / setpoint * 1e2)
    x.append(fwd[1] / setpoint * 1e2)

In [ ]:
fwd.shape

In [ ]:
thresholds

In [ ]:
np.array(drives) * factor1

In [ ]:
plt.plot(fwds[0][1])

In [ ]:
# Interpolate the thresholds as a function of the drives
from scipy.interpolate import interp1d

x_temp = np.array(drives) * factor1
y = np.array(thresholds)
y = np.nan_to_num(y)


# Create interpolation function
interp_func = interp1d(x_temp, y, kind='linear')  # Try 'linear' if issues persist

# Define a denser set of x values
x_dense = np.linspace(min(x_temp), max(x_temp), 100)

# Apply the interpolation function
y_dense = interp_func(x_dense)

# Plot the results
plt.figure(figsize=[3.,3.])
plt.scatter(y , x_temp, color='red', label="Threshold")
plt.plot(y_dense , x_dense, label="Interpolation", linestyle='--')
plt.ylabel('Drive Amplitude (nm)')
plt.xlabel('Setpoint (%)')
plt.legend()
plt.tight_layout()
plt.xlim(0, 100)

# plt.savefig('output/{} threhold curve.png'.format(save), dpi=400, bbox_inches='tight', pad_inches=0.1)


In [ ]:
# index = 1, 5, 8, 13

# colors = ['r', 'g', 'b', 'orange']
fig, ax = plt.subplots(1, 2, figsize=[9., 3])
for i in range(len(fwds)):
    ax[0].plot(x[i], fwds[i][2], label='Drive={:.0f} nm'.format(drives[i]*factor1))
    ax[1].plot(x[i], fwds[i][0] * 1e9 - 5250, label='Drive={:.0f} nm'.format(drives[i]*factor1))
    
    # plt.axvline(thresholds[ind], linestyle='--')
ax[1].legend(bbox_to_anchor=[1.1, 1])
ax[0].set_xlim(20, 100)
ax[0].set_ylim(20, 180)
ax[0].axhline(85, linestyle='--', color='gray')
ax[0].set_ylabel('Phase (degrees)')
ax[0].set_xlabel('Setpoint (%)')
ax[1].set_xlim(20, 100)
ax[1].set_ylabel('Amplitude (nm)')
ax[1].set_xlabel('Setpoint (%)')
plt.tight_layout()
# plt.savefig('output/{} threhold FD curve.png'.format(save), dpi=400, bbox_inches='tight', pad_inches=0.1)


In [ ]:
# index = 1, 5, 8, 13

# colors = ['r', 'g', 'b', 'orange']
fig, ax = plt.subplots(1, 2, figsize=[9., 3])
for i in range(len(fwds)):
    ax[0].plot(x[i], fwds[i][2], label='Drive={:.0f} nm'.format(drives[i]*factor1))
    ax[1].plot(x[i], fwds[i][1], label='Drive={:.0f} nm'.format(drives[i]*factor1))
    
    # plt.axvline(thresholds[ind], linestyle='--')
ax[1].legend(bbox_to_anchor=[1.1, 1])
ax[0].set_xlim(20, 100)
ax[0].set_ylim(20, 180)
ax[0].axhline(85, linestyle='--', color='gray')
ax[0].set_ylabel('Phase (degrees)')
ax[0].set_xlabel('Setpoint (%)')
ax[1].set_xlim(20, 100)
ax[1].set_ylabel('Amplitude (nm)')
ax[1].set_xlabel('Setpoint (%)')
plt.tight_layout()
# plt.savefig('output/{} threhold FD curve.png'.format(save), dpi=400, bbox_inches='tight', pad_inches=0.1)


In [ ]:
for i in range(len(fwds)):
    print(fwds[i][1][-1] / fwds[i][1][0])

In [ ]:
index = 1, 5, 8, 13

colors = ['r', 'g', 'b', 'orange']
plt.figure(figsize=[4., 3])
for i, ind in enumerate(index):
    plt.plot(x[ind], fwds[ind][2], color=colors[i], label='Drive={:.0f} nm'.format(drives[ind]*factor1))
    # plt.axvline(thresholds[ind], linestyle='--')
plt.legend(loc='upper right')
plt.xlim(20, 100)
plt.ylim(20, 180)
plt.axhline(85, linestyle='--', color='gray')
plt.ylabel('Phase (degrees)')
plt.xlabel('Setpoint (%)')
# plt.savefig('output/{} threhold FD curve.png'.format(save), dpi=400, bbox_inches='tight', pad_inches=0.1)


In [ ]:
traces = param_clean['traces']
traces.shape

In [ ]:
plt.plot(traces[28, 3, 1])

## Build approach curve library

In [ ]:
"""
Approach-curve library + cross-drive interpolation.

Strategy
--------
Every measured approach curve is single-valued in z when parameterized by
the slow z-piezo. Store curves in physical units (nm, deg) per drive A0.

To interpolate to an arbitrary drive A0*:
  1) Resample each curve onto a common normalized grid u = d / A0,
     storing the normalized amplitude a = A / A0 and the phase phi.
  2) Linearly interpolate a(u, A0) and phi(u, A0) in A0 at fixed u.
  3) Convert back: d = u * A0*,  A = a * A0*.

Optional force F(d) is carried through if provided.

Public API
----------
FDCurve1D                 : single-curve interpolator with evaluate(d) -> (A, phi, F)
ApproachCurveLibrary      : add curves, query at any drive, returns FDCurve1D
make_normalized_fd_lookup : adapter that wraps the library in the normalized-units
                            API expected by ScannerFD (matches calibrate_scanner.py)
detect_bistability        : utility to flag curves with phase jumps before fitting
"""
from __future__ import annotations

import numpy as np
from scipy.interpolate import interp1d
from typing import Optional, Tuple, Dict, List, Callable


# -----------------------------------------------------------------------------
# Single-curve evaluator
# -----------------------------------------------------------------------------

class FDCurve1D:
    """Interpolated A(d), phi(d), F(d) at a single drive amplitude.

    Conventions
    -----------
    Inputs in any consistent unit system (typically nm, degrees).
    d increases away from the surface; the largest d in the curve is treated
    as the "free" point, A(d_max) ~ A0.

    Out-of-range behavior
    ---------------------
    d > d_max : A clamped to A0 (free), phi clamped to phi(d_max), F -> 0.
    d < d_min : edge-clamped (no extrapolation into untrusted near-contact regime).
    """

    def __init__(self, d, A, phi, F=None, *, A0: Optional[float] = None,
                 kind: str = 'linear'):
        d = np.asarray(d, dtype=float)
        A = np.asarray(A, dtype=float)
        phi = np.asarray(phi, dtype=float)
        F = np.zeros_like(d) if F is None else np.asarray(F, dtype=float)

        order = np.argsort(d)
        self.d   = d[order]
        self.A   = A[order]
        self.phi = phi[order]
        self.F   = F[order]
        self.A0 = float(self.A[-1]) if A0 is None else float(A0)
        self.kind = kind

        self._A = interp1d(
            self.d, self.A, kind=kind, bounds_error=False,
            fill_value=(self.A[0], self.A0),  # left = nearest, right = free
        )
        self._phi = interp1d(
            self.d, self.phi, kind=kind, bounds_error=False,
            fill_value=(self.phi[0], self.phi[-1]),
        )
        self._F = interp1d(
            self.d, self.F, kind=kind, bounds_error=False,
            fill_value=(self.F[0], 0.0),
        )

    def evaluate(self, d):
        """Return (A, phi, F). Scalar in -> scalar out; array in -> array out."""
        d_arr = np.atleast_1d(d).astype(float)
        A   = self._A(d_arr)
        phi = self._phi(d_arr)
        F   = self._F(d_arr)
        if np.isscalar(d):
            return float(A[0]), float(phi[0]), float(F[0])
        return A, phi, F


# -----------------------------------------------------------------------------
# Library
# -----------------------------------------------------------------------------

class ApproachCurveLibrary:
    """Collection of approach curves at multiple drive amplitudes (nm).

    Each curve is stored as 1D arrays (d_nm, A_nm, phi_deg, optional F).
    The curves do NOT need to share a common d-grid or have the same length.
    """

    def __init__(self):
        self._curves: Dict[float, Dict[str, Optional[np.ndarray]]] = {}

    # ----- ingestion -----

    def add(self, drive_nm: float, d_nm, A_nm, phi_deg, F=None) -> None:
        """Add one approach curve at a known drive amplitude (in nm).

        Parameters
        ----------
        drive_nm : free-amplitude drive setting for this curve, nm.
        d_nm     : 1D array of z-piezo positions, nm. Increases away from surface.
        A_nm     : 1D array of measured oscillation amplitudes, nm.
        phi_deg  : 1D array of measured phases, degrees.
        F        : optional 1D array (e.g. peak force or virial), arbitrary units.
        """
        d = np.asarray(d_nm,    dtype=float).ravel()
        A = np.asarray(A_nm,    dtype=float).ravel()
        phi = np.asarray(phi_deg, dtype=float).ravel()
        if not (d.shape == A.shape == phi.shape):
            raise ValueError("d, A, phi must be the same length")
        F_arr = None
        if F is not None:
            F_arr = np.asarray(F, dtype=float).ravel()
            if F_arr.shape != d.shape:
                raise ValueError("F must match d length")

        # Drop NaNs and sort by d ascending
        mask = np.isfinite(d) & np.isfinite(A) & np.isfinite(phi)
        if F_arr is not None:
            mask &= np.isfinite(F_arr)
        d, A, phi = d[mask], A[mask], phi[mask]
        if F_arr is not None:
            F_arr = F_arr[mask]
        order = np.argsort(d)

        self._curves[float(drive_nm)] = {
            'd':   d[order],
            'A':   A[order],
            'phi': phi[order],
            'F':   None if F_arr is None else F_arr[order],
        }

    @classmethod
    def from_dict(cls, by_drive: Dict[float, Dict[str, np.ndarray]]) -> 'ApproachCurveLibrary':
        """Bulk-construct from {drive_nm: {'d':..., 'A':..., 'phi':..., 'F':...}}."""
        lib = cls()
        for drv, c in by_drive.items():
            lib.add(drv, c['d'], c['A'], c['phi'], F=c.get('F'))
        return lib

    # ----- inspection -----

    @property
    def drives(self) -> np.ndarray:
        return np.array(sorted(self._curves.keys()))

    def get_raw(self, drive_nm: float) -> Dict[str, Optional[np.ndarray]]:
        return self._curves[float(drive_nm)]

    # ----- normalization -----

    def normalized_grid(
        self,
        n_u: int = 400,
        u_min: Optional[float] = None,
        u_max: Optional[float] = None,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """Resample all curves onto a common u = d/A0 grid.

        Returns
        -------
        drives   : (n_drives,)        sorted drive amplitudes [nm]
        u_grid   : (n_u,)             common normalized distance grid
        a_grid   : (n_drives, n_u)    normalized amplitude  A/A0
        phi_grid : (n_drives, n_u)    phase [deg]
        F_grid   : (n_drives, n_u)    force (zero where F not supplied)
        """
        drives = self.drives
        if drives.size < 2:
            raise ValueError("Library needs at least two drives to interpolate.")
        umins = [self._curves[d]['d'].min() / d for d in drives]
        umaxs = [self._curves[d]['d'].max() / d for d in drives]

        u_min = max(umins) if u_min is None else float(u_min)
        u_max = min(umaxs) if u_max is None else float(u_max)
        if not (u_max > u_min):
            raise ValueError(
                f"No common u-range across curves "
                f"(largest u_min={max(umins):.3g}, smallest u_max={min(umaxs):.3g}). "
                "Trim outlier curves or pass u_min/u_max manually."
            )
        u_grid = np.linspace(u_min, u_max, int(n_u))

        nD = drives.size
        a_grid   = np.empty((nD, n_u))
        phi_grid = np.empty((nD, n_u))
        F_grid   = np.zeros((nD, n_u))

        for i, drv in enumerate(drives):
            c = self._curves[drv]
            u = c['d'] / drv
            a = c['A'] / drv
            a_grid[i]   = np.interp(u_grid, u, a)
            phi_grid[i] = np.interp(u_grid, u, c['phi'])
            if c['F'] is not None:
                F_grid[i] = np.interp(u_grid, u, c['F'])

        return drives, u_grid, a_grid, phi_grid, F_grid

    # ----- interpolation across drives -----

    def fd_at_drive(self, drive_target_nm: float, n_u: int = 400) -> FDCurve1D:
        """Build an FDCurve1D for an arbitrary target drive amplitude (nm).

        Linear interpolation in (u, A0). Outside [drives.min, drives.max],
        clamps to the nearest curve (no extrapolation across drive).
        """
        drives, u_grid, a_grid, phi_grid, F_grid = self.normalized_grid(n_u=n_u)

        if drive_target_nm <= drives[0]:
            a_t, phi_t, F_t = a_grid[0], phi_grid[0], F_grid[0]
        elif drive_target_nm >= drives[-1]:
            a_t, phi_t, F_t = a_grid[-1], phi_grid[-1], F_grid[-1]
        else:
            idx = int(np.searchsorted(drives, drive_target_nm) - 1)
            d_lo, d_hi = drives[idx], drives[idx + 1]
            w = (drive_target_nm - d_lo) / (d_hi - d_lo)
            a_t   = (1 - w) * a_grid[idx]   + w * a_grid[idx + 1]
            phi_t = (1 - w) * phi_grid[idx] + w * phi_grid[idx + 1]
            F_t   = (1 - w) * F_grid[idx]   + w * F_grid[idx + 1]

        # Convert back to physical d, A
        d_phys = u_grid * float(drive_target_nm)
        A_phys = a_t   * float(drive_target_nm)
        return FDCurve1D(d_phys, A_phys, phi_t, F=F_t, A0=float(drive_target_nm))

    # convenience alias
    def __call__(self, drive_target_nm: float) -> FDCurve1D:
        return self.fd_at_drive(drive_target_nm)


# -----------------------------------------------------------------------------
# Adapter to the normalized-units API expected by ScannerFD
# -----------------------------------------------------------------------------

def make_normalized_fd_lookup(
    library: ApproachCurveLibrary,
    conv_L: float,
    conv_A: Optional[float] = None,
    conv_F: float = 1.0,
    n_u: int = 400,
) -> Callable[[float], FDCurve1D]:
    """Wrap a physical-units library in a callable returning FDCurve1D in
    the *normalized* units the ScannerFD expects internally.

    Parameters
    ----------
    library : ApproachCurveLibrary in physical units (nm, deg).
    conv_L  : length conversion (d_norm = d_nm * conv_L). Same as scanner.conversion['L'].
    conv_A  : amplitude conversion (A_norm = A_nm * conv_A). Defaults to conv_L.
    conv_F  : force conversion (use scanner.conversion['F'] if you have one).
    n_u     : grid size for the normalized resampling step.

    The returned callable takes a drive in *physical* nm and returns an
    FDCurve1D whose evaluate(d_hat) accepts and returns *normalized* values.
    Use this directly as `fd_lookup` in calibrate_scanner.calibrate(...).
    """
    if conv_A is None:
        conv_A = conv_L

    def lookup(drive_nm: float) -> FDCurve1D:
        fd_phys = library.fd_at_drive(float(drive_nm), n_u=n_u)
        return FDCurve1D(
            d=fd_phys.d * conv_L,
            A=fd_phys.A * conv_A,
            phi=fd_phys.phi,
            F=fd_phys.F * conv_F,
            A0=float(drive_nm) * conv_A,
        )
    return lookup


# -----------------------------------------------------------------------------
# Diagnostics
# -----------------------------------------------------------------------------

def detect_bistability(
    library: ApproachCurveLibrary,
    phase_jump_threshold_deg: float = 25.0,
    amp_jump_threshold_rel: float = 0.15,
) -> Dict[float, Dict[str, float]]:
    """Flag curves where phi or A/A0 jumps more than thresholds between
    adjacent samples in d. These are likely to be bistable and should be
    inspected (or split into branches) before being put into the library.

    Returns {drive: {'max_dphi_deg': ..., 'max_da_rel': ..., 'd_at_jump': ...}}
    only for curves that exceed at least one threshold.
    """
    flags: Dict[float, Dict[str, float]] = {}
    for drv in library.drives:
        c = library.get_raw(drv)
        d, A, phi = c['d'], c['A'], c['phi']
        if d.size < 3:
            continue
        dphi = np.abs(np.diff(phi))
        da_rel = np.abs(np.diff(A)) / float(drv)
        i_phi = int(np.argmax(dphi)) if dphi.size else 0
        i_a   = int(np.argmax(da_rel)) if da_rel.size else 0
        if dphi[i_phi] > phase_jump_threshold_deg or da_rel[i_a] > amp_jump_threshold_rel:
            phi_severity = dphi[i_phi] / phase_jump_threshold_deg
            a_severity   = da_rel[i_a] / amp_jump_threshold_rel
            i_jump = i_phi if phi_severity > a_severity else i_a
            flags[float(drv)] = {
                'max_dphi_deg': float(dphi.max()),
                'max_da_rel':   float(da_rel.max()),
                'd_at_jump':    float(d[i_jump]),
            }
    return flags




In [ ]:
# -----------------------------------------------------------------------------
# Example
# -----------------------------------------------------------------------------
if __name__ == '__main__':
    # Suppose you measured 15 approach curves: drive in nm, then 1D arrays
    # of (z_piezo_nm, A_nm, phi_deg) per drive.
    #
    # measured = {
    #     2.0:  {'d': z_arr_2,   'A': A_arr_2,   'phi': phi_arr_2},
    #     8.0:  {...},
    #     ...
    #     115.0:{...},
    # }
    #
    # lib = ApproachCurveLibrary.from_dict(measured)
    #
    # # quick sanity check
    # bist = detect_bistability(lib)
    # for drv, info in bist.items():
    #     print(f"Drive {drv} nm: jump near d={info['d_at_jump']:.2f} nm, "
    #           f"max d-phi={info['max_dphi_deg']:.1f} deg")
    #
    # # query at any drive
    # fd_phys = lib(40.0)               # FDCurve1D in nm/deg
    # A, phi, F = fd_phys.evaluate(2.5) # at d = 2.5 nm
    #
    # # plug into calibrate_scanner
    # from calibrate_scanner import calibrate
    # fd_lookup = make_normalized_fd_lookup(
    #     lib,
    #     conv_L=scanner.conversion['L'],
    #     conv_A=scanner.conversion['A'],
    # )
    # fit = calibrate(scanner, fd_lookup, h_truth_hat, exps)
    pass


## Calibrate the scanner

In [ ]:
"""
Calibrate a ScannerFD digital twin against experimental scan lines.

Free parameters fit (theta, length 6):
    a_drive, b_drive          drive_sim    = a_drive * drive_exp    + b_drive
    a_setpoint, b_setpoint    setpoint_sim = a_setpoint * setpoint_exp + b_setpoint
    tau_A                     amplitude / phase demodulator time constant (1st-order LPF)
    tau_z                     Z-actuator time constant (1st-order LPF)

Held fixed per scan line:
    I_gain                    taken from each experiment record
    P_gain                    0 by default; expose as an extra theta entry if needed

Inputs the user supplies:
    scanner       : a ScannerFD instance configured with nominal physical params
    fd_lookup(A)  : callable returning an FDCurve1D for a given drive amplitude (nm)
                    (interpolate your approach-curve library here)
    h_truth_hat   : 1D ground-truth topography (already in the normalized length
                    units used by the scanner, i.e. h * conversion['L'])
    experiments   : list of dicts (one per scan line). Use flatten_experiments(...)
                    to build it from a packed (D, S, I, 4, P) array.
"""
from __future__ import annotations

import numpy as np
from scipy.optimize import least_squares
from typing import Callable, Iterable, List, Dict, Optional, Tuple


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def wrap_phase_diff(phi_a_deg, phi_b_deg) -> np.ndarray:
    """Phase difference (a - b) wrapped to (-180, 180] degrees."""
    d = np.asarray(phi_a_deg, dtype=float) - np.asarray(phi_b_deg, dtype=float)
    return (d + 180.0) % 360.0 - 180.0


def set_scanner_drive(scanner, A_nm: float, fd_model) -> None:
    """Switch scanner free amplitude and FD lookup in place."""
    scanner.param['A'] = float(A_nm)
    scanner.param_norm['A'] = float(A_nm) * scanner.conversion['A']
    scanner.param_norm['C3'] = scanner.param_norm['A'] / scanner.param_norm['Q']
    scanner.fd_model = fd_model


# -----------------------------------------------------------------------------
# Forward model: one simulated line, both directions
# -----------------------------------------------------------------------------

def simulate_trace_retrace(
    scanner,
    fd_lookup: Callable[[float], object],
    h_truth_hat: np.ndarray,
    drive_nm: float,
    setpoint_frac: float,
    I_gain: float,
    tau_A: float,
    tau_z: float,
    P_gain: float = 0.0,
    tau_phi: Optional[float] = None,
    burn_in: int = 128,
    scan_speed_hat: float = 1.0,
    dx_hat: float = 1.0,
) -> Dict[str, np.ndarray]:
    """Simulate one scan line forward (trace) and backward (retrace).

    Returns a dict with h_{trace,retrace}, A_{trace,retrace}, phi_{trace,retrace},
    each of length len(h_truth_hat). The retrace arrays are returned in the
    same x-order as the trace (i.e. left-to-right) for direct comparison.
    """
    fd_model = fd_lookup(drive_nm)
    set_scanner_drive(scanner, drive_nm, fd_model)

    if tau_phi is None:
        tau_phi = tau_A

    common = dict(
        setpoint=setpoint_frac,
        scan_speed_hat=scan_speed_hat,
        dx_hat=dx_hat,
        P=P_gain,
        I=I_gain,
        A_filter_order=1, tau_A=tau_A,
        phi_filter_order=1, tau_phi=tau_phi,
        z_filter_order=1, tau_z=tau_z,
    )

    # --- Trace: pad with first sample, scan left-to-right, drop the burn-in ---
    pad_t = np.full(burn_in, h_truth_hat[0], dtype=float)
    h_in_t = np.concatenate([pad_t, h_truth_hat])
    out_t = scanner.scan_line_pi(h_in_t, **common)

    h_trace   = out_t['d_hat'][burn_in:]
    A_trace   = out_t['A_hat'][burn_in:]
    phi_trace = out_t['phi_deg'][burn_in:]

    # --- Retrace: scan reversed h, carry controller state from end of trace ---
    pad_r = np.full(burn_in, h_truth_hat[-1], dtype=float)
    h_in_r = np.concatenate([pad_r, h_truth_hat[::-1]])
    out_r = scanner.scan_line_pi(
        h_in_r,
        d_cmd_init_hat=out_t['d_cmd_final_hat'],
        d_act_init_hat=out_t['d_final_hat'],
        A_meas_init_hat=out_t['A_meas_final_hat'],
        phi_meas_init_deg=out_t['phi_meas_final_deg'],
        integ_init=out_t['integ_final'],
        **common,
    )
    # Drop burn-in, then reverse so x-order matches the trace.
    h_retrace   = out_r['d_hat'][burn_in:][::-1]
    A_retrace   = out_r['A_hat'][burn_in:][::-1]
    phi_retrace = out_r['phi_deg'][burn_in:][::-1]

    return dict(
        h_trace=h_trace,     h_retrace=h_retrace,
        A_trace=A_trace,     A_retrace=A_retrace,
        phi_trace=phi_trace, phi_retrace=phi_retrace,
    )


# -----------------------------------------------------------------------------
# Pack / unpack experimental data
# -----------------------------------------------------------------------------

def flatten_experiments(
    data: np.ndarray,
    drives: np.ndarray,
    setpoints: np.ndarray,
    I_gains: np.ndarray,
    channel_order: Tuple[str, ...] = (
        'h_trace', 'h_retrace', 'phi_trace', 'phi_retrace'
    ),
) -> List[Dict]:
    """Convert a packed (n_drive, n_setpoint, n_Igain, n_channel, n_pix) array
    into a flat list of per-line dicts.

    The default channel_order matches your description "trace and retrace of
    heights + phases" interpreted as (h_trace, h_retrace, phi_trace, phi_retrace).
    Override channel_order if your packing differs.
    """
    data = np.asarray(data)
    nD, nS, nI, nC, nP = data.shape
    if nC != len(channel_order):
        raise ValueError(f"Expected {len(channel_order)} channels, got {nC}")
    if not (len(drives) == nD and len(setpoints) == nS and len(I_gains) == nI):
        raise ValueError("drives / setpoints / I_gains lengths must match data shape")

    rows: List[Dict] = []
    for i in range(nD):
        for j in range(nS):
            for k in range(nI):
                line = {ch: np.asarray(data[i, j, k, ci, :], dtype=float)
                        for ci, ch in enumerate(channel_order)}
                rows.append({
                    'drive':    float(drives[i]),
                    'setpoint': float(setpoints[j]),
                    'I_gain':   float(I_gains[k]),
                    **line,
                })
    return rows


# -----------------------------------------------------------------------------
# Cost function
# -----------------------------------------------------------------------------

def make_residual_fn(
    scanner,
    fd_lookup: Callable[[float], object],
    h_truth_hat: np.ndarray,
    experiments: List[Dict],
    weights: Optional[Dict[str, float]] = None,
    scan_speed_hat: float = 1.0,
    dx_hat: float = 1.0,
    burn_in: int = 128,
    huge_penalty: float = 1e6,
):
    """Build a residual vector: stacked (h_trace, h_retrace, phi_trace, phi_retrace)
    differences across all experimental lines. Phase residuals are wrapped.

    Default weights make 1 deg of phase ~ 0.01 normalized length unit.
    """
    if weights is None:
        weights = {'h': 1.0, 'phi': 0.01}

    h_truth_hat = np.asarray(h_truth_hat, dtype=float)
    n_pix = h_truth_hat.size

    def residual(theta: np.ndarray) -> np.ndarray:
        a_d, b_d, a_s, b_s, log_tau_A, log_tau_z = theta
        tau_A = float(np.exp(log_tau_A))
        tau_z = float(np.exp(log_tau_z))

        chunks: List[np.ndarray] = []
        for exp in experiments:
            drive_sim    = a_d * exp['drive']    + b_d
            setpoint_sim = a_s * exp['setpoint'] + b_s

            # Refuse pathological setpoints early (saves a sim call)
            if not (0.01 < setpoint_sim < 0.999) or drive_sim <= 0.0:
                chunks.append(np.full(4 * n_pix, huge_penalty))
                continue

            try:
                sim = simulate_trace_retrace(
                    scanner, fd_lookup, h_truth_hat,
                    drive_nm=float(drive_sim),
                    setpoint_frac=float(setpoint_sim),
                    I_gain=float(exp['I_gain']),
                    tau_A=tau_A, tau_z=tau_z,
                    burn_in=burn_in,
                    scan_speed_hat=scan_speed_hat,
                    dx_hat=dx_hat,
                )
            except Exception:
                # FD lookup outside library, NaN on a branch jump, etc.
                chunks.append(np.full(4 * n_pix, huge_penalty))
                continue

            r_h_t = (sim['h_trace']   - exp['h_trace'])   * weights['h']
            r_h_r = (sim['h_retrace'] - exp['h_retrace']) * weights['h']
            r_p_t = wrap_phase_diff(sim['phi_trace'],   exp['phi_trace'])   * weights['phi']
            r_p_r = wrap_phase_diff(sim['phi_retrace'], exp['phi_retrace']) * weights['phi']
            chunks.extend([r_h_t, r_h_r, r_p_t, r_p_r])

        return np.concatenate(chunks)

    return residual


# -----------------------------------------------------------------------------
# Top-level driver
# -----------------------------------------------------------------------------

def calibrate(
    scanner,
    fd_lookup: Callable[[float], object],
    h_truth_hat: np.ndarray,
    experiments: List[Dict],
    x0: Optional[np.ndarray] = None,
    bounds: Optional[Tuple[Iterable[float], Iterable[float]]] = None,
    weights: Optional[Dict[str, float]] = None,
    scan_speed_hat: float = 1.0,
    dx_hat: float = 1.0,
    burn_in: int = 128,
    verbose: int = 2,
    max_nfev: int = 200,
) -> Dict:
    """Fit (a_drive, b_drive, a_setpoint, b_setpoint, tau_A, tau_z).

    h_truth_hat : 1D ground-truth profile in the same normalized length units
                  the scanner uses internally (i.e. physical_h * conversion['L']).
    experiments : list of dicts with keys
                  drive (nm, in EXPERIMENTAL units), setpoint (fraction, in
                  EXPERIMENTAL units), I_gain, h_trace, h_retrace,
                  phi_trace, phi_retrace.
    fd_lookup   : drive_nm_simulator -> FDCurve1D (interpolate your library).

    Returns a dict with the fitted parameters and the scipy result object.
    """
    if x0 is None:
        # Identity transforms; tau_A, tau_z log-space initial guesses.
        x0 = np.array([1.0, 0.0, 1.0, 0.0, np.log(1e-3), np.log(1e-4)])

    if bounds is None:
        # Generous but bounded; tighten if you have priors on calibration error.
        lo = [0.5, -20.0, 0.7, -0.20, np.log(1e-7), np.log(1e-7)]
        hi = [2.0,  20.0, 1.5,  0.20, np.log(1e0),  np.log(1e0)]
        bounds = (lo, hi)

    res_fn = make_residual_fn(
        scanner, fd_lookup, h_truth_hat, experiments,
        weights=weights, scan_speed_hat=scan_speed_hat,
        dx_hat=dx_hat, burn_in=burn_in,
    )

    sol = least_squares(
        res_fn, x0, bounds=bounds,
        method='trf', x_scale='jac',
        verbose=verbose, max_nfev=max_nfev,
    )

    a_d, b_d, a_s, b_s, log_tau_A, log_tau_z = sol.x
    return {
        'a_drive':    float(a_d),
        'b_drive':    float(b_d),
        'a_setpoint': float(a_s),
        'b_setpoint': float(b_s),
        'tau_A':      float(np.exp(log_tau_A)),
        'tau_z':      float(np.exp(log_tau_z)),
        'cost':       float(sol.cost),
        'theta':      sol.x,
        'result':     sol,
    }


# -----------------------------------------------------------------------------
# Diagnostics
# -----------------------------------------------------------------------------

def evaluate_fit(
    fit: Dict,
    scanner,
    fd_lookup: Callable[[float], object],
    h_truth_hat: np.ndarray,
    experiments: List[Dict],
    scan_speed_hat: float = 1.0,
    dx_hat: float = 1.0,
    burn_in: int = 128,
) -> List[Dict]:
    """Re-run the simulator at the fitted theta for each experiment line.
    Useful for plotting sim-vs-exp diagnostics after calibration.
    """
    a_d, b_d = fit['a_drive'], fit['b_drive']
    a_s, b_s = fit['a_setpoint'], fit['b_setpoint']
    tau_A, tau_z = fit['tau_A'], fit['tau_z']

    out = []
    for exp in experiments:
        sim = simulate_trace_retrace(
            scanner, fd_lookup, h_truth_hat,
            drive_nm=a_d * exp['drive'] + b_d,
            setpoint_frac=a_s * exp['setpoint'] + b_s,
            I_gain=exp['I_gain'],
            tau_A=tau_A, tau_z=tau_z,
            burn_in=burn_in,
            scan_speed_hat=scan_speed_hat, dx_hat=dx_hat,
        )
        out.append({'exp': exp, 'sim': sim})
    return out




In [ ]:
# -----------------------------------------------------------------------------
# Example usage
# -----------------------------------------------------------------------------
if __name__ == '__main__':
    # 1) Build scanner with nominal (uncalibrated) physical params
    # scanner = ScannerFD(params=..., conversions=..., fd_model=<placeholder>)
    #
    # 2) Build an FD lookup over your library, e.g. nearest-drive selection
    #    or a SciPy interpolator that returns an FDCurve1D-like object.
    # def fd_lookup(drive_nm):
    #     return interpolate_fd_library(drive_nm)
    #
    # 3) Pack your data and metadata
    # data shape: (n_drive, n_setpoint, n_Igain, 4, n_pix)
    # channels in order: h_trace, h_retrace, phi_trace, phi_retrace
    # exps = flatten_experiments(data, drives_nm, setpoints_frac, I_gains)
    #
    # 4) Run calibration
    # fit = calibrate(scanner, fd_lookup, h_truth_hat, exps,
    #                 scan_speed_hat=..., dx_hat=...)
    # print(fit)
    #
    # 5) Inspect residuals on a held-out condition (different drive or setpoint
    #    than any in `exps`) before trusting the fit.
    pass


## FD calibration codes

In [ ]:
"""
fd_only_calibration.py
======================

FD-only calibration for the SPM digital twin.

Inputs are a set of *experimental* point-FD curves taken at different
drive amplitudes.  Each curve is a 1D array (length N, e.g. 1007) of
amplitude and phase, optionally with z-piezo position.  The "setpoint
axis" used for fitting is the normalized amplitude:

        setpoint_pct  =  A_meas / A_free * 100

This avoids needing absolute z-piezo calibration and matches how the
attractive-to-repulsive transition is usually plotted (see Drive vs
Threshold-Setpoint plot).

The calibrator runs the existing RK45 approach-curve simulator from
`spm_fd_multidrive_alignment.py`, reparameterizes the simulated FD
curve onto the same setpoint axis, and minimises a weighted loss made
of three terms:

    L = w_phase  * MSE( phi_sim(setpoint) - phi_exp(setpoint) )
      + w_free   * (phi_sim_free - phi_exp_free)^2
      + w_thresh * (sp_thresh_sim - sp_thresh_exp)^2

Parameters that can be fit (any subset, controlled by FDCalibrationConfig.fit_keys):

    Material:   E_star, H, a0, R
    Cantilever: Q, k
    Aux:        phase_offset_deg, drive_scale

Recommended first pass:  fit ("E_star", "H", "a0", "phase_offset_deg")
with Q locked (use a thermal-tune Q) and drive_scale = 1.

Author: digital-twin alignment helper.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, minimize

# # Re-use the existing simulator. Adjust the import path if needed.
# from spm_fd_multidrive_alignment import (
#     NormalizedSPMParams,
#     RK45Options,
#     SPMRK45ApproachModel,
#     build_normalized_spm_params,
# )


# ============================================================
# 1) DATA CONTAINER
# ============================================================
@dataclass
class FDCalibrationCurve:
    """One experimental FD curve at a fixed drive amplitude.

    The fitter only needs (amplitude_nm, phase_deg) and the drive in SI.
    Setpoint% is auto-computed from amplitude / A_free, and the threshold
    setpoint (attractive→repulsive transition) is auto-detected from a
    sharp downward jump in phase across `threshold_phase_deg` if not
    supplied explicitly.

    All arrays must be 1-D and the same length.
    """
    drive_si: float                                  # m, drive amplitude at the cantilever
    amplitude_nm: np.ndarray                         # nm
    phase_deg: np.ndarray                            # deg (in [0, 180] convention)
    z_piezo_nm: Optional[np.ndarray] = None          # optional, z-piezo position (nm)
    setpoint_pct: Optional[np.ndarray] = None        # optional, default = A/A_free*100
    threshold_setpoint_pct: Optional[float] = None   # optional; auto-detected if None
    A_free_nm: Optional[float] = None                # optional; auto = high-percentile of A
    free_phase_deg: Optional[float] = None           # optional; auto = mean phase at high SP
    label: str = ""

    def __post_init__(self):
        self.amplitude_nm = np.asarray(self.amplitude_nm, dtype=float).ravel()
        self.phase_deg = np.asarray(self.phase_deg, dtype=float).ravel()
        if self.amplitude_nm.shape != self.phase_deg.shape:
            raise ValueError("amplitude_nm and phase_deg must have the same shape")
        if self.z_piezo_nm is not None:
            self.z_piezo_nm = np.asarray(self.z_piezo_nm, dtype=float).ravel()
        if self.A_free_nm is None:
            # robust free-amplitude estimate: mean of top 5% of amplitude
            top = np.percentile(self.amplitude_nm, 95.0)
            self.A_free_nm = float(np.mean(self.amplitude_nm[self.amplitude_nm >= top]))
        if self.A_free_nm <= 0:
            raise ValueError(f"Could not estimate A_free_nm for drive={self.drive_si}")
        if self.setpoint_pct is None:
            self.setpoint_pct = self.amplitude_nm / self.A_free_nm * 100.0
        else:
            self.setpoint_pct = np.asarray(self.setpoint_pct, dtype=float).ravel()
        if self.free_phase_deg is None:
            top_sp = np.percentile(self.setpoint_pct, 95.0)
            mask = self.setpoint_pct >= top_sp
            if np.any(mask):
                self.free_phase_deg = float(np.mean(self.phase_deg[mask]))
            else:
                self.free_phase_deg = float(np.mean(self.phase_deg))


# ============================================================
# 2) TRANSITION DETECTION
# ============================================================
def detect_threshold_setpoint(
    setpoint_pct: np.ndarray,
    phase_deg: np.ndarray,
    threshold_phase_deg: float = 90.0,
    jump_min_deg: float = 15.0,
    smooth_window: int = 3,
) -> Optional[float]:
    """
    Find the setpoint at which phase jumps downward across `threshold_phase_deg`.

    Walks the curve in approach order (descending setpoint) and looks for
    a discrete drop of at least `jump_min_deg` over `smooth_window` samples
    that crosses `threshold_phase_deg`.  Returns the linearly interpolated
    setpoint at the crossing, or None if no clear jump exists.
    """
    sp = np.asarray(setpoint_pct, dtype=float)
    ph = np.asarray(phase_deg, dtype=float)
    order = np.argsort(-sp)
    sp_o = sp[order]
    ph_o = ph[order]
    w = max(1, int(smooth_window))
    if ph_o.size < w + 1:
        return None
    diff = ph_o[w:] - ph_o[:-w]
    crossings = np.where(
        (ph_o[:-w] >= threshold_phase_deg)
        & (ph_o[w:] < threshold_phase_deg)
        & (diff <= -jump_min_deg)
    )[0]
    if crossings.size == 0:
        return None
    i = int(crossings[0])
    a = ph_o[i]
    b = ph_o[i + w]
    if b == a:
        return float(0.5 * (sp_o[i] + sp_o[i + w]))
    t = (threshold_phase_deg - a) / (b - a)
    t = float(np.clip(t, 0.0, 1.0))
    return float(sp_o[i] + t * (sp_o[i + w] - sp_o[i]))


# ============================================================
# 3) FIT CONFIG
# ============================================================
@dataclass
class FDCalibrationConfig:
    # Which keys to fit. Anything not listed is held at base_params value.
    fit_keys: Tuple[str, ...] = ("E_star", "H", "a0", "phase_offset_deg")

    # Loss weights
    phase_weight: float = 1.0
    free_phase_weight: float = 2.0
    threshold_weight: float = 6.0

    # Region of phase curve to compare. Avoids the bistable jump itself
    # (handled by the threshold term) and the noisy pre-engagement tail.
    setpoint_min_pct: float = 30.0
    setpoint_max_pct: float = 92.0

    # Threshold detection on experimental + simulated curves.
    threshold_phase_deg: float = 90.0
    jump_min_deg: float = 15.0
    threshold_smooth_window: int = 3

    # Branch selection for sim → exp interpolation
    # "auto"        : sort by setpoint and interp (works above threshold)
    # "attractive"  : keep only the high-d_hat branch (before transition)
    # "repulsive"   : keep only the low-d_hat branch (after transition)
    interp_branch: str = "auto"

    # Simulator grid
    n_d_hat_grid: int = 160
    d_end_hat: float = -8.0
    d_start_factor: float = 1.25  # d_start_hat = drive_hat * d_start_factor

    # Numerics
    bistable_jump_penalty: float = 100.0   # penalty if exp shows a jump but sim doesn't


# ============================================================
# 4) FD-ONLY CALIBRATOR
# ============================================================
class FDOnlyCalibrator:
    """
    Hierarchical FD-only calibration:
        1. Auto-detect threshold setpoints in experimental curves.
        2. Forward-simulate each FD curve under candidate parameters.
        3. Score (phi_sim vs phi_exp on the setpoint axis,
                  free_phase asymptote,
                  threshold setpoint).
        4. Optimise via differential evolution + Powell polish.
    """

    _MATERIAL_KEYS = ("E_star", "H", "a0", "R", "Q", "k")
    _AUX_KEYS = ("phase_offset_deg", "drive_scale")

    def __init__(
        self,
        fd_curves: Sequence[FDCalibrationCurve],
        base_params: Dict[str, float],
        conversions: Dict[str, float],
        solver: Optional[RK45Options] = None,
        config: Optional[FDCalibrationConfig] = None,
    ):
        self.fd_curves = list(fd_curves)
        self.base_params = dict(base_params)
        self.conversions = dict(conversions)
        self.solver = solver if solver is not None else RK45Options()
        self.config = config if config is not None else FDCalibrationConfig()

        # Auto-fill missing thresholds from data
        for fd in self.fd_curves:
            if fd.threshold_setpoint_pct is None:
                fd.threshold_setpoint_pct = detect_threshold_setpoint(
                    fd.setpoint_pct, fd.phase_deg,
                    self.config.threshold_phase_deg,
                    self.config.jump_min_deg,
                    self.config.threshold_smooth_window,
                )

    # ---------------- parameter assembly ----------------
    def _build_sim_params(self, theta: Dict[str, float]) -> Dict[str, float]:
        p = dict(self.base_params)
        for k in self._MATERIAL_KEYS:
            if k in theta:
                p[k] = float(theta[k])
        return p

    # ---------------- forward simulation ----------------
    def simulate_curve(self, drive_si: float, theta: Dict[str, float]):
        """Run one approach curve.  Returns d_hat, A_hat, phi_deg (with phase offset applied)."""
        p = self._build_sim_params(theta)
        drive_eff = float(drive_si) * float(theta.get("drive_scale", 1.0))
        spm_params, param_norm = build_normalized_spm_params(
            params=p,
            conversions=self.conversions,
            drive_si=drive_eff,
            omega_drive_hat=1.0,
            omega_ref_hat=1.0,
        )
        model = SPMRK45ApproachModel(spm_params, solver=self.solver)
        d_start = float(param_norm["A"]) * float(self.config.d_start_factor)
        curve = model.approach_curve(
            d_start_hat=d_start,
            d_end_hat=float(self.config.d_end_hat),
            n_points=int(self.config.n_d_hat_grid),
            z0_hat=0.0, v0_hat=0.0, warm_start=True,
        )
        phi = curve["phi_deg"] + float(theta.get("phase_offset_deg", 0.0))
        return curve["d_hat"], curve["A_hat"], phi

    # ---------------- sim → setpoint axis ----------------
    @staticmethod
    def _sim_to_setpoint_axis(d_hat: np.ndarray, A_hat: np.ndarray, phi: np.ndarray):
        """Re-parameterize (d_hat, A, phi) onto setpoint% = A/A_free*100."""
        far = int(np.argmax(d_hat))
        A_free_hat = float(A_hat[far])
        if A_free_hat <= 0 or not np.isfinite(A_free_hat):
            return None, None, None, np.nan
        sp_sim = A_hat / A_free_hat * 100.0
        return sp_sim, phi, d_hat, A_free_hat

    @staticmethod
    def _split_branches(sp_sim, phi_sim, d_sim, threshold_phase_deg: float = 90.0):
        """Split the natural approach trajectory at the first phase-jump across threshold."""
        order = np.argsort(-d_sim)
        sp = sp_sim[order]
        ph = phi_sim[order]
        ds = d_sim[order]
        trans = None
        for i in range(1, ph.size):
            if ph[i - 1] >= threshold_phase_deg > ph[i]:
                trans = i
                break
        if trans is None:
            return (sp, ph, ds), (None, None, None)
        return (sp[:trans], ph[:trans], ds[:trans]), (sp[trans:], ph[trans:], ds[trans:])

    @classmethod
    def _interp_phase(cls, sp_sim, phi_sim, d_sim, sp_query, branch: str = "auto",
                      threshold_phase_deg: float = 90.0):
        """Interpolate sim phase at query setpoints."""
        if branch == "auto":
            order = np.argsort(sp_sim)
            sp_s = sp_sim[order]
            ph_s = phi_sim[order]
            keep = np.concatenate([[True], np.diff(sp_s) > 1e-9])
            sp_s = sp_s[keep]
            ph_s = ph_s[keep]
            return np.interp(sp_query, sp_s, ph_s, left=np.nan, right=np.nan)
        attr, rep = cls._split_branches(sp_sim, phi_sim, d_sim, threshold_phase_deg)
        if branch == "attractive":
            sp_b, ph_b, _ = attr
        elif branch == "repulsive":
            if rep[0] is None:
                return np.full_like(np.asarray(sp_query, dtype=float), np.nan)
            sp_b, ph_b, _ = rep
        else:
            raise ValueError(f"Unknown branch: {branch}")
        order = np.argsort(sp_b)
        sp_s = sp_b[order]
        ph_s = ph_b[order]
        keep = np.concatenate([[True], np.diff(sp_s) > 1e-9])
        return np.interp(sp_query, sp_s[keep], ph_s[keep], left=np.nan, right=np.nan)

    @classmethod
    def _find_sim_threshold(cls, sp_sim, phi_sim, d_sim, threshold_phase_deg: float = 90.0):
        """Find the setpoint at which the sim's phase first crosses threshold downward."""
        order = np.argsort(-d_sim)
        sp = sp_sim[order]
        ph = phi_sim[order]
        for i in range(1, ph.size):
            if ph[i - 1] >= threshold_phase_deg > ph[i]:
                if ph[i] == ph[i - 1]:
                    return float(0.5 * (sp[i - 1] + sp[i]))
                t = (threshold_phase_deg - ph[i - 1]) / (ph[i] - ph[i - 1])
                return float(sp[i - 1] + t * (sp[i] - sp[i - 1]))
        return np.nan

    # ---------------- loss ----------------
    def loss(self, theta: Dict[str, float]) -> float:
        cfg = self.config
        total = 0.0
        n_terms = 0
        for fd in self.fd_curves:
            try:
                d_hat, A_hat, phi = self.simulate_curve(fd.drive_si, theta)
            except Exception:
                return 1e20
            sp_sim, phi_sim, d_sim, A_free_hat = self._sim_to_setpoint_axis(d_hat, A_hat, phi)
            if sp_sim is None:
                return 1e20

            # (a) phase curve term
            if cfg.phase_weight > 0:
                m = (fd.setpoint_pct >= cfg.setpoint_min_pct) & (fd.setpoint_pct <= cfg.setpoint_max_pct)
                if np.any(m):
                    phi_at = self._interp_phase(
                        sp_sim, phi_sim, d_sim, fd.setpoint_pct[m],
                        branch=cfg.interp_branch,
                        threshold_phase_deg=cfg.threshold_phase_deg,
                    )
                    valid = np.isfinite(phi_at)
                    if np.any(valid):
                        r = phi_at[valid] - fd.phase_deg[m][valid]
                        total += cfg.phase_weight * float(np.mean(r * r))
                        n_terms += 1

            # (b) free phase asymptote
            if cfg.free_phase_weight > 0:
                far_sim = float(phi_sim[int(np.argmax(d_sim))])
                r = far_sim - float(fd.free_phase_deg)
                total += cfg.free_phase_weight * float(r * r)
                n_terms += 1

            # (c) threshold term
            if cfg.threshold_weight > 0 and fd.threshold_setpoint_pct is not None:
                sp_th_sim = self._find_sim_threshold(sp_sim, phi_sim, d_sim, cfg.threshold_phase_deg)
                if np.isfinite(sp_th_sim):
                    r = sp_th_sim - float(fd.threshold_setpoint_pct)
                    total += cfg.threshold_weight * float(r * r)
                else:
                    total += cfg.threshold_weight * float(cfg.bistable_jump_penalty)
                n_terms += 1

        return float(total / max(n_terms, 1))

    # ---------------- fit ----------------
    def fit(
        self,
        bounds: Dict[str, Tuple[float, float]],
        x0: Optional[Dict[str, float]] = None,
        method: str = "differential_evolution",
        de_maxiter: int = 50,
        seed: int = 0,
        workers: int = 1,
        polish: bool = True,
        verbose: bool = False,
        callback=None,
    ) -> Dict[str, Any]:
        names = list(bounds.keys())
        bnds = [bounds[n] for n in names]

        def vec_to_theta(x):
            return {n: float(x[i]) for i, n in enumerate(names)}

        def obj(x):
            v = self.loss(vec_to_theta(x))
            if verbose:
                print(f"  loss={v:.5f}  theta={vec_to_theta(x)}")
            return v

        if method == "differential_evolution":
            res = differential_evolution(
                obj, bounds=bnds, maxiter=de_maxiter, seed=seed,
                workers=workers, polish=False, callback=callback,
            )
            x_best = res.x.copy()
            f_best = float(res.fun)
            if polish:
                res_local = minimize(obj, x_best, method="Powell", bounds=bnds)
                if float(res_local.fun) < f_best:
                    x_best = res_local.x.copy()
                    f_best = float(res_local.fun)
        else:
            x0_arr = (np.array([x0[n] for n in names])
                      if x0 is not None else
                      np.array([0.5 * (b[0] + b[1]) for b in bnds]))
            res = minimize(obj, x0_arr, method=method, bounds=bnds)
            x_best = res.x.copy()
            f_best = float(res.fun)

        theta_best = vec_to_theta(x_best)
        return {
            "theta_best": theta_best,
            "loss_best": f_best,
            "names": names,
            "x_best": x_best,
            "result": res,
            "calibrated_params": self.locked_params(theta_best),
        }

    def locked_params(self, theta: Dict[str, float]) -> Dict[str, float]:
        """Return the base_params dict with material values overridden by theta."""
        out = dict(self.base_params)
        for k in self._MATERIAL_KEYS:
            if k in theta:
                out[k] = float(theta[k])
        return out

    # ---------------- visualization ----------------
    def plot_fit(self, theta, figsize=(11, 8), only_drives=None, ax=None):
        if ax is None:
            fig, ax = plt.subplots(figsize=figsize)
        else:
            fig = ax.figure
        curves = self.fd_curves if only_drives is None else [
            c for c in self.fd_curves
            if any(np.isclose(c.drive_si, d, rtol=1e-3) for d in only_drives)
        ]
        colors = plt.cm.viridis(np.linspace(0, 0.92, max(len(curves), 1)))
        for fd, col in zip(curves, colors):
            d_hat, A_hat, phi = self.simulate_curve(fd.drive_si, theta)
            sp_sim, phi_sim, d_sim, _ = self._sim_to_setpoint_axis(d_hat, A_hat, phi)
            label_drive = fd.label or f"drive={fd.drive_si*1e9:.1f} nm"
            ax.plot(fd.setpoint_pct, fd.phase_deg, "-", color=col, lw=1.6,
                    label=f"Exp {label_drive}")
            order = np.argsort(-d_sim)
            ax.plot(sp_sim[order], phi_sim[order], "--", color=col, lw=1.2, alpha=0.7)
            if fd.threshold_setpoint_pct is not None:
                ax.axvline(fd.threshold_setpoint_pct, color=col, ls=":", alpha=0.4)
            sp_th_sim = self._find_sim_threshold(sp_sim, phi_sim, d_sim,
                                                 self.config.threshold_phase_deg)
            if np.isfinite(sp_th_sim):
                ax.axvline(sp_th_sim, color=col, ls="-.", alpha=0.4)
        ax.set_xlabel("Setpoint (%)")
        ax.set_ylabel("Phase (deg)")
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 180)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, loc="best", ncol=2)
        ax.set_title(f"FD-only calibration  (loss={self.loss(theta):.3g})")
        fig.tight_layout()
        return fig, ax

    def plot_threshold_curve(self, theta, figsize=(6.5, 6.5), ax=None):
        if ax is None:
            fig, ax = plt.subplots(figsize=figsize)
        else:
            fig = ax.figure
        drives_nm = []
        sp_th_exp = []
        sp_th_sim = []
        for fd in self.fd_curves:
            if fd.threshold_setpoint_pct is None:
                continue
            d_hat, A_hat, phi = self.simulate_curve(fd.drive_si, theta)
            sp_sim, phi_sim, d_sim, _ = self._sim_to_setpoint_axis(d_hat, A_hat, phi)
            sp_th = self._find_sim_threshold(sp_sim, phi_sim, d_sim,
                                             self.config.threshold_phase_deg)
            drives_nm.append(fd.drive_si * 1e9)
            sp_th_exp.append(fd.threshold_setpoint_pct)
            sp_th_sim.append(sp_th)
        drives_nm = np.array(drives_nm)
        sp_th_exp = np.array(sp_th_exp)
        sp_th_sim = np.array(sp_th_sim)
        ax.plot(sp_th_exp, drives_nm, "ro", ms=10, label="Exp threshold")
        ax.plot(sp_th_sim, drives_nm, "b^", ms=10, label="Sim threshold")
        for x_e, x_s, y in zip(sp_th_exp, sp_th_sim, drives_nm):
            if np.isfinite(x_s):
                ax.plot([x_e, x_s], [y, y], "k-", lw=0.5, alpha=0.4)
        ax.set_xlabel("Threshold setpoint (%)")
        ax.set_ylabel("Drive amplitude (nm)")
        ax.legend()
        ax.grid(alpha=0.3)
        ax.set_title("Attractive→repulsive threshold")
        fig.tight_layout()
        return fig, ax

    def report(self, theta):
        """Per-curve diagnostics — useful for spotting which drives don't match."""
        rows = []
        for fd in self.fd_curves:
            d_hat, A_hat, phi = self.simulate_curve(fd.drive_si, theta)
            sp_sim, phi_sim, d_sim, A_free_hat = self._sim_to_setpoint_axis(d_hat, A_hat, phi)
            far_sim = float(phi_sim[int(np.argmax(d_sim))])
            sp_th_sim = self._find_sim_threshold(
                sp_sim, phi_sim, d_sim, self.config.threshold_phase_deg
            )
            m = (fd.setpoint_pct >= self.config.setpoint_min_pct) & (
                fd.setpoint_pct <= self.config.setpoint_max_pct
            )
            phi_at = self._interp_phase(
                sp_sim, phi_sim, d_sim, fd.setpoint_pct[m],
                branch=self.config.interp_branch,
                threshold_phase_deg=self.config.threshold_phase_deg,
            )
            valid = np.isfinite(phi_at)
            rmse = (
                float(np.sqrt(np.mean((phi_at[valid] - fd.phase_deg[m][valid]) ** 2)))
                if np.any(valid)
                else float("nan")
            )
            rows.append({
                "drive_nm": fd.drive_si * 1e9,
                "A_free_exp_nm": fd.A_free_nm,
                "A_free_sim_nm": A_free_hat,  # already in nm units in this normalization
                "free_phase_exp": fd.free_phase_deg,
                "free_phase_sim": far_sim,
                "thresh_exp_pct": fd.threshold_setpoint_pct,
                "thresh_sim_pct": sp_th_sim,
                "phase_rmse_deg": rmse,
            })
        return rows


# ============================================================
# 5) BOUNDS HELPER
# ============================================================
def suggest_bounds(
    base_params: Dict[str, float],
    fit_keys: Sequence[str] = ("E_star", "H", "a0", "phase_offset_deg"),
) -> Dict[str, Tuple[float, float]]:
    """Conservative bounds keyed off the user's base_params."""
    b: Dict[str, Tuple[float, float]] = {}
    if "E_star" in fit_keys:
        E0 = base_params["E_star"]
        b["E_star"] = (E0 * 0.2, E0 * 5.0)
    if "H" in fit_keys:
        H0 = base_params["H"]
        b["H"] = (H0 * 0.2, H0 * 5.0)
    if "a0" in fit_keys:
        a0_0 = base_params["a0"]
        b["a0"] = (a0_0 * 0.3, a0_0 * 3.0)
    if "R" in fit_keys:
        R0 = base_params["R"]
        b["R"] = (R0 * 0.5, R0 * 2.0)
    if "Q" in fit_keys:
        Q0 = base_params["Q"]
        b["Q"] = (Q0 * 0.7, Q0 * 1.3)
    if "k" in fit_keys:
        k0 = base_params["k"]
        b["k"] = (k0 * 0.5, k0 * 2.0)
    if "phase_offset_deg" in fit_keys:
        b["phase_offset_deg"] = (-30.0, 30.0)
    if "drive_scale" in fit_keys:
        b["drive_scale"] = (0.7, 1.4)
    return b

# ============================================================
# 6) DATA LOADER FOR THE COMMON 15-CURVES LAYOUT
# ============================================================
def build_curves_from_arrays(
    drives_si: Sequence[float],
    amplitude_nm: np.ndarray,        # shape (n_drive, n_pixel)
    phase_deg: np.ndarray,           # shape (n_drive, n_pixel)
    z_piezo_nm: Optional[np.ndarray] = None,    # shape (n_drive, n_pixel) or None
    threshold_pct: Optional[Sequence[Optional[float]]] = None,
    A_free_nm: Optional[Sequence[float]] = None,
    labels: Optional[Sequence[str]] = None,
) -> List[FDCalibrationCurve]:
    """Convenience loader: turn 2D arrays of (n_drive, n_pixel) into FDCalibrationCurve list."""
    drives_si = list(drives_si)
    amp = np.asarray(amplitude_nm, dtype=float)
    ph = np.asarray(phase_deg, dtype=float)
    if amp.shape != ph.shape:
        raise ValueError("amplitude_nm and phase_deg must share shape")
    if amp.shape[0] != len(drives_si):
        raise ValueError("amplitude_nm.shape[0] must equal len(drives_si)")
    z = np.asarray(z_piezo_nm, dtype=float) if z_piezo_nm is not None else None
    out: List[FDCalibrationCurve] = []
    for i, d in enumerate(drives_si):
        out.append(FDCalibrationCurve(
            drive_si=float(d),
            amplitude_nm=amp[i],
            phase_deg=ph[i],
            z_piezo_nm=None if z is None else z[i],
            threshold_setpoint_pct=None if threshold_pct is None else threshold_pct[i],
            A_free_nm=None if A_free_nm is None else float(A_free_nm[i]),
            label="" if labels is None else str(labels[i]),
        ))
    return out

## FD calibration test

In [ ]:

# # ----- 1. Base params and conversions (same as your main script) -----
# base_params = {
#     "k": 1.0,
#     "A": 100e-9,                    # placeholder, gets overwritten per curve
#     "d0": 100e-9 * 1.25,
#     "a0": 0.164e-9,
#     "R": 20e-9,
#     "f0": 75e3,
#     "Q": 200,
#     "E_star": 1 / ((1 - 0.3 ** 2) / 130e9 + (1 - 0.3 ** 2) / 1.2e9),
#     "H": 7.1e-20,
# }
# conversions = {
#     "k": 1,
#     "A": 1e9,
#     "d0": 1e9,
#     "a0": 1e9,
#     "R": 1e9,
#     "f0": 1 / (2 * np.pi * base_params["f0"]),
#     "Q": 1,
#     "E_star": 1e-9 / base_params["k"],
#     "H": 1 / (base_params["k"] * (1e-9) ** 2),
# }
# solver = RK45Options(
#     N_cycles=20, max_windows=40, settle_windows=1,
#     tol_A=1e-4, tol_phi_deg=1e-2, consecutive=2,
#     demod_per_ref_cycle=30, rtol=1e-6, atol=1e-9,
#     dt0=0.02, dt_min=1e-10, dt_max=0.3, max_steps_per_window=200000,
# )

# # ----- 2. Load YOUR 15 experimental FD curves -----
# # Replace the dummy block below with your real loader.
# # Expected shapes: drives_si (15,), amplitude_nm (15, 1007), phase_deg (15, 1007)
# drives_si = np.array(
#     [7, 13, 18, 24, 29, 34, 39, 45, 50, 53, 66, 79, 92, 105, 117], dtype=float
# ) * 1e-9
# n_pix = 1007
# amplitude_nm = np.empty((len(drives_si), n_pix))
# phase_deg = np.empty((len(drives_si), n_pix))
# # >>> REPLACE THIS BLOCK with your actual loaded arrays <<<
# # e.g.:
# # data = np.load("fd_calibration_curves.npz")
# # amplitude_nm = data["amplitude_nm"]
# # phase_deg    = data["phase_deg"]
# # drives_si    = data["drives_si"]
# for i, drv in enumerate(drives_si):
#     # mock approach with synthetic noise for demo
#     sp = np.linspace(95, 5, n_pix)
#     amplitude_nm[i] = drv * 1e9 * sp / 100.0
#     phase_deg[i] = 90.0 + 15.0 * np.exp(-((sp - 70) ** 2) / 200.0) - 50.0 * (sp < 50)
# fd_curves = build_curves_from_arrays(drives_si, amplitude_nm, phase_deg)

# # ----- 3. Configure and build the calibrator -----
# cfg = FDCalibrationConfig(
#     fit_keys=("E_star", "H", "a0", "phase_offset_deg"),
#     phase_weight=1.0,
#     free_phase_weight=2.0,
#     threshold_weight=8.0,
#     setpoint_min_pct=30.0,
#     setpoint_max_pct=92.0,
#     threshold_phase_deg=90.0,
#     jump_min_deg=15.0,
#     n_d_hat_grid=160,
#     d_end_hat=-8.0,
#     interp_branch="auto",
# )
# cal = FDOnlyCalibrator(fd_curves, base_params, conversions, solver, cfg)

# print("Detected experimental thresholds:")
# for fd in cal.fd_curves:
#     print(f"  drive={fd.drive_si*1e9:6.1f} nm  threshold={fd.threshold_setpoint_pct}")

# # ----- 4. Stage 1: fit phase_offset alone (cheap warm-up) -----
# bounds_warm = {"phase_offset_deg": (-30.0, 30.0)}
# res_warm = cal.fit(bounds=bounds_warm, method="differential_evolution",
#                     de_maxiter=15, polish=True, seed=0)
# print(f"\n[stage 1] phase offset only: loss={res_warm['loss_best']:.4f}  "
#       f"theta={res_warm['theta_best']}")

# # ----- 5. Stage 2: fit material (E_star, H, a0) holding phase_offset from stage 1 -----
# cal2_cfg = FDCalibrationConfig(**{**cfg.__dict__,
#                                   "fit_keys": ("E_star", "H", "a0")})
# cal2 = FDOnlyCalibrator(fd_curves,
#                         base_params={**base_params,
#                                      "phase_offset_lock": res_warm["theta_best"]["phase_offset_deg"]},
#                         conversions=conversions, solver=solver, config=cal2_cfg)
# # apply the locked phase offset by injecting it into theta of every loss eval:
# fixed_phase_offset = res_warm["theta_best"]["phase_offset_deg"]
# orig_loss = cal2.loss
# cal2.loss = lambda th: orig_loss({**th, "phase_offset_deg": fixed_phase_offset})
# bounds_mat = suggest_bounds(base_params, ("E_star", "H", "a0"))
# res_mat = cal2.fit(bounds=bounds_mat, method="differential_evolution",
#                    de_maxiter=40, polish=True, seed=0, verbose=False)
# print(f"\n[stage 2] material only: loss={res_mat['loss_best']:.4f}  "
#       f"theta={res_mat['theta_best']}")

# # ----- 6. Stage 3: joint refinement of all four params (tight bounds around stage 1+2) -----
# th_seed = {**res_mat["theta_best"], "phase_offset_deg": fixed_phase_offset}
# bounds_joint = {}
# for k, v in th_seed.items():
#     if k == "phase_offset_deg":
#         bounds_joint[k] = (v - 5.0, v + 5.0)
#     else:
#         bounds_joint[k] = (v * 0.7, v * 1.4)
# res_final = cal.fit(bounds=bounds_joint, method="Powell", x0=th_seed, polish=False)
# print(f"\n[stage 3] joint polish: loss={res_final['loss_best']:.4f}  "
#       f"theta={res_final['theta_best']}")

# # ----- 7. Diagnostics & plots -----
# rows = cal.report(res_final["theta_best"])
# print("\nPer-curve diagnostics:")
# print(f"{'drive(nm)':>10}  {'A_free_exp':>10}  {'A_free_sim':>10}  "
#       f"{'phi0_exp':>9}  {'phi0_sim':>9}  {'th_exp':>7}  {'th_sim':>7}  {'rmse(deg)':>9}")
# for r in rows:
#     thx = "----" if r["thresh_exp_pct"] is None else f"{r['thresh_exp_pct']:7.2f}"
#     ths = "----" if not np.isfinite(r["thresh_sim_pct"]) else f"{r['thresh_sim_pct']:7.2f}"
#     print(f"{r['drive_nm']:10.2f}  {r['A_free_exp_nm']:10.3f}  {r['A_free_sim_nm']:10.3f}  "
#           f"{r['free_phase_exp']:9.2f}  {r['free_phase_sim']:9.2f}  "
#           f"{thx}  {ths}  {r['phase_rmse_deg']:9.3f}")

# cal.plot_fit(res_final["theta_best"])
# cal.plot_threshold_curve(res_final["theta_best"])
# plt.show()

# # ----- 8. Use the calibrated params downstream -----
# calibrated = res_final["calibrated_params"]
# phase_offset_calibrated = res_final["theta_best"].get("phase_offset_deg", 0.0)
# drive_scale_calibrated = res_final["theta_best"].get("drive_scale", 1.0)

# print("\n=== LOCKED PARAMETERS for stage 2 (imaging-grid calibration) ===")
# print(calibrated)
# print(f"phase_offset_deg = {phase_offset_calibrated:.3f}")
# print(f"drive_scale      = {drive_scale_calibrated:.3f}")

In [ ]:
drives_si = np.array(drives)[:-1] * factor * 1e-9
drives_si

In [ ]:
N = len(drives_si)

drives_si = np.array(drives)[:-1] * factor * 1e-9
print(len(drives_si))

L = 1007 // 20

amplitude_nm = np.zeros([N, L])
phase_deg = np.zeros([N, L])
height_nm = np.zeros([N, L])


for i in range(N):
    height_nm[i] = fwds[i][0][-L:] - np.min(fwds[i][0][-L:])
    amplitude_nm[i] = fwds[i][1][-L:] * 1e-9
    phase_deg[i] = fwds[i][2][-L:]
    # height_nm[i] = fwds[i][0][::4][:L]
    # amplitude_nm[i] = fwds[i][1][::4][:L] * 1e-9
    # phase_deg[i] = fwds[i][2][::4][:L]

In [ ]:
plt.plot(height_nm[-1], phase_deg[-1], '.-')

In [ ]:

# ----- 1. Base params and conversions (same as your main script) -----
base_params = {
    "k": 1.0,
    "A": 100e-9,                    # placeholder, gets overwritten per curve
    "d0": 100e-9 * 1.25,
    "a0": 0.164e-9,
    "R": 20e-9,
    "f0": 75e3,
    "Q": 200,
    "E_star": 1 / ((1 - 0.3 ** 2) / 130e9 + (1 - 0.3 ** 2) / 1.2e9),
    "H": 7.1e-20,
}
conversions = {
    "k": 1,
    "A": 1e9,
    "d0": 1e9,
    "a0": 1e9,
    "R": 1e9,
    "f0": 1 / (2 * np.pi * base_params["f0"]),
    "Q": 1,
    "E_star": 1e-9 / base_params["k"],
    "H": 1 / (base_params["k"] * (1e-9) ** 2),
}
solver = RK45Options(
    N_cycles=20, max_windows=40, settle_windows=1,
    tol_A=1e-4, tol_phi_deg=1e-2, consecutive=2,
    demod_per_ref_cycle=30, rtol=1e-6, atol=1e-9,
    dt0=0.02, dt_min=1e-10, dt_max=0.3, max_steps_per_window=200000,
)

# ----- 2. Load YOUR 15 experimental FD curves -----
N = len(drives_si)

drives_si = np.array(drives)[:-1] * factor * 1e-9
drives_si

amplitude_nm = np.zeros([N, 1007])
phase_deg = np.zeros([N, 1007])
height_nm = np.zeros([N, 1007])


for i in range(N):
    height_nm[i] = fwds[i][0][:1007]
    amplitude_nm[i] = fwds[i][1][:1007] * 1e-9
    phase_deg[i] = fwds[i][2][:1007]

fd_curves = build_curves_from_arrays(drives_si, amplitude_nm, phase_deg)

# ----- 3. Configure and build the calibrator -----
cfg = FDCalibrationConfig(
    fit_keys=("E_star", "H", "a0", "phase_offset_deg"),
    phase_weight=1.0,
    free_phase_weight=2.0,
    threshold_weight=8.0,
    setpoint_min_pct=30.0,
    setpoint_max_pct=92.0,
    threshold_phase_deg=90.0,
    jump_min_deg=15.0,
    n_d_hat_grid=160,
    d_end_hat=-8.0,
    interp_branch="auto",
)
cal = FDOnlyCalibrator(fd_curves, base_params, conversions, solver, cfg)

print("Detected experimental thresholds:")
for fd in cal.fd_curves:
    print(f"  drive={fd.drive_si*1e9:6.1f} nm  threshold={fd.threshold_setpoint_pct}")

# ----- 4. Stage 1: fit phase_offset alone (cheap warm-up) -----
bounds_warm = {"phase_offset_deg": (-30.0, 30.0)}
res_warm = cal.fit(bounds=bounds_warm, method="differential_evolution",
                    de_maxiter=15, polish=True, seed=0)
print(f"\n[stage 1] phase offset only: loss={res_warm['loss_best']:.4f}  "
      f"theta={res_warm['theta_best']}")

# ----- 5. Stage 2: fit material (E_star, H, a0) holding phase_offset from stage 1 -----
cal2_cfg = FDCalibrationConfig(**{**cfg.__dict__,
                                  "fit_keys": ("E_star", "H", "a0")})
cal2 = FDOnlyCalibrator(fd_curves,
                        base_params={**base_params,
                                     "phase_offset_lock": res_warm["theta_best"]["phase_offset_deg"]},
                        conversions=conversions, solver=solver, config=cal2_cfg)
# apply the locked phase offset by injecting it into theta of every loss eval:
fixed_phase_offset = res_warm["theta_best"]["phase_offset_deg"]
orig_loss = cal2.loss
cal2.loss = lambda th: orig_loss({**th, "phase_offset_deg": fixed_phase_offset})
bounds_mat = suggest_bounds(base_params, ("E_star", "H", "a0"))
res_mat = cal2.fit(bounds=bounds_mat, method="differential_evolution",
                   de_maxiter=40, polish=True, seed=0, verbose=False)
print(f"\n[stage 2] material only: loss={res_mat['loss_best']:.4f}  "
      f"theta={res_mat['theta_best']}")

# ----- 6. Stage 3: joint refinement of all four params (tight bounds around stage 1+2) -----
th_seed = {**res_mat["theta_best"], "phase_offset_deg": fixed_phase_offset}
bounds_joint = {}
for k, v in th_seed.items():
    if k == "phase_offset_deg":
        bounds_joint[k] = (v - 5.0, v + 5.0)
    else:
        bounds_joint[k] = (v * 0.7, v * 1.4)
res_final = cal.fit(bounds=bounds_joint, method="Powell", x0=th_seed, polish=False)
print(f"\n[stage 3] joint polish: loss={res_final['loss_best']:.4f}  "
      f"theta={res_final['theta_best']}")

# ----- 7. Diagnostics & plots -----
rows = cal.report(res_final["theta_best"])
print("\nPer-curve diagnostics:")
print(f"{'drive(nm)':>10}  {'A_free_exp':>10}  {'A_free_sim':>10}  "
      f"{'phi0_exp':>9}  {'phi0_sim':>9}  {'th_exp':>7}  {'th_sim':>7}  {'rmse(deg)':>9}")
for r in rows:
    thx = "----" if r["thresh_exp_pct"] is None else f"{r['thresh_exp_pct']:7.2f}"
    ths = "----" if not np.isfinite(r["thresh_sim_pct"]) else f"{r['thresh_sim_pct']:7.2f}"
    print(f"{r['drive_nm']:10.2f}  {r['A_free_exp_nm']:10.3f}  {r['A_free_sim_nm']:10.3f}  "
          f"{r['free_phase_exp']:9.2f}  {r['free_phase_sim']:9.2f}  "
          f"{thx}  {ths}  {r['phase_rmse_deg']:9.3f}")

cal.plot_fit(res_final["theta_best"])
cal.plot_threshold_curve(res_final["theta_best"])
plt.show()

# ----- 8. Use the calibrated params downstream -----
calibrated = res_final["calibrated_params"]
phase_offset_calibrated = res_final["theta_best"].get("phase_offset_deg", 0.0)
drive_scale_calibrated = res_final["theta_best"].get("drive_scale", 1.0)

print("\n=== LOCKED PARAMETERS for stage 2 (imaging-grid calibration) ===")
print(calibrated)
print(f"phase_offset_deg = {phase_offset_calibrated:.3f}")
print(f"drive_scale      = {drive_scale_calibrated:.3f}")

In [ ]:
"""
fd_priors.py
============

Turn a standard `params` dict (handbook / nominal cantilever + sample values)
into a usable Bayesian-flavored prior for the FD calibration:

    1. parameter GRIDS centered on the prior  (for surrogate library build)
    2. parameter BOUNDS for DE / minimize
    3. initial guess X0 for local optimizers
    4. optional Gaussian PRIOR PENALTY added to the loss

Conventions
-----------
- For parameters known on a log scale (E_star, H), the prior is log-normal:
        log10(theta) ~ Normal(log10(mu), sigma_dex)
  with sigma expressed in *decades* (e.g. 0.3 dex = factor of 2 std-dev).
- For parameters known on a linear scale (a0, R, Q), the prior is normal:
        theta ~ Normal(mu, sigma_factor * mu)
  with sigma_factor as a fractional width (e.g. 0.3 = ±30% std-dev).
- For aux knobs (phase_offset_deg, drive_scale), the prior is normal with an
  absolute sigma in deg or unitless ratio.

Usage
-----
    from fd_priors import make_default_prior

    prior = make_default_prior(params)        # uses your handbook params dict

    grids   = prior.grids(("E_star","H","a0"), n_points=6, n_sigma=2.5)
    bounds  = prior.bounds(("E_star","H","a0","phase_offset_deg"), n_sigma=3.0)
    x0      = prior.x0(    ("E_star","H","a0","phase_offset_deg"))

    # build library centered on the prior:
    lib = build_fd_param_library(..., E_grid=grids["E_star"],
                                  H_grid=grids["H"], a0_grid=grids["a0"], ...)

    # optionally add a soft penalty to the calibrator loss:
    cal = FDSurrogateCalibrator(lib, fd_curves, cfg)
    cal.loss = prior.wrap_loss(cal.loss, weight=0.05)

    # warm-started local fit (no DE needed if priors are good):
    res = cal.fit(bounds=bounds, x0=x0, method="L-BFGS-B")
"""

from __future__ import annotations
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, Optional, Sequence, Tuple

import numpy as np


# ============================================================
# 1) DEFAULT PRIOR WIDTHS (overridable)
# ============================================================
# Log-scale standard deviations expressed in DECADES (dex).
# 0.30 dex = factor 2 spread,  0.18 dex = factor 1.5 spread.
DEFAULT_LOG_SIGMA_DEX = {
    "E_star": 0.30,    # ±factor 2:   sample modulus is often uncertain by 2x
    "H":      0.30,    # Hamaker varies wildly; broad prior
}

# Linear-scale fractional widths (sigma / mu).
DEFAULT_LIN_SIGMA_FRAC = {
    "a0": 0.30,        # ±30%
    "R":  0.20,        # ±20% (tip radius from manufacturer)
    "Q":  0.10,        # ±10% (Q from thermal tune is usually tight)
    "k":  0.15,        # ±15% (cantilever spring constant)
    "f0": 0.01,        # ±1% (f0 well known from frequency sweep)
}

# Absolute (unitful) sigmas for auxiliary knobs.
DEFAULT_ABS_SIGMA = {
    "phase_offset_deg": 8.0,    # deg
    "drive_scale":      0.10,   # unitless multiplicative factor
}

# Which parameters live on a log scale (for grid layout & prior shape).
DEFAULT_LOG_SCALE = {
    "E_star": True,
    "H":      True,
    "a0":     False,
    "R":      False,
    "Q":      False,
    "k":      False,
    "f0":     False,
    "phase_offset_deg": False,
    "drive_scale":      False,
}


# ============================================================
# 2) PRIOR DATA CLASS
# ============================================================
@dataclass
class MaterialPrior:
    """
    Lightweight container for prior beliefs over the calibration parameters.

    Attributes
    ----------
    values : Dict[str, float]
        Prior mean (point estimate) for each parameter.
    log_sigma_dex : Dict[str, float]
        Std-dev in decades, used for log-scaled parameters.
    lin_sigma_frac : Dict[str, float]
        Fractional std-dev (sigma/mu) for linear-scaled positive parameters.
    abs_sigma : Dict[str, float]
        Absolute std-dev for auxiliary knobs (phase_offset_deg, drive_scale).
    log_scale : Dict[str, bool]
        Whether the parameter lives on a log scale.
    """
    values: Dict[str, float]
    log_sigma_dex: Dict[str, float] = field(default_factory=dict)
    lin_sigma_frac: Dict[str, float] = field(default_factory=dict)
    abs_sigma: Dict[str, float] = field(default_factory=dict)
    log_scale: Dict[str, bool] = field(default_factory=dict)

    # ---- internals ----
    def _is_log(self, name: str) -> bool:
        return bool(self.log_scale.get(name, DEFAULT_LOG_SCALE.get(name, False)))

    def _sigma(self, name: str) -> float:
        """Return std-dev *in the natural coordinates of the parameter*.

        For log-scale parameters this is in decades.
        For linear positive parameters it's an absolute value (frac * mu).
        For aux parameters it's an absolute value (already in deg / unitless).
        """
        if self._is_log(name):
            return float(self.log_sigma_dex.get(name, DEFAULT_LOG_SIGMA_DEX.get(name, 0.3)))
        if name in DEFAULT_ABS_SIGMA or name in self.abs_sigma:
            return float(self.abs_sigma.get(name, DEFAULT_ABS_SIGMA.get(name, 1.0)))
        mu = float(self.values[name])
        frac = float(self.lin_sigma_frac.get(name, DEFAULT_LIN_SIGMA_FRAC.get(name, 0.2)))
        return frac * abs(mu)

    # ---- accessor 1:  initial guess ----
    def x0(self, fit_keys: Sequence[str]) -> Dict[str, float]:
        return {k: float(self.values[k]) for k in fit_keys}

    # ---- accessor 2:  bounds ----
    def bounds(self, fit_keys: Sequence[str], n_sigma: float = 3.0,
               clip_positive: bool = True) -> Dict[str, Tuple[float, float]]:
        out: Dict[str, Tuple[float, float]] = {}
        for k in fit_keys:
            mu = float(self.values[k])
            sig = self._sigma(k)
            if self._is_log(k):
                lo = float(10 ** (np.log10(mu) - n_sigma * sig))
                hi = float(10 ** (np.log10(mu) + n_sigma * sig))
            else:
                lo = mu - n_sigma * sig
                hi = mu + n_sigma * sig
            if clip_positive and k in ("E_star", "H", "a0", "R", "Q", "k", "f0"):
                lo = max(lo, 1e-30)
            out[k] = (float(lo), float(hi))
        return out

    # ---- accessor 3:  surrogate library grids ----
    def grids(self, fit_keys: Sequence[str], n_points: int = 6,
              n_sigma: float = 2.5) -> Dict[str, np.ndarray]:
        out: Dict[str, np.ndarray] = {}
        for k in fit_keys:
            mu = float(self.values[k])
            sig = self._sigma(k)
            if self._is_log(k):
                lo = 10 ** (np.log10(mu) - n_sigma * sig)
                hi = 10 ** (np.log10(mu) + n_sigma * sig)
                out[k] = np.geomspace(lo, hi, int(n_points))
            else:
                lo = mu - n_sigma * sig
                hi = mu + n_sigma * sig
                if k in ("E_star", "H", "a0", "R", "Q", "k", "f0"):
                    lo = max(lo, 1e-30)
                out[k] = np.linspace(lo, hi, int(n_points))
        return out

    # ---- accessor 4:  Gaussian penalty ----
    def penalty(self, theta: Dict[str, float], weight: float = 1.0) -> float:
        """
        Sum of squared standardized deviations from the prior mean.

        Returns weight * sum_i ( (theta_i - mu_i) / sigma_i )^2.
        For log-scale params, deviations are computed in log10 space.
        Only parameters that are BOTH in `theta` AND in `self.values` contribute.
        """
        tot = 0.0
        for k, v in theta.items():
            if k not in self.values:
                continue
            mu = float(self.values[k])
            sig = self._sigma(k)
            if sig <= 0:
                continue
            if self._is_log(k):
                if v <= 0 or mu <= 0:
                    continue
                z = (np.log10(v) - np.log10(mu)) / sig
            else:
                z = (float(v) - mu) / sig
            tot += float(z * z)
        return float(weight) * tot

    def wrap_loss(self, loss_fn: Callable[[Dict[str, float]], float],
                   weight: float = 1.0) -> Callable[[Dict[str, float]], float]:
        """Return a new loss = data_loss + weight * prior_penalty."""
        def wrapped(theta: Dict[str, float]) -> float:
            return float(loss_fn(theta)) + self.penalty(theta, weight=weight)
        return wrapped

    # ---- diagnostic ----
    def report(self, fit_keys: Optional[Sequence[str]] = None) -> str:
        keys = list(fit_keys) if fit_keys is not None else list(self.values.keys())
        lines = ["MaterialPrior summary"]
        lines.append(f"  {'param':>18s}  {'mu':>14s}  {'sigma':>14s}  {'scale':>5s}  "
                     f"{'-3sigma':>14s}  {'+3sigma':>14s}")
        for k in keys:
            mu = float(self.values[k])
            sig = self._sigma(k)
            scale = "log" if self._is_log(k) else "lin"
            if self._is_log(k):
                lo = 10 ** (np.log10(mu) - 3 * sig)
                hi = 10 ** (np.log10(mu) + 3 * sig)
            else:
                lo = mu - 3 * sig
                hi = mu + 3 * sig
            lines.append(f"  {k:>18s}  {mu:14.4g}  {sig:14.4g}  {scale:>5s}  "
                         f"{lo:14.4g}  {hi:14.4g}")
        return "\n".join(lines)


# ============================================================
# 3) BUILDER FROM A STANDARD `params` DICT
# ============================================================
def make_default_prior(
    params: Dict[str, float],
    log_sigma_dex_overrides: Optional[Dict[str, float]] = None,
    lin_sigma_frac_overrides: Optional[Dict[str, float]] = None,
    abs_sigma_overrides: Optional[Dict[str, float]] = None,
    include_aux: bool = True,
) -> MaterialPrior:
    """
    Build a sensible MaterialPrior from your standard params dict.

    Parameters
    ----------
    params : the same dict you pass to build_normalized_spm_params, e.g.
        {'k', 'A', 'd0', 'a0', 'R', 'f0', 'Q', 'E_star', 'H', ...}
        Only physically-meaningful keys (E_star, H, a0, R, Q, k, f0) become priors.
    log_sigma_dex_overrides, lin_sigma_frac_overrides, abs_sigma_overrides :
        Override defaults per-key.  Use these to tighten priors you trust
        (e.g. Q from thermal tune) and loosen those you don't (sample modulus).
    include_aux : if True, also seed phase_offset_deg=0 and drive_scale=1.0
        so they can be fit alongside material params.
    """
    keys_of_interest = ("E_star", "H", "a0", "R", "Q", "k", "f0")
    values: Dict[str, float] = {}
    for k in keys_of_interest:
        if k in params:
            values[k] = float(params[k])
    if include_aux:
        values["phase_offset_deg"] = 0.0
        values["drive_scale"] = 1.0

    log_sigma = dict(DEFAULT_LOG_SIGMA_DEX)
    if log_sigma_dex_overrides:
        log_sigma.update(log_sigma_dex_overrides)

    lin_sigma = dict(DEFAULT_LIN_SIGMA_FRAC)
    if lin_sigma_frac_overrides:
        lin_sigma.update(lin_sigma_frac_overrides)

    abs_sigma = dict(DEFAULT_ABS_SIGMA)
    if abs_sigma_overrides:
        abs_sigma.update(abs_sigma_overrides)

    log_scale = dict(DEFAULT_LOG_SCALE)

    return MaterialPrior(
        values=values,
        log_sigma_dex=log_sigma,
        lin_sigma_frac=lin_sigma,
        abs_sigma=abs_sigma,
        log_scale=log_scale,
    )




In [ ]:
# # ============================================================
# # 4) END-TO-END EXAMPLE
# # ============================================================
# if __name__ == "__main__":
#     # ----- the example params from the user prompt -----
#     drive = 100e-9
#     params = {
#         "k": 1,
#         "A": drive,
#         "d0": drive * 1.25,
#         "a0": 0.164e-9,
#         "R": 20e-9,
#         "f0": 75e3,
#         "Q": 200,
#         "E_star": 1 / ((1 - 0.3 ** 2) / 130e9 + (1 - 0.3 ** 2) / 1.2e9),
#         "H": 7.1e-20,
#     }

#     # ----- build a prior with tighter Q (we trust thermal tune) and looser H -----
#     prior = make_default_prior(
#         params,
#         lin_sigma_frac_overrides={"Q": 0.05, "R": 0.30},
#         log_sigma_dex_overrides={"H": 0.5},
#     )
#     print(prior.report(("E_star", "H", "a0", "R", "Q", "phase_offset_deg")))

#     # ----- 1. grids for the surrogate library -----
#     fit_mat = ("E_star", "H", "a0")
#     grids = prior.grids(fit_mat, n_points=6, n_sigma=2.5)
#     print("\nLibrary grids:")
#     for k, g in grids.items():
#         print(f"  {k:>8}: [{g[0]:.4g} ... {g[-1]:.4g}]   {g}")

#     # ----- 2. bounds for the optimizer -----
#     fit_all = ("E_star", "H", "a0", "phase_offset_deg")
#     bounds = prior.bounds(fit_all, n_sigma=3.0)
#     print("\nBounds:")
#     for k, (lo, hi) in bounds.items():
#         print(f"  {k:>18}: [{lo:.4g}, {hi:.4g}]")

#     # ----- 3. initial guess for warm-started local optimization -----
#     x0 = prior.x0(fit_all)
#     print("\nInitial guess x0:", x0)

#     # ----- 4. prior penalty (try a few candidate thetas) -----
#     print("\nPenalty examples (n_sigma deviations):")
#     for label, theta in [
#         ("at prior mean", x0),
#         ("E* x2",         {**x0, "E_star": x0["E_star"] * 2.0}),
#         ("a0 +50%",       {**x0, "a0": x0["a0"] * 1.5}),
#         ("phase off=15",  {**x0, "phase_offset_deg": 15.0}),
#     ]:
#         print(f"  {label:>15}:  penalty(weight=1) = {prior.penalty(theta):.3f}")

#     # ----- 5. wiring it into the surrogate calibrator -----
#     print("""
#     Workflow integration
#     --------------------
#     from fd_surrogate_calibration import (
#         build_fd_param_library, FDSurrogateCalibrator, SurrogateCalibrationConfig,
#     )
#     from fd_only_calibration import FDCalibrationCurve

#     # build the surrogate library on the prior-centered grids
#     lib = build_fd_param_library(
#         base_params=params, conversions=conversions,
#         E_grid=grids["E_star"], H_grid=grids["H"], a0_grid=grids["a0"],
#         drives_si=drives_si, solver=solver,
#         n_d_hat=140, d_start_factor=1.25, d_end_hat=-7.0,
#         n_jobs=8, cache_path="fd_param_library.npz",
#     )

#     cal = FDSurrogateCalibrator(lib, fd_curves, SurrogateCalibrationConfig())
#     cal.loss = prior.wrap_loss(cal.loss, weight=0.05)   # soft Bayesian penalty

#     # warm-started local fit; no DE needed:
#     res = cal.fit(bounds=bounds, x0=x0, method="L-BFGS-B")
#     print("theta_best =", res["theta_best"])
#     """)


In [ ]:
"""
fd_only_calibration.py
======================

FD-only calibration for the SPM digital twin.

Inputs are a set of *experimental* point-FD curves taken at different
drive amplitudes.  Each curve is a 1D array (length N, e.g. 1007) of
amplitude and phase, optionally with z-piezo position.  The "setpoint
axis" used for fitting is the normalized amplitude:

        setpoint_pct  =  A_meas / A_free * 100

This avoids needing absolute z-piezo calibration and matches how the
attractive-to-repulsive transition is usually plotted (see Drive vs
Threshold-Setpoint plot).

The calibrator runs the existing RK45 approach-curve simulator from
`spm_fd_multidrive_alignment.py`, reparameterizes the simulated FD
curve onto the same setpoint axis, and minimises a weighted loss made
of three terms:

    L = w_phase  * MSE( phi_sim(setpoint) - phi_exp(setpoint) )
      + w_free   * (phi_sim_free - phi_exp_free)^2
      + w_thresh * (sp_thresh_sim - sp_thresh_exp)^2

Parameters that can be fit (any subset, controlled by FDCalibrationConfig.fit_keys):

    Material:   E_star, H, a0, R
    Cantilever: Q, k
    Aux:        phase_offset_deg, drive_scale

Recommended first pass:  fit ("E_star", "H", "a0", "phase_offset_deg")
with Q locked (use a thermal-tune Q) and drive_scale = 1.

Author: digital-twin alignment helper.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, minimize

# # Re-use the existing simulator. Adjust the import path if needed.
# from spm_fd_multidrive_alignment import (
#     NormalizedSPMParams,
#     RK45Options,
#     SPMRK45ApproachModel,
#     build_normalized_spm_params,
# )


# ============================================================
# 1) DATA CONTAINER
# ============================================================
@dataclass
class FDCalibrationCurve:
    """One experimental FD curve at a fixed drive amplitude.

    The fitter only needs (amplitude_nm, phase_deg) and the drive in SI.
    Setpoint% is auto-computed from amplitude / A_free, and the threshold
    setpoint (attractive→repulsive transition) is auto-detected from a
    sharp downward jump in phase across `threshold_phase_deg` if not
    supplied explicitly.

    All arrays must be 1-D and the same length.
    """
    drive_si: float                                  # m, drive amplitude at the cantilever
    amplitude_nm: np.ndarray                         # nm
    phase_deg: np.ndarray                            # deg (in [0, 180] convention)
    z_piezo_nm: Optional[np.ndarray] = None          # optional, z-piezo position (nm)
    setpoint_pct: Optional[np.ndarray] = None        # optional, default = A/A_free*100
    threshold_setpoint_pct: Optional[float] = None   # optional; auto-detected if None
    A_free_nm: Optional[float] = None                # optional; auto = high-percentile of A
    free_phase_deg: Optional[float] = None           # optional; auto = mean phase at high SP
    label: str = ""

    def __post_init__(self):
        self.amplitude_nm = np.asarray(self.amplitude_nm, dtype=float).ravel()
        self.phase_deg = np.asarray(self.phase_deg, dtype=float).ravel()
        if self.amplitude_nm.shape != self.phase_deg.shape:
            raise ValueError("amplitude_nm and phase_deg must have the same shape")
        if self.z_piezo_nm is not None:
            self.z_piezo_nm = np.asarray(self.z_piezo_nm, dtype=float).ravel()
        if self.A_free_nm is None:
            # robust free-amplitude estimate: mean of top 5% of amplitude
            top = np.percentile(self.amplitude_nm, 95.0)
            self.A_free_nm = float(np.mean(self.amplitude_nm[self.amplitude_nm >= top]))
        if self.A_free_nm <= 0:
            raise ValueError(f"Could not estimate A_free_nm for drive={self.drive_si}")
        if self.setpoint_pct is None:
            self.setpoint_pct = self.amplitude_nm / self.A_free_nm * 100.0
        else:
            self.setpoint_pct = np.asarray(self.setpoint_pct, dtype=float).ravel()
        if self.free_phase_deg is None:
            top_sp = np.percentile(self.setpoint_pct, 95.0)
            mask = self.setpoint_pct >= top_sp
            if np.any(mask):
                self.free_phase_deg = float(np.mean(self.phase_deg[mask]))
            else:
                self.free_phase_deg = float(np.mean(self.phase_deg))


# ============================================================
# 2) TRANSITION DETECTION
# ============================================================
def detect_threshold_setpoint(
    setpoint_pct: np.ndarray,
    phase_deg: np.ndarray,
    threshold_phase_deg: float = 90.0,
    jump_min_deg: float = 15.0,
    smooth_window: int = 3,
) -> Optional[float]:
    """
    Find the setpoint at which phase jumps downward across `threshold_phase_deg`.

    Walks the curve in approach order (descending setpoint) and looks for
    a discrete drop of at least `jump_min_deg` over `smooth_window` samples
    that crosses `threshold_phase_deg`.  Returns the linearly interpolated
    setpoint at the crossing, or None if no clear jump exists.
    """
    sp = np.asarray(setpoint_pct, dtype=float)
    ph = np.asarray(phase_deg, dtype=float)
    order = np.argsort(-sp)
    sp_o = sp[order]
    ph_o = ph[order]
    w = max(1, int(smooth_window))
    if ph_o.size < w + 1:
        return None
    diff = ph_o[w:] - ph_o[:-w]
    crossings = np.where(
        (ph_o[:-w] >= threshold_phase_deg)
        & (ph_o[w:] < threshold_phase_deg)
        & (diff <= -jump_min_deg)
    )[0]
    if crossings.size == 0:
        return None
    i = int(crossings[0])
    a = ph_o[i]
    b = ph_o[i + w]
    if b == a:
        return float(0.5 * (sp_o[i] + sp_o[i + w]))
    t = (threshold_phase_deg - a) / (b - a)
    t = float(np.clip(t, 0.0, 1.0))
    return float(sp_o[i] + t * (sp_o[i + w] - sp_o[i]))


# ============================================================
# 3) FIT CONFIG
# ============================================================
@dataclass
class FDCalibrationConfig:
    # Which keys to fit. Anything not listed is held at base_params value.
    fit_keys: Tuple[str, ...] = ("E_star", "H", "a0", "phase_offset_deg")

    # Loss weights
    phase_weight: float = 1.0
    free_phase_weight: float = 2.0
    threshold_weight: float = 6.0

    # Region of phase curve to compare. Avoids the bistable jump itself
    # (handled by the threshold term) and the noisy pre-engagement tail.
    setpoint_min_pct: float = 30.0
    setpoint_max_pct: float = 92.0

    # Threshold detection on experimental + simulated curves.
    threshold_phase_deg: float = 90.0
    jump_min_deg: float = 15.0
    threshold_smooth_window: int = 3

    # Branch selection for sim → exp interpolation
    # "auto"        : sort by setpoint and interp (works above threshold)
    # "attractive"  : keep only the high-d_hat branch (before transition)
    # "repulsive"   : keep only the low-d_hat branch (after transition)
    interp_branch: str = "auto"

    # Simulator grid
    n_d_hat_grid: int = 160
    d_end_hat: float = -8.0
    d_start_factor: float = 1.25  # d_start_hat = drive_hat * d_start_factor

    # Numerics
    bistable_jump_penalty: float = 100.0   # penalty if exp shows a jump but sim doesn't


# ============================================================
# 4) FD-ONLY CALIBRATOR
# ============================================================
class FDOnlyCalibrator:
    """
    Hierarchical FD-only calibration:
        1. Auto-detect threshold setpoints in experimental curves.
        2. Forward-simulate each FD curve under candidate parameters.
        3. Score (phi_sim vs phi_exp on the setpoint axis,
                  free_phase asymptote,
                  threshold setpoint).
        4. Optimise via differential evolution + Powell polish.
    """

    _MATERIAL_KEYS = ("E_star", "H", "a0", "R", "Q", "k")
    _AUX_KEYS = ("phase_offset_deg", "drive_scale")

    def __init__(
        self,
        fd_curves: Sequence[FDCalibrationCurve],
        base_params: Dict[str, float],
        conversions: Dict[str, float],
        solver: Optional[RK45Options] = None,
        config: Optional[FDCalibrationConfig] = None,
    ):
        self.fd_curves = list(fd_curves)
        self.base_params = dict(base_params)
        self.conversions = dict(conversions)
        self.solver = solver if solver is not None else RK45Options()
        self.config = config if config is not None else FDCalibrationConfig()

        # Auto-fill missing thresholds from data
        for fd in self.fd_curves:
            if fd.threshold_setpoint_pct is None:
                fd.threshold_setpoint_pct = detect_threshold_setpoint(
                    fd.setpoint_pct, fd.phase_deg,
                    self.config.threshold_phase_deg,
                    self.config.jump_min_deg,
                    self.config.threshold_smooth_window,
                )

    # ---------------- parameter assembly ----------------
    def _build_sim_params(self, theta: Dict[str, float]) -> Dict[str, float]:
        p = dict(self.base_params)
        for k in self._MATERIAL_KEYS:
            if k in theta:
                p[k] = float(theta[k])
        return p

    # ---------------- forward simulation ----------------
    def simulate_curve(self, drive_si: float, theta: Dict[str, float]):
        """Run one approach curve.  Returns d_hat, A_hat, phi_deg (with phase offset applied)."""
        p = self._build_sim_params(theta)
        drive_eff = float(drive_si) * float(theta.get("drive_scale", 1.0))
        spm_params, param_norm = build_normalized_spm_params(
            params=p,
            conversions=self.conversions,
            drive_si=drive_eff,
            omega_drive_hat=1.0,
            omega_ref_hat=1.0,
        )
        model = SPMRK45ApproachModel(spm_params, solver=self.solver)
        d_start = float(param_norm["A"]) * float(self.config.d_start_factor)
        curve = model.approach_curve(
            d_start_hat=d_start,
            d_end_hat=float(self.config.d_end_hat),
            n_points=int(self.config.n_d_hat_grid),
            z0_hat=0.0, v0_hat=0.0, warm_start=True,
        )
        phi = curve["phi_deg"] + float(theta.get("phase_offset_deg", 0.0))
        return curve["d_hat"], curve["A_hat"], phi

    # ---------------- sim → setpoint axis ----------------
    @staticmethod
    def _sim_to_setpoint_axis(d_hat: np.ndarray, A_hat: np.ndarray, phi: np.ndarray):
        """Re-parameterize (d_hat, A, phi) onto setpoint% = A/A_free*100."""
        far = int(np.argmax(d_hat))
        A_free_hat = float(A_hat[far])
        if A_free_hat <= 0 or not np.isfinite(A_free_hat):
            return None, None, None, np.nan
        sp_sim = A_hat / A_free_hat * 100.0
        return sp_sim, phi, d_hat, A_free_hat

    @staticmethod
    def _split_branches(sp_sim, phi_sim, d_sim, threshold_phase_deg: float = 90.0):
        """Split the natural approach trajectory at the first phase-jump across threshold."""
        order = np.argsort(-d_sim)
        sp = sp_sim[order]
        ph = phi_sim[order]
        ds = d_sim[order]
        trans = None
        for i in range(1, ph.size):
            if ph[i - 1] >= threshold_phase_deg > ph[i]:
                trans = i
                break
        if trans is None:
            return (sp, ph, ds), (None, None, None)
        return (sp[:trans], ph[:trans], ds[:trans]), (sp[trans:], ph[trans:], ds[trans:])

    @classmethod
    def _interp_phase(cls, sp_sim, phi_sim, d_sim, sp_query, branch: str = "auto",
                      threshold_phase_deg: float = 90.0):
        """Interpolate sim phase at query setpoints."""
        if branch == "auto":
            order = np.argsort(sp_sim)
            sp_s = sp_sim[order]
            ph_s = phi_sim[order]
            keep = np.concatenate([[True], np.diff(sp_s) > 1e-9])
            sp_s = sp_s[keep]
            ph_s = ph_s[keep]
            return np.interp(sp_query, sp_s, ph_s, left=np.nan, right=np.nan)
        attr, rep = cls._split_branches(sp_sim, phi_sim, d_sim, threshold_phase_deg)
        if branch == "attractive":
            sp_b, ph_b, _ = attr
        elif branch == "repulsive":
            if rep[0] is None:
                return np.full_like(np.asarray(sp_query, dtype=float), np.nan)
            sp_b, ph_b, _ = rep
        else:
            raise ValueError(f"Unknown branch: {branch}")
        order = np.argsort(sp_b)
        sp_s = sp_b[order]
        ph_s = ph_b[order]
        keep = np.concatenate([[True], np.diff(sp_s) > 1e-9])
        return np.interp(sp_query, sp_s[keep], ph_s[keep], left=np.nan, right=np.nan)

    @classmethod
    def _find_sim_threshold(cls, sp_sim, phi_sim, d_sim, threshold_phase_deg: float = 90.0):
        """Find the setpoint at which the sim's phase first crosses threshold downward."""
        order = np.argsort(-d_sim)
        sp = sp_sim[order]
        ph = phi_sim[order]
        for i in range(1, ph.size):
            if ph[i - 1] >= threshold_phase_deg > ph[i]:
                if ph[i] == ph[i - 1]:
                    return float(0.5 * (sp[i - 1] + sp[i]))
                t = (threshold_phase_deg - ph[i - 1]) / (ph[i] - ph[i - 1])
                return float(sp[i - 1] + t * (sp[i] - sp[i - 1]))
        return np.nan

    # ---------------- loss ----------------
    def loss(self, theta: Dict[str, float]) -> float:
        cfg = self.config
        total = 0.0
        n_terms = 0
        for fd in self.fd_curves:
            try:
                d_hat, A_hat, phi = self.simulate_curve(fd.drive_si, theta)
            except Exception:
                return 1e20
            sp_sim, phi_sim, d_sim, A_free_hat = self._sim_to_setpoint_axis(d_hat, A_hat, phi)
            if sp_sim is None:
                return 1e20

            # (a) phase curve term
            if cfg.phase_weight > 0:
                m = (fd.setpoint_pct >= cfg.setpoint_min_pct) & (fd.setpoint_pct <= cfg.setpoint_max_pct)
                if np.any(m):
                    phi_at = self._interp_phase(
                        sp_sim, phi_sim, d_sim, fd.setpoint_pct[m],
                        branch=cfg.interp_branch,
                        threshold_phase_deg=cfg.threshold_phase_deg,
                    )
                    valid = np.isfinite(phi_at)
                    if np.any(valid):
                        r = phi_at[valid] - fd.phase_deg[m][valid]
                        total += cfg.phase_weight * float(np.mean(r * r))
                        n_terms += 1

            # (b) free phase asymptote
            if cfg.free_phase_weight > 0:
                far_sim = float(phi_sim[int(np.argmax(d_sim))])
                r = far_sim - float(fd.free_phase_deg)
                total += cfg.free_phase_weight * float(r * r)
                n_terms += 1

            # (c) threshold term
            if cfg.threshold_weight > 0 and fd.threshold_setpoint_pct is not None:
                sp_th_sim = self._find_sim_threshold(sp_sim, phi_sim, d_sim, cfg.threshold_phase_deg)
                if np.isfinite(sp_th_sim):
                    r = sp_th_sim - float(fd.threshold_setpoint_pct)
                    total += cfg.threshold_weight * float(r * r)
                else:
                    total += cfg.threshold_weight * float(cfg.bistable_jump_penalty)
                n_terms += 1

        return float(total / max(n_terms, 1))

    # ---------------- fit ----------------
    def fit(
        self,
        bounds: Dict[str, Tuple[float, float]],
        x0: Optional[Dict[str, float]] = None,
        method: str = "differential_evolution",
        de_maxiter: int = 50,
        seed: int = 0,
        workers: int = 1,
        polish: bool = True,
        verbose: bool = False,
        callback=None,
    ) -> Dict[str, Any]:
        names = list(bounds.keys())
        bnds = [bounds[n] for n in names]

        def vec_to_theta(x):
            return {n: float(x[i]) for i, n in enumerate(names)}

        def obj(x):
            v = self.loss(vec_to_theta(x))
            if verbose:
                print(f"  loss={v:.5f}  theta={vec_to_theta(x)}")
            return v

        if method == "differential_evolution":
            res = differential_evolution(
                obj, bounds=bnds, maxiter=de_maxiter, seed=seed,
                workers=workers, polish=False, callback=callback,
            )
            x_best = res.x.copy()
            f_best = float(res.fun)
            if polish:
                res_local = minimize(obj, x_best, method="Powell", bounds=bnds)
                if float(res_local.fun) < f_best:
                    x_best = res_local.x.copy()
                    f_best = float(res_local.fun)
        else:
            x0_arr = (np.array([x0[n] for n in names])
                      if x0 is not None else
                      np.array([0.5 * (b[0] + b[1]) for b in bnds]))
            res = minimize(obj, x0_arr, method=method, bounds=bnds)
            x_best = res.x.copy()
            f_best = float(res.fun)

        theta_best = vec_to_theta(x_best)
        return {
            "theta_best": theta_best,
            "loss_best": f_best,
            "names": names,
            "x_best": x_best,
            "result": res,
            "calibrated_params": self.locked_params(theta_best),
        }

    def locked_params(self, theta: Dict[str, float]) -> Dict[str, float]:
        """Return the base_params dict with material values overridden by theta."""
        out = dict(self.base_params)
        for k in self._MATERIAL_KEYS:
            if k in theta:
                out[k] = float(theta[k])
        return out

    # ---------------- visualization ----------------
    def plot_fit(self, theta, figsize=(11, 8), only_drives=None, ax=None):
        if ax is None:
            fig, ax = plt.subplots(figsize=figsize)
        else:
            fig = ax.figure
        curves = self.fd_curves if only_drives is None else [
            c for c in self.fd_curves
            if any(np.isclose(c.drive_si, d, rtol=1e-3) for d in only_drives)
        ]
        colors = plt.cm.viridis(np.linspace(0, 0.92, max(len(curves), 1)))
        for fd, col in zip(curves, colors):
            d_hat, A_hat, phi = self.simulate_curve(fd.drive_si, theta)
            sp_sim, phi_sim, d_sim, _ = self._sim_to_setpoint_axis(d_hat, A_hat, phi)
            label_drive = fd.label or f"drive={fd.drive_si*1e9:.1f} nm"
            ax.plot(fd.setpoint_pct, fd.phase_deg, "-", color=col, lw=1.6,
                    label=f"Exp {label_drive}")
            order = np.argsort(-d_sim)
            ax.plot(sp_sim[order], phi_sim[order], "--", color=col, lw=1.2, alpha=0.7)
            if fd.threshold_setpoint_pct is not None:
                ax.axvline(fd.threshold_setpoint_pct, color=col, ls=":", alpha=0.4)
            sp_th_sim = self._find_sim_threshold(sp_sim, phi_sim, d_sim,
                                                 self.config.threshold_phase_deg)
            if np.isfinite(sp_th_sim):
                ax.axvline(sp_th_sim, color=col, ls="-.", alpha=0.4)
        ax.set_xlabel("Setpoint (%)")
        ax.set_ylabel("Phase (deg)")
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 180)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, loc="best", ncol=2)
        ax.set_title(f"FD-only calibration  (loss={self.loss(theta):.3g})")
        fig.tight_layout()
        return fig, ax

    def plot_threshold_curve(self, theta, figsize=(6.5, 6.5), ax=None):
        if ax is None:
            fig, ax = plt.subplots(figsize=figsize)
        else:
            fig = ax.figure
        drives_nm = []
        sp_th_exp = []
        sp_th_sim = []
        for fd in self.fd_curves:
            if fd.threshold_setpoint_pct is None:
                continue
            d_hat, A_hat, phi = self.simulate_curve(fd.drive_si, theta)
            sp_sim, phi_sim, d_sim, _ = self._sim_to_setpoint_axis(d_hat, A_hat, phi)
            sp_th = self._find_sim_threshold(sp_sim, phi_sim, d_sim,
                                             self.config.threshold_phase_deg)
            drives_nm.append(fd.drive_si * 1e9)
            sp_th_exp.append(fd.threshold_setpoint_pct)
            sp_th_sim.append(sp_th)
        drives_nm = np.array(drives_nm)
        sp_th_exp = np.array(sp_th_exp)
        sp_th_sim = np.array(sp_th_sim)
        ax.plot(sp_th_exp, drives_nm, "ro", ms=10, label="Exp threshold")
        ax.plot(sp_th_sim, drives_nm, "b^", ms=10, label="Sim threshold")
        for x_e, x_s, y in zip(sp_th_exp, sp_th_sim, drives_nm):
            if np.isfinite(x_s):
                ax.plot([x_e, x_s], [y, y], "k-", lw=0.5, alpha=0.4)
        ax.set_xlabel("Threshold setpoint (%)")
        ax.set_ylabel("Drive amplitude (nm)")
        ax.legend()
        ax.grid(alpha=0.3)
        ax.set_title("Attractive→repulsive threshold")
        fig.tight_layout()
        return fig, ax

    def report(self, theta):
        """Per-curve diagnostics — useful for spotting which drives don't match."""
        rows = []
        for fd in self.fd_curves:
            d_hat, A_hat, phi = self.simulate_curve(fd.drive_si, theta)
            sp_sim, phi_sim, d_sim, A_free_hat = self._sim_to_setpoint_axis(d_hat, A_hat, phi)
            far_sim = float(phi_sim[int(np.argmax(d_sim))])
            sp_th_sim = self._find_sim_threshold(
                sp_sim, phi_sim, d_sim, self.config.threshold_phase_deg
            )
            m = (fd.setpoint_pct >= self.config.setpoint_min_pct) & (
                fd.setpoint_pct <= self.config.setpoint_max_pct
            )
            phi_at = self._interp_phase(
                sp_sim, phi_sim, d_sim, fd.setpoint_pct[m],
                branch=self.config.interp_branch,
                threshold_phase_deg=self.config.threshold_phase_deg,
            )
            valid = np.isfinite(phi_at)
            rmse = (
                float(np.sqrt(np.mean((phi_at[valid] - fd.phase_deg[m][valid]) ** 2)))
                if np.any(valid)
                else float("nan")
            )
            rows.append({
                "drive_nm": fd.drive_si * 1e9,
                "A_free_exp_nm": fd.A_free_nm,
                "A_free_sim_nm": A_free_hat,  # already in nm units in this normalization
                "free_phase_exp": fd.free_phase_deg,
                "free_phase_sim": far_sim,
                "thresh_exp_pct": fd.threshold_setpoint_pct,
                "thresh_sim_pct": sp_th_sim,
                "phase_rmse_deg": rmse,
            })
        return rows


# ============================================================
# 5) BOUNDS HELPER
# ============================================================
def suggest_bounds(
    base_params: Dict[str, float],
    fit_keys: Sequence[str] = ("E_star", "H", "a0", "phase_offset_deg"),
) -> Dict[str, Tuple[float, float]]:
    """Conservative bounds keyed off the user's base_params."""
    b: Dict[str, Tuple[float, float]] = {}
    if "E_star" in fit_keys:
        E0 = base_params["E_star"]
        b["E_star"] = (E0 * 0.2, E0 * 5.0)
    if "H" in fit_keys:
        H0 = base_params["H"]
        b["H"] = (H0 * 0.2, H0 * 5.0)
    if "a0" in fit_keys:
        a0_0 = base_params["a0"]
        b["a0"] = (a0_0 * 0.3, a0_0 * 3.0)
    if "R" in fit_keys:
        R0 = base_params["R"]
        b["R"] = (R0 * 0.5, R0 * 2.0)
    if "Q" in fit_keys:
        Q0 = base_params["Q"]
        b["Q"] = (Q0 * 0.7, Q0 * 1.3)
    if "k" in fit_keys:
        k0 = base_params["k"]
        b["k"] = (k0 * 0.5, k0 * 2.0)
    if "phase_offset_deg" in fit_keys:
        b["phase_offset_deg"] = (-30.0, 30.0)
    if "drive_scale" in fit_keys:
        b["drive_scale"] = (0.7, 1.4)
    return b


# ============================================================
# 6) DATA LOADER FOR THE COMMON 15-CURVES LAYOUT
# ============================================================
def build_curves_from_arrays(
    drives_si: Sequence[float],
    amplitude_nm: np.ndarray,        # shape (n_drive, n_pixel)
    phase_deg: np.ndarray,           # shape (n_drive, n_pixel)
    z_piezo_nm: Optional[np.ndarray] = None,    # shape (n_drive, n_pixel) or None
    threshold_pct: Optional[Sequence[Optional[float]]] = None,
    A_free_nm: Optional[Sequence[float]] = None,
    labels: Optional[Sequence[str]] = None,
) -> List[FDCalibrationCurve]:
    """Convenience loader: turn 2D arrays of (n_drive, n_pixel) into FDCalibrationCurve list."""
    drives_si = list(drives_si)
    amp = np.asarray(amplitude_nm, dtype=float)
    ph = np.asarray(phase_deg, dtype=float)
    if amp.shape != ph.shape:
        raise ValueError("amplitude_nm and phase_deg must share shape")
    if amp.shape[0] != len(drives_si):
        raise ValueError("amplitude_nm.shape[0] must equal len(drives_si)")
    z = np.asarray(z_piezo_nm, dtype=float) if z_piezo_nm is not None else None
    out: List[FDCalibrationCurve] = []
    for i, d in enumerate(drives_si):
        out.append(FDCalibrationCurve(
            drive_si=float(d),
            amplitude_nm=amp[i],
            phase_deg=ph[i],
            z_piezo_nm=None if z is None else z[i],
            threshold_setpoint_pct=None if threshold_pct is None else threshold_pct[i],
            A_free_nm=None if A_free_nm is None else float(A_free_nm[i]),
            label="" if labels is None else str(labels[i]),
        ))
    return out





In [ ]:
# # ============================================================
# # 7) EXAMPLE WORKFLOW
# # ============================================================
# if __name__ == "__main__":
#     # ----- 1. Base params and conversions (same as your main script) -----
#     base_params = {
#         "k": 1.0,
#         "A": 100e-9,                    # placeholder, gets overwritten per curve
#         "d0": 100e-9 * 1.25,
#         "a0": 0.164e-9,
#         "R": 20e-9,
#         "f0": 75e3,
#         "Q": 200,
#         "E_star": 1 / ((1 - 0.3 ** 2) / 130e9 + (1 - 0.3 ** 2) / 1.2e9),
#         "H": 7.1e-20,
#     }
#     conversions = {
#         "k": 1,
#         "A": 1e9,
#         "d0": 1e9,
#         "a0": 1e9,
#         "R": 1e9,
#         "f0": 1 / (2 * np.pi * base_params["f0"]),
#         "Q": 1,
#         "E_star": 1e-9 / base_params["k"],
#         "H": 1 / (base_params["k"] * (1e-9) ** 2),
#     }
#     solver = RK45Options(
#         N_cycles=20, max_windows=40, settle_windows=1,
#         tol_A=1e-4, tol_phi_deg=1e-2, consecutive=2,
#         demod_per_ref_cycle=30, rtol=1e-6, atol=1e-9,
#         dt0=0.02, dt_min=1e-10, dt_max=0.3, max_steps_per_window=200000,
#     )

#     # ----- 2. Load YOUR 15 experimental FD curves -----
#     # Replace the dummy block below with your real loader.
#     # Expected shapes: drives_si (15,), amplitude_nm (15, 1007), phase_deg (15, 1007)
#     drives_si = np.array(
#         [7, 13, 18, 24, 29, 34, 39, 45, 50, 53, 66, 79, 92, 105, 117], dtype=float
#     ) * 1e-9
#     n_pix = 1007
#     amplitude_nm = np.empty((len(drives_si), n_pix))
#     phase_deg = np.empty((len(drives_si), n_pix))
#     # >>> REPLACE THIS BLOCK with your actual loaded arrays <<<
#     # e.g.:
#     # data = np.load("fd_calibration_curves.npz")
#     # amplitude_nm = data["amplitude_nm"]
#     # phase_deg    = data["phase_deg"]
#     # drives_si    = data["drives_si"]
#     for i, drv in enumerate(drives_si):
#         # mock approach with synthetic noise for demo
#         sp = np.linspace(95, 5, n_pix)
#         amplitude_nm[i] = drv * 1e9 * sp / 100.0
#         phase_deg[i] = 90.0 + 15.0 * np.exp(-((sp - 70) ** 2) / 200.0) - 50.0 * (sp < 50)
#     fd_curves = build_curves_from_arrays(drives_si, amplitude_nm, phase_deg)

#     # ----- 3. Configure and build the calibrator -----
#     cfg = FDCalibrationConfig(
#         fit_keys=("E_star", "H", "a0", "phase_offset_deg"),
#         phase_weight=1.0,
#         free_phase_weight=2.0,
#         threshold_weight=8.0,
#         setpoint_min_pct=30.0,
#         setpoint_max_pct=92.0,
#         threshold_phase_deg=90.0,
#         jump_min_deg=15.0,
#         n_d_hat_grid=160,
#         d_end_hat=-8.0,
#         interp_branch="auto",
#     )
#     cal = FDOnlyCalibrator(fd_curves, base_params, conversions, solver, cfg)

#     print("Detected experimental thresholds:")
#     for fd in cal.fd_curves:
#         print(f"  drive={fd.drive_si*1e9:6.1f} nm  threshold={fd.threshold_setpoint_pct}")

#     # ----- 4. Stage 1: fit phase_offset alone (cheap warm-up) -----
#     bounds_warm = {"phase_offset_deg": (-30.0, 30.0)}
#     res_warm = cal.fit(bounds=bounds_warm, method="differential_evolution",
#                         de_maxiter=15, polish=True, seed=0)
#     print(f"\n[stage 1] phase offset only: loss={res_warm['loss_best']:.4f}  "
#           f"theta={res_warm['theta_best']}")

#     # ----- 5. Stage 2: fit material (E_star, H, a0) holding phase_offset from stage 1 -----
#     cal2_cfg = FDCalibrationConfig(**{**cfg.__dict__,
#                                       "fit_keys": ("E_star", "H", "a0")})
#     cal2 = FDOnlyCalibrator(fd_curves,
#                             base_params={**base_params,
#                                          "phase_offset_lock": res_warm["theta_best"]["phase_offset_deg"]},
#                             conversions=conversions, solver=solver, config=cal2_cfg)
#     # apply the locked phase offset by injecting it into theta of every loss eval:
#     fixed_phase_offset = res_warm["theta_best"]["phase_offset_deg"]
#     orig_loss = cal2.loss
#     cal2.loss = lambda th: orig_loss({**th, "phase_offset_deg": fixed_phase_offset})
#     bounds_mat = suggest_bounds(base_params, ("E_star", "H", "a0"))
#     res_mat = cal2.fit(bounds=bounds_mat, method="differential_evolution",
#                        de_maxiter=40, polish=True, seed=0, verbose=False)
#     print(f"\n[stage 2] material only: loss={res_mat['loss_best']:.4f}  "
#           f"theta={res_mat['theta_best']}")

#     # ----- 6. Stage 3: joint refinement of all four params (tight bounds around stage 1+2) -----
#     th_seed = {**res_mat["theta_best"], "phase_offset_deg": fixed_phase_offset}
#     bounds_joint = {}
#     for k, v in th_seed.items():
#         if k == "phase_offset_deg":
#             bounds_joint[k] = (v - 5.0, v + 5.0)
#         else:
#             bounds_joint[k] = (v * 0.7, v * 1.4)
#     res_final = cal.fit(bounds=bounds_joint, method="Powell", x0=th_seed, polish=False)
#     print(f"\n[stage 3] joint polish: loss={res_final['loss_best']:.4f}  "
#           f"theta={res_final['theta_best']}")

#     # ----- 7. Diagnostics & plots -----
#     rows = cal.report(res_final["theta_best"])
#     print("\nPer-curve diagnostics:")
#     print(f"{'drive(nm)':>10}  {'A_free_exp':>10}  {'A_free_sim':>10}  "
#           f"{'phi0_exp':>9}  {'phi0_sim':>9}  {'th_exp':>7}  {'th_sim':>7}  {'rmse(deg)':>9}")
#     for r in rows:
#         thx = "----" if r["thresh_exp_pct"] is None else f"{r['thresh_exp_pct']:7.2f}"
#         ths = "----" if not np.isfinite(r["thresh_sim_pct"]) else f"{r['thresh_sim_pct']:7.2f}"
#         print(f"{r['drive_nm']:10.2f}  {r['A_free_exp_nm']:10.3f}  {r['A_free_sim_nm']:10.3f}  "
#               f"{r['free_phase_exp']:9.2f}  {r['free_phase_sim']:9.2f}  "
#               f"{thx}  {ths}  {r['phase_rmse_deg']:9.3f}")

#     cal.plot_fit(res_final["theta_best"])
#     cal.plot_threshold_curve(res_final["theta_best"])
#     plt.show()

#     # ----- 8. Use the calibrated params downstream -----
#     calibrated = res_final["calibrated_params"]
#     phase_offset_calibrated = res_final["theta_best"].get("phase_offset_deg", 0.0)
#     drive_scale_calibrated = res_final["theta_best"].get("drive_scale", 1.0)

#     print("\n=== LOCKED PARAMETERS for stage 2 (imaging-grid calibration) ===")
#     print(calibrated)
#     print(f"phase_offset_deg = {phase_offset_calibrated:.3f}")
#     print(f"drive_scale      = {drive_scale_calibrated:.3f}")

In [ ]:
"""
fd_surrogate_calibration.py
===========================

Surrogate-accelerated FD calibration of material parameters (E_star, H, a0).

Strategy
--------
1. BUILD a coarse parameter library ONCE:
     for each (E_star_i, H_j, a0_k) on a regular grid,
     for each drive in drives_si,
        run the RK45 approach simulator -> store (A_hat[d_hat], phi[d_hat])
   The result is a 5-D table:  (n_E, n_H, n_a0, n_drive, n_d_hat)
   This step is parallelizable and cacheable to .npz.

2. AT OPTIMIZATION TIME, evaluate the loss by interpolating in the
   library (scipy.RegularGridInterpolator over (E_star, H, a0)).  No
   ODE solves.  A loss eval costs milliseconds, so DE / Powell run
   in seconds.

3. OPTIONAL VERIFICATION: at the end, run one full-fidelity simulation
   at the optimum and confirm the surrogate's prediction.

This pairs with `fd_only_calibration.py` (same FDCalibrationCurve schema,
same loss decomposition: phase-curve + free-phase + threshold).

Compute budget guideline
------------------------
Default grid 6 x 6 x 6 = 216 param combos x 15 drives = 3240 RK45 sims.
At ~0.5 s per sim (numba-jit, 160 d_hat points) -> ~25 min serial,
~3 min on 8 cores.  Re-use forever.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple

import os
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, minimize
from scipy.interpolate import RegularGridInterpolator

# from spm_fd_multidrive_alignment import (
#     RK45Options,
#     SPMRK45ApproachModel,
#     build_normalized_spm_params,
# )

# # Re-use the FDCalibrationCurve / detect_threshold_setpoint from the
# # previous module so we don't duplicate the data schema.
# from fd_only_calibration import (
#     FDCalibrationCurve,
#     detect_threshold_setpoint,
# )


# ============================================================
# 1) PARAMETER LIBRARY
# ============================================================
@dataclass
class FDParamLibrary:
    """5-D pre-computed FD library indexed by (E_star, H, a0, drive, d_hat)."""
    E_grid: np.ndarray         # (n_E,)
    H_grid: np.ndarray         # (n_H,)
    a0_grid: np.ndarray        # (n_a0,)
    drives_si: np.ndarray      # (n_drive,)  drive amplitudes in m
    d_hat_grid: np.ndarray     # (n_d_hat,)  shared distance grid
    A_table: np.ndarray        # (n_E, n_H, n_a0, n_drive, n_d_hat)
    phi_table: np.ndarray      # same shape, phase in deg (NO offset applied)
    base_params: Dict[str, float]
    conversions: Dict[str, float]
    solver: Dict[str, Any] = field(default_factory=dict)
    attrs: Dict[str, Any] = field(default_factory=dict)

    # ---------- save / load ----------
    def save(self, path: str) -> None:
        np.savez_compressed(
            path,
            E_grid=self.E_grid, H_grid=self.H_grid, a0_grid=self.a0_grid,
            drives_si=self.drives_si, d_hat_grid=self.d_hat_grid,
            A_table=self.A_table, phi_table=self.phi_table,
            base_params_keys=np.array(list(self.base_params.keys())),
            base_params_vals=np.array(list(self.base_params.values()), dtype=float),
            conversions_keys=np.array(list(self.conversions.keys())),
            conversions_vals=np.array(list(self.conversions.values()), dtype=float),
        )

    @classmethod
    def load(cls, path: str) -> "FDParamLibrary":
        z = np.load(path, allow_pickle=False)
        bp = {str(k): float(v) for k, v in zip(z["base_params_keys"], z["base_params_vals"])}
        cv = {str(k): float(v) for k, v in zip(z["conversions_keys"], z["conversions_vals"])}
        return cls(
            E_grid=z["E_grid"], H_grid=z["H_grid"], a0_grid=z["a0_grid"],
            drives_si=z["drives_si"], d_hat_grid=z["d_hat_grid"],
            A_table=z["A_table"], phi_table=z["phi_table"],
            base_params=bp, conversions=cv,
        )

    # ---------- geometry ----------
    @property
    def shape(self):
        return self.A_table.shape  # (n_E, n_H, n_a0, n_drive, n_d_hat)


# ============================================================
# 2) BUILDER
# ============================================================
def _simulate_one(args):
    """Top-level helper for joblib / multiprocessing pickling."""
    (E, H, a0, drv, base_params, conversions, solver_kw,
     d_start_hat, d_end_hat, n_d_hat) = args
    p = dict(base_params)
    p["E_star"] = float(E)
    p["H"] = float(H)
    p["a0"] = float(a0)
    spm_params, param_norm = build_normalized_spm_params(
        params=p, conversions=conversions, drive_si=float(drv),
        omega_drive_hat=1.0, omega_ref_hat=1.0,
    )
    solver = RK45Options(**solver_kw)
    model = SPMRK45ApproachModel(spm_params, solver=solver)
    d0 = float(param_norm["A"]) * float(d_start_hat)
    curve = model.approach_curve(
        d_start_hat=d0, d_end_hat=float(d_end_hat),
        n_points=int(n_d_hat), z0_hat=0.0, v0_hat=0.0, warm_start=True,
    )
    # Re-sample onto a fixed d_hat grid so all curves share the same axis.
    d_grid = np.linspace(float(d_start_hat * float(param_norm["A"])), float(d_end_hat), int(n_d_hat))
    order = np.argsort(curve["d_hat"])
    d_sorted = curve["d_hat"][order]
    A_sorted = curve["A_hat"][order]
    phi_sorted = curve["phi_deg"][order]
    A_resamp = np.interp(d_grid, d_sorted, A_sorted)
    phi_resamp = np.interp(d_grid, d_sorted, phi_sorted)
    return A_resamp, phi_resamp, d_grid


def build_fd_param_library(
    base_params: Dict[str, float],
    conversions: Dict[str, float],
    E_grid: np.ndarray,
    H_grid: np.ndarray,
    a0_grid: np.ndarray,
    drives_si: np.ndarray,
    solver: Optional[RK45Options] = None,
    n_d_hat: int = 160,
    d_start_factor: float = 1.25,   # d_start_hat = drive_hat * factor
    d_end_hat: float = -8.0,
    n_jobs: int = 1,
    verbose: bool = True,
    cache_path: Optional[str] = None,
) -> FDParamLibrary:
    """
    Build an FD library on the (E_star, H, a0) x drive grid.

    If `cache_path` exists, the library is loaded from there and the build
    is skipped.  After building, the library is saved to `cache_path`.
    """
    if cache_path is not None and os.path.exists(cache_path):
        if verbose:
            print(f"[lib] loading cached library from {cache_path}")
        return FDParamLibrary.load(cache_path)

    E_grid = np.asarray(E_grid, dtype=float).ravel()
    H_grid = np.asarray(H_grid, dtype=float).ravel()
    a0_grid = np.asarray(a0_grid, dtype=float).ravel()
    drives_si = np.asarray(drives_si, dtype=float).ravel()

    n_E, n_H, n_a0, n_drv = E_grid.size, H_grid.size, a0_grid.size, drives_si.size
    total = n_E * n_H * n_a0 * n_drv

    if solver is None:
        solver = RK45Options()
    solver_kw = dict(
        N_cycles=solver.N_cycles, max_windows=solver.max_windows,
        settle_windows=solver.settle_windows, tol_A=solver.tol_A,
        tol_phi_deg=solver.tol_phi_deg, consecutive=solver.consecutive,
        demod_per_ref_cycle=solver.demod_per_ref_cycle, rtol=solver.rtol,
        atol=solver.atol, dt0=solver.dt0, dt_min=solver.dt_min, dt_max=solver.dt_max,
        max_steps_per_window=solver.max_steps_per_window,
    )

    # the d_hat grid depends on drive (via d_start_hat = drive_hat * factor),
    # so per-drive we'll use a normalized [0,1] axis and store d_hat too.
    # For interpolation simplicity we pick a SINGLE d_hat grid that spans
    # the largest drive's d_start; smaller drives get clipped/extrapolated.
    largest_drive_hat = float(drives_si.max() * conversions["A"])
    d_start_hat_global = largest_drive_hat * float(d_start_factor)
    d_hat_global = np.linspace(d_start_hat_global, float(d_end_hat), int(n_d_hat))

    A_table = np.full((n_E, n_H, n_a0, n_drv, n_d_hat), np.nan, dtype=float)
    phi_table = np.full_like(A_table, np.nan)

    # build job list
    jobs = []
    idx_map = []
    for iE, E in enumerate(E_grid):
        for iH, H in enumerate(H_grid):
            for ia, a0v in enumerate(a0_grid):
                for idr, drv in enumerate(drives_si):
                    jobs.append((
                        E, H, a0v, drv,
                        base_params, conversions, solver_kw,
                        d_start_factor, d_end_hat, n_d_hat,
                    ))
                    idx_map.append((iE, iH, ia, idr))

    if verbose:
        print(f"[lib] building {total} sims  "
              f"(n_E={n_E}, n_H={n_H}, n_a0={n_a0}, n_drive={n_drv}, n_d_hat={n_d_hat})  "
              f"on {n_jobs} job(s)")

    t0 = time.time()
    if n_jobs == 1:
        for k, args in enumerate(jobs):
            A_resamp, phi_resamp, d_grid = _simulate_one(args)
            iE, iH, ia, idr = idx_map[k]
            A_table[iE, iH, ia, idr, :] = np.interp(d_hat_global, d_grid, A_resamp,
                                                     left=np.nan, right=np.nan)
            phi_table[iE, iH, ia, idr, :] = np.interp(d_hat_global, d_grid, phi_resamp,
                                                       left=np.nan, right=np.nan)
            if verbose and (k + 1) % max(1, total // 20) == 0:
                el = time.time() - t0
                eta = el / (k + 1) * (total - k - 1)
                print(f"  [{k+1:5d}/{total}]  elapsed {el:6.1f}s  ETA {eta:6.1f}s")
    else:
        try:
            from joblib import Parallel, delayed
        except ImportError as exc:
            raise RuntimeError("Install joblib (pip install joblib) to use n_jobs > 1") from exc
        results = Parallel(n_jobs=n_jobs, verbose=10 if verbose else 0)(
            delayed(_simulate_one)(args) for args in jobs
        )
        for k, (A_resamp, phi_resamp, d_grid) in enumerate(results):
            iE, iH, ia, idr = idx_map[k]
            A_table[iE, iH, ia, idr, :] = np.interp(d_hat_global, d_grid, A_resamp,
                                                     left=np.nan, right=np.nan)
            phi_table[iE, iH, ia, idr, :] = np.interp(d_hat_global, d_grid, phi_resamp,
                                                       left=np.nan, right=np.nan)

    el = time.time() - t0
    if verbose:
        print(f"[lib] done in {el:.1f}s  ({el/total*1000:.1f} ms/sim)")

    lib = FDParamLibrary(
        E_grid=E_grid, H_grid=H_grid, a0_grid=a0_grid,
        drives_si=drives_si, d_hat_grid=d_hat_global,
        A_table=A_table, phi_table=phi_table,
        base_params=dict(base_params), conversions=dict(conversions),
        solver=solver_kw,
        attrs={"d_start_factor": float(d_start_factor),
               "d_end_hat": float(d_end_hat), "n_d_hat": int(n_d_hat),
               "build_time_s": float(el)},
    )
    if cache_path is not None:
        lib.save(cache_path)
        if verbose:
            print(f"[lib] saved to {cache_path}")
    return lib


# ============================================================
# 3) SURROGATE INTERPOLATOR
# ============================================================
class FDSurrogate:
    """
    Cheap evaluator: given (E_star, H, a0) -> per-drive (A_hat[d], phi[d]) curves.

    Uses scipy RegularGridInterpolator (linear) over the 3-D parameter grid;
    drive index and d_hat are sampled discretely.
    """

    def __init__(self, library: FDParamLibrary, method: str = "linear"):
        self.lib = library
        # Interpolators: one per (drive_idx, channel) returning the (n_d_hat,) vector.
        # Trick: pack (n_drive, n_d_hat) into the trailing dims and let RGI broadcast.
        # values shape -> (n_E, n_H, n_a0, n_drive*n_d_hat)
        n_E, n_H, n_a0, n_drv, n_d = library.A_table.shape
        A_flat = library.A_table.reshape(n_E, n_H, n_a0, n_drv * n_d)
        phi_flat = library.phi_table.reshape(n_E, n_H, n_a0, n_drv * n_d)
        pts = (library.E_grid, library.H_grid, library.a0_grid)
        self._interp_A = RegularGridInterpolator(pts, A_flat, method=method,
                                                   bounds_error=False, fill_value=np.nan)
        self._interp_phi = RegularGridInterpolator(pts, phi_flat, method=method,
                                                    bounds_error=False, fill_value=np.nan)
        self._n_drv = n_drv
        self._n_d = n_d

    def evaluate(self, E_star: float, H: float, a0: float):
        """Return (A_table[n_drive, n_d], phi_table[n_drive, n_d]) at (E*, H, a0)."""
        q = np.array([[float(E_star), float(H), float(a0)]])
        A = self._interp_A(q).reshape(self._n_drv, self._n_d)
        phi = self._interp_phi(q).reshape(self._n_drv, self._n_d)
        return A, phi


# ============================================================
# 4) SURROGATE CALIBRATOR
# ============================================================
@dataclass
class SurrogateCalibrationConfig:
    fit_keys: Tuple[str, ...] = ("E_star", "H", "a0", "phase_offset_deg")
    phase_weight: float = 1.0
    free_phase_weight: float = 2.0
    threshold_weight: float = 6.0
    setpoint_min_pct: float = 30.0
    setpoint_max_pct: float = 92.0
    threshold_phase_deg: float = 90.0
    jump_min_deg: float = 15.0
    bistable_jump_penalty: float = 100.0
    interp_branch: str = "auto"


class FDSurrogateCalibrator:
    """
    Same loss decomposition as FDOnlyCalibrator, but evaluates the forward
    model via interpolation in the precomputed library, NOT via the ODE.
    """

    def __init__(self,
                 library: FDParamLibrary,
                 fd_curves: Sequence[FDCalibrationCurve],
                 config: Optional[SurrogateCalibrationConfig] = None,
                 surrogate_method: str = "linear"):
        self.lib = library
        self.fd_curves = list(fd_curves)
        self.config = config if config is not None else SurrogateCalibrationConfig()
        self.surr = FDSurrogate(library, method=surrogate_method)

        # Map each experimental curve to a library drive index (nearest).
        self._drive_idx = []
        for fd in self.fd_curves:
            i = int(np.argmin(np.abs(self.lib.drives_si - float(fd.drive_si))))
            err_pct = abs(self.lib.drives_si[i] - fd.drive_si) / fd.drive_si * 100.0
            if err_pct > 5.0:
                print(f"[warn] drive {fd.drive_si*1e9:.2f} nm not in library "
                      f"(nearest = {self.lib.drives_si[i]*1e9:.2f} nm, err {err_pct:.1f}%)")
            self._drive_idx.append(i)

        # Auto-detect thresholds in the experimental data once.
        for fd in self.fd_curves:
            if fd.threshold_setpoint_pct is None:
                fd.threshold_setpoint_pct = detect_threshold_setpoint(
                    fd.setpoint_pct, fd.phase_deg,
                    self.config.threshold_phase_deg,
                    self.config.jump_min_deg,
                )

    # ---- branch-aware sim → setpoint axis interpolation ----
    @staticmethod
    def _sim_to_setpoint(d_hat, A_hat, phi):
        far = int(np.argmax(d_hat))
        A_free = float(A_hat[far])
        if A_free <= 0 or not np.isfinite(A_free):
            return None, None, None, np.nan
        return A_hat / A_free * 100.0, phi, d_hat, A_free

    @staticmethod
    def _interp_phase_at(sp_sim, phi_sim, sp_query):
        order = np.argsort(sp_sim)
        sp_s = sp_sim[order]
        phi_s = phi_sim[order]
        keep = np.concatenate([[True], np.diff(sp_s) > 1e-9])
        return np.interp(sp_query, sp_s[keep], phi_s[keep], left=np.nan, right=np.nan)

    @staticmethod
    def _find_threshold_sp(sp_sim, phi_sim, d_sim, threshold_phase_deg=90.0):
        order = np.argsort(-d_sim)
        sp = sp_sim[order]
        ph = phi_sim[order]
        for i in range(1, ph.size):
            if ph[i - 1] >= threshold_phase_deg > ph[i]:
                if ph[i] == ph[i - 1]:
                    return float(0.5 * (sp[i - 1] + sp[i]))
                t = (threshold_phase_deg - ph[i - 1]) / (ph[i] - ph[i - 1])
                return float(sp[i - 1] + t * (sp[i] - sp[i - 1]))
        return np.nan

    # ---- loss ----
    def loss(self, theta: Dict[str, float]) -> float:
        cfg = self.config
        E = float(theta.get("E_star", self.lib.base_params["E_star"]))
        H = float(theta.get("H", self.lib.base_params["H"]))
        a0 = float(theta.get("a0", self.lib.base_params["a0"]))
        phase_offset = float(theta.get("phase_offset_deg", 0.0))

        try:
            A_all, phi_all = self.surr.evaluate(E, H, a0)
        except Exception:
            return 1e20
        if not np.all(np.isfinite(A_all)) or not np.all(np.isfinite(phi_all)):
            return 1e20

        d_sim = self.lib.d_hat_grid
        total = 0.0
        n_terms = 0

        for k, fd in enumerate(self.fd_curves):
            j = self._drive_idx[k]
            A_hat = A_all[j]
            phi = phi_all[j] + phase_offset
            sp_sim, phi_sim, d_arr, A_free_hat = self._sim_to_setpoint(d_sim, A_hat, phi)
            if sp_sim is None:
                return 1e20

            # phase curve term
            if cfg.phase_weight > 0:
                m = (fd.setpoint_pct >= cfg.setpoint_min_pct) & (fd.setpoint_pct <= cfg.setpoint_max_pct)
                if np.any(m):
                    phi_at = self._interp_phase_at(sp_sim, phi_sim, fd.setpoint_pct[m])
                    valid = np.isfinite(phi_at)
                    if np.any(valid):
                        r = phi_at[valid] - fd.phase_deg[m][valid]
                        total += cfg.phase_weight * float(np.mean(r * r))
                        n_terms += 1

            # free-phase term
            if cfg.free_phase_weight > 0:
                far_sim = float(phi_sim[int(np.argmax(d_arr))])
                r = far_sim - float(fd.free_phase_deg)
                total += cfg.free_phase_weight * float(r * r)
                n_terms += 1

            # threshold term
            if cfg.threshold_weight > 0 and fd.threshold_setpoint_pct is not None:
                sp_th_sim = self._find_threshold_sp(sp_sim, phi_sim, d_arr, cfg.threshold_phase_deg)
                if np.isfinite(sp_th_sim):
                    r = sp_th_sim - float(fd.threshold_setpoint_pct)
                    total += cfg.threshold_weight * float(r * r)
                else:
                    total += cfg.threshold_weight * cfg.bistable_jump_penalty
                n_terms += 1

        return float(total / max(n_terms, 1))

    # ---- fit ----
    def fit(self, bounds: Dict[str, Tuple[float, float]],
            method: str = "differential_evolution",
            de_maxiter: int = 60, seed: int = 0, polish: bool = True,
            x0: Optional[Dict[str, float]] = None, verbose: bool = False):
        names = list(bounds.keys())
        bnds = [bounds[n] for n in names]

        def vec_to_theta(x):
            return {n: float(x[i]) for i, n in enumerate(names)}

        def obj(x):
            v = self.loss(vec_to_theta(x))
            if verbose:
                print(f"  loss={v:.5f}  theta={vec_to_theta(x)}")
            return v

        if method == "differential_evolution":
            res = differential_evolution(obj, bounds=bnds, maxiter=de_maxiter, seed=seed,
                                          polish=False)
            x_best = res.x.copy()
            f_best = float(res.fun)
            if polish:
                res2 = minimize(obj, x_best, method="Powell", bounds=bnds)
                if float(res2.fun) < f_best:
                    x_best, f_best = res2.x.copy(), float(res2.fun)
        else:
            x0_arr = (np.array([x0[n] for n in names])
                      if x0 is not None else
                      np.array([0.5 * (b[0] + b[1]) for b in bnds]))
            res = minimize(obj, x0_arr, method=method, bounds=bnds)
            x_best = res.x.copy()
            f_best = float(res.fun)

        return {
            "theta_best": vec_to_theta(x_best),
            "loss_best": f_best,
            "result": res,
        }

    # ---- diagnostic plot ----
    def plot_fit(self, theta, figsize=(11, 8), only_drive_indices=None):
        E = float(theta.get("E_star", self.lib.base_params["E_star"]))
        H = float(theta.get("H", self.lib.base_params["H"]))
        a0 = float(theta.get("a0", self.lib.base_params["a0"]))
        po = float(theta.get("phase_offset_deg", 0.0))
        A_all, phi_all = self.surr.evaluate(E, H, a0)
        d_sim = self.lib.d_hat_grid

        fig, ax = plt.subplots(figsize=figsize)
        idxs = range(len(self.fd_curves)) if only_drive_indices is None else only_drive_indices
        colors = plt.cm.viridis(np.linspace(0, 0.92, len(list(idxs))))
        for col, k in zip(colors, idxs):
            fd = self.fd_curves[k]
            j = self._drive_idx[k]
            A_hat, phi = A_all[j], phi_all[j] + po
            sp_sim, phi_sim, d_arr, _ = self._sim_to_setpoint(d_sim, A_hat, phi)
            ax.plot(fd.setpoint_pct, fd.phase_deg, "-", color=col, lw=1.6,
                    label=f"Exp d={fd.drive_si*1e9:.0f}nm")
            order = np.argsort(-d_arr)
            ax.plot(sp_sim[order], phi_sim[order], "--", color=col, lw=1.2, alpha=0.75)
        ax.set_xlabel("Setpoint (%)")
        ax.set_ylabel("Phase (deg)")
        ax.set_xlim(0, 100); ax.set_ylim(0, 180)
        ax.grid(alpha=0.3); ax.legend(fontsize=8, ncol=2)
        ax.set_title(f"Surrogate fit  loss={self.loss(theta):.3g}  "
                      f"(E={E:.2g}, H={H:.2g}, a0={a0:.2e})")
        fig.tight_layout()
        return fig, ax




In [ ]:
# # ============================================================
# # 5) END-TO-END WORKFLOW EXAMPLE
# # ============================================================
# if __name__ == "__main__":
#     # ---- (i) base setup, identical to your main script ----
#     base_params = {
#         "k": 1.0, "A": 100e-9, "d0": 100e-9 * 1.25,
#         "a0": 0.164e-9, "R": 20e-9, "f0": 75e3, "Q": 200,
#         "E_star": 1 / ((1 - 0.3 ** 2) / 130e9 + (1 - 0.3 ** 2) / 1.2e9),
#         "H": 7.1e-20,
#     }
#     conversions = {
#         "k": 1, "A": 1e9, "d0": 1e9, "a0": 1e9, "R": 1e9,
#         "f0": 1 / (2 * np.pi * base_params["f0"]), "Q": 1,
#         "E_star": 1e-9 / base_params["k"],
#         "H": 1 / (base_params["k"] * (1e-9) ** 2),
#     }
#     solver = RK45Options(N_cycles=15, max_windows=25, settle_windows=1,
#                           tol_A=2e-4, tol_phi_deg=2e-2, consecutive=2,
#                           demod_per_ref_cycle=24, rtol=1e-5, atol=1e-8,
#                           dt0=0.02, dt_min=1e-10, dt_max=0.3,
#                           max_steps_per_window=120000)

#     # drives_si = np.array([7, 13, 18, 24, 29, 34, 39, 45, 50, 53,
#     #                       66, 79, 92, 105, 117], dtype=float) * 1e-9

#     # ---- (ii) BUILD library ONCE (cached after first run) ----
#     # Use a SUBSET of 5-6 drives for material fitting; library stores them all.
#     E0 = base_params["E_star"]
#     H0 = base_params["H"]
#     a0_0 = base_params["a0"]
#     E_grid = np.geomspace(E0 * 0.4, E0 * 2.5, 6)     # log-spaced (~3 decades robust)
#     H_grid = np.geomspace(H0 * 0.4, H0 * 2.5, 6)
#     a0_grid = np.linspace(a0_0 * 0.6, a0_0 * 1.8, 6)  # linear
#     lib = build_fd_param_library(
#         base_params=base_params, conversions=conversions,
#         E_grid=E_grid, H_grid=H_grid, a0_grid=a0_grid,
#         drives_si=drives_si, solver=solver,
#         n_d_hat=140, d_start_factor=1.25, d_end_hat=-7.0,
#         n_jobs=1,                    # bump to 4-8 if joblib available
#         cache_path="fd_param_library.npz",
#         verbose=True,
#     )

#     # ---- (iii) load YOUR experimental FD curves ----
#     # Replace this stub with your actual loader (see fast-calibration script).
#     n_pix = L
#     # amplitude_nm = np.empty((drives_si.size, n_pix))
#     # phase_deg = np.empty((drives_si.size, n_pix))
#     # for i, drv in enumerate(drives_si):
#     #     sp = np.linspace(95, 5, n_pix)
#     #     amplitude_nm[i] = drv * 1e9 * sp / 100.0
#     #     phase_deg[i] = 90.0 + 18.0 * np.exp(-((sp - 70) ** 2) / 200.0) - 55.0 * (sp < 50)
#     fd_curves = [
#         FDCalibrationCurve(drive_si=float(drives_si[i]),
#                             amplitude_nm=amplitude_nm[i], phase_deg=phase_deg[i])
#         for i in range(drives_si.size)
#     ]

#     # ---- (iv) instantiate surrogate calibrator and FIT (seconds!) ----
#     cfg = SurrogateCalibrationConfig(
#         fit_keys=("E_star", "H", "a0", "phase_offset_deg"),
#         phase_weight=1.0, free_phase_weight=2.0, threshold_weight=8.0,
#         setpoint_min_pct=30.0, setpoint_max_pct=92.0,
#     )
#     cal = FDSurrogateCalibrator(lib, fd_curves, cfg, surrogate_method="linear")
#     bounds = {
#         "E_star": (E_grid.min(), E_grid.max()),
#         "H":      (H_grid.min(), H_grid.max()),
#         "a0":     (a0_grid.min(), a0_grid.max()),
#         "phase_offset_deg": (-20.0, 20.0),
#     }

#     t0 = time.time()
#     res = cal.fit(bounds, method="differential_evolution", de_maxiter=80,
#                    seed=0, polish=True, verbose=False)
#     print(f"[fit] done in {time.time()-t0:.1f}s    loss={res['loss_best']:.4f}")
#     print("theta_best =", res["theta_best"])

#     # ---- (v) optional verification: one full-fidelity sim at the optimum ----
#     # from fd_only_calibration import FDOnlyCalibrator, FDCalibrationConfig
#     # verify_cfg = FDCalibrationConfig(**cfg.__dict__)
#     # verify = FDOnlyCalibrator(fd_curves, base_params, conversions, solver, verify_cfg)
#     # full_loss = verify.loss(res["theta_best"])
#     # print(f"surrogate-loss = {res['loss_best']:.4f}, full-fidelity-loss = {full_loss:.4f}")

#     cal.plot_fit(res["theta_best"])
#     plt.show()


In [ ]:
"""
fd_surrogate_calibration.py
===========================

Surrogate-accelerated FD calibration of material parameters (E_star, H, a0).

Strategy
--------
1. BUILD a coarse parameter library ONCE:
     for each (E_star_i, H_j, a0_k) on a regular grid,
     for each drive in drives_si,
        run the RK45 approach simulator -> store (A_hat[d_hat], phi[d_hat])
   The result is a 5-D table:  (n_E, n_H, n_a0, n_drive, n_d_hat)
   This step is parallelizable and cacheable to .npz.

2. AT OPTIMIZATION TIME, evaluate the loss by interpolating in the
   library (scipy.RegularGridInterpolator over (E_star, H, a0)).  No
   ODE solves.  A loss eval costs milliseconds, so DE / Powell run
   in seconds.

3. OPTIONAL VERIFICATION: at the end, run one full-fidelity simulation
   at the optimum and confirm the surrogate's prediction.

This pairs with `fd_only_calibration.py` (same FDCalibrationCurve schema,
same loss decomposition: phase-curve + free-phase + threshold).

Compute budget guideline
------------------------
Default grid 6 x 6 x 6 = 216 param combos x 15 drives = 3240 RK45 sims.
At ~0.5 s per sim (numba-jit, 160 d_hat points) -> ~25 min serial,
~3 min on 8 cores.  Re-use forever.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple

import os
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, minimize
from scipy.interpolate import RegularGridInterpolator

# from spm_fd_multidrive_alignment import (
#     RK45Options,
#     SPMRK45ApproachModel,
#     build_normalized_spm_params,
# )

# # Re-use the FDCalibrationCurve / detect_threshold_setpoint from the
# # previous module so we don't duplicate the data schema.
# from fd_only_calibration import (
#     FDCalibrationCurve,
#     detect_threshold_setpoint,
# )


# ============================================================
# 1) PARAMETER LIBRARY
# ============================================================
@dataclass
class FDParamLibrary:
    """5-D pre-computed FD library indexed by (E_star, H, a0, drive, d_hat)."""
    E_grid: np.ndarray         # (n_E,)
    H_grid: np.ndarray         # (n_H,)
    a0_grid: np.ndarray        # (n_a0,)
    drives_si: np.ndarray      # (n_drive,)  drive amplitudes in m
    d_hat_grid: np.ndarray     # (n_d_hat,)  shared distance grid
    A_table: np.ndarray        # (n_E, n_H, n_a0, n_drive, n_d_hat)
    phi_table: np.ndarray      # same shape, phase in deg (NO offset applied)
    base_params: Dict[str, float]
    conversions: Dict[str, float]
    solver: Dict[str, Any] = field(default_factory=dict)
    attrs: Dict[str, Any] = field(default_factory=dict)

    # ---------- save / load ----------
    def save(self, path: str) -> None:
        np.savez_compressed(
            path,
            E_grid=self.E_grid, H_grid=self.H_grid, a0_grid=self.a0_grid,
            drives_si=self.drives_si, d_hat_grid=self.d_hat_grid,
            A_table=self.A_table, phi_table=self.phi_table,
            base_params_keys=np.array(list(self.base_params.keys())),
            base_params_vals=np.array(list(self.base_params.values()), dtype=float),
            conversions_keys=np.array(list(self.conversions.keys())),
            conversions_vals=np.array(list(self.conversions.values()), dtype=float),
        )

    @classmethod
    def load(cls, path: str) -> "FDParamLibrary":
        z = np.load(path, allow_pickle=False)
        bp = {str(k): float(v) for k, v in zip(z["base_params_keys"], z["base_params_vals"])}
        cv = {str(k): float(v) for k, v in zip(z["conversions_keys"], z["conversions_vals"])}
        return cls(
            E_grid=z["E_grid"], H_grid=z["H_grid"], a0_grid=z["a0_grid"],
            drives_si=z["drives_si"], d_hat_grid=z["d_hat_grid"],
            A_table=z["A_table"], phi_table=z["phi_table"],
            base_params=bp, conversions=cv,
        )

    # ---------- geometry ----------
    @property
    def shape(self):
        return self.A_table.shape  # (n_E, n_H, n_a0, n_drive, n_d_hat)


# ============================================================
# 2) BUILDER
# ============================================================
def _simulate_one(args):
    """Top-level helper for joblib / multiprocessing pickling."""
    (E, H, a0, drv, base_params, conversions, solver_kw,
     d_start_hat, d_end_hat, n_d_hat) = args
    p = dict(base_params)
    p["E_star"] = float(E)
    p["H"] = float(H)
    p["a0"] = float(a0)
    spm_params, param_norm = build_normalized_spm_params(
        params=p, conversions=conversions, drive_si=float(drv),
        omega_drive_hat=1.0, omega_ref_hat=1.0,
    )
    solver = RK45Options(**solver_kw)
    model = SPMRK45ApproachModel(spm_params, solver=solver)
    d0 = float(param_norm["A"]) * float(d_start_hat)
    curve = model.approach_curve(
        d_start_hat=d0, d_end_hat=float(d_end_hat),
        n_points=int(n_d_hat), z0_hat=0.0, v0_hat=0.0, warm_start=True,
    )
    # Re-sample onto a fixed *ascending* d_hat grid so all curves share the same axis
    # and so that np.interp / RegularGridInterpolator behave correctly downstream.
    d_grid = np.linspace(float(d_end_hat), float(d_start_hat * float(param_norm["A"])), int(n_d_hat))
    order = np.argsort(curve["d_hat"])
    d_sorted = curve["d_hat"][order]
    A_sorted = curve["A_hat"][order]
    phi_sorted = curve["phi_deg"][order]
    A_resamp = np.interp(d_grid, d_sorted, A_sorted)
    phi_resamp = np.interp(d_grid, d_sorted, phi_sorted)
    return A_resamp, phi_resamp, d_grid


def build_fd_param_library(
    base_params: Dict[str, float],
    conversions: Dict[str, float],
    E_grid: np.ndarray,
    H_grid: np.ndarray,
    a0_grid: np.ndarray,
    drives_si: np.ndarray,
    solver: Optional[RK45Options] = None,
    n_d_hat: int = 160,
    d_start_factor: float = 1.25,   # d_start_hat = drive_hat * factor
    d_end_hat: float = -8.0,
    n_jobs: int = 1,
    verbose: bool = True,
    cache_path: Optional[str] = None,
) -> FDParamLibrary:
    """
    Build an FD library on the (E_star, H, a0) x drive grid.

    If `cache_path` exists, the library is loaded from there and the build
    is skipped.  After building, the library is saved to `cache_path`.
    """
    if cache_path is not None and os.path.exists(cache_path):
        if verbose:
            print(f"[lib] loading cached library from {cache_path}")
        return FDParamLibrary.load(cache_path)

    E_grid = np.asarray(E_grid, dtype=float).ravel()
    H_grid = np.asarray(H_grid, dtype=float).ravel()
    a0_grid = np.asarray(a0_grid, dtype=float).ravel()
    drives_si = np.asarray(drives_si, dtype=float).ravel()

    n_E, n_H, n_a0, n_drv = E_grid.size, H_grid.size, a0_grid.size, drives_si.size
    total = n_E * n_H * n_a0 * n_drv

    if solver is None:
        solver = RK45Options()
    solver_kw = dict(
        N_cycles=solver.N_cycles, max_windows=solver.max_windows,
        settle_windows=solver.settle_windows, tol_A=solver.tol_A,
        tol_phi_deg=solver.tol_phi_deg, consecutive=solver.consecutive,
        demod_per_ref_cycle=solver.demod_per_ref_cycle, rtol=solver.rtol,
        atol=solver.atol, dt0=solver.dt0, dt_min=solver.dt_min, dt_max=solver.dt_max,
        max_steps_per_window=solver.max_steps_per_window,
    )

    # the d_hat grid depends on drive (via d_start_hat = drive_hat * factor),
    # so per-drive we'll use a normalized [0,1] axis and store d_hat too.
    # For interpolation simplicity we pick a SINGLE d_hat grid that spans
    # the largest drive's d_start; smaller drives get clipped/extrapolated.
    largest_drive_hat = float(drives_si.max() * conversions["A"])
    d_start_hat_global = largest_drive_hat * float(d_start_factor)
    # ASCENDING d_hat grid (low to high). Smaller drives only cover the lower
    # portion of this range; the upper portion will be NaN for those drives.
    d_hat_global = np.linspace(float(d_end_hat), d_start_hat_global, int(n_d_hat))

    A_table = np.full((n_E, n_H, n_a0, n_drv, n_d_hat), np.nan, dtype=float)
    phi_table = np.full_like(A_table, np.nan)

    # build job list
    jobs = []
    idx_map = []
    for iE, E in enumerate(E_grid):
        for iH, H in enumerate(H_grid):
            for ia, a0v in enumerate(a0_grid):
                for idr, drv in enumerate(drives_si):
                    jobs.append((
                        E, H, a0v, drv,
                        base_params, conversions, solver_kw,
                        d_start_factor, d_end_hat, n_d_hat,
                    ))
                    idx_map.append((iE, iH, ia, idr))

    if verbose:
        print(f"[lib] building {total} sims  "
              f"(n_E={n_E}, n_H={n_H}, n_a0={n_a0}, n_drive={n_drv}, n_d_hat={n_d_hat})  "
              f"on {n_jobs} job(s)")

    t0 = time.time()
    if n_jobs == 1:
        for k, args in enumerate(jobs):
            A_resamp, phi_resamp, d_grid = _simulate_one(args)
            iE, iH, ia, idr = idx_map[k]
            A_table[iE, iH, ia, idr, :] = np.interp(d_hat_global, d_grid, A_resamp,
                                                     left=np.nan, right=np.nan)
            phi_table[iE, iH, ia, idr, :] = np.interp(d_hat_global, d_grid, phi_resamp,
                                                       left=np.nan, right=np.nan)
            if verbose and (k + 1) % max(1, total // 20) == 0:
                el = time.time() - t0
                eta = el / (k + 1) * (total - k - 1)
                print(f"  [{k+1:5d}/{total}]  elapsed {el:6.1f}s  ETA {eta:6.1f}s")
    else:
        try:
            from joblib import Parallel, delayed
        except ImportError as exc:
            raise RuntimeError("Install joblib (pip install joblib) to use n_jobs > 1") from exc
        results = Parallel(n_jobs=n_jobs, verbose=10 if verbose else 0)(
            delayed(_simulate_one)(args) for args in jobs
        )
        for k, (A_resamp, phi_resamp, d_grid) in enumerate(results):
            iE, iH, ia, idr = idx_map[k]
            A_table[iE, iH, ia, idr, :] = np.interp(d_hat_global, d_grid, A_resamp,
                                                     left=np.nan, right=np.nan)
            phi_table[iE, iH, ia, idr, :] = np.interp(d_hat_global, d_grid, phi_resamp,
                                                       left=np.nan, right=np.nan)

    el = time.time() - t0
    if verbose:
        print(f"[lib] done in {el:.1f}s  ({el/total*1000:.1f} ms/sim)")

    lib = FDParamLibrary(
        E_grid=E_grid, H_grid=H_grid, a0_grid=a0_grid,
        drives_si=drives_si, d_hat_grid=d_hat_global,
        A_table=A_table, phi_table=phi_table,
        base_params=dict(base_params), conversions=dict(conversions),
        solver=solver_kw,
        attrs={"d_start_factor": float(d_start_factor),
               "d_end_hat": float(d_end_hat), "n_d_hat": int(n_d_hat),
               "build_time_s": float(el)},
    )
    if cache_path is not None:
        lib.save(cache_path)
        if verbose:
            print(f"[lib] saved to {cache_path}")
    return lib


# ============================================================
# 3) SURROGATE INTERPOLATOR
# ============================================================
class FDSurrogate:
    """
    Cheap evaluator: given (E_star, H, a0) -> per-drive (A_hat[d], phi[d]) curves.

    Uses scipy RegularGridInterpolator (linear) over the 3-D parameter grid;
    drive index and d_hat are sampled discretely.
    """

    def __init__(self, library: FDParamLibrary, method: str = "linear"):
        self.lib = library
        # Interpolators: one per (drive_idx, channel) returning the (n_d_hat,) vector.
        # Trick: pack (n_drive, n_d_hat) into the trailing dims and let RGI broadcast.
        # values shape -> (n_E, n_H, n_a0, n_drive*n_d_hat)
        n_E, n_H, n_a0, n_drv, n_d = library.A_table.shape
        A_flat = library.A_table.reshape(n_E, n_H, n_a0, n_drv * n_d)
        phi_flat = library.phi_table.reshape(n_E, n_H, n_a0, n_drv * n_d)
        pts = (library.E_grid, library.H_grid, library.a0_grid)
        self._interp_A = RegularGridInterpolator(pts, A_flat, method=method,
                                                   bounds_error=False, fill_value=np.nan)
        self._interp_phi = RegularGridInterpolator(pts, phi_flat, method=method,
                                                    bounds_error=False, fill_value=np.nan)
        self._n_drv = n_drv
        self._n_d = n_d

    def evaluate(self, E_star: float, H: float, a0: float):
        """Return (A_table[n_drive, n_d], phi_table[n_drive, n_d]) at (E*, H, a0)."""
        q = np.array([[float(E_star), float(H), float(a0)]])
        A = self._interp_A(q).reshape(self._n_drv, self._n_d)
        phi = self._interp_phi(q).reshape(self._n_drv, self._n_d)
        return A, phi


# ============================================================
# 4) SURROGATE CALIBRATOR
# ============================================================
@dataclass
class SurrogateCalibrationConfig:
    fit_keys: Tuple[str, ...] = ("E_star", "H", "a0", "phase_offset_deg")
    phase_weight: float = 1.0
    free_phase_weight: float = 2.0
    threshold_weight: float = 6.0
    setpoint_min_pct: float = 30.0
    setpoint_max_pct: float = 92.0
    threshold_phase_deg: float = 90.0
    jump_min_deg: float = 15.0
    bistable_jump_penalty: float = 100.0
    interp_branch: str = "auto"


class FDSurrogateCalibrator:
    """
    Same loss decomposition as FDOnlyCalibrator, but evaluates the forward
    model via interpolation in the precomputed library, NOT via the ODE.
    """

    def __init__(self,
                 library: FDParamLibrary,
                 fd_curves: Sequence[FDCalibrationCurve],
                 config: Optional[SurrogateCalibrationConfig] = None,
                 surrogate_method: str = "linear"):
        self.lib = library
        self.fd_curves = list(fd_curves)
        self.config = config if config is not None else SurrogateCalibrationConfig()
        self.surr = FDSurrogate(library, method=surrogate_method)

        # Map each experimental curve to a library drive index (nearest).
        self._drive_idx = []
        for fd in self.fd_curves:
            i = int(np.argmin(np.abs(self.lib.drives_si - float(fd.drive_si))))
            err_pct = abs(self.lib.drives_si[i] - fd.drive_si) / fd.drive_si * 100.0
            if err_pct > 5.0:
                print(f"[warn] drive {fd.drive_si*1e9:.2f} nm not in library "
                      f"(nearest = {self.lib.drives_si[i]*1e9:.2f} nm, err {err_pct:.1f}%)")
            self._drive_idx.append(i)

        # Auto-detect thresholds in the experimental data once.
        for fd in self.fd_curves:
            if fd.threshold_setpoint_pct is None:
                fd.threshold_setpoint_pct = detect_threshold_setpoint(
                    fd.setpoint_pct, fd.phase_deg,
                    self.config.threshold_phase_deg,
                    self.config.jump_min_deg,
                )

    # ---- branch-aware sim → setpoint axis interpolation ----
    @staticmethod
    def _sim_to_setpoint(d_hat, A_hat, phi):
        # Mask out any NaN entries (small drives have NaN at high d_hat in the
        # global library grid because they don't reach those distances).
        finite = np.isfinite(A_hat) & np.isfinite(phi) & np.isfinite(d_hat)
        if not np.any(finite):
            return None, None, None, np.nan
        d_v = d_hat[finite]
        A_v = A_hat[finite]
        phi_v = phi[finite]
        far = int(np.argmax(d_v))
        A_free = float(A_v[far])
        if A_free <= 0 or not np.isfinite(A_free):
            return None, None, None, np.nan
        return A_v / A_free * 100.0, phi_v, d_v, A_free

    @staticmethod
    def _interp_phase_at(sp_sim, phi_sim, sp_query):
        order = np.argsort(sp_sim)
        sp_s = sp_sim[order]
        phi_s = phi_sim[order]
        keep = np.concatenate([[True], np.diff(sp_s) > 1e-9])
        return np.interp(sp_query, sp_s[keep], phi_s[keep], left=np.nan, right=np.nan)

    @staticmethod
    def _find_threshold_sp(sp_sim, phi_sim, d_sim, threshold_phase_deg=90.0):
        order = np.argsort(-d_sim)
        sp = sp_sim[order]
        ph = phi_sim[order]
        for i in range(1, ph.size):
            if ph[i - 1] >= threshold_phase_deg > ph[i]:
                if ph[i] == ph[i - 1]:
                    return float(0.5 * (sp[i - 1] + sp[i]))
                t = (threshold_phase_deg - ph[i - 1]) / (ph[i] - ph[i - 1])
                return float(sp[i - 1] + t * (sp[i] - sp[i - 1]))
        return np.nan

    # ---- loss ----
    def loss(self, theta: Dict[str, float]) -> float:
        cfg = self.config
        E = float(theta.get("E_star", self.lib.base_params["E_star"]))
        H = float(theta.get("H", self.lib.base_params["H"]))
        a0 = float(theta.get("a0", self.lib.base_params["a0"]))
        phase_offset = float(theta.get("phase_offset_deg", 0.0))

        try:
            A_all, phi_all = self.surr.evaluate(E, H, a0)
        except Exception:
            return 1e20

        d_sim = self.lib.d_hat_grid
        total = 0.0
        n_terms = 0

        for k, fd in enumerate(self.fd_curves):
            j = self._drive_idx[k]
            A_hat = A_all[j]
            phi = phi_all[j] + phase_offset
            sp_sim, phi_sim, d_arr, A_free_hat = self._sim_to_setpoint(d_sim, A_hat, phi)
            if sp_sim is None:
                # surrogate returned no valid samples for this drive; skip it
                continue

            # phase curve term
            if cfg.phase_weight > 0:
                m = (fd.setpoint_pct >= cfg.setpoint_min_pct) & (fd.setpoint_pct <= cfg.setpoint_max_pct)
                if np.any(m):
                    phi_at = self._interp_phase_at(sp_sim, phi_sim, fd.setpoint_pct[m])
                    valid = np.isfinite(phi_at)
                    if np.any(valid):
                        r = phi_at[valid] - fd.phase_deg[m][valid]
                        total += cfg.phase_weight * float(np.mean(r * r))
                        n_terms += 1

            # free-phase term
            if cfg.free_phase_weight > 0:
                far_sim = float(phi_sim[int(np.argmax(d_arr))])
                r = far_sim - float(fd.free_phase_deg)
                total += cfg.free_phase_weight * float(r * r)
                n_terms += 1

            # threshold term
            if cfg.threshold_weight > 0 and fd.threshold_setpoint_pct is not None:
                sp_th_sim = self._find_threshold_sp(sp_sim, phi_sim, d_arr, cfg.threshold_phase_deg)
                if np.isfinite(sp_th_sim):
                    r = sp_th_sim - float(fd.threshold_setpoint_pct)
                    total += cfg.threshold_weight * float(r * r)
                else:
                    total += cfg.threshold_weight * cfg.bistable_jump_penalty
                n_terms += 1

        return float(total / max(n_terms, 1))

    # ---- fit ----
    def fit(self, bounds: Dict[str, Tuple[float, float]],
            method: str = "differential_evolution",
            de_maxiter: int = 60, seed: int = 0, polish: bool = True,
            x0: Optional[Dict[str, float]] = None, verbose: bool = False):
        names = list(bounds.keys())
        bnds = [bounds[n] for n in names]

        def vec_to_theta(x):
            return {n: float(x[i]) for i, n in enumerate(names)}

        def obj(x):
            v = self.loss(vec_to_theta(x))
            if verbose:
                print(f"  loss={v:.5f}  theta={vec_to_theta(x)}")
            return v

        if method == "differential_evolution":
            res = differential_evolution(obj, bounds=bnds, maxiter=de_maxiter, seed=seed,
                                          polish=False)
            x_best = res.x.copy()
            f_best = float(res.fun)
            if polish:
                res2 = minimize(obj, x_best, method="Powell", bounds=bnds)
                if float(res2.fun) < f_best:
                    x_best, f_best = res2.x.copy(), float(res2.fun)
        else:
            x0_arr = (np.array([x0[n] for n in names])
                      if x0 is not None else
                      np.array([0.5 * (b[0] + b[1]) for b in bnds]))
            res = minimize(obj, x0_arr, method=method, bounds=bnds)
            x_best = res.x.copy()
            f_best = float(res.fun)

        return {
            "theta_best": vec_to_theta(x_best),
            "loss_best": f_best,
            "result": res,
        }

    # ---- diagnostic plot ----
    def plot_fit(self, theta, figsize=(11, 8), only_drive_indices=None):
        E = float(theta.get("E_star", self.lib.base_params["E_star"]))
        H = float(theta.get("H", self.lib.base_params["H"]))
        a0 = float(theta.get("a0", self.lib.base_params["a0"]))
        po = float(theta.get("phase_offset_deg", 0.0))
        A_all, phi_all = self.surr.evaluate(E, H, a0)
        d_sim = self.lib.d_hat_grid

        fig, ax = plt.subplots(figsize=figsize)
        idxs = range(len(self.fd_curves)) if only_drive_indices is None else only_drive_indices
        colors = plt.cm.viridis(np.linspace(0, 0.92, len(list(idxs))))
        for col, k in zip(colors, idxs):
            fd = self.fd_curves[k]
            j = self._drive_idx[k]
            A_hat, phi = A_all[j], phi_all[j] + po
            sp_sim, phi_sim, d_arr, _ = self._sim_to_setpoint(d_sim, A_hat, phi)
            ax.plot(fd.setpoint_pct, fd.phase_deg, "-", color=col, lw=1.6,
                    label=f"Exp d={fd.drive_si*1e9:.0f}nm")
            if sp_sim is None:
                # surrogate had no valid samples for this drive; show exp only
                continue
            order = np.argsort(-d_arr)
            ax.plot(sp_sim[order], phi_sim[order], "--", color=col, lw=1.2, alpha=0.75)
        ax.set_xlabel("Setpoint (%)")
        ax.set_ylabel("Phase (deg)")
        ax.set_xlim(0, 100); ax.set_ylim(0, 180)
        ax.grid(alpha=0.3); ax.legend(fontsize=8, ncol=2)
        ax.set_title(f"Surrogate fit  loss={self.loss(theta):.3g}  "
                      f"(E={E:.2g}, H={H:.2g}, a0={a0:.2e})")
        fig.tight_layout()
        return fig, ax




In [ ]:

# ============================================================
# 5) END-TO-END WORKFLOW EXAMPLE
# ============================================================
if __name__ == "__main__":
    # ---- (i) base setup, identical to your main script ----
    base_params = {
        "k": 1.0, "A": 100e-9, "d0": 100e-9 * 1.25,
        "a0": 0.164e-9, "R": 20e-9, "f0": 75e3, "Q": 200,
        "E_star": 1 / ((1 - 0.3 ** 2) / 130e9 + (1 - 0.3 ** 2) / 1.2e9),
        "H": 7.1e-20,
    }
    conversions = {
        "k": 1, "A": 1e9, "d0": 1e9, "a0": 1e9, "R": 1e9,
        "f0": 1 / (2 * np.pi * base_params["f0"]), "Q": 1,
        "E_star": 1e-9 / base_params["k"],
        "H": 1 / (base_params["k"] * (1e-9) ** 2),
    }
    solver = RK45Options(N_cycles=15, max_windows=25, settle_windows=1,
                          tol_A=2e-4, tol_phi_deg=2e-2, consecutive=2,
                          demod_per_ref_cycle=24, rtol=1e-5, atol=1e-8,
                          dt0=0.02, dt_min=1e-10, dt_max=0.3,
                          max_steps_per_window=120000)

    # drives_si = np.array([7, 13, 18, 24, 29, 34, 39, 45, 50, 53,
    #                       66, 79, 92, 105, 117], dtype=float) * 1e-9

    # ---- (ii) BUILD library ONCE (cached after first run) ----
    # Use a SUBSET of 5-6 drives for material fitting; library stores them all.
    E0 = base_params["E_star"]
    H0 = base_params["H"]
    a0_0 = base_params["a0"]
    E_grid = np.geomspace(E0 * 0.4, E0 * 2.5, 7)     # log-spaced (~3 decades robust)
    H_grid = np.geomspace(H0 * 0.4, H0 * 2.5, 7)
    a0_grid = np.linspace(a0_0 * 0.6, a0_0 * 1.8, 7)  # linear
    lib = build_fd_param_library(
        base_params=base_params, conversions=conversions,
        E_grid=E_grid, H_grid=H_grid, a0_grid=a0_grid,
        drives_si=drives_si, solver=solver,
        n_d_hat=101, d_start_factor=1.25, d_end_hat=10,
        n_jobs=1,                    # bump to 4-8 if joblib available
        cache_path="fd_param_library.npz",
        verbose=True,
    )

    # ---- (iii) load YOUR experimental FD curves ----
    # Replace this stub with your actual loader (see fast-calibration script).
    n_pix = L
    # amplitude_nm = np.empty((drives_si.size, n_pix))
    # phase_deg = np.empty((drives_si.size, n_pix))
    # for i, drv in enumerate(drives_si):
    #     sp = np.linspace(95, 5, n_pix)
    #     amplitude_nm[i] = drv * 1e9 * sp / 100.0
    #     phase_deg[i] = 90.0 + 18.0 * np.exp(-((sp - 70) ** 2) / 200.0) - 55.0 * (sp < 50)
    fd_curves = [
        FDCalibrationCurve(drive_si=float(drives_si[i]),
                            amplitude_nm=amplitude_nm[i], phase_deg=phase_deg[i])
        for i in range(drives_si.size)
    ]

    # ---- (iv) instantiate surrogate calibrator and FIT (seconds!) ----
    cfg = SurrogateCalibrationConfig(
        fit_keys=("E_star", "H", "a0", "phase_offset_deg"),
        phase_weight=1.0, free_phase_weight=2.0, threshold_weight=8.0,
        setpoint_min_pct=30.0, setpoint_max_pct=92.0,
    )
    cal = FDSurrogateCalibrator(lib, fd_curves, cfg, surrogate_method="linear")
    bounds = {
        "E_star": (E_grid.min(), E_grid.max()),
        "H":      (H_grid.min(), H_grid.max()),
        "a0":     (a0_grid.min(), a0_grid.max()),
        "phase_offset_deg": (-20.0, 20.0),
    }

    t0 = time.time()
    res = cal.fit(bounds, method="differential_evolution", de_maxiter=80,
                   seed=0, polish=True, verbose=False)
    print(f"[fit] done in {time.time()-t0:.1f}s    loss={res['loss_best']:.4f}")
    print("theta_best =", res["theta_best"])

    # ---- (v) optional verification: one full-fidelity sim at the optimum ----
    # from fd_only_calibration import FDOnlyCalibrator, FDCalibrationConfig
    # verify_cfg = FDCalibrationConfig(**cfg.__dict__)
    # verify = FDOnlyCalibrator(fd_curves, base_params, conversions, solver, verify_cfg)
    # full_loss = verify.loss(res["theta_best"])
    # print(f"surrogate-loss = {res['loss_best']:.4f}, full-fidelity-loss = {full_loss:.4f}")

    cal.plot_fit(res["theta_best"])
    plt.show()

In [ ]:

# ============================================================
# 5) END-TO-END WORKFLOW EXAMPLE
# ============================================================
if __name__ == "__main__":
    # ---- (i) base setup, identical to your main script ----
    base_params = {
        "k": 35.0, "A": 100e-9, "d0": 100e-9 * 1.25,
        "a0": 0.164e-9, "R": 15e-9, "f0": 300e3, "Q": 175,
        "E_star": 1 / ((1 - 0.3 ** 2) / 130e9 + (1 - 0.3 ** 2) / 1.2e9),
        "H": 7.1e-20,
    }
    conversions = {
        "k": 1, "A": 1e9, "d0": 1e9, "a0": 1e9, "R": 1e9,
        "f0": 1 / (2 * np.pi * base_params["f0"]), "Q": 1,
        "E_star": 1e-9 / base_params["k"],
        "H": 1 / (base_params["k"] * (1e-9) ** 2),
    }
    solver = RK45Options(N_cycles=15, max_windows=25, settle_windows=1,
                          tol_A=2e-4, tol_phi_deg=2e-2, consecutive=2,
                          demod_per_ref_cycle=24, rtol=1e-5, atol=1e-8,
                          dt0=0.02, dt_min=1e-10, dt_max=0.3,
                          max_steps_per_window=120000)

    # drives_si = np.array([7, 13, 18, 24, 29, 34, 39, 45, 50, 53,
    #                       66, 79, 92, 105, 117], dtype=float) * 1e-9

    # ---- (ii) BUILD library ONCE (cached after first run) ----
    # Use a SUBSET of 5-6 drives for material fitting; library stores them all.
    E0 = base_params["E_star"]
    H0 = base_params["H"]
    a0_0 = base_params["a0"]
    E_grid = np.geomspace(E0 * 0.4, E0 * 2.5, 11)     # log-spaced (~3 decades robust)
    H_grid = np.geomspace(H0 * 0.4, H0 * 2.5, 11)
    a0_grid = np.linspace(a0_0 * 0.6, a0_0 * 1.8, 11)  # linear
    lib = build_fd_param_library(
        base_params=base_params, conversions=conversions,
        E_grid=E_grid, H_grid=H_grid, a0_grid=a0_grid,
        drives_si=drives_si, solver=solver,
        n_d_hat=101, d_start_factor=1.25, d_end_hat=10,
        n_jobs=8,                    # bump to 4-8 if joblib available
        cache_path=None,
        verbose=True,
    )

    # ---- (iii) load YOUR experimental FD curves ----
    # Replace this stub with your actual loader (see fast-calibration script).
    n_pix = L
    # amplitude_nm = np.empty((drives_si.size, n_pix))
    # phase_deg = np.empty((drives_si.size, n_pix))
    # for i, drv in enumerate(drives_si):
    #     sp = np.linspace(95, 5, n_pix)
    #     amplitude_nm[i] = drv * 1e9 * sp / 100.0
    #     phase_deg[i] = 90.0 + 18.0 * np.exp(-((sp - 70) ** 2) / 200.0) - 55.0 * (sp < 50)
    fd_curves = [
        FDCalibrationCurve(drive_si=float(drives_si[i]),
                            amplitude_nm=amplitude_nm[i], phase_deg=phase_deg[i])
        for i in range(drives_si.size)
    ]

    # ---- (iv) instantiate surrogate calibrator and FIT (seconds!) ----
    cfg = SurrogateCalibrationConfig(
        fit_keys=("E_star", "H", "a0", "phase_offset_deg"),
        phase_weight=1.0, free_phase_weight=2.0, threshold_weight=8.0,
        setpoint_min_pct=30.0, setpoint_max_pct=92.0,
    )
    cal = FDSurrogateCalibrator(lib, fd_curves, cfg, surrogate_method="linear")
    bounds = {
        "E_star": (E_grid.min(), E_grid.max()),
        "H":      (H_grid.min(), H_grid.max()),
        "a0":     (a0_grid.min(), a0_grid.max()),
        "phase_offset_deg": (-20.0, 20.0),
    }

    t0 = time.time()
    res = cal.fit(bounds, method="differential_evolution", de_maxiter=80,
                   seed=0, polish=True, verbose=False)
    print(f"[fit] done in {time.time()-t0:.1f}s    loss={res['loss_best']:.4f}")
    print("theta_best =", res["theta_best"])

    # ---- (v) optional verification: one full-fidelity sim at the optimum ----
    # from fd_only_calibration import FDOnlyCalibrator, FDCalibrationConfig
    # verify_cfg = FDCalibrationConfig(**cfg.__dict__)
    # verify = FDOnlyCalibrator(fd_curves, base_params, conversions, solver, verify_cfg)
    # full_loss = verify.loss(res["theta_best"])
    # print(f"surrogate-loss = {res['loss_best']:.4f}, full-fidelity-loss = {full_loss:.4f}")

    cal.plot_fit(res["theta_best"])
    plt.show()

In [ ]:
params

In [ ]:
res['theta_best']

In [ ]:
drives_si = drives_si[:-1]
len(drives_si)

In [ ]:
fd_curves = [
    FDCalibrationCurve(drive_si=float(drives_si[i]),
                        amplitude_nm=amplitude_nm[i], phase_deg=phase_deg[i])
    for i in range(drives_si.size)
]

# ---- (iv) instantiate surrogate calibrator and FIT (seconds!) ----
cfg = SurrogateCalibrationConfig(
    fit_keys=("E_star", "H", "a0", "phase_offset_deg"),
    phase_weight=1.0, free_phase_weight=1.0, threshold_weight=8.0,
    setpoint_min_pct=30.0, setpoint_max_pct=92.0,
)
cal = FDSurrogateCalibrator(lib, fd_curves, cfg, surrogate_method="linear")
bounds = {
    "E_star": (E_grid.min(), E_grid.max()),
    "H":      (H_grid.min(), H_grid.max()),
    "a0":     (a0_grid.min(), a0_grid.max()),
    "phase_offset_deg": (-20.0, 20.0),
}

t0 = time.time()
res = cal.fit(bounds, method="differential_evolution", de_maxiter=80,
               seed=0, polish=True, verbose=False)
print(f"[fit] done in {time.time()-t0:.1f}s    loss={res['loss_best']:.4f}")
print("theta_best =", res["theta_best"])

# ---- (v) optional verification: one full-fidelity sim at the optimum ----
# from fd_only_calibration import FDOnlyCalibrator, FDCalibrationConfig
# verify_cfg = FDCalibrationConfig(**cfg.__dict__)
# verify = FDOnlyCalibrator(fd_curves, base_params, conversions, solver, verify_cfg)
# full_loss = verify.loss(res["theta_best"])
# print(f"surrogate-loss = {res['loss_best']:.4f}, full-fidelity-loss = {full_loss:.4f}")

cal.plot_fit(res["theta_best"])
plt.xlim(40, 120)
plt.show()

In [ ]:
lib   = build_fd_param_library(base_params, conversions, E_grid, H_grid, a0_grid,
                                drives_si, solver=solver, n_jobs=8,
                                cache_path="fd_param_library.npz")
cal   = FDSurrogateCalibrator(lib, fd_curves, cfg)
result = cal.fit(bounds, method="differential_evolution", de_maxiter=80)

In [ ]:
# from codes.fd_priors.py import make_default_prior
# from codes.fd_surrogate_calibration.py import build_fd_param_library, FDSurrogateCalibrator, SurrogateCalibrationConfig


In [ ]:
from codes.fd_priors import make_default_prior
from codes.fd_surrogate_calibration import build_fd_param_library, FDSurrogateCalibrator, SurrogateCalibrationConfig

prior  = make_default_prior(params)
grids  = prior.grids(("E_star","H","a0"), n_points=6, n_sigma=2.5)
bounds = prior.bounds(("E_star","H","a0","phase_offset_deg"), n_sigma=3.0)
x0     = prior.x0(    ("E_star","H","a0","phase_offset_deg"))

lib = build_fd_param_library(
    base_params=params, conversions=conversions,
    E_grid=grids["E_star"], H_grid=grids["H"], a0_grid=grids["a0"],
    drives_si=drives_si, solver=solver,
    cache_path="fd_param_library.npz",
)

cal = FDSurrogateCalibrator(lib, fd_curves, SurrogateCalibrationConfig())
cal.loss = prior.wrap_loss(cal.loss, weight=0.05)        # soft prior penalty
res = cal.fit(bounds=bounds, x0=x0, method="L-BFGS-B")   # warm-started local fit

In [ ]:
"""
fd_drive_transformation.py
==========================

With ALL material parameters held fixed (E_star, H, a0, R, Q assumed
correct), find the transformation

        drive_sim = f(drive_exp)

that makes the simulator agree with the experimental FD curves.

Why this is now a one-dimensional problem
-----------------------------------------
Material/cantilever physics is locked in.  The simulator's free
amplitude, attractive lobe, bistability threshold, and repulsive slope
are all *determined functions of drive*.  So for each experimental
curve at drive_exp_i, there is a single drive_sim_i that best
reproduces it.  Aggregating those pairs and fitting a parametric form
gives the calibration.

Workflow
--------
    1. Build a 1-D drive surrogate (FD curves at many drive values, fixed material).
       Cached to .npz.
    2. For each experimental curve, 1-D-minimise over drive_sim against
       the same loss decomposition you've been using (phase curve + free
       phase + threshold).
    3. Fit drive_sim_i = f(drive_exp_i) — try 'proportional', 'affine',
       and 'polynomial' models, pick the simplest one that captures the
       data.
    4. Optional joint refinement: re-fit the transformation parameters
       by minimising the total loss over all curves simultaneously.
"""

from __future__ import annotations

import os
import time
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, minimize_scalar

# from spm_fd_multidrive_alignment import (
#     RK45Options,
#     SPMRK45ApproachModel,
#     build_normalized_spm_params,
# )
# from fd_only_calibration import (
#     FDCalibrationCurve,
#     detect_threshold_setpoint,
# )


# ============================================================
# 1) 1-D DRIVE SURROGATE
# ============================================================
@dataclass
class Drive1DSurrogate:
    """Pre-computed FD curves on a dense drive grid, material held fixed."""
    drives_si: np.ndarray            # (n_drv,) ascending, in m
    d_hat_grid: np.ndarray           # (n_d_hat,) ascending
    A_table: np.ndarray              # (n_drv, n_d_hat) -- NaN where out of range
    phi_table: np.ndarray            # (n_drv, n_d_hat)
    fixed_params: Dict[str, float]
    conversions: Dict[str, float]
    attrs: Dict[str, Any] = field(default_factory=dict)

    # ---- linear interp in drive ----
    def evaluate(self, drive_si: float) -> Tuple[np.ndarray, np.ndarray]:
        d = float(drive_si)
        drives = self.drives_si
        if d <= drives[0]:
            return self.A_table[0].copy(), self.phi_table[0].copy()
        if d >= drives[-1]:
            return self.A_table[-1].copy(), self.phi_table[-1].copy()
        idx = int(np.searchsorted(drives, d) - 1)
        f = (d - drives[idx]) / (drives[idx + 1] - drives[idx])
        A = (1 - f) * self.A_table[idx] + f * self.A_table[idx + 1]
        phi = (1 - f) * self.phi_table[idx] + f * self.phi_table[idx + 1]
        return A, phi

    # ---- save / load ----
    def save(self, path: str) -> None:
        np.savez_compressed(
            path,
            drives_si=self.drives_si, d_hat_grid=self.d_hat_grid,
            A_table=self.A_table, phi_table=self.phi_table,
            fixed_params_keys=np.array(list(self.fixed_params.keys())),
            fixed_params_vals=np.array(list(self.fixed_params.values()), dtype=float),
            conversions_keys=np.array(list(self.conversions.keys())),
            conversions_vals=np.array(list(self.conversions.values()), dtype=float),
        )

    @classmethod
    def load(cls, path: str) -> "Drive1DSurrogate":
        z = np.load(path, allow_pickle=False)
        fp = {str(k): float(v) for k, v in zip(z["fixed_params_keys"], z["fixed_params_vals"])}
        cv = {str(k): float(v) for k, v in zip(z["conversions_keys"], z["conversions_vals"])}
        return cls(
            drives_si=z["drives_si"], d_hat_grid=z["d_hat_grid"],
            A_table=z["A_table"], phi_table=z["phi_table"],
            fixed_params=fp, conversions=cv,
        )


def build_drive1d_surrogate(
    fixed_params: Dict[str, float],
    conversions: Dict[str, float],
    drives_si: Sequence[float],
    solver: Optional[RK45Options] = None,
    n_d_hat: int = 140,
    d_start_factor: float = 1.25,
    d_end_hat: float = -7.0,
    cache_path: Optional[str] = None,
    verbose: bool = True,
) -> Drive1DSurrogate:
    """Build a 1-D drive surrogate with material parameters held fixed."""
    if cache_path is not None and os.path.exists(cache_path):
        if verbose:
            print(f"[surrogate] loading cached surrogate from {cache_path}")
        return Drive1DSurrogate.load(cache_path)

    drives_si = np.sort(np.asarray(list(drives_si), dtype=float))
    if solver is None:
        solver = RK45Options()

    # Shared ascending d_hat grid sized to the largest drive.
    largest_drive_hat = float(drives_si.max() * conversions["A"])
    d_global = np.linspace(float(d_end_hat),
                            largest_drive_hat * float(d_start_factor),
                            int(n_d_hat))
    A_table = np.full((drives_si.size, n_d_hat), np.nan)
    phi_table = np.full_like(A_table, np.nan)

    t0 = time.time()
    for i, drv in enumerate(drives_si):
        spm_params, param_norm = build_normalized_spm_params(
            params=fixed_params, conversions=conversions, drive_si=float(drv),
            omega_drive_hat=1.0, omega_ref_hat=1.0,
        )
        model = SPMRK45ApproachModel(spm_params, solver=solver)
        d0 = float(param_norm["A"]) * float(d_start_factor)
        curve = model.approach_curve(
            d_start_hat=d0, d_end_hat=float(d_end_hat),
            n_points=int(n_d_hat), z0_hat=0.0, v0_hat=0.0, warm_start=True,
        )
        order = np.argsort(curve["d_hat"])
        d_sorted = curve["d_hat"][order]
        A_sorted = curve["A_hat"][order]
        phi_sorted = curve["phi_deg"][order]
        A_table[i] = np.interp(d_global, d_sorted, A_sorted, left=np.nan, right=np.nan)
        phi_table[i] = np.interp(d_global, d_sorted, phi_sorted, left=np.nan, right=np.nan)
        if verbose:
            print(f"  [{i+1}/{drives_si.size}] drive={drv*1e9:6.2f} nm  "
                  f"({(time.time()-t0):.1f}s)")
    if verbose:
        print(f"[surrogate] built in {time.time()-t0:.1f}s")

    surr = Drive1DSurrogate(
        drives_si=drives_si, d_hat_grid=d_global,
        A_table=A_table, phi_table=phi_table,
        fixed_params=dict(fixed_params), conversions=dict(conversions),
        attrs={"d_start_factor": float(d_start_factor),
               "d_end_hat": float(d_end_hat), "n_d_hat": int(n_d_hat)},
    )
    if cache_path is not None:
        surr.save(cache_path)
        if verbose:
            print(f"[surrogate] saved to {cache_path}")
    return surr


# ============================================================
# 2) PER-CURVE LOSS  (phase + free-phase + threshold)
# ============================================================
@dataclass
class DriveLossConfig:
    setpoint_min_pct: float = 30.0
    setpoint_max_pct: float = 92.0
    threshold_phase_deg: float = 90.0
    phase_weight: float = 1.0
    free_phase_weight: float = 2.0
    threshold_weight: float = 6.0


def _curve_to_setpoint(d_hat, A_hat, phi):
    """Mask NaN, compute setpoint% = A/A_free*100."""
    finite = np.isfinite(A_hat) & np.isfinite(phi) & np.isfinite(d_hat)
    if not np.any(finite):
        return None, None, None, np.nan
    d = d_hat[finite]
    A = A_hat[finite]
    p = phi[finite]
    far = int(np.argmax(d))
    A_free = float(A[far])
    if A_free <= 0 or not np.isfinite(A_free):
        return None, None, None, np.nan
    return A / A_free * 100.0, p, d, A_free


def _find_sim_threshold(sp, ph, d, threshold_phase_deg=90.0):
    order = np.argsort(-d)
    sp_o, ph_o = sp[order], ph[order]
    for i in range(1, ph_o.size):
        if ph_o[i - 1] >= threshold_phase_deg > ph_o[i]:
            if ph_o[i] == ph_o[i - 1]:
                return float(0.5 * (sp_o[i - 1] + sp_o[i]))
            t = (threshold_phase_deg - ph_o[i - 1]) / (ph_o[i] - ph_o[i - 1])
            return float(sp_o[i - 1] + t * (sp_o[i] - sp_o[i - 1]))
    return np.nan


def per_curve_drive_loss(
    surrogate: Drive1DSurrogate,
    drive_sim: float,
    fd: FDCalibrationCurve,
    phase_offset_deg: float = 0.0,
    cfg: Optional[DriveLossConfig] = None,
) -> float:
    """Loss between sim curve at `drive_sim` and one experimental curve."""
    if cfg is None:
        cfg = DriveLossConfig()
    A, phi = surrogate.evaluate(drive_sim)
    sp_sim, phi_sim, d_arr, A_free = _curve_to_setpoint(
        surrogate.d_hat_grid, A, phi + phase_offset_deg
    )
    if sp_sim is None:
        return 1e20

    total = 0.0
    n = 0

    # phase curve term
    if cfg.phase_weight > 0:
        m = (fd.setpoint_pct >= cfg.setpoint_min_pct) & (fd.setpoint_pct <= cfg.setpoint_max_pct)
        if np.any(m):
            order = np.argsort(sp_sim)
            sp_s, ph_s = sp_sim[order], phi_sim[order]
            keep = np.concatenate([[True], np.diff(sp_s) > 1e-9])
            phi_at = np.interp(fd.setpoint_pct[m], sp_s[keep], ph_s[keep],
                                left=np.nan, right=np.nan)
            valid = np.isfinite(phi_at)
            if np.any(valid):
                r = phi_at[valid] - fd.phase_deg[m][valid]
                total += cfg.phase_weight * float(np.mean(r * r))
                n += 1

    # free phase term
    if cfg.free_phase_weight > 0:
        far_phi = float(phi_sim[int(np.argmax(d_arr))])
        r = far_phi - float(fd.free_phase_deg)
        total += cfg.free_phase_weight * float(r * r)
        n += 1

    # threshold term
    if cfg.threshold_weight > 0 and fd.threshold_setpoint_pct is not None:
        sp_th_sim = _find_sim_threshold(sp_sim, phi_sim, d_arr, cfg.threshold_phase_deg)
        if np.isfinite(sp_th_sim):
            r = sp_th_sim - float(fd.threshold_setpoint_pct)
            total += cfg.threshold_weight * float(r * r)
            n += 1

    return float(total / max(n, 1))


# ============================================================
# 3) PER-CURVE BEST-DRIVE SEARCH
# ============================================================
def find_drive_sim_per_curve(
    surrogate: Drive1DSurrogate,
    fd_curves: Sequence[FDCalibrationCurve],
    phase_offset_deg: float = 0.0,
    cfg: Optional[DriveLossConfig] = None,
    search_lo_factor: float = 0.5,
    search_hi_factor: float = 2.0,
    verbose: bool = True,
) -> Dict[str, np.ndarray]:
    """For each curve, find the drive_sim that minimises the per-curve loss."""
    if cfg is None:
        cfg = DriveLossConfig()
    drives_exp = np.array([fd.drive_si for fd in fd_curves], dtype=float)
    drives_sim_best = np.empty_like(drives_exp)
    losses = np.empty_like(drives_exp)

    drv_lo = surrogate.drives_si.min()
    drv_hi = surrogate.drives_si.max()

    for k, fd in enumerate(fd_curves):
        # bounded scalar minimization within the surrogate range
        # (centered on the experimental drive)
        lo = max(drv_lo, fd.drive_si * search_lo_factor)
        hi = min(drv_hi, fd.drive_si * search_hi_factor)
        if lo >= hi:
            lo, hi = drv_lo, drv_hi

        def obj(d):
            return per_curve_drive_loss(surrogate, float(d), fd,
                                         phase_offset_deg=phase_offset_deg, cfg=cfg)

        res = minimize_scalar(obj, bounds=(lo, hi), method="bounded",
                               options={"xatol": 1e-13})
        drives_sim_best[k] = float(res.x)
        losses[k] = float(res.fun)
        if verbose:
            print(f"  [{k+1:2d}/{len(fd_curves)}] drive_exp={fd.drive_si*1e9:7.2f} nm  "
                  f"-> drive_sim={drives_sim_best[k]*1e9:7.2f} nm   loss={losses[k]:.4f}")
    return {"drives_exp": drives_exp,
            "drives_sim": drives_sim_best,
            "losses": losses}


# ============================================================
# 4) FIT A PARAMETRIC DRIVE TRANSFORMATION
# ============================================================
@dataclass
class DriveTransformation:
    model: str                       # 'proportional' | 'affine' | 'polynomial'
    coeffs: np.ndarray               # ascending order (c0, c1, c2, ...)

    def apply(self, drive_exp):
        d = np.asarray(drive_exp, dtype=float)
        if self.model == "proportional":
            return self.coeffs[0] * d
        if self.model == "affine":
            return self.coeffs[0] + self.coeffs[1] * d
        if self.model == "polynomial":
            return np.polyval(self.coeffs[::-1], d)
        raise ValueError(f"unknown model: {self.model}")

    def __repr__(self):
        if self.model == "proportional":
            return f"d_sim = {self.coeffs[0]:.5f} * d_exp"
        if self.model == "affine":
            return f"d_sim = {self.coeffs[1]:.5f} * d_exp + ({self.coeffs[0]:+.4e})"
        terms = " + ".join([f"{c:.4e}*d_exp^{i}" for i, c in enumerate(self.coeffs)])
        return f"d_sim = {terms}"


def fit_drive_transformation(
    drives_exp: np.ndarray,
    drives_sim: np.ndarray,
    model: str = "affine",
    degree: int = 2,
) -> Tuple[DriveTransformation, Dict[str, float]]:
    """
    Fit drive_sim = f(drive_exp) via least squares.

    Returns the fit and a small stats dict with R^2 and residual MAD.
    """
    de = np.asarray(drives_exp, dtype=float)
    ds = np.asarray(drives_sim, dtype=float)

    if model == "proportional":
        a = float(np.dot(de, ds) / np.dot(de, de))
        coeffs = np.array([a])
        pred = a * de
    elif model == "affine":
        a, b = np.polyfit(de, ds, 1)
        coeffs = np.array([float(b), float(a)])      # [intercept, slope]
        pred = a * de + b
    elif model == "polynomial":
        c_desc = np.polyfit(de, ds, int(degree))
        coeffs = c_desc[::-1].astype(float)
        pred = np.polyval(c_desc, de)
    else:
        raise ValueError(model)

    resid = ds - pred
    ss_res = float(np.sum(resid ** 2))
    ss_tot = float(np.sum((ds - np.mean(ds)) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    mad = float(np.median(np.abs(resid - np.median(resid))) * 1.4826)
    rmse = float(np.sqrt(np.mean(resid ** 2)))
    return (DriveTransformation(model, coeffs),
            {"r2": r2, "rmse": rmse, "mad": mad,
             "max_abs_resid": float(np.max(np.abs(resid)))})


# ============================================================
# 5) JOINT REFINEMENT (refines transformation params against total loss)
# ============================================================
def joint_refine_transformation(
    surrogate: Drive1DSurrogate,
    fd_curves: Sequence[FDCalibrationCurve],
    initial: DriveTransformation,
    phase_offset_deg: float = 0.0,
    cfg: Optional[DriveLossConfig] = None,
    method: str = "Powell",
) -> Tuple[DriveTransformation, float]:
    """Refine transformation coefficients by minimising the sum of per-curve losses."""
    if cfg is None:
        cfg = DriveLossConfig()
    model = initial.model
    x0 = initial.coeffs.copy()
    drives_exp = np.array([fd.drive_si for fd in fd_curves], dtype=float)
    drv_lo, drv_hi = surrogate.drives_si.min(), surrogate.drives_si.max()

    def obj(x):
        t = DriveTransformation(model, np.asarray(x, dtype=float))
        ds = t.apply(drives_exp)
        if np.any(ds < drv_lo * 0.5) or np.any(ds > drv_hi * 1.5):
            return 1e20
        total, n = 0.0, 0
        for k, fd in enumerate(fd_curves):
            total += per_curve_drive_loss(surrogate, float(ds[k]), fd,
                                            phase_offset_deg=phase_offset_deg, cfg=cfg)
            n += 1
        return total / max(n, 1)

    res = minimize(obj, x0, method=method)
    return DriveTransformation(model, np.asarray(res.x, dtype=float)), float(res.fun)


# ============================================================
# 6) DIAGNOSTIC PLOT
# ============================================================
def plot_drive_transformation(
    drives_exp: np.ndarray,
    drives_sim: np.ndarray,
    transform: Optional[DriveTransformation] = None,
    losses: Optional[np.ndarray] = None,
    figsize: Tuple[float, float] = (11, 4.5),
):
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    ax = axes[0]
    de_nm = drives_exp * 1e9
    ds_nm = drives_sim * 1e9
    ax.plot(de_nm, ds_nm, "ro", ms=8, label="per-curve best d_sim")
    lo, hi = de_nm.min(), de_nm.max()
    pad = 0.05 * (hi - lo + 1e-9)
    line = np.linspace(lo - pad, hi + pad, 100)
    ax.plot(line, line, "k:", lw=1.0, label="identity (slope=1)")
    if transform is not None:
        ax.plot(line, transform.apply(line * 1e-9) * 1e9, "b--", lw=1.6,
                label=f"fit: {transform}")
    ax.set_xlabel("Experimental drive (nm)")
    ax.set_ylabel("Simulator-equivalent drive (nm)")
    ax.set_title("Drive transformation")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

    ax = axes[1]
    if transform is None:
        resid = ds_nm - de_nm
        ax.set_ylabel("d_sim - d_exp (nm)")
    else:
        pred_nm = transform.apply(drives_exp) * 1e9
        resid = ds_nm - pred_nm
        ax.set_ylabel("d_sim - fit(d_exp) (nm)")
    ax.axhline(0, color="k", lw=0.8)
    ax.plot(de_nm, resid, "ks", ms=7, label="residuals")
    if losses is not None:
        ax2 = ax.twinx()
        ax2.plot(de_nm, losses, "g+", ms=10, label="per-curve loss")
        ax2.set_ylabel("per-curve loss", color="g")
        ax2.tick_params(axis="y", colors="g")
    ax.set_xlabel("Experimental drive (nm)")
    ax.set_title("Residuals")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left", fontsize=9)

    fig.tight_layout()
    return fig, axes




In [ ]:
# from fd_drive_transformation import (
#     build_drive1d_surrogate, find_drive_sim_per_curve,
#     fit_drive_transformation, joint_refine_transformation,
#     plot_drive_transformation, DriveLossConfig,
# )

# 1. one-time surrogate build (~30 sims at fixed material)
surr = build_drive1d_surrogate(
    fixed_params=params, conversions=conversions,
    drives_si=np.geomspace(5e-9, 250e-9, 51),
    solver=solver, cache_path=None,
)

# 2. per-curve best-drive search
out = find_drive_sim_per_curve(surr, fd_curves,
                                phase_offset_deg=-13)

# 3. fit a parametric form to the (drive_exp, drive_sim) pairs
for model in ("proportional", "affine", "polynomial"):
    t, stats = fit_drive_transformation(out["drives_exp"], out["drives_sim"], model=model, degree=7)
    print(f"{model}: {t}  R^2={stats['r2']:.4f}")

In [ ]:
def transform_d(d_exp):
    d_sim = -1.2341e-07*d_exp**0 + \
    3.5079e+01*d_exp**1 + \
    -3.3320e+09*d_exp**2 + \
    1.5728e+17*d_exp**3 + \
    -3.9171e+24*d_exp**4 + \
    5.2492e+31*d_exp**5 + \
    -3.5737e+38*d_exp**6 + \
    9.6856e+44*d_exp**7  
    return d_sim

In [ ]:
plt.plot(drives_si*1e9, transform_d(drives_si)*1e9, '.-')

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
de_nm = out["drives_exp"] * 1e9
ds_nm = out["drives_sim"] * 1e9
axes[0].plot(de_nm, ds_nm, "ro", ms=8)
axes[0].plot(de_nm, de_nm, "k:", label="identity")
axes[0].set_xlabel("drive_exp (nm)"); axes[0].set_ylabel("drive_sim (nm)")
axes[0].grid(alpha=0.3); axes[0].legend()
axes[1].plot(de_nm, out["losses"], "ks-")
axes[1].set_xlabel("drive_exp (nm)"); axes[1].set_ylabel("per-curve loss")
axes[1].set_yscale("log"); axes[1].grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

# also: did any drive_sim hit the search boundary?
lo, hi = surr.drives_si.min(), surr.drives_si.max()
hit_lo = out["drives_sim"] < lo * 1.02
hit_hi = out["drives_sim"] > hi * 0.98
print("hit lower bound:", np.where(hit_lo)[0])
print("hit upper bound:", np.where(hit_hi)[0])

In [ ]:
# ============================================================
# 7) END-TO-END EXAMPLE
# ============================================================
if __name__ == "__main__":
    # Material parameters assumed correct (replace with your fit / handbook).
    fixed_params = {
        "k": 1.0, "A": 100e-9, "d0": 100e-9 * 1.25,
        "a0": 0.164e-9, "R": 20e-9, "f0": 75e3, "Q": 200,
        "E_star": 1 / ((1 - 0.3 ** 2) / 130e9 + (1 - 0.3 ** 2) / 1.2e9),
        "H": 7.1e-20,
    }
    conversions = {
        "k": 1, "A": 1e9, "d0": 1e9, "a0": 1e9, "R": 1e9,
        "f0": 1 / (2 * np.pi * fixed_params["f0"]), "Q": 1,
        "E_star": 1e-9 / fixed_params["k"],
        "H": 1 / (fixed_params["k"] * (1e-9) ** 2),
    }
    solver = RK45Options(
        N_cycles=15, max_windows=25, settle_windows=1,
        tol_A=2e-4, tol_phi_deg=2e-2, consecutive=2,
        demod_per_ref_cycle=24, rtol=1e-5, atol=1e-8,
        dt0=0.02, dt_min=1e-10, dt_max=0.3, max_steps_per_window=120000,
    )

    # ---- (i) Build a 1-D drive surrogate on a DENSE drive grid ----
    drives_surr = np.geomspace(3e-9, 150e-9, 30)        # 30 drives, ~5 minutes
    surr = build_drive1d_surrogate(
        fixed_params=fixed_params, conversions=conversions,
        drives_si=drives_surr, solver=solver,
        n_d_hat=140, d_start_factor=1.25, d_end_hat=-7.0,
        cache_path="drive1d_surrogate.npz", verbose=True,
    )

    # ---- (ii) Load YOUR experimental FD curves ----
    # Stub data so the file runs standalone; replace with your loader.
    drives_exp_si = np.array(
        [7, 13, 18, 24, 29, 34, 39, 45, 50, 53, 66, 79, 92, 105, 117], dtype=float
    ) * 1e-9
    n_pix = 1007
    fd_curves: List[FDCalibrationCurve] = []
    for drv in drives_exp_si:
        sp = np.linspace(95, 5, n_pix)
        amp = drv * 1e9 * sp / 100.0
        ph = 90.0 + 18.0 * np.exp(-((sp - 70) ** 2) / 200.0) - 55.0 * (sp < 50)
        fd_curves.append(FDCalibrationCurve(drive_si=float(drv),
                                              amplitude_nm=amp, phase_deg=ph))

    # ---- (iii) Per-curve best-drive search ----
    cfg = DriveLossConfig(setpoint_min_pct=30.0, setpoint_max_pct=92.0,
                           threshold_phase_deg=90.0,
                           phase_weight=1.0, free_phase_weight=2.0, threshold_weight=8.0)
    phase_offset = 0.0       # set from your fast-calibration result
    out = find_drive_sim_per_curve(surr, fd_curves,
                                     phase_offset_deg=phase_offset, cfg=cfg)

    # ---- (iv) Fit parametric transformations and compare ----
    print("\nFit comparison:")
    for model in ("proportional", "affine"):
        t, stats = fit_drive_transformation(out["drives_exp"], out["drives_sim"], model=model)
        print(f"  {model:>13}:  {t}      R^2={stats['r2']:.4f}  RMSE={stats['rmse']*1e9:.3f} nm")

    # default to affine (most common useful case)
    transform, stats = fit_drive_transformation(out["drives_exp"], out["drives_sim"], model="affine")

    # ---- (v) Optional joint refinement ----
    refined, refined_loss = joint_refine_transformation(
        surr, fd_curves, initial=transform, phase_offset_deg=phase_offset, cfg=cfg,
    )
    print(f"\nJoint-refined transform: {refined}   total loss = {refined_loss:.4f}")

    # ---- (vi) Plot ----
    plot_drive_transformation(out["drives_exp"], out["drives_sim"],
                                transform=refined, losses=out["losses"])
    plt.show()

    # ---- (vii) How to use the result downstream ----
    # In your scanner / library builder, replace the raw experimental drive
    # by transform.apply(drive_exp_in_m):
    #     drive_si_for_simulator = refined.apply(drive_exp_si)
    # This becomes the drive value passed to ScannerFD2D / FDLibrary2D.
